
## UW-Whitewater Scouting Parser (portable version)

Parses UW-Whitewater's own schedule/box-score, each scouted opponent's schedule and FastScout "ScoutBuilder" game-plan PDF, and (further down) play-by-play + video-tagging exports -- turning them into player/lineup-level scouting data, cross-checked against actual results. Ported from a Databricks notebook to run locally via Databricks Connect (Playwright/asyncio/subprocess-based scraping runs on the local machine, not the remote cluster); Databricks-only APIs (`dbutils`, `display()`, widgets, Spark table writes) were swapped for plain Python equivalents everywhere.

### Configuration

Edit `INPUT_DIR`/`OUTPUT_DIR`/`USE_LLM`/`reference_date_str` below for your local setup -- everything downstream reads from these instead of hardcoded paths or Databricks widgets.

In [2]:
# UW-Whitewater Schedule and Scout Report Parser -- Portable Version
#
# Converted from the Databricks notebook. All Databricks-specific dependencies replaced:
# - /Volumes/... and /Workspace/... paths -> configurable INPUT_DIR / OUTPUT_DIR below
# - dbutils.widgets -> plain variables below (edit these directly -- no widget UI outside Databricks)
# - spark.createDataFrame / saveAsTable -> removed (CSV export only)
# - display() -> print()
# - dbutils -> removed
# - ai_query() -> OpenAI client (via player_comparison module, when USE_LLM is enabled)

import email
import glob
import json
import logging
import math
import os
import re
import sys
from collections import Counter
from datetime import datetime
from email import policy
from io import StringIO

import pandas as pd
from bs4 import BeautifulSoup

# --- Configuration -- edit these to match your local setup ---
INPUT_DIR = "./inputs"    # directory containing MHTML/PDF input files
OUTPUT_DIR = "../data"    # directory to write CSV output files
USE_LLM = False            # set True to enable LLM-based player comparisons (requires OPENAI_API_KEY)
reference_date_str = "2025-11-19"   # games on/after this date are treated as not yet played; the first
                                     # scouted UWW game on/after it is flagged as the upcoming game
reference_date = datetime.strptime(reference_date_str, "%Y-%m-%d")
before_scout = "yes"      # "yes" = run as if the UPCOMING opponent's own scouting report does not exist yet,
                         # even when a "*_scout.html"/"*_scout.pdf" file for that game IS sitting in
                         # INPUT_DIR. Reports for games BEFORE the upcoming one are still used normally, and
                         # nothing is deleted from disk -- the upcoming game's report is simply filtered out
                         # of every scout-file lookup (and never re-downloaded) for this run. "no" = normal
                         # behaviour: use every report found.
_before_scout_enabled = str(before_scout).strip().lower() in {"yes", "y", "true", "1"}

os.makedirs(OUTPUT_DIR, exist_ok=True)


# --- Scout-report file discovery (honours `before_scout` above) ------------------------------------------
# Every cell that looks for scouting reports goes through find_scout_files() instead of globbing
# "*_scout.*" directly, so the `before_scout` switch has exactly ONE place to take effect rather than
# needing the same filter repeated (and kept in sync) at each glob site -- the "two places that both claim
# to mean the same thing" pattern that has bitten this notebook before.
_SCOUT_FILENAME_DATE_RE = re.compile(r"^(\d{1,2})_(\d{1,2})_(\d{2,4})\s")


def scout_file_game_date(path):
    """Parse the game date out of a "<M>_<D>_<YY> <Away> @ <Home>_scout.<ext>" filename.

    Returns a datetime, or None when the filename doesn't carry a parseable date."""
    m = _SCOUT_FILENAME_DATE_RE.match(os.path.basename(path))
    if not m:
        return None
    month, day, year = (int(g) for g in m.groups())
    if year < 100:
        year += 2000
    try:
        return datetime(year, month, day)
    except ValueError:
        return None


def find_scout_files(directory, extensions=("pdf", "html")):
    """Glob the scouting reports in `directory`, dropping the upcoming game's own report when
    `before_scout` is "yes".

    "Upcoming game" is identified by DATE rather than by opponent name, because this runs before the
    upcoming opponent has been resolved: reference_date is by definition the cutoff for "not yet played",
    so any report whose filename date is on/after it belongs to the upcoming matchup (or a later one that
    is out of scope for this run anyway). Reports dated strictly before reference_date -- the opponent's
    earlier games, and every previously-scouted opponent -- are always kept."""
    paths = []
    for ext in extensions:
        paths.extend(glob.glob(f"{directory}/*_scout.{ext}"))
    paths = sorted(paths)
    if not _before_scout_enabled:
        return paths

    kept, dropped, undated = [], [], []
    for p in paths:
        game_date = scout_file_game_date(p)
        if game_date is None:
            # Fail open: an undated filename can't be attributed to a specific game, so keep it rather
            # than silently dropping a report that may belong to a past opponent.
            undated.append(p)
            kept.append(p)
        elif game_date >= reference_date:
            dropped.append(p)
        else:
            kept.append(p)
    if dropped:
        print(f"  [before_scout=yes] Ignoring {len(dropped)} scouting report(s) dated on/after "
              f"{reference_date_str}: {[os.path.basename(p) for p in dropped]}")
    if undated:
        print(f"  [before_scout=yes] WARNING: could not read a game date from "
              f"{[os.path.basename(p) for p in undated]} -- keeping these; rename them to the "
              f"'<M>_<D>_<YY> <Away> @ <Home>_scout.<ext>' convention if one is the upcoming game's report.")
    return kept


print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"LLM comparisons: {'enabled' if USE_LLM else 'disabled'}")
print(f"Reference date: {reference_date_str}")
print(f"Before scout: {before_scout} -- "
      + ("ignoring the upcoming opponent's own scouting report for this run"
         if _before_scout_enabled else "using every scouting report found"))

# Hosted URL of the Streamlit scouting app (e.g. "https://uww-scouting.streamlit.app"). The scouting brief links
# every condensed section into the app with it; leave blank and the brief prints "(in the app)" instead of links.
#
# SET THIS. With it blank, EVERY link in the emailed brief is dead text -- player names, "Clips, actions and
# spots", "All of our lineups", all of it. The staff gets a brief that says "(in the app)" 17+ times with no
# way to get there. Put the deployed URL in the string below (it wins over the environment variable), or
# export UWW_APP_URL before running.
APP_BASE_URL = "" or os.environ.get("UWW_APP_URL", "")

if APP_BASE_URL:
    # A bare host is a RELATIVE href once it lands in the brief -- the browser resolves it against
    # wherever the file sits, so the link opens file:///.../uwwmensbball-new.streamlit.app/?page=...
    # instead of the app. Also repairs the usual scheme typos (https//host, https:/host).
    _raw_app_url = APP_BASE_URL
    APP_BASE_URL = APP_BASE_URL.strip().strip('"\'').rstrip("/")
    APP_BASE_URL = re.sub(r"^(https?)(?::/{0,2}|/{1,2})", r"\1://", APP_BASE_URL, flags=re.I)
    if not re.match(r"^https?://", APP_BASE_URL, flags=re.I):
        APP_BASE_URL = "https://" + APP_BASE_URL
    APP_BASE_URL = APP_BASE_URL.rstrip("/")
    if APP_BASE_URL != _raw_app_url:
        print(f"  APP_BASE_URL normalized: {_raw_app_url!r} -> {APP_BASE_URL}")
    print(f"Brief links -> {APP_BASE_URL}")
else:
    print("WARNING: APP_BASE_URL is blank -- the scouting brief will render every app link as dead "
          "\"(in the app)\" text. Set APP_BASE_URL in this cell (or the UWW_APP_URL env var) to make the "
          "brief's links actually open the app.")


Input directory: ./inputs
Output directory: ../data
LLM comparisons: disabled
Reference date: 2025-11-19
Before scout: yes -- ignoring the upcoming opponent's own scouting report for this run



### Legacy install/restart cells (disabled)

These two cells mirror the original Databricks notebook's `%pip install lxml` + `dbutils.library.restartPython()` steps, kept only for parity -- both are commented out here since the portable setup installs dependencies via `requirements.txt` ahead of time, and there's no Databricks kernel to restart outside the web UI.

In [4]:
# %pip install -q lxml

In [5]:
# dbutils.library.restartPython()


### Parse every team's schedule; scrape live from FastScout with a local backup fallback

The largest cell in this notebook -- kept as one block rather than split further, since it's a single, carefully-debugged live-scraping engine and separating it risks pulling apart tightly-coupled state (like the shared Playwright session) from the functions that manage it. It:

* Defines the MHTML/HTML loading helpers (`load_html_snapshot`, `_save_scraped_html`) shared by every other scraping step in this notebook (schedules, scouting-report PDFs, play-by-play, video tagging).
* Defines the one shared, lazily-opened, self-healing FastScout Playwright session (`run_in_fastscout_session`) -- reused everywhere else a live scrape is needed instead of opening a new browser per call.
* Scrapes UW-Whitewater's own schedule and every opponent's schedule live (skipping the live scrape when a local backup file already exists), falling back to a local `"<Team> - Schedule.mhtml"`/`".html"` snapshot when live scraping is unavailable or fails.
* Downloads any missing scouting-report PDF straight from FastScout.
* Produces `schedule`, `team_schedules`, and `scouted_opponents`, used throughout the rest of the notebook.

In [7]:
# Every team's own FastScout schedule snapshot (MHTML) lives in INPUT_DIR alongside per-game pbp/video/box
# MHTML files and scout-report PDFs -- process every "<Team> - Schedule.mhtml" file found there (not just
# UWW's). Filtering strictly on the "- Schedule.mhtml" suffix (not just "*.mhtml") matters here: INPUT_DIR is
# a single flat portable folder (unlike the original Databricks setup, which kept per-game MHTMLs in a
# separate volume from the team-schedule snapshots), so a bare "*.mhtml" glob would also match per-game files
# like "11_14_25 UW-Whitewater @ St. Thomas (TX)_box.mhtml" -- which also contains "whitewater" in its name
# and would otherwise be mistaken for UWW's own schedule snapshot below. Each real schedule file has the same
# 3 <table> structure: [0] game-by-game schedule/results, [1] season player box-score stats, [2] empty
# (template for future games).
import asyncio
import concurrent.futures
import subprocess
import time
import traceback
from urllib.parse import urljoin, urlparse, parse_qs

schedules_dir = INPUT_DIR
# Glob both ".mhtml" (a manually-exported/uploaded snapshot) and ".html" (this notebook's own live-scrape
# cache -- see _save_scraped_html) -- the same *_scout.pdf/*_scout.html duality already used for reports.
schedule_mhtml_paths = sorted(
    p for p in glob.glob(f"{schedules_dir}/*.mhtml") + glob.glob(f"{schedules_dir}/*.html")
    # "(?:_\d{4})?" makes the "_<season start year>" suffix optional, so this matches both a legacy
    # filename (no suffix) and a new one saved with _add_season_suffix_to_path.
    if re.search(r"-\s*Schedule(?:_\d{4})?\.(mhtml|html)$", os.path.basename(p), re.IGNORECASE)
)
print(f"Found {len(schedule_mhtml_paths)} schedule MHTML file(s) in {schedules_dir}:")
for p in schedule_mhtml_paths:
    print(" -", os.path.basename(p))


def load_mhtml_html(path):
    """Extract the embedded text/html part from a saved MHTML web page archive."""
    with open(path, "rb") as f:
        raw = f.read()
    msg = email.message_from_bytes(raw, policy=policy.default)
    for part in msg.walk():
        if part.get_content_type() == "text/html":
            charset = part.get_content_charset() or "utf-8"
            return part.get_payload(decode=True).decode(charset, errors="replace")
    return None


def load_html_snapshot(path):
    """Load a saved HTML snapshot from disk, handling both a genuine MHTML web-page archive (from a
    browser's "Save as Webpage, Single File", parsed via load_mhtml_html) AND a plain rendered-HTML file (as
    saved by this notebook's own live-scrape caching -- see _save_scraped_html below) -- mirrors the same
    duality already established for scouting reports ("*_scout.pdf" vs "*_scout.html")."""
    if path.lower().endswith(".mhtml"):
        return load_mhtml_html(path)
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return f.read()


def _save_scraped_html(html, dest_path, label):
    """Save raw scraped HTML to disk, the same way scouting reports are already cached as "*_scout.html" --
    confirmed by the user: they want everything this notebook scrapes persisted this way, not just held in
    memory for the current run, so a future run has a local backup if live scraping is unavailable or fails.
    Read back via load_html_snapshot (NOT load_mhtml_html directly -- this is plain HTML, not a real
    multipart MHTML archive)."""
    try:
        with open(dest_path, "w", encoding="utf-8") as f:
            f.write(html)
        print(f"    [save] {label} -> {os.path.basename(dest_path)}")
    except Exception as save_error:
        print(f"    [save] Could not save {label} to {dest_path}: {type(save_error).__name__}: {save_error}")


def split_opponent(text):
    m = re.match(r"^(.*?)(\d+-\d+)$", str(text).strip())
    return (m.group(1).strip(), m.group(2)) if m else (text, None)


def split_result(text):
    m = re.match(r"^([WL])(\d+)-(\d+)$", str(text).strip())
    if not m:
        return (None, None, None)
    return (m.group(1), int(m.group(2)), int(m.group(3)))


FASTSCOUT_ORIGIN = "https://fastscout.fastmodelsports.com"


def _resolve_team_link(href):
    """Normalize a team link href to an absolute fastscout.fastmodelsports.com URL. Live-rendered SPA pages
    often use RELATIVE hrefs (e.g. "/teams/<id>") for internal client-side-routed links, unlike an exported
    MHTML snapshot's browser-resolved absolute hrefs -- urljoin makes both cases resolve the same way."""
    return href if href.startswith("http") else urljoin(FASTSCOUT_ORIGIN, href)


# Games that have a scouting report -- one "*_scout.pdf" file per scouted matchup, named
# "<date> <Team A> @ <Team B>_scout.pdf", sitting alongside the schedule MHTMLs in INPUT_DIR. Used below
# to filter each team's own schedule down to only the games that have a matching scout report.
volume_dir = INPUT_DIR
# Confirmed by the user: no PDF is needed at all -- scouting reports auto-downloaded from FastScout are now
# saved as "*_scout.html" (the live report page's own rendered DOM) instead of trying to reproduce a PDF via
# Chromium's print pipeline, which never worked reliably across headless/headed and every print-media
# variation tried. Manually-uploaded reports stay as "*_scout.pdf" (from before this change) -- glob both so
# either format counts as "this opponent already has a report".
# Routed through find_scout_files() (Configuration cell) so the `before_scout` switch is honoured here:
# with before_scout="yes" the upcoming game's own report is left out of this list entirely, which in turn
# keeps it out of scouted_opponents_for() / _scout_pdf_already_exists() below.
scout_pdf_files = find_scout_files(volume_dir)


def scouted_opponents_for(team_name, scout_files):
    team_key = team_name.split()[0].lower()
    opponents = []
    for p in scout_files:
        # Confirmed by a live run: this only stripped "_scout.pdf" -- never updated when "*_scout.html"
        # downloads were added above, so any HTML-sourced report kept its "_scout.html" suffix baked into
        # the parsed opponent name (e.g. "Ripon Red Hawks_scout.html"). That garbled name then never matches
        # the real schedule's plain "Ripon Red Hawks" opponent text below, silently dropping that game from
        # the scouted-opponents filter -- confirmed by a live run where exactly the HTML-sourced AWAY-game
        # opponents (whose name ends up on the right side of " @ ", where the suffix lands) vanished from
        # UWW's own scouted schedule. Strip either extension.
        name = re.sub(r"_scout\.(pdf|html)$", "", os.path.basename(p), flags=re.IGNORECASE)
        name = re.sub(r"^\d+_\d+_\d+\s+", "", name)
        if " @ " not in name:
            continue
        left, right = [side.strip() for side in name.split(" @ ", 1)]
        if team_key in left.lower():
            opponents.append(right)
        elif team_key in right.lower():
            opponents.append(left)
    return opponents


# Parse the schedule dates (format from FastScout is like "Sat, Nov 16") into comparable datetimes.
# CONFIRMED CHANGE (requested): this used to hardcode "2025 if month >= 8 else 2026" -- a single
# season baked directly into the parser, silently wrong the moment a schedule snapshot from a
# DIFFERENT season is loaded (e.g. loading 2024-25 data alongside 2025-26). Confirmed live: every
# schedule page FastScout renders carries its own season directly on the page, in a "seasonDropdown"
# element showing text like "2025-2026" -- read via extract_season_start_year() below instead of
# assumed. _DEFAULT_SEASON_START_YEAR is kept as a fallback ONLY for the rare case a page's season
# text can't be found/parsed, so a run never hard-fails over this specifically.
_DEFAULT_SEASON_START_YEAR = 2025

def extract_season_start_year(soup, fallback=None):
    """Read a FastScout team page's own season-selector text (e.g. "2025-2026") directly off the page,
    via its #seasonDropdown element, instead of assuming one hardcoded season for every page. Returns
    the season's START year (e.g. 2025 for "2025-2026") as an int, or `fallback` if the element isn't
    present or its text doesn't parse -- confirmed present on both UWW's own schedule page and every
    opponent schedule page checked so far, but not asserted as always-guaranteed to exist."""
    el = soup.find(id="seasonDropdown")
    if el is not None:
        m = re.search(r"(\d{4})\s*-\s*\d{2,4}", el.get_text(" ", strip=True))
        if m:
            return int(m.group(1))
    return fallback

def parse_schedule_date(date_str, season_start_year=None):
    """`season_start_year` is the ACADEMIC year the season started in (e.g. 2025 for "2025-2026") --
    pass the value extract_season_start_year() read off the specific page this date came from, so a
    date from a 2024-25 schedule and one from a 2025-26 schedule each resolve to their own real
    calendar year rather than both being forced through the same assumption. Falls back to
    _DEFAULT_SEASON_START_YEAR when no season_start_year is given (callers that can't easily thread a
    specific one through -- see the call sites further down this notebook)."""
    try:
        parsed = datetime.strptime(date_str.strip(), "%a, %b %d")
        _syear = season_start_year if season_start_year is not None else _DEFAULT_SEASON_START_YEAR
        year = _syear if parsed.month >= 8 else _syear + 1
        return parsed.replace(year=year)
    except (ValueError, AttributeError):
        return None

def _add_season_suffix_to_path(path, html):
    """Insert "_<season start year>" before the file extension, e.g. "UW-Whitewater - Schedule.html" ->
    "UW-Whitewater - Schedule_2025.html" -- read directly from the page's own season-selector text (see
    extract_season_start_year), so a schedule snapshot saved for one season can never collide with, or
    get silently confused with, one saved for a different season under the same base filename."""
    year = extract_season_start_year(BeautifulSoup(html, "lxml"), fallback=_DEFAULT_SEASON_START_YEAR)
    root, ext = os.path.splitext(path)
    return f"{root}_{year}{ext}"


def build_team_schedule_from_html(html, source_label):
    """Parse one team's own FastScout schedule page HTML (from an MHTML snapshot OR a live scrape) into a
    cleaned schedule DataFrame."""
    page_soup = BeautifulSoup(html, "lxml")
    page_tables = page_soup.find_all("table")

    team_name_raw = page_soup.find("h1").get_text(strip=True)
    team_name = re.match(r"^([^\d]+)", team_name_raw).group(1).strip()

    sched_raw = pd.read_html(StringIO(str(page_tables[0])))[0]

    row_els = [r for r in page_tables[0].find_all("tr") if r.find_all("td")]
    opponent_urls, game_urls, video_urls = [], [], []
    for row_el in row_els:
        hrefs = [a["href"] for a in row_el.find_all("a", href=True)]
        # Match on the "/teams/" path alone (not requiring the full domain) so RELATIVE hrefs from a live
        # SPA render match too -- see _resolve_team_link.
        fastscout_team_links = [h for h in hrefs if "/teams/" in h and "identity.hudl.com" not in h]
        opponent_links = [h for h in fastscout_team_links if "/games/" not in h]
        boxscore_links = [h for h in fastscout_team_links if "/boxscore" in h]
        video_links = [h for h in hrefs if "synergysports.com/video" in h]
        opponent_urls.append(_resolve_team_link(opponent_links[0]) if opponent_links else None)
        game_urls.append(_resolve_team_link(boxscore_links[0]) if boxscore_links else None)
        video_urls.append(video_links[0] if video_links else None)
    sched_raw["opponent_url"] = opponent_urls
    sched_raw["game_url"] = game_urls
    sched_raw["video_url"] = video_urls

    team_schedule = pd.DataFrame()
    team_schedule["date"] = sched_raw["Date"]
    opp_split = sched_raw["Opponent"].apply(split_opponent)
    team_schedule["opponent"] = opp_split.apply(lambda x: x[0])
    team_schedule["team"] = team_name
    team_schedule["location"] = sched_raw["Location"]
    team_schedule["opponent_url"] = sched_raw["opponent_url"]
    team_schedule["game_url"] = sched_raw["game_url"]
    team_schedule["video_url"] = sched_raw["video_url"]
    res_split = sched_raw["Result"].apply(split_result)
    team_schedule["outcome"] = res_split.apply(lambda x: x[0])
    team_schedule["team_score"] = res_split.apply(lambda x: x[1])
    team_schedule["opponent_score"] = res_split.apply(lambda x: x[2])
    team_schedule["point_margin"] = team_schedule["team_score"] - team_schedule["opponent_score"]

    # This file's OWN real season, read from the page itself rather than assumed -- see
    # extract_season_start_year(). Stored as a real column (not just a local variable used once here) so
    # it travels with these rows through every later concat/filter, and any later code that needs to
    # resolve one of THIS team's own dates again (e.g. re-parsing a date pulled from a specific row) can
    # use the season that row actually came from, instead of falling back to a single assumed default.
    _season_start_year = extract_season_start_year(page_soup, fallback=_DEFAULT_SEASON_START_YEAR)
    team_schedule["season"] = f"{_season_start_year}-{str(_season_start_year + 1)[-2:]}"
    team_schedule["_parsed_date"] = team_schedule["date"].apply(lambda d: parse_schedule_date(d, _season_start_year))
    is_primary_team = "whitewater" in team_name.lower()

    if is_primary_team:
        scouted_opponents = scouted_opponents_for(team_name, scout_pdf_files)
        is_scouted = team_schedule["opponent"].apply(
            lambda opp: any(re.search(re.escape(short), opp, re.IGNORECASE) for short in scouted_opponents)
        )

        # Determine upcoming from the FULL schedule (not filtered to scouted-only) so that a new
        # opponent whose scout report hasn't been downloaded yet still gets flagged as upcoming and
        # triggers the live-scrape download below.
        team_schedule["Upcoming"] = "No"
        upcoming_idx = None
        upcoming_candidates = team_schedule[team_schedule["_parsed_date"] >= reference_date]
        if not upcoming_candidates.empty:
            upcoming_idx = upcoming_candidates["_parsed_date"].idxmin()
        if upcoming_idx is not None:
            team_schedule.loc[upcoming_idx, "Upcoming"] = "Yes"

        # Keep scouted opponents (played games) + the upcoming game (even if not yet scouted)
        keep_mask = (is_scouted & (team_schedule["_parsed_date"] < reference_date)) | (team_schedule.index == upcoming_idx)
        team_schedule = team_schedule[keep_mask].reset_index(drop=True)
        summary_note = f"scouted opponents: {scouted_opponents}"
    else:
        team_schedule = team_schedule[team_schedule["_parsed_date"] < reference_date].reset_index(drop=True)
        team_schedule["Upcoming"] = "No"
        summary_note = f"all games before {reference_date_str}"

    team_schedule = team_schedule.drop(columns=["_parsed_date"])

    # CONFIRMED BUG (fixed here): this nulls every column NOT in the allowlist below for the upcoming
    # (not-yet-played) row -- correct for score/outcome columns, which really would be a leak, but
    # "season" wasn't in the allowlist when it was added, so it silently got nulled out here too. Usually
    # invisible (any OTHER row's "season" value was still fine to read), but when the upcoming game is
    # the ONLY row left after filtering -- exactly the "reference_date before UWW's first game" case --
    # .iloc[0] hits this nulled row directly, and int(str(None).split("-")[0]) raised "ValueError:
    # invalid literal for int() with base 10: 'None'" two cells down. "season" isn't leaky information
    # the way a score/outcome is -- it's added to the allowlist alongside the other non-result columns.
    team_schedule.loc[
        team_schedule["Upcoming"] == "Yes",
        team_schedule.columns.difference(["date", "opponent", "Upcoming", "team", "location", "opponent_url", "game_url", "video_url", "season"]),
    ] = None

    print(f"  {team_name} ({source_label}): {len(team_schedule)} game(s) -- {summary_note}")
    return team_schedule, page_soup, page_tables


def build_team_schedule(schedule_path):
    """Parse one team's own saved FastScout schedule snapshot (a genuine ".mhtml" export OR this notebook's
    own live-scrape ".html" cache -- see load_html_snapshot) into a cleaned schedule DataFrame."""
    html = load_html_snapshot(schedule_path)
    return build_team_schedule_from_html(html, source_label=os.path.basename(schedule_path))


def login_to_fastscout(page, username, password, timeout_ms=20000):
    """Log into FastScout via Hudl's Auth0 Universal Login flow. Selectors avoid the dynamic
    React-generated ids/names seen in a saved snapshot of this flow (e.g. "uniId_:r0:") since those
    regenerate every session -- input[type=...] plus button role/name are stable across sessions instead.

    Each step is wrapped separately and re-raises with the URL at that point PLUS any visible on-page error
    text (Auth0's Universal Login shows invalid-credential/MFA/CAPTCHA errors as text on the page itself,
    not as an HTTP error or a distinct exception type) -- otherwise every failure mode collapses into the
    same generic timeout with no way to tell WHY the login didn't go through.
    """

    def _page_error_text():
        for selector in ('[role="alert"]', ".error-message", "#error-element-password", "#error-element-username"):
            try:
                text = page.locator(selector).first.inner_text(timeout=1000)
                if text and text.strip():
                    return text.strip()
            except Exception:
                continue
        return None

    try:
        email_input = page.locator('input[type="email"]')
        email_input.wait_for(timeout=timeout_ms)
        email_input.fill(username)
        # An UNANCHORED "continue" regex also matches Hudl's "Continue with Google/Facebook/Apple" social
        # login buttons on this page, which Playwright's strict mode rejects as an ambiguous match (4
        # elements). Anchoring to the exact button text disambiguates it from those.
        page.get_by_role("button", name=re.compile(r"^continue$", re.IGNORECASE)).click()
    except Exception as e:
        error_text = _page_error_text()
        raise RuntimeError(
            f"FastScout login failed at the EMAIL step (url={page.url}): {type(e).__name__}: {e}"
            + (f" -- page showed: {error_text!r}" if error_text else "")
        ) from e

    try:
        password_input = page.locator('input[type="password"]')
        password_input.wait_for(timeout=timeout_ms)
        password_input.fill(password)
        page.get_by_role("button", name=re.compile(r"^(continue|log ?in)$", re.IGNORECASE)).click()
    except Exception as e:
        error_text = _page_error_text()
        raise RuntimeError(
            f"FastScout login failed at the PASSWORD step (url={page.url}): {type(e).__name__}: {e}"
            + (f" -- page showed: {error_text!r}" if error_text else "")
        ) from e

    try:
        page.wait_for_url(re.compile(r"fastscout\.fastmodelsports\.com"), timeout=timeout_ms)
    except Exception as e:
        error_text = _page_error_text()
        raise RuntimeError(
            "FastScout login did not redirect back to fastscout.fastmodelsports.com after submitting "
            f"credentials (still at url={page.url}): {type(e).__name__}: {e}"
            + (f" -- page showed: {error_text!r}" if error_text else "")
            + " -- this usually means the username/password was rejected (wrong credentials, an MFA prompt, "
            "or a CAPTCHA), not a code bug."
        ) from e


def _goto_with_auth_retry(page, url, wait_selector, timeout_ms=30000):
    """Navigate to url and wait for wait_selector to appear. FastScout's auth-guard redirect to
    identity.hudl.com is ASYNC client-side JS -- observed to fire well AFTER "domcontentloaded" (and even
    "networkidle") have already been reached, so checking page.url immediately after goto() returns is
    unreliable and can miss it entirely (page.url still showed the ORIGINAL url right after goto(), yet
    ended up on identity.hudl.com by the time wait_for_selector's own timeout had elapsed). Instead, let
    wait_for_selector run its full course; if it fails AND we're on identity.hudl.com by then (checked AFTER
    that wait, when the async redirect has had time to actually happen), log in and retry once."""
    page.goto(url, wait_until="domcontentloaded", timeout=timeout_ms)
    try:
        page.wait_for_selector(wait_selector, timeout=timeout_ms)
        return
    except Exception:
        if "access_token=" in page.url:
            # A SILENT SSO re-auth (valid Hudl session cookies already present, so no interactive
            # email/password form was ever shown) can complete its ENTIRE identity.hudl.com -> callback ->
            # "#access_token=..." redirect dance WHILE this wait_for_selector call was already polling --
            # confirmed by a live run's own Playwright action log showing exactly that sequence happen
            # mid-wait. By the time it lands back on fastscout.fastmodelsports.com with the token in the URL
            # hash, most/all of the original timeout budget is already spent, leaving the SPA no time to
            # actually consume that token and render the page. This is NOT the "stuck needing interactive
            # login" case below (we're not on identity.hudl.com) -- just give it one more full timeout
            # window to finish rendering on its own instead of failing immediately.
            print(f"    [auth-retry] landed on an in-progress SSO callback ({page.url}) while waiting on {url} -- giving it a fresh wait instead of failing")
            page.wait_for_selector(wait_selector, timeout=timeout_ms)
            return
        if "identity.hudl.com" not in page.url:
            raise

    print(f"    [auth-retry] got redirected to identity.hudl.com while waiting on {url} -- logging in and retrying")
    login_to_fastscout(page, fastscout_username, fastscout_password, timeout_ms=timeout_ms)
    print(f"    [auth-retry] login_to_fastscout returned, now at {page.url} -- re-navigating to {url}")
    page.goto(url, wait_until="domcontentloaded", timeout=timeout_ms)
    print(f"    [auth-retry] re-navigated, now at {page.url} -- waiting for {wait_selector!r}")
    try:
        page.wait_for_selector(wait_selector, timeout=timeout_ms)
    except Exception:
        # This is the SECOND failure (after an already-completed login) -- dump whatever is actually
        # visible on screen, since a bare timeout gives no clue whether this is a stuck spinner, a repeat
        # MFA/consent prompt, or something else entirely looping back through the auth provider.
        try:
            visible_text = page.locator("body").inner_text(timeout=2000)[:800]
        except Exception as text_error:
            visible_text = f"<could not read body text: {type(text_error).__name__}: {text_error}>"
        print(
            f"    [auth-retry] STILL stuck after retry -- url={page.url}, title={page.title()!r}\n"
            f"    [auth-retry] visible body text (first 800 chars): {visible_text!r}"
        )
        raise


def _click_tab_by_text(page, label, timeout_ms=30000):
    """Click a nav tab/link by its exact visible text (case-insensitive) within the already-loaded FastScout
    SPA. Direct page.goto() to a sub-route URL (e.g. '/games', '/documents') gets silently redirected back to
    '/analytics/dashboard' -- FastScout only renders those views via client-side navigation (an in-app click
    on the corresponding nav tab), not a fresh full-page load. Confirmed by observation: navigating straight
    to '.../games?...' consistently lands on '.../analytics/dashboard' instead, showing the SAME efficiency
    panel regardless of which team's page was requested."""
    page.get_by_text(re.compile(rf"^{re.escape(label)}$", re.IGNORECASE)).first.click(timeout=timeout_ms)


def scrape_rendered_html(page, url, wait_selector="#myTeamSchedule", timeout_ms=30000, save_path=None):
    """Navigate to a FastScout team's base page, then click the 'SCHEDULE' nav tab to reach its schedule
    table client-side -- see _click_tab_by_text for why a direct URL to the schedule sub-route doesn't work.
    If save_path is given, caches the scraped HTML there (see _save_scraped_html) the same way scouting
    reports are cached."""
    print(f"    [scrape] navigating to {url}")
    try:
        _goto_with_auth_retry(page, url, "text=SCHEDULE", timeout_ms)
        print(f"    [scrape] landed on {page.url} -- clicking 'SCHEDULE' tab")
        _click_tab_by_text(page, "SCHEDULE", timeout_ms)
        page.wait_for_selector(wait_selector, timeout=timeout_ms)
        # The container can become visible before its rows finish an async data fetch triggered by the tab
        # click -- wait for an actual <tr> inside it too, not just the container itself.
        page.wait_for_selector(f"{wait_selector} tr", timeout=timeout_ms)
        # This page ALSO renders a second table just below the schedule -- the season player box-score
        # stats table (tables[1] in the next cell) -- which loads via its OWN separate async fetch that the
        # waits above don't cover at all (they only target #myTeamSchedule, i.e. tables[0]). Confirmed by a
        # live run: tables[0] came back fully populated but tables[1] had zero data rows -- same "captured
        # before the async widget finished loading" issue already seen with the ScoutBuilder boxscore Tile.
        # Each populated player row renders its name in a "<div data-id=\"...\">" cell (header cells use
        # col/label/statkey attributes instead, never data-id), so wait on that as a stable marker that this
        # second table has actually loaded before capturing the page.
        #
        # Confirmed by a live run: for 3 opponents this STILL timed out even though the selector resolved to
        # 30+ matching elements on every single poll (e.g. 32 for Aurora) -- Playwright just never considered
        # the first one "visible". That's the signature of a virtualized/lazy-rendered table (react-window
        # style): rows already exist in the DOM with real data (a data-id already set), but with a
        # collapsed/zero-size bounding box until actually scrolled into view -- the exact same lazy-render
        # behavior already confirmed for the ScoutBuilder boxscore Tile, which needed an explicit
        # scroll_into_view_if_needed() to force it to render. Scroll the stats table's own header row into
        # view first (its rows share the same "stat-table-row" class already seen on the header) to trigger
        # that, before waiting on its data cells.
        try:
            page.locator("tr.stat-table-row").first.scroll_into_view_if_needed(timeout=5000)
        except Exception:
            pass
    except Exception as e:
        raise RuntimeError(
            f"Could not reach the schedule view from {url} (actual url={page.url}, title={page.title()!r}): "
            f"{type(e).__name__}: {e}"
        ) from e

    # Confirmed by a live run: for EVERY opponent, this wait timed out even though the schedule table
    # ({wait_selector} and its <tr> rows, both awaited above) had already loaded successfully -- the
    # elements it found (e.g. 30 for Ripon, showing a player name like "Olin Zellmer") belong to a
    # DIFFERENT widget entirely (a roster/top-players panel elsewhere on the team page), not the schedule
    # table's own rows. This step only exists for the season player box-score STATS table (tables[1]),
    # which build_team_schedule_from_html (the only consumer of this function's result) never reads --
    # it only ever uses tables[0], the schedule table already confirmed loaded above. Making a failure here
    # non-fatal instead of raising means an opponent's schedule (already successfully captured) no longer
    # gets thrown away and replaced by a usually-nonexistent local backup file just because this unrelated,
    # unused-by-this-caller widget didn't finish loading.
    try:
        page.wait_for_selector("td div[data-id]", timeout=timeout_ms)
    except Exception as e:
        print(f"    [scrape] (non-fatal) second stats table never loaded, proceeding with schedule table only: {type(e).__name__}: {e}")
    print(f"    [scrape] landed on {page.url}")
    html = page.content()
    if save_path:
        # Same season-suffixed save as UW-Whitewater's own schedule above -- see _add_season_suffix_to_path.
        _save_scraped_html(html, _add_season_suffix_to_path(save_path, html), "scraped opponent schedule")
    return html


def _ensure_playwright_ready():
    """Import playwright.sync_api, installing the pip package and/or the Chromium browser binary first if
    either isn't already present. Both the scouting-report-PDF download step and the opponent-schedule
    scraping step below need this, so it's centralized here instead of duplicated in each."""
    try:
        from playwright.sync_api import sync_playwright
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "playwright"])
        from playwright.sync_api import sync_playwright

    # The "playwright" PIP PACKAGE and the actual Chromium BROWSER BINARY install separately, and can drift
    # out of sync after a fresh pip install, an environment rebuild, or a Playwright version bump -- that's
    # exactly what the "Looks like Playwright was just installed or updated -- please run playwright install"
    # message means. Running this every time is a fast no-op once the browser is already downloaded, so
    # there's no need to run it manually in a separate terminal.
    subprocess.check_call([sys.executable, "-m", "playwright", "install", "chromium"])
    return sync_playwright


FASTSCOUT_DOCS_LEAGUE = "NCAAB-III"
FASTSCOUT_DOCS_SEASON = "25"


def _ensure_fastscout_login(page, timeout_ms=30000):
    """Log into FastScout if not already authenticated. Uses a KNOWN, always-valid URL ("myTeam" is a
    literal alias FastScout resolves to whichever team the logged-in account belongs to) to reliably trigger
    Hudl's auth redirect, rather than navigating to an arbitrary opponent's page first and hoping the
    redirect behaves consistently.

    IMPORTANT: after a fresh login, FastScout always lands on
    ".../teams/myTeam/analytics/dashboard?league=...&season=..." regardless of what URL originally triggered
    it -- so callers must always explicitly (re-)navigate to wherever they actually want afterward. This
    function only guarantees the session is authenticated, not that the page is showing anything useful.

    Confirmed by the user hitting a "season=25 -> season=26" redirect on this exact "myTeam" URL: FastScout's
    "myTeam" alias appears to resolve to the ACCOUNT'S currently-active season, silently overriding whatever
    `season=` this URL requested once real-world time moves past that season (e.g. the 2025-26 season this
    notebook analyzes was already over by the time this ran). Not fatal by itself (login still succeeds),
    but risky: any view reached via "myTeam" AFTER this redirect -- e.g. scrape_uww_live_schedule's SCHEDULE
    tab click below -- could then silently show the WRONG season's data instead of raising an error. Log a
    loud, explicit warning whenever the landed season doesn't match FASTSCOUT_DOCS_SEASON, rather than
    letting that redirect pass by unnoticed in the ordinary "landed on {page.url}" print below.
    """
    known_url = f"https://fastscout.fastmodelsports.com/teams/myTeam/analytics/dashboard?league={FASTSCOUT_DOCS_LEAGUE}&season={FASTSCOUT_DOCS_SEASON}"
    print(f"    [login] navigating to {known_url}")
    _goto_with_auth_retry(page, known_url, "text=SCHEDULE", timeout_ms)
    print(f"    [login] confirmed logged in, landed on {page.url}")

    landed_season = parse_qs(urlparse(page.url).query).get("season", [None])[0]
    if landed_season is not None and landed_season != FASTSCOUT_DOCS_SEASON:
        print(
            f"    [login] WARNING: requested season={FASTSCOUT_DOCS_SEASON!r} but FastScout's \"myTeam\" "
            f"redirect landed on season={landed_season!r} instead -- this account's \"current\" season has "
            f"moved on. Any view reached via \"myTeam\" from here (e.g. the live schedule scrape below) may "
            f"now be showing season {landed_season} data instead of season {FASTSCOUT_DOCS_SEASON}. Verify "
            "the scraped schedule's games actually belong to the intended season before trusting this run's output."
        )

    # Diagnostic: persist this login's own LANDING page HTML too -- the "myTeam" analytics/dashboard view,
    # captured BEFORE any caller clicks away to another tab (e.g. scrape_uww_live_schedule's SCHEDULE click).
    # Confirmed by inspecting the already-saved SCHEDULE tab snapshot: the "Top Rebounders/Scorers/3PT/FT"
    # leaderboard tiles the user asked about do NOT appear there -- so if they exist anywhere on this account,
    # this dashboard landing page (nothing currently captures it) is the next most likely place. One-time save
    # per session bootstrap (this function only runs once per lazily-opened session -- see
    # run_in_fastscout_session) -- cheap, and gives a real snapshot to search instead of guessing selectors.
    try:
        _save_scraped_html(
            page.content(), os.path.join(schedules_dir, "myTeam - Analytics Dashboard.html"),
            "myTeam analytics dashboard (diagnostic)",
        )
    except Exception as dashboard_save_error:
        print(f"    [login] (non-fatal) could not save analytics/dashboard diagnostic snapshot: {type(dashboard_save_error).__name__}: {dashboard_save_error}")


def scrape_uww_live_schedule(page, timeout_ms=30000, save_path=None):
    """Scrape UW-Whitewater's own schedule table live from FastScout, instead of relying on a manually
    exported/uploaded schedule MHTML snapshot. Assumes the page is already on a loaded, authenticated team
    page (e.g. via _ensure_fastscout_login) -- just clicks the 'SCHEDULE' nav tab from there (see
    _click_tab_by_text for why a direct URL to this sub-route doesn't work). If save_path is given, caches
    the scraped HTML there (see _save_scraped_html) the same way scouting reports are cached."""
    print("    [scrape] clicking 'SCHEDULE' tab")
    _click_tab_by_text(page, "SCHEDULE", timeout_ms)
    page.wait_for_selector("#myTeamSchedule", timeout=timeout_ms)
    # The container can become visible before its rows finish an async data fetch triggered by the tab
    # click -- wait for an actual <tr> inside it too, not just the container itself.
    try:
        page.wait_for_selector("#myTeamSchedule tr", timeout=timeout_ms)
    except Exception as e:
        raise RuntimeError(
            f"'#myTeamSchedule' appeared but never got any <tr> rows within {timeout_ms}ms (url={page.url}): "
            f"{type(e).__name__}: {e}"
        ) from e
    print(f"    [scrape] landed on {page.url}")
    html = page.content()
    if save_path:
        # CONFIRMED CHANGE (requested): save with a "_<season start year>" suffix (e.g.
        # "..._2025.html") so a snapshot saved for one season never collides with, or gets mistaken
        # for, one saved for a different season under the exact same base filename.
        _save_scraped_html(html, _add_season_suffix_to_path(save_path, html), "UW-Whitewater's own scraped schedule")
    return html


# FastScout login lives behind Hudl's identity provider. Credentials are resolved from
# FASTSCOUT_USERNAME / FASTSCOUT_PASSWORD environment variables, optionally loaded from a local ".env"
# file (via python-dotenv) sitting next to this notebook. A ".env" file (git-ignored!) would look like:
#   FASTSCOUT_USERNAME=you@example.com
#   FASTSCOUT_PASSWORD=your-password
try:
    from dotenv import load_dotenv

    load_dotenv()
except ModuleNotFoundError:
    pass

fastscout_username = os.environ.get("FASTSCOUT_USERNAME")
fastscout_password = os.environ.get("FASTSCOUT_PASSWORD")

if not (fastscout_username and fastscout_password):
    print(
        "No FastScout credentials found (checked FASTSCOUT_USERNAME/FASTSCOUT_PASSWORD env vars / .env) -- "
        "live scraping will be skipped and every opponent will fall back to its local backup MHTML."
    )

# Synergy Sports Tech (auth.synergysportstech.com) is a SEPARATE identity provider from FastScout/Hudl --
# confirmed by a live run that it does NOT share FastScout's username/password (login_to_synergy in the
# video-tagging helpers cell below submitted the FastScout credentials and got a real "Invalid username or
# password" back from Synergy's own server). Resolved the same way, from its own SYNERGY_USERNAME /
# SYNERGY_PASSWORD env vars / ".env" entries:
#   SYNERGY_USERNAME=you@example.com
#   SYNERGY_PASSWORD=your-synergy-password
synergy_username = os.environ.get("SYNERGY_USERNAME")
synergy_password = os.environ.get("SYNERGY_PASSWORD")

if not (synergy_username and synergy_password):
    print(
        "No Synergy credentials found (checked SYNERGY_USERNAME/SYNERGY_PASSWORD env vars / .env) -- "
        "live video-clip scraping will fail for any game not already cached locally."
    )


def find_backup_mhtml(opponent_name, schedules_dir):
    """Find a saved schedule snapshot for this opponent in schedules_dir, matched loosely by name -- either a
    genuine ".mhtml" export or this notebook's own live-scrape ".html" cache (see _save_scraped_html)."""
    for path in sorted(glob.glob(f"{schedules_dir}/*.mhtml") + glob.glob(f"{schedules_dir}/*.html")):
        base = re.sub(r"(_schedule(?:_\d{4})?\.(mhtml|html)$|\s*-\s*Schedule(?:_\d{4})?\.(mhtml|html)$)", "", os.path.basename(path), flags=re.IGNORECASE)
        if base.lower() in opponent_name.lower():
            return path
    return None


# --- One shared FastScout Playwright session, reused by every scraping step below AND by the live-scrape
# fallbacks in the opponent-pbp/video cells further down -- instead of each opening and closing its own
# browser+thread. Confirmed by the user hitting a Windows greenlet "MemoryError" inside Playwright's own
# dispatch loop after many independent open/close cycles accumulated across repeated cell re-runs in one
# long-lived kernel: this notebook previously opened up to 5 separate sessions per full run (UWW schedule,
# scout PDFs, opponent schedules, plus the 2 pbp/video live-scrape fallbacks) -- now just 1, opened lazily on
# first use and left open/reused for the rest of this kernel session (call close_fastscout_session() to
# explicitly tear it down early if needed; otherwise a kernel restart cleans it up).
_fastscout_session = {"executor": None, "playwright": None, "browser": None, "page": None}


def run_in_fastscout_session(fn):
    """Run fn(page) against the single shared, lazily-opened FastScout session. All Playwright sync-API calls
    for a given browser/page must run on the SAME thread that created them (a brand-new thread has no
    asyncio event loop of its own, sidestepping Playwright's sync API refusing to start on a thread that
    already has one -- see the historical comment this replaced, preserved in git history), so this always
    dispatches through one persistent ThreadPoolExecutor(max_workers=1) worker thread -- created once and
    reused for every call -- rather than a fresh executor + browser + login per call. The Windows
    WindowsProactorEventLoopPolicy swap (needed because Jupyter/ipykernel forces WindowsSelectorEventLoopPolicy
    process-wide, which breaks Playwright's Node-driver asyncio subprocess) only needs to happen once, at
    first-use bootstrap, and is restored immediately after -- not on every call."""
    def _bootstrap():
        original_policy = asyncio.get_event_loop_policy() if sys.platform == "win32" else None
        if sys.platform == "win32":
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        try:
            sync_playwright = _ensure_playwright_ready()
            playwright = sync_playwright().start()
            browser = playwright.chromium.launch(headless=True)
            page = browser.new_page()
            _ensure_fastscout_login(page)
            _fastscout_session.update(playwright=playwright, browser=browser, page=page)
        finally:
            if sys.platform == "win32":
                asyncio.set_event_loop_policy(original_policy)

    def _discard_stale_session():
        for key, closer in (("browser", "close"), ("playwright", "stop")):
            obj = _fastscout_session.get(key)
            if obj is not None:
                try:
                    getattr(obj, closer)()
                except Exception:
                    pass
        _fastscout_session.update(playwright=None, browser=None, page=None)

    def _job():
        if _fastscout_session["page"] is None:
            _bootstrap()
        # A single automatic retry wasn't always enough -- confirmed by the user hitting the SAME
        # "Page.goto: Connection closed while reading from the driver" error again right after a first
        # reopen-and-retry (a flaky Windows-side Playwright driver subprocess can take more than one restart
        # to recover). Retry up to MAX_DEAD_SESSION_RETRIES times, with a short pause before each fresh
        # bootstrap to let the OS fully release the previous browser/driver process first, instead of giving
        # up (or leaving the shared session permanently broken) after only one attempt.
        MAX_DEAD_SESSION_RETRIES = 2
        for attempt in range(MAX_DEAD_SESSION_RETRIES + 1):
            try:
                return fn(_fastscout_session["page"])
            except Exception as e:
                # The shared browser/driver can die BETWEEN calls (a crash, a timeout, or a Windows-side
                # Playwright driver subprocess issue) even though it was fine at bootstrap. There was
                # previously no liveness check at all, so once the underlying browser died, EVERY subsequent
                # call in this kernel session would keep failing the same way. Treat any exception
                # mentioning a dead connection/driver/target as that signal: discard the stale session and
                # retry against a freshly-bootstrapped one.
                msg = str(e).lower()
                is_dead_session = any(s in msg for s in ("connection closed", "target closed", "browser has been closed", "driver"))
                if is_dead_session and attempt < MAX_DEAD_SESSION_RETRIES:
                    print(f"    [session] Detected a dead FastScout browser session (attempt {attempt + 1}/{MAX_DEAD_SESSION_RETRIES}) -- reopening and retrying.")
                    _discard_stale_session()
                    time.sleep(2)
                    _bootstrap()
                    continue
                raise

    if _fastscout_session["executor"] is None:
        _fastscout_session["executor"] = concurrent.futures.ThreadPoolExecutor(max_workers=1)
    return _fastscout_session["executor"].submit(_job).result()


def close_fastscout_session():
    """Explicitly tear down the shared session (browser + driver + worker thread) to reclaim resources
    without a full kernel restart. Not required between runs -- the session is left open and reused by
    default."""
    def _job():
        if _fastscout_session["browser"] is not None:
            _fastscout_session["browser"].close()
        if _fastscout_session["playwright"] is not None:
            _fastscout_session["playwright"].stop()
        _fastscout_session.update(playwright=None, browser=None, page=None)

    executor = _fastscout_session["executor"]
    if executor is not None:
        executor.submit(_job).result()
        executor.shutdown(wait=False)
        _fastscout_session["executor"] = None


# 1) UW-Whitewater's own schedule: try scraping it LIVE from FastScout first (via the "myTeam" alias, which
#    resolves to whichever team the logged-in account belongs to -- no need to know UWW's own team id), the
#    same live-scrape-with-MHTML-backup pattern already used for every opponent below. Falls back to the
#    local MHTML snapshot if no credentials are set or the live scrape fails for any reason.
uww_mhtml_path = next((p for p in schedule_mhtml_paths if "whitewater" in os.path.basename(p).lower()), None)

# CONFIRMED BUG (fixed here): FASTSCOUT_DOCS_SEASON was hardcoded to "25" (the 2025-26 season) in every
# live-scrape URL used to auto-download a missing scout report -- meaning the "Documents" page those
# URLs point to is ALWAYS the current season's, regardless of which season this run is actually
# processing. This is the real root cause behind two connected, real symptoms: before the game_date fix
# elsewhere in this cell, a WRONG (2025) game_date happened to coincidentally match SOMETHING on that
# same (2025) Documents page, so a scout report was downloaded and saved -- just the WRONG opponent's
# CURRENT-season report, mislabeled under the archived game's filename. After that fix, the CORRECT
# (2024) game_date can no longer match anything on a Documents page that only ever shows 2025-26, so
# nothing gets saved at all -- an honest failure instead of a silent wrong one, but still a failure.
# If a local schedule snapshot already exists for UWW, its own season (read the same way as everywhere
# else in this notebook -- see extract_season_start_year) is a far better guess than the hardcoded
# default, since it directly reflects the season this run is actually processing.
#
# IMPORTANT CAVEAT: this makes the live-scrape URL point at the RIGHT season, but it does NOT guarantee
# FastScout's live site actually serves a Documents listing for an archived past season at all -- that's
# a question about the external service's own data retention, which this notebook has no way to verify.
# If auto-download still finds nothing after this fix, the reliable path for a historical/archived
# season is to provide the scout report file directly in INPUT_DIR rather than rely on auto-download.
if uww_mhtml_path is not None:
    try:
        _fss_html = load_html_snapshot(uww_mhtml_path)
        _fss_year = extract_season_start_year(BeautifulSoup(_fss_html, "lxml"), fallback=None)
        if _fss_year is not None:
            FASTSCOUT_DOCS_SEASON = str(_fss_year)[-2:]
            print(f"Detected season {_fss_year} from UWW's local schedule snapshot -- using "
                  f"FASTSCOUT_DOCS_SEASON={FASTSCOUT_DOCS_SEASON!r} for any live-scrape URLs below "
                  f"(was hardcoded to '25').")
    except Exception as _fss_e:
        print(f"Could not read UWW's local schedule file to auto-detect FASTSCOUT_DOCS_SEASON "
              f"({type(_fss_e).__name__}: {_fss_e}) -- leaving it at the hardcoded default "
              f"({FASTSCOUT_DOCS_SEASON!r}).")
uww_html = None
if fastscout_username and fastscout_password:
    # Skip the live scrape entirely when a local schedule file already exists for UWW itself -- same
    # skip-check applied to every opponent below (see _run_fastscout_scrape_session's find_backup_mhtml
    # check), just using uww_mhtml_path (already resolved above via the same "whitewater" filename match)
    # instead of re-deriving it. Confirmed by the user: once a schedule has been captured once, there's no
    # need to pay for a fresh (slow, resource-heavy) live scrape on every subsequent run.
    if uww_mhtml_path is not None:
        print(f"Skipping live scrape for UW-Whitewater's own schedule -- a local schedule file already exists: {os.path.basename(uww_mhtml_path)}")
    else:
        try:
            _uww_save_path = os.path.join(schedules_dir, "UW-Whitewater - Schedule.html")
            uww_html = run_in_fastscout_session(lambda page: scrape_uww_live_schedule(page, save_path=_uww_save_path))
            print("Scraped UW-Whitewater's own schedule live from FastScout (teams/myTeam/games).")
        except Exception as e:
            print(f"Could not scrape UWW's own live schedule, falling back to local MHTML: {type(e).__name__}: {e}")

if uww_html is None and uww_mhtml_path is None:
    raise FileNotFoundError(
        f"No UW-Whitewater schedule MHTML found in {schedules_dir}, and live scraping was unavailable or failed."
    )

# 1b) Before building UWW's (scouted-games-only) schedule, proactively download any MISSING scouting-report
#     PDF straight from each opponent's own FastScout "documents" page, using the same authenticated session
#     -- rather than requiring these to be manually collected ahead of time. This has to run on the RAW
#     (unfiltered) schedule, since the whole point is to discover games that don't have a scout PDF locally
#     YET; build_team_schedule_from_html's own scouted-games filter can't include a game until AFTER this
#     step has filled in its missing report.
def parse_raw_schedule_rows(html):
    """Parse a team's own FastScout schedule page HTML into (date, parsed_date, opponent, location,
    opponent_url) rows, with NO scouted-games filtering applied -- unlike build_team_schedule_from_html,
    which only returns games that ALREADY have a matching scout PDF. Used solely to discover which games are
    missing a report so one can be downloaded before that filter runs for real."""
    page_soup = BeautifulSoup(html, "lxml")
    page_tables = page_soup.find_all("table")
    sched_raw = pd.read_html(StringIO(str(page_tables[0])))[0]

    # CONFIRMED BUG (fixed here): parsed_date below used to call parse_schedule_date(r["Date"]) with no
    # season_start_year override, silently falling back to _DEFAULT_SEASON_START_YEAR (2025) regardless
    # of which season this HTML actually is. build_team_schedule_from_html() already reads this page's
    # real season correctly (see extract_season_start_year()), but THIS function runs earlier -- it's
    # what discovers which games need a scout report downloaded in the first place -- and never got the
    # same fix applied. Confirmed as the root cause of a real, reported case: a 2024-25 game on 11/12/24
    # got its auto-downloaded scout report saved as "11_12_25 ..._scout.html" -- a full year off -- because
    # the date used to NAME that file came from this function's un-overridden parsed_date. Fixed the same
    # way as build_team_schedule_from_html: read this page's own season directly, from the same page_soup
    # already built two lines up.
    _season_start_year = extract_season_start_year(page_soup, fallback=_DEFAULT_SEASON_START_YEAR)

    row_els = [r for r in page_tables[0].find_all("tr") if r.find_all("td")]
    opponent_urls = []
    for row_el in row_els:
        hrefs = [a["href"] for a in row_el.find_all("a", href=True)]
        fastscout_team_links = [h for h in hrefs if "/teams/" in h and "identity.hudl.com" not in h]
        opponent_links = [h for h in fastscout_team_links if "/games/" not in h]
        opponent_urls.append(_resolve_team_link(opponent_links[0]) if opponent_links else None)
    sched_raw["opponent_url"] = opponent_urls

    rows = []
    for _, r in sched_raw.iterrows():
        opponent, _ = split_opponent(r["Opponent"])
        rows.append({
            "date": r["Date"],
            "parsed_date": parse_schedule_date(r["Date"], _season_start_year),
            "opponent": opponent,
            "location": r["Location"],
            "opponent_url": r["opponent_url"],
        })
    return pd.DataFrame(rows)


def _scout_pdf_already_exists(opponent, scout_files):
    opponent_key = opponent.split()[0].lower()
    return any(opponent_key in os.path.basename(p).lower() for p in scout_files)


def _find_matching_scout_document_row(docs_soup, game_date):
    """Within an opponent's FastScout '/documents' page HTML, find the <tr> whose 'Game Date' column matches
    game_date (comparing month/day only, since the visible text format varies). Returns (row_element,
    row_index_within_body_rows) on a match, or (None, seen_dates) with whatever date text WAS found, so a
    non-match can still be explained rather than just failing silently."""
    docs_table = docs_soup.find("table")
    if docs_table is None:
        raise RuntimeError("No <table> found on the documents page.")

    headers = [th.get_text(strip=True) for th in docs_table.find_all("th")]
    try:
        date_col_idx = next(i for i, h in enumerate(headers) if "game date" in h.lower())
    except StopIteration:
        raise RuntimeError(f"No 'Game Date' column found in the documents table headers: {headers}")

    body_rows = [r for r in docs_table.find_all("tr") if r.find_all("td")]
    seen_dates = []
    for row_idx, row_el in enumerate(body_rows):
        cells = row_el.find_all("td")
        if date_col_idx >= len(cells):
            continue
        cell_text = cells[date_col_idx].get_text(strip=True)
        seen_dates.append(cell_text)
        for fmt in ("%m/%d/%Y", "%m/%d/%y", "%b %d, %Y", "%B %d, %Y", "%a, %b %d, %Y", "%a, %b %d"):
            try:
                parsed = datetime.strptime(cell_text, fmt)
            except ValueError:
                continue
            if parsed.month == game_date.month and parsed.day == game_date.day:
                return row_el, row_idx
    return None, seen_dates


def _download_matching_scout_pdf(page, docs_url, opponent, game_date, location, timeout_ms=30000):
    # docs_url's "/documents?..." deep link doesn't render directly via page.goto() (see _click_tab_by_text)
    # -- land on the opponent's own base team page first, then click through to their documents/scouts tab.
    opponent_base_url = docs_url.split("/documents")[0]
    try:
        _goto_with_auth_retry(page, opponent_base_url, "text=SCHEDULE", timeout_ms)
    except Exception as e:
        raise RuntimeError(
            f"Could not load {opponent}'s team page ({opponent_base_url}) (actual url={page.url}, "
            f"title={page.title()!r}): {type(e).__name__}: {e}"
        ) from e

    # Confirmed from saved snapshots: an opponent's tab pointing at this "/documents?..." URL is labeled
    # "SCOUTS" (MY OWN team's equivalent is "SELF SCOUTS") -- but the app ALSO has a global top-nav link
    # also labeled "SCOUTS" (pointing at a different library page), so a plain text match risks clicking the
    # wrong one. Click by href instead of by text to avoid that ambiguity -- but confirmed by an actual run
    # that an EXACT match against the absolute docs_url silently never matched (it landed on the global
    # "/library/opponents" page instead, meaning it fell through to the text-based fallback below and hit
    # the wrong "SCOUTS" link): a live SPA render authors this anchor's href ATTRIBUTE as a relative path
    # (e.g. "/teams/<id>/documents?..."), and browsers don't rewrite the raw attribute value to absolute --
    # only the resolved ".href" PROPERTY -- so an exact match against our absolute docs_url can never hit it.
    # Match on a *suffix* instead, which works whether this specific anchor's href happens to be authored as
    # relative or absolute.
    docs_path = docs_url.replace(FASTSCOUT_ORIGIN, "")
    try:
        page.locator(f'a[href$="{docs_path}"]').first.click(timeout=5000)
        clicked = True
    except Exception:
        clicked = False

    if not clicked:
        for label in ("SCOUTS", "SELF SCOUTS", "DOCUMENTS", "SCOUTING REPORTS"):
            try:
                _click_tab_by_text(page, label, timeout_ms=5000)
                break
            except Exception:
                continue
        else:
            try:
                visible_text = page.locator("body").inner_text(timeout=2000)[:800]
            except Exception:
                visible_text = "<could not read body text>"
            raise RuntimeError(
                f"Could not find a Documents/Scouts tab on {opponent}'s page (url={page.url}, "
                f"title={page.title()!r}) -- visible body text (first 800 chars): {visible_text!r}"
            )

    try:
        page.wait_for_selector("table", timeout=timeout_ms)
    except Exception as e:
        raise RuntimeError(
            f"No <table> appeared after clicking a Documents/Scouts tab for {opponent} (url={page.url}, "
            f"title={page.title()!r}): {type(e).__name__}: {e} -- this may mean no scouting reports exist "
            f"for {opponent} under league={FASTSCOUT_DOCS_LEAGUE!r} season={FASTSCOUT_DOCS_SEASON!r}."
        ) from e

    docs_soup = BeautifulSoup(page.content(), "lxml")
    try:
        matched_row_el, row_idx_or_seen_dates = _find_matching_scout_document_row(docs_soup, game_date)
    except Exception as e:
        # Surface the ACTUAL url/title Playwright ended up on, not just docs_url we asked for -- a table
        # with the wrong headers (e.g. a stats/efficiency panel instead of a documents list) usually means
        # the SPA redirected/rendered a different view than the one we navigated to, and the requested vs.
        # actual url diverging is the key signal for that.
        raise RuntimeError(
            f"Could not find a 'Game Date' column on {docs_url} (actual url={page.url}, title={page.title()!r}): "
            f"{type(e).__name__}: {e}"
        ) from e
    if matched_row_el is None:
        raise RuntimeError(
            f"No row matched game date {game_date.strftime('%b %d')} on {docs_url} -- 'Game Date' values "
            f"seen on the page: {row_idx_or_seen_dates}"
        )
    matched_row_idx = row_idx_or_seen_dates

    # Confirmed by the user: no PDF is needed at all -- auto-downloaded reports are saved as the live report
    # page's own rendered HTML instead (see below), so this filename ends in "_scout.html", not "_scout.pdf".
    matchup = f"{opponent} @ UW-Whitewater" if str(location).strip().lower() == "home" else f"UW-Whitewater @ {opponent}"
    filename = f"{game_date.month}_{game_date.day}_{game_date.strftime('%y')} {matchup}_scout.html"
    dest_path = os.path.join(schedules_dir, filename)

    # games_needing_scout_pdf was already filtered against scout_pdf_files (which now globs both "*_scout.pdf"
    # and "*_scout.html") up front, but that check is a fuzzy match on the OPPONENT'S first name-word against
    # ANY existing report filename -- it can't know the EXACT filename this specific game would produce until
    # game_date/matchup are resolved (both only available here, mid-function). Re-check the precise dest_path
    # too, as a second, exact line of defense, before doing any browser work at all.
    if os.path.exists(dest_path):
        print(f"  [{opponent}] scout report already exists in {schedules_dir} -- skipping download: {filename}")
        return dest_path

    # Prefer a direct <a href="...pdf"> link in the matched row -- fetch it with the browser context's own
    # authenticated cookies (page.context.request) rather than clicking through the UI.
    pdf_hrefs = [a["href"] for a in matched_row_el.find_all("a", href=True) if ".pdf" in a["href"].lower()]
    if pdf_hrefs:
        response = page.context.request.get(pdf_hrefs[0])
        if not response.ok:
            raise RuntimeError(f"GET {pdf_hrefs[0]} returned HTTP {response.status}")
        with open(dest_path, "wb") as f:
            f.write(response.body())
        return dest_path

    # No download button here after all (confirmed by the user) -- the team-name cell's edit icon carries
    # the report's numeric id directly, e.g. id="pencil-729471" -> https://fastscout.fastmodelsports.com/
    # report/729471. Extracting it from the already-parsed row avoids any ambiguous click target entirely.
    pencil_icon = matched_row_el.find(id=re.compile(r"^pencil-\d+$"))
    if pencil_icon is None:
        raise RuntimeError(
            f"Could not find a report id (an element with id='pencil-<id>') in the matched row for "
            f"{opponent} (url={page.url})."
        )
    report_id = pencil_icon["id"].split("-", 1)[1]
    # Confirmed the boxscore widget still stays empty no matter how long we wait, scroll it into view, or add
    # the SAME "?league=...&season=..." query params every other FastScout URL in this notebook carries --
    # none of that fixed it when reaching the report via a fresh page.goto() (a hard full-page load). That
    # matches the EXACT pattern _click_tab_by_text already documented for other sub-routes in this app: a
    # hard page load doesn't carry over whatever client-side app/router state a normal in-app navigation
    # would leave in place, and this SPA only fully renders some views when reached via an actual in-app
    # click. report_url is kept only for the direct-PDF-response check right below (a raw HTTP fetch, not a
    # page navigation, so it's unaffected either way) -- the actual navigation into the report happens further
    # down by CLICKING the pencil icon in the still-live documents-list page instead of goto()'ing this URL.
    report_url = f"{FASTSCOUT_ORIGIN}/report/{report_id}?league={FASTSCOUT_DOCS_LEAGUE}&season={FASTSCOUT_DOCS_SEASON}"

    response = page.context.request.get(report_url)
    if not response.ok:
        raise RuntimeError(f"GET {report_url} returned HTTP {response.status}")
    content_type = response.headers.get("content-type", "")
    if "pdf" in content_type.lower():
        with open(dest_path, "wb") as f:
            f.write(response.body())
        return dest_path

    # Not a direct PDF response -- the report renders as an HTML page instead, with the SAME information a
    # manually-exported PDF would have. Rather than writing a parallel HTML-parsing path, render the
    # currently-loaded page straight to a real PDF file via headless Chromium's native print engine
    # (page.pdf() always uses "print" media -- and the report's own CSS already has "hidden-print" classes
    # on UI chrome like the edit icon, so this should closely match what a real "Print"/"Export" action
    # would produce). The saved file then flows through the EXACT SAME PDF-parsing pipeline
    # (read_boxscore_table, extract_pdf_season_stats, etc.) as any manually-provided scout PDF.
    try:
        # CLICK into the report from the live documents-list page (still open in `page` from the matching
        # step above) instead of page.goto()'ing report_url -- see the comment where report_url is built for
        # why a hard full-page load leaves the boxscore widget permanently empty. This is an in-app
        # client-side navigation, exactly like a real user opening the report would trigger.
        #
        # Confirmed from a saved snapshot's own markup: the pencil <i> icon itself has NO click handler --
        # its ancestor <tr class="... scout-row cursor-pointer ..."> is the real click target (the whole row
        # is clickable; "cursor-pointer" on the row, not the icon, is the tell). A prior attempt clicked the
        # icon directly (even with force=True) and it silently did nothing -- no exception, but page.url
        # never changed, so the code went on to save the STILL-open documents-list page as the "report" HTML
        # (confirmed by the user opening the saved file and finding the reports LIST, not an actual report).
        # Click the containing row instead, and explicitly VERIFY the navigation actually happened afterward
        # -- silently saving the wrong page if it doesn't is exactly the bug just described, so this must
        # raise loudly rather than let that repeat.
        row_locator = page.locator(f"tr:has(#pencil-{report_id})").first
        try:
            row_locator.scroll_into_view_if_needed(timeout=5000)
        except Exception:
            pass
        row_locator.click(timeout=timeout_ms)
        try:
            page.wait_for_url(re.compile(r"/report/\d+"), timeout=timeout_ms)
        except Exception:
            pass
        try:
            page.wait_for_load_state("networkidle", timeout=5000)
        except Exception:
            pass
        if "/report/" not in page.url:
            raise RuntimeError(
                f"Clicking the documents-list row for report {report_id} did not navigate into the report "
                f"(still at url={page.url}, title={page.title()!r}) -- refusing to save this page, since it "
                f"would silently be the wrong content (the documents list itself, not the report)."
            )

        # Confirmed by the user: no PDF is needed at all -- every attempt at reproducing a clean PDF via
        # Chromium's print pipeline (headless page.pdf() under default/print media, forcing window.print(),
        # manual CSS/JS hacks under screen media, clicking the real "#print" icon in a headed browser) failed
        # in a different way each time (blank body, leftover chrome frame, broken layout, or a blocking native
        # OS dialog Playwright can't dismiss). Save the live report's own rendered HTML directly instead --
        # parse_scout_html_elements() (Cell 8) reconstructs the same element schema straight from this DOM,
        # which is actually MORE reliable than pdfplumber's text-clustering heuristics on a printed PDF.
        #
        # The season boxscore ("Tile boxscore") specifically is populated by an async API call after the
        # initial page shell loads -- confirmed empty in a "Save Page As" MHTML snapshot for exactly that
        # reason. Wait for its actual text content to appear (there's no <table> tag anywhere in this app at
        # all, so the earlier "wait_for_selector('table')" calls above never actually caught anything real)
        # before saving.
        # Confirmed by a real live download: waiting (even 30s) without ever SCROLLING to the boxscore's
        # page did NOT populate it -- it stayed completely empty. This report renders across 5 separate
        # print-style "pages" (react-grid-layout items), with the boxscore on the LAST one -- a very common
        # pattern for this kind of layout is to lazy-load/virtualize off-screen pages and only fetch a
        # widget's data once it's actually scrolled into view (e.g. via IntersectionObserver). Explicitly
        # scroll the boxscore tile into view first to trigger that, before waiting for its content.
        try:
            page.locator(".Tile.boxscore").scroll_into_view_if_needed(timeout=5000)
        except Exception:
            pass

        # Confirmed fixed once (a real downloaded report showed real boxscore content) but NOT reliable --
        # a later 6-opponent run had this populate for only 1 of 6. Root cause: waiting for ".fa-spin" to
        # clear (below) is an ABSENCE-of-loading inference, not a positive content check -- if the scroll
        # above fires before whatever actually triggers the boxscore's async fetch (a race condition, since
        # this is a one-shot scroll immediately followed by polling), there's simply no spinner to see in the
        # first place, so that loop finds 0 spinners on its very FIRST check and declares victory even though
        # the boxscore table is still completely empty -- confirmed exactly by that run (5 of 6 opponents
        # saved with an empty boxscore, no exception, no spinner ever observed). Wait for an ACTUAL populated
        # row in the boxscore table itself instead, retrying the scroll a few times in case an earlier
        # attempt didn't land while the fetch was still in flight.
        boxscore_loaded = False
        for _scroll_attempt in range(5):
            try:
                page.wait_for_selector(".Tile.boxscore table tbody tr", timeout=6000)
                boxscore_loaded = True
                break
            except Exception:
                try:
                    page.locator(".Tile.boxscore").scroll_into_view_if_needed(timeout=5000)
                except Exception:
                    pass
        if not boxscore_loaded:
            print(f"    [{opponent}] boxscore table never populated after retrying the scroll {5} times -- saving anyway (season-stat cells relying on it will be incomplete).")

        # The 4 playerStatTable tiles (Top Rebounders/Scorers/3PT/FT Shooters) each load via their OWN
        # independent async call too -- confirmed one ("Top Rebounders") was still showing its loading
        # spinner (a "<i class=\"fa fa-refresh fa-spin\">" icon, with an empty <tbody>) even after the
        # boxscore had already finished. Poll for ANY ".fa-spin" element left anywhere on the page as a
        # secondary check covering all of them at once (this absence-of-loading signal is fine here since the
        # boxscore's OWN readiness is now verified by positive content above, not inferred from this alone).
        for _attempt in range(10):
            _spinner_count = page.evaluate("document.querySelectorAll('.fa-spin').length")
            if _spinner_count == 0:
                break
            page.wait_for_timeout(3000)
        else:
            print(f"    [{opponent}] {_spinner_count} loading spinner(s) still visible after 30s -- saving anyway (some stat tiles may be incomplete).")

        with open(dest_path, "w", encoding="utf-8") as f:
            f.write(page.content())
    except Exception as e:
        raise RuntimeError(
            f"GET {report_url} was not a PDF (content-type={content_type!r}), and saving its rendered HTML "
            f"also failed (url={page.url}, title={page.title()!r}): {type(e).__name__}: {e}"
        ) from e
    return dest_path


raw_uww_games = parse_raw_schedule_rows(uww_html if uww_html is not None else load_html_snapshot(uww_mhtml_path))

# Only games already played (parsed_date < reference_date) need a report for retrospective scouting, PLUS
# the single NEXT upcoming game (closest parsed_date >= reference_date) for pre-game prep -- not every game
# still remaining on the schedule after that. Mirrors the same "Upcoming" selection build_team_schedule_from_html
# uses for the primary team, computed independently here since this runs on the unfiltered raw schedule.
valid_dates = raw_uww_games.loc[raw_uww_games["parsed_date"].notna(), "parsed_date"]
upcoming_dates = valid_dates[valid_dates >= reference_date]
next_upcoming_date = upcoming_dates.min() if not upcoming_dates.empty else None


def _is_past_or_next_upcoming(d):
    if d is None:
        return False
    if d < reference_date:
        return True
    return next_upcoming_date is not None and d == next_upcoming_date


in_scope_mask = raw_uww_games["parsed_date"].apply(_is_past_or_next_upcoming)
n_future_excluded = (raw_uww_games["parsed_date"].notna() & ~in_scope_mask & (raw_uww_games["parsed_date"] >= reference_date)).sum()
if n_future_excluded:
    print(f"Excluding {n_future_excluded} game(s) scheduled beyond the next upcoming game -- not downloading those reports yet.")

# before_scout="yes" means "run as if the upcoming opponent's report doesn't exist". find_scout_files()
# already filters that report out of scout_pdf_files -- but that alone would make the upcoming game look
# MISSING to the check below and trigger a fresh download of the very report being ignored (re-creating it
# on disk, where the next re-glob would pick it back up). Drop the upcoming game from the download scope too.
if _before_scout_enabled and next_upcoming_date is not None:
    in_scope_mask = in_scope_mask & (raw_uww_games["parsed_date"] != next_upcoming_date)
    print(f"[before_scout=yes] Not downloading a scouting report for the upcoming "
          f"{next_upcoming_date:%m/%d/%y} game.")

games_needing_scout_pdf = raw_uww_games[
    raw_uww_games["opponent_url"].notna()
    & raw_uww_games["parsed_date"].notna()
    & in_scope_mask
    & ~raw_uww_games["opponent"].apply(lambda o: _scout_pdf_already_exists(o, scout_pdf_files))
]

if not fastscout_username or not fastscout_password:
    if not games_needing_scout_pdf.empty:
        print(
            f"\n{len(games_needing_scout_pdf)} game(s) are missing a local '*_scout.pdf' report and could be "
            "downloaded automatically from FastScout, but no FASTSCOUT_USERNAME/FASTSCOUT_PASSWORD were "
            "found -- skipping."
        )
elif raw_uww_games.empty:
    print("\nNo games were found in UWW's raw schedule at all (live scrape or MHTML) -- nothing to download.")
elif games_needing_scout_pdf.empty:
    n_missing_url = raw_uww_games["opponent_url"].isna().sum()
    n_missing_date = raw_uww_games["parsed_date"].isna().sum()
    print(
        f"\nEvery game already has a local scouting-report PDF, or is missing required data -- nothing to "
        f"download from FastScout. ({len(raw_uww_games)} raw game(s) found; {n_missing_url} missing "
        f"opponent_url, {n_missing_date} missing a parseable date.)"
    )
else:
    print(f"\nDownloading {len(games_needing_scout_pdf)} missing scouting-report PDF(s) from FastScout:")

    def _download_scout_pdfs(login_page):
        downloaded = {}
        for _, game_row in games_needing_scout_pdf.iterrows():
            opponent, opponent_url = game_row["opponent"], game_row["opponent_url"]
            game_date, location = game_row["parsed_date"], game_row["location"]
            # opponent_url already carries its own "?league=...&season=..." query string (e.g.
            # ".../teams/<id>?league=NCAAB-III&season=25"), so naively appending
            # "/documents?league=...&season=..." after it lands "/documents" INSIDE that first
            # query string instead of as a real path segment, producing a malformed URL. Strip
            # any existing query string before appending the documents path.
            opponent_base_url = opponent_url.split("?")[0].rstrip("/")
            docs_url = f"{opponent_base_url}/documents?league={FASTSCOUT_DOCS_LEAGUE}&season={FASTSCOUT_DOCS_SEASON}"
            try:
                saved_path = _download_matching_scout_pdf(login_page, docs_url, opponent, game_date, location)
                downloaded[opponent] = saved_path
                print(f"  [{opponent}] saved {os.path.basename(saved_path)}")
            except Exception as doc_error:
                print(f"  [{opponent}] could not download scouting report: {type(doc_error).__name__}: {doc_error}")
        return downloaded

    try:
        run_in_fastscout_session(_download_scout_pdfs)
    except Exception as download_session_error:
        print(
            "  Could not start an authenticated FastScout session for PDF downloads: "
            f"{type(download_session_error).__name__}: {download_session_error}"
        )
        traceback.print_exc()

    # Re-glob so build_team_schedule_from_html's scouted-games filter (right below) picks up whatever PDF(s)
    # were just downloaded.
    # Confirmed by the user: no PDF is needed at all -- scouting reports auto-downloaded from FastScout are now
# saved as "*_scout.html" (the live report page's own rendered DOM) instead of trying to reproduce a PDF via
# Chromium's print pipeline, which never worked reliably across headless/headed and every print-media
# variation tried. Manually-uploaded reports stay as "*_scout.pdf" (from before this change) -- glob both so
# either format counts as "this opponent already has a report".
# Routed through find_scout_files() (Configuration cell) so the `before_scout` switch is honoured here:
# with before_scout="yes" the upcoming game's own report is left out of this list entirely, which in turn
# keeps it out of scouted_opponents_for() / _scout_pdf_already_exists() below.
scout_pdf_files = find_scout_files(volume_dir)

if uww_html is not None:
    uww_team_schedule, soup, tables = build_team_schedule_from_html(uww_html, source_label="scraped live (myTeam)")
else:
    uww_team_schedule, soup, tables = build_team_schedule(uww_mhtml_path)
# The next cell (season player box-score stats) expects "soup"/"tables" for UWW specifically.
team_schedules = [uww_team_schedule]

# UWW's own real season, as a single reusable value -- every row of uww_team_schedule carries the same
# "season" (it all comes from ONE schedule snapshot, which represents one team's one season), so rather
# than thread a per-row season through every later call site that needs to re-parse one of UWW's own
# dates (the upcoming matchup date, a specific game's date when checking for a local PBP/video file,
# etc.), compute it ONCE here and reuse it everywhere below instead of the single-season assumption
# those call sites used to rely on implicitly.
# Reads the first NON-NULL "season" value rather than a bare .iloc[0] -- defensive on top of the
# allowlist fix a few lines up, rather than relying on that being the only thing standing between this
# and the same crash: any future column-nulling logic added above that forgets "season" the same way
# fails safely into the fallback below instead of crashing here again.
_uww_season_values = uww_team_schedule["season"].dropna() if "season" in uww_team_schedule.columns else pd.Series(dtype=object)
try:
    uww_season_start_year = int(str(_uww_season_values.iloc[0]).split("-")[0]) if not _uww_season_values.empty else _DEFAULT_SEASON_START_YEAR
except (ValueError, IndexError):
    uww_season_start_year = _DEFAULT_SEASON_START_YEAR
# Exposed at module level (not just inside build_team_schedule_from_html's local scope) since a later cell
# (identifying the upcoming opponent's own prior-game tendencies) also needs UWW's scouted-opponent list. The
# original notebook cell references a bare "scouted_opponents" name that was never actually assigned at module
# level either -- another pre-existing gap in the source notebook, filled in here the same way as
# opponent_from_scout_filename() above.
scouted_opponents = scouted_opponents_for(uww_team_schedule["team"].iloc[0], scout_pdf_files) if not uww_team_schedule.empty else []

# 2) For every opponent that shows up in UWW's own (scouted, date-capped) schedule, scrape their live
#    FastScout team page via opponent_url -- falling back to a local " - Schedule.mhtml" backup on failure.
opponents = (
    uww_team_schedule[["opponent", "opponent_url"]]
    .dropna(subset=["opponent_url"])
    .drop_duplicates(subset=["opponent_url"])
)
print(f"\nFetching {len(opponents)} opponent schedule(s) (scrape opponent_url, MHTML backup on failure):")

scraped_html_by_opponent = {}
if fastscout_username and fastscout_password and opponents.empty:
    print(
        f"  No opponents with a resolvable opponent_url were found in {os.path.abspath(schedules_dir)} -- "
        "skipping the authenticated FastScout session entirely (every opponent will fall back to its local "
        "backup MHTML instead, if one exists). uww_team_schedule likely ended up empty because it's filtered "
        "down to only SCOUTED games, and that filter comes from '*_scout.pdf' files found in INPUT_DIR -- if "
        "none are found there, every game gets filtered out. Double check that INPUT_DIR (the absolute path "
        "above) actually contains all the '*_scout.pdf' scouting-report files alongside the schedule "
        "MHTMLs, not just the schedule MHTMLs themselves."
    )

if fastscout_username and fastscout_password and not opponents.empty:
    def _run_fastscout_scrape_session(login_page):
        # See run_in_fastscout_session above for why this needs to run inside a dedicated worker thread
        # with a temporarily-swapped WindowsProactorEventLoopPolicy on Windows.
        session_results = {}
        for _, opp_row in opponents.iterrows():
            opponent_name, opponent_url = opp_row["opponent"], opp_row["opponent_url"]
            # Skip the live scrape entirely when a local schedule file already exists for this opponent --
            # either a genuine manually-uploaded ".mhtml" export OR this notebook's own live-scrape ".html"
            # cache from a PRIOR run (find_backup_mhtml matches both). Confirmed by the user: once a
            # schedule has been captured once, there's no need to pay for a fresh (slow, resource-heavy)
            # live scrape on every subsequent run. This only skips the ATTEMPT -- the fallback loop below
            # still loads that same file via find_backup_mhtml, so the resulting team_schedules entry is
            # unchanged either way.
            if find_backup_mhtml(opponent_name, schedules_dir):
                print(f"  Skipping live scrape for {opponent_name} -- a local schedule file already exists.")
                continue
            try:
                opp_save_path = os.path.join(schedules_dir, f"{opponent_name} - Schedule.html")
                session_results[opponent_name] = scrape_rendered_html(login_page, opponent_url, save_path=opp_save_path)
            except Exception as scrape_error:
                print(f"  Could not scrape {opponent_name} ({opponent_url}): {type(scrape_error).__name__}: {scrape_error}")
        return session_results

    try:
        scraped_html_by_opponent = run_in_fastscout_session(_run_fastscout_scrape_session)
    except Exception as session_error:
        print(f"  Could not start an authenticated FastScout browser session: {type(session_error).__name__}: {session_error}")
        traceback.print_exc()

for _, opp_row in opponents.iterrows():
    opponent_name, opponent_url = opp_row["opponent"], opp_row["opponent_url"]
    opponent_html = scraped_html_by_opponent.get(opponent_name)
    if opponent_html:
        opp_schedule, _, _ = build_team_schedule_from_html(opponent_html, source_label="scraped live")
        team_schedules.append(opp_schedule)
        continue

    backup_path = find_backup_mhtml(opponent_name, schedules_dir)
    if backup_path:
        print(f"  Falling back to local backup MHTML for {opponent_name}: {os.path.basename(backup_path)}")
        opp_schedule, _, _ = build_team_schedule(backup_path)
        team_schedules.append(opp_schedule)
    else:
        print(f"  No backup MHTML found for {opponent_name} either -- skipping this opponent.")

schedule = pd.concat(team_schedules, ignore_index=True) if team_schedules else pd.DataFrame()
print(schedule)

Found 22 schedule MHTML file(s) in ./inputs:
 - Alma Scots - Schedule.html
 - Aurora Spartans - Schedule.html
 - Carroll (WI) Pioneers - Schedule.html
 - Coe Kohawks - Schedule.html
 - Elmhurst Bluejays - Schedule.html
 - Eureka Red Devils - Schedule.html
 - Hope Flying Dutchmen - Schedule.html
 - Lawrence Vikings - Schedule.html
 - Loras Duhawks - Schedule.html
 - Ripon Red Hawks - Schedule.html
 - Simpson Storm - Schedule.html
 - St. Thomas (TX) Celts - Schedule.html
 - UW-Eau Claire Blugolds - Schedule.html
 - UW-La Crosse Eagles - Schedule.html
 - UW-Oshkosh Titans - Schedule.html
 - UW-Platteville Pioneers - Schedule.html
 - UW-River Falls Falcons - Schedule.html
 - UW-Stevens Point Pointers - Schedule.html
 - UW-Stout Blue Devils - Schedule.html
 - UW-Whitewater - Schedule.mhtml
 - UW-Whitewater - Schedule_2023.html
 - UW-Whitewater - Schedule_2024.html
  [before_scout=yes] Ignoring 21 scouting report(s) dated on/after 2025-11-19: ['11_19_25 Aurora Spartans @ UW-Whitewater_scout.


### Preview `schedule`

In [9]:
schedule

,date,opponent,team,location,opponent_url,game_url,video_url,outcome,team_score,opponent_score,point_margin,season,Upcoming
0,"Fri, Nov 7",Ripon Red Hawks,UW-Whitewater Warhawks,Away,https://fastscout.fastmodelsports.com/teams/NY...,https://fastscout.fastmodelsports.com/teams/YN...,https://editor-web.synergysports.com/video?pla...,W,76.0,58.0,18.0,2025-26,No
1,"Fri, Nov 14",St. Thomas (TX) Celts,UW-Whitewater Warhawks,Neutral Court,https://fastscout.fastmodelsports.com/teams/an...,https://fastscout.fastmodelsports.com/teams/YN...,https://editor-web.synergysports.com/video?pla...,L,73.0,81.0,-8.0,2025-26,No
2,"Sat, Nov 15",Eureka Red Devils,UW-Whitewater Warhawks,Neutral Court,https://fastscout.fastmodelsports.com/teams/jV...,https://fastscout.fastmodelsports.com/teams/YN...,https://editor-web.synergysports.com/video?pla...,W,116.0,73.0,43.0,2025-26,No
3,"Wed, Nov 19",Aurora Spartans,UW-Whitewater Warhawks,Home,https://fastscout.fastmodelsports.com/teams/Ux...,https://fastscout.fastmodelsports.com/teams/YN...,https://editor-web.synergysports.com/video?pla...,None,NaN,NaN,NaN,2025-26,Yes
4,"Fri, Nov 7",UW-Whitewater Warhawks,Ripon Red Hawks,Home,https://fastscout.fastmodelsports.com/teams/YN...,https://fastscout.fastmodelsports.com/teams/NY...,https://editor-web.synergysports.com/video?pla...,L,58.0,76.0,-18.0,2025-26,No
5,"Mon, Nov 10",Green Bay Phoenix EXH,Ripon Red Hawks,Away,https://fastscout.fastmodelsports.com/teams/nQ...,https://fastscout.fastmodelsports.com/teams/NY...,https://editor-web.synergysports.com/video?pla...,L,63.0,83.0,-20.0,2025-26,No
6,"Wed, Nov 12",St. Norbert Green Knights,Ripon Red Hawks,Away,https://fastscout.fastmodelsports.com/teams/md...,https://fastscout.fastmodelsports.com/teams/NY...,https://editor-web.synergysports.com/video?pla...,L,43.0,67.0,-24.0,2025-26,No
7,"Tue, Nov 18",Lakeland Muskies,Ripon Red Hawks,Home,https://fastscout.fastmodelsports.com/teams/2V...,https://fastscout.fastmodelsports.com/teams/NY...,https://editor-web.synergysports.com/video?pla...,L,73.0,77.0,-4.0,2025-26,No
8,"Sat, Nov 8",Biblical Studies (TX) Ambassadors,St. Thomas (TX) Celts,Home,None,https://fastscout.fastmodelsports.com/teams/an...,None,W,89.0,55.0,34.0,2025-26,No
9,"Tue, Nov 11",East Texas Baptist Tigers,St. Thomas (TX) Celts,Away,https://fastscout.fastmodelsports.com/teams/Sw...,https://fastscout.fastmodelsports.com/teams/an...,https://editor-web.synergysports.com/video?pla...,W,72.0,64.0,8.0,2025-26,No



### Extract UW-Whitewater's season player box-score stats

Reads the second `<table>` on the schedule page (`tables[1]`, captured while parsing UWW's own schedule in the previous cell) -- season-long per-player averages, not per-game data.

In [11]:
stats_raw = pd.read_html(StringIO(str(tables[1])))[0]

# This table comes from FastScout's live-rendered "season player box-score" widget (see the extensive
# lazy-load/virtualization workarounds for it elsewhere in this notebook -- it's a genuinely quirky piece of
# markup to scrape). Confirmed downstream (the Streamlit app crashing with "truth value of a Series is
# ambiguous" the moment it read a "PTS" cell out of this table): pd.read_html can come back with two columns
# sharing the exact same header text -- e.g. a sortable-column control or a hidden group label that doesn't
# survive read_html's flattening, so what LOOKS like one header cell in the browser produces two identically-
# named columns here. Any df[col] or row[col] access on a duplicated name then silently returns a Series
# instead of a scalar instead of raising, so this is worth catching loudly right at the source rather than
# letting it surface downstream as a cryptic app crash.
_dupe_cols = stats_raw.columns[stats_raw.columns.duplicated()].unique().tolist()
if _dupe_cols:
    print(f"WARNING: stats_raw has duplicate column name(s) {_dupe_cols} -- keeping the FIRST occurrence of "
          f"each and dropping the rest. If the dropped column actually held different data (e.g. a genuine "
          f"second stat under the same header text) this silently loses it -- inspect stats_raw.columns and "
          f"the raw `tables[1]` HTML directly if that seems possible for this table.")
    stats_raw = stats_raw.loc[:, ~stats_raw.columns.duplicated()]

# Drop the leading unnamed/blank columns (they held player headshot/jersey-icon cells with no text)
stats = stats_raw.drop(columns=[c for c in stats_raw.columns if c.startswith("Unnamed")])
stats = stats.rename(columns={"#": "jersey_number"})

print(stats)

    jersey_number                  PLAYER  GP-GS   MIN      FGM-A    FG%  \
0            14.0            Mikey Wildes    4-0     2    0.0-0.2     0%   
1            24.0           Richie Warren  29-24    22    3.5-8.1  43.2%   
2             0.0            Isaac Verges  27-26    23    3.5-6.7  52.5%   
3            23.0          Mauryon Turner      -     -          -      -   
4            23.0         Maurquis Turner    3-0     3    0.7-1.0  66.7%   
5            40.0  Tyshawn Teague-Johnson   18-1    11    1.7-3.9  42.9%   
6            35.0           Rashad Rogers   19-2     9    0.8-1.7  48.5%   
7             5.0         Isaiah Robinson   20-0     8    0.4-1.0  33.3%   
8            12.0              Jake Quast   28-0    16    2.0-4.0  51.4%   
9            30.0           Matthew McKay    3-0     2    0.7-0.7   100%   
10            2.0           Kelton McEwen   28-0    11    1.1-2.6  41.7%   
11           21.0            Brock Marino  28-27    21    3.2-6.1  52.3%   
12          


### Investigate `playerStatTable` tile widgets (Top Rebounders/Scorers/3PT/FT)

The season stats table above (`tables[1]`) is FastScout's plain per-player-averages `<table>` -- it does NOT cover the separate "Top Rebounders" / "Top Scorers" / "Top 3PT" / "Top FT" leaderboard TILES that also appear somewhere on this schedule page (a different UI widget, presumably div/card-based rather than a `<table>`, so `pd.read_html` over `page_tables` would never see it). This diagnostic cell re-parses the same saved schedule snapshot (`uww_mhtml_path`) with a fresh, full-page `BeautifulSoup` parse and searches for any element that looks like those tiles -- by class name and by known label text -- so their real markup can be confirmed before writing extraction logic for them. Run this cell (it only reads the already-saved local backup file, no live scrape needed) and share the printed output.

In [13]:
# --- Diagnostic: locate the "Top Rebounders/Scorers/3PT/FT" tile widgets on the schedule page -------------
# These are presumed to be a SEPARATE UI widget from the plain season stats table (tables[1], extracted
# above) -- likely rendered as div/card elements, not a <table>, so pd.read_html over page_tables never sees
# them. Re-parse the FULL page (not just the <table> elements already captured in `tables`) and search two
# ways since the exact markup isn't confirmed yet: (1) any element whose class attribute mentions "stat"
# (case-insensitive) that ISN'T a <table> itself, and (2) any element whose own text contains one of the
# known tile labels. Prints just enough of each match (tag, classes, a short text preview) to identify the
# real selector to parse against, without assuming a specific structure upfront.
tile_investigation_html = load_html_snapshot(uww_mhtml_path)
tile_investigation_soup = BeautifulSoup(tile_investigation_html, "lxml")

TILE_LABELS = ["Top Rebounders", "Top Scorers", "Top 3PT", "Top FT", "Top Assists", "Leaders"]

print("Elements with a 'stat'-like class attribute (excluding <table> itself):")
stat_class_matches = [
    el for el in tile_investigation_soup.find_all(True, class_=True)
    if el.name != "table" and any("stat" in c.lower() for c in el.get("class", []))
]
print(f"  found {len(stat_class_matches)}")
for el in stat_class_matches[:15]:
    preview = el.get_text(" ", strip=True)[:80]
    print(f"  <{el.name} class={el.get('class')}> -- text preview: {preview!r}")

print(f"\nElements whose text contains a known tile label {TILE_LABELS}:")
found_any = False
for label in TILE_LABELS:
    matches = tile_investigation_soup.find_all(string=re.compile(re.escape(label), re.IGNORECASE))
    if matches:
        found_any = True
    for txt in matches[:5]:
        ancestor = txt.parent
        for _ in range(3):
            if ancestor is None or ancestor.get("class") or ancestor.parent is None:
                break
            ancestor = ancestor.parent
        if ancestor is not None:
            print(f"  [{label}] <{ancestor.name} class={ancestor.get('class')}> -- text preview: {ancestor.get_text(' ', strip=True)[:120]!r}")
if not found_any:
    print("  (none found in this saved snapshot -- these tiles may not exist on this page, may use "
          "different wording, or may not have been present/loaded when this snapshot was captured)")

print(f"\nTotal <table> elements on this page: {len(tile_investigation_soup.find_all('table'))}")

Elements with a 'stat'-like class attribute (excluding <table> itself):
  found 27
  <tr class=['stat-table-row', 'stat-table-row']> -- text preview: '# PLAYER GP-GS MIN FGM-A FG% 3PM-A 3P% 3P-R FTM-A FT% PTS ORB DRB REB AST TO STL'
  <tr class=['stat-table-row', 'stat-table-row', 'no-border-row']> -- text preview: '14 Mikey Wildes 4 - 0 2 0.0 - 0.2 0% 0.0 - 0.0 - 0% 0.0 - 0.0 - 0.0 0.0 0.2 0.2 '
  <tr class=['stat-table-row', 'stat-table-row', 'no-border-row']> -- text preview: '24 Richie Warren 29 - 24 22 3.5 - 8.1 43.2% 0.3 - 1.2 25% 15.3% 1.3 - 2.1 65% 8.'
  <tr class=['stat-table-row', 'stat-table-row', 'no-border-row']> -- text preview: '0 Isaac Verges 27 - 26 23 3.5 - 6.7 52.5% 0.4 - 1.2 37.5% 17.7% 2.3 - 3.3 71.6% '
  <tr class=['stat-table-row', 'stat-table-row', 'no-border-row']> -- text preview: '23 Mauryon Turner - - - - - - - - - - - - - - - - - -'
  <tr class=['stat-table-row', 'stat-table-row', 'no-border-row']> -- text preview: '23 Maurquis Turner 3 - 0 3 0.7 - 1.0 66.7


### Investigate the `myTeam` analytics/dashboard landing page for the same tile widgets

The diagnostic above found nothing on the SCHEDULE tab -- no `playerStatTable`-style tile widget, and no "Top Rebounders/Scorers/3PT/FT" text anywhere on that page. `_ensure_fastscout_login` (Cell 4) now also saves a diagnostic snapshot of the login's actual LANDING page -- `.../teams/myTeam/analytics/dashboard?...` -- to `myTeam - Analytics Dashboard.html` every time a live FastScout session bootstraps, since that's a different page from SCHEDULE that nothing has captured or searched yet. This cell re-runs the same tile-detection search against that file, once it exists locally (requires a live session with credentials to have run at least once).

In [15]:
dashboard_snapshot_path = os.path.join(schedules_dir, "myTeam - Analytics Dashboard.html")

if not os.path.exists(dashboard_snapshot_path):
    print(f"'{os.path.basename(dashboard_snapshot_path)}' doesn't exist yet -- run a live FastScout session with credentials first, then re-run this cell.")
else:
    dashboard_html = load_html_snapshot(dashboard_snapshot_path)
    dashboard_soup = BeautifulSoup(dashboard_html, "lxml")
    dash_tile_labels = ["Top Rebounders", "Top Scorers", "Top 3PT", "Top FT", "Top Assists", "Leaders"]

    print("Elements with a 'stat'-like class attribute (excluding <table> itself):")
    dash_stat_matches = [
        el for el in dashboard_soup.find_all(True, class_=True)
        if el.name != "table" and any("stat" in c.lower() for c in el.get("class", []))
    ]
    print(f"  found {len(dash_stat_matches)}")
    for el in dash_stat_matches[:15]:
        preview = el.get_text(" ", strip=True)[:80]
        print(f"  <{el.name} class={el.get('class')}> -- text preview: {preview!r}")

    print(f"\nElements whose text contains a known tile label {dash_tile_labels}:")
    dash_found_any = False
    for label in dash_tile_labels:
        matches = dashboard_soup.find_all(string=re.compile(re.escape(label), re.IGNORECASE))
        if matches:
            dash_found_any = True
        for txt in matches[:5]:
            ancestor = txt.parent
            for _ in range(3):
                if ancestor is None or ancestor.get("class") or ancestor.parent is None:
                    break
                ancestor = ancestor.parent
            if ancestor is not None:
                print(f"  [{label}] <{ancestor.name} class={ancestor.get('class')}> -- text preview: {ancestor.get_text(' ', strip=True)[:120]!r}")
    if not dash_found_any:
        print("  (none found on the analytics/dashboard page either)")

    print(f"\nTotal <table> elements on this page: {len(dashboard_soup.find_all('table'))}")

Elements with a 'stat'-like class attribute (excluding <table> itself):
  found 4
  <div class=['Tile', 'teamStatsLineup', 'top-left-tile']> -- text preview: 'LINEUP STATS Loading...'
  <div class=['teamStatsLineup']> -- text preview: 'Loading...'
  <div class=['Tile', 'teamAdvancedStatsTable', 'top-left-tile']> -- text preview: 'ADVANCED STATS Loading...'
  <div class=['teamAdvancedStatsTable']> -- text preview: 'Loading...'

Elements whose text contains a known tile label ['Top Rebounders', 'Top Scorers', 'Top 3PT', 'Top FT', 'Top Assists', 'Leaders']:
  [Leaders] <a class=['helvetica-neue-bold', 'uppercase']> -- text preview: 'Leaders'

Total <table> elements on this page: 0


In [16]:
dashboard_snapshot_path = os.path.join(schedules_dir, "myTeam - Analytics Dashboard.html")

if not os.path.exists(dashboard_snapshot_path):
    print(f"'{os.path.basename(dashboard_snapshot_path)}' doesn't exist yet -- run a live FastScout session with credentials first, then re-run this cell.")
else:
    dashboard_html = load_html_snapshot(dashboard_snapshot_path)
    dashboard_soup = BeautifulSoup(dashboard_html, "lxml")
    dash_tile_labels = ["Top Rebounders", "Top Scorers", "Top 3PT", "Top FT", "Top Assists", "Leaders"]

    print("Elements with a 'stat'-like class attribute (excluding <table> itself):")
    dash_stat_matches = [
        el for el in dashboard_soup.find_all(True, class_=True)
        if el.name != "table" and any("stat" in c.lower() for c in el.get("class", []))
    ]
    print(f"  found {len(dash_stat_matches)}")
    for el in dash_stat_matches[:15]:
        preview = el.get_text(" ", strip=True)[:80]
        print(f"  <{el.name} class={el.get('class')}> -- text preview: {preview!r}")

    print(f"\nElements whose text contains a known tile label {dash_tile_labels}:")
    dash_found_any = False
    for label in dash_tile_labels:
        matches = dashboard_soup.find_all(string=re.compile(re.escape(label), re.IGNORECASE))
        if matches:
            dash_found_any = True
        for txt in matches[:5]:
            ancestor = txt.parent
            for _ in range(3):
                if ancestor is None or ancestor.get("class") or ancestor.parent is None:
                    break
                ancestor = ancestor.parent
            if ancestor is not None:
                print(f"  [{label}] <{ancestor.name} class={ancestor.get('class')}> -- text preview: {ancestor.get_text(' ', strip=True)[:120]!r}")
    if not dash_found_any:
        print("  (none found on the analytics/dashboard page either)")

    print(f"\nTotal <table> elements on this page: {len(dashboard_soup.find_all('table'))}")

Elements with a 'stat'-like class attribute (excluding <table> itself):
  found 4
  <div class=['Tile', 'teamStatsLineup', 'top-left-tile']> -- text preview: 'LINEUP STATS Loading...'
  <div class=['teamStatsLineup']> -- text preview: 'Loading...'
  <div class=['Tile', 'teamAdvancedStatsTable', 'top-left-tile']> -- text preview: 'ADVANCED STATS Loading...'
  <div class=['teamAdvancedStatsTable']> -- text preview: 'Loading...'

Elements whose text contains a known tile label ['Top Rebounders', 'Top Scorers', 'Top 3PT', 'Top FT', 'Top Assists', 'Leaders']:
  [Leaders] <a class=['helvetica-neue-bold', 'uppercase']> -- text preview: 'Leaders'

Total <table> elements on this page: 0


In [17]:
dashboard_snapshot_path = os.path.join(schedules_dir, "myTeam - Analytics Dashboard.html")

if not os.path.exists(dashboard_snapshot_path):
    print(
        f"'{os.path.basename(dashboard_snapshot_path)}' doesn't exist yet in {schedules_dir} -- it's only saved "
        "the first time a live FastScout session actually bootstraps (Cell 4's _ensure_fastscout_login), which "
        "requires FASTSCOUT_USERNAME/FASTSCOUT_PASSWORD to be set. Run any cell that live-scrapes at least once "
        "with credentials, then re-run this cell."
    )
else:
    dashboard_html = load_html_snapshot(dashboard_snapshot_path)
    dashboard_soup = BeautifulSoup(dashboard_html, "lxml")
    dash_tile_labels = ["Top Rebounders", "Top Scorers", "Top 3PT", "Top FT", "Top Assists", "Leaders"]

    print("Elements with a 'stat'-like class attribute (excluding <table> itself):")
    dash_stat_matches = [
        el for el in dashboard_soup.find_all(True, class_=True)
        if el.name != "table" and any("stat" in c.lower() for c in el.get("class", []))
    ]
    print(f"  found {len(dash_stat_matches)}")
    for el in dash_stat_matches[:15]:
        preview = el.get_text(" ", strip=True)[:80]
        print(f"  <{el.name} class={el.get('class')}> -- text preview: {preview!r}")

    print(f"\nElements whose text contains a known tile label {dash_tile_labels}:")
    dash_found_any = False
    for label in dash_tile_labels:
        matches = dashboard_soup.find_all(string=re.compile(re.escape(label), re.IGNORECASE))
        if matches:
            dash_found_any = True
        for txt in matches[:5]:
            ancestor = txt.parent
            for _ in range(3):
                if ancestor is None or ancestor.get("class") or ancestor.parent is None:
                    break
                ancestor = ancestor.parent
            if ancestor is not None:
                print(f"  [{label}] <{ancestor.name} class={ancestor.get('class')}> -- text preview: {ancestor.get_text(' ', strip=True)[:120]!r}")
    if not dash_found_any:
        print("  (none found on the analytics/dashboard page either)")

    print(f"\nTotal <table> elements on this page: {len(dashboard_soup.find_all('table'))}")

Elements with a 'stat'-like class attribute (excluding <table> itself):
  found 4
  <div class=['Tile', 'teamStatsLineup', 'top-left-tile']> -- text preview: 'LINEUP STATS Loading...'
  <div class=['teamStatsLineup']> -- text preview: 'Loading...'
  <div class=['Tile', 'teamAdvancedStatsTable', 'top-left-tile']> -- text preview: 'ADVANCED STATS Loading...'
  <div class=['teamAdvancedStatsTable']> -- text preview: 'Loading...'

Elements whose text contains a known tile label ['Top Rebounders', 'Top Scorers', 'Top 3PT', 'Top FT', 'Top Assists', 'Leaders']:
  [Leaders] <a class=['helvetica-neue-bold', 'uppercase']> -- text preview: 'Leaders'

Total <table> elements on this page: 0


In [18]:
dashboard_snapshot_path = os.path.join(schedules_dir, "myTeam - Analytics Dashboard.html")

if not os.path.exists(dashboard_snapshot_path):
    print(
        f"'{os.path.basename(dashboard_snapshot_path)}' doesn't exist yet in {schedules_dir} -- it's only saved "
        "the first time a live FastScout session actually bootstraps (Cell 4's _ensure_fastscout_login), which "
        "requires FASTSCOUT_USERNAME/FASTSCOUT_PASSWORD to be set. Run any cell that live-scrapes at least once "
        "with credentials, then re-run this cell."
    )
else:
    dashboard_html = load_html_snapshot(dashboard_snapshot_path)
    dashboard_soup = BeautifulSoup(dashboard_html, "lxml")
    dash_tile_labels = ["Top Rebounders", "Top Scorers", "Top 3PT", "Top FT", "Top Assists", "Leaders"]

    print("Elements with a 'stat'-like class attribute (excluding <table> itself):")
    dash_stat_matches = [
        el for el in dashboard_soup.find_all(True, class_=True)
        if el.name != "table" and any("stat" in c.lower() for c in el.get("class", []))
    ]
    print(f"  found {len(dash_stat_matches)}")
    for el in dash_stat_matches[:15]:
        preview = el.get_text(" ", strip=True)[:80]
        print(f"  <{el.name} class={el.get('class')}> -- text preview: {preview!r}")

    print(f"\nElements whose text contains a known tile label {dash_tile_labels}:")
    dash_found_any = False
    for label in dash_tile_labels:
        matches = dashboard_soup.find_all(string=re.compile(re.escape(label), re.IGNORECASE))
        if matches:
            dash_found_any = True
        for txt in matches[:5]:
            ancestor = txt.parent
            for _ in range(3):
                if ancestor is None or ancestor.get("class") or ancestor.parent is None:
                    break
                ancestor = ancestor.parent
            if ancestor is not None:
                print(f"  [{label}] <{ancestor.name} class={ancestor.get('class')}> -- text preview: {ancestor.get_text(' ', strip=True)[:120]!r}")
    if not dash_found_any:
        print("  (none found on the analytics/dashboard page either)")

    print(f"\nTotal <table> elements on this page: {len(dashboard_soup.find_all('table'))}")

Elements with a 'stat'-like class attribute (excluding <table> itself):
  found 4
  <div class=['Tile', 'teamStatsLineup', 'top-left-tile']> -- text preview: 'LINEUP STATS Loading...'
  <div class=['teamStatsLineup']> -- text preview: 'Loading...'
  <div class=['Tile', 'teamAdvancedStatsTable', 'top-left-tile']> -- text preview: 'ADVANCED STATS Loading...'
  <div class=['teamAdvancedStatsTable']> -- text preview: 'Loading...'

Elements whose text contains a known tile label ['Top Rebounders', 'Top Scorers', 'Top 3PT', 'Top FT', 'Top Assists', 'Leaders']:
  [Leaders] <a class=['helvetica-neue-bold', 'uppercase']> -- text preview: 'Leaders'

Total <table> elements on this page: 0



### Summarize UW-Whitewater's season record and home/away/neutral splits

`schedule` spans every team found in `schedules_dir` (UWW's own games plus each opponent's) -- filtered down to just UWW's own rows before aggregating the win/loss record and average point-margin splits by location.

In [20]:
# "schedule" now spans every team found in schedules_dir (UWW's own games plus any opponent's, e.g.
# Elmhurst) -- this record/splits summary is specifically about UW-Whitewater, so filter down to just its
# rows before aggregating; otherwise the win/loss counts and splits below would blend in other teams' games too.
uww_schedule = schedule[schedule["team"] == "UW-Whitewater Warhawks"]

record_counts = uww_schedule["outcome"].value_counts()
print("Overall record (W-L):", f"{record_counts.get('W', 0)}-{record_counts.get('L', 0)}")

split_summary = (
    uww_schedule.groupby(["location", "outcome"])
    .size()
    .unstack(fill_value=0)
    .assign(games=lambda d: d.sum(axis=1))
)
print(split_summary.reset_index())

avg_margin_by_location = uww_schedule.groupby("location")["point_margin"].mean().round(1)
print(avg_margin_by_location.reset_index().rename(columns={"point_margin": "avg_point_margin"}))

Overall record (W-L): 2-1
outcome       location  L  W  games
0                 Away  0  1      1
1        Neutral Court  1  1      2
        location  avg_point_margin
0           Away              18.0
1           Home               NaN
2  Neutral Court              17.5



### Parse each opponent's FastScout "ScoutBuilder" game-plan report

Scouting reports are sourced exclusively from PDF exports (MHTML snapshots leave live-data stat widgets empty). Since `ai_parse_document()` only runs on serverless AI Functions compute, this falls back to `pdfplumber` and reconstructs the same `section_header`/`text`/`table` element schema used by every downstream cell, looping over every `"*_scout.pdf"` file found in `INPUT_DIR`.

In [22]:
# FastScout "ScoutBuilder" game-plan reports, now sourced EXCLUSIVELY from PDF exports -- MHTML is retired.
# PDF exports render everything server-side before printing, so their tables are fully populated (unlike MHTML
# "Save Page As" snapshots, whose live-data stat widgets are empty placeholders). The original ai_parse_document()
# approach fails on this classic cluster because that routine only runs on serverless AI Functions compute, so this
# cell falls back to pdfplumber and reconstructs the same element schema used downstream: section_header / text / table.
# Loop through every "*_scout.pdf" file in the inputs volume so newly added reports are picked up automatically
# without touching this code. 
try:
    import pdfplumber
except ModuleNotFoundError:
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pdfplumber"])
    import pdfplumber

logging.getLogger("pdfminer").setLevel(logging.ERROR)

GAME_PLAN_HEADERS = {
    "TEAM STRENGTHS", "KEYS TO VICTORY",
    "Overall Defensive Scheme", "Attacking their man defense", "Ball Screen & DHO Defense",
    "Ball Screen Actions & Reads", "Speciality Defensive Notes", "Potential Adjustments",
    "Defending Their Action", "Overall OFFENSIVE SCHEME", "Ball Screen Actions & Personnel",
    "Ball Screen Coverage(s)", "ELOB & SLOB",
}
PLAYER_LINE_RE = re.compile(r"^#\d+\s*•")
PAGE_NUM_RE = re.compile(r"^\d+\s+of\s+\d+$")

volume_dir = INPUT_DIR
# Confirmed by the user: no PDF is ever needed -- reports auto-downloaded going forward are saved as the live
# report page's own rendered HTML ("*_scout.html", see Cell 4) instead of a Chromium-printed PDF, which never
# rendered cleanly no matter the approach tried. Manually-uploaded reports from before this change stay as
# "*_scout.pdf". Both are parsed below (by their own dedicated parser) into the identical element schema, so
# every downstream cell keeps working unchanged regardless of which format a given opponent's report is in.
# find_scout_files() (Configuration cell) applies the `before_scout` switch -- with before_scout="yes" the
# upcoming game's report is excluded here too, so no scout_reports entry is built for that opponent and
# every downstream cell behaves exactly as it does for an opponent whose report hasn't been made yet.
scout_pdf_files = find_scout_files(volume_dir, extensions=("pdf",))
scout_html_files = find_scout_files(volume_dir, extensions=("html",))
scout_report_files = sorted(scout_pdf_files + scout_html_files)
print(f"Found {len(scout_pdf_files)} PDF scout report(s) and {len(scout_html_files)} HTML scout report(s):")
for f in scout_report_files:
    print(" -", os.path.basename(f))


def normalize_text(text):
    text = text.replace("• ", "•").replace(" •", "•")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def grouped_lines(page):
    # Bucketing words by a bare round(word["top"]) is fragile: the LEFT half's baseline ("Player Notes:")
    # and the RIGHT half's baseline ("Keys to Defending:") of the SAME visual header line can land on
    # different sub-pixel tops that round to ADJACENT integers (e.g. 556 vs 557) rather than the same one.
    # When that happens, the two halves get split into two separate one-sided rows, the "Player Notes" +
    # "Keys to Defending" combined-header detection below never fires on either row, and that whole player's
    # notes/keys bullets get silently dropped (confirmed for Michael Asman and Kolby Williams on the Ripon
    # PDF's STARTERS page -- their header row split 556/557 and 684/685). Cluster words within a small
    # vertical tolerance into the same row instead of relying on exact/rounded top equality.
    split_x = page.width / 2
    words = page.extract_words()
    ordered = sorted(words, key=lambda w: (w["top"], w["x0"]))
    clusters = []
    row_top_tolerance = 2
    for word in ordered:
        if clusters and abs(word["top"] - clusters[-1]["top"]) <= row_top_tolerance:
            clusters[-1]["words"].append(word)
        else:
            clusters.append({"top": word["top"], "words": [word]})

    rows = []
    for cluster in clusters:
        line_words = sorted(cluster["words"], key=lambda w: w["x0"])
        rows.append({
            "top": cluster["top"],
            "left": normalize_text(" ".join(w["text"] for w in line_words if w["x0"] < split_x)),
            "right": normalize_text(" ".join(w["text"] for w in line_words if w["x0"] >= split_x)),
            "full": normalize_text(" ".join(w["text"] for w in line_words)),
        })
    return rows


def parse_game_plan_page(page, elements):
    segments = []
    for side_name in ["left", "right"]:
        current = None
        for row in grouped_lines(page):
            if row["top"] < 80:
                continue
            text = row[side_name]
            if not text or PAGE_NUM_RE.fullmatch(text) or text in {"STARTERS", "BENCH"}:
                continue
            header_candidate = text.rstrip(" -•")
            if header_candidate in GAME_PLAN_HEADERS:
                current = {
                    "top": row["top"],
                    "side": 0 if side_name == "left" else 1,
                    "header": header_candidate,
                    "texts": [],
                }
                segments.append(current)
            elif text.upper() == text and len(text) > 8 and not re.match(r"^\d+\.", text):
                continue
            elif current is not None:
                current["texts"].append(text)

    for segment in sorted(segments, key=lambda s: (s["top"], s["side"])):
        elements.append(("section_header", segment["header"]))
        for text in segment["texts"]:
            elements.append(("text", text))


def parse_roster_page(page, elements):
    in_notes_block = False
    for row in grouped_lines(page):
        if row["top"] < 80:
            continue
        left, right, full = row["left"], row["right"], row["full"]
        if PAGE_NUM_RE.fullmatch(full):
            continue
        if full in {"STARTERS", "BENCH"}:
            elements.append(("section_header", full))
            in_notes_block = False
            continue
        if PLAYER_LINE_RE.match(left or full):
            elements.append(("text", left or full))
            in_notes_block = False
            continue
        if (
            left.startswith("GP-GS")
            or full.startswith("GP-GS")
            or left.startswith("Last Season")
            or full.startswith("Last Season")
            or re.match(r"^\d{2}-\d{2}\s*\(", left or full)
            or re.match(r"^\d{2}-\d{2}\s*\(", full)
        ):
            continue
        if "Player Notes" in full or "Keys to Defending" in full:
            in_notes_block = True
            continue
        if in_notes_block:
            if left:
                elements.append(("section_header", "Player Notes"))
                elements.append(("text", left))
            if right:
                elements.append(("section_header", "Keys to Defending"))
                elements.append(("text", right))


def parse_boxscore_page(page, elements):
    lines = [
        row["full"].replace("", "").strip()
        for row in grouped_lines(page)
        if row["top"] >= 80 and row["full"].strip()
    ]
    box_idx = next((i for i, line in enumerate(lines) if "BOXSCORE" in line.upper()), None)
    header_idx = next((i for i in range(box_idx + 1, len(lines)) if lines[i].startswith("# PLAYER ")), None) if box_idx is not None else None
    if box_idx is None or header_idx is None:
        return

    stop_idx = next(
        (
            i for i in range(header_idx + 1, len(lines))
            if lines[i].startswith("Top Scorers")
            or lines[i].startswith("3PT Shooters")
            or PAGE_NUM_RE.fullmatch(lines[i])
        ),
        len(lines),
    )
    header_tokens = lines[header_idx].split()
    stat_cols = header_tokens[2:]
    rows = []
    for line in lines[header_idx + 1:stop_idx]:
        tokens = line.split()
        if len(tokens) < 2 + len(stat_cols):
            continue
        name = " ".join(tokens[1:-len(stat_cols)])
        if name == "TeamTotal":
            name = "Team Total"
        rows.append([tokens[0], name] + tokens[-len(stat_cols):])

    if rows:
        box_df = pd.DataFrame(rows, columns=["#", "PLAYER"] + stat_cols)
        elements.append(("section_header", lines[box_idx]))
        elements.append(("table", box_df.to_html(index=False)))


def parse_scout_pdf_elements(path):
    elements = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text() or ""
            if "BOXSCORE" in page_text:
                parse_boxscore_page(page, elements)
            elif "STARTERS" in page_text or "BENCH" in page_text or re.search(r"#\d+•", page_text):
                parse_roster_page(page, elements)
            else:
                parse_game_plan_page(page, elements)

    # pd.DataFrame([]) on an empty list produces a DataFrame with NO COLUMNS at all (not just 0 rows) --
    # explicitly pin the expected columns so a PDF that yields zero elements (e.g. a page layout these
    # heuristics don't recognize) still reports "0 elements, 0 tables" downstream instead of crashing with
    # a KeyError on element_type.
    return pd.DataFrame(
        [
            {"element_index": idx, "element_type": element_type, "element_content": element_content}
            for idx, (element_type, element_content) in enumerate(elements)
        ],
        columns=["element_index", "element_type", "element_content"],
    )


# NOTE: home/away-aware opponent-name extraction for scout report filenames -- ported as-is from the original
# notebook cell, which called an "opponent_from_scout_filename()" that was never actually defined there
# either (a pre-existing bug in the source notebook, not introduced by this portable conversion). This local
# helper fills that gap using the same "<date> <A> @ <B>_scout.<ext>" filename convention documented below --
# handles both the legacy "_scout.pdf" and the new "_scout.html" extension.
def opponent_from_scout_filename(path):
    name = re.sub(r"_scout\.(pdf|html)$", "", os.path.basename(path))
    name = re.sub(r"^\d+_\d+_\d+\s+", "", name)
    if " @ " not in name:
        return name
    left, right = [side.strip() for side in name.split(" @ ", 1)]
    return right if "whitewater" in left.lower() else left


def normalize_html_text(text):
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def _has_exact_class(tag, cls):
    # BeautifulSoup's class_=lambda predicate is invoked once PER CLASS TOKEN (as a bare string), not once
    # per tag with the full class list -- so a naive "cls in tag['class']" substring/membership check done
    # the wrong way (e.g. "Tile" in "EditableTile") can silently match an unrelated wrapper class too. Do the
    # membership check explicitly against the tag's own parsed class LIST instead, so only an exact token
    # match counts (confirmed: without this, "Tile" incorrectly matched "EditableTile" and duplicated every
    # game-plan/roster element).
    classes = tag.get("class")
    return bool(classes) and cls in classes


def _draft_editor_blocks(tile):
    """Every paragraph/bullet line inside a Tile's rich-text (Draft.js) content shares the class
    'public-DraftStyleDefault-block', whether it's wrapped in an <li> (bulleted/numbered list) or a plain
    <div> (unformatted lines, e.g. KEYS TO VICTORY's numbered lines) -- selecting on that class uniformly
    covers both cases in visual top-to-bottom order, instead of special-casing <li> vs plain paragraphs."""
    draft = tile.find(class_="public-DraftEditor-content")
    if draft is None:
        return []
    blocks = [b for b in draft.find_all(True) if _has_exact_class(b, "public-DraftStyleDefault-block")]
    return [normalize_html_text(b.get_text(" ", strip=True)) for b in blocks]


def parse_scout_html_elements(path):
    """HTML counterpart to parse_scout_pdf_elements() -- reconstructs the identical element schema
    (element_index, element_type in {section_header, text, table}, element_content) directly from the
    ScoutBuilder report's own live DOM (saved by Cell 4's download step), instead of from a printed PDF.
    Confirmed structurally MORE reliable than pdfplumber's text-clustering heuristics: game-plan bullets,
    player notes, and keys-to-defending are each their own clean DOM node here, with no PDF column-merging
    to work around (the Eureka-style merged notes/keys blob that split_combined_notes_keys() exists for in
    Cell 9 simply can't happen with this source).
    """
    with open(path, "r", encoding="utf-8") as f:
        html = f.read()
    soup = BeautifulSoup(html, "html.parser")
    printable = soup.find(class_="PrintableNode")
    elements = []
    if printable is None:
        return pd.DataFrame([], columns=["element_index", "element_type", "element_content"])

    # ---- game-plan tiles: any exact-class "Tile" whose title span matches a known header, in document order.
    # Confirmed the site does NOT reliably split these across pages the same way every time (unlike the PDF
    # export) -- parsing the whole PrintableNode in one pass sidesteps relying on any particular page boundary.
    for tile in printable.find_all("div"):
        if not _has_exact_class(tile, "Tile"):
            continue
        title_el = tile.find("span", class_="scout-tile-title")
        if title_el is None:
            continue
        header = normalize_html_text(title_el.get_text(strip=True))
        if header not in GAME_PLAN_HEADERS:
            continue
        elements.append(("section_header", header))
        for block in _draft_editor_blocks(tile):
            elements.append(("text", block))

    # ---- roster tiles: STARTERS/BENCH section headers interleaved with each player's own "playerGroup" tile,
    # in document order (this interleaving matters -- Cell 9 infers Starter vs Bench role from each player's
    # position relative to the BENCH header).
    for tile in printable.find_all("div"):
        if not _has_exact_class(tile, "Tile"):
            continue
        classes = tile.get("class")
        if "section" in classes:
            header_text = normalize_html_text(tile.get_text(" ", strip=True))
            if header_text in ("STARTERS", "BENCH"):
                elements.append(("section_header", header_text))
        elif "playerGroup" in classes:
            info_span = tile.find("span", class_="player-info-line")
            if info_span is None:
                continue
            info_div = info_span.find("div", class_=lambda c: c and "display-flex" in c)
            # Direct-child <span>s only (jersey/name/pos/height/weight/class in order) -- reading them
            # positionally (rather than via .stripped_strings, which silently DROPS an empty span, e.g. a
            # player missing a weight) keeps all 6 fields and 5 "•" separators intact even when one field is
            # blank, matching Cell 9's PLAYER_LINE_RE exactly (it already tolerates an empty weight group).
            field_spans = info_div.find_all("span", recursive=False) if info_div else []
            fields = [normalize_html_text(s.get_text(" ", strip=True)) for s in field_spans]
            elements.append(("text", " • ".join(fields)))

            for text_tile in tile.find_all("div"):
                if not (_has_exact_class(text_tile, "Tile") and _has_exact_class(text_tile, "text")):
                    continue
                blocks = [b for b in _draft_editor_blocks(text_tile) if b]
                if not blocks:
                    continue
                label = blocks[0].lower()
                if label.startswith("player notes"):
                    section_name = "Player Notes"
                elif label.startswith("keys to defending"):
                    section_name = "Keys to Defending"
                else:
                    continue
                elements.append(("section_header", section_name))
                for block in blocks:
                    if block.lower().startswith(section_name.lower()):
                        continue
                    elements.append(("text", block))

    # ---- season boxscore: confirmed against a real live-downloaded report -- once loaded (Cell 4's download
    # step explicitly waits for it), it's a genuine <table> (not a div-grid as originally guessed), with the
    # same PLAYER/Team Total/Opponent row shape as the PDF version. Emit the same (section_header, "table")
    # element pair the PDF path produces (a "...BOXSCORE" header immediately followed by a table element) so
    # extract_team_totals_from_pdf() downstream picks it up unchanged regardless of source format.
    boxscore_tile = next(
        (t for t in printable.find_all("div") if _has_exact_class(t, "Tile") and _has_exact_class(t, "boxscore")),
        None,
    )
    if boxscore_tile is not None:
        boxscore_table = boxscore_tile.find("table")
        boxscore_tbody = boxscore_table.find("tbody") if boxscore_table is not None else None
        if boxscore_tbody is not None and boxscore_tbody.find("tr") is not None:
            title_el = boxscore_tile.find("span", class_="scout-tile-title")
            header_text = normalize_html_text(title_el.get_text(strip=True)) if title_el else "BOXSCORE"
            elements.append(("section_header", header_text))
            elements.append(("table", str(boxscore_table)))
        else:
            print(
                f"    NOTE: '{os.path.basename(path)}' boxscore Tile has no populated table yet (still "
                f"loading, or the wait in Cell 4 didn't catch it this time) -- season-stat cells relying on "
                f"it will be incomplete for this opponent."
            )

    return pd.DataFrame(
        [
            {"element_index": idx, "element_type": element_type, "element_content": element_content}
            for idx, (element_type, element_content) in enumerate(elements)
        ],
        columns=["element_index", "element_type", "element_content"],
    )


scout_reports = {}  # opponent short name (from filename) -> parsed element table (element_index, element_type, element_content)
for path in scout_report_files:
    # Filenames are either "<date> UW-Whitewater @ <Opponent>_scout.<ext>" (away game) or
    # "<date> <Opponent> @ UW-Whitewater_scout.<ext>" (home game, e.g. the Aurora report) -- reuse the
    # same home/away-aware extraction defined above instead of always taking the text after "@", which
    # would mislabel every home game's report as "UW-Whitewater". Dispatch to the parser matching this
    # specific file's format -- older manually-uploaded reports are PDFs, auto-downloaded ones are HTML.
    opponent_short = opponent_from_scout_filename(path)
    parser_fn = parse_scout_html_elements if path.lower().endswith(".html") else parse_scout_pdf_elements
    elements_df = parser_fn(path)
    scout_reports[opponent_short] = elements_df
    n_tables = (elements_df["element_type"] == "table").sum()
    print(f"  Parsed '{opponent_short}' ({os.path.splitext(path)[1]}): {len(elements_df)} elements, {n_tables} tables")

# Flag any opponent that only has an MHTML report -- since MHTML is no longer used as a source at all, they're
# excluded from every downstream cell until a PDF version is added for them too.
mhtml_only_files = sorted(glob.glob(f"{volume_dir}/*_scout.mhtml"))
mhtml_opponents = {re.search(r"@ (.+)_scout\.mhtml$", os.path.basename(f)).group(1) for f in mhtml_only_files}
dropped_opponents = mhtml_opponents - set(scout_reports)
if dropped_opponents:
    print(f"\nWARNING: no PDF scout report exists yet for {sorted(dropped_opponents)} -- "
          f"MHTML is retired as a source, so these opponents are excluded from the analysis below.")

  [before_scout=yes] Ignoring 21 scouting report(s) dated on/after 2025-11-19: ['11_19_25 Aurora Spartans @ UW-Whitewater_scout.html', '11_25_25 Simpson Storm @ UW-Whitewater_scout.html', '12_10_25 UW-Whitewater @ Lawrence Vikings_scout.html', '12_13_25 Carroll (WI) Pioneers @ UW-Whitewater_scout.html', '12_19_25 UW-Whitewater @ Hope Flying Dutchmen_scout.html', '12_20_25 UW-Whitewater @ Alma Scots_scout.html', '12_2_25 Elmhurst Bluejays @ UW-Whitewater_scout.html', '12_30_25 UW-Whitewater @ Coe Kohawks_scout.html', '1_10_26 UW-Whitewater @ UW-River Falls Falcons_scout.html', '1_14_26 UW-Whitewater @ UW-La Crosse Eagles_scout.html', '1_17_26 UW-Stout Blue Devils @ UW-Whitewater_scout.html', '1_21_26 UW-Whitewater @ UW-Platteville Pioneers_scout.html', '1_24_26 UW-Eau Claire Blugolds @ UW-Whitewater_scout.html', '1_3_26 UW-Oshkosh Titans @ UW-Whitewater_scout.html', '1_7_26 UW-Stevens Point Pointers @ UW-Whitewater_scout.html', '2_11_26 UW-Platteville Pioneers @ UW-Whitewater_scout.html


### Extract each opponent's roster, player notes, and keys to defending

Parses the `"#<jersey> • <name> • <pos> • <height> • <weight> • <class>"` identity lines out of each scouting report's parsed elements, along with the "Player Notes" and "Keys to Defending" text that follows each player. Starter/bench role is read from the report's own "BENCH" section header when present, falling back to roster order (1st-5th = Starter) for reports that omit it.

In [24]:
# Player identity lines in the PDF's parsed text look like "#<jersey> \u2022 <name> \u2022 <pos> \u2022 <height> \u2022 <weight> \u2022
# <class>", each followed by a "Player Notes:" section_header + text and a "Keys to Defending" section_header +
# text. Starter/bench split (and roster size) varies by opponent, so it's derived from the "BENCH" section
# header's position when the PDF actually prints one. Some PDFs (e.g. Eureka) never print a "BENCH" header at
# all even though later players still have Player Notes/Keys to Defending filled in (Damuzha Moore, Jacob
# Gonzalez) -- for those, fall back to roster order: the 1st-5th players listed are Starters, 6th onward Bench.
PLAYER_LINE_RE = re.compile(r"^#(\d+)\s*\u2022\s*(.+?)\s*\u2022\s*([A-Z/]+)\s*\u2022\s*(\d+'\d+\")\s*\u2022\s*(.*?)\s*\u2022\s*([A-Z]+)$")
STARTERS_BY_ORDER_CUTOFF = 5

import json
from datetime import date

# The schedule's "date" strings (e.g. "Fri, Nov 14") carry NO year at all -- a men's basketball season spans
# two calendar years, so the year has to be inferred from the season boundary rather than read off the string.
# CONFIRMED CHANGE (requested): this used to hardcode a single "2025-26 season" assumption in its own
# SEASON_START_YEAR, separate from (and inconsistent in RETURN TYPE with) the other parse_schedule_date
# defined earlier in this notebook -- that one returns a datetime, this one returns a plain date, and
# every call site further down this notebook was written against THIS one's date-returning behavior
# (e.g. comparing against reference_date.date()). Kept as its own function rather than consolidated into
# the earlier one specifically to preserve that return type -- the fix here is the same as the other
# one's: accept an explicit season_start_year instead of a single hardcoded constant, so a caller can
# pass uww_season_start_year (computed once, right after UWW's own schedule is parsed, from the real
# season read off the page itself -- see extract_season_start_year) instead of silently assuming one
# season for every date resolved here.
def parse_schedule_date(date_str, season_start_year=None):
    """'Fri, Nov 14' -> date(2025, 11, 14) (or whatever year season_start_year resolves to)."""
    if pd.isna(date_str):
        return None
    parsed = datetime.strptime(str(date_str).strip(), "%a, %b %d")
    _syear = season_start_year if season_start_year is not None else _DEFAULT_SEASON_START_YEAR
    year = _syear if parsed.month >= 8 else _syear + 1
    return date(year, parsed.month, parsed.day)

# A few entries (e.g. Eureka's Damuzha Moore, Jacob Gonzalez) don't render "Player Notes"/"Keys to Defending" as
# separate labeled sections at all -- both labels AND both fields' text come through as ONE run-on text element,
# because the PDF's two side-by-side columns got merged with their lines interleaved by the text extractor.
# Splitting that reliably with regex isn't feasible (the interleaving order isn't consistent), so an LLM call
# separates the blob back into the two original fields based on their content -- notes describe playing style,
# keys are short defensive coaching instructions. (Portable: uses a plain OpenAI-compatible client instead of
# Databricks' ai_query() SQL function -- see USE_LLM / OPENAI_API_KEY / AI_MODEL / OPENAI_BASE_URL.)
def split_combined_notes_keys(blob):
    if not USE_LLM:
        return "", ""
    prompt = (
        "The following text is a run-on merge of two DIFFERENT scouting-report fields for one basketball "
        "player, with their lines interleaved: 'Player Notes' (describes how the player plays -- shooting, "
        "driving, position role, etc.) and 'Keys to Defending' (short defensive coaching instructions on how "
        "to guard him -- e.g. 'keep in front', 'box out', 'be a helper', 'anticipate drive', 'get off into "
        "gaps'). Split the text below back into its two original fields, preserving the original wording "
        "exactly (do not paraphrase, do not add words), and drop the literal labels 'Player Notes:' / "
        "'Keys to Defending' themselves from the output. Respond with ONLY a JSON object of the exact shape "
        '{"player_notes": "...", "keys_to_defending": "..."}, no other text.'
        f"\n\nTEXT: {blob}"
    )
    try:
        from openai import OpenAI

        client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL") or None)
        response = client.chat.completions.create(
            model=os.environ.get("AI_MODEL", "gpt-4o-mini"),
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
        )
        parsed = json.loads(response.choices[0].message.content)
    except Exception as llm_error:
        print(f"  LLM split failed for a combined notes/keys blob: {type(llm_error).__name__}: {llm_error} -- leaving both fields blank.")
        return "", ""
    return parsed.get("player_notes", "").strip(), parsed.get("keys_to_defending", "").strip()

# Reuse the schedule as the source of truth for each game's date (same fuzzy opponent-name match used later to
# resolve each scouted opponent's game number), rather than re-parsing it out of the PDF filename. Returns a
# real datetime.date (via parse_schedule_date) rather than the schedule's raw display string, since every
# "game_date" column built from this function (player_profiles, pbp_events, and everything grouped from pbp_events
# -- pbp_box_score, lineup_box_score) is meant for date arithmetic/sorting, not just display.
# CONFIRMED BUG (fixed here): this called parse_schedule_date() with no season_start_year override,
# silently falling back to _DEFAULT_SEASON_START_YEAR (2025) regardless of which season the MATCHED row
# actually belongs to. schedule already carries a real per-row "season" column (see
# extract_season_start_year() / build_team_schedule_from_html()), so the fix reads the season directly
# off the matched row itself, rather than assuming one season for every caller of this function --
# game_date_for() feeds player_profiles/pbp_events/pbp_box_score/lineup_box_score's own "game_date"
# columns (per this function's own docstring above), so a wrong year here would have propagated well
# beyond just a filename.
def game_date_for(opponent_short):
    matches = schedule[schedule["opponent"].str.contains(re.escape(opponent_short), case=False)]
    if matches.empty:
        return None
    _gdf_row = matches.iloc[0]
    _gdf_season_start_year = None
    if "season" in schedule.columns and pd.notna(_gdf_row.get("season")):
        try:
            _gdf_season_start_year = int(str(_gdf_row["season"]).split("-")[0])
        except (ValueError, TypeError):
            _gdf_season_start_year = None
    return parse_schedule_date(_gdf_row["date"], _gdf_season_start_year)

def extract_roster_from_pdf(elements_df, opponent):
    elements = elements_df.to_dict("records")
    n = len(elements)
    bench_idx = next(
        (e["element_index"] for e in elements if e["element_type"] == "section_header" and e["element_content"].strip() == "BENCH"),
        None,
    )
    game_date = game_date_for(opponent)

    rows = []
    player_seq = 0
    i = 0
    while i < n:
        el = elements[i]
        if el["element_type"] == "text":
            m = PLAYER_LINE_RE.match(el["element_content"].strip())
            if m:
                jersey, name, pos, height, weight, cls = m.groups()
                player_seq += 1
                notes, keys, combined_blob = [], [], None
                j = i + 1
                while j < n:
                    nel = elements[j]
                    if nel["element_type"] == "text" and PLAYER_LINE_RE.match(nel["element_content"].strip()):
                        break
                    if nel["element_type"] == "section_header" and nel["element_content"].strip() in ("STARTERS", "BENCH"):
                        break
                    if nel["element_type"] == "text":
                        content = nel["element_content"].strip()
                        prev = elements[j - 1]
                        if prev["element_type"] == "section_header" and prev["element_content"].startswith("Player Notes"):
                            notes.append(content)
                        elif prev["element_type"] == "section_header" and prev["element_content"].startswith("Keys to Defending"):
                            keys.append(content)
                        elif "player notes" in content.lower() and "keys to defending" in content.lower():
                            combined_blob = content
                    j += 1
                if not notes and not keys and combined_blob:
                    notes_text, keys_text = split_combined_notes_keys(combined_blob)
                else:
                    notes_text, keys_text = " ".join(notes), " ".join(keys)
                if bench_idx is not None:
                    role = "Starter" if el["element_index"] < bench_idx else "Bench"
                else:
                    role = "Starter" if player_seq <= STARTERS_BY_ORDER_CUTOFF else "Bench"
                rows.append({
                    "opponent": opponent,
                    "game_date": game_date,
                    "jersey_number": f"#{jersey}",
                    "name": name.strip(),
                    "position": pos,
                    "height": height,
                    "weight": (weight.strip() or None),
                    "class_year": cls,
                    "role": role,
                    "player_notes": notes_text,
                    "keys_to_defending": keys_text,
                    # Every row here came from parsing an actual scouting-report player entry (jersey/name/pos
                    # line + its Player Notes / Keys to Defending text) -- as opposed to a player later recovered
                    # ONLY from the season boxscore table (see player_profiles), who was never individually
                    # scouted. Carried forward through player_profiles so downstream cells that judge free-text
                    # scouting language (e.g. the LLM comparison) can restrict themselves to real scouted players.
                    "has_scouting_report": True,
                })
                i = j
                continue
        i += 1
    return pd.DataFrame(rows)

roster_cols = ["opponent", "game_date", "jersey_number", "name", "position", "height", "weight", "class_year", "role", "player_notes", "keys_to_defending", "has_scouting_report"]
all_rosters = pd.concat(
    [extract_roster_from_pdf(df, opponent) for opponent, df in scout_reports.items()],
    ignore_index=True,
) if scout_reports else pd.DataFrame(columns=roster_cols)
print(all_rosters)

                 opponent   game_date jersey_number               name  \
0   St. Thomas (TX) Celts  2025-11-14           #10      Angel Johnson   
1   St. Thomas (TX) Celts  2025-11-14            #1     Corey Thompson   
2   St. Thomas (TX) Celts  2025-11-14            #0     Nathan Kongolo   
3   St. Thomas (TX) Celts  2025-11-14           #12   Nicholas Buffalo   
4   St. Thomas (TX) Celts  2025-11-14           #25    Charles Gitonga   
5   St. Thomas (TX) Celts  2025-11-14           #33     Garret Rodgers   
6   St. Thomas (TX) Celts  2025-11-14            #5        Reyce Allen   
7   St. Thomas (TX) Celts  2025-11-14            #3       Brennan Webb   
8   St. Thomas (TX) Celts  2025-11-14            #2         Omar Gayle   
9       Eureka Red Devils  2025-11-15           #12     Jaxson Provost   
10      Eureka Red Devils  2025-11-15            #5        Micah Bruer   
11      Eureka Red Devils  2025-11-15            #4         Ben Carter   
12      Eureka Red Devils  2025-11-15 


### Extract UW-Whitewater's offensive and defensive game plan for each opponent

Pulls the "TEAM STRENGTHS" / "KEYS TO VICTORY" / defensive-scheme / offensive-scheme sections out of each scouting report's parsed elements, grouped under a consistent `section_group` label so they can be compared across opponents.

In [26]:
headers_in_order = [
    "TEAM STRENGTHS", "KEYS TO VICTORY",
    "Overall Defensive Scheme", "Attacking their man defense", "Ball Screen & DHO Defense",
    "Ball Screen Actions & Reads", "Speciality Defensive Notes", "Potential Adjustments",
    "Defending Their Action", "Overall OFFENSIVE SCHEME", "Ball Screen Actions & Personnel",
    "Ball Screen Coverage(s)", "Potential Adjustments", "ELOB & SLOB",
]
section_group = {
    "TEAM STRENGTHS": "Game Plan Overview", "KEYS TO VICTORY": "Game Plan Overview",
    "Overall Defensive Scheme": "Offensive Game Plan (vs. Opponent Defense)",
    "Attacking their man defense": "Offensive Game Plan (vs. Opponent Defense)",
    "Ball Screen & DHO Defense": "Offensive Game Plan (vs. Opponent Defense)",
    "Ball Screen Actions & Reads": "Offensive Game Plan (vs. Opponent Defense)",
    "Speciality Defensive Notes": "Offensive Game Plan (vs. Opponent Defense)",
    "Defending Their Action": "Defensive Game Plan (vs. Opponent Offense)",
    "Overall OFFENSIVE SCHEME": "Defensive Game Plan (vs. Opponent Offense)",
    "Ball Screen Actions & Personnel": "Defensive Game Plan (vs. Opponent Offense)",
    "Ball Screen Coverage(s)": "Defensive Game Plan (vs. Opponent Offense)",
    "ELOB & SLOB": "Defensive Game Plan (vs. Opponent Offense)",
}
# PDF elements already separate section_header from text cleanly (no banner/page-chrome noise mixed in, unlike
# flattened MHTML text) -- reconstruct the game plan by bucketing consecutive "text" elements under the most
# recent matching "section_header", stopping once the roster ("STARTERS") begins.
def extract_game_plan_from_pdf(elements_df, opponent):
    buckets, order = {}, []
    current_label = None
    adj_seen = 0
    for el in elements_df.to_dict("records"):
        if el["element_type"] == "section_header":
            content = el["element_content"].strip()
            if content == "STARTERS":
                break
            if content in headers_in_order:
                if content == "Potential Adjustments":
                    adj_seen += 1
                    label = f"Potential Adjustments ({'Defense' if adj_seen == 1 else 'Offense'})"
                    category = "Offensive Game Plan (vs. Opponent Defense)" if adj_seen == 1 else "Defensive Game Plan (vs. Opponent Offense)"
                else:
                    label, category = content, section_group[content]
                current_label = label
                buckets[label] = []
                order.append((category, label))
            else:
                current_label = None  # banner/divider header (e.g. "RIPON DEFENSE") we don't care about
        elif el["element_type"] == "text" and current_label is not None:
            txt = el["element_content"].strip()
            if txt:
                buckets[current_label].append(txt)
    return pd.DataFrame([
        {"opponent": opponent, "category": category, "topic": label, "notes": " | ".join(buckets[label])}
        for category, label in order
    ])

game_plan_cols = ["opponent", "category", "topic", "notes"]
all_game_plans = pd.concat(
    [extract_game_plan_from_pdf(df, opponent) for opponent, df in scout_reports.items()],
    ignore_index=True,
) if scout_reports else pd.DataFrame(columns=game_plan_cols)
pd.set_option("display.max_colwidth", 150)
print(all_game_plans)

                 opponent                                    category  \
0   St. Thomas (TX) Celts                          Game Plan Overview   
1   St. Thomas (TX) Celts                          Game Plan Overview   
2   St. Thomas (TX) Celts  Offensive Game Plan (vs. Opponent Defense)   
3   St. Thomas (TX) Celts  Offensive Game Plan (vs. Opponent Defense)   
4   St. Thomas (TX) Celts  Offensive Game Plan (vs. Opponent Defense)   
5   St. Thomas (TX) Celts  Offensive Game Plan (vs. Opponent Defense)   
6   St. Thomas (TX) Celts  Offensive Game Plan (vs. Opponent Defense)   
7   St. Thomas (TX) Celts  Offensive Game Plan (vs. Opponent Defense)   
8   St. Thomas (TX) Celts  Defensive Game Plan (vs. Opponent Offense)   
9   St. Thomas (TX) Celts  Defensive Game Plan (vs. Opponent Offense)   
10  St. Thomas (TX) Celts  Defensive Game Plan (vs. Opponent Offense)   
11  St. Thomas (TX) Celts  Defensive Game Plan (vs. Opponent Offense)   
12  St. Thomas (TX) Celts  Defensive Game Plan (vs.


### Team efficiency narrative (deprecated -- PDF exports don't carry it)

The FASTINTELLIGENCE panel (ORtg/DRtg/Pace + percentile narrative) is a ScoutBuilder UI-only widget that never appears in the printable/PDF report. Kept as an empty placeholder with the same schema as before so nothing downstream breaks, now that scouting reports are sourced exclusively from PDFs.

In [28]:
# The "FASTINTELLIGENCE" panel (ORtg/DRtg/Pace + national-percentile narrative) is ScoutBuilder's UI-only
# "Scout Assistant" widget -- it is NOT part of the printable report, so it never appears in the PDF export
# (confirmed: 0 of the Ripon PDF's 133 parsed elements mention ORtg/DRtg/FASTINTELLIGENCE). This metric is no
# longer available now that scouting reports are sourced exclusively from PDFs. Kept as an empty placeholder
# (same schema as before) so nothing downstream breaks.

# all_team_stats = pd.DataFrame(columns=["opponent", "ORtg", "DRtg", "Pace", "offense_narrative", "defense_narrative"])

print(
    "Team efficiency narrative (ORtg/DRtg/Pace/FASTINTELLIGENCE) is not captured in the PDF export -- it's a\n"
    "UI-only panel that isn't part of the printable ScoutBuilder report, and is no longer available now that\n"
    "scouting reports are sourced exclusively from PDFs."
)

Team efficiency narrative (ORtg/DRtg/Pace/FASTINTELLIGENCE) is not captured in the PDF export -- it's a
UI-only panel that isn't part of the printable ScoutBuilder report, and is no longer available now that
scouting reports are sourced exclusively from PDFs.



### Cross-reference each scouted opponent's report against the actual result

Since the FASTINTELLIGENCE narrative is gone (see previous cell), this instead reads each opponent's season-average points scored/allowed straight from their PDF's own boxscore table ("Team Total" and "Opponent" rows) to sanity-check the scouting report against what actually happened.

In [30]:
# The FASTINTELLIGENCE narrative used previously is gone (see prior cell), so this now cross-references each
# opponent's season-average PPG scored/allowed straight from their PDF's "...BOXSCORE" table's "Team Total"
# and "Opponent" rows -- a different underlying data source than the old narrative, so exact figures may differ
# from what MHTML-based reports previously showed.
def _dedupe_boxscore_columns(df):
    """read_boxscore_table below builds its DataFrame from a plain Python `headers` list via
    `pd.DataFrame(records, columns=headers)` -- unlike pd.read_html elsewhere in this notebook, this gets NO
    automatic duplicate-column mangling from pandas at all, so two header cells that happen to render the
    same text (confirmed to happen on the live-rendered ScoutBuilder boxscore widget -- see the app-side fix
    this mirrors) silently produce a DataFrame with two identically-named columns. Any later df[col] or
    row[col] access on that name then returns a Series instead of a scalar without raising, which is exactly
    the "truth value of a Series is ambiguous" crash this was tracked down from. Dedupe once here, right at
    construction, rather than downstream wherever it happens to first get accessed."""
    if df.columns.duplicated().any():
        dupes = df.columns[df.columns.duplicated()].unique().tolist()
        print(f"WARNING: boxscore table has duplicate column name(s) {dupes} -- keeping the first occurrence "
              f"of each and dropping the rest.")
        df = df.loc[:, ~df.columns.duplicated()]
    return df


def read_boxscore_table(html_str):
    """Parse a FastScout PDF boxscore <table> into a DataFrame. On some (not all) opponents' PDFs, a player's
    full name is split across two separate <td> cells (first name, last name) while the header row only
    allocates a single <th>PLAYER</th> slot for it -- silently shifting every later column (FG%, 3P%, etc.) one
    position to the right of its real header when read naively via pd.read_html(). Detect that per-row and
    merge the split name cells back into one before assigning headers, so every stat column lines up correctly
    regardless of which layout a given PDF used."""
    table_soup = BeautifulSoup(html_str, "html.parser")
    header_row = table_soup.find("tr")
    header_cells = header_row.find_all("th")

    # The LIVE ScoutBuilder HTML boxscore table (Cell 8's HTML path, as opposed to a pdfplumber-extracted PDF
    # table) wraps every REAL header/data cell in its own "cell-content" div, and ALSO carries decorative
    # leading cells (a blank "table-gutter" cell + a sort-handle cell) that never get one -- confirmed against
    # a real live-downloaded report: naive positional header/cell alignment (the PDF-oriented path below)
    # silently misaligns every column by those decorative cells, since headers ends up 2 shorter than the
    # actual <td> count per row for a totally unrelated reason than the split-name case it was built for.
    # Detect that shape via "cell-content" and filter on it directly instead of position.
    if any(th.find(class_="cell-content") is not None for th in header_cells):
        headers = []
        for th in header_cells:
            content_div = th.find(class_="cell-content")
            if content_div is not None:
                headers.append(normalize_html_text(content_div.get_text(" ", strip=True)))
        body = table_soup.find("tbody")
        body_rows = body.find_all("tr") if body else table_soup.find_all("tr")[1:]
        records = []
        for tr in body_rows:
            cells = []
            for td in tr.find_all("td"):
                content_div = td.find(class_="cell-content")
                cells.append(normalize_html_text(content_div.get_text(" ", strip=True)) if content_div is not None else None)
            cells = [c for c in cells if c is not None]
            records.append(cells[: len(headers)])
        _boxscore_df = pd.DataFrame(records, columns=headers)
        return _dedupe_boxscore_columns(_boxscore_df)

    # --- PDF-oriented path (pdfplumber's plain-text cells have no "cell-content" wrapper at all) ---
    header_cells_text = [th.get_text(strip=True) for th in header_cells]
    # Drop trailing decorative header cell(s) with no real stat name -- this is sometimes a blank "" and
    # sometimes a private-use-area icon glyph (e.g. "\uf107", presumably a sort-arrow icon that pdfplumber
    # extracted as text) depending on the PDF, but either way it never has any alphanumeric content.
    headers = header_cells_text[:]
    while headers and not any(c.isalnum() for c in headers[-1]):
        headers = headers[:-1]
    name_idx = headers.index("PLAYER")
    body = table_soup.find("tbody")
    body_rows = body.find_all("tr") if body else table_soup.find_all("tr")[1:]
    records = []
    for tr in body_rows:
        cells = [td.get_text(strip=True) for td in tr.find_all("td")]
        if len(cells) == len(headers) + 1:
            cells = cells[:name_idx] + [" ".join(c for c in cells[name_idx:name_idx + 2] if c)] + cells[name_idx + 2:]
        records.append(cells[: len(headers)])
    _boxscore_df = pd.DataFrame(records, columns=headers)
    return _dedupe_boxscore_columns(_boxscore_df)


def extract_team_totals_from_pdf(elements_df, opponent):
    rows = elements_df.to_dict("records")
    box_idx = next(
        (i for i, r in enumerate(rows) if r["element_type"] == "section_header" and "BOXSCORE" in r["element_content"].upper()),
        None,
    )
    if box_idx is None or rows[box_idx + 1]["element_type"] != "table":
        return None
    stats_df = read_boxscore_table(rows[box_idx + 1]["element_content"]).rename(columns={"PLAYER": "name"})
    # See the analogous fix in the player-tagging cell above -- pdfplumber can truncate "Team Total" down to just
    # "Team", so also fall back to the jersey-number column ("#" == "-" for the aggregate row) to find it.
    team_row = stats_df[(stats_df["name"] == "Team Total") | (stats_df.get("#") == "-")]
    opp_row = stats_df[stats_df["name"] == "Opponent"]
    if team_row.empty:
        print(f"  Skipping '{opponent}': no 'Team Total' row in its boxscore table.")
        return None
    if opp_row.empty:
        print(f"  Note: '{opponent}' boxscore has no 'Opponent' row (points allowed) -- reporting team_ppg only.")
    return {
        "opponent": opponent,
        "team_ppg": float(team_row["PTS"].iloc[0]),
        "opp_ppg_allowed": float(opp_row["PTS"].iloc[0]) if not opp_row.empty else None,
    }

team_totals = pd.DataFrame([
    r for r in (extract_team_totals_from_pdf(df, opponent) for opponent, df in scout_reports.items()) if r
])

# CONFIRMED CHANGE (requested): stop using ANY team or player statistics sourced from the scouting
# report, parser-wide. Root cause of a real, reported leak: these PDF scout reports are explicitly
# labeled "Last Season" and carry the PRIOR YEAR's numbers -- Ripon's real per-game PPG/points-allowed
# had nothing to do with any game either team has played this year, and no reference_date fix can make a
# stale, undated PDF summary trustworthy. team_ppg/opp_ppg_allowed are blanked immediately after parsing
# (team_totals is kept for the opponent-identity/schedule-matching logic below, which doesn't need real
# stats). The ONLY legitimate source for these numbers from here on is the PBP-derived override further
# down this notebook (see "Override the upcoming opponent's team_totals"), which uses real, current-
# season game data instead.
if not team_totals.empty:
    team_totals["team_ppg"] = None
    team_totals["opp_ppg_allowed"] = None

# "Keys to Victory" notes use basketball terminology rather than stat names directly ("Ball Security" ==
# turnovers, "Own the Paint"/"Bully them on the glass" == rebounding, etc). Map that terminology to the UWW
# season stat columns it corresponds to, then surface UWW's season-long "Team Total" (their own average) vs.
# "Opponent" (what teams average against UWW) rows from `stats` for whichever columns a given note touches on --
# there's no per-game UWW box score in this pipeline (the PDF boxscore only covers the OPPONENT's roster), so the
# season averages are the best available context for whether that emphasis is one UWW has actually executed on.
KEYS_TO_VICTORY_STAT_MAP = {
    # --- Ball Security / Turnovers (TO) ---
    "ball security": ["TO"], "turnover": ["TO"], "protect the ball": ["TO"], "take care of the ball": ["TO"],
    "limit turnovers": ["TO"], "careless": ["TO"],
    # --- Rebounding (REB, ORB, DRB) ---
    "own the paint": ["REB", "ORB", "DRB"], "bully": ["REB", "ORB", "DRB"], "glass": ["REB", "ORB", "DRB"],
    "rebound": ["REB", "ORB", "DRB"], "board": ["REB", "ORB", "DRB"], "second chance": ["ORB"],
    "crash": ["REB", "ORB", "DRB"],
    # --- Three-Point Shooting (3PM-A, 3P%) ---
    "three": ["3PM-A", "3P%"], "3 pt": ["3PM-A", "3P%"], "3pt": ["3PM-A", "3P%"],
    "perimeter shooting": ["3PM-A", "3P%"], "spacing": ["3PM-A", "3P%"],
    "shooting ability": ["3PM-A", "3P%"], "shooting team": ["3PM-A", "3P%"],
    "sniper": ["3PM-A", "3P%"], "will shoot": ["3PM-A", "3P%"],
    # --- Free Throws (FTM-A, FT%) ---
    "free throw": ["FTM-A", "FT%"], "ft line": ["FTM-A", "FT%"], "getting to ft": ["FTM-A", "FT%"],
    # --- Fouls / Discipline (PF) ---
    "foul": ["PF"], "wall up": ["PF"], "drawing fouls": ["PF"],
    # --- Ball Movement / Assists (AST) ---
    "assist": ["AST"], "ball movement": ["AST"], "share the ball": ["AST"],
    "playmaking": ["AST"], "playmaker": ["AST"], "create": ["AST"],
    # --- Perimeter Defense / Ball Pressure (STL) ---
    "steal": ["STL"], "press capable": ["STL", "TO"], "full court press": ["STL", "TO"],
    "force turnovers": ["STL"], "force to's": ["STL"],
    "guard your yard": ["STL"], "keep the ball in front": ["STL"], "guard 1 on 1": ["STL"],
    "early gap": ["STL"], "help side": ["STL"], "active hands": ["STL"],
    "physical & aggressive on ball": ["STL"], "on ball defensively": ["STL"],
    "pressure": ["STL"],
    # --- Paint Protection / Blocks (BLK) ---
    "block": ["BLK"], "protect the rim": ["BLK"], "paint protection": ["BLK"],
    # --- Scoring Inside (FG2M, FG2A, FG2% — derived as FGM-FG3M, FGA-FG3A) ---
    "dominate the paint": ["FG2M", "FG2A", "FG2%"], "attack the paint": ["FG2M", "FG2A", "FG2%"],
    "live in the paint": ["FG2M", "FG2A", "FG2%"], "attack the basket": ["FG2M", "FG2A", "FG2%"],
    "scoring at the rim": ["FG2M", "FG2A", "FG2%"], "get to rim": ["FG2M", "FG2A", "FG2%"],
    "attack the rim": ["FG2M", "FG2A", "FG2%"], "get to the rim": ["FG2M", "FG2A", "FG2%"],
    # --- Field Goal Efficiency (FGM-A, FG%) — overall ---
    "limit their scoring": ["FGM-A", "FG%"],
}

# --- Side attribution: Is this phrase about what UWW does (offense/proactive) or containing the opponent? ---
PHRASE_SIDE = {
    "ball security": "UWW", "turnover": "UWW", "protect the ball": "UWW",
    "take care of the ball": "UWW", "limit turnovers": "UWW", "careless": "UWW",
    "own the paint": "UWW", "bully": "UWW", "glass": "UWW",
    "rebound": "UWW", "board": "UWW", "second chance": "UWW", "crash": "UWW",
    "box out": "OPP", "keep off glass": "OPP",
    "three": "UWW", "3 pt": "OPP", "3pt": "OPP", "perimeter shooting": "UWW", "spacing": "UWW",
    "shooting ability": "OPP", "shooting team": "OPP", "sniper": "OPP", "will shoot": "OPP",
    "close out": "OPP", "closeout": "OPP", "run off the line": "OPP",
    "free throw": "UWW", "ft line": "UWW", "getting to ft": "UWW",
    "don't foul": "OPP", "keep them off": "OPP",
    "foul": "OPP", "wall up": "OPP", "drawing fouls": "OPP",
    "assist": "UWW", "ball movement": "UWW", "share the ball": "UWW",
    "playmaking": "UWW", "playmaker": "UWW", "create": "UWW",
    "steal": "OPP", "press capable": "OPP", "full court press": "OPP",
    "force turnovers": "OPP", "force to's": "OPP",
    "guard your yard": "OPP", "keep the ball in front": "OPP", "guard 1 on 1": "OPP",
    "early gap": "OPP", "help side": "OPP", "active hands": "OPP",
    "physical & aggressive on ball": "OPP", "on ball defensively": "OPP",
    "pressure": "OPP",
    "block": "OPP", "protect the rim": "OPP", "paint protection": "OPP",
    "attack the paint": "UWW", "live in the paint": "UWW",
    "dominate the paint": "UWW", "attack the basket": "UWW",
    "scoring at the rim": "UWW", "get to rim": "UWW",
    "attack the rim": "UWW", "get to the rim": "UWW",
    "limit their scoring": "OPP",
    "take away": "OPP", "funnel": "OPP", "deny": "OPP", "contain": "OPP",
    "limit": "OPP", "contest": "OPP", "make them": "OPP", "load up": "OPP",
    "transition defense": "OPP", "fight over": "OPP", "switch": "OPP",
    "trap": "OPP", "double team": "OPP", "coverage": "OPP",
    "don't help off": "OPP", "take away personnel": "OPP",
    "run the floor": "UWW", "push tempo": "UWW", "fast break": "UWW",
    "score in transition": "UWW", "finish": "UWW", "execute": "UWW",
    "dominate": "UWW", "impose": "UWW", "push the pace": "UWW",
}
STAT_LABELS = {
    "TO": "Turnovers/gm", "REB": "Rebounds/gm", "ORB": "Off. rebounds/gm", "DRB": "Def. rebounds/gm",
    "AST": "Assists/gm", "STL": "Steals/gm", "BLK": "Blocks/gm", "3PM-A": "3PM-A/gm", "3P%": "3P%",
    "FGM-A": "FGM-A/gm", "FG%": "FG%", "FTM-A": "FTM-A/gm", "FT%": "FT%", "PF": "Fouls/gm",
}
# uww_team_totals = stats[stats["PLAYER"] == "Team Total"].iloc[0] if (stats["PLAYER"] == "Team Total").any() else None
uww_team_totals = None
# uww_opp_totals = stats[stats["PLAYER"] == "Opponent"].iloc[0] if (stats["PLAYER"] == "Opponent").any() else None
uww_opp_totals = None

def stat_cols_for_note(note_text):
    text = str(note_text).lower()
    cols = []
    for phrase, stat_cols in KEYS_TO_VICTORY_STAT_MAP.items():
        if phrase in text:
            for c in stat_cols:
                if c not in cols:
                    cols.append(c)
    return cols

def print_keys_to_victory_stats(note_text):
    cols = stat_cols_for_note(note_text)
    if not cols or uww_team_totals is None:
        return
    print("    Relevant UWW season averages (Team Total = UWW's own avg, Opponent = what teams average against UWW):")
    for c in cols:
        print(f"      {STAT_LABELS.get(c, c)}: UWW {uww_team_totals[c]}  |  Opponent avg {uww_opp_totals[c]}")

# CONFIRMED BUG (fixed here): my first pass at removing the PDF-stat comparison deleted this ENTIRE
# section, including scouted_game_comparison itself -- which broke a separate, LEGITIMATE downstream cell
# (the win/loss-splits-by-"Keys to Victory"-category cell) that only ever reads its `opponent` and
# `outcome` columns, both of which come from the real schedule, not the PDF. That cell crashed with
# "NameError: name 'scouted_game_comparison' is not defined" the very next run. Restored below: every
# column sourced from the REAL schedule (opponent, date, location, outcome, actual scores, point_margin)
# stays; only the PDF-derived season-average PPG comparison columns are permanently None (they inherit
# that from team_totals's blanking above) and guarded with pd.notna() so they read as "N/A" in the
# printout instead of crashing on None -- consistent with the opp_ppg_allowed side, which already had to
# handle a sometimes-missing value the same way even before this change.
comparison_rows = []
for _, tt in team_totals.iterrows():
    opponent = tt["opponent"]
    matches = schedule[schedule["opponent"].str.contains(re.escape(opponent), case=False)]
    if matches.empty:
        print(f"No schedule match found for scouted opponent '{opponent}' -- skipping.")
        continue
    game_result = matches.iloc[0]

    comparison_rows.append({
        "opponent": opponent,
        "date": game_result["date"],
        "location": game_result["location"],
        "outcome": game_result["outcome"],
        "opp_season_avg_ppg": tt["team_ppg"],
        "opp_actual_ppg": game_result["opponent_score"],
        "opp_ppg_vs_average": (
            round(game_result["opponent_score"] - tt["team_ppg"], 1) if pd.notna(tt["team_ppg"]) else None
        ),
        "uww_points": game_result["team_score"],
        "opp_season_avg_ppg_allowed": tt["opp_ppg_allowed"],
        "uww_ppg_vs_opp_avg_allowed": (
            round(game_result["team_score"] - tt["opp_ppg_allowed"], 1) if pd.notna(tt["opp_ppg_allowed"]) else None
        ),
        "point_margin": game_result["point_margin"],
    })

scouted_game_comparison = pd.DataFrame(comparison_rows)
if scouted_game_comparison.empty:
    print("No opponents with a schedule match -- nothing to cross-reference.")
else:
    print(scouted_game_comparison)
    print("\n(opp_season_avg_ppg / opp_ppg_vs_average / opp_season_avg_ppg_allowed / uww_ppg_vs_opp_avg_allowed "
          "are always blank now -- team/player statistics from the scouting report are no longer used; see "
          "the blanking earlier in this cell.)")
    for _, row in scouted_game_comparison.iterrows():
        keys_to_victory = all_game_plans.loc[
            (all_game_plans["opponent"] == row["opponent"]) & (all_game_plans["topic"] == "KEYS TO VICTORY"), "notes"
        ]
        print(f"\n{row['opponent']} ({row['date']}, {row['location']}): UWW {row['uww_points']} - "
              f"{row['opponent']} {row['opp_actual_ppg']} ({row['outcome']}, margin {row['point_margin']:+.0f})")
        if not keys_to_victory.empty:
            print("  Pre-game keys to victory:", keys_to_victory.iloc[0])
            print_keys_to_victory_stats(keys_to_victory.iloc[0])
        if pd.notna(row["opp_ppg_vs_average"]):
            print(f"  Opponent scored {row['opp_ppg_vs_average']:+.1f} vs their season-average PPG (from the PDF box score).")
        else:
            print("  No season-average PPG available for comparison (scouting-report statistics are not used).")
        if pd.notna(row["uww_ppg_vs_opp_avg_allowed"]):
            print(f"  UWW scored {row['uww_ppg_vs_opp_avg_allowed']:+.1f} vs the opponent's season-average points allowed.")
        else:
            print("  No season-average points-allowed figure available for comparison (scouting-report statistics are not used).")

                opponent         date       location outcome  \
0  St. Thomas (TX) Celts  Fri, Nov 14  Neutral Court       L   
1      Eureka Red Devils  Sat, Nov 15  Neutral Court       W   
2        Ripon Red Hawks   Fri, Nov 7           Away       W   

  opp_season_avg_ppg  opp_actual_ppg opp_ppg_vs_average  uww_points  \
0               None            81.0               None        73.0   
1               None            73.0               None       116.0   
2               None            58.0               None        76.0   

  opp_season_avg_ppg_allowed uww_ppg_vs_opp_avg_allowed  point_margin  
0                       None                       None          -8.0  
1                       None                       None          43.0  
2                       None                       None          18.0  

(opp_season_avg_ppg / opp_ppg_vs_average / opp_season_avg_ppg_allowed / uww_ppg_vs_opp_avg_allowed are always blank now -- team/player statistics from the scouting repor


### Win/loss splits by Keys-to-Victory category, with side attribution

Tags each matched "Keys to Victory" phrase with a side -- UWW (what Whitewater does proactively: attack, score, rebound, share) or OPP (what Whitewater does to contain the opponent: guard, force turnovers, limit, pressure) -- then breaks down win/loss outcomes by category and side.

In [32]:
# Win/loss splits by "Keys to Victory" category WITH SIDE ATTRIBUTION (UWW vs OPP).
# Each matched phrase now carries a "side" from PHRASE_SIDE: UWW = what Whitewater proactively does (offense/
# hustle), OPP = what Whitewater does to CONTAIN the opponent (defense/discipline). This answers the coaching
# question: "When we emphasize attacking the rim ourselves (UWW) vs limiting THEIR rim attacks (OPP), which
# approach correlates with winning?"
# The stat-category grouping is unchanged (rebounding columns still roll up to "Rebounding"), but now each
# game-category row also carries its side, so splits can be cut both ways.
STAT_COL_CATEGORY = {
    "TO": "Ball Security / Turnovers", "STL": "Perimeter Defense / Ball Pressure",
    "REB": "Rebounding", "ORB": "Rebounding", "DRB": "Rebounding",
    "3PM-A": "Three-Point Shooting", "3P%": "Three-Point Shooting",
    "FTM-A": "Free Throws", "FT%": "Free Throws",
    "PF": "Fouls / Discipline",
    "AST": "Ball Movement / Assists",
    "BLK": "Paint Protection / Blocks",
    "FG2M": "Scoring Inside", "FG2A": "Scoring Inside", "FG2%": "Scoring Inside",
    "FGM-A": "Field Goal Efficiency", "FG%": "Field Goal Efficiency",
}

SIDE_DISPLAY_LABELS = {
    ("Ball Security / Turnovers", "UWW"): "UWW: Protect the Ball",
    ("Ball Security / Turnovers", "OPP"): "OPP: Force Turnovers",
    ("Rebounding", "UWW"): "UWW: Crash the Boards",
    ("Rebounding", "OPP"): "OPP: Limit Their Rebounding",
    ("Three-Point Shooting", "UWW"): "UWW: Hit Our Threes",
    ("Three-Point Shooting", "OPP"): "OPP: Contest Their Shooting",
    ("Free Throws", "UWW"): "UWW: Get to the FT Line",
    ("Free Throws", "OPP"): "OPP: Keep Them Off the Line",
    ("Fouls / Discipline", "UWW"): "UWW: Stay Disciplined",
    ("Fouls / Discipline", "OPP"): "OPP: They Draw Fouls",
    ("Ball Movement / Assists", "UWW"): "UWW: Share the Ball",
    ("Ball Movement / Assists", "OPP"): "OPP: Disrupt Their Ball Movement",
    ("Perimeter Defense / Ball Pressure", "UWW"): "UWW: Create Pressure",
    ("Perimeter Defense / Ball Pressure", "OPP"): "OPP: On-Ball Defense",
    ("Paint Protection / Blocks", "UWW"): "UWW: Protect Our Rim",
    ("Paint Protection / Blocks", "OPP"): "OPP: Limit Their Interior",
    ("Scoring Inside", "UWW"): "UWW: Attack the Paint",
    ("Scoring Inside", "OPP"): "OPP: Limit Their Inside Scoring",
    ("Field Goal Efficiency", "UWW"): "UWW: Efficient Shooting",
    ("Field Goal Efficiency", "OPP"): "OPP: Limit Their FG Efficiency",
}

def categories_for_note(note_text):
    """Return list of (category, side) tuples matched from note text."""
    text = str(note_text).lower()
    seen = set()
    results = []
    for phrase, stat_cols in KEYS_TO_VICTORY_STAT_MAP.items():
        if phrase in text:
            side = PHRASE_SIDE.get(phrase, "UWW")  # default to UWW if not explicitly listed
            for c in stat_cols:
                cat = STAT_COL_CATEGORY.get(c)
                if cat and (cat, side) not in seen:
                    seen.add((cat, side))
                    results.append((cat, side))
    return sorted(results)

def stat_cols_for_note(note_text):
    """Legacy helper: just the stat columns (no side), used by downstream How We Stack Up."""
    text = str(note_text).lower()
    cols = []
    for phrase, stat_cols in KEYS_TO_VICTORY_STAT_MAP.items():
        if phrase in text:
            for c in stat_cols:
                if c not in cols:
                    cols.append(c)
    return cols

game_category_rows = []
for _, row in scouted_game_comparison.iterrows():
    keys_to_victory = all_game_plans.loc[
        (all_game_plans["opponent"] == row["opponent"]) & (all_game_plans["topic"] == "KEYS TO VICTORY"), "notes"
    ]
    if keys_to_victory.empty:
        continue
    cat_sides = categories_for_note(keys_to_victory.iloc[0])
    if not cat_sides:
        game_category_rows.append({"opponent": row["opponent"], "outcome": row["outcome"], "category": "(no matched category)", "side": ""})
    for cat, side in cat_sides:
        game_category_rows.append({"opponent": row["opponent"], "outcome": row["outcome"], "category": cat, "side": side})

game_categories = pd.DataFrame(game_category_rows)
print("Per-game 'Keys to Victory' categories detected WITH SIDE ATTRIBUTION:")
print("  UWW = what Whitewater does proactively (attack, score, rebound, share)")
print("  OPP = what Whitewater does to contain the opponent (guard, force TOs, limit, pressure)")
print(game_categories)

if not game_categories.empty:
    # Granular splits: by category + side
    game_categories["display_label"] = game_categories.apply(
        lambda r: SIDE_DISPLAY_LABELS.get((r["category"], r["side"]), f"{r['side']}: {r['category']}"), axis=1
    )
    splits = (
        game_categories.groupby(["category", "side", "display_label"])["outcome"]
        .agg(games="count", wins=lambda s: (s == "W").sum(), losses=lambda s: (s == "L").sum())
        .reset_index()
        .sort_values(["category", "side"], ascending=[True, True])
    )
    splits["win_pct"] = (splits["wins"] / splits["games"]).round(3)
    print("\nWin/loss splits by category + side (UWW vs OPP emphasis):")
    print(splits[["display_label", "side", "category", "games", "wins", "losses", "win_pct"]])
else:
    print("No 'Keys to Victory' notes with a matched category yet.")

Per-game 'Keys to Victory' categories detected WITH SIDE ATTRIBUTION:
  UWW = what Whitewater does proactively (attack, score, rebound, share)
  OPP = what Whitewater does to contain the opponent (guard, force TOs, limit, pressure)
                opponent outcome                           category side
0  St. Thomas (TX) Celts       L          Ball Security / Turnovers  UWW
1  St. Thomas (TX) Celts       L              Field Goal Efficiency  OPP
2  St. Thomas (TX) Celts       L                         Rebounding  UWW
3      Eureka Red Devils       W  Perimeter Defense / Ball Pressure  OPP
4      Eureka Red Devils       W                         Rebounding  UWW
5        Ripon Red Hawks       W                         Rebounding  UWW
6        Ripon Red Hawks       W                     Scoring Inside  UWW

Win/loss splits by category + side (UWW vs OPP emphasis):
                    display_label side                           category  \
0           UWW: Protect the Ball  UWW          


### Keys-to-Victory category reference

Documents WHY each category exists: which Keys-to-Victory phrases trigger it, which stat column(s) it's graded on, and cases where a phrase is deliberately left unmapped because no stat in a season box score can measure it. Update this table any time a new KTV phrase pattern shows up.

In [34]:
# --- Keys-to-Victory category reference -----------------------------------------------------------------------
# Documents WHY each category exists: which Keys-to-Victory phrases trigger it, which stat column(s) it's graded
# on, and the reasoning -- including cases where a phrase is deliberately left unmapped because no stat in a
# season box score can measure it. Update this table any time KEYS_TO_VICTORY_STAT_MAP / STAT_COL_CATEGORY change.
# Last updated: expanded keywords after reviewing all 6 opponents' KTV + Team Strengths notes for unmapped phrases.
KTV_CATEGORY_REFERENCE = [
    {
        "category": "Ball Security / Turnovers", "stat_cols": "TO",
        "example_phrases": "ball security, turnover, protect/take care of the ball, limit turnovers, careless",
        "why_chosen": "Directly named -- \"ball security\"/\"turnovers\" have a 1:1 stat column (TO), no proxy needed.",
    },
    {
        "category": "Rebounding", "stat_cols": "REB, ORB, DRB",
        "example_phrases": "own the paint, bully/dominate the glass, rebound, board, second chance, crash (the glass)",
        "why_chosen": "\"Glass\"/\"board\"/\"crash\"/\"own the paint\" possession language in this scouting vocabulary is "
                      "always about winning the rebounding battle, not shot-making -- REB/ORB/DRB are the direct stats.",
    },
    {
        "category": "Three-Point Shooting", "stat_cols": "3PM-A, 3P%",
        "example_phrases": "three, 3 pt, 3pt, perimeter shooting, spacing, shooting ability, shooting team, "
                           "will shoot, sniper",
        "why_chosen": "Direct stat match for perimeter shot volume/efficiency. Expanded with \"3 pt\"/\"shooting "
                      "ability\"/\"shooting team\" from Elmhurst notes (\"High level 3 pt shooting team\") and "
                      "\"will shoot\" from Eureka (\"All 5 Will Shoot\").",
    },
    {
        "category": "Free Throws", "stat_cols": "FTM-A, FT%",
        "example_phrases": "free throw, ft line, getting to ft",
        "why_chosen": "Direct stat match. Added \"ft line\" / \"getting to ft\" from Aurora (\"getting to FT line\").",
    },
    {
        "category": "Fouls / Discipline", "stat_cols": "PF",
        "example_phrases": "foul, wall up, drawing fouls",
        "why_chosen": "Direct stat match for foul-discipline emphasis. Added \"wall up\" (Aurora) and \"drawing "
                      "fouls\" (Aurora \"Great at drawing fouls\").",
    },
    {
        "category": "Ball Movement / Assists", "stat_cols": "AST",
        "example_phrases": "assist, ball movement, share the ball, playmaking, playmaker, create",
        "why_chosen": "Direct stat match for offensive ball-sharing emphasis. Added \"playmaking\"/\"playmaker\"/ "
                      "\"create\" from Simpson/Ripon notes (\"2 playmaking guards\", \"Multiple guys that can create\").",
    },
    {
        "category": "Paint Protection / Blocks", "stat_cols": "BLK",
        "example_phrases": "block, protect the rim, paint protection",
        "why_chosen": "Direct stat match for interior shot-blocking emphasis.",
    },
    {
        "category": "Perimeter Defense / Ball Pressure", "stat_cols": "STL",
        "example_phrases": "steal, press capable, full court press, force turnovers, force to's, guard your yard, "
                           "keep the ball in front, guard 1 on 1, early gap, help side, active hands, pressure, "
                           "physical & aggressive on ball, on ball defensively",
        "why_chosen": "Renamed from \"Ball Pressure / Steals\" once Aurora/Eureka/Elmhurst introduced CONTAINMENT "
                      "phrasing (\"guard your yard\", \"keep the ball in front\", \"guard 1 on 1\") alongside the "
                      "original steal-gambling phrasing (\"press\", \"force turnovers\"). Both are point-of-attack, "
                      "on-ball defensive emphases. STL is the only stat in the season box score that reflects "
                      "defensive activity at all. Added \"pressure\" (data-driven keys), \"physical & aggressive on "
                      "ball\" / \"on ball defensively\" (Elmhurst \"Physical & Aggressive on ball defensively\"), "
                      "and \"force to's\" (Ripon). NOTE: \"press\" changed to \"press capable\"/\"full court press\" to "
                      "avoid false-positive substring matches (e.g. \"pressure\" contains \"press\").",
    },
    {
        "category": "Scoring Inside", "stat_cols": "FG2M, FG2A, FG2%",
        "example_phrases": "dominate the paint, attack the paint, live in the paint, attack the basket, "
                           "scoring at the rim, get to rim, attack the rim, get to the rim",
        "why_chosen": "Interior-scoring emphasis -- uses derived 2PT FG stats (FGM-FG3M, FGA-FG3A) as a "
                      "direct measure of inside scoring rather than overall FG efficiency. Does NOT include "
                      "free throws. Added from Aurora/Simpson/Ripon/Elmhurst/St. Thomas phrasing.",
    },
    {
        "category": "Field Goal Efficiency", "stat_cols": "FGM-A, FG%",
        "example_phrases": "limit their scoring",
        "why_chosen": "Overall shooting efficiency emphasis -- used when the scouting note is about limiting "
                      "opponent scoring broadly rather than specifically inside or from 3PT range.",
    },
    {
        "category": "(intentionally unmapped)", "stat_cols": "-",
        "example_phrases": "communication screening action / communicate screens & actions (Ripon, Elmhurst), "
                           "take away personnel tendencies (Simpson), heavy ball screen usage (Ripon), "
                           "will be their 4th game (Eureka)",
        "why_chosen": "No corresponding stat exists in a season box score for screen-navigation communication, "
                      "opponent-personnel-specific keys, scheme-specific ball screen usage, or schedule context. "
                      "These stay qualitative-only in the game-plan notes and don't produce a category row.",
    },
]

pd.set_option("display.max_colwidth", 300)
print(pd.DataFrame(KTV_CATEGORY_REFERENCE))

                             category         stat_cols  \
0           Ball Security / Turnovers                TO   
1                          Rebounding     REB, ORB, DRB   
2                Three-Point Shooting        3PM-A, 3P%   
3                         Free Throws        FTM-A, FT%   
4                  Fouls / Discipline                PF   
5             Ball Movement / Assists               AST   
6           Paint Protection / Blocks               BLK   
7   Perimeter Defense / Ball Pressure               STL   
8                      Scoring Inside  FG2M, FG2A, FG2%   
9               Field Goal Efficiency        FGM-A, FG%   
10           (intentionally unmapped)                 -   

                                                                                                                                                                                                                 example_phrases  \
0                                                              


### Build per-player scouting profiles: position, height, and playing-style tags

Builds a comparable `player_profiles` entry for every scouted player -- normalized position group, height in inches, and a set of playing-style tags mined from their free-text scouting notes (player notes + keys to defending). Box-score-only players with no scouting writeup are added with a default Bench role so they aren't silently dropped from later lineup analysis.

In [36]:
# Build a comparable "player profile" for every scouted player: normalized position group, height in inches,
# and a set of playing-style tags mined from their free-text scouting notes (player_notes + keys_to_defending).
def parse_height_inches(h):
    m = re.match(r"(\d+)'(\d+)\"?", str(h))
    return int(m.group(1)) * 12 + int(m.group(2)) if m else None

def normalize_position(pos):
    pos = str(pos).upper()
    if "G" in pos and "F" in pos:
        return "Wing"
    if "G" in pos:
        return "Guard"
    if "F" in pos or "C" in pos:
        return "Forward/Post"
    return "Unknown"

NOTES_TAG_KEYWORDS = {
    "catch_and_shoot": ["c&s", "c & s", "catch & shoot", "catch and shoot", "quick release", "spot-up", "spot up"],
    "pull_up_shooter": ["pull up", "pull-up", "mid range", "mid-range", "go to=mid", "step back"],
    "three_point_shooter": ["3's", "3pt", "three", "sniper", "shooter", "shoot"],
    "slasher_driver": ["driver", "drive", "gets to rim", "attacks the rim", "rhd", "lhd", "finish", "attack"],
    "post_scorer": ["post game", "back to the basket", "ls/rh", "post", "power post"],
    "rebounder": ["rebound", "board"],
    "playmaker": ["playmaker", "assist", "creator", "create", "distributor", "main creator", "ball mover", "ball move"],
    "physical_finisher": ["physical", "strong", "bully", "through his defender"],
    "high_usage": ["main creator", "go to"],
    "cutter": ["back cut", "curl"],
}

KEYS_TAG_KEYWORDS = {
    "deny_catch_and_shoot": [
        "c&s", "c & s", "closeout", "close out", "chase", "high hand", "early hand", "arrive on the catch",
        "stunt", "pick and pop", "pick & pop",
    ],
    "anticipate_move": ["anticipate", "antcipate", "antipipate"],
    "keep_in_front": ["keep in front", "keep him in front", "contest", "active hands", "vision off ball",
                       "stay in front", "step up", "spin back"],
    "help_defense": ["help", "gap", "wedge", "talk switches"],
    "box_out_priority": ["box out", "boxout", "keep off glass", "off the glass"],
    "post_defense": ["nls", "no ls/rh", "no rs/lh", "post defense", "front the post", "limit post touches", "post touches"],
    "physical_discipline": ["be physical", "stay down", "do not foul", "no fouls", "no foul", "wall up"],
    "pressure_disrupt": ["pressure", "presure", "disrupt", "speed up"],
    "transition_defense": ["locate in transition", "transition"],
    "deny_cuts": ["back cut"],
    "screen_navigation": ["fight over screens", "over screens", "under screens", "pops after screens"],
}

def tag_player(text, keywords):
    text = text.lower()
    text = re.sub(r"\bnon[- ]shooter\b", "", text)
    return {tag for tag, kws in keywords.items() if any(kw in text for kw in kws)}

player_profiles = all_rosters.copy()
player_profiles["height_inches"] = player_profiles["height"].apply(parse_height_inches)
player_profiles["position_group"] = player_profiles["position"].apply(normalize_position)
player_profiles["notes_tags"] = player_profiles["player_notes"].apply(lambda t: tag_player(t, NOTES_TAG_KEYWORDS))
player_profiles["keys_tags"] = player_profiles["keys_to_defending"].apply(lambda t: tag_player(t, KEYS_TAG_KEYWORDS))
player_profiles["notes_tags_display"] = player_profiles["notes_tags"].apply(lambda s: ", ".join(sorted(s)) if s else "")
player_profiles["keys_tags_display"] = player_profiles["keys_tags"].apply(lambda s: ", ".join(sorted(s)) if s else "")

# The season box score is the "table" element immediately following the "...BOXSCORE" section header. It covers
# the WHOLE roster (including deep bench players never mentioned in the scouting notes) plus a team-total row.
def extract_pdf_season_stats(elements_df, opponent):
    rows = elements_df.to_dict("records")
    box_idx = next(
        (i for i, r in enumerate(rows) if r["element_type"] == "section_header" and "BOXSCORE" in r["element_content"].upper()),
        None,
    )
    if box_idx is None or rows[box_idx + 1]["element_type"] != "table":
        print(f"No season boxscore table found for '{opponent}'.")
        return pd.DataFrame()

    stats_df = read_boxscore_table(rows[box_idx + 1]["element_content"])
    stats_df = stats_df.rename(columns={"#": "jersey_number", "PLAYER": "name"})
    # pdfplumber's table extraction sometimes truncates a multi-word name cell down to its first word (e.g. this
    # PDF's own "Team Total" row comes through as just "Team") -- so filtering on the exact string "Team Total"
    # alone can silently let that aggregate row through into player_profiles (and downstream FG%/3P% consumers
    # like player_comparison.py's parse_pct(), which then chokes on a "made-attempted" string like "617-1511").
    # Its jersey number is reliably "-" regardless of the name-cell truncation, so filter on that instead.
    stats_df = stats_df[stats_df["jersey_number"].astype(str).str.strip() != "-"]
    stats_df = stats_df[stats_df["name"] != "Opponent"]
    stats_df["jersey_number"] = "#" + stats_df["jersey_number"].astype(str)
    stats_df.insert(0, "opponent", opponent)
    return stats_df

pdf_season_stats = pd.concat(
    [extract_pdf_season_stats(df, opponent) for opponent, df in scout_reports.items()],
    ignore_index=True,
) if scout_reports else pd.DataFrame()

missing = pdf_season_stats.merge(
    player_profiles[["opponent", "jersey_number"]], on=["opponent", "jersey_number"], how="left", indicator=True
)
box_only = missing[missing["_merge"] == "left_only"][["opponent", "jersey_number", "name"]].drop_duplicates()

if not box_only.empty:
    box_only = box_only.copy()
    box_only["game_date"] = box_only["opponent"].apply(game_date_for)
    box_only["position"] = None
    box_only["height"] = None
    box_only["weight"] = None
    box_only["class_year"] = None
    box_only["role"] = "Bench"
    box_only["player_notes"] = ""
    box_only["keys_to_defending"] = ""
    box_only["height_inches"] = None
    box_only["position_group"] = "Unknown"
    box_only["notes_tags"] = [set() for _ in range(len(box_only))]
    box_only["keys_tags"] = [set() for _ in range(len(box_only))]
    box_only["notes_tags_display"] = ""
    box_only["keys_tags_display"] = ""
    box_only["has_scouting_report"] = False
    print(f"Adding {len(box_only)} box-score-only player(s) with no scouting writeup (defaulted to role=Bench):")
    print(box_only[["opponent", "jersey_number", "name"]].to_string(index=False))
    player_profiles = pd.concat([player_profiles, box_only[player_profiles.columns.tolist()]], ignore_index=True)
else:
    print("No box-score-only players found -- every player in the season boxscore already has a roster/notes entry.")

# CONFIRMED CHANGE (requested): this used to left-join real season stats from the PDF's boxscore table
# onto player_profiles. Root cause of a real, reported leak: these scout report PDFs are explicitly
# labeled "Last Season" and contain the PRIOR YEAR's per-player numbers -- e.g. Ripon's real stats had
# nothing to do with any game they've played this year, and no reference_date fix can make a stale,
# undated PDF summary trustworthy. Stopped using ANY team or player statistics sourced from the scouting
# report, parser-wide: stat_cols is added here as all-null instead of merged in from pdf_season_stats, so
# player_profiles has the columns every downstream consumer expects, but they start empty. The ONLY
# legitimate source for these numbers from here on is the PBP-derived override further down this
# notebook (see "Override the upcoming opponent's player_profiles stats"), which uses real, current-
# season game data instead. pdf_season_stats itself is left in place above (still parsed, unused for
# stats) since box_only -- the "player has box-score data but no separate notes/tags writeup" backfill a
# few lines up -- still needs it to know which players exist at all.
stat_cols = ["MIN", "FG%", "3PM-A", "3P%", "FTM-A", "FT%", "REB", "AST", "TO", "STL", "BLK", "PTS"]
for _col in stat_cols:
    player_profiles[_col] = None

print("Scouting-report player statistics are disabled by design (PDF reports are last-season data) -- "
      "player_profiles stat columns start blank and are only filled in by the PBP-derived override "
      "further down this notebook, from real current-season games.")

print(player_profiles[[
    "opponent", "jersey_number", "name", "role", "notes_tags_display", "keys_tags_display", "PTS", "REB", "AST", "FG%", "3P%",
]])

Adding 14 box-score-only player(s) with no scouting writeup (defaulted to role=Bench):
             opponent jersey_number            name
St. Thomas (TX) Celts           #23      Jaden Ross
St. Thomas (TX) Celts           #11 Tamarcus Butler
St. Thomas (TX) Celts           #13   Legborsi Mato
    Eureka Red Devils           #35  Max Richardson
    Eureka Red Devils           #20      Nolan Kerr
    Eureka Red Devils           #22      Tony Mabon
    Eureka Red Devils           #15 Amare Brokemond
    Eureka Red Devils            #3   Blake Logsdon
    Eureka Red Devils            #0   Colin DeLaere
    Eureka Red Devils            #1   Dylan Logsdon
      Ripon Red Hawks           #23     Keenan Rahn
      Ripon Red Hawks           #11       Sam Leoni
      Ripon Red Hawks           #20     Anton Kilde
      Ripon Red Hawks            #2    Caeden Holly
Scouting-report player statistics are disabled by design (PDF reports are last-season data) -- player_profiles stat columns start bla


### Break out Keys-to-Victory categories by opponent starter vs. bench role

For each per-game Keys-to-Victory category (from the win/loss-splits cell above), breaks out the scouted opponent's own production in that category by role -- Starter vs. Bench -- to show whether an emphasis like "own the glass" was really driven by starters or bench depth.

In [38]:
# For each per-game "Keys to Victory" category (from the win/loss-splits cell above), break out the SCOUTED
# OPPONENT's own production in that category by role -- Starter vs. Bench -- using the season stats merged onto
# player_profiles. This shows whether an emphasis like "own the glass" was really about containing the
# opponent's starting five or their bench unit.
CATEGORY_TO_STAT_COLS = {}
for col, cat in STAT_COL_CATEGORY.items():
    CATEGORY_TO_STAT_COLS.setdefault(cat, []).append(col)

COUNT_STATS = {"TO", "REB", "AST", "STL", "BLK", "PTS"}

def parse_numeric_stat(val):
    s = str(val).strip()
    if s in ("", "-", "nan", "None"):
        return None
    if s.endswith("%"):
        return float(s.rstrip("%"))
    if "-" in s and not s.startswith("-"):
        try:
            return float(s.split("-")[0])  # compound "made-attempted" string -- use the made count
        except ValueError:
            return None
    try:
        return float(s)
    except ValueError:
        return None

role_breakdown_rows = []
for _, row in game_categories.iterrows():
    if row["category"] == "(no matched category)":
        continue
    stat_cols = [c for c in CATEGORY_TO_STAT_COLS.get(row["category"], []) if c in player_profiles.columns]
    if not stat_cols:
        continue
    opp_players = player_profiles[player_profiles["opponent"] == row["opponent"]]
    for role in ["Starter", "Bench"]:
        role_players = opp_players[opp_players["role"] == role]
        if role_players.empty:
            continue
        for col in stat_cols:
            vals = role_players[col].apply(parse_numeric_stat)
            if vals.notna().any():
                role_breakdown_rows.append({
                    "opponent": row["opponent"], "outcome": row["outcome"], "category": row["category"],
                    "role": role, "stat": col, "players_with_data": int(vals.notna().sum()),
                    "avg_per_player": round(vals.mean(), 2),
                    "role_total": round(vals.sum(), 2) if col in COUNT_STATS else None,
                })

role_breakdown = pd.DataFrame(
    role_breakdown_rows,
    columns=["opponent", "outcome", "category", "role", "stat", "players_with_data", "avg_per_player", "role_total"],
)
print("Opponent production by role (Starter vs Bench) for the stat(s) behind each matched Keys-to-Victory category:")
print(role_breakdown)

count_stat_rows = role_breakdown[role_breakdown["stat"].isin(COUNT_STATS)]
if not count_stat_rows.empty:
    pivot = count_stat_rows.pivot_table(
        index=["opponent", "category", "stat"], columns="role", values="role_total", aggfunc="first"
    ).reset_index()
    for r in ["Starter", "Bench"]:
        if r not in pivot.columns:
            pivot[r] = 0.0
    pivot["starter_share"] = (pivot["Starter"] / (pivot["Starter"] + pivot["Bench"]).replace(0, pd.NA)).round(3)
    print("\nStarter vs Bench totals for count-type stats, side-by-side (starter_share near 1 = concentrated among "
          "starters; near 0 = bench-driven):")
    print(pivot.sort_values(["opponent", "category", "stat"]))
else:
    print("No count-type stats available yet to compare Starter vs Bench totals for the matched categories.")

Opponent production by role (Starter vs Bench) for the stat(s) behind each matched Keys-to-Victory category:
Empty DataFrame
Columns: [opponent, outcome, category, role, stat, players_with_data, avg_per_player, role_total]
Index: []
No count-type stats available yet to compare Starter vs Bench totals for the matched categories.



### Play-by-play parsing functions

Each game's play-by-play is a separate MHTML snapshot of FastScout's playByPlay page (same "Save as Webpage, Single File" format as the season schedule), uploaded as `"<date> UW-Whitewater @ <Opponent>_pbp.mhtml"` (or `"<date> <Opponent> @ UW-Whitewater_pbp.mhtml"` for home games). `parse_pbp_html`/`parse_pbp_mhtml` parse that table; `scrape_pbp_live` is the live-scrape fallback, reused by every PBP-fetching cell further down.

In [40]:
# --- Play-by-play (PBP) parsing functions -------------------------------------------------------------------
# Each game's play-by-play is a separate MHTML snapshot of FastScout's playByPlay page (same "Save as Webpage,
# Single File" format as the season schedule), uploaded as "<date> UW-Whitewater @ <Opponent>_pbp.mhtml" (or
# "<date> <Opponent> @ UW-Whitewater_pbp.mhtml" for home games).
def parse_pbp_html(html):
    """Core play-by-play table parsing, factored out of parse_pbp_mhtml() so it can run on HTML from either
    source: a manually-exported/uploaded MHTML snapshot, OR a live Playwright page.content() capture -- see
    scrape_pbp_live() below, which the user confirmed is reachable via a small path change to a game's own
    "game_url" (already available per-game via team_schedules/opp_schedule -- see the schedule-scraping cell
    and the opponent-schedule cell)."""
    soup = BeautifulSoup(html, "lxml")
    tables = soup.find_all("table")
    raw_df = pd.read_html(StringIO(str(tables[0])))[0]
    # The table's own header row is "Time | <Team A> | Score | <Team B>" -- it NAMES the two teams. That
    # was being thrown away by the rename below, which is fine while there's a roster to match player
    # names against, but resolve_self_column() (opponent prior-games cell) has no roster to use when the
    # opponent has no scouting report, and with nothing to match it silently defaulted to "uww_text" --
    # a coin flip that swaps the two teams' events. Keep the original labels so that fallback has
    # something deterministic to read. Stored on .attrs rather than as columns so every existing caller
    # (which indexes by the fixed names below) is untouched.
    _pbp_header = [str(c).strip() for c in raw_df.columns]
    raw_df.columns = ["time_raw", "uww_text", "score_raw", "opp_text"]

    def _pbp_team_label(idx):
        """The header text at `idx`, or None when pandas invented one (no <th> row) -- an invented
        "Unnamed: 1"/"0" label names no team, and treating it as one would be worse than admitting we
        don't know."""
        if idx >= len(_pbp_header):
            return None
        label = _pbp_header[idx]
        if not label or label.lower().startswith("unnamed") or label.isdigit():
            return None
        return label

    raw_df.attrs["column_teams"] = {"uww_text": _pbp_team_label(1), "opp_text": _pbp_team_label(3)}
    return raw_df


def parse_pbp_mhtml(path):
    return parse_pbp_html(load_html_snapshot(path))


def scrape_pbp_live(page, game_url, timeout_ms=30000, save_path=None):
    """Navigate to a game's own FastScout play-by-play page and capture its rendered event table.

    Confirmed by a live run: the previous approach -- swap the trailing "/boxscore" segment of `game_url`
    for "/playbyplay" and goto() straight there -- does NOT work. FastScout's SPA only renders this kind of
    sub-route via an in-app tab click, exactly like the already-documented behavior in _click_tab_by_text's
    own docstring above ("Direct page.goto() to a sub-route URL ... gets silently redirected back to
    '/analytics/dashboard'"). A direct goto() to ".../playbyplay?..." gets redirected away from the game
    page entirely, so the FIRST "table" wait (8s) never found one -- and then the fallback tab click ALSO
    timed out (30s) because there was no "Play By Play" tab on whatever page it actually landed on. Always
    goto() the game's own ORIGINAL `game_url` (the boxscore URL -- a valid full-page-load entry point, same
    as every other team/opponent page navigation in this notebook) and click the "Play By Play" nav tab
    in-app from there, instead of ever attempting a direct URL swap. If save_path is given, caches the
    scraped HTML there (see _save_scraped_html in Cell 4) -- the same persist-everything-scraped pattern
    already used for scouting reports and team schedules -- so a future run can fall back to it via
    parse_pbp_mhtml (through load_html_snapshot) without needing to live-scrape again.
    """
    _goto_with_auth_retry(page, game_url, "text=Play By Play", timeout_ms)
    _click_tab_by_text(page, "Play By Play", timeout_ms)
    page.wait_for_selector("table", timeout=timeout_ms)
    html = page.content()
    if save_path:
        _save_scraped_html(html, save_path, "scraped play-by-play")
    return parse_pbp_html(html)

### Classify each raw play-by-play text string into an event type

The 4-column layout is "Time | UW-Whitewater | Score | \<Opponent\>" -- almost every row has exactly one of the two team-text columns filled in (one atomic event per row), alongside the running score. A handful of event strings are team-level with no player name at all (a shot-clock/backcourt turnover, or a held-ball jump ball) -- those get matched by exact text instead of a regex.

In [42]:
# The 4-column layout is "Time | UW-Whitewater | Score | <Opponent>" -- almost every row has exactly ONE of the
# two team-text columns filled in (one atomic event per row), alongside the running score. A few event strings
# are TEAM-level with no player name at all (a shot-clock/backcourt turnover, or a held-ball jump ball).
EVENT_PATTERNS = [
    ("jump_ball_won", re.compile(r"^(?P<player>.+?) Wins Jump Ball$")),
    ("jump_ball_lost", re.compile(r"^(?P<player>.+?) Loses Jump Ball$")),
    # A held ball can be logged with any parenthetical reason ("(Held Ball)", "(Block Tie Up)", ...).
    # TEAM_LEVEL_EXACT only ever listed "(Held Ball)", so every other variant fell through to the
    # unclassified fallback and became its own fake player -- "Jump Ball (Block Tie Up)" reached the box
    # score with more total minutes than any real player on the roster. No player group on purpose.
    ("jump_ball_held", re.compile(r"^Jump Ball \(.+\)$")),
    ("made_shot", re.compile(r"^(?P<player>.+?) Makes (?P<shot_type>\d)PT(?: (?P<shot_desc>.+))?$")),
    ("missed_shot", re.compile(r"^(?P<player>.+?) Misses (?P<shot_type>\d)PT(?: (?P<shot_desc>.+))?$")),
    ("rebound_offensive", re.compile(r"^(?P<player>.+?) Offensive Rebound$")),
    ("rebound_defensive", re.compile(r"^(?P<player>.+?) Defensive Rebound$")),
    ("team_deadball_rebound_offensive", re.compile(r"^(?P<player>.+?) Offensive Deadball Rebound$")),
    ("team_deadball_rebound_defensive", re.compile(r"^(?P<player>.+?) Defensive Deadball Rebound$")),
    ("assist", re.compile(r"^(?P<player>.+?) Assists$")),
    ("steal", re.compile(r"^(?P<player>.+?) Steals$")),
    ("block", re.compile(r"^(?P<player>.+?) Blocks$")),
    # The parenthetical turnover-type suffix (e.g. "(Bad Pass)") is present for some individual turnovers but
    # MISSING entirely for others (e.g. "Corey Thompson Turnover") -- making the suffix optional handles both.
    ("turnover", re.compile(r"^(?P<player>.+?) Turnover(?: \((?P<turnover_type>.+)\))?$")),
    # A bare TEAM-level turnover with a parenthetical type but NO player attached (e.g. "Turnover (Offensive
    # Foul)") previously fell all the way through to the "unclassified" catch-all below, whose fallback sets
    # player = the ENTIRE raw text -- so the literal string "Turnover (Offensive Foul)" ended up as its own
    # "player" row in the reconstructed box score (confirmed: exactly this string, in exactly this shape,
    # showed up in a real game's box score). TEAM_LEVEL_EXACT only covered the bare "Turnover" case with no
    # parenthetical at all -- this regex generalizes to ANY bare "Turnover (TYPE)" variant instead of needing
    # every possible type enumerated individually. No `(?P<player>...)` group here on purpose: leaving
    # "player" out of this match's groupdict() means the merged pbp_events row gets player=NaN naturally
    # (pandas fills a missing dict key with NaN when building the DataFrame), which is exactly what makes the
    # box-score builder's `pbp_events["player"].notna()` filter correctly exclude it.
    ("turnover", re.compile(r"^Turnover(?: \((?P<turnover_type>.+)\))?$")),
    # foul_type was required to be "<something> Foul", so a bare "<Player> Commits Foul" -- a real,
    # common line with no foul type recorded -- never matched, and the fallback turned the WHOLE string
    # into a player name ("Damyen Jackson Commits Foul" appeared as a person, with minutes). Making the
    # descriptor optional classifies those correctly AND credits the foul to the right player, rather
    # than just discarding them.
    ("foul", re.compile(r"^(?P<player>.+?) Commits (?P<foul_type>.*?Foul)$")),
    ("free_throw_made", re.compile(r"^(?P<player>.+?) Makes Free Throw \((?P<ft_num>\d+) of (?P<ft_total>\d+)\)$")),
    ("free_throw_missed", re.compile(r"^(?P<player>.+?) Misses Free Throw \((?P<ft_num>\d+) of (?P<ft_total>\d+)\)$")),
    ("sub_in", re.compile(r"^(?P<player>.+?) Subs In$")),
    ("sub_out", re.compile(r"^(?P<player>.+?) Subs Out$")),
    # A timeout is called by a TEAM or an official ("Official TV Timeout"), never by a roster player, so
    # the caller is deliberately not captured as `player` -- it stays in raw_text. Keeping it meant
    # "Official TV" and every team name showed up wherever player names are enumerated.
    ("timeout", re.compile(r"^.+? Timeout$")),
    ("ejected", re.compile(r"^(?P<player>.+?) Ejected$")),
]
TEAM_LEVEL_EXACT = {
    "Turnover": {"event_type": "turnover", "player": None, "turnover_type": "Team"},
    # A few games log a bare team rebound with NO player prefix at all -- classified into the same
    # team_deadball_rebound_* buckets as the other team-level rebound format.
    "Offensive Rebound": {"event_type": "team_deadball_rebound_offensive", "player": None},
    "Defensive Rebound": {"event_type": "team_deadball_rebound_defensive", "player": None},
    "Jump Ball (Held Ball)": {"event_type": "jump_ball_held", "player": None},
    # A foul logged with neither a player nor a type.
    "Commits Foul": {"event_type": "foul", "player": None},
}


def classify_event(text):
    text = text.strip()
    if text in TEAM_LEVEL_EXACT:
        return dict(TEAM_LEVEL_EXACT[text])
    for event_type, pattern in EVENT_PATTERNS:
        m = pattern.match(text)
        if m:
            return {"event_type": event_type, **m.groupdict()}
    # CONFIRMED BUG (fixed here): this used to return `player=text`, so any line the patterns above don't
    # recognise became a PLAYER named after the raw event string -- with its own box-score row, its own
    # minutes from the lineup reconstruction, and a place in the leaderboards. Keep the text in raw_text
    # (build_pbp_events already stores it) and leave `player` empty, so an unrecognised line is counted and
    # reported but can never masquerade as a person. Add a pattern above for anything that shows up here.
    return {"event_type": "unclassified", "player": None}

### Turn the raw 4-column play-by-play table into one row per event

`build_pbp_events` carries the running score forward onto every row and labels each row with which team it belongs to. `self_column` must be resolved per-file (see `resolve_self_column` later on) since FastScout puts the exporting team's own events in a fixed column regardless of home/away, but which raw column that is varies by whose account captured the snapshot.

In [44]:
def parse_time_to_seconds(time_raw):
    """'19:58 (H1)' -> ('H1', 1198)."""
    if pd.isna(time_raw):
        return None, None
    m = re.match(r"(\d+):(\d+)\s*\((\w+)\)", str(time_raw).strip())
    if not m:
        return None, None
    minutes, seconds, period = m.groups()
    return period, int(minutes) * 60 + int(seconds)


def build_pbp_events(raw_df, opponent, game_date, self_team="UW-Whitewater", self_column="uww_text"):
    """One row per event (or per period-start/end marker), carrying the running score after that event.
    `self_team` labels whichever column `self_column` points at. Defaults match UW-Whitewater's own game files,
    where the "uww_text" column is always UWW's own events. For an opponent's OWN schedule snapshot (e.g.
    scouting an opponent's games before they face Whitewater), FastScout puts the exporting team's own events in
    a FIXED column regardless of home/away -- but which raw column that is varies file-to-file depending on
    whose FastScout account captured it, so the caller must resolve `self_column` empirically (e.g. by matching
    known roster player names) rather than assume "uww_text"."""
    rows = []
    current_period = None
    for i, r in raw_df.iterrows():
        period, seconds = parse_time_to_seconds(r["time_raw"])
        if period:
            current_period = period
        score_raw = r["score_raw"]
        score_m = re.match(r"^(\d+)-(\d+)$", str(score_raw).strip()) if pd.notna(score_raw) else None
        if score_m is None:
            # e.g. "Start 1st Half" / "End 2nd Half" -- no team/player/score attached to these marker rows
            rows.append({
                "opponent": opponent, "game_date": game_date, "event_order": i, "period": current_period,
                "time_remaining": None, "time_remaining_seconds": None, "team": None,
                "event_type": "period_marker", "raw_text": str(score_raw).strip(),
                "uww_score": None, "opp_score": None,
            })
            continue
        uww_score, opp_score = int(score_m.group(1)), int(score_m.group(2))
        opp_column = "opp_text" if self_column == "uww_text" else "uww_text"
        for team_label, text in [(self_team, r[self_column]), (opponent, r[opp_column])]:
            if pd.notna(text) and str(text).strip():
                rows.append({
                    "opponent": opponent, "game_date": game_date, "event_order": i, "period": current_period,
                    "time_remaining": r["time_raw"], "time_remaining_seconds": seconds, "team": team_label,
                    "raw_text": str(text).strip(), "uww_score": uww_score, "opp_score": opp_score,
                    **classify_event(str(text).strip()),
                })
    return pd.DataFrame(rows)

### Extract the opponent name from a `_pbp.mhtml` filename

In [46]:
def opponent_from_pbp_filename(path):
    """Home/away-aware opponent extraction -- mirrors opponent_from_scout_filename() used for the scout PDFs.
    Filenames are "<date> <Away> @ <Home>_pbp.mhtml" (or "..._pbp.html" for a live-scraped/cached file -- see
    _save_scraped_html), so the correct opponent is whichever side of "@" ISN'T "UW-Whitewater", not simply
    "everything after @" -- that naive approach is right for away games ("UW-Whitewater @ Ripon" -> "Ripon")
    but wrong for home games ("Aurora @ UW-Whitewater" would otherwise extract "UW-Whitewater" itself as the
    opponent).

    CONFIRMED BUG (fixed here): this only ever stripped the "_pbp.mhtml" suffix, never "_pbp.html" -- so for
    every ".html"-sourced file (any auto-downloaded/live-scraped PBP, which is most of them going forward)
    the extension silently rode along attached to whichever side of "@" this function returns. For an AWAY
    game specifically, that's the side actually returned (left == "UW-Whitewater", so `right` -- e.g. "Ripon
    Red Hawks_pbp.html" -- comes back instead of "Ripon Red Hawks"), corrupting the opponent name used to key
    every downstream table (pbp_events, pbp_box_score, lineup_stints, etc.) for that game. HOME games
    happened to come out clean by accident (the garbage suffix landed on `right`, which isn't the branch
    returned when left != "UW-Whitewater"), which is why this only ever broke away games -- confirmed via the
    box-score reconciliation diagnostic above flagging every single away game and zero home games."""
    name = re.sub(r"_pbp\.(mhtml|html)$", "", os.path.basename(path))
    name = re.sub(r"^\d+_\d+_\d+\s+", "", name)
    left, right = [side.strip() for side in name.split(" @ ", 1)]
    return right if left == "UW-Whitewater" else left


# --- A GAME IS (opponent, game_date), NEVER opponent ALONE ---------------------------------------
# CONFIRMED BUG (fixed here): every table below used the short opponent name as a game's identity.
# That silently merges a home-and-home (or a third conference meeting) into ONE game: box-score
# stats get summed across both meetings, the games-played denominator counts them once, and the
# lineup-stint clock -- diffed within ("opponent", "period") while `event_order` restarts at 0 each
# game -- interleaves the two meetings and re-counts the same seconds, inflating minutes many-fold.
# Reconciling uww_pbp_box_score against uww_schedule showed this exactly: UW-La Crosse came out
# 206-209 (63+65+78 vs 60+68+81), i.e. three real games stacked into one row.
#
# GAME_KEYS is the grouping/merge key every per-game computation must use from here on.
GAME_KEYS = ["opponent", "game_date"]


def game_date_from_pbp_filename(path):
    """The game's own date, read from the '<m>_<d>_<yy> ' prefix these files are named with.

    This is the ONLY per-file source of truth for WHICH meeting a play-by-play file covers.
    game_date_for() cannot answer that -- it fuzzy-matches the schedule on opponent name and takes
    .iloc[0], so for a rematch it always returns the FIRST meeting's date. Returns None when a file
    carries no date prefix, so the caller can fall back (loudly) rather than guessing silently.
    """
    from datetime import date as _date
    m = re.match(r"^(\d+)_(\d+)_(\d+)\s", os.path.basename(path))
    if not m:
        return None
    month, day, yy = (int(g) for g in m.groups())
    return _date(2000 + yy, month, day)


### Identify the upcoming opponent

Read the single "Upcoming" row off `schedule` (built in Cell 4) to get the opponent's full name, then match it against `scouted_opponents` to get the short name used throughout the rest of this notebook (filenames, `player_profiles`, etc).

In [48]:
# Identify the upcoming opponent from the schedule
# CONFIRMED BUG (fixed here): this assumed there's ALWAYS a row flagged "Upcoming" (a next game to
# prepare for), AND that whichever opponent it is always already has a scout report on file. Neither
# holds for a fully historical/archived season run: reference_date set after an already-played game
# (the normal way to mark a past game as "already happened") makes the SCHEDULE'S NEXT game "Upcoming"
# whether or not any data has actually been prepared for it -- and for an old, archived opponent there
# is no live version of their page left to auto-scrape a report from either. .iloc[0] on no "Upcoming"
# row raised IndexError; next() with no scouted_opponents match raised StopIteration. Both are now
# handled the same way the rest of this notebook already treats "not enough data yet" --
# upcoming_game/upcoming_opponent/upcoming_opponent_short fall back to None with a clear explanation,
# instead of crashing the whole run over what is actually a normal state for a purely historical load.
_upcoming_rows = schedule[schedule["Upcoming"] == "Yes"]
if _upcoming_rows.empty:
    print("No game on/after reference_date found in the schedule -- nothing to identify as \"upcoming\" "
          "(expected for a fully historical/archived season run). Downstream cells that rely on an "
          "upcoming opponent will be skipped or come back empty rather than crash.")
    upcoming_game = None
    upcoming_opponent = None
    upcoming_opponent_short = None
else:
    upcoming_game = _upcoming_rows.iloc[0]
    upcoming_opponent = upcoming_game["opponent"]
    upcoming_opponent_short = next(
        (s for s in scouted_opponents if re.search(re.escape(s), upcoming_opponent, re.IGNORECASE)), None
    )
    if upcoming_opponent_short is None:
        # CONFIRMED BUG (fixed here): leaving this as None switched OFF the entire upcoming-game
        # pipeline, not just the scouting-report parts of it. upcoming_opponent_short is the key every
        # downstream cell keys on -- opp_team_schedule, prev_games, the "_pbp"/"_video" glob patterns,
        # pbp_events_upcoming's self_team label, the player_profiles and team_totals PBP overrides -- so
        # None meant no opponent season leaders and no opponent team stats either, even though NONE of
        # that data comes from a scouting report: it is all reconstructed from the opponent's own
        # play-by-play. Reported live with before_scout="yes": the Stats & Analysis page came up blank.
        #
        # Having no report for the next opponent is a NORMAL state, not a failure -- it is every
        # opponent's state until their report gets built, and it is exactly what before_scout="yes"
        # reproduces on purpose. Fall back to the schedule's own opponent text as the short name. That
        # is the same string the auto-downloader builds scout/pbp filenames from (see
        # _download_matching_scout_pdf's `matchup`), so the glob patterns downstream still match. When a
        # report DOES exist, the scouted_opponents match above still wins, so nothing changes.
        upcoming_opponent_short = upcoming_opponent
        print(f"No scout report on file for the next scheduled opponent ({upcoming_opponent}) -- using "
              f"the schedule's own name for them as the short name, so their prior-game PBP (season "
              f"leaders, team stats, lineups) is still built. Only the scouting-report-sourced output "
              f"(game plan, keys to victory, player notes, tag-based comparisons) will be missing.")

No scout report on file for the next scheduled opponent (Aurora Spartans) -- using the schedule's own name for them as the short name, so their prior-game PBP (season leaders, team stats, lineups) is still built. Only the scouting-report-sourced output (game plan, keys to victory, player notes, tag-based comparisons) will be missing.



### Roster pages: photo, jersey number, position, height, class year -- bio fields only, never stats

FastScout has a team roster page at the same kind of URL as the schedule/games page (a team's base URL with
`/roster`), for UW-Whitewater and for every opponent. This scrapes it for **photo, jersey number, position,
height and class year only** -- any table on that page that looks like a stat table (PPG, RPG, minutes, ...)
is dropped before anything is read, so no game or season stat is ever pulled from these pages.

Selectors (`#myTeamRoster`, the "ROSTER" nav tab) are best-effort guesses modeled on the schedule tab's own
pattern, since this environment has no live FastScout access to confirm them against. If they're wrong, this
cell reports exactly what it tried and falls back safely -- `uww_live_rosters.csv` comes back empty and
downstream tables (Personnel Details) keep using the sample placeholder, rather than the run failing.

In [50]:
# --- Roster pages: player photo, number, position, height, class year -- for UWW and every opponent ------
# CONFIRMED CHANGE (requested): FastScout has a team roster page separate from the scouting report, at the
# same kind of URL as a team's schedule/games page -- a team's own base URL with "/roster" instead of
# "/games". It shows a photo, jersey number, position, height and class year per player. The user was
# explicit: pull ONLY those bio fields, never any per-game or season stat number that page might also show
# (PPG, RPG, minutes, etc.) -- _ROSTER_STATS_HEADERS below exists specifically to keep that promise, by
# dropping any table that looks like a stat table before any player data is read out of the page at all.
#
# SELECTORS ARE BEST-EFFORT GUESSES, not yet confirmed against a live page (this environment has no
# FastScout credentials or network access to verify against). They are modeled directly on two things this
# notebook already knows for certain:
#   1. The SCHEDULE tab's own pattern (see login_to_fastscout / scrape_rendered_html above): a nav tab
#      clicked by its visible text ("SCHEDULE"), landing in a container with a predictable id
#      ("#myTeamSchedule"). "ROSTER" / "#myTeamRoster" follow that same naming convention.
#   2. The exact bullet-separated player-line format FastScout's OWN PDF roster export already uses --
#      "#<jersey> \u2022 <name> \u2022 <pos> \u2022 <height> \u2022 <weight> \u2022 <class>" (see PLAYER_LINE_RE in the
#      scouting-report PDF parser earlier in this notebook). Since that PDF is generated by FastScout from
#      this same underlying roster data, the live page's player rows are a reasonable bet to carry the same
#      shape. _ROSTER_LINE_RE below is that same pattern, loosened to tolerate different bullet characters
#      and optional weight, and matched against the whole page's flattened text rather than assuming a
#      specific tile/DOM structure -- so it degrades gracefully (matches nothing, rather than matching
#      wrong) if the live markup turns out to differ.
# If either guess is wrong, this cell fails safe: it prints exactly what it tried and why it came back
# empty, and every downstream consumer already treats a missing/empty uww_live_rosters.csv as "no live
# roster yet" (falls back to the sample Personnel Details table), not a hard error.

_ROS_OUT = "uww_live_rosters"
from urllib.parse import unquote  # for decoding the player name FastScout encodes into its photo CDN URLs
_ROSTER_STATS_HEADERS = {
    "pts", "reb", "ast", "stl", "blk", "to", "tov", "pf", "fg", "fg%", "3p", "3p%", "3pt", "ft", "ft%",
    "ppg", "rpg", "apg", "spg", "bpg", "mpg", "gp", "gs", "min", "eff",
}
# Same bullet-separated shape as the PDF roster's PLAYER_LINE_RE, loosened for an unknown live separator
# and an optional weight field between height and class. Height accepts both "6-2" and "6'2\"" spellings.
_ROSTER_LINE_RE = re.compile(
    r"#(\d{1,2})\s*[\u2022|\u00b7]\s*([A-Za-z][A-Za-z.'\-\u00e9\u00f1 ]+?)\s*[\u2022|\u00b7]\s*"
    r"([A-Z]{1,4}(?:/[A-Z]{1,4})?)\s*[\u2022|\u00b7]\s*"
    r"(\d{1,2}[-'\u2019]\d{1,2}\"?)\s*(?:[\u2022|\u00b7]\s*\d{2,3}\s*(?:lbs?)?\s*)?[\u2022|\u00b7]\s*"
    r"([A-Za-z]{2,10})\.?"
)


def _ros_normalize_height(raw):
    """"6'2\"" or "6\u20192" -> "6-2"; "6-2" passes through unchanged -- matches the hyphenated convention
    every other height value in this pipeline already uses."""
    m = re.match(r"(\d{1,2})[-'\u2019](\d{1,2})", str(raw))
    return f"{m.group(1)}-{m.group(2)}" if m else str(raw)


def _ros_norm_name(name):
    return re.sub(r"\s+", " ", str(name)).strip().lower()


# CONFIRMED BY A LIVE RUN: FastScout's headshot CDN URL carries the player's own name in its path --
# "https://stats-assets.fastmodelsports.com/images/personnel/Aurora/2025-2026/Josh%20Tinney/1763067644351"
# -- which means a photo can be attached to a player by NAME, not by which position it happens to occupy on
# the page. That matters because the position-based pairing this cell originally used was confirmed wrong on
# a live run (Mekhi Doby's row carried Josh Tinney's photo), so name-matching replaces it as the primary
# method; positional pairing is now only a fallback, used solely for images whose URL doesn't carry a name
# (already excluded from that fallback pool if it turned out to be a genuine 1:1 correspondence, this would
# be unnecessary -- it isn't, so treat any positionally-assigned photo as lower-confidence than a name match).
_ROSTER_PHOTO_NAME_RE = re.compile(r"/personnel/[^/]+/[^/]+/([^/]+)/[^/]+/?$")


def _ros_photo_owner(url):
    """The player name encoded in a FastScout headshot CDN URL, if the URL has that shape. None for a URL
    that doesn't match -- e.g. a generic silhouette placeholder with no player-specific path."""
    if not url:
        return None
    m = _ROSTER_PHOTO_NAME_RE.search(url)
    if not m:
        return None
    return unquote(m.group(1)).replace("+", " ").strip()


# CONFIRMED BUG (fixed here): the roster page is a JavaScript app -- a saved capture of UW-Whitewater's page
# had the nav chrome, the "PLAYERS PER ROW" toggle and the sort dropdown, and 398 characters of text in total:
# not one player. The old waits looked for "#myTeamRoster", which does not exist on this page at all (the only
# roster-ish id is "roster-page-sort-dropdown"), so both waits timed out and the shell was captured anyway.
# Wait for actual player content instead -- FastScout's own headshot URLs (/images/personnel/...) are the
# reliable tell, with the bullet player line as a backup for a player with no photo.
_ROS_PLAYER_IMG_SELECTOR = "img[src*='/images/personnel/']"


def _ros_wait_for_players(page, timeout_ms=30000):
    """Poll until the roster actually renders. Returns the number of player photos found (-1 when players are
    on the page as text but have no photos, 0 when nothing rendered before the timeout)."""
    try:
        page.wait_for_load_state("networkidle", timeout=timeout_ms)
    except Exception:
        pass
    waited = 0
    while waited < timeout_ms:
        try:
            found = int(page.evaluate(
                f"document.querySelectorAll({_ROS_PLAYER_IMG_SELECTOR!r}).length"))
        except Exception:
            found = 0
        if found:
            return found
        try:
            body_text = page.evaluate("document.body.innerText") or ""
        except Exception:
            body_text = ""
        if re.search(r"#\d{1,2}\s*[\u2022|\u00b7]", body_text):
            return -1
        try:
            page.mouse.wheel(0, 1200)  # nudge anything that renders lazily on scroll
        except Exception:
            pass
        page.wait_for_timeout(500)
        waited += 500
    return 0


def _ros_open_list_view(page):
    """The roster page has a PLAYERS PER ROW toggle; the 1-per-row view lays each player out as a row with his
    number, position, height and class beside the photo, which is the shape this parser reads best."""
    try:
        if page.locator("#players-per-row-1").count():
            page.click("#players-per-row-1", timeout=5000)
            page.wait_for_timeout(750)
    except Exception:
        pass


def _ros_cards_from_dom(soup, team_label):
    """Fallback parse: one row per headshot, taking the player's NAME from the image URL (FastScout encodes it
    there) and the rest from the text around the image. Works when the page renders player details as separate
    elements rather than one bullet-separated line."""
    rows = []
    seen = set()
    for img in soup.select(_ROS_PLAYER_IMG_SELECTOR):
        src = img.get("src") or ""
        owner = _ros_photo_owner(src)
        if not owner or _ros_norm_name(owner) in seen:
            continue
        seen.add(_ros_norm_name(owner))
        node, text = img, ""
        for _ in range(5):  # climb until the surrounding block has more than just the image
            node = node.parent
            if node is None:
                break
            text = node.get_text(" ", strip=True)
            if len(text) > len(owner) + 6:
                break
        jersey = re.search(r"#\s*(\d{1,2})\b", text)
        pos = re.search(r"\b([A-Z]{1,2}(?:/[A-Z]{1,2})?)\b(?!\w)", text.replace(owner, ""))
        height = re.search(r"\b(\d{1,2}[-'\u2019]\d{1,2})\b", text)
        year = re.search(r"\b(Fr|So|Jr|Sr|Gr|R-Fr|R-So|R-Jr|R-Sr)\.?\b", text)
        rows.append({
            "team": team_label, "jersey_number": jersey.group(1) if jersey else None,
            "name": owner, "position": pos.group(1) if pos else None,
            "height": _ros_normalize_height(height.group(1)) if height else None,
            "class_year": year.group(1) if year else None,
            "photo_url": _resolve_team_link(src), "source": "FastScout roster page",
        })
    return rows


def parse_roster_html(html, team_label):
    """Bio fields only, by design: strip any table that looks like a stat table BEFORE reading anything
    else off the page, then regex-match the bullet-separated player lines in the remaining flattened text.
    Photos are attached by the player name encoded in the image's own CDN URL (_ros_photo_owner) whenever
    the URL carries one; page position is used only as a fallback for a photo whose URL doesn't name its
    owner, and that fallback is exactly the case a live run showed CAN mismatch -- so a positionally-assigned
    photo is flagged in the diagnostic count below rather than trusted silently."""
    soup = BeautifulSoup(html, "lxml")
    for tbl in soup.find_all("table"):
        header_text = " ".join(c.get_text(" ", strip=True).lower() for c in tbl.find_all(["th", "td"])[:20])
        header_tokens = set(re.findall(r"[a-z%]+", header_text))
        if header_tokens & _ROSTER_STATS_HEADERS:
            tbl.decompose()

    text = re.sub(r"\s+", " ", soup.get_text(" ", strip=True))
    matches = list(_ROSTER_LINE_RE.finditer(text))
    photo_urls = [
        img.get("src") for img in soup.find_all("img")
        if img.get("src") and not re.search(r"logo|icon|crest|badge", img.get("src"), re.IGNORECASE)
    ]
    photo_by_owner = {}
    unnamed_photos = []
    for _src in photo_urls:
        _owner = _ros_photo_owner(_src)
        if _owner:
            photo_by_owner[_ros_norm_name(_owner)] = _src
        else:
            unnamed_photos.append(_src)
    _unnamed_iter = iter(unnamed_photos)
    _n_by_name = _n_by_position = 0

    rows = []
    seen_jerseys = set()
    for m in matches:
        jersey, name, pos, height_raw, class_year = m.groups()
        # The same bullet line can appear twice if the page repeats a player card (e.g. a "starters" and a
        # "full roster" panel on one page) -- keep the first occurrence only, per team.
        if jersey in seen_jerseys:
            continue
        seen_jerseys.add(jersey)
        _photo = photo_by_owner.get(_ros_norm_name(name))
        if _photo is not None:
            _n_by_name += 1
        else:
            _photo = next(_unnamed_iter, None)
            if _photo is not None:
                _n_by_position += 1
        rows.append({
            "team": team_label, "jersey_number": jersey, "name": name.strip(),
            "position": pos.strip(), "height": _ros_normalize_height(height_raw),
            "class_year": class_year.strip().rstrip("."),
            "photo_url": _resolve_team_link(_photo) if _photo else None,
            "source": "FastScout roster page",
        })
    if _n_by_position:
        print(f"    [roster] {team_label}: {_n_by_name} photo(s) matched by name in the URL, "
              f"{_n_by_position} fell back to page position (lower confidence -- spot-check these).")
    if not rows:
        rows = _ros_cards_from_dom(soup, team_label)
        if rows:
            print(f"    [roster] {team_label}: no bullet-separated player lines on this page -- read "
                  f"{len(rows)} player(s) from the headshot URLs and the text around them instead.")
    return pd.DataFrame(rows, columns=["team", "jersey_number", "name", "position", "height", "class_year",
                                       "photo_url", "source"])


def scrape_roster_rendered_html(page, url, wait_selector=None, timeout_ms=30000, save_path=None):
    """Live-scrape one team's roster page: navigate to its base team URL, click the 'ROSTER' nav tab (see
    _click_tab_by_text -- a direct URL to the sub-route doesn't render it, same as '/games'/'/documents'),
    then read the rendered HTML. Falls back to returning whatever HTML is on screen if wait_selector's
    guessed id never appears, rather than failing outright on a selector guess that may be wrong."""
    print(f"    [scrape] navigating to {url}")
    _goto_with_auth_retry(page, url, "text=ROSTER", timeout_ms)
    print(f"    [scrape] landed on {page.url} -- clicking 'ROSTER' tab")
    _click_tab_by_text(page, "ROSTER", timeout_ms)
    _ros_open_list_view(page)
    _found = _ros_wait_for_players(page, timeout_ms)
    if _found == 0:
        print(f"    [scrape] no player photos or player lines rendered within {timeout_ms}ms -- capturing the "
              f"page anyway, but expect this team to come back with no roster rows.")
    html = page.content()
    if save_path:
        _save_scraped_html(html, _add_season_suffix_to_path(save_path, html), "roster page")
    return html


def scrape_uww_live_roster(page, timeout_ms=30000, save_path=None):
    """UWW's own roster -- assumes page is already on an authenticated team page (post-login), same
    assumption scrape_uww_live_schedule makes for the SCHEDULE tab."""
    print("    [scrape] clicking 'ROSTER' tab")
    _click_tab_by_text(page, "ROSTER", timeout_ms)
    _ros_open_list_view(page)
    _found = _ros_wait_for_players(page, timeout_ms)
    if _found == 0:
        print(f"    [scrape] no player photos or player lines rendered within {timeout_ms}ms -- capturing the "
              f"page anyway, but expect this team to come back with no roster rows.")
    html = page.content()
    if save_path:
        _save_scraped_html(html, _add_season_suffix_to_path(save_path, html), "UW-Whitewater's own roster")
    return html


def _ros_local_backup(team_label):
    """A previously-saved roster snapshot for this team, either a genuine MHTML export or this cell's own
    live-scrape cache -- same skip-the-live-scrape-if-we-already-have-one convention as the schedule cell."""
    key = team_label.split()[0].lower()
    for p in sorted(glob.glob(f"{schedules_dir}/*.html") + glob.glob(f"{schedules_dir}/*.mhtml")):
        base = os.path.basename(p)
        if re.search(r"-\s*Roster", base, re.IGNORECASE) and key in base.lower():
            return p
    return None


_ros_problems = []

# CONFIRMED BUG (fixed here): UWW came back with ZERO roster rows while every opponent scraped fine. UWW was
# the one target with no URL -- it clicked "ROSTER" on whatever page the session already had open, which after
# login_to_fastscout is ".../teams/myTeam/analytics/dashboard". That ANALYTICS view has its own sub-nav and no
# ROSTER tab, so the click failed, the per-target except swallowed it, and our own players silently had no
# photos in the brief. UWW now navigates to its own TEAM page first, exactly like every opponent does -- the
# "myTeam" alias works in that URL the same way it does for the dashboard.
#
# Targets: UWW itself, every opponent already resolved for the schedule (the `opponents` DataFrame built
# above), plus the upcoming opponent specifically -- `opponents` only covers SCOUTED games, so with
# before_scout="yes" the upcoming opponent would otherwise be skipped even though their roster is exactly
# what's needed to prep for them.
_ROS_UWW_URL = (f"{FASTSCOUT_ORIGIN}/teams/myTeam?league={FASTSCOUT_DOCS_LEAGUE}&season={FASTSCOUT_DOCS_SEASON}")
_ros_targets = [(uww_team_schedule["team"].iloc[0] if not uww_team_schedule.empty else "UW-Whitewater",
                 _ROS_UWW_URL)]
if not opponents.empty:
    for _, _r in opponents.iterrows():
        _ros_targets.append((_r["opponent"], _r["opponent_url"]))
if upcoming_opponent and upcoming_opponent not in {t for t, _ in _ros_targets}:
    _up_rows = raw_uww_games[raw_uww_games["opponent"] == upcoming_opponent] if "raw_uww_games" in dir() else pd.DataFrame()
    if not _up_rows.empty and pd.notna(_up_rows.iloc[0].get("opponent_url")):
        _ros_targets.append((upcoming_opponent, _up_rows.iloc[0]["opponent_url"]))
    else:
        _ros_problems.append(f"could not resolve a team URL for the upcoming opponent ({upcoming_opponent}) "
                             f"-- their roster was skipped this run.")

_seen_teams = set()
_ros_targets = [t for t in _ros_targets if not (t[0] in _seen_teams or _seen_teams.add(t[0]))]
print(f"Roster targets this run: {[t for t, _ in _ros_targets]}")


def _run_roster_scrape_session(login_page):
    results = {}
    for _team_label, _team_url in _ros_targets:
        if _ros_local_backup(_team_label):
            print(f"  Skipping live roster scrape for {_team_label} -- a local roster file already exists.")
            continue
        try:
            _save_path = os.path.join(schedules_dir, f"{_team_label} - Roster.html")
            if _team_url is None:
                results[_team_label] = scrape_uww_live_roster(login_page, save_path=_save_path)
            else:
                try:
                    results[_team_label] = scrape_roster_rendered_html(login_page, _team_url, save_path=_save_path)
                except Exception as _nav_error:
                    if _team_url != _ROS_UWW_URL:
                        raise
                    # Our own team only: if the "myTeam" team-page URL doesn't take, fall back to clicking
                    # ROSTER on whatever authenticated page the session already has open.
                    print(f"  {_team_label}: team-page URL didn't work ({type(_nav_error).__name__}: "
                          f"{_nav_error}) -- trying the ROSTER tab on the current page instead.")
                    results[_team_label] = scrape_uww_live_roster(login_page, save_path=_save_path)
        except Exception as _scrape_error:
            # Loud, not swallowed: a failure here is why a team ends up with no photos at all, and the brief
            # can only report the symptom.
            _ros_problems.append(f"{_team_label}: roster scrape failed ({type(_scrape_error).__name__}: "
                                 f"{_scrape_error}) -- no photos or bio details for this team this run.")
            print(f"  Could not scrape roster for {_team_label}: {type(_scrape_error).__name__}: {_scrape_error}")
    return results


scraped_roster_html = {}
if not (fastscout_username and fastscout_password):
    _ros_problems.append("no FastScout credentials (FASTSCOUT_USERNAME/FASTSCOUT_PASSWORD) -- live roster "
                         "scrape skipped for every team; only local ' - Roster' backups (if any) were used.")
elif not _ros_targets:
    _ros_problems.append("no roster targets resolved (no schedule loaded yet) -- nothing to scrape.")
else:
    try:
        scraped_roster_html = run_in_fastscout_session(_run_roster_scrape_session)
    except Exception as _session_error:
        _ros_problems.append(f"could not start an authenticated FastScout session for rosters: "
                             f"{type(_session_error).__name__}: {_session_error}")

_ros_frames = []
for _team_label, _team_url in _ros_targets:
    _html = scraped_roster_html.get(_team_label)
    if _html is None:
        _backup = _ros_local_backup(_team_label)
        if _backup:
            _html = load_html_snapshot(_backup)
    if _html is None:
        continue
    try:
        _parsed = parse_roster_html(_html, _team_label)
        if _parsed.empty:
            _ros_problems.append(f"{_team_label}: page loaded but no player lines matched the expected "
                                 f"bullet pattern -- the live page's format likely differs from the PDF "
                                 f"roster's; _ROSTER_LINE_RE needs adjusting once the real markup is visible.")
        _ros_frames.append(_parsed)
    except Exception as _parse_error:
        _ros_problems.append(f"{_team_label}: could not parse roster HTML: "
                             f"{type(_parse_error).__name__}: {_parse_error}")

live_rosters = (pd.concat(_ros_frames, ignore_index=True) if _ros_frames else
               pd.DataFrame(columns=["team", "jersey_number", "name", "position", "height", "class_year",
                                     "photo_url", "source"]))

print(f"\nlive_rosters: {len(live_rosters)} player(s) across {live_rosters['team'].nunique() if not live_rosters.empty else 0} team(s).")
for _team_label, _ in _ros_targets:
    _n = int((live_rosters["team"] == _team_label).sum()) if not live_rosters.empty else 0
    if _n == 0:
        print(f"  {_team_label}: NO roster rows -- check the saved '{_team_label} - Roster.html' in "
              f"{schedules_dir} to see what the page actually returned.")
if not live_rosters.empty:
    print(live_rosters.groupby("team").size().to_string())
if _ros_problems:
    print("\nRoster problems:")
    for _p in _ros_problems:
        print(f"  - {_p}")


Roster targets this run: ['UW-Whitewater Warhawks', 'Ripon Red Hawks', 'St. Thomas (TX) Celts', 'Eureka Red Devils', 'Aurora Spartans']
  Skipping live roster scrape for UW-Whitewater Warhawks -- a local roster file already exists.
  Skipping live roster scrape for Ripon Red Hawks -- a local roster file already exists.
  Skipping live roster scrape for St. Thomas (TX) Celts -- a local roster file already exists.
  Skipping live roster scrape for Eureka Red Devils -- a local roster file already exists.
  Skipping live roster scrape for Aurora Spartans -- a local roster file already exists.
    [roster] UW-Whitewater Warhawks: 19 photo(s) matched by name in the URL, 2 fell back to page position (lower confidence -- spot-check these).
    [roster] St. Thomas (TX) Celts: 20 photo(s) matched by name in the URL, 1 fell back to page position (lower confidence -- spot-check these).
    [roster] Eureka Red Devils: 25 photo(s) matched by name in the URL, 6 fell back to page position (lower confi


### Locate the opponent's own schedule backup file

`schedule` only has ONE row per opponent (their single game vs UWW), so it can't tell us what this opponent's own games looked like before they played Whitewater. `_schedule_file_for` finds their own "\<Team\> - Schedule.mhtml"/".html" backup file on disk -- used only as a fallback (see the next cell, which prefers the schedule Cell 4 already scraped/parsed live).

In [52]:
# `schedule` only has ONE row per opponent (their single game vs UWW), so it can't tell us what THIS
# opponent's own games looked like before they played Whitewater. Load their own FastScout team schedule page
# instead -- saved the same way as UW-Whitewater's own ("<Opponent> - Schedule.mhtml"), parsed identically to
# the schedule-parsing cell above.
def _schedule_file_for(opponent_full_name, opponent_short_hint, volume_dir):
    """Find this opponent's own '<Team> - Schedule.mhtml' backup file, tolerant of short-vs-full naming --
    confirmed by a live run: scouted_opponents (and upcoming_opponent_short derived from it) now holds FULL
    team names (e.g. 'Elmhurst Bluejays') since auto-downloaded HTML scout reports carry the schedule's full
    opponent name in their filename, but the manually-uploaded backup schedule MHTMLs predate that change and
    still use the SHORT team name (e.g. 'Elmhurst - Schedule.mhtml') -- an exact '{short} - Schedule.mhtml'
    match no longer finds them. Match on whichever side is a substring of the other instead."""
    # Glob both ".mhtml" (a manually-exported/uploaded snapshot) and ".html" (this notebook's own
    # live-scrape cache -- see _save_scraped_html in Cell 4).
    for p in glob.glob(f"{volume_dir}/* - Schedule.mhtml") + glob.glob(f"{volume_dir}/* - Schedule.html"):
        file_team = re.sub(r"\s*-\s*Schedule\.(mhtml|html)$", "", os.path.basename(p), flags=re.IGNORECASE)
        if file_team.lower() in opponent_full_name.lower() or opponent_short_hint.lower() in file_team.lower():
            return p
    return f"{volume_dir}/{opponent_short_hint} - Schedule.mhtml"  # fall back to the old guess, for the warning message below


### Resolve the opponent's parsed schedule from Cell 4

Cell 4 already scraped and parsed every opponent's own schedule into `team_schedules` this run (live from FastScout, with a local MHTML backup only as a last resort) -- reuse that in-memory result instead of re-deriving a file path and re-parsing HTML from scratch. A live-scraped opponent schedule IS cached to disk too, via `scrape_rendered_html`'s `save_path` in Cell 4 -- this cell just prefers the in-memory `team_schedules` result already parsed THIS run.

In [54]:
# Live-scraped opponent schedules ARE cached to disk too (scrape_rendered_html's save_path writes
# "<Team> - Schedule.html" via _save_scraped_html). This cell just prefers the in-memory team_schedules
# result already parsed THIS run.
# CONFIRMED BUG (fixed here): upcoming_opponent/upcoming_opponent_short can now legitimately be None
# (see the "Identify the upcoming opponent" cell above -- a fully historical run with no next game, or
# one where the next scheduled opponent has no scout report on file). .lower() on either raised
# AttributeError. Skips this lookup the same way the rest of this notebook already treats "no upcoming
# opponent identified" -- opp_team_schedule falls back to None rather than crashing.
if upcoming_opponent is None or upcoming_opponent_short is None:
    opp_team_schedule = None
else:
    opp_team_schedule = next(
        (
            ts for ts in team_schedules
            if not ts.empty and (
                ts["team"].iloc[0].lower() in upcoming_opponent.lower()
                or upcoming_opponent_short.lower() in ts["team"].iloc[0].lower()
            )
        ),
        None,
    )

### Build `opp_schedule`

Prefer the live-scraped/parsed entry already sitting in `team_schedules` (from Cell 4) over a local backup file -- a live scrape never gets written back to disk as "<Team> - Schedule.mhtml", so a file-only lookup would report "not found" even right after a fully successful live scrape.

In [56]:
if opp_team_schedule is not None:
    opp_schedule = pd.DataFrame()
    opp_schedule["date"] = opp_team_schedule["date"]
    opp_schedule["game_date"] = opp_schedule["date"].apply(parse_schedule_date)
    opp_schedule["opponent"] = opp_team_schedule["opponent"]
    opp_schedule["outcome"] = opp_team_schedule["outcome"]
    opp_schedule["team_score"] = opp_team_schedule["team_score"]
    opp_schedule["opponent_score"] = opp_team_schedule["opponent_score"]
    # team_schedules (built in Cell 4 from the live scrape/MHTML) already carries each game's FastScout/Synergy
    # links straight from that row's own <a href> tags -- confirmed by the user: video_url is a direct link to
    # that game's full film on Synergy (e.g. "https://editor-web.synergysports.com/video?playlistUrl=...").
    # Carry it through here (dropped in the manual column selection above) so prev_games below exposes a
    # clickable video link per prior game even when no separately-exported/tagged "_video.mhtml" file exists.
    opp_schedule["video_url"] = opp_team_schedule["video_url"]
    opp_schedule["game_url"] = opp_team_schedule["game_url"]
    # Needed to build an accurate "<Away> @ <Home>" matchup string for live-scrape cache filenames below
    # (mirrors the same location-based matchup naming already used for scout-report downloads in Cell 4).
    opp_schedule["location"] = opp_team_schedule["location"]
elif upcoming_opponent is None or upcoming_opponent_short is None:
    # CONFIRMED BUG (fixed here): _schedule_file_for() calls .lower() on both of these unconditionally --
    # they can now legitimately be None (see the "Identify the upcoming opponent" cell), which raised
    # AttributeError. Same fallback the rest of this notebook already uses for "no upcoming opponent".
    print("No upcoming opponent identified (or no scout report on file for them) -- skipping opponent "
          "schedule parsing and prior-game pbp analysis.")
    opp_schedule = pd.DataFrame(columns=["date", "game_date", "opponent", "outcome", "team_score", "opponent_score"])
else:
    # team_schedules truly has no entry for this opponent (e.g. it wasn't in UWW's own scouted schedule at
    # all this run) -- fall back to the old local "<Team> - Schedule.mhtml" backup file lookup.
    opp_schedule_path = _schedule_file_for(upcoming_opponent, upcoming_opponent_short, volume_dir)
    if not os.path.exists(opp_schedule_path):
        print(f"WARNING: Opponent schedule file not found: {opp_schedule_path}")
        print(f"    Upload '{upcoming_opponent_short} - Schedule.mhtml' to enable full scouting analysis.")
        print(f"    Skipping opponent schedule parsing and prior-game pbp analysis for {upcoming_opponent_short}.")
        opp_schedule = pd.DataFrame(columns=["date", "game_date", "opponent", "outcome", "team_score", "opponent_score"])
    else:
        opp_html = load_html_snapshot(opp_schedule_path)
        opp_schedule_raw = pd.read_html(StringIO(str(BeautifulSoup(opp_html, "lxml").find_all("table")[0])))[0]

        opp_schedule = pd.DataFrame()
        opp_schedule["date"] = opp_schedule_raw["Date"]
        opp_schedule["game_date"] = opp_schedule["date"].apply(parse_schedule_date)
        opp_schedule["opponent"] = opp_schedule_raw["Opponent"].apply(lambda x: split_opponent(x)[0])
        res_split = opp_schedule_raw["Result"].apply(split_result)
        opp_schedule["outcome"] = res_split.apply(lambda x: x[0])
        opp_schedule["team_score"] = res_split.apply(lambda x: x[1])
        opp_schedule["opponent_score"] = res_split.apply(lambda x: x[2])

### Find the opponent's games before they played UW-Whitewater

Also print each game's raw video link and check whether a local `_pbp`/`_video` file already exists for it -- these are the games the next two cells try to fill in via a live scrape when a file is missing.

In [58]:
if opp_schedule.empty:
    prev_games = pd.DataFrame(columns=["date", "game_date", "opponent", "outcome", "team_score", "opponent_score"])
else:
    # Whitewater's own matchup date -- games strictly before that are what to check for existing _pbp/_video
    # files (the opponent's games against teams other than Whitewater). Read this straight from UWW's OWN
    # schedule row (upcoming_game, already resolved above) rather than searching for a "vs Whitewater" row
    # inside the OPPONENT's own schedule -- confirmed by a live run: when opp_schedule comes from
    # team_schedules, that entry was already filtered (in Cell 4) to games strictly before reference_date,
    # which can exclude the Whitewater matchup itself if it falls ON OR AFTER reference_date from the
    # opponent's side (e.g. Elmhurst's Dec 2 game vs a Dec 1 reference_date) -- so searching for it inside
    # opp_schedule can come up empty even though the date is already known independently.
    whitewater_date = parse_schedule_date(upcoming_game["date"], uww_season_start_year)
    if whitewater_date is None:
        raise ValueError(f"Could not parse UW-Whitewater's own matchup date for {upcoming_opponent_short} from {upcoming_game['date']!r}.")

    # CONFIRMED BUG (fixed here): this only ever compared against whitewater_date -- the date UWW itself
    # plays this opponent -- with no consideration of reference_date at all. reference_date represents
    # "today" for the WHOLE scouting exercise, not just for UWW's own games (pbp_events already respects
    # this via its own reference_date filter) -- when reference_date falls BEFORE whitewater_date, which
    # is the normal case (you scout an opponent ahead of actually playing them), this let the opponent's
    # own games between reference_date and whitewater_date leak in as if they'd already happened.
    # Confirmed as the root cause of a real, reported case: reference_date set before Ripon's own season
    # had even started, and the app still showed real scoring-distribution numbers for Ripon -- every
    # opponent-scouting table downstream (pbp_events_upcoming, pbp_box_score_upcoming, the
    # player_profiles/team_totals overrides below, and the opponent-history halves of the Pace & Style /
    # Runs KTV cards) traces back to this one line. The cutoff is now whichever of the two dates is
    # EARLIER: the matchup itself, or "today".
    _prev_games_cutoff = min(whitewater_date, reference_date.date())
    prev_games = opp_schedule[opp_schedule["game_date"] < _prev_games_cutoff].reset_index(drop=True)
    print(f"{upcoming_opponent_short}'s games before facing UW-Whitewater on {whitewater_date}:")
    print(prev_games)
    if "video_url" in prev_games.columns:
        print("\nVideo links for these games (raw full-game film, independent of any manually-tagged _video.mhtml export):")
        for _, _pg_row in prev_games.iterrows():
            # game_date can come through as either datetime.date or datetime.datetime/Timestamp depending on
            # how opp_schedule was sourced -- pd.Timestamp(...) normalizes either into something .date() works
            # on, rather than assuming one specific type (confirmed by a live run: plain .date() raised
            # "'datetime.date' object has no attribute 'date'" here).
            print(f"  {pd.Timestamp(_pg_row['game_date']).date()} vs {_pg_row['opponent']}: {_pg_row['video_url'] or '(no video_url)'}")

    # Check for _pbp and _video files for each previous game. Filenames are "<m>_<d>_<yy> ..." (no leading zeros).
    missing_files = []
    for _, row in prev_games.iterrows():
        # "%-m"/"%-d" (no-leading-zero month/day) are a glibc-only strftime extension -- Windows' CRT
        # strftime rejects them outright with "ValueError: Invalid format string". Since this notebook runs
        # via Databricks Connect from a local Windows machine, build the no-leading-zero month/day manually
        # instead (plain int formatting has no platform-specific behavior), keeping "%y" (a portable, standard
        # strftime directive) for the 2-digit year.
        game_date_str = f"{row['game_date'].month}_{row['game_date'].day}_{row['game_date'].strftime('%y')}"
        # "_pbp.*" (not just "_pbp.mhtml") so this also finds this notebook's own live-scrape ".html" cache
        # (see _save_scraped_html in Cell 4), the same broad wildcard already used for "_video.*" below.
        pbp_pattern = f"{volume_dir}/{game_date_str}*{upcoming_opponent_short}*_pbp.*"
        video_pattern = f"{volume_dir}/{game_date_str}*{upcoming_opponent_short}*_video.*"
        pbp_files = glob.glob(pbp_pattern)
        video_files = glob.glob(video_pattern)
        if not pbp_files or not video_files:
            missing_files.append({
                "date": row["game_date"], "opponent": row["opponent"],
                "pbp_found": bool(pbp_files), "video_found": bool(video_files)
            })

    if missing_files:
        print("\nMissing _pbp or _video files for the following games before the Whitewater matchup:")
        for mf in missing_files:
            print(f"  {mf['date']} vs {mf['opponent']}: pbp_found={mf['pbp_found']}, video_found={mf['video_found']}")
    else:
        print("\nAll required _pbp and _video files found for the upcoming opponent's previous games before Whitewater.")

Aurora Spartans's games before facing UW-Whitewater on 2025-11-19:
          date   game_date                          opponent outcome  \
0   Fri, Nov 7  2025-11-07  Gustavus Adolphus Golden Gusties       L   
1   Sat, Nov 8  2025-11-08          Bethany Lutheran Vikings       W   
2  Wed, Nov 12  2025-11-12      North Central (IL) Cardinals       W   
3  Sat, Nov 15  2025-11-15           Benedictine (IL) Eagles       L   

   team_score  opponent_score  \
0        79.0            84.0   
1        84.0            79.0   
2        65.0            55.0   
3        76.0            87.0   

                                                                                                                                                                video_url  \
0  https://editor-web.synergysports.com/video?playlistUrl=https%3a%2f%2fbasketball.synergysportstech.com%2fapi%2fgames%2f68dc5147e106e7b276af199b%2ffullgamevideo%2fclips   
1  https://editor-web.synergysports.com/video?playlistUrl=ht

### Live-scrape any missing play-by-play files as a fallback

Reuses the one shared FastScout Playwright session from Cell 4 (`run_in_fastscout_session`) rather than opening its own browser+thread.

In [60]:
if not opp_schedule.empty:
    # Determine which raw column ("uww_text" or "opp_text") actually holds THIS opponent's own events for each pbp
    # file -- FastScout puts the exporting team's events in a fixed column regardless of home/away, but WHICH
    # column that is depends on whose account captured the snapshot. Resolve it per file by matching known
    # roster player names instead of assuming.
    known_names = set(player_profiles.loc[player_profiles["opponent"] == upcoming_opponent_short, "name"].dropna())

    def resolve_self_column(raw_df, known_names):
        uww_matches = sum(any(name in str(t) for name in known_names) for t in raw_df["uww_text"].dropna())
        opp_matches = sum(any(name in str(t) for name in known_names) for t in raw_df["opp_text"].dropna())
        if uww_matches or opp_matches:
            return "uww_text" if uww_matches >= opp_matches else "opp_text"

        # Neither column matched a single known name. The usual cause is that known_names is EMPTY --
        # it's built from player_profiles, which is built from scouting reports, so an opponent with no
        # report on file (the normal pre-scout state, and what before_scout="yes" reproduces) has none.
        # The old code returned "uww_text" here purely because 0 >= 0, silently assigning this
        # opponent's events to whichever column happened to be first. Read the table's own header
        # instead -- it names both teams (see parse_pbp_html).
        _rsc_labels = raw_df.attrs.get("column_teams") or {}
        _rsc_target = str(upcoming_opponent_short or "").strip().casefold()
        _rsc_first_word = _rsc_target.split()[0] if _rsc_target else ""
        for _rsc_col in ("uww_text", "opp_text"):
            _rsc_label = str(_rsc_labels.get(_rsc_col) or "").strip().casefold()
            if not _rsc_label:
                continue
            if _rsc_target and (_rsc_target in _rsc_label or _rsc_label in _rsc_target
                                or (_rsc_first_word and _rsc_first_word in _rsc_label)):
                return _rsc_col
        print(f"  WARNING: could not tell which play-by-play column holds {upcoming_opponent_short}'s "
              f"own events -- no roster names to match (no scouting report on file) and the table's "
              f"header columns ({_rsc_labels}) don't name them either. Defaulting to 'uww_text'; if "
              f"this game's numbers look like they belong to the other team, that's why.")
        return "uww_text"

    # Find which prior games are missing a local "_pbp.mhtml" file AND have a usable "game_url" to live-scrape
    # instead (only present when opp_schedule was sourced from team_schedules -- see above).
    games_needing_live_pbp = []
    for _, row in prev_games.iterrows():
        game_date_str = f"{row['game_date'].month}_{row['game_date'].day}_{row['game_date'].strftime('%y')}"
        pbp_pattern = f"{volume_dir}/{game_date_str}*{upcoming_opponent_short}*_pbp.*"
        if not glob.glob(pbp_pattern) and "game_url" in row.index and pd.notna(row.get("game_url")):
            games_needing_live_pbp.append(row)

    # Confirmed by the user: a game's own play-by-play page is reachable via a small path change to its
    # "game_url" (see scrape_pbp_live() in the pbp-parsing cell above) -- live-scrape it for any prior game
    # missing a local "_pbp.mhtml" file, the same live-scrape-with-fallback pattern Cell 4 already uses for
    # schedules and scout reports. One shared session covers every game that needs this, rather than opening
    # a new browser per game.
    live_pbp_by_game = {}
    if games_needing_live_pbp and fastscout_username and fastscout_password:
        # One run_in_fastscout_session call PER GAME (not one call looping over all games) -- run_in_fastscout_
        # session's own retry-on-dead-session logic (Cell 4) can only kick in on a call it directly wraps, so
        # if every game were scraped inside a single call, a mid-batch dead browser/driver would silently fail
        # every remaining game with no chance to self-heal. Reuses the ONE shared FastScout Playwright session
        # (opened lazily on first use) instead of opening its own separate browser+thread per game.
        for g_row in games_needing_live_pbp:
            try:
                # Same location-based "<Away> @ <Home>" matchup naming as scout-report downloads (Cell 4) and
                # the video-clip cache below, so the saved file matches the manually-uploaded naming
                # convention closely enough for the glob patterns above to find it on a future run.
                g_date_str = f"{g_row['game_date'].month}_{g_row['game_date'].day}_{g_row['game_date'].strftime('%y')}"
                if str(g_row.get("location", "")).strip().lower() == "home":
                    matchup = f"{g_row['opponent']} @ {upcoming_opponent_short}"
                else:
                    matchup = f"{upcoming_opponent_short} @ {g_row['opponent']}"
                pbp_save_path = f"{volume_dir}/{g_date_str} {matchup}_pbp.html"
                live_pbp_by_game[g_row["game_date"]] = run_in_fastscout_session(
                    lambda page, url=g_row["game_url"], sp=pbp_save_path: scrape_pbp_live(page, url, save_path=sp)
                )
            except Exception as pbp_scrape_error:
                print(f"  Could not live-scrape pbp for {g_row['opponent']} ({g_row['game_url']}): {type(pbp_scrape_error).__name__}: {pbp_scrape_error}")

else:
    # CONFIRMED BUG (fixed here): known_names is only ever defined inside the branch above -- when
    # opp_schedule is empty (no upcoming opponent identified, or no scout report on file for them), it
    # never gets assigned at all, and a later cell (the shot-tendency breakdown) reading it raised
    # "NameError: name 'known_names' is not defined". A well-formed empty fallback here, matching the
    # pattern already used elsewhere in this notebook (e.g. pbp_events_upcoming's own empty fallback).
    known_names = set()

### Parse play-by-play events for the opponent's prior games

Builds `pbp_events_upcoming` from every local (or just-cached, live-scraped) `_pbp` file found for these prior games.

In [62]:
if opp_schedule.empty:
    pbp_events_upcoming = pd.DataFrame(columns=[
        "opponent", "game_date", "event_order", "period", "time_remaining",
        "time_remaining_seconds", "team", "event_type", "raw_text",
        "uww_score", "opp_score", "player", "shot_type", "shot_desc",
        "turnover_type", "foul_type", "ft_num", "ft_total"
    ])
else:
    # Run play-by-play parsing for the upcoming opponent's previous games
    pbp_events_list = []
    for _, row in prev_games.iterrows():
        # Same Windows strftime portability fix as above -- "%-m"/"%-d" aren't supported by Windows' CRT.
        game_date_str = f"{row['game_date'].month}_{row['game_date'].day}_{row['game_date'].strftime('%y')}"
        pbp_pattern = f"{volume_dir}/{game_date_str}*{upcoming_opponent_short}*_pbp.*"
        pbp_files = glob.glob(pbp_pattern)
        raw_dfs = [parse_pbp_mhtml(path) for path in pbp_files]
        if not raw_dfs and row["game_date"] in live_pbp_by_game:
            raw_dfs = [live_pbp_by_game[row["game_date"]]]
        for raw_df in raw_dfs:
            self_column = resolve_self_column(raw_df, known_names)
            # `self_team`/`self_column` = the upcoming opponent's own events (whichever raw column actually has
            # their players); `opponent` = row["opponent"] -- whoever they actually played in THIS game, NOT
            # literally "UW-Whitewater", since Whitewater isn't in these games at all.
            events = build_pbp_events(
                raw_df, row["opponent"], row["game_date"], self_team=upcoming_opponent_short, self_column=self_column
            )
            pbp_events_list.append(events)

    # CONFIRMED BUG (fixed here): a bare pd.DataFrame() has ZERO columns, unlike the well-formed empty
    # fallback in the "opp_schedule.empty" branch above -- so if prev_games was non-empty but EVERY one of
    # those games failed to produce PBP data (no local _pbp file, and not in live_pbp_by_game either -- e.g.
    # an opponent whose games are so early in the season that this data simply isn't available yet), the
    # resulting pbp_events_upcoming had no "opponent" column at all, and the very next cell's
    # pbp_events_upcoming["opponent"] lookup crashed with KeyError: 'opponent' instead of just being empty.
    _PBP_EVENTS_UPCOMING_COLS = [
        "opponent", "game_date", "event_order", "period", "time_remaining",
        "time_remaining_seconds", "team", "event_type", "raw_text",
        "uww_score", "opp_score", "player", "shot_type", "shot_desc",
        "turnover_type", "foul_type", "ft_num", "ft_total",
    ]
    pbp_events_upcoming = (
        pd.concat(pbp_events_list, ignore_index=True) if pbp_events_list
        else pd.DataFrame(columns=_PBP_EVENTS_UPCOMING_COLS)
    )
    if not pbp_events_list:
        print(f"  No PBP data found/scraped for any of {upcoming_opponent_short}'s {len(prev_games)} game(s) before UWW -- pbp_events_upcoming is empty but well-formed.")
    print(pbp_events_upcoming.head(20))

                            opponent   game_date  event_order period  \
0   Gustavus Adolphus Golden Gusties  2025-11-07            0   None   
1   Gustavus Adolphus Golden Gusties  2025-11-07            1     H1   
2   Gustavus Adolphus Golden Gusties  2025-11-07            2     H1   
3   Gustavus Adolphus Golden Gusties  2025-11-07            3     H1   
4   Gustavus Adolphus Golden Gusties  2025-11-07            4     H1   
5   Gustavus Adolphus Golden Gusties  2025-11-07            5     H1   
6   Gustavus Adolphus Golden Gusties  2025-11-07            6     H1   
7   Gustavus Adolphus Golden Gusties  2025-11-07            7     H1   
8   Gustavus Adolphus Golden Gusties  2025-11-07            8     H1   
9   Gustavus Adolphus Golden Gusties  2025-11-07            9     H1   
10  Gustavus Adolphus Golden Gusties  2025-11-07           10     H1   
11  Gustavus Adolphus Golden Gusties  2025-11-07           11     H1   
12  Gustavus Adolphus Golden Gusties  2025-11-07           12   


### Shared video-tagging helper functions

Hoisted here (rather than defined inline where first used) so cell run-order doesn't matter -- `parse_video_mhtml`, `expand_clip_to_subevents`, `global_align`, `RESULT_TO_KEY`, and `FREE_THROW_EVENT_TYPES` are reused both by the opponent's-own-games attachment below and by the later cell that attaches video tags onto UWW's own `pbp_events`.

In [64]:
# --- Shared video-tagging helper functions, hoisted here so cell run-order doesn't matter -------------------
# Self-contained imports -- this cell is meant to work regardless of run order (see title above), so it
# shouldn't rely on another cell (e.g. the Configuration cell) having already run and left these in the
# global namespace. Confirmed by a live run: this cell raised "NameError: name 're' is not defined" during a
# fresh top-to-bottom run of the whole notebook.
import os
import re
from io import StringIO

import pandas as pd
from bs4 import BeautifulSoup

RESULT_TO_KEY = {
    "Make 2 Pts": "made_shot", "Make 3 Pts": "made_shot",
    "Miss 2 Pts": "missed_shot", "Miss 3 Pts": "missed_shot",
    "Turnover": "turnover",
    "Free Throw": "free_throw",
    "Foul": "foul", "Non Shooting Foul": "foul",
}
AND1_SHOT_RE = re.compile(r"Make (2|3) Pts Foul")
FREE_THROW_EVENT_TYPES = {"free_throw_made", "free_throw_missed"}

# Confirmed-by-video-review manual overrides -- keyed by (opponent_short, video clip "No.") -> the pbp_events
# "event_order" it actually corresponds to.
CONFIRMED_CLIP_EVENT_OVERRIDES = {
    ("Ripon", 120): 271,
    ("Ripon", 164): 371,
}


def expand_clip_to_subevents(row):
    """One video clip can represent more than one real pbp event (an \"And-1\" makes a shot + a free throw) --
    return a list of event_type sub-events (made_shot/missed_shot/turnover/free_throw) for this clip, in the
    order they happened."""
    if row["Result"] in RESULT_TO_KEY:
        return [RESULT_TO_KEY[row["Result"]]]
    if row["Result"] in ("1 Pts", "0 Pts"):
        events = ["made_shot"] if AND1_SHOT_RE.search(row["Description"]) else []
        events.append("free_throw")
        return events
    return []


def global_align(n, m, compatible, pos_cost):
    """pbp side has n items (indices 0..n-1, already sorted by event_order), video side has m items (indices
    0..m-1, already sorted by video_clip_number). Returns {pbp_index: video_index} for the order-preserving
    pairing that maximizes total match count (a huge fixed bonus per match dominates the DP), using summed
    `pos_cost` only as a tiebreaker among otherwise-equally-good maximal alignments."""
    BIG_BONUS = 1000.0
    score = [[0.0] * (m + 1) for _ in range(n + 1)]
    choice = [[0] * (m + 1) for _ in range(n + 1)]  # 0=matched (diagonal), 1=skip pbp item, 2=skip video item
    for i in range(1, n + 1):
        choice[i][0] = 1
    for j in range(1, m + 1):
        choice[0][j] = 2
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            best, best_choice = score[i - 1][j], 1
            if score[i][j - 1] > best:
                best, best_choice = score[i][j - 1], 2
            if compatible(i - 1, j - 1):
                diag = score[i - 1][j - 1] + BIG_BONUS - pos_cost(i - 1, j - 1)
                if diag >= best:  # ties prefer matching over skipping
                    best, best_choice = diag, 0
            score[i][j], choice[i][j] = best, best_choice

    pairs = {}
    i, j = n, m
    while i > 0 and j > 0:
        c = choice[i][j]
        if c == 0:
            pairs[i - 1] = j - 1
            i, j = i - 1, j - 1
        elif c == 1:
            i -= 1
        else:
            j -= 1
    return pairs


def parse_video_mhtml_html(html):
    """Core clip-table parsing logic, factored out of parse_video_mhtml() so it can run on HTML from either
    source: a manually-exported/uploaded MHTML snapshot, OR a live Playwright page.content() capture --
    confirmed by the user: a game's own "video_url" (already available per-game via team_schedules/
    opp_schedule -- see the schedule-scraping cell and the opponent-schedule cell above) renders this EXACT
    same clip-tagging table when opened, so a live scrape of that URL is a drop-in replacement for the
    manual "_video.mhtml" export/upload step. See scrape_video_clips_live() below for the live-scrape path.
    """
    soup = BeautifulSoup(html, "lxml")
    parsed_tables = [pd.read_html(StringIO(str(t)))[0] for t in soup.find_all("table")]
    # The file also contains a saved "Edit"/playlist metadata table with no "Description" column -- skip it.
    clips = next(t for t in parsed_tables if "Description" in t.columns)
    clips = clips.copy()
    clips["player"] = clips["Player"].astype(str).str.strip()
    return clips[["No.", "Result", "Description", "player", "Team", "Duration"]]


def parse_video_mhtml(path):
    return parse_video_mhtml_html(load_html_snapshot(path))


def login_to_synergy(page, username, password, timeout_ms=30000):
    """Log into Synergy Sports Tech's own identity provider (auth.synergysportstech.com) -- a separate login
    from FastScout/Hudl with its OWN credentials (see synergy_username/synergy_password in the
    schedule-scraping cell above; confirmed NOT the same as FastScout's). Selectors are best-effort guesses
    at a single-page username+password form; errors include the actual url/visible-error-text to help
    correct them if needed.
    """
    def _page_error_text():
        for sel in ["[class*='error' i]", "[class*='alert' i]", "[role='alert']"]:
            try:
                text = page.locator(sel).first.inner_text(timeout=1000)
                if text and text.strip():
                    return text.strip()
            except Exception:
                continue
        return None

    # Diagnostic: save the pristine login page HTML so its real form markup can be inspected directly if a
    # selector below turns out to be wrong, instead of guessing blindly from a bare timeout.
    try:
        _save_scraped_html(
            page.content(), os.path.join(schedules_dir, "Synergy - Login Page (diagnostic).html"),
            "Synergy login page (diagnostic)",
        )
    except Exception:
        pass

    try:
        username_input = page.locator(
            'input[type="email"], input[name="Username"], input[name="username"], input[id*="username" i], '
            'input[id*="email" i]'
        ).first
        username_input.wait_for(timeout=timeout_ms)
        # Log which field actually got matched (name/id/type only -- never the credential value itself) so a
        # wrong-field match is distinguishable from a genuine server-side credential rejection.
        print(f"    [synergy-login] username field matched: name={username_input.get_attribute('name')!r} id={username_input.get_attribute('id')!r} type={username_input.get_attribute('type')!r}")
        username_input.fill(username)
    except Exception as e:
        error_text = _page_error_text()
        raise RuntimeError(
            f"Synergy login failed finding the username field (url={page.url}): {type(e).__name__}: {e}"
            + (f" -- page showed: {error_text!r}" if error_text else "")
        ) from e

    try:
        password_input = page.locator('input[type="password"]').first
        password_input.wait_for(timeout=timeout_ms)
        print(f"    [synergy-login] password field matched: name={password_input.get_attribute('name')!r} id={password_input.get_attribute('id')!r}")
        password_input.fill(password)
    except Exception as e:
        error_text = _page_error_text()
        raise RuntimeError(
            f"Synergy login failed finding the password field (url={page.url}): {type(e).__name__}: {e}"
            + (f" -- page showed: {error_text!r}" if error_text else "")
        ) from e

    try:
        page.get_by_role("button", name=re.compile(r"^(log ?in|sign ?in|continue|submit)$", re.IGNORECASE)).first.click(timeout=timeout_ms)
    except Exception as e:
        error_text = _page_error_text()
        raise RuntimeError(
            f"Synergy login failed clicking the submit button (url={page.url}): {type(e).__name__}: {e}"
            + (f" -- page showed: {error_text!r}" if error_text else "")
        ) from e

    try:
        page.wait_for_url(lambda url: "/Account/Login" not in url, timeout=timeout_ms)
    except Exception as e:
        error_text = _page_error_text()
        raise RuntimeError(
            f"Synergy login did not leave the login page after submitting credentials (still at "
            f"url={page.url}): {type(e).__name__}: {e}" + (f" -- page showed: {error_text!r}" if error_text else "")
        ) from e


def scrape_video_clips_live(page, video_url, timeout_ms=30000, save_path=None):
    """Navigate to a game's own FastScout "video_url" (the Synergy-embedded clip-tagging player) and parse
    its rendered clip table -- a live-scrape replacement for a manually-exported "_video.mhtml" snapshot.
    Expects an already-authenticated FastScout Playwright `page`. If save_path is given, caches the scraped
    HTML there (see _save_scraped_html) so a future run can fall back to it via parse_video_mhtml instead of
    live-scraping again.

    Navigating here can redirect to Synergy's own login (auth.synergysportstech.com), a separate identity
    provider from FastScout/Hudl -- _attempt() detects that and logs in via login_to_synergy using Synergy's
    own synergy_username/synergy_password before retrying. Media-blocking (_attempt(block_media=True)/False)
    is a separate defensive measure against the video-editor SPA buffering full-game video in the background.
    """
    def _attempt(block_media):
        handler = None
        if block_media:
            def handler(route):
                req = route.request
                if req.resource_type == "media" or re.search(r"\.(mp4|m3u8|ts|webm|mov)(\?|$)", req.url, re.IGNORECASE):
                    route.abort()
                else:
                    route.continue_()
            page.route("**/*", handler)
        try:
            page.goto(video_url, wait_until="domcontentloaded", timeout=timeout_ms)
            try:
                page.wait_for_selector("table", timeout=timeout_ms)
            except Exception:
                if "auth.synergysportstech.com" not in page.url:
                    raise
                print(f"    [video] Redirected to Synergy's own login ({page.url}) -- logging in with Synergy's own credentials and retrying.")
                login_to_synergy(page, synergy_username, synergy_password, timeout_ms=timeout_ms)
                page.goto(video_url, wait_until="domcontentloaded", timeout=timeout_ms)
                page.wait_for_selector("table", timeout=timeout_ms)
            return page.content()
        finally:
            if handler is not None:
                page.unroute("**/*", handler)

    try:
        html = _attempt(block_media=True)
    except Exception as block_error:
        # Only reached if the media-blocked attempt itself failed -- retry once, fully unblocked, rather than
        # let a media-blocking regression silently break every video-clip scrape. If the underlying browser/
        # driver is actually dead (the connection-closed case this function was written to avoid), this
        # unblocked retry will fail the same way and its exception still propagates up to
        # run_in_fastscout_session's own dead-session retry logic, same as before this fallback existed.
        print(f"    [video] Media-blocked scrape failed ({type(block_error).__name__}: {block_error}) -- retrying without blocking media.")
        html = _attempt(block_media=False)

    if save_path:
        _save_scraped_html(html, save_path, "scraped video-tagging clips")
    return parse_video_mhtml_html(html)


### Attach video-tagging clip descriptions onto the opponent's prior-game events

Reuses the helpers from the cell above to align each `_video` export's clip descriptions onto `pbp_events_upcoming`. Only the per-game team resolution differs from the later UWW version: these are the opponent's own games against a varying actual opponent each time, not always UW-Whitewater vs. them.

In [66]:
# --- Attach video-tagging clip descriptions onto pbp_events_upcoming, same method as pbp_events uses --------
# Reuses parse_video_mhtml / expand_clip_to_subevents / global_align / RESULT_TO_KEY / FREE_THROW_EVENT_TYPES
# from the cell above. Only the per-game team resolution differs here: these are the opponent's OWN games
# against a VARYING actual opponent each time, not always UW-Whitewater vs a fixed opponent, and there's no
# lineup reconstruction for these games.
video_desc_col = pd.Series(index=pbp_events_upcoming.index, dtype=object)
video_result_col = pd.Series(index=pbp_events_upcoming.index, dtype=object)
video_player_col = pd.Series(index=pbp_events_upcoming.index, dtype=object)
video_clip_number_col = pd.Series(index=pbp_events_upcoming.index, dtype=float)

# Same Windows strftime portability fix used in the opponent-schedule cell above -- "%-m"/"%-d" (no-leading-
# zero month/day) aren't supported by Windows' CRT strftime.
def _game_date_str(game_date):
    return f"{game_date.month}_{game_date.day}_{game_date.strftime('%y')}"


# Find which prior games are missing a local "_video.*" file AND have a usable "video_url" to live-scrape
# instead (only present when opp_schedule was sourced from team_schedules -- see the opponent-schedule cell).
games_needing_live_video = []
for _, row in prev_games.iterrows():
    video_matches = glob.glob(f"{volume_dir}/{_game_date_str(row['game_date'])}*{upcoming_opponent_short}*_video.*")
    if not video_matches and "video_url" in row.index and pd.notna(row.get("video_url")):
        games_needing_live_video.append(row)

# Confirmed by the user: a game's own video-clip tagging table renders directly at its "video_url" (see
# scrape_video_clips_live() in the video-tagging helpers cell above) -- live-scrape it for any prior game
# missing a local "_video.mhtml" file, the same live-scrape-with-fallback pattern Cell 4 already uses for
# schedules and scout reports. One shared session covers every game that needs this.
live_clips_by_game = {}
if games_needing_live_video and fastscout_username and fastscout_password:
    # One run_in_fastscout_session call PER GAME (not one call looping over all games) -- run_in_fastscout_
    # session's own retry-on-dead-session logic (Cell 4) can only kick in on a call it directly wraps, so if
    # every game were scraped inside a single call, a mid-batch dead browser/driver would silently fail every
    # remaining game with no chance to self-heal. Reuses the ONE shared FastScout Playwright session (opened
    # lazily on first use) instead of opening its own separate browser+thread per game.
    for g_row in games_needing_live_video:
        try:
            # Same location-based "<Away> @ <Home>" matchup naming as scout-report downloads (Cell 4) and
            # the pbp cache above, so the saved file matches the manually-uploaded naming convention closely
            # enough for the glob pattern above to find it on a future run.
            g_date_str = _game_date_str(g_row["game_date"])
            if str(g_row.get("location", "")).strip().lower() == "home":
                matchup = f"{g_row['opponent']} @ {upcoming_opponent_short}"
            else:
                matchup = f"{upcoming_opponent_short} @ {g_row['opponent']}"
            video_save_path = f"{volume_dir}/{g_date_str} {matchup}_video.html"
            live_clips_by_game[g_row["game_date"]] = run_in_fastscout_session(
                lambda page, url=g_row["video_url"], sp=video_save_path: scrape_video_clips_live(page, url, save_path=sp)
            )
        except Exception as video_scrape_error:
            print(f"  Could not live-scrape video clips for {g_row['opponent']} ({g_row['video_url']}): {type(video_scrape_error).__name__}: {video_scrape_error}")

games_with_video = []
# Per-game accounting for why a prior game ends up with no tagged shots. Every skip below used to be a bare
# `continue`, so a game that produced nothing looked identical to a game that was never on the schedule --
# and the app could only report how many games DID have tagged video, never which ones didn't or why.
video_status_rows = []
for _, row in prev_games.iterrows():
    video_matches = glob.glob(f"{volume_dir}/{_game_date_str(row['game_date'])}*{upcoming_opponent_short}*_video.*")
    actual_opponent = row["opponent"]
    game_mask = (pbp_events_upcoming["opponent"] == actual_opponent) & (pbp_events_upcoming["game_date"] == row["game_date"])
    _vs_row = {
        "game_date": row["game_date"], "vs": actual_opponent,
        "video_file": os.path.basename(video_matches[0]) if video_matches else ("(live-scraped)" if row["game_date"] in live_clips_by_game else ""),
        "pbp_rows": int(game_mask.sum()), "clips": 0, "matched": 0, "why": "",
    }
    if video_matches:
        clips = parse_video_mhtml(video_matches[0])
    elif row["game_date"] in live_clips_by_game:
        clips = live_clips_by_game[row["game_date"]]
    else:
        # No local "<m>_<d>_<yy>*<Opponent>*_video.*" file, and either no video_url on the schedule row or
        # the live scrape failed. Note that the glob needs BOTH the no-leading-zero date and the opponent's
        # short name in the filename -- "01_10_26 ..." or a file named with their full name won't be found.
        _vs_row["why"] = "no _video file found and no live scrape"
        video_status_rows.append(_vs_row)
        continue
    if _vs_row["pbp_rows"] == 0:
        # Clips exist but there are no play-by-play rows to hang them on: the _pbp file is missing for this
        # game, or its opponent string doesn't match this schedule row's spelling.
        _vs_row["clips"] = len(clips)
        _vs_row["why"] = "no pbp events for this game (missing _pbp file, or opponent name mismatch)"
        video_status_rows.append(_vs_row)
        continue
    total_clips_n = len(clips)

    opp_all_events = pbp_events_upcoming[game_mask]
    player_team_lookup = opp_all_events.dropna(subset=["player"]).drop_duplicates("player").set_index("player")["team"]
    known_teams = set(player_team_lookup.unique())
    known_players = set(player_team_lookup.index)

    def normalize_player_name(name):
        if name in known_players:
            return name
        for p in known_players:
            if p.casefold() == str(name).casefold():
                return p
        return name

    def committing_team_for(fouled_player):
        fouled_team = player_team_lookup.get(fouled_player)
        others = known_teams - {fouled_team}
        return next(iter(others)) if len(others) == 1 else None

    subevent_rows = []
    for _, clip_row in clips.iterrows():
        clip_player = normalize_player_name(clip_row["player"])
        for event_type in expand_clip_to_subevents(clip_row):
            match_key = committing_team_for(clip_player) if event_type == "foul" else clip_player
            subevent_rows.append({
                "match_key": match_key, "event_type": event_type,
                "Description": clip_row["Description"], "video_result": clip_row["Result"],
                "video_clip_player": clip_player, "video_clip_number": clip_row["No."],
            })
    matchable_clips = pd.DataFrame(
        subevent_rows, columns=["match_key", "event_type", "Description", "video_result", "video_clip_player", "video_clip_number"]
    ).sort_values("video_clip_number").reset_index(drop=True)

    target_event_types = {"made_shot", "missed_shot", "turnover", "foul"} | FREE_THROW_EVENT_TYPES
    opp_events = pbp_events_upcoming[game_mask & pbp_events_upcoming["event_type"].isin(target_event_types)].sort_values("event_order").copy()
    opp_events["match_event_type"] = opp_events["event_type"].where(~opp_events["event_type"].isin(FREE_THROW_EVENT_TYPES), "free_throw")
    opp_events["match_key"] = opp_events["team"].where(opp_events["event_type"] == "foul", opp_events["player"])
    total_pbp_n = pbp_events_upcoming.loc[game_mask, "event_order"].max()

    pbp_orig_index = opp_events.index.tolist()
    pbp_list = [
        {"event_order": r["event_order"], "match_event_type": r["match_event_type"], "match_key": r["match_key"]}
        for _, r in opp_events.iterrows()
    ]
    video_list = matchable_clips.to_dict("records")
    n, m = len(pbp_list), len(video_list)

    def compatible(i, j):
        p, v = pbp_list[i], video_list[j]
        return p["match_event_type"] == v["event_type"] and p["match_key"] == v["match_key"]

    def pos_cost(i, j):
        return abs(pbp_list[i]["event_order"] / total_pbp_n - video_list[j]["video_clip_number"] / total_clips_n)

    pairs = global_align(n, m, compatible, pos_cost)
    for i, j in pairs.items():
        orig_idx = pbp_orig_index[i]
        v = video_list[j]
        video_desc_col.loc[orig_idx] = v["Description"]
        video_result_col.loc[orig_idx] = v["video_result"]
        video_player_col.loc[orig_idx] = v["video_clip_player"]
        video_clip_number_col.loc[orig_idx] = v["video_clip_number"]

    games_with_video.append(actual_opponent)
    _vs_row.update({"clips": total_clips_n, "matched": len(pairs),
                    "why": "" if pairs else "clips found but none aligned to a pbp event"})
    video_status_rows.append(_vs_row)
    print(f"  {actual_opponent}: matched {len(pairs)}/{len(pbp_list)} eligible pbp events to {len(video_list)} video sub-events ({total_clips_n} clips)")

pbp_events_upcoming["video_description"] = video_desc_col
pbp_events_upcoming["video_result"] = video_result_col
pbp_events_upcoming["video_player"] = video_player_col
pbp_events_upcoming["video_clip_number"] = video_clip_number_col

# One table answering "which of their prior games actually contributed tagged shots, and why not the rest".
video_status = pd.DataFrame(video_status_rows)
if not video_status.empty:
    _n_ok = int((video_status["matched"] > 0).sum())
    print(f"\nTagged-video coverage for {upcoming_opponent_short}: {_n_ok} of {len(prev_games)} game(s) "
          f"before facing UWW contributed tagged shots.")
    print(video_status.to_string(index=False))
    _gaps = video_status[video_status["why"] != ""]
    if not _gaps.empty:
        print(f"\n{len(_gaps)} game(s) contributed nothing -- reasons above. The most common cause is a "
              f"filename the glob can't see: it needs BOTH the no-leading-zero date "
              f"(\"1_10_26\", not \"01_10_26\") and \"{upcoming_opponent_short}\" in the name, "
              f"e.g. \"1_10_26 {upcoming_opponent_short} @ Coe_video.mhtml\".")

print(f"\nGames with a video-tagging file: {games_with_video}")
print(f"pbp_events_upcoming rows with a matched video description: {pbp_events_upcoming['video_description'].notna().sum()} of {len(pbp_events_upcoming)}")
if not pbp_events_upcoming.empty:
    print(pbp_events_upcoming[pbp_events_upcoming["video_description"].notna()].head(20))
else:
    print("WARNING: No opponent prior-game events to display (opponent schedule file not loaded).")

  Gustavus Adolphus Golden Gusties: matched 208/236 eligible pbp events to 221 video sub-events (230 clips)
  Bethany Lutheran Vikings: matched 200/215 eligible pbp events to 208 video sub-events (208 clips)
  North Central (IL) Cardinals: matched 190/205 eligible pbp events to 196 video sub-events (208 clips)
  Benedictine (IL) Eagles: matched 266/286 eligible pbp events to 270 video sub-events (284 clips)

Tagged-video coverage for Aurora Spartans: 4 of 4 game(s) before facing UWW contributed tagged shots.
 game_date                               vs                                                            video_file  pbp_rows  clips  matched why
2025-11-07 Gustavus Adolphus Golden Gusties 11_7_25 Gustavus Adolphus Golden Gusties @ Aurora Spartans_video.html       549    230      208    
2025-11-08         Bethany Lutheran Vikings         11_8_25 Bethany Lutheran Vikings @ Aurora Spartans_video.html       532    208      200    
2025-11-12     North Central (IL) Cardinals    11_12_2


### Scout the upcoming opponent's own tendencies from their prior games

Splits `pbp_events_upcoming` into the opponent's own events vs. their opponents' events, then summarizes shot-type volume/efficiency by play type and contest level -- the opponent's own tendencies heading into the Whitewater game.

In [68]:
# --- Scout the upcoming opponent's own tendencies from their games before facing Whitewater -------------------
_safe_display = lambda df: print(df) if not df.empty else print("  (no data)")

elmhurst_events = pbp_events_upcoming[pbp_events_upcoming["team"] == upcoming_opponent_short].copy()
opponent_events = pbp_events_upcoming[
    pbp_events_upcoming["team"].notna() & (pbp_events_upcoming["team"] != upcoming_opponent_short)
].copy()

print(f"{upcoming_opponent_short}'s own events across their {prev_games.shape[0]} games before Whitewater: {len(elmhurst_events)}")
print(f"Their opponents' events across those same games: {len(opponent_events)}\n")

def normalize_elmhurst_player(name):
    if name in known_names:
        return name
    for p in known_names:
        if p.casefold() == str(name).casefold():
            return p
    return name

def extract_play_segment(description, player):
    if pd.isna(description) or pd.isna(player):
        return None
    segments = [s.strip() for s in description.split(" > ")]
    player_norm = normalize_elmhurst_player(player)
    last_player_idx = None
    for idx, seg in enumerate(segments):
        m = re.match(r"^\d+\s+(.+)$", seg)
        if m and normalize_elmhurst_player(m.group(1)) == player_norm:
            last_player_idx = idx
    if last_player_idx is not None and last_player_idx + 1 < len(segments):
        return segments[last_player_idx + 1]
    return segments[1] if len(segments) > 1 else None

def extract_guarded(description):
    if pd.isna(description):
        return None
    if "Guarded" in description:
        return "Yes"
    if "Open" in description:
        return "No"
    return "N/A"

shots = elmhurst_events[elmhurst_events["event_type"].isin(["made_shot", "missed_shot"])].copy()
shots["made"] = shots["event_type"] == "made_shot"
shots["play_type"] = shots.apply(lambda r: extract_play_segment(r["video_description"], r["player"]), axis=1)
shots["guarded"] = shots["video_description"].apply(extract_guarded)
shot_profile = (
    shots.groupby(["shot_type", "shot_desc", "play_type", "guarded"])
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
shot_profile["fg_pct"] = (shot_profile["makes"] / shot_profile["attempts"] * 100).round(1)
shot_profile = shot_profile.sort_values("attempts", ascending=False).reset_index(drop=True)
print(f"{upcoming_opponent_short}'s shot-type tendencies (volume + efficiency, tagged by play_type/guarded) across their last {prev_games.shape[0]} games:")
_safe_display(shot_profile)

play_type_tendency = shots.groupby("play_type").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
play_type_tendency["fg_pct"] = (play_type_tendency["makes"] / play_type_tendency["attempts"] * 100).round(1)
play_type_tendency = play_type_tendency.sort_values("attempts", ascending=False).reset_index(drop=True)
print(f"\n{upcoming_opponent_short}'s shot attempts by play type:")
_safe_display(play_type_tendency)

guarded_tendency = shots.groupby("guarded").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
guarded_tendency["fg_pct"] = (guarded_tendency["makes"] / guarded_tendency["attempts"] * 100).round(1)
guarded_tendency = guarded_tendency.sort_values("attempts", ascending=False).reset_index(drop=True)
print(f"\n{upcoming_opponent_short}'s shot attempts by contest level:")
_safe_display(guarded_tendency)

turnovers = elmhurst_events[elmhurst_events["event_type"] == "turnover"].copy()
turnovers["play_type"] = turnovers.apply(lambda r: extract_play_segment(r["video_description"], r["player"]), axis=1)
turnover_profile = (
    turnovers.groupby(["turnover_type", "play_type"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
print(f"\n{upcoming_opponent_short}'s turnover types ({len(turnovers)} total, tagged by play_type):")
_safe_display(turnover_profile)

fouls = elmhurst_events[elmhurst_events["event_type"] == "foul"]
foul_profile = (
    fouls.groupby(["foul_type", "video_description"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
print(f"\n{upcoming_opponent_short}'s foul types ({len(fouls)} total, with video_description):")
_safe_display(foul_profile)

made_shots = elmhurst_events[elmhurst_events["event_type"] == "made_shot"].copy()
made_shots["points"] = made_shots["shot_type"].astype(float)
made_fts = elmhurst_events[elmhurst_events["event_type"] == "free_throw_made"].copy()
made_fts["points"] = 1.0
scoring = pd.concat([made_shots[["player", "points"]], made_fts[["player", "points"]]], ignore_index=True)
top_scorers = scoring.groupby("player")["points"].sum().sort_values(ascending=False).reset_index()
top_scorers.columns = ["player", f"total_points_across_{prev_games.shape[0]}_games"]
print(f"\n{upcoming_opponent_short}'s top scorers across their last {prev_games.shape[0]} games (from play-by-play):")
_safe_display(top_scorers.head(10))

Aurora Spartans's own events across their 4 games before Whitewater: 1117
Their opponents' events across those same games: 1082

Aurora Spartans's shot-type tendencies (volume + efficiency, tagged by play_type/guarded) across their last 4 games:
   shot_type                shot_desc          play_type guarded  attempts  \
0          3                Jump Shot            Spot-Up     Yes        30   
1          3                Jump Shot            Spot-Up      No        23   
2          2                    Layup                Cut     N/A        13   
3          2                    Layup            Post-Up     N/A        11   
4          2                    Layup         Transition     N/A        11   
5          2                    Layup                ISO     N/A         9   
6          2            Driving Layup         Transition     N/A         9   
7          2                Jump Shot            Spot-Up     N/A         6   
8          2                Jump Shot   P&R Ball Han


### Compare PBP-derived opponent tendencies to UWW's own scouting keys

Cross-checks the tendencies just derived from actual play-by-play against UWW's own scouting-report notes (`TEAM STRENGTHS` / `KEYS TO VICTORY`) for this opponent, to see whether the scouted keys hold up against what they've actually done on the floor.

In [70]:
# --- Compare our PBP-derived findings to UWW's own scouting keys for the upcoming opponent ---------------------
_safe_display = lambda df: print(df) if not df.empty else print("  (no data)")

elmhurst_plan = all_game_plans[
    (all_game_plans["opponent"] == upcoming_opponent_short) & (all_game_plans["topic"].isin(["TEAM STRENGTHS", "KEYS TO VICTORY"]))
]
print(f"UWW's own scouting notes for {upcoming_opponent_short} [SOURCE: SCOUTING REPORT]:")
for _, r in elmhurst_plan.iterrows():
    print(f"  {r['topic']}: {r['notes']}")

# Rebounding split, for the "DOMINATE THE PAINT" key -- covers both individual and team-level rebound events.
rebounds = elmhurst_events[elmhurst_events["event_type"].str.contains("rebound", na=False)]
reb_split = rebounds["event_type"].apply(lambda x: "offensive" if "offensive" in x else "defensive").value_counts().reset_index()
reb_split.columns = ["rebound_type", "count"]

# Blocks recorded AGAINST them by their opponents -- another paint-imposition signal.
blocks_against = opponent_events[opponent_events["event_type"] == "block"]

paint_attempts = shot_profile.loc[shot_profile["shot_desc"].str.contains("Layup", na=False), "attempts"].sum()
three_pt = shot_profile[shot_profile["shot_type"] == "3"]
three_pt_attempts, three_pt_makes = three_pt["attempts"].sum(), three_pt["makes"].sum()
measured_3p_pct = round(three_pt_makes / three_pt_attempts * 100, 1) if three_pt_attempts else None

print("\n--- Comparing PBP evidence to the stated Keys to Victory [SOURCE: SCOUTING REPORT] ---\n")

print("KEY 1 [SCOUTING REPORT] -- COMMUNICATE SCREENS & ACTIONS:")
print("  Not directly measurable from play-by-play event types (no screen-action tagging) -- this is a")
print("  communication/technique key tied to their offensive scheme (per the scout's own Offensive Scheme notes),")
print("  not something the event log alone can confirm or refute.\n")

print("KEY 2 [SCOUTING REPORT] -- DOMINATE THE PAINT:")
print(f"  Their own rebounding split across their {prev_games.shape[0]} games: {reb_split.to_dict('records')}")
print(f"  Their layup-area attempts (Layup + Driving Layup): {paint_attempts} of {shot_profile['attempts'].sum()} total FGA")
print(f"  Blocks recorded against them by opponents: {len(blocks_against)}")
print("  Compare against their own Defensive Scheme notes and layup-area conversion rate to judge whether this key")
print("  is supported by the evidence.\n")

print("KEY 3 [SCOUTING REPORT] -- GUARD 1 ON 1:")
print(f"  Measured 3PT jump-shot rate: {three_pt_attempts} attempts at {measured_3p_pct}%.")
print(f"  Scoring balance (top 5 scorers): {top_scorers.head(5).to_dict('records')}")
print("  Compare against the scout's own TEAM STRENGTHS note on shooting/scoring balance to judge whether")
print("  over-helping creates open catch-and-shoot looks that straight man coverage would limit.")

# --- New Keys to Victory, derived directly from PBP data/tendencies rather than the written scouting report --
# CONFIRMED BUG (fixed here): DATA-KEY 1's recommendation was a hardcoded sentence -- "load up P&R roll coverage
# and transition defense" -- printed no matter WHICH actions came out on top. On Aurora it sat under
# "ISO: 60.0% on 20; Cut: 55.0% on 20", neither of which is a P&R roll or transition, so the evidence and the
# instruction on the same key contradicted each other. Each recommendation is now built from the actions the
# data actually picked, via _PT_DEFENSIVE_CALL below.
#
# CONFIRMED BUG (fixed here): DATA-KEY 3 ranked turnover triggers by RAW COUNT. Spot-Up is also their most-used
# action (79 shots), so it tops the turnover list by volume alone -- which then contradicted DATA-KEY 2 telling
# us to let them keep running it. Now ranked by turnover RATE per use of the action (TO / (shots + TO)), with a
# volume floor, so "trigger" means an action that is actually loose rather than one that is merely common.
#
# CONFIRMED BUG (fixed here): "No Play Type" -- the tagger's own placeholder for an untagged clip -- was being
# ranked as if it were an action ("No Play Type: 7 of 52 turnovers"). A coach can't scheme against it.
# Placeholder labels are dropped from every ranking below.
#
# CONFIRMED BUG (fixed here): "Both well above their overall clip" was asserted without checking. An action
# now has to beat (or trail) the overall FG% by _PT_MIN_GAP points to be called out at all.
print("\n\n--- New Keys to Victory, derived from PBP data/tendencies [SOURCE: PBP-DERIVED] ---\n")

_PT_PLACEHOLDERS = {"", "nan", "none", "no play type", "unknown", "n/a", "not tagged"}
_PT_MIN_ATTEMPTS = 15   # shots on an action before its FG% is worth a key
_PT_MIN_USES_FOR_TO = 20  # shots + turnovers on an action before its turnover RATE is worth a key
_PT_MIN_GAP = 5.0       # FG% points above/below their overall clip before an action is "well" above/below

# What to actually DO against each action. Keyed on lower-cased substrings of the tagger's play-type labels,
# checked in order, so "P&R Roll Man" hits the roll-man entry before the generic "p&r" one.
_PT_DEFENSIVE_CALL = [
    ("roll man", "tag the roll man early from the weak side and don't give up the pocket pass"),
    ("ball handler", "pick one ball-screen coverage and stay in it -- keep the handler out of the middle"),
    ("p&r", "pick one ball-screen coverage and stay in it -- keep the handler out of the middle"),
    ("iso", "keep a gap, show help at the level of the ball and make him score over a crowd"),
    ("isolation", "keep a gap, show help at the level of the ball and make him score over a crowd"),
    ("cut", "jump to the ball on every pass and see man and ball -- no back-cuts behind ball-watchers"),
    ("transition", "sprint back, stop the ball first and match up second"),
    ("post", "front or three-quarter the post and send a timed dig from the passer"),
    ("off screen", "lock and trail off the screen and switch nothing we haven't called"),
    ("hand off", "get into the handoff and force it away from the middle"),
    ("handoff", "get into the handoff and force it away from the middle"),
    ("spot", "close out short and under control -- contest without flying by"),
    ("put back", "hit a body on every shot before going to the ball"),
    ("offensive rebound", "hit a body on every shot before going to the ball"),
]


def _pt_is_placeholder(label):
    return str(label).strip().lower() in _PT_PLACEHOLDERS


def _pt_call(label):
    low = str(label).lower()
    for needle, call in _PT_DEFENSIVE_CALL:
        if needle in low:
            return call
    return f"make {label} a named item in the defensive walkthrough"


overall_fg_pct = round(shots["made"].mean() * 100, 1)
_pt_named = play_type_tendency[~play_type_tendency["play_type"].apply(_pt_is_placeholder)]
high_volume_types = _pt_named[_pt_named["attempts"] >= _PT_MIN_ATTEMPTS]

most_efficient = (high_volume_types[high_volume_types["fg_pct"] >= overall_fg_pct + _PT_MIN_GAP]
                  .sort_values("fg_pct", ascending=False).head(2))
least_efficient = (high_volume_types[high_volume_types["fg_pct"] <= overall_fg_pct - _PT_MIN_GAP]
                   .sort_values("fg_pct").head(2))

# Turnover RATE per use of each action. A use is a shot or a turnover on that action.
_to_named = turnovers[~turnovers["play_type"].apply(_pt_is_placeholder)]
_to_counts = _to_named.groupby("play_type").size().rename("turnovers")
turnover_rates = (play_type_tendency.set_index("play_type")[["attempts"]]
                  .join(_to_counts, how="outer").fillna(0))
turnover_rates = turnover_rates[~turnover_rates.index.to_series().apply(_pt_is_placeholder)]
turnover_rates["uses"] = turnover_rates["attempts"] + turnover_rates["turnovers"]
turnover_rates["to_rate"] = (100 * turnover_rates["turnovers"]
                             / turnover_rates["uses"].replace(0, float("nan"))).round(1)
_team_uses = float(turnover_rates["uses"].sum()) or 1.0
team_to_rate = round(100 * float(turnover_rates["turnovers"].sum()) / _team_uses, 1)
top_turnover_triggers = (turnover_rates[(turnover_rates["uses"] >= _PT_MIN_USES_FOR_TO)
                                        & (turnover_rates["to_rate"] > team_to_rate)]
                         .sort_values("to_rate", ascending=False).head(2))


def _pt_list(frame):
    return "; ".join(f"{r['play_type']}: {r['fg_pct']}% on {int(r['attempts'])} attempts"
                     for _, r in frame.iterrows())


def _pt_calls(frame):
    return " ".join(f"{r['play_type']}: {_pt_call(r['play_type'])}." for _, r in frame.iterrows())


_derived = []
if not most_efficient.empty:
    _derived.append({
        "title": "Take away their most efficient high-volume actions",
        "supporting_stats": _pt_list(most_efficient),
        "recommendation": (f"Each is {_PT_MIN_GAP:.0f}+ points above their {overall_fg_pct}% overall clip on "
                           f"{_PT_MIN_ATTEMPTS}+ attempts. " + _pt_calls(most_efficient)),
    })
if not least_efficient.empty:
    _derived.append({
        "title": "Funnel them into their worst high-volume looks",
        "supporting_stats": _pt_list(least_efficient),
        "recommendation": (f"Each is {_PT_MIN_GAP:.0f}+ points below their {overall_fg_pct}% overall clip -- "
                           "don't help off these actions; make them keep taking them."),
    })
if not top_turnover_triggers.empty:
    _derived.append({
        "title": "Pressure the actions they turn it over on",
        "supporting_stats": "; ".join(
            f"{pt}: {int(r['turnovers'])} TO in {int(r['uses'])} uses ({r['to_rate']}% vs {team_to_rate}% "
            f"across all actions)" for pt, r in top_turnover_triggers.iterrows()),
        "recommendation": ("Loosest actions per use, not the most common ones -- load ball pressure and "
                           "gap help onto these specifically rather than pressing everything."),
    })

for _i, _k in enumerate(_derived, start=1):
    print(f"DATA-KEY {_i} [PBP-DERIVED] -- {_k['title'].upper()}:")
    print(f"  {_k['supporting_stats']}")
    print(f"  {_k['recommendation']}\n")
if not _derived:
    print("  No action cleared the volume and gap floors -- no PBP-derived keys this week.")

# One row per key, tagged with an explicit `source` so downstream consumers can tell these from the written
# report's keys. Keyed by `opponent` so future opponents' keys can append to the same table.
pbp_derived_keys = pd.DataFrame(
    [{"opponent": upcoming_opponent_short, "key_number": _i, "source": "PBP-DERIVED", **_k}
     for _i, _k in enumerate(_derived, start=1)],
    columns=["opponent", "key_number", "title", "source", "supporting_stats", "recommendation"],
)
print("\nStructured PBP-derived keys, ready for CSV export:")
_safe_display(pbp_derived_keys)

UWW's own scouting notes for Aurora Spartans [SOURCE: SCOUTING REPORT]:

--- Comparing PBP evidence to the stated Keys to Victory [SOURCE: SCOUTING REPORT] ---

KEY 1 [SCOUTING REPORT] -- COMMUNICATE SCREENS & ACTIONS:
  Not directly measurable from play-by-play event types (no screen-action tagging) -- this is a
  communication/technique key tied to their offensive scheme (per the scout's own Offensive Scheme notes),
  not something the event log alone can confirm or refute.

KEY 2 [SCOUTING REPORT] -- DOMINATE THE PAINT:
  Their own rebounding split across their 4 games: [{'rebound_type': 'defensive', 'count': 88}, {'rebound_type': 'offensive', 'count': 77}]
  Their layup-area attempts (Layup + Driving Layup): 97 of 208 total FGA
  Blocks recorded against them by opponents: 15
  Compare against their own Defensive Scheme notes and layup-area conversion rate to judge whether this key
  is supported by the evidence.

KEY 3 [SCOUTING REPORT] -- GUARD 1 ON 1:
  Measured 3PT jump-shot rat


### Diagnose the opponent's weakest video-tagged play type

Same method as UWW's own play-type diagnosis (later in the notebook) -- surfaces which play type the opponent has been least efficient at across their games before facing Whitewater, using video-tagged clip data.

In [72]:
# --- Diagnose the upcoming opponent's weakest video-tagged play type, same method as UWW's own diagnosis -------
_safe_display = lambda df: print(df) if not df.empty else print("  (no data)")
# Play type is whatever chained segment comes right after the LAST "<jersey#> <player name>" token in
# video_description that matches the shooter -- an earlier segment may belong to a DIFFERENT player (e.g. the
# screener/passer who set up the shot), so anchoring on the shooter's own name-token avoids misattributing
# someone else's action. Self-contained name normalization (via `known_names`, their own roster from the
# schedule-loading cell above) rather than reusing the video-attach cell's `normalize_player_name` closure,
# since that one is left pointing at whichever game it last iterated over.
def normalize_elmhurst_name(name, names):
    if name in names:
        return name
    for p in names:
        if p.casefold() == str(name).casefold():
            return p
    return name

def extract_play_type(description, player, names):
    if pd.isna(description) or pd.isna(player):
        return None
    segments = [s.strip() for s in description.split(" > ")]
    player_norm = normalize_elmhurst_name(player, names)
    last_player_idx = None
    for idx, seg in enumerate(segments):
        m = re.match(r"^\d+\s+(.+)$", seg)
        if m and normalize_elmhurst_name(m.group(1), names) == player_norm:
            last_player_idx = idx
    if last_player_idx is not None and last_player_idx + 1 < len(segments):
        return segments[last_player_idx + 1]
    return segments[1] if len(segments) > 1 else None

elmhurst_shot_rows = elmhurst_events[
    elmhurst_events["event_type"].isin(["made_shot", "missed_shot"]) & elmhurst_events["video_description"].notna()
].copy()
elmhurst_shot_rows["play_type"] = elmhurst_shot_rows.apply(
    lambda r: extract_play_type(r["video_description"], r["player"], known_names), axis=1, result_type='reduce'
)
elmhurst_shot_rows["made"] = elmhurst_shot_rows["event_type"] == "made_shot"

play_type_summary = elmhurst_shot_rows.groupby("play_type").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
play_type_summary["fg_pct"] = (100 * play_type_summary["makes"] / play_type_summary["attempts"]).round(1)
play_type_summary = play_type_summary.sort_values("attempts", ascending=False).reset_index(drop=True)
print(f"{upcoming_opponent_short}'s video-tagged play types, across their {prev_games.shape[0]} games before Whitewater "
      f"({len(elmhurst_shot_rows)} video-matched attempts):\n")
_safe_display(play_type_summary)

HIGH_VOLUME_MIN_ATTEMPTS = 10
high_volume = play_type_summary[play_type_summary["attempts"] >= HIGH_VOLUME_MIN_ATTEMPTS]
if not high_volume.empty:
    weakest = high_volume.sort_values("fg_pct").iloc[0]
else:
    weakest = pd.Series({"play_type": "(none)", "fg_pct": 0, "makes": 0, "attempts": 0})
print(f"Weakest high-volume play type (>= {HIGH_VOLUME_MIN_ATTEMPTS} attempts): '{weakest['play_type']}' at "
      f"{weakest['fg_pct']}% ({int(weakest['makes'])}/{int(weakest['attempts'])})\n")

# The tagger's own vocabulary doesn't name every shot; these two labels mark where it runs out. Kept as
# named constants so the "best shot type" logic can recognise a residual bucket instead of presenting it as
# a real shot type.
UNCLASSIFIED_SHOT_MECHANIC = "Unclassified (no mechanic tag)"
NO_CONTEST_TAG = "Not tagged (contest recorded only on catch-and-shoot)"


def extract_shot_mechanic(description):
    """Which kind of shot this was, from the video tagger's own chained description.

    The first three tests are the tagger's SHOT MECHANIC vocabulary. Everything else used to fall through to
    a bucket called "Other" -- which was 19% of all tagged shots and, at 58.9%, the most efficient bucket on
    the board, so it kept winning "best shot type" while telling a coach nothing. Reading the raw tags, it was
    cuts to the rim, putbacks off the offensive glass, and post-ups.

    Those three are tested AFTER the mechanic tests, not before: they describe how a shot was CREATED rather
    than how it was released, and a post-up that finishes as a jumper should still count as a jumper. Checked
    against real tagged data -- "Cut" and "Offensive Rebound" appear in zero already-classified shots, and
    "Post-Up" in 169, all of which keep their existing (more specific) label under this ordering. Adding the
    tier shrinks the residual from 586 shots to 14.
    """
    if pd.isna(description):
        return None
    d = str(description)
    if "No Dribble Jumper" in d:
        return "Catch-and-shoot"
    if "Dribble Jumper" in d:
        return "Pull-up off the dribble"
    if "To Basket" in d:
        return "Drive to the basket"
    # --- fallback tier: shot ORIGIN, for tags carrying no mechanic keyword at all ---
    if "Offensive Rebound" in d:
        return "Putback off the offensive glass"
    if "Cut" in d:
        return "Cut to the basket"
    if "Post-Up" in d:
        return "Post-up"
    return UNCLASSIFIED_SHOT_MECHANIC


def extract_contest(description):
    """Defender contest, which the tagger records ONLY on catch-and-shoot jumpers.

    Verified across 3,039 tagged shots: every one of the 1,069 catch-and-shoot attempts carries Guarded or
    Open, and not one of the other 1,970 does. So a missing contest tag does not mean the shot was a drive --
    the previous label said "(drive, no contest tag)", which mislabelled every cut, putback and post-up as a
    drive. It means the contest dimension simply does not apply to this shot type.
    """
    if pd.isna(description):
        return None
    d = str(description)
    if "Guarded" in d:
        return "Guarded"
    if "Open" in d:
        return "Open"
    return NO_CONTEST_TAG

def extract_distance(description):
    if pd.isna(description):
        return None
    for tag in ["Long/3pt", "Medium/17' to <3p", "Short to < 17'"]:
        if tag in description:
            return tag
    return "N/A"

weak_type_rows = elmhurst_shot_rows[elmhurst_shot_rows["play_type"] == weakest["play_type"]].copy()
weak_type_rows["shot_mechanic"] = weak_type_rows["video_description"].apply(extract_shot_mechanic)
weak_type_rows["contest"] = weak_type_rows["video_description"].apply(extract_contest)
weak_type_rows["distance"] = weak_type_rows["video_description"].apply(extract_distance)

print(f"'{weakest['play_type']}' shot-quality breakdown ({len(weak_type_rows)} attempts):\n")

mechanic_summary = weak_type_rows.groupby("shot_mechanic").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
mechanic_summary["fg_pct"] = (100 * mechanic_summary["makes"] / mechanic_summary["attempts"]).round(1)
print("By shot mechanic:")
_safe_display(mechanic_summary.sort_values("attempts", ascending=False))

contest_summary = weak_type_rows.groupby("contest").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
contest_summary["fg_pct"] = (100 * contest_summary["makes"] / contest_summary["attempts"]).round(1)
print("\nBy contest level:")
_safe_display(contest_summary.sort_values("attempts", ascending=False))

distance_summary = weak_type_rows.groupby("distance").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
distance_summary["fg_pct"] = (100 * distance_summary["makes"] / distance_summary["attempts"]).round(1)
print("\nBy shot distance:")
_safe_display(distance_summary.sort_values("attempts", ascending=False))

player_weak_type = weak_type_rows.groupby("player").agg(
    attempts=("made", "count"), makes=("made", "sum"),
    pct_catch_and_shoot=("shot_mechanic", lambda s: round(100 * (s == "Catch-and-shoot").mean(), 1)),
    pct_guarded=("contest", lambda s: round(100 * (s == "Guarded").mean(), 1)),
).reset_index()
player_weak_type["fg_pct"] = (100 * player_weak_type["makes"] / player_weak_type["attempts"]).round(1)
player_weak_type = player_weak_type.sort_values("attempts", ascending=False)
print(f"\nPer-player '{weakest['play_type']}' volume/efficiency, with catch-and-shoot% and guarded% of those attempts:")
_safe_display(player_weak_type[["player", "attempts", "makes", "fg_pct", "pct_catch_and_shoot", "pct_guarded"]])

Aurora Spartans's video-tagged play types, across their 4 games before Whitewater (218 video-matched attempts):

            play_type  attempts  makes  fg_pct
0             Spot-Up        79     22    27.8
1          Transition        34     18    52.9
2                 Cut        20     11    55.0
3                 ISO        20     12    60.0
4    P&R Ball Handler        20      7    35.0
5            Hand Off        15      5    33.3
6             Post-Up        13      4    30.8
7   Offensive Rebound         6      5    83.3
8        No Play Type         4      3    75.0
9        P&R Roll Man         4      2    50.0
10         Off Screen         3      2    66.7
Weakest high-volume play type (>= 10 attempts): 'Spot-Up' at 27.8% (22/79)

'Spot-Up' shot-quality breakdown (79 attempts):

By shot mechanic:
             shot_mechanic  attempts  makes  fg_pct
0          Catch-and-shoot        53     13    24.5
2  Pull-up off the dribble        14      3    21.4
1      Drive to the bask


### Load player comparison algorithms and compute tag rarity weights

Portable replacement for `%run UW Whitewater Player Comparison Algorithms` -- imports `build_player_comparison_artifacts` from `player_comparison.py` (reloaded each run so edits to that file are picked up), and identifies the target opponent (the most recently scouted game) to compare its players against every other scouted opponent's roster.

In [74]:
# Portable replacement for "%run UW Whitewater Player Comparison Algorithms" -- imports the same helper logic
# from player_comparison.py, sitting alongside this notebook. Force a reload so re-running this cell always
# picks up the latest edits to player_comparison.py, even though the module is already cached in sys.modules
# from an earlier run in this same kernel session.
import importlib
import player_comparison
importlib.reload(player_comparison)
from player_comparison import build_player_comparison_artifacts, PLAYER_COMPARISON_ALGORITHMS_VERSION

# USE_LLM is set in the Configuration cell at the top -- defaults to False so player comparisons run on
# tag-based similarity only, without any LLM calls or cost. Set USE_LLM=True to also run the LLM-based
# comparison (cells below).
use_llm = USE_LLM

comparison_artifacts = build_player_comparison_artifacts(
    schedule=schedule,
    scout_reports=scout_reports,
    player_profiles=player_profiles,
    cache_path=os.path.join(OUTPUT_DIR, "_cache", "llm_player_comparison_cache.jsonl"),
    use_llm=use_llm,
    # Build the comparisons FOR the upcoming opponent (defined in the "Identify the upcoming opponent" cell
    # above). Without this the module defaults to "whichever scouted opponent sits latest on the schedule",
    # which silently produced a uww_player_comparisons full of rows for a DIFFERENT team -- the app's Player
    # Details dialog then matched none of the upcoming opponent's roster players and showed "No comparable
    # player found" against a CSV that was not empty at all.
    upcoming_opponent=upcoming_opponent_short,
)
globals().update(comparison_artifacts)

# Fail loudly rather than quietly shipping comparisons for the wrong team -- this exact mismatch reached the
# app once already and looked like an app bug from there.
if target_opponent != upcoming_opponent_short:
    print(f"WARNING: comparison target is {target_opponent!r}, not the upcoming opponent "
          f"{upcoming_opponent_short!r} -- uww_player_comparisons will not match the app's roster view.")

print(f"Loaded player comparison algorithms (version {PLAYER_COMPARISON_ALGORITHMS_VERSION}). use_llm={use_llm}\n")
print("Scouted opponents and their game number on the schedule:")
for opp, num in sorted(scout_game_numbers.items(), key=lambda x: (x[1] is None, x[1])):
    print(f"  Game #{num}: {opp}")

if target_opponent is None:
    print("\nWARNING: no scouted opponent has a resolvable game number on the schedule, so there's no valid target for a similarity comparison yet.")
else:
    print(f"\nTarget (most recently scouted game): {target_opponent} (game #{target_game_number})")
    print("Previous scouted opponents to compare against:", previous_opponents)

print("\nplayer_notes-derived tag frequency and rarity-based importance weight (rarer tags count more toward similarity):")
print(notes_tag_importance_df)

print("\nkeys_to_defending-derived tag frequency and rarity-based importance weight:")
print(keys_tag_importance_df)

LLM comparison skipped (use_llm=False) -- using tag-based similarity only.
Loaded player comparison algorithms (version 2026-07-29-portable). use_llm=False

Scouted opponents and their game number on the schedule:
  Game #1: Ripon Red Hawks
  Game #2: St. Thomas (TX) Celts
  Game #3: Eureka Red Devils

Target (most recently scouted game): Aurora Spartans (game #None)
Previous scouted opponents to compare against: ['St. Thomas (TX) Celts', 'Eureka Red Devils', 'Ripon Red Hawks']

player_notes-derived tag frequency and rarity-based importance weight (rarer tags count more toward similarity):
                   tag  count  importance_weight
0       slasher_driver      8              2.440
1  three_point_shooter      8              2.440
2            rebounder      4              3.028
3      catch_and_shoot      2              3.539
4           high_usage      1              3.944
5            playmaker      1              3.944
6          post_scorer      1              3.944
7      pull


### Surface tag-based best match per target-opponent player

Displays each target player's single most comparable previously-scouted player, ranked by the tag-based similarity score (playing-style tags, notes/keys overlap, and stat similarity) computed in the cell above.

In [76]:
stat_cols_display = [c for c in best_matches.columns if c.startswith("target_") or c.startswith("compared_")]
stat_cols_display = [c for c in stat_cols_display if c.split("_")[-1] in {"PTS", "REB", "AST", "FG%", "3P%"}]

# CONFIRMED BUG (fixed here): for the very first scouted game of the season there's no PREVIOUS opponent to
# compare players against yet (previous_opponents is empty), so build_player_comparison_artifacts() hands
# back an empty, columnless best_matches. Every column reference below then raised "KeyError: None of
# [Index([...])] are in the [columns]" instead of just explaining that there's nothing to compare against
# yet -- an expected, one-time state at the start of a season, not a real failure.
_bm_required_cols = {
    "target_player", "target_position", "target_opponent", "target_game_date",
    "compared_player", "compared_opponent", "compared_game_date",
    "compared_position", "similarity_score", "stat_similarity", "shared_notes_tags", "shared_keys_tags",
}
if best_matches.empty or not _bm_required_cols <= set(best_matches.columns):
    print(f"No player comparisons available for {target_opponent} yet -- there's no previously-scouted "
          f"opponent on record to compare their players against. Expected for the first scouted game of "
          f"the season; comparisons will start appearing once a second opponent has been scouted.")
else:
    print(best_matches[[
        "target_player", "target_position", "target_opponent", "target_game_date",
        "compared_player", "compared_opponent", "compared_game_date",
        "compared_position", "similarity_score", "stat_similarity", "shared_notes_tags", "shared_keys_tags",
    ] + stat_cols_display])

    print(f"Most comparable previously-scouted player for each {target_opponent} player "
          f"(scouted {scout_game_dates.get(target_opponent)}):\n")
    for _, row in best_matches.iterrows():
        notes_note = f" -- shared NOTES tags: {row['shared_notes_tags']}" if row["shared_notes_tags"] else " -- no shared notes tags"
        keys_note = f"; shared KEYS tags: {row['shared_keys_tags']}" if row["shared_keys_tags"] else "; no shared keys tags"
        print(f"{row['target_player']} ({row['target_position']}) -> {row['compared_player']} of "
              f"{row['compared_opponent']} on {row['compared_game_date']} ({row['compared_position']}), "
              f"score={row['similarity_score']}{notes_note}{keys_note}")
        if pd.notna(row.get("stat_similarity")):
            print(f"    Season-stat similarity contribution: {row['stat_similarity']} (0-1 scale, weighted x{stat_weight} into the score above)")
        else:
            print("    No comparable season stats for this pair -- stat similarity contributed nothing to the score.")
        if pd.notna(row.get("compared_PTS")):
            print(f"    {row['compared_player']}'s season averages: {row['compared_PTS']} PTS, {row['compared_REB']} REB, "
                  f"{row['compared_AST']} AST, {row['compared_FG%']} FG%, {row['compared_3P%']} 3P%")
        else:
            print(f"    No season stats available for {row['compared_player']} (no PDF scout report for {row['compared_opponent']}).")
        if pd.notna(row.get("target_PTS")):
            print(f"    {row['target_player']}'s season averages: {row['target_PTS']} PTS, {row['target_REB']} REB, "
                  f"{row['target_AST']} AST, {row['target_FG%']} FG%, {row['target_3P%']} 3P%")
        else:
            print(f"    No season stats available for {row['target_player']} (no PDF scout report for {target_opponent} yet).")


No player comparisons available for Aurora Spartans yet -- there's no previously-scouted opponent on record to compare their players against. Expected for the first scouted game of the season; comparisons will start appearing once a second opponent has been scouted.



### Compare players using an LLM over their full scouting-note text

An alternative to the tag-based comparison above: sends each target/candidate player's full scouting-note text to an LLM for a similarity judgment. Only runs when `USE_LLM=True` in the Configuration cell (requires `OPENAI_API_KEY`) -- skipped by default.

In [78]:
if not use_llm:
    print("LLM-based comparison was skipped (USE_LLM=False). Set the USE_LLM config variable to True to run this cell.")
else:
    if skipped_targets or skipped_candidates:
        print(f"Skipping {skipped_targets} target player(s) and {skipped_candidates} candidate player(s) with no "
              "scouting report text (box-score-only players) -- the LLM comparison only runs on scouted players.\n")

    print(llm_player_comparison.sort_values(["target_player", "llm_notes_similarity_score"], ascending=[True, False]))

LLM-based comparison was skipped (USE_LLM=False). Set the USE_LLM config variable to True to run this cell.



### Surface the LLM's best match per player, and compare it to the tag-based pick

Shows whether the LLM's top match agrees with the tag-based pick from earlier. Also skipped when `USE_LLM=False`.

In [80]:
if not use_llm:
    print("LLM-based comparison was skipped (USE_LLM=False). Set the USE_LLM config variable to True to run this cell.")
else:
    print(combined[[
        "target_player", "target_position", "compared_player", "compared_opponent",
        "llm_notes_similarity_score", "llm_notes_shared_traits", "llm_keys_similarity_score", "llm_keys_shared_traits",
        "tag_based_pick", "tag_based_score", "picks_agree",
    ]])

    print(f"LLM-based best match per {target_opponent} player -- playing-style (notes) and defensive-approach (keys) "
          "judged as SEPARATE dimensions:\n")
    for _, row in combined.iterrows():
        agree_note = "agrees with" if row["picks_agree"] else "DIFFERS from"
        print(f"{row['target_player']} ({row['target_position']}) -> {row['compared_player']} of "
              f"{row['compared_opponent']} (keyword-tag approach {agree_note} this pick: {row['tag_based_pick']}, score={row['tag_based_score']})")
        print(f"    [Notes] score={row['llm_notes_similarity_score']:.1f}/10 -- shared traits: {row['llm_notes_shared_traits']}")
        print(f"      {row['llm_notes_rationale']}")
        print(f"    [Keys]  score={row['llm_keys_similarity_score']:.1f}/10 -- shared traits: {row['llm_keys_shared_traits']}")
        print(f"      {row['llm_keys_rationale']}\n")

LLM-based comparison was skipped (USE_LLM=False). Set the USE_LLM config variable to True to run this cell.



### Blend the tag-based and LLM similarity scores

Combines both scores into one ranked list per target player. Skipped when `USE_LLM=False`, since there's no LLM score to blend in.

In [82]:
if not use_llm:
    print("LLM-based comparison was skipped (USE_LLM=False), so there is no LLM score to blend with the tag-based "
          "score. Set the USE_LLM config variable to True to run this cell.")
else:
    print(blended_best_matches[[
        "target_player", "target_position", "compared_player", "compared_opponent",
        "blended_score", "tag_similarity_score", "llm_similarity_score",
        "matches_tag_pick", "matches_llm_pick",
        "shared_notes_tags", "shared_keys_tags", "llm_notes_shared_traits", "llm_keys_shared_traits",
    ]])

    print(f"Blended (tag + LLM average) best match per {target_opponent} player, ranked highest to lowest:\n")
    for rank, (_, row) in enumerate(blended_best_matches.iterrows(), start=1):
        tag_note = "same as tag-only pick" if row["matches_tag_pick"] else f"tag-only picked {row['tag_only_pick']}"
        llm_note = "same as LLM-only pick" if row["matches_llm_pick"] else f"LLM-only picked {row['llm_only_pick']}"
        print(f"{rank}. {row['target_player']} ({row['target_position']}) -> {row['compared_player']} of "
              f"{row['compared_opponent']}, blended={row['blended_score']:.2f} "
              f"(tag={row['tag_similarity_score']:.2f}, llm={row['llm_similarity_score']:.1f}/10)")
        print(f"    {tag_note}; {llm_note}")
        print(f"    Tag-based shared NOTES tags: {row['shared_notes_tags'] or '(none)'}")
        print(f"    Tag-based shared KEYS tags: {row['shared_keys_tags'] or '(none)'}")
        print(f"    LLM shared NOTES traits: {row['llm_notes_shared_traits']}")
        print(f"    LLM shared KEYS traits: {row['llm_keys_shared_traits']}\n")

LLM-based comparison was skipped (USE_LLM=False), so there is no LLM score to blend with the tag-based score. Set the USE_LLM config variable to True to run this cell.



### Display best matches

Prints `best_matches` (the tag-based result) on its own -- a plain re-display of the table already shown two cells above, kept as a quick standalone reference.

In [84]:
print(best_matches)

Empty DataFrame
Columns: []
Index: []



### Parse play-by-play MHTML files into a unified event log

Runs `parse_pbp_mhtml`/`build_pbp_events`/`opponent_from_pbp_filename` (defined earlier) over every `"*_pbp.mhtml"` file in `INPUT_DIR`, resolving each file's `self_column` per-game, to build `pbp_events` -- UW-Whitewater's own play-by-play across every scouted game so far.

In [86]:
# --- Play-by-play (PBP) data -----------------------------------------------------------------------------
# Parsing functions (parse_pbp_mhtml, classify_event, build_pbp_events, opponent_from_pbp_filename) live in the
# "Play-by-play parsing functions" cell above. This cell just runs them over every "*_pbp.mhtml" file in
# INPUT_DIR (same pattern as the "*_scout.pdf" loop above) so newly added games are picked up automatically.

# Filter to "*UW-Whitewater*_pbp.mhtml" rather than the broader "*_pbp.mhtml" -- INPUT_DIR may also hold pbp
# files for OTHER teams' games that don't involve UW-Whitewater at all (uploaded while cross-scouting an
# opponent's other games). Those have no "UW-Whitewater" column in their 4-column layout at all, so parsing
# them here would silently misread one of the other two teams' event text as if it were UWW's.
pbp_files = sorted(glob.glob(f"{volume_dir}/*UW-Whitewater*_pbp.mhtml") + glob.glob(f"{volume_dir}/*UW-Whitewater*_pbp.html"))

# Live-scrape+cache any scouted UWW game missing a local pbp file, using its "game_url" from
# uww_team_schedule -- same pattern as the opponent-prior-games pbp cell above.
def _game_date_str(game_date):
    return f"{game_date.month}_{game_date.day}_{game_date.strftime('%y')}"

games_needing_live_uww_pbp = []
if not uww_team_schedule.empty:
    for _, g_row in uww_team_schedule.iterrows():
        g_date = parse_schedule_date(g_row["date"], uww_season_start_year) if pd.notna(g_row.get("date")) else None
        if g_date is None or pd.isna(g_row.get("game_url")):
            continue
        opp_short = next(
            (s for s in scouted_opponents if re.search(re.escape(s), str(g_row["opponent"]), re.IGNORECASE)), None
        )
        if opp_short is None or glob.glob(f"{volume_dir}/{_game_date_str(g_date)}*{opp_short}*_pbp.*"):
            continue
        games_needing_live_uww_pbp.append((g_row, g_date, opp_short))

if games_needing_live_uww_pbp and fastscout_username and fastscout_password:
    # One session call per game so a dead session can self-heal mid-batch (see the pbp cell above).
    for g_row, g_date, opp_short in games_needing_live_uww_pbp:
        try:
            if str(g_row.get("location", "")).strip().lower() == "home":
                matchup = f"{g_row['opponent']} @ UW-Whitewater"
            else:
                matchup = f"UW-Whitewater @ {g_row['opponent']}"
            pbp_save_path = f"{volume_dir}/{_game_date_str(g_date)} {matchup}_pbp.html"
            run_in_fastscout_session(
                lambda page, url=g_row["game_url"], sp=pbp_save_path: scrape_pbp_live(page, url, save_path=sp)
            )
        except Exception as pbp_scrape_error:
            print(f"  Could not live-scrape UWW's own pbp for {g_row['opponent']} ({g_row['game_url']}): {type(pbp_scrape_error).__name__}: {pbp_scrape_error}")

    # Re-glob so the freshly-cached ".html" file(s) are picked up by the parsing loop below.
    pbp_files = sorted(glob.glob(f"{volume_dir}/*UW-Whitewater*_pbp.mhtml") + glob.glob(f"{volume_dir}/*UW-Whitewater*_pbp.html"))
elif games_needing_live_uww_pbp:
    print(
        f"{len(games_needing_live_uww_pbp)} scouted UWW game(s) are missing a local '_pbp' file and could be "
        "live-scraped, but no FASTSCOUT_USERNAME/FASTSCOUT_PASSWORD were found -- skipping."
    )

print(f"Found {len(pbp_files)} UW-Whitewater play-by-play file(s):")
for f in pbp_files:
    print(" -", os.path.basename(f))

# Build a roster set for auto-detecting column swaps -- UWW season stats' second column is the player name.
_player_col = stats.columns[1]
uww_roster_names = set(stats[_player_col].dropna().tolist()) - {"Team Total", "Opponent"}

pbp_events_list = []
for path in pbp_files:
    opponent_short = opponent_from_pbp_filename(path)
    raw_df = parse_pbp_mhtml(path)

    # Auto-detect if UWW's events ended up in the wrong column. FastScout's 4-column layout is
    # "Time | LeftTeam | Score | RightTeam", and which side UWW is on depends on whether they're home or away
    # in that particular export -- it does NOT always match our assumed "uww_text" = column 2 convention.
    # Cross-reference extracted player names from each text column against the known UWW roster.
    def _extract_player_names(series):
        names = set()
        for text in series.dropna():
            parsed = classify_event(str(text).strip())
            if parsed.get("player"):
                names.add(parsed["player"])
        return names

    uww_col_players = _extract_player_names(raw_df["uww_text"])
    opp_col_players = _extract_player_names(raw_df["opp_text"])
    uww_in_uww_col = len(uww_col_players & uww_roster_names)
    uww_in_opp_col = len(opp_col_players & uww_roster_names)

    if uww_in_opp_col > uww_in_uww_col:
        # Columns are swapped: UWW events are in "opp_text" and opponent events in "uww_text".
        # Swap both text column VALUES and reverse the score format ("OppScore-UWWScore" -> "UWWScore-OppScore")
        # so build_pbp_events' defaults (self_column="uww_text", score group(1)=UWW) work correctly.
        raw_df["uww_text"], raw_df["opp_text"] = raw_df["opp_text"].copy(), raw_df["uww_text"].copy()
        raw_df["score_raw"] = raw_df["score_raw"].apply(
            lambda s: "-".join(reversed(str(s).strip().split("-"))) if pd.notna(s) and re.match(r"^\d+-\d+$", str(s).strip()) else s
        )
        print(f"  WARNING: Detected swapped columns for '{opponent_short}' (UWW roster found in right column) -- auto-corrected")

    # Date this game from its OWN filename, not from game_date_for() -- see
    # game_date_from_pbp_filename(). Without this, a rematch inherits the first meeting's date and
    # collapses into it in every groupby downstream.
    file_game_date = game_date_from_pbp_filename(path)
    if file_game_date is None:
        file_game_date = game_date_for(opponent_short)
        print(f"    WARNING: '{os.path.basename(path)}' has no '<m>_<d>_<yy> ' date prefix -- falling "
              f"back to the schedule's FIRST '{opponent_short}' meeting ({file_game_date}). If UWW "
              f"plays this opponent more than once, rename the file so the two games stay distinct.")
    events = build_pbp_events(raw_df, opponent_short, file_game_date)
    n_unclassified = (events["event_type"] == "unclassified").sum()
    print(f"  Parsed '{opponent_short}': {len(events)} events" + (f" ({n_unclassified} UNCLASSIFIED -- inspect raw_text)" if n_unclassified else ""))
    pbp_events_list.append(events)

pbp_events = pd.concat(pbp_events_list, ignore_index=True) if pbp_events_list else pd.DataFrame()

# CONFIRMED BUG (fixed here): pbp_files (above) is a pure filesystem glob with no reference_date awareness at
# all -- it picks up EVERY local "*_pbp.mhtml"/"*_pbp.html" file regardless of that game's date, so pbp_events
# (and everything built from it downstream: uww_pbp_box_score, uww_lineup_stints, the video-tagging attachment
# in the next section, scoring runs, clutch events...) silently included games on/after reference_date whenever
# local files for them already existed on disk. In normal usage (reference_date left at the real "today") this
# never shows up, since future games' PBP files genuinely don't exist yet -- but when reference_date is set
# earlier than the files actually available (e.g. simulating an earlier point in an already-completed season,
# exactly how this was caught), pbp_events ends up scoped to "every game with a local file" instead of "every
# game played so far," which threw off several season-average stats downstream (confirmed: UWW's own turnovers/
# game on the Upcoming Game page). Filtered here, once, at the source, rather than patching every downstream
# consumer individually. Keeps rows with no resolved game_date rather than dropping them -- an unresolved date
# is a different, separate problem (see game_date_for()) and silently discarding that data isn't the fix for it.
if not pbp_events.empty and "game_date" in pbp_events.columns:
    _pbp_events_before = len(pbp_events)
    pbp_events = pbp_events[pbp_events["game_date"].isna() | (pbp_events["game_date"] < reference_date.date())].reset_index(drop=True)
    _n_dropped = _pbp_events_before - len(pbp_events)
    if _n_dropped:
        print(f"Dropped {_n_dropped} pbp_events row(s) with a game_date on/after reference_date ({reference_date_str}) -- not yet \'played\' in this simulation.")

# Anything still unclassified is invisible in every downstream table now that it carries no player --
# so list the distinct raw strings here, which is what a new EVENT_PATTERNS entry gets written from.
if not pbp_events.empty and (pbp_events["event_type"] == "unclassified").any():
    _unc = pbp_events.loc[pbp_events["event_type"] == "unclassified", "raw_text"].value_counts()
    print(f"\n{int(_unc.sum())} unclassified event(s) across {len(_unc)} distinct string(s) -- "
          f"add a pattern to EVENT_PATTERNS for any of these that should be counted:")
    print(_unc.head(20).to_string())
else:
    print("\nEvery play-by-play line was classified.")

print(pbp_events.head(20))

Found 28 UW-Whitewater play-by-play file(s):
 - 11_14_25 UW-Whitewater @ St. Thomas (TX) Celts_pbp.html
 - 11_15_25 UW-Whitewater @ Eureka Red Devils_pbp.html
 - 11_19_25 Aurora Spartans @ UW-Whitewater_pbp.html
 - 11_25_25 Simpson Storm @ UW-Whitewater_pbp.html
 - 11_7_25 UW-Whitewater @ Ripon Red Hawks_pbp.html
 - 12_10_25 UW-Whitewater @ Lawrence Vikings_pbp.html
 - 12_13_25 Carroll (WI) Pioneers @ UW-Whitewater_pbp.html
 - 12_19_25 UW-Whitewater @ Hope Flying Dutchmen_pbp.html
 - 12_20_25 UW-Whitewater @ Alma Scots_pbp.html
 - 12_2_25 Elmhurst Bluejays @ UW-Whitewater_pbp.html
 - 12_30_25 UW-Whitewater @ Coe Kohawks_pbp.html
 - 1_10_26 UW-Whitewater @ UW-River Falls Falcons_pbp.html
 - 1_14_26 UW-Whitewater @ UW-La Crosse Eagles_pbp.html
 - 1_17_26 UW-Stout Blue Devils @ UW-Whitewater_pbp.html
 - 1_21_26 UW-Whitewater @ UW-Platteville Pioneers_pbp.html
 - 1_24_26 UW-Eau Claire Blugolds @ UW-Whitewater_pbp.html
 - 1_28_26 UW-Whitewater @ UW-Oshkosh Titans_pbp.html
 - 1_3_26 UW-Oshko


### Load coach-tagged "recap" CSVs: per-clip Text Overlay notes (play calls, execution grades)

Some games have a companion `"<matchup>_recap.csv"` export from the video-tagging tool -- one row per
tagged clip, with a `Text Overlay` field holding the coach's own note for that specific play (an offensive
play call and how it was executed, or a defensive breakdown of what went right/wrong). Each row also carries
enough identifying info (period, game clock, team, player) to attach it onto the matching `pbp_events` row
for that same play, so the note shows up in context on the play-by-play rather than as a disconnected list.

Every note is ALSO kept in its own standalone `coach_notes` table regardless of whether it successfully
matched a `pbp_events` row -- a handful of clips have no captured game clock (a between-play/summary note),
which can't be matched to a specific play but shouldn't be silently discarded either; season-wide analytics
(most-called plays, common flagged themes) read from this full table, not just the matched subset.

In [88]:
# --- Playbook catalog: the team's own play index (Hudl "Plays" page, saved as MHTML) -------------------
# Every other source names a play differently -- the season play log writes 'Panther-4 "P4"', a coach's
# clip note might say "P-4", "P4" or "PANTHER 4" -- so without a canonical list they count as separate
# plays everywhere downstream. This catalog is that list: the play's real name, the SERIES it belongs to,
# the shorthand the staff actually types, and a set of normalized match keys the app resolves against.
import io

PLAYS_CATALOG_COLS = ["season", "team", "series", "play_name", "base_name", "play_family", "aliases", "match_keys"]


def _play_norm(text):
    """Match key: uppercase, letters and digits only -- "P-4"/"P 4"/"p4" all collapse to "P4"."""
    return re.sub(r"[^A-Z0-9]", "", str(text).upper())


plays_catalog_paths = sorted(
    p for p in glob.glob(f"{INPUT_DIR}/*.mhtml") + glob.glob(f"{INPUT_DIR}/*.html")
    if re.search(r"plays\.(mhtml|html)$", os.path.basename(p), re.IGNORECASE)
)
print(f"Found {len(plays_catalog_paths)} playbook catalog file(s):")
for p in plays_catalog_paths:
    print(" -", os.path.basename(p))

_pc_frames = []
for path in plays_catalog_paths:
    try:
        _pc_html = load_html_snapshot(path)
    except Exception as e:
        print(f"  Could not read {os.path.basename(path)}: {type(e).__name__}: {e}")
        continue
    if not _pc_html:
        continue
    try:
        _pc_tables = pd.read_html(io.StringIO(_pc_html))
    except ValueError:
        print(f"  {os.path.basename(path)}: no HTML tables found -- skipping.")
        continue
    # The page renders one filter row and one data table; take whichever table actually holds play rows
    # (a real season value like "2025-26" and a play name) rather than trusting table order.
    for _t in _pc_tables:
        if _t.shape[1] < 6:
            continue
        _sub = _t.iloc[:, 2:6].copy()
        _sub.columns = ["season", "team", "series", "play_name"]
        _sub = _sub[
            _sub["season"].astype(str).str.match(r"^\d{4}-\d{2}$", na=False)
            & _sub["play_name"].notna()
            & (_sub["team"].astype(str).str.upper().str.strip() != "ALL")
        ]
        if not _sub.empty:
            _pc_frames.append(_sub)

if _pc_frames:
    plays_catalog = pd.concat(_pc_frames, ignore_index=True)
    plays_catalog = plays_catalog.drop_duplicates(subset=["season", "team", "play_name"]).reset_index(drop=True)
    for _c in ("season", "team", "series", "play_name"):
        plays_catalog[_c] = plays_catalog[_c].astype(str).str.strip()
    # Shorthand the staff types lives in quotes inside the play name: 'Panther-4 "P4"' -> alias "P4",
    # base name "Panther-4". A play whose whole name is quoted (the ELOB set '"20"') has no base name --
    # its family falls back to the SERIES rather than being left blank.
    plays_catalog["aliases"] = plays_catalog["play_name"].apply(
        lambda n: "|".join(re.findall(r'"([^"]+)"', str(n)))
    )
    plays_catalog["base_name"] = plays_catalog["play_name"].apply(
        lambda n: re.sub(r'"[^"]*"', "", str(n)).strip()
    )
    plays_catalog["play_family"] = plays_catalog.apply(
        lambda r: (re.match(r"[A-Za-z]+", r["base_name"]).group(0) if re.match(r"[A-Za-z]+", r["base_name"]) else r["series"]),
        axis=1,
    )
    plays_catalog["match_keys"] = plays_catalog.apply(
        lambda r: "|".join(sorted({
            k for k in (
                [_play_norm(r["play_name"]), _play_norm(r["base_name"])]
                + [_play_norm(a) for a in str(r["aliases"]).split("|") if a]
            ) if k
        })),
        axis=1,
    )
    plays_catalog = plays_catalog[PLAYS_CATALOG_COLS]
    print(f"\nParsed {len(plays_catalog)} play(s) across {plays_catalog['series'].nunique()} series "
          f"and {plays_catalog['play_family'].nunique()} family/families.")
    print(plays_catalog[["series", "play_family", "play_name", "aliases"]].to_string(index=False))
else:
    plays_catalog = pd.DataFrame(columns=PLAYS_CATALOG_COLS)
    print("No playbook catalog parsed -- play calls will be used exactly as they appear in the source data.")


Found 1 playbook catalog file(s):
 - UW-Whitewater - Plays.mhtml

Parsed 27 play(s) across 9 series and 17 family/families.
    series play_family          play_name aliases
      ELOB        ELOB               "20"      20
      Snap        Snap     20 Snap (flip)        
      Snap        Snap          30 (flip)        
      Snap        Snap            40 Snap        
      Over     Cheetah            Cheetah        
    Bursts       Cobra              Cobra        
    Bursts         Fox                Fox        
Transition         Get                Get        
  Specials    Kentucky           Kentucky        
  Specials     Panther     Panther 2 "P2"      P2
  Specials     Panther     Panther-4 "P4"      P4
  Specials     Panther        Panther "P"       P
      Over         Pin          Pin Flair        
   Entries      Pistol             Pistol        
Transition       Pitch              Pitch        
    Bursts       Rhino              Rhino        
      SLOB        Snap    

In [89]:
# --- Parse "*_recap.csv" (single-game coach notes) and "*_plays_*.csv" (season-wide play-call log) exports,
# and attach them onto pbp_events ---------------------------------------------------------------------------
# CONFIRMED CHANGE (requested): season-wide play-call logs ("*_plays_*.csv") are retired. Play calls now come from
# INPUT_DIR/uww_plays.csv and INPUT_DIR/opponent_plays.csv, decoded and joined in the "Play calls" cell after the
# video-tagging attach. This cell now reads single-game coach recaps only; the season-wide branch below stays
# for any old multi-date recap file but no longer has a glob feeding it log exports.
recap_files = sorted(glob.glob(f"{volume_dir}/*_recap.csv"))
print(f"Found {len(recap_files)} coach-note/play-log CSV(s):")
for f in recap_files:
    print(" -", os.path.basename(f))


def _recap_period_label(pd_val):
    """Recap CSVs use a bare half number in "Pd." (1, 2, 3+ for OT) -- pbp_events uses "H1"/"H2"/"OT"/"OT2"
    (whatever token build_pbp_events pulled out of the raw PBP source's own "MM:SS (TOKEN)" format). Map the
    bare number onto that same convention so notes line up with pbp_events' own "period" values."""
    try:
        n = int(float(pd_val))
    except (TypeError, ValueError):
        return None
    if n <= 0:
        return None
    if n <= 2:
        return f"H{n}"
    return "OT" if n == 3 else f"OT{n - 2}"


def _clock_to_seconds(clock_val):
    """"10:46" -> 646. Returns None for a blank/unparseable clock (some clips -- e.g. a between-play summary
    note -- have no captured game clock at all)."""
    m = re.match(r"^(\d+):(\d+)", str(clock_val).strip())
    return int(m.group(1)) * 60 + int(m.group(2)) if m else None


def _normalize_for_match(text):
    """Collapse ALL whitespace and lowercase, so "Ripon Red Hawks" and "Ripon Redhawks" -- a real, confirmed
    spelling inconsistency between one of these CSVs' own "Team" column and the schedule/scout-file spelling
    used to build scouted_opponents -- compare as equal. A plain substring check fails on exactly this kind
    of whitespace difference even though it's obviously the same opponent."""
    return re.sub(r"\s+", "", str(text)).lower()


# Date -> short opponent name, built from UWW's own schedule -- used to resolve each ROW's own opponent in a
# season-wide play-call log (one file covering many games), where a single "whole file belongs to one
# opponent" assumption (used for the single-game recap files below) doesn't hold.
_date_to_opponent = {}
if not uww_team_schedule.empty:
    for _, _sched_row in uww_team_schedule.iterrows():
        _d = parse_schedule_date(_sched_row.get("date"), uww_season_start_year)
        if _d is None:
            continue
        _full_opp = str(_sched_row.get("opponent", ""))
        _short_match = next((s for s in scouted_opponents if re.search(re.escape(s), _full_opp, re.IGNORECASE)), None)
        _date_to_opponent[_d] = _short_match or _full_opp


_recap_rows = []
for path in recap_files:
    try:
        recap_df = pd.read_csv(path)
    except Exception as e:
        print(f"  Could not read {os.path.basename(path)}: {type(e).__name__}: {e}")
        continue

    required_cols = {"Player", "Team", "Pd.", "Clock", "Text Overlay", "Result"}
    missing = required_cols - set(recap_df.columns)
    if missing:
        print(f"  {os.path.basename(path)}: missing expected column(s) {missing} -- skipping.")
        continue

    # Section-header rows ("DEFENSIVE CLIPS" / "OFFENSIVE CLIPS" / a play-family header like "Panther Series"
    # in the season-wide log) and any row with no real note/tag carry no Player at all -- drop those first.
    recap_df = recap_df[recap_df["Player"].notna() & (recap_df["Player"].astype(str).str.strip() != "")].copy()
    recap_df = recap_df[recap_df["Text Overlay"].notna() & (recap_df["Text Overlay"].astype(str).str.strip() != "")]
    if recap_df.empty:
        continue

    _is_season_wide = "Date" in recap_df.columns and pd.to_datetime(recap_df["Date"], errors="coerce").dt.date.nunique() > 1

    if _is_season_wide:
        # Season-wide play-call log: each ROW belongs to a different game, so resolve opponent per-row from
        # that row's own "Date" (matched against UWW's own schedule) rather than assuming one opponent for
        # the whole file. Also: this file's "Text Overlay" is a clean PLAY NAME (e.g. "Panther - Elmhurst",
        # "Twins Right Swirl - Ripon"), not free-text coach commentary like the single-game recap files use
        # -- so it goes into its own "play_call" field instead of "coach_note", rather than force two
        # different kinds of content through a format built for one of them.
        recap_df["_parsed_date"] = pd.to_datetime(recap_df["Date"], errors="coerce").dt.date
        # CONFIRMED BUG (fixed here): this file has ALL of a season's play-call rows in one CSV -- unlike
        # every other data source in this notebook (pbp_events, video files, coaching flags), nothing here
        # ever checked a row's own date against reference_date. Set reference_date before a game has been
        # played and its play calls -- if already logged in this file -- showed up anyway. Dropped here,
        # before opponent resolution, so a coach can't end up seeing tendencies from a game that, as of
        # reference_date, hasn't happened yet.
        _n_future = int((recap_df["_parsed_date"].notna() & (recap_df["_parsed_date"] >= reference_date.date())).sum())
        recap_df = recap_df[recap_df["_parsed_date"].isna() | (recap_df["_parsed_date"] < reference_date.date())]
        if _n_future:
            print(f"  {os.path.basename(path)}: dropped {_n_future} row(s) dated on/after reference_date ({reference_date_str}).")
        recap_df["opponent"] = recap_df["_parsed_date"].map(_date_to_opponent)
        _unresolved = recap_df["opponent"].isna().sum()
        recap_df = recap_df[recap_df["opponent"].notna()]
        if _unresolved:
            print(f"  {os.path.basename(path)}: {_unresolved} row(s) had a date that didn't match any UWW schedule game -- skipped.")
        if recap_df.empty:
            continue
        recap_df["team"] = recap_df["Team"].apply(lambda t: "UW-Whitewater" if "whitewater" in str(t).lower() else None)
        recap_df = recap_df[recap_df["team"].notna()]  # this log is offense-only (no "Team" = opponent rows observed)
        recap_df["period"] = recap_df["Pd."].apply(_recap_period_label)
        recap_df["time_remaining_seconds"] = recap_df["Clock"].apply(_clock_to_seconds)
        recap_df["player"] = recap_df["Player"].astype(str).str.strip()
        # Play name = the text before the first " - "-style delimiter in Text Overlay (handles the observed
        # inconsistent spacing, e.g. "Panther - Elmhurst" and "Over Action- Pin to Flare..." alike).
        recap_df["play_call"] = recap_df["Text Overlay"].astype(str).apply(lambda t: re.split(r"\s*-\s*", t.strip(), maxsplit=1)[0].strip())
        # The same "Text Overlay" field also carries clock/situation tags ("End of Half", "Timeout") that
        # aren't called plays at all. Left in, they rank in the app's play-call breakdown as if the staff
        # ran them -- and "End of Half" ranks high purely because every game has one. Drop those rows here:
        # with no coach_note either, they carry nothing else worth keeping.
        _NON_PLAY_CALL_PATTERNS = (
            r"^end\s+of\b",
            r"^(half|halftime|game|period|quarter|ot\d*|overtime)$",
            r"^(time\s*out|timeout|to)$",
            r"^(dead\s*ball|jump\s*ball|tip\s*off|tipoff)$",
            r"^(shot\s*clock|clock)\b",
            r"^(free\s*throws?|ft)$",
            r"^(n/?a|none|unknown|tbd|misc|other|untagged)$",
        )
        # Canonicalize against the playbook catalog parsed just above: the tagger writes the same set a
        # different way in nearly every clip ("P4", "P-4", "PANTHER 4") and often appends the OUTCOME to
        # the name ("P4 Good"), which split one play into a row per spelling and per result downstream.
        # Store the catalog's own name so every consumer of uww_coach_notes agrees on one play per set.
        _PLAY_QUALIFIER_WORDS = {
            "good", "bad", "great", "ok", "okay", "nice", "poor",
            "make", "made", "makes", "miss", "missed", "misses", "score", "scored", "bucket",
            "and1", "and-1", "foul", "fouled", "to", "turnover", "tov", "execution", "exec",
        }
        _play_lookup = {}
        if not plays_catalog.empty:
            for _, _cat_row in plays_catalog.iterrows():
                for _k in [k for k in str(_cat_row["match_keys"]).split("|") if k]:
                    _play_lookup.setdefault(_k, str(_cat_row["play_name"]))

        def _canonical_play(raw):
            """Catalog name for a tagged call, else the call with any trailing outcome word removed.
            Qualifiers are stripped only from the END and only from the known list -- a general
            "longest prefix in the catalog" rule would rewrite the real play "Twins Swirl" into the
            different real play "Twins"."""
            _toks = str(raw or "").strip().split()
            _hit = _play_lookup.get(_play_norm(raw))
            if _hit:
                return _hit
            while len(_toks) > 1 and _toks[-1].strip("().,+-\"'").lower() in _PLAY_QUALIFIER_WORDS:
                _toks.pop()
            _stripped = " ".join(_toks)
            _hit = _play_lookup.get(_play_norm(_stripped))
            if _hit:
                return _hit
            # Not in the catalog (a combination tagged in clips but never filed as its own play, e.g.
            # "Twins Right Swirl"): keep the wording, normalize the CASE. Coaches type the same call as
            # "TWINS SWIRL" one clip and "Twins Swirl" the next, and those grouped as two plays.
            _PLAY_ACRONYMS = {"ELOB", "SLOB", "BLOB", "ATO", "DHO", "ISO", "PNR", "OB", "UCLA", "STS"}
            _cased = []
            for _tok in _stripped.split():
                if _tok.upper().strip("()\"'") in _PLAY_ACRONYMS:
                    _cased.append(_tok.upper())
                elif len(_tok) <= 3 or any(_ch.isdigit() for _ch in _tok):
                    _cased.append(_tok.upper() if _tok.isupper() else _tok)
                else:
                    _cased.append(_tok[:1].upper() + _tok[1:].lower())
            return " ".join(_cased)

        _pre_canon = recap_df["play_call"].copy()
        recap_df["play_call"] = recap_df["play_call"].apply(_canonical_play)
        _n_canon = int((_pre_canon.astype(str) != recap_df["play_call"].astype(str)).sum())
        if _n_canon:
            print(f"  {os.path.basename(path)}: normalized {_n_canon} play call(s) to their catalog names.")

        _is_non_play = recap_df["play_call"].astype(str).str.strip().str.lower().apply(
            lambda t: (not t) or any(re.search(_p, re.sub(r"\s+", " ", t)) for _p in _NON_PLAY_CALL_PATTERNS)
        )
        if int(_is_non_play.sum()):
            print(f"  {os.path.basename(path)}: dropped {int(_is_non_play.sum())} non-play row(s) "
                  f"({', '.join(sorted(set(recap_df.loc[_is_non_play, 'play_call'].astype(str)))[:5])}).")
        recap_df = recap_df[~_is_non_play]
        if recap_df.empty:
            continue
        recap_df["coach_note"] = None
        recap_df["result"] = recap_df["Result"].astype(str).str.strip()
        recap_df["clip_side"] = "Offense"
        _recap_rows.append(recap_df[[
            "opponent", "period", "time_remaining_seconds", "team", "player", "result", "coach_note", "play_call", "clip_side",
        ]])
        print(f"  {os.path.basename(path)}: {len(recap_df)} play-call log row(s) across {recap_df['opponent'].nunique()} opponent(s)")
        continue

    # CONFIRMED BUG (fixed here): same leak as the season-wide log above, for the single-game recap case --
    # nothing here checked this file's own game date against reference_date before including it. These
    # files follow the same "<m>_<d>_<yy> ..." filename convention the PBP/video files use, so their date
    # is available the same way, via game_date_from_pbp_filename() (it only reads the filename prefix, so
    # it works on any file named that way regardless of what comes after).
    _recap_file_date = game_date_from_pbp_filename(path)
    if _recap_file_date is not None and _recap_file_date >= reference_date.date():
        print(f"  {os.path.basename(path)}: game date {_recap_file_date} is on/after reference_date ({reference_date_str}) -- skipping.")
        continue

    # Single-game recap: whole file belongs to one opponent, resolved from whichever non-UWW "Team" value
    # appears (e.g. "Ripon Redhawks") -- whitespace-normalized match against scouted_opponents.
    _opp_team_vals = recap_df.loc[
        ~recap_df["Team"].astype(str).str.contains("whitewater", case=False, na=False), "Team"
    ].dropna().unique()
    _recap_opponent = None
    for _tv in _opp_team_vals:
        _tv_norm = _normalize_for_match(_tv)
        _match = next(
            (s for s in scouted_opponents if _normalize_for_match(s) in _tv_norm or _tv_norm in _normalize_for_match(s)),
            None,
        )
        if _match:
            _recap_opponent = _match
            break
    if _recap_opponent is None:
        print(f"  Could not resolve which scouted opponent {os.path.basename(path)} belongs to (Team values seen: {list(_opp_team_vals)}) -- skipping.")
        continue

    recap_df["opponent"] = _recap_opponent
    recap_df["team"] = recap_df["Team"].apply(lambda t: "UW-Whitewater" if "whitewater" in str(t).lower() else _recap_opponent)
    recap_df["period"] = recap_df["Pd."].apply(_recap_period_label)
    recap_df["time_remaining_seconds"] = recap_df["Clock"].apply(_clock_to_seconds)
    recap_df["player"] = recap_df["Player"].astype(str).str.strip()
    recap_df["coach_note"] = recap_df["Text Overlay"].astype(str).str.strip()
    recap_df["play_call"] = None
    recap_df["result"] = recap_df["Result"].astype(str).str.strip()
    recap_df["clip_side"] = recap_df["team"].apply(lambda t: "Offense" if t == "UW-Whitewater" else "Defense")

    _recap_rows.append(recap_df[[
        "opponent", "period", "time_remaining_seconds", "team", "player", "result", "coach_note", "play_call", "clip_side",
    ]])
    print(f"  {os.path.basename(path)}: {len(recap_df)} coach note(s) for opponent '{_recap_opponent}'")

coach_notes = pd.concat(_recap_rows, ignore_index=True) if _recap_rows else pd.DataFrame(
    columns=["opponent", "period", "time_remaining_seconds", "team", "player", "result", "coach_note", "play_call", "clip_side"]
)

# One possession, one row. The same play can be tagged in BOTH a single-game recap ("PANTHER EXECUTION,
# BIG = WALK YOUR MAN UP") and the season play-call log, and two play-log exports can overlap each other --
# each of which produced a separate row here for one real possession, double-counting it in every play-call
# breakdown downstream. Collapse on the possession key, keeping the first non-null value of each field so
# the merged row carries BOTH the coach's note and the structured play_call. Rows with no clock can't be
# identified as the same possession, so they pass through untouched rather than being merged on a guess.
if not coach_notes.empty and "time_remaining_seconds" in coach_notes.columns:
    _cn_key = ["opponent", "period", "time_remaining_seconds", "team", "player"]
    _cn_timed = coach_notes[coach_notes["time_remaining_seconds"].notna()].copy()
    _cn_untimed = coach_notes[coach_notes["time_remaining_seconds"].isna()]
    _cn_before = len(_cn_timed)
    if _cn_before:
        _cn_timed = (
            _cn_timed.groupby(_cn_key, dropna=False, as_index=False, sort=False)
            .agg({c: "first" for c in coach_notes.columns if c not in _cn_key})
        )
        # groupby().first() skips nulls per column, which is exactly what merges a note-only row and a
        # play-log-only row for the same possession into one complete row.
        if _cn_before != len(_cn_timed):
            print(f"\nMerged {_cn_before - len(_cn_timed)} duplicate possession row(s) "
                  f"(same play tagged in more than one export).")
    coach_notes = pd.concat([_cn_timed, _cn_untimed], ignore_index=True)[list(coach_notes.columns)]

# Attach coach_note/play_call onto the matching pbp_events row by (opponent, period, time_remaining_seconds,
# team, player) -- pbp_events already carries a clean time_remaining_seconds column (parsed from the raw PBP
# source's own "MM:SS (PERIOD)" text), so match on THAT rather than pbp_events' "time_remaining" column,
# which is that unparsed raw string (e.g. "10:46 (H1)") and would never equal the recap's plain "10:46".
# A small number of clips have no clock at all (time_remaining_seconds is None) and simply won't match any
# row -- their note is still preserved in the standalone coach_notes table above, just not linked in-line.
for _c in ("coach_note", "play_call"):
    if _c in pbp_events.columns:
        pbp_events = pbp_events.drop(columns=[_c])
if not coach_notes.empty and not pbp_events.empty:
    _matchable_notes = coach_notes[coach_notes["time_remaining_seconds"].notna() & coach_notes["period"].notna()]
    _join_keys = ["opponent", "period", "time_remaining_seconds", "team", "player"]
    pbp_events = pbp_events.merge(
        _matchable_notes[_join_keys + ["coach_note", "play_call"]].drop_duplicates(subset=_join_keys),
        on=_join_keys, how="left",
    )
    _n_matched = int((pbp_events["coach_note"].notna() | pbp_events["play_call"].notna()).sum())
    print(
        f"\nAttached {_n_matched} coach note(s)/play-call(s) onto pbp_events out of {len(coach_notes)} parsed "
        f"({len(coach_notes) - len(_matchable_notes)} had no usable clock and can't be linked to a specific "
        "play; the rest may not match if a player name is spelled differently between sources)."
    )
    # If the match rate is suspiciously low, print exactly what didn't line up -- side-by-side against a
    # sample of pbp_events' own keys for the same opponent(s) -- rather than leaving "why" as a guessing game.
    if _n_matched < len(_matchable_notes):
        _unmatched = _matchable_notes.merge(
            pbp_events[_join_keys].drop_duplicates(), on=_join_keys, how="left", indicator=True
        )
        _unmatched = _unmatched[_unmatched["_merge"] == "left_only"]
        if not _unmatched.empty:
            print(f"\n{len(_unmatched)} note(s) did NOT find a matching pbp_events row. First few unmatched note keys:")
            print(_unmatched[_join_keys].head(8).to_string(index=False))
            _sample_opp = _unmatched["opponent"].iloc[0]
            print(f"\nFor comparison, actual pbp_events keys for opponent \'{_sample_opp}\' (first 8 rows with a player):")
            _sample_pbp = pbp_events[(pbp_events["opponent"] == _sample_opp) & pbp_events["player"].notna()]
            print(_sample_pbp[_join_keys].head(8).to_string(index=False))
            print(
                "\nCompare the two tables above column-by-column -- the mismatch (differently-spelled player "
                "name, a period token that doesn\'t match, an opponent string that isn\'t identical) should be "
                "visible directly. A common cause: this game\'s local _pbp file used a different exact player-"
                "name spelling than the recap CSV (e.g. a nickname or suffix like \'Jr\')."
            )
else:
    pbp_events["coach_note"] = None
    pbp_events["play_call"] = None


Found 0 coach-note/play-log CSV(s):



### Reconstruct the on-court 5-man lineup for both teams at every event

Walks each game's `pbp_events` in order, tracking substitutions to derive `uww_lineup`/`opp_lineup` -- the 5 players on the floor for each team at the moment of every event.

In [91]:
# --- On-court 5-man lineups for both teams, at every point in the play-by-play -----------------------------
# Reconstructed purely from substitution events ("Subs In"/"Subs Out") plus a starting lineup inferred from each
# player's FIRST event of the game: if a player's first action is anything other than "Subs In", they were
# already on the floor at tip-off (a starter); if their first action IS "Subs In", they came off the bench.
# Dead-ball substitutions almost always swap multiple players for a team at the EXACT same game-clock time, so
# subs are applied as an atomic batch per (opponent, team, period, time_remaining_seconds) -- applying them one
# row at a time would otherwise show a transient 4-man lineup between an "out" row and its matching "in" row.
# Excludes every TEAM-level event whose "player" field is actually a team name, not a roster player.
TEAM_LEVEL_EVENT_TYPES = {"team_deadball_rebound_offensive", "team_deadball_rebound_defensive", "timeout"}
real_player_events = pbp_events[
    pbp_events["player"].notna() & (~pbp_events["event_type"].isin(TEAM_LEVEL_EVENT_TYPES))
].sort_values("event_order")

def starting_lineup(team_events):
    first_seen = team_events.groupby("player").first()
    return set(first_seen[first_seen["event_type"] != "sub_in"].index)

uww_lineup_col = pd.Series(index=pbp_events.index, dtype=object)
opp_lineup_col = pd.Series(index=pbp_events.index, dtype=object)

for (opponent, game_date), opp_rows in pbp_events.groupby(GAME_KEYS, dropna=False):
    opp_real = real_player_events[
        (real_player_events["opponent"] == opponent) & (real_player_events["game_date"] == game_date)
    ]

    # Period start anchors are shared by BOTH teams: the true boundary for a period is the earliest real event
    # across EITHER team tagged with that period, not just one team's own subset.
    period_starts = opp_real.dropna(subset=["period"]).groupby("period")["event_order"].min().sort_values()
    periods_in_order = period_starts.index.tolist()

    for team_label, target_col in [("UW-Whitewater", uww_lineup_col), (opponent, opp_lineup_col)]:
        team_events = opp_real[opp_real["team"] == team_label]

        # Re-infer the starting five FRESH at the start of EVERY period (H1, H2, OT...), not just tip-off.
        changes = []
        for period in periods_in_order:
            period_events = team_events[team_events["period"] == period].sort_values("event_order")
            starters = starting_lineup(period_events)
            if len(starters) != 5:
                print(f"WARNING: {opponent} {game_date}/{team_label}/{period} starting-lineup detection found {len(starters)} "
                      f"players (expected 5): {sorted(starters)}")

            # Apply substitutions one at a time in chronological order, but only RECORD a new change-point once
            # the running set settles back at exactly 5.
            current = set(starters)
            changes.append((period_starts[period] - 1, frozenset(current)))
            sub_events = period_events[period_events["event_type"].isin(["sub_in", "sub_out"])].sort_values("event_order")
            for _, row in sub_events.iterrows():
                if row["event_type"] == "sub_in":
                    current.add(row["player"])
                else:
                    current.discard(row["player"])
                if len(current) == 5:
                    changes.append((row["event_order"], frozenset(current)))
            if len(current) != 5:
                print(f"WARNING: {opponent} {game_date}/{team_label}/{period} never settled back to a 5-man lineup by the "
                      f"last substitution (ended with {len(current)}): {sorted(current)}")

        changes_df = pd.DataFrame(changes, columns=["event_order", "lineup"]).sort_values("event_order")
        target_rows = opp_rows.sort_values("event_order")
        merged = pd.merge_asof(target_rows[["event_order"]], changes_df, on="event_order", direction="backward")
        lineup_strings = merged["lineup"].apply(lambda s: ", ".join(sorted(s)) if isinstance(s, frozenset) else None)
        lineup_strings.index = target_rows.index
        target_col.loc[target_rows.index] = lineup_strings

pbp_events["uww_lineup"] = uww_lineup_col
pbp_events["opp_lineup"] = opp_lineup_col

print(pbp_events[["opponent", "period", "time_remaining", "team", "event_type", "raw_text", "uww_lineup", "opp_lineup"]].head(30))

print("\nLineup size check (every non-null value should be exactly 5 players):")
print(" uww_lineup sizes:", pbp_events["uww_lineup"].dropna().apply(lambda s: len(s.split(", "))).value_counts().to_dict())
print(" opp_lineup sizes:", pbp_events["opp_lineup"].dropna().apply(lambda s: len(s.split(", "))).value_counts().to_dict())

                 opponent period time_remaining                   team  \
0   St. Thomas (TX) Celts   None           None                   None   
1   St. Thomas (TX) Celts     H1     19:58 (H1)  St. Thomas (TX) Celts   
2   St. Thomas (TX) Celts     H1     19:58 (H1)          UW-Whitewater   
3   St. Thomas (TX) Celts     H1     19:34 (H1)  St. Thomas (TX) Celts   
4   St. Thomas (TX) Celts     H1     19:22 (H1)          UW-Whitewater   
5   St. Thomas (TX) Celts     H1     19:22 (H1)  St. Thomas (TX) Celts   
6   St. Thomas (TX) Celts     H1     19:16 (H1)  St. Thomas (TX) Celts   
7   St. Thomas (TX) Celts     H1     19:14 (H1)          UW-Whitewater   
8   St. Thomas (TX) Celts     H1     19:09 (H1)          UW-Whitewater   
9   St. Thomas (TX) Celts     H1     19:09 (H1)  St. Thomas (TX) Celts   
10  St. Thomas (TX) Celts     H1     19:05 (H1)  St. Thomas (TX) Celts   
11  St. Thomas (TX) Celts     H1     19:05 (H1)  St. Thomas (TX) Celts   
12  St. Thomas (TX) Celts     H1     1


### Classify each on-court opponent lineup using scouting-report tags

Adds `opp_lineup_summary`, describing the specific opponent lineup on the floor at any moment: role composition (Starter/Bench), position composition, and each player's most common notes/keys tags from `player_profiles`. No equivalent exists for `uww_lineup` -- UW-Whitewater isn't a scouted opponent, so there's no scouting-report data to classify our own lineups with.

In [93]:
# --- Classify each on-court OPPONENT 5-man unit using its players' scouting-report data -----------------------
# One new field, opp_lineup_summary, describing the specific opponent lineup on the floor: role composition
# (Starter/Bench), position composition, and each player's most common notes_tags/keys_tags. There's no
# equivalent field for uww_lineup -- UW-Whitewater is "us", not a scouted opponent, so player_profiles has no
# scouting-report rows for our own roster to classify it with.
def summarize_lineup(opponent, lineup_str):
    if pd.isna(lineup_str):
        return None
    names = lineup_str.split(", ")
    rows = player_profiles[(player_profiles["opponent"] == opponent) & (player_profiles["name"].isin(names))]
    if rows.empty:
        return f"No scouting data found for: {', '.join(names)}"
    unmatched = [n for n in names if n not in set(rows["name"])]

    role_summary = " / ".join(f"{count} {role}" for role, count in rows["role"].value_counts().items())
    pos_summary = ", ".join(f"{count} {pos}" for pos, count in rows["position_group"].value_counts().items())

    notes_tag_counts = Counter(tag for tags in rows["notes_tags"] for tag in tags)
    keys_tag_counts = Counter(tag for tags in rows["keys_tags"] for tag in tags)
    top_notes = ", ".join(f"{tag} x{n}" for tag, n in notes_tag_counts.most_common(3))
    top_keys = ", ".join(f"{tag} x{n}" for tag, n in keys_tag_counts.most_common(3))

    parts = [
        role_summary,
        f"Pos: {pos_summary}",
        f"Style: {top_notes}" if top_notes else "Style: (no tagged traits)",
        f"Defend: {top_keys}" if top_keys else "Defend: (no tagged traits)",
    ]
    if unmatched:
        parts.append(f"No scouting match: {', '.join(unmatched)}")
    return " | ".join(parts)

unique_opp_lineups = pbp_events[["opponent", "opp_lineup"]].drop_duplicates().dropna().copy()
unique_opp_lineups["opp_lineup_summary"] = unique_opp_lineups.apply(
    lambda r: summarize_lineup(r["opponent"], r["opp_lineup"]), axis=1
)
# Drop any stale opp_lineup_summary from a previous run before merging, so re-running this cell overwrites
# cleanly instead of colliding into opp_lineup_summary_x/_y.
pbp_events = pbp_events.drop(columns=["opp_lineup_summary"], errors="ignore").merge(unique_opp_lineups, on=["opponent", "opp_lineup"], how="left")

print(
    pbp_events[["opponent", "opp_lineup", "opp_lineup_summary"]]
    .drop_duplicates(subset=["opponent", "opp_lineup"])
    .sort_values("opponent")
)

                   opponent  \
835       Eureka Red Devils   
895       Eureka Red Devils   
889       Eureka Red Devils   
878       Eureka Red Devils   
844       Eureka Red Devils   
818       Eureka Red Devils   
803       Eureka Red Devils   
775       Eureka Red Devils   
760       Eureka Red Devils   
678       Eureka Red Devils   
628       Eureka Red Devils   
597       Eureka Red Devils   
581       Eureka Red Devils   
536       Eureka Red Devils   
534       Eureka Red Devils   
556       Eureka Red Devils   
502       Eureka Red Devils   
444       Eureka Red Devils   
507       Eureka Red Devils   
1045        Ripon Red Hawks   
1063        Ripon Red Hawks   
1370        Ripon Red Hawks   
1295        Ripon Red Hawks   
1285        Ripon Red Hawks   
1281        Ripon Red Hawks   
1076        Ripon Red Hawks   
1267        Ripon Red Hawks   
1250        Ripon Red Hawks   
1229        Ripon Red Hawks   
1154        Ripon Red Hawks   
1146        Ripon Red Hawks   
1133    


### Attach video-tagging clip descriptions onto `pbp_events`

Same global order-preserving alignment used for the opponent's prior games earlier, applied here to UWW's own `pbp_events` against each `_video` export. The next cell just displays the result.

In [95]:
# --- Attach video-tagging clip descriptions ("*_video.mhtml") onto pbp_events ---------------------------------
# The video-tagging export logs one row per CLIP, with a chained action "Description" (e.g. "3 Seth Bunders >
# Off Screen > ... > Miss 3 Pts") -- one row per POSSESSION-ENDING action, not one row per raw play-by-play
# event, and it carries no game-clock/timestamp. So there's no direct key to join on. Instead, run ONE global
# order-preserving (monotonic) alignment across the whole opponent's clip log against the whole list of
# attempted pbp events at once, using compatible() as a hard gate (event_type + player/team match, plus an
# on-court lineup check for fouls) and maximizing the total number of matches.
# RESULT_TO_KEY, AND1_SHOT_RE, FREE_THROW_EVENT_TYPES, expand_clip_to_subevents, global_align, and
# parse_video_mhtml are reused from the "Shared video-tagging helper functions" cell above.

def _game_date_str(game_date):
    return f"{game_date.month}_{game_date.day}_{game_date.strftime('%y')}"

# CONFIRMED BUG (fixed here): this cell used to align every "*_video" file found on disk against
# pbp_events regardless of date, including games on/after reference_date. pbp_events itself is already
# restricted to game_date < reference_date (see the "Play-by-play (PBP) data" cell), so a
# post-reference-date video file's own game simply isn't in pbp_events to align against -- that's more
# than wasted work, too: if UWW plays the SAME opponent twice (once before reference_date, once after),
# the date-scoped game_mask below finds no rows for the post-reference-date file and silently falls
# back to aligning its clips against the OTHER, earlier meeting's events instead -- misattributing an
# entire game's worth of video clips to the wrong game. Dropping post-reference-date files up front
# avoids that. A file with no parseable date is kept (same "can't tell, so don't guess" stance the
# WARNING branch further down already takes for ambiguous dates) -- only a CONFIRMED on/after-date file
# gets dropped.
def _drop_post_reference_date_videos(files):
    dated = [(f, game_date_from_pbp_filename(f.replace("_video.", "_pbp."))) for f in files]
    kept = [f for f, d in dated if d is None or d < reference_date.date()]
    n_dropped = len(files) - len(kept)
    if n_dropped:
        print(f"Skipping {n_dropped} video-tagging file(s) dated on/after reference_date ({reference_date_str}).")
    return kept

video_files = sorted(glob.glob(f"{volume_dir}/*_video.mhtml") + glob.glob(f"{volume_dir}/*_video.html"))
video_files = _drop_post_reference_date_videos(video_files)

# Live-scrape+cache any scouted UWW game missing a local video-tagging file, using its "video_url" from
# uww_team_schedule -- same pattern as the opponent-prior-games video cell above.
games_needing_live_uww_video = []
if not uww_team_schedule.empty:
    for _, g_row in uww_team_schedule.iterrows():
        g_date = parse_schedule_date(g_row["date"], uww_season_start_year) if pd.notna(g_row.get("date")) else None
        if g_date is None or pd.isna(g_row.get("video_url")):
            continue
        opp_short = next(
            (s for s in scouted_opponents if re.search(re.escape(s), str(g_row["opponent"]), re.IGNORECASE)), None
        )
        if opp_short is None or glob.glob(f"{volume_dir}/{_game_date_str(g_date)}*{opp_short}*_video.*"):
            continue
        games_needing_live_uww_video.append((g_row, g_date, opp_short))

if games_needing_live_uww_video and fastscout_username and fastscout_password:
    for g_row, g_date, opp_short in games_needing_live_uww_video:
        try:
            if str(g_row.get("location", "")).strip().lower() == "home":
                matchup = f"{g_row['opponent']} @ UW-Whitewater"
            else:
                matchup = f"UW-Whitewater @ {g_row['opponent']}"
            video_save_path = f"{volume_dir}/{_game_date_str(g_date)} {matchup}_video.html"
            run_in_fastscout_session(
                lambda page, url=g_row["video_url"], sp=video_save_path: scrape_video_clips_live(page, url, save_path=sp)
            )
        except Exception as video_scrape_error:
            print(f"  Could not live-scrape UWW's own video clips for {g_row['opponent']} ({g_row['video_url']}): {type(video_scrape_error).__name__}: {video_scrape_error}")

    # Re-glob so the freshly-cached ".html" file(s) are picked up by the alignment loop below.
    video_files = sorted(glob.glob(f"{volume_dir}/*_video.mhtml") + glob.glob(f"{volume_dir}/*_video.html"))
    video_files = _drop_post_reference_date_videos(video_files)
elif games_needing_live_uww_video:
    print(
        f"{len(games_needing_live_uww_video)} scouted UWW game(s) are missing a local '_video' file and could "
        "be live-scraped, but no FASTSCOUT_USERNAME/FASTSCOUT_PASSWORD were found -- skipping."
    )

print(f"Found {len(video_files)} video-tagging file(s):")
for f in video_files:
    print(" -", os.path.basename(f))

video_desc_col = pd.Series(index=pbp_events.index, dtype=object)
video_result_col = pd.Series(index=pbp_events.index, dtype=object)
video_player_col = pd.Series(index=pbp_events.index, dtype=object)
video_clip_number_col = pd.Series(index=pbp_events.index, dtype=float)

# CONFIRMED_CLIP_EVENT_OVERRIDES is reused from the "Shared video-tagging helper functions" cell above.
opponents_with_video = []
for path in video_files:
    # Home/away-aware extraction (mirrors opponent_from_pbp_filename above); matches ".mhtml" or ".html".
    _vname = re.sub(r"_video\.(mhtml|html)$", "", os.path.basename(path), flags=re.IGNORECASE)
    _vname = re.sub(r"^\d+_\d+_\d+\s+", "", _vname)
    if " @ " in _vname:
        _left, _right = [side.strip() for side in _vname.split(" @ ", 1)]
        opponent_short = _right if _left == "UW-Whitewater" else _left
    else:
        opponent_short = _vname
    opponents_with_video.append(opponent_short)
    # A video file covers ONE game. Scope the alignment to that game (by its own filename date), or a
    # rematch's clips get aligned against both meetings' events at once.
    video_game_date = game_date_from_pbp_filename(path.replace("_video.", "_pbp."))
    clips = parse_video_mhtml(path)
    total_clips_n = len(clips)

    game_mask = pbp_events["opponent"] == opponent_short
    if video_game_date is not None and (game_mask & (pbp_events["game_date"] == video_game_date)).any():
        game_mask &= pbp_events["game_date"] == video_game_date
    elif pbp_events.loc[game_mask, "game_date"].nunique() > 1:
        print(f"    WARNING: '{os.path.basename(path)}' has no usable date prefix but "
              f"{opponent_short} has {pbp_events.loc[game_mask, 'game_date'].nunique()} games -- "
              f"clips will be aligned across all of them.")
    opp_all_events = pbp_events[game_mask]
    player_team_lookup = opp_all_events.dropna(subset=["player"]).drop_duplicates("player").set_index("player")["team"]
    known_teams = set(player_team_lookup.unique())
    known_players = set(player_team_lookup.index)

    def normalize_player_name(name):
        if name in known_players:
            return name
        for p in known_players:
            if p.casefold() == str(name).casefold():
                return p
        return name

    def committing_team_for(fouled_player):
        fouled_team = player_team_lookup.get(fouled_player)
        others = known_teams - {fouled_team}
        return next(iter(others)) if len(others) == 1 else None

    subevent_rows = []
    for _, clip_row in clips.iterrows():
        clip_player = normalize_player_name(clip_row["player"])
        for event_type in expand_clip_to_subevents(clip_row):
            match_key = committing_team_for(clip_player) if event_type == "foul" else clip_player
            subevent_rows.append({
                "match_key": match_key, "event_type": event_type,
                "Description": clip_row["Description"], "video_result": clip_row["Result"],
                "video_clip_player": clip_player, "video_clip_number": clip_row["No."],
            })
    matchable_clips = pd.DataFrame(
        subevent_rows, columns=["match_key", "event_type", "Description", "video_result", "video_clip_player", "video_clip_number"]
    )

    target_event_types = {"made_shot", "missed_shot", "turnover", "foul"} | FREE_THROW_EVENT_TYPES
    opp_events = pbp_events[game_mask & pbp_events["event_type"].isin(target_event_types)].sort_values("event_order").copy()
    opp_events["match_event_type"] = opp_events["event_type"].where(
        ~opp_events["event_type"].isin(FREE_THROW_EVENT_TYPES), "free_throw"
    )
    opp_events["match_key"] = opp_events["team"].where(opp_events["event_type"] == "foul", opp_events["player"])
    total_pbp_n = pbp_events.loc[game_mask, "event_order"].max()

    matchable_clips = matchable_clips.sort_values("video_clip_number").reset_index(drop=True)
    opp_events = opp_events.sort_values("event_order")
    pbp_orig_index = opp_events.index.tolist()

    pbp_list = []
    for _, row in opp_events.iterrows():
        uww_set = set(row["uww_lineup"].split(", ")) if pd.notna(row["uww_lineup"]) else None
        opp_set = set(row["opp_lineup"].split(", ")) if pd.notna(row["opp_lineup"]) else None
        pbp_list.append({
            "event_order": row["event_order"], "match_event_type": row["match_event_type"],
            "match_key": row["match_key"], "uww_lineup_set": uww_set, "opp_lineup_set": opp_set,
        })
    video_list = matchable_clips.to_dict("records")
    n, m = len(pbp_list), len(video_list)

    def compatible(i, j):
        p, v = pbp_list[i], video_list[j]
        if p["match_event_type"] != v["event_type"] or p["match_key"] != v["match_key"]:
            return False
        if v["event_type"] == "foul":
            fouled_player = v["video_clip_player"]
            fouled_team = player_team_lookup.get(fouled_player)
            lineup_set = p["uww_lineup_set"] if fouled_team == "UW-Whitewater" else p["opp_lineup_set"]
            if lineup_set is not None and fouled_player not in lineup_set:
                return False
        return True

    def pos_cost(i, j):
        return abs(pbp_list[i]["event_order"] / total_pbp_n - video_list[j]["video_clip_number"] / total_clips_n)

    n_matched = 0
    n_ft_matched = 0
    n_ft_total = int((opp_events["match_event_type"] == "free_throw").sum())
    n_foul_matched = 0
    n_foul_total = int((opp_events["match_event_type"] == "foul").sum())
    pairs = global_align(n, m, compatible, pos_cost)

    pbp_order_to_i = {p["event_order"]: idx for idx, p in enumerate(pbp_list)}
    video_no_to_j = {v["video_clip_number"]: idx for idx, v in enumerate(video_list)}
    for (ov_opponent, ov_clip_no), ov_event_order in CONFIRMED_CLIP_EVENT_OVERRIDES.items():
        if ov_opponent != opponent_short or ov_clip_no not in video_no_to_j or ov_event_order not in pbp_order_to_i:
            continue
        override_i, override_j = pbp_order_to_i[ov_event_order], video_no_to_j[ov_clip_no]
        pairs = {i: j for i, j in pairs.items() if i != override_i and j != override_j}
        pairs[override_i] = override_j

    for i, j in pairs.items():
        pbp_idx = pbp_orig_index[i]
        clip = video_list[j]
        video_desc_col.loc[pbp_idx] = clip["Description"]
        video_result_col.loc[pbp_idx] = clip["video_result"]
        video_player_col.loc[pbp_idx] = clip["video_clip_player"]
        video_clip_number_col.loc[pbp_idx] = clip["video_clip_number"]
        n_matched += 1
        if pbp_list[i]["match_event_type"] == "free_throw":
            n_ft_matched += 1
        if pbp_list[i]["match_event_type"] == "foul":
            n_foul_matched += 1

    print(
        f"  {opponent_short}: matched {n_matched}/{len(opp_events)} made/missed-shot, turnover, free-throw & foul "
        f"pbp_events rows to a video clip description via a single global order-preserving alignment across the "
        f"whole game (of which {n_ft_matched}/{n_ft_total} are free throws, and {n_foul_matched}/{n_foul_total} "
        f"are fouls). video log had {len(matchable_clips)} taggable sub-events of these types."
    )

pbp_events["video_description"] = video_desc_col
pbp_events["video_result"] = video_result_col
pbp_events["video_clip_number"] = video_clip_number_col
pbp_events["video_player"] = video_player_col
print(
    pbp_events[["opponent", "event_order", "team", "player", "event_type", "raw_text","video_clip_number", "video_result", "video_player", "video_description"]]
    .head(20)
)

ATTEMPTED_EVENT_TYPES = {"made_shot", "missed_shot", "turnover", "foul"} | FREE_THROW_EVENT_TYPES
has_video_mask = pbp_events["opponent"].isin(opponents_with_video)
unmatched_mask = has_video_mask & pbp_events["event_type"].isin(ATTEMPTED_EVENT_TYPES) & pbp_events["video_description"].isna()
n_unattributed_turnover = (unmatched_mask & pbp_events["player"].isna()).sum()
unmatched = pbp_events[unmatched_mask & pbp_events["player"].notna()].sort_values(["opponent", "event_order"])
print(
    f"\n{len(unmatched)} still-unmatched row(s) among attempted event types across {len(opponents_with_video)} "
    f"game(s) with a video-tagging file ({n_unattributed_turnover} bare team-level turnover(s) with no player "
    f"excluded -- never matchable):"
)
for opponent_short in opponents_with_video:
    print(f"  {opponent_short}: {(unmatched['opponent'] == opponent_short).sum()} unmatched")
print(unmatched[["opponent", "event_order", "period", "time_remaining", "team", "player", "event_type", "raw_text"]])

matched_clip_numbers = set(video_clip_number_col.dropna().astype(int))
unmatched_clips = pd.DataFrame()
for path in video_files:
    # Home/away-aware extraction (mirrors opponent_from_pbp_filename above); matches ".mhtml" or ".html".
    _vname = re.sub(r"_video\.(mhtml|html)$", "", os.path.basename(path), flags=re.IGNORECASE)
    _vname = re.sub(r"^\d+_\d+_\d+\s+", "", _vname)
    if " @ " in _vname:
        _left, _right = [side.strip() for side in _vname.split(" @ ", 1)]
        opponent_short = _right if _left == "UW-Whitewater" else _left
    else:
        opponent_short = _vname
    clips = parse_video_mhtml(path)
    NEVER_MATCHED_RESULTS = {"Run Offense", "No Violation", "Kicked Ball"}
    unmatched_c = clips[
        ~clips["No."].astype(int).isin(matched_clip_numbers) & ~clips["Result"].isin(NEVER_MATCHED_RESULTS)
    ]
    if not unmatched_c.empty:
        unmatched_c = unmatched_c.copy()
        unmatched_c["opponent"] = opponent_short
        unmatched_clips = pd.concat([unmatched_clips, unmatched_c], ignore_index=True)
if unmatched_clips.empty:
    print("No unmatched clips found in the video-tagging files.")
else:
    print("\nUnmatched clips from the video-tagging file(s):")
    print(unmatched_clips[["opponent", "No.", "player", "Result", "Description", "Team", "Duration"]])

Skipping 65 video-tagging file(s) dated on/after reference_date (2025-11-19).
Found 23 video-tagging file(s):
 - 11_11_25 Monmouth (IL) Fighting Scots @ Eureka Red Devils_video.html
 - 11_11_25 St. Thomas (TX) Celts @ East Texas Baptist Tigers_video.html
 - 11_12_25 Aurora Spartans @ North Central (IL) Cardinals_video.html
 - 11_12_25 Hope Flying Dutchmen @ Elmhurst Bluejays_video.html
 - 11_12_25 North Park Vikings @ Alma Scots_video.html
 - 11_14_25 Eureka Red Devils @ Washington-St. Louis Bears_video.html
 - 11_14_25 Loras Duhawks @ Blackburn Beavers_video.html
 - 11_14_25 UW-Oshkosh Titans @ Maryville (TN) Scots_video.html
 - 11_14_25 UW-Whitewater @ St. Thomas (TX) Celts_video.html
 - 11_15_25 Aurora Spartans @ Benedictine (IL) Eagles_video.html
 - 11_15_25 Carthage Firebirds @ Alma Scots_video.html
 - 11_15_25 UW-Oshkosh Titans @ Illinois Wesleyan Titans_video.html
 - 11_15_25 UW-Whitewater @ Eureka Red Devils_video.html
 - 11_15_25 Wisconsin Lutheran Warriors @ Elmhurst Bluejays

In [96]:
print(pbp_events[['period','time_remaining','event_order','uww_score','opp_score','team','player','event_type','raw_text','shot_type','shot_desc','video_clip_number','video_description','video_player','video_result']])

     period time_remaining  event_order  uww_score  opp_score  \
0      None           None            0        NaN        NaN   
1        H1     19:58 (H1)            1        0.0        0.0   
2        H1     19:58 (H1)            2        0.0        0.0   
3        H1     19:34 (H1)            3        0.0        2.0   
4        H1     19:22 (H1)            4        0.0        2.0   
...     ...            ...          ...        ...        ...   
1417     H2     00:15 (H2)          477       76.0       58.0   
1418     H2     00:15 (H2)          478       76.0       58.0   
1419     H2     00:01 (H2)          479       76.0       58.0   
1420     H2     00:00 (H2)          480       76.0       58.0   
1421     H2           None          481        NaN        NaN   

                       team           player                       event_type  \
0                      None              NaN                    period_marker   
1     St. Thomas (TX) Celts  Charles Gitonga             


### Reconstruct and validate a per-game box score from play-by-play

Aggregates player-level PBP events into a real box score, then sanity-checks each game's reconstructed final score against the schedule's own recorded result.

In [98]:
# --- Reconstruct a real single-game box score from the play-by-play, and sanity-check it against the schedule's
# final scores ------------------------------------------------------------------------------------------------
TEAM_LEVEL_EVENT_TYPES_BOX = {"team_deadball_rebound_offensive", "team_deadball_rebound_defensive", "timeout"}
player_events = pbp_events[
    pbp_events["player"].notna() & (~pbp_events["event_type"].isin(TEAM_LEVEL_EVENT_TYPES_BOX))
].copy()

player_events["points"] = player_events.apply(
    lambda row: int(row["shot_type"]) if row["event_type"] == "made_shot" else (1 if row["event_type"] == "free_throw_made" else 0),
    axis=1,
)
player_events["is_fgm"] = player_events["event_type"] == "made_shot"
player_events["is_fga"] = player_events["event_type"].isin(["made_shot", "missed_shot"])
player_events["is_3pm"] = player_events["is_fgm"] & (player_events["shot_type"] == "3")
player_events["is_3pa"] = player_events["is_fga"] & (player_events["shot_type"] == "3")
player_events["is_ftm"] = player_events["event_type"] == "free_throw_made"
player_events["is_fta"] = player_events["event_type"].isin(["free_throw_made", "free_throw_missed"])
player_events["is_oreb"] = player_events["event_type"] == "rebound_offensive"
player_events["is_dreb"] = player_events["event_type"] == "rebound_defensive"
player_events["is_ast"] = player_events["event_type"] == "assist"
player_events["is_stl"] = player_events["event_type"] == "steal"
player_events["is_blk"] = player_events["event_type"] == "block"
player_events["is_to"] = player_events["event_type"] == "turnover"
player_events["is_pf"] = player_events["event_type"] == "foul"

pbp_box_score = player_events.groupby(["opponent", "game_date", "team", "player"]).agg(
    PTS=("points", "sum"), FGM=("is_fgm", "sum"), FGA=("is_fga", "sum"),
    FG3M=("is_3pm", "sum"), FG3A=("is_3pa", "sum"), FTM=("is_ftm", "sum"), FTA=("is_fta", "sum"),
    OREB=("is_oreb", "sum"), DREB=("is_dreb", "sum"), AST=("is_ast", "sum"), STL=("is_stl", "sum"),
    BLK=("is_blk", "sum"), TO=("is_to", "sum"), PF=("is_pf", "sum"),
).reset_index()

# CONFIRMED GAP (fixed here): a bare team-level turnover (e.g. "Turnover (Offensive Foul)" with no player
# attached -- see classify_event's dedicated pattern for this) is correctly excluded from player_events above
# via the player.notna() filter, since there's no player to attribute it to. But that also means it never
# showed up ANYWHERE in the box score -- not on a player's line, and not accounted for at all -- so the box
# score's total turnover count silently undercounted the real game total by however many of these occurred.
# Surfaced here as a synthetic "TEAM" row per (opponent, game_date, team) instead, so these are visible rather
# than silently dropped.
_team_level_turnovers = pbp_events[pbp_events["player"].isna() & (pbp_events["event_type"] == "turnover")]
if not _team_level_turnovers.empty:
    _team_to_rows = _team_level_turnovers.groupby(["opponent", "game_date", "team"]).size().reset_index(name="TO")
    _team_to_rows["player"] = "TEAM"
    for _stat_col in ["PTS", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA", "OREB", "DREB", "AST", "STL", "BLK", "PF"]:
        _team_to_rows[_stat_col] = 0
    pbp_box_score = pd.concat([pbp_box_score, _team_to_rows], ignore_index=True)
    print(f"Added {len(_team_to_rows)} synthetic TEAM row(s) for {int(_team_to_rows['TO'].sum())} bare (unattributed) team-level turnover(s) that would otherwise be missing from the box score entirely.")

pbp_box_score["REB"] = pbp_box_score["OREB"] + pbp_box_score["DREB"]
pbp_box_score["FG%"] = (100 * pbp_box_score["FGM"] / pbp_box_score["FGA"]).round(1)
pbp_box_score["3P%"] = (100 * pbp_box_score["FG3M"] / pbp_box_score["FG3A"]).round(1)
pbp_box_score["FT%"] = (100 * pbp_box_score["FTM"] / pbp_box_score["FTA"]).round(1)

starters_by_game = {}
for (opponent, game_date), group in pbp_events.groupby(GAME_KEYS, dropna=False):
    first_row = group.sort_values("event_order").iloc[0]
    starters_by_game[(opponent, game_date)] = {
        "UW-Whitewater": set(first_row["uww_lineup"].split(", ")) if pd.notna(first_row["uww_lineup"]) else set(),
        opponent: set(first_row["opp_lineup"].split(", ")) if pd.notna(first_row["opp_lineup"]) else set(),
    }
pbp_box_score["started"] = pbp_box_score.apply(
    lambda row: row["player"] in starters_by_game.get((row["opponent"], row["game_date"]), {}).get(row["team"], set()), axis=1
)

pbp_team_totals = pbp_box_score.groupby(["opponent", "game_date", "team"])["PTS"].sum().reset_index()
print("Validating PBP-reconstructed final scores against the schedule:\n")
# Matched on BOTH opponent and date. A name-only match returns the first meeting for every rematch,
# so a merged game would silently "validate" against the wrong row (or appear to be off by exactly
# the other meeting's score) instead of being reported.
_n_ok = _n_bad = 0
for (opponent, game_date) in pbp_events[GAME_KEYS].dropna(subset=["opponent"]).drop_duplicates().itertuples(index=False):
    cand = schedule[schedule["opponent"].str.contains(re.escape(opponent), case=False, na=False)].copy()
    cand = cand[cand["date"].apply(parse_schedule_date) == game_date] if game_date is not None else cand
    if cand.empty:
        print(f"  {opponent} {game_date}: no matching schedule row found -- can't validate.")
        continue
    sched_row = cand.iloc[0]
    sel = (pbp_team_totals["opponent"] == opponent) & (pbp_team_totals["game_date"] == game_date)
    uww_pts = pbp_team_totals[sel & (pbp_team_totals["team"] == "UW-Whitewater")]["PTS"]
    opp_pts = pbp_team_totals[sel & (pbp_team_totals["team"] == opponent)]["PTS"]
    uww_val = int(uww_pts.iloc[0]) if not uww_pts.empty else None
    opp_val = int(opp_pts.iloc[0]) if not opp_pts.empty else None
    ok = (uww_val, opp_val) == (sched_row["team_score"], sched_row["opponent_score"])
    _n_ok, _n_bad = _n_ok + ok, _n_bad + (not ok)
    status = "OK" if ok else "MISMATCH -- check parsing for this game"
    print(f"  {opponent} {game_date}: PBP {uww_val}-{opp_val} vs. schedule "
          f"{sched_row['team_score']}-{sched_row['opponent_score']} [{status}]")
    for team_label in ["UW-Whitewater", opponent]:
        starters_list = sorted(starters_by_game.get((opponent, game_date), {}).get(team_label, set()))
        print(f"    {team_label} starters: {', '.join(starters_list) if starters_list else '(not detected)'}")
print(f"\n{_n_ok} game(s) reconcile exactly, {_n_bad} do not.")
_dupes = pbp_box_score.groupby(["opponent", "game_date", "team", "player"]).size()
_dupes = _dupes[_dupes > 1]
if len(_dupes):
    print(f"WARNING: {len(_dupes)} duplicated (opponent, game_date, team, player) key(s) -- two files for one game?")

print(pbp_box_score.sort_values(["opponent", "PTS"], ascending=[True, False])[[
    "opponent", "game_date", "team", "player", "started", "PTS", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA",
    "OREB", "DREB", "REB", "AST", "STL", "BLK", "TO", "PF", "FG%", "3P%", "FT%",
]])

# --- Cross-validate the PBP-reconstructed box score against the official per-game box-score snapshot ----------
box_files = sorted(glob.glob(f"{volume_dir}/*_box.mhtml"))
print(f"\nFound {len(box_files)} official box-score file(s) to cross-validate against:")
for f in box_files:
    print(" -", os.path.basename(f))

def parse_box_mhtml(path):
    html = load_mhtml_html(path)
    soup = BeautifulSoup(html, "lxml")
    tables = soup.find_all("table")
    score_df = pd.read_html(StringIO(str(tables[0])))[0]
    team_order = score_df.iloc[:, 2].tolist()
    frames = []
    for team_name, table in zip(team_order, tables[1:]):
        pdf = pd.read_html(StringIO(str(table)))[0]
        pdf["team"] = team_name
        frames.append(pdf)
    box_df = pd.concat(frames, ignore_index=True)
    box_df["started"] = box_df["PLAYER"].astype(str).str.endswith("*")
    box_df["player"] = box_df["PLAYER"].astype(str).str.rstrip("*").str.strip()
    box_df = box_df[~box_df["player"].str.contains("Team Total", case=False)]
    for pair_col, (m_col, a_col) in {"FGM-A": ("FGM", "FGA"), "3PM-A": ("FG3M", "FG3A"), "FTM-A": ("FTM", "FTA")}.items():
        split = box_df[pair_col].astype(str).replace("-", "0-0").str.split("-", n=1, expand=True)
        box_df[m_col] = pd.to_numeric(split[0], errors="coerce").fillna(0)
        box_df[a_col] = pd.to_numeric(split[1], errors="coerce").fillna(0)
    for c in ["PTS", "REB", "AST", "TO", "STL", "BLK", "PF"]:
        box_df[c] = pd.to_numeric(box_df[c], errors="coerce").fillna(0)
    return box_df[["team", "player", "started", "PTS", "REB", "AST", "TO", "STL", "BLK", "PF",
                   "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA"]]

box_stat_cols = ["PTS", "REB", "AST", "TO", "STL", "BLK", "PF", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA"]
print("\nValidating PBP-reconstructed per-player box score against the official box-score file, where available:\n")
for path in box_files:
    m = re.search(r"@ (.+)_box\.mhtml$", os.path.basename(path))
    opponent_short = m.group(1) if m else os.path.basename(path)
    official_box = parse_box_mhtml(path)
    pbp_slice = pbp_box_score[pbp_box_score["opponent"] == opponent_short]
    if pbp_slice.empty:
        print(f"{opponent_short}: no PBP-reconstructed box score found for this opponent -- skipping.\n")
        continue

    merged = official_box.merge(pbp_slice, on=["team", "player"], how="outer", suffixes=("_official", "_pbp"), indicator=True)
    only_official = merged[merged["_merge"] == "left_only"]
    only_pbp = merged[merged["_merge"] == "right_only"]
    both = merged[merged["_merge"] == "both"]

    n_stat_mismatch = 0
    n_started_mismatch = 0
    for _, r in both.iterrows():
        stat_mismatches = [c for c in box_stat_cols if r[f"{c}_official"] != r[f"{c}_pbp"]]
        started_mismatch = r["started_official"] != r["started_pbp"]
        if stat_mismatches or started_mismatch:
            detail = []
            if stat_mismatches:
                detail.append("; ".join(f"{c} official={r[f'{c}_official']} vs pbp={r[f'{c}_pbp']}" for c in stat_mismatches))
            if started_mismatch:
                detail.append(f"started official={r['started_official']} vs pbp={r['started_pbp']}")
            print(f"  MISMATCH {opponent_short} {r['team']} {r['player']}: {'; '.join(detail)}")
            n_stat_mismatch += bool(stat_mismatches)
            n_started_mismatch += bool(started_mismatch)
    for _, r in only_official.iterrows():
        print(f"  {opponent_short} {r['team']} {r['player']}: in official box score but missing from PBP reconstruction")
    for _, r in only_pbp.iterrows():
        print(f"  {opponent_short} {r['team']} {r['player']}: in PBP reconstruction but missing from official box score")

    status = (
        "OK" if not (n_stat_mismatch or n_started_mismatch or len(only_official) or len(only_pbp))
        else f"{n_stat_mismatch} stat mismatch(es), {n_started_mismatch} started mismatch(es), "
             f"{len(only_official)} missing-from-PBP, {len(only_pbp)} extra-in-PBP"
    )
    print(f"{opponent_short}: {len(both)} player(s) compared against the official box score -- [{status}]\n")

Added 1 synthetic TEAM row(s) for 1 bare (unattributed) team-level turnover(s) that would otherwise be missing from the box score entirely.
Validating PBP-reconstructed final scores against the schedule:

  St. Thomas (TX) Celts 2025-11-14: PBP 73-81 vs. schedule 73.0-81.0 [OK]
    UW-Whitewater starters: Brock Marino, Collin Madson, Isaac Verges, JR Lukenbill, Luke Bara
    St. Thomas (TX) Celts starters: Angel Johnson, Charles Gitonga, Corey Thompson, Nathan Kongolo, Nicholas Buffalo
  Eureka Red Devils 2025-11-15: PBP 116-73 vs. schedule 116.0-73.0 [OK]
    UW-Whitewater starters: Brock Marino, Collin Madson, Isaac Verges, JR Lukenbill, Luke Bara
    Eureka Red Devils starters: Andrew Coker, Ben Carter, Colin DeLaere, Jaxson Provost, Micah Bruer
  Ripon Red Hawks 2025-11-07: PBP 76-58 vs. schedule 76.0-58.0 [OK]
    UW-Whitewater starters: Brock Marino, Collin Madson, Isaac Verges, JR Lukenbill, Luke Bara
    Ripon Red Hawks starters: CJ Brown, Kolby Williams, Michael Asman, Sam Leo


### Diagnostic: which played/scouted games are missing a box score, and why

Cross-references every game in `uww_team_schedule` (UWW's own scouted + played games -- the set the
Streamlit app expects a box score for) against `pbp_box_score`, and prints a specific reason for any gap:
no local `_pbp` file was found for that game, a live-scrape wasn't attempted (no FastScout credentials),
or the game's `opponent` value in `pbp_events` doesn't match how it's spelled in the schedule (the PBP
builder derives `opponent` from the PBP filename itself, via `opponent_from_pbp_filename()` -- a mismatch
there means the game's events exist in `pbp_events` but never get matched up with its schedule row anywhere
downstream, including in the exported CSVs the app reads from). Run this after the box-score cell above to
catch a silently-missing game before it shows up as a mystery in the app instead of here.

In [100]:
# --- Diagnostic: reconcile uww_team_schedule (what the app expects a box score for) against pbp_box_score
# (what actually got built) -- prints a specific reason for every gap instead of leaving it to be discovered
# later as a blank "Box Score" section in the Streamlit app.
if uww_team_schedule.empty:
    print("uww_team_schedule is empty -- nothing to reconcile.")
else:
    _pbp_opponents_found = set(pbp_events["opponent"].dropna().unique()) if not pbp_events.empty else set()
    _box_opponents_found = set(pbp_box_score["opponent"].dropna().unique()) if not pbp_box_score.empty else set()
    _has_creds = bool(fastscout_username and fastscout_password)

    _gaps = []
    for _, _g in uww_team_schedule.iterrows():
        if _g.get("Upcoming") == "Yes":
            continue  # hasn't been played yet -- not expected to have a box score
        _opp_full = str(_g.get("opponent", ""))
        _opp_short = next((s for s in scouted_opponents if re.search(re.escape(s), _opp_full, re.IGNORECASE)), None)
        if _opp_short is None:
            _gaps.append((_opp_full, "not in scouted_opponents -- no scout report was matched for this game at all"))
            continue
        if _opp_short in _box_opponents_found:
            continue  # has a box score -- nothing to report
        _g_date = parse_schedule_date(_g["date"], uww_season_start_year) if pd.notna(_g.get("date")) else None
        _local_pbp = glob.glob(f"{volume_dir}/*{_opp_short}*_pbp.*") if _g_date is None else glob.glob(
            f"{volume_dir}/{_g_date.month}_{_g_date.day}_{_g_date.strftime('%y')}*{_opp_short}*_pbp.*"
        )
        if _opp_short in _pbp_opponents_found:
            reason = (
                f"PBP events exist under opponent name mismatch -- check whether any of "
                f"{sorted(_pbp_opponents_found)} was actually meant to be '{_opp_short}' "
                f"(opponent_from_pbp_filename() derives this from the _pbp file's own filename)"
            )
        elif _local_pbp:
            reason = f"a local PBP file exists ({[os.path.basename(p) for p in _local_pbp]}) but produced no box score rows -- check its parsing for errors/UNCLASSIFIED events"
        elif not _has_creds:
            reason = "no local _pbp file found, and no FASTSCOUT_USERNAME/FASTSCOUT_PASSWORD were set to attempt a live scrape"
        else:
            reason = "no local _pbp file found, and the live-scrape fallback either wasn't attempted or failed -- check the earlier cell's output for this opponent's name"
        _gaps.append((_opp_short, reason))

    if not _gaps:
        print("Every played, scouted game in uww_team_schedule has a box score in pbp_box_score. No gaps found.")
    else:
        print(f"{len(_gaps)} played game(s) are missing a box score:\n")
        for opp, reason in _gaps:
            print(f"  {opp}: {reason}")


Every played, scouted game in uww_team_schedule has a box score in pbp_box_score. No gaps found.



### Reconstruct a single-game box score by 5-man lineup

Same reconstruction as the cell above, but aggregated by the 5-man lineup on the floor (`uww_lineup`/`opp_lineup`) instead of by individual player -- validated the same way, against the schedule's final scores.

In [102]:
# --- Single-game box score aggregated by 5-MAN LINEUP instead of by individual player -----------------------
lineup_events = pbp_events[pbp_events["event_type"] != "period_marker"].copy()
lineup_events["lineup"] = lineup_events.apply(
    lambda row: row["uww_lineup"] if row["team"] == "UW-Whitewater" else row["opp_lineup"], axis=1
)

lineup_events["points"] = lineup_events.apply(
    lambda row: int(row["shot_type"]) if row["event_type"] == "made_shot" else (1 if row["event_type"] == "free_throw_made" else 0),
    axis=1,
)
lineup_events["is_fgm"] = lineup_events["event_type"] == "made_shot"
lineup_events["is_fga"] = lineup_events["event_type"].isin(["made_shot", "missed_shot"])
lineup_events["is_3pm"] = lineup_events["is_fgm"] & (lineup_events["shot_type"] == "3")
lineup_events["is_3pa"] = lineup_events["is_fga"] & (lineup_events["shot_type"] == "3")
lineup_events["is_ftm"] = lineup_events["event_type"] == "free_throw_made"
lineup_events["is_fta"] = lineup_events["event_type"].isin(["free_throw_made", "free_throw_missed"])
lineup_events["is_oreb"] = lineup_events["event_type"].isin(["rebound_offensive", "team_deadball_rebound_offensive"])
lineup_events["is_dreb"] = lineup_events["event_type"].isin(["rebound_defensive", "team_deadball_rebound_defensive"])
lineup_events["is_ast"] = lineup_events["event_type"] == "assist"
lineup_events["is_stl"] = lineup_events["event_type"] == "steal"
lineup_events["is_blk"] = lineup_events["event_type"] == "block"
lineup_events["is_to"] = lineup_events["event_type"] == "turnover"
lineup_events["is_pf"] = lineup_events["event_type"] == "foul"

lineup_box_score = lineup_events.dropna(subset=["lineup"]).groupby(["opponent", "game_date", "team", "lineup"]).agg(
    PTS=("points", "sum"), FGM=("is_fgm", "sum"), FGA=("is_fga", "sum"),
    FG3M=("is_3pm", "sum"), FG3A=("is_3pa", "sum"), FTM=("is_ftm", "sum"), FTA=("is_fta", "sum"),
    OREB=("is_oreb", "sum"), DREB=("is_dreb", "sum"), AST=("is_ast", "sum"), STL=("is_stl", "sum"),
    BLK=("is_blk", "sum"), TO=("is_to", "sum"), PF=("is_pf", "sum"),
).reset_index()
lineup_box_score["REB"] = lineup_box_score["OREB"] + lineup_box_score["DREB"]
lineup_box_score["FG%"] = (100 * lineup_box_score["FGM"] / lineup_box_score["FGA"]).round(1)
lineup_box_score["3P%"] = (100 * lineup_box_score["FG3M"] / lineup_box_score["FG3A"]).round(1)
lineup_box_score["FT%"] = (100 * lineup_box_score["FTM"] / lineup_box_score["FTA"]).round(1)

stint_source = pbp_events[pbp_events["event_type"] != "period_marker"].sort_values(GAME_KEYS + ["event_order"]).copy()
prev_uww_lineup = stint_source.groupby(GAME_KEYS, dropna=False)["uww_lineup"].shift(1)
prev_opp_lineup = stint_source.groupby(GAME_KEYS, dropna=False)["opp_lineup"].shift(1)
stint_changed = (stint_source["uww_lineup"] != prev_uww_lineup) | (stint_source["opp_lineup"] != prev_opp_lineup)
stint_source["stint_num"] = stint_changed.fillna(True).groupby([stint_source[k] for k in GAME_KEYS]).cumsum()
stint_source["prev_uww_score"] = stint_source.groupby(GAME_KEYS, dropna=False)["uww_score"].shift(1).fillna(0)
stint_source["prev_opp_score"] = stint_source.groupby(GAME_KEYS, dropna=False)["opp_score"].shift(1).fillna(0)
# The game clock never runs backwards inside a period, so take a running minimum before differencing.
# Without it, ANY out-of-order row makes (prev - now) positive on the way back down and .clip(lower=0)
# keeps that phantom elapsed time while discarding the compensating negative -- silently inventing
# minutes. Grouping on the game (not just the opponent) is what stops two meetings interleaving here.
stint_source["clock"] = stint_source.groupby(GAME_KEYS + ["period"], dropna=False)["time_remaining_seconds"].cummin()
stint_source["prev_time_remaining_seconds"] = stint_source.groupby(GAME_KEYS + ["period"], dropna=False)["clock"].shift(1)
stint_source["seconds_elapsed"] = (stint_source["prev_time_remaining_seconds"] - stint_source["clock"]).clip(lower=0).fillna(0)

stints_for_box = stint_source.groupby(GAME_KEYS + ["stint_num", "uww_lineup", "opp_lineup"]).agg(
    end_uww_score=("uww_score", "last"), end_opp_score=("opp_score", "last"),
    start_prev_uww_score=("prev_uww_score", "first"), start_prev_opp_score=("prev_opp_score", "first"),
    stint_seconds=("seconds_elapsed", "sum"),
).reset_index()
stints_for_box["uww_margin_change"] = (
    (stints_for_box["end_uww_score"] - stints_for_box["start_prev_uww_score"]) - (stints_for_box["end_opp_score"] - stints_for_box["start_prev_opp_score"])
)
stints_for_box["stint_minutes"] = (stints_for_box["stint_seconds"] / 60).round(2)

uww_minutes_margin = stints_for_box.groupby(GAME_KEYS + ["uww_lineup"]).agg(
    MIN=("stint_minutes", "sum"), **{"+/-": ("uww_margin_change", "sum")}
).reset_index().rename(columns={"uww_lineup": "lineup"})
uww_minutes_margin["team"] = "UW-Whitewater"

opp_minutes_margin = stints_for_box.groupby(GAME_KEYS + ["opp_lineup"]).agg(
    MIN=("stint_minutes", "sum"), **{"+/-": ("uww_margin_change", lambda s: -s.sum())}
).reset_index().rename(columns={"opp_lineup": "lineup"})
opp_minutes_margin["team"] = opp_minutes_margin["opponent"]

minutes_margin = pd.concat([uww_minutes_margin, opp_minutes_margin], ignore_index=True)
# CONFIRMED BUG (fixed here): for the first game of the season, lineup reconstruction can come back
# with uww_lineup/opp_lineup entirely NaN for the whole game (e.g. no logged substitution events yet to
# build a lineup sequence from). groupby() drops an all-NaN key entirely, so uww_minutes_margin/
# opp_minutes_margin end up EMPTY -- and pandas infers an empty/all-NaN "lineup" column as float64
# rather than object (string). lineup_box_score's own "lineup" column is always a real object/string
# dtype (built from dropna()'d string values earlier in this cell), so the two sides' dtypes disagreed
# and the merge below raised "ValueError: You are trying to merge on float64 and object columns for key
# 'lineup'." Casting both sides to the same dtype right before merging fixes this regardless of which
# side (if either) ends up empty.
lineup_box_score["lineup"] = lineup_box_score["lineup"].astype(object)
minutes_margin["lineup"] = minutes_margin["lineup"].astype(object)
lineup_box_score = lineup_box_score.merge(minutes_margin, on=GAME_KEYS + ["team", "lineup"], how="left")

lineup_team_totals = lineup_box_score.groupby(GAME_KEYS + ["team"])["PTS"].sum().reset_index()
print("Validating lineup-level PTS totals against the schedule:\n")
for (opponent, game_date) in pbp_events[GAME_KEYS].dropna(subset=["opponent"]).drop_duplicates().itertuples(index=False):
    cand = schedule[schedule["opponent"].str.contains(re.escape(opponent), case=False, na=False)].copy()
    cand = cand[cand["date"].apply(parse_schedule_date) == game_date] if game_date is not None else cand
    if cand.empty:
        continue
    sched_row = cand.iloc[0]
    sel = (lineup_team_totals["opponent"] == opponent) & (lineup_team_totals["game_date"] == game_date)
    uww_pts = lineup_team_totals[sel & (lineup_team_totals["team"] == "UW-Whitewater")]["PTS"]
    opp_pts = lineup_team_totals[sel & (lineup_team_totals["team"] == opponent)]["PTS"]
    uww_val = int(uww_pts.iloc[0]) if not uww_pts.empty else None
    opp_val = int(opp_pts.iloc[0]) if not opp_pts.empty else None
    status = "OK" if (uww_val, opp_val) == (sched_row["team_score"], sched_row["opponent_score"]) else "MISMATCH -- check lineup attribution for this game"
    print(f"  {opponent} {game_date}: lineup box score {uww_val}-{opp_val} vs. schedule {sched_row['team_score']}-{sched_row['opponent_score']} [{status}]")

print(lineup_box_score.sort_values(["opponent", "team", "PTS"], ascending=[True, True, False])[[
    "opponent", "game_date", "team", "lineup", "MIN", "+/-", "PTS", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA",
    "OREB", "DREB", "REB", "AST", "STL", "BLK", "TO", "PF", "FG%", "3P%", "FT%",
]])

Validating lineup-level PTS totals against the schedule:

  St. Thomas (TX) Celts 2025-11-14: lineup box score 73-81 vs. schedule 73.0-81.0 [OK]
  Eureka Red Devils 2025-11-15: lineup box score 116-73 vs. schedule 116.0-73.0 [OK]
  Ripon Red Hawks 2025-11-07: lineup box score 76-58 vs. schedule 76.0-58.0 [OK]
                  opponent   game_date               team  \
3        Eureka Red Devils  2025-11-15  Eureka Red Devils   
1        Eureka Red Devils  2025-11-15  Eureka Red Devils   
12       Eureka Red Devils  2025-11-15  Eureka Red Devils   
2        Eureka Red Devils  2025-11-15  Eureka Red Devils   
10       Eureka Red Devils  2025-11-15  Eureka Red Devils   
..                     ...         ...                ...   
94   St. Thomas (TX) Celts  2025-11-14      UW-Whitewater   
100  St. Thomas (TX) Celts  2025-11-14      UW-Whitewater   
106  St. Thomas (TX) Celts  2025-11-14      UW-Whitewater   
107  St. Thomas (TX) Celts  2025-11-14      UW-Whitewater   
108  St. Thomas (T


### Aggregate the upcoming opponent's 5-man lineup season box scores

Uses `pbp_events_upcoming` (the opponent's own games, before facing Whitewater) to build their season-aggregated 5-man lineup box scores -- their body of work reconstructed the same way as the cells above, just on the opponent's own games instead of UWW's.

In [104]:
# --- Season-aggregated 5-man lineup box scores for the UPCOMING OPPONENT -----------------------------------
# Uses pbp_events_upcoming to build the upcoming opponent's 5-man lineup season box scores -- their body of
# work BEFORE they face Whitewater, reconstructed the same way as the cells above but on the opponent's own games.
TEAM_LEVEL_UPCOMING = {"team_deadball_rebound_offensive", "team_deadball_rebound_defensive", "timeout"}

if pbp_events_upcoming.empty:
    upcoming_lineup_season = pd.DataFrame()
    print(f"No PBP data available for {upcoming_opponent_short} -- "
          "lineup season box scores will populate once PBP files are uploaded.")
else:
    real_up = pbp_events_upcoming[
        pbp_events_upcoming["player"].notna() & (~pbp_events_upcoming["event_type"].isin(TEAM_LEVEL_UPCOMING))
    ].sort_values("event_order")

    def _starting_lineup(team_events):
        first_seen = team_events.groupby("player").first()
        return set(first_seen[first_seen["event_type"] != "sub_in"].index)

    self_lineup_col = pd.Series(index=pbp_events_upcoming.index, dtype=object)
    their_lineup_col = pd.Series(index=pbp_events_upcoming.index, dtype=object)

    # The upcoming opponent has the same problem on their own schedule -- e.g. three meetings with
    # one conference rival all carry the same `opponent` value.
    for (opp_name, opp_game_date), opp_rows in pbp_events_upcoming.groupby(GAME_KEYS, dropna=False):
        opp_real = real_up[(real_up["opponent"] == opp_name) & (real_up["game_date"] == opp_game_date)]
        period_starts = opp_real.dropna(subset=["period"]).groupby("period")["event_order"].min().sort_values()
        periods_in_order = period_starts.index.tolist()

        for team_label, target_col in [(upcoming_opponent_short, self_lineup_col), (opp_name, their_lineup_col)]:
            team_events = opp_real[opp_real["team"] == team_label]
            changes = []
            for period in periods_in_order:
                period_events = team_events[team_events["period"] == period].sort_values("event_order")
                starters = _starting_lineup(period_events)
                current = set(starters)
                changes.append((period_starts[period] - 1, frozenset(current)))
                sub_events = period_events[period_events["event_type"].isin(["sub_in", "sub_out"])].sort_values("event_order")
                for _, row in sub_events.iterrows():
                    if row["event_type"] == "sub_in":
                        current.add(row["player"])
                    else:
                        current.discard(row["player"])
                    if len(current) == 5:
                        changes.append((row["event_order"], frozenset(current)))

            if changes:
                changes_df = pd.DataFrame(changes, columns=["event_order", "lineup"]).sort_values("event_order")
                target_rows = opp_rows.sort_values("event_order")
                merged = pd.merge_asof(target_rows[["event_order"]], changes_df, on="event_order", direction="backward")
                lineup_strings = merged["lineup"].apply(lambda s: ", ".join(sorted(s)) if isinstance(s, frozenset) else None)
                lineup_strings.index = target_rows.index
                target_col.loc[target_rows.index] = lineup_strings

    pbp_up = pbp_events_upcoming.copy()
    pbp_up["self_lineup"] = self_lineup_col
    pbp_up["their_lineup"] = their_lineup_col

    # CONFIRMED CHANGE (requested): self_lineup used to live only on this cell's local `pbp_up` copy, so
    # nothing downstream of THIS cell could look up "which 5-man unit was on the floor" for a given
    # possession -- including the play-calls cell, which needs exactly that to cross-reference a decoded
    # play call with the personnel grouping running it. Persisted onto the real pbp_events_upcoming here so
    # it survives past this cell. (uww_lineup/opp_lineup already work the same way for UWW's own games --
    # see the "On-court 5-man lineups" cell earlier -- this just closes the equivalent gap on the opponent
    # side, which is why the play-calls cell was moved to run after this one.)
    pbp_events_upcoming["self_lineup"] = self_lineup_col

    stint_src = pbp_up[pbp_up["event_type"] != "period_marker"].sort_values(GAME_KEYS + ["event_order"]).copy()
    prev_self = stint_src.groupby(GAME_KEYS, dropna=False)["self_lineup"].shift(1)
    prev_their = stint_src.groupby(GAME_KEYS, dropna=False)["their_lineup"].shift(1)
    stint_changed = (stint_src["self_lineup"] != prev_self) | (stint_src["their_lineup"] != prev_their)
    stint_src["stint_num"] = stint_changed.fillna(True).groupby([stint_src[k] for k in GAME_KEYS]).cumsum()
    stint_src["prev_self_score"] = stint_src.groupby(GAME_KEYS, dropna=False)["uww_score"].shift(1).fillna(0)
    stint_src["prev_their_score"] = stint_src.groupby(GAME_KEYS, dropna=False)["opp_score"].shift(1).fillna(0)
    stint_src["clock"] = stint_src.groupby(GAME_KEYS + ["period"], dropna=False)["time_remaining_seconds"].cummin()
    stint_src["prev_time_remaining_seconds"] = stint_src.groupby(GAME_KEYS + ["period"], dropna=False)["clock"].shift(1)
    stint_src["seconds_elapsed"] = (stint_src["prev_time_remaining_seconds"] - stint_src["clock"]).clip(lower=0).fillna(0)

    stints = stint_src.groupby(GAME_KEYS + ["stint_num", "self_lineup"]).agg(
        end_self_score=("uww_score", "last"), end_their_score=("opp_score", "last"),
        start_prev_self_score=("prev_self_score", "first"), start_prev_their_score=("prev_their_score", "first"),
        stint_seconds=("seconds_elapsed", "sum"),
    ).reset_index()
    stints["margin_change"] = (
        (stints["end_self_score"] - stints["start_prev_self_score"]) -
        (stints["end_their_score"] - stints["start_prev_their_score"])
    )
    stints["stint_minutes"] = (stints["stint_seconds"] / 60).round(2)

    minutes_margin = stints.groupby(GAME_KEYS + ["self_lineup"]).agg(
        MIN=("stint_minutes", "sum"), **{"+/-": ("margin_change", "sum")}
    ).reset_index().rename(columns={"self_lineup": "lineup"})

    lu_events = pbp_up[
        (pbp_up["team"] == upcoming_opponent_short) & pbp_up["self_lineup"].notna()
    ].copy()
    lu_events["lineup"] = lu_events["self_lineup"]
    lu_events["points"] = lu_events.apply(
        lambda r: int(r["shot_type"]) if r["event_type"] == "made_shot" else (1 if r["event_type"] == "free_throw_made" else 0), axis=1)
    lu_events["is_fgm"] = lu_events["event_type"] == "made_shot"
    lu_events["is_fga"] = lu_events["event_type"].isin(["made_shot", "missed_shot"])
    lu_events["is_3pm"] = lu_events["is_fgm"] & (lu_events["shot_type"] == "3")
    lu_events["is_3pa"] = lu_events["is_fga"] & (lu_events["shot_type"] == "3")
    lu_events["is_ftm"] = lu_events["event_type"] == "free_throw_made"
    lu_events["is_fta"] = lu_events["event_type"].isin(["free_throw_made", "free_throw_missed"])
    lu_events["is_oreb"] = lu_events["event_type"].isin(["rebound_offensive", "team_deadball_rebound_offensive"])
    lu_events["is_dreb"] = lu_events["event_type"].isin(["rebound_defensive", "team_deadball_rebound_defensive"])
    lu_events["is_ast"] = lu_events["event_type"] == "assist"
    lu_events["is_stl"] = lu_events["event_type"] == "steal"
    lu_events["is_blk"] = lu_events["event_type"] == "block"
    lu_events["is_to"] = lu_events["event_type"] == "turnover"
    lu_events["is_pf"] = lu_events["event_type"] == "foul"

    per_game_lineup = lu_events.groupby(GAME_KEYS + ["lineup"], as_index=False).agg(
        PTS=("points", "sum"), FGM=("is_fgm", "sum"), FGA=("is_fga", "sum"),
        FG3M=("is_3pm", "sum"), FG3A=("is_3pa", "sum"), FTM=("is_ftm", "sum"), FTA=("is_fta", "sum"),
        OREB=("is_oreb", "sum"), DREB=("is_dreb", "sum"), AST=("is_ast", "sum"), STL=("is_stl", "sum"),
        BLK=("is_blk", "sum"), TO=("is_to", "sum"), PF=("is_pf", "sum"),
    )
    per_game_lineup["REB"] = per_game_lineup["OREB"] + per_game_lineup["DREB"]
    per_game_lineup = per_game_lineup.merge(minutes_margin, on=GAME_KEYS + ["lineup"], how="left")

    upcoming_lineup_season = per_game_lineup.groupby("lineup").agg(
        GP=("game_date", "nunique"),
        MIN=("MIN", "sum"),
        **{"+/-": ("+/-", "sum")},
        PTS=("PTS", "sum"), FGM=("FGM", "sum"), FGA=("FGA", "sum"),
        FG3M=("FG3M", "sum"), FG3A=("FG3A", "sum"), FTM=("FTM", "sum"), FTA=("FTA", "sum"),
        OREB=("OREB", "sum"), DREB=("DREB", "sum"), REB=("REB", "sum"),
        AST=("AST", "sum"), STL=("STL", "sum"), BLK=("BLK", "sum"), TO=("TO", "sum"), PF=("PF", "sum"),
    ).reset_index()
    upcoming_lineup_season["FG%"] = (100 * upcoming_lineup_season["FGM"] / upcoming_lineup_season["FGA"]).round(1)
    upcoming_lineup_season["3P%"] = (100 * upcoming_lineup_season["FG3M"] / upcoming_lineup_season["FG3A"]).round(1)
    upcoming_lineup_season["FT%"] = (100 * upcoming_lineup_season["FTM"] / upcoming_lineup_season["FTA"]).round(1)
    upcoming_lineup_season["MIN"] = upcoming_lineup_season["MIN"].round(1)
    upcoming_lineup_season = upcoming_lineup_season.sort_values("MIN", ascending=False).reset_index(drop=True)

    print(f"{upcoming_opponent_short} season 5-man lineup box scores "
          f"({upcoming_lineup_season['GP'].max()} game(s) of PBP data, "
          f"{len(upcoming_lineup_season)} distinct units):")
    print(upcoming_lineup_season[[
        "lineup", "GP", "MIN", "+/-", "PTS", "FGM", "FGA", "FG%",
        "FG3M", "FG3A", "3P%", "FTM", "FTA", "FT%",
        "OREB", "DREB", "REB", "AST", "STL", "BLK", "TO", "PF",
    ]])

Aurora Spartans season 5-man lineup box scores (4 game(s) of PBP data, 73 distinct units):
                                                                                lineup  \
0       Devon Richardson, Gevon Grant, Jeffery Hillmer, Juan Madrigal, Zerrick Johnson   
1   Bryden Gryzmala, Devon Richardson, Jeffery Hillmer, Juan Madrigal, Zerrick Johnson   
2                  Cullen Rauls, Gevon Grant, Larry Carthan, Mekhi Doby, Robert Hutson   
3   Bryden Gryzmala, Devon Richardson, Jeffery Hillmer, Robert Hutson, Zerrick Johnson   
4      Bryden Gryzmala, Jeffery Hillmer, Juan Madrigal, Larry Carthan, Zerrick Johnson   
..                                                                                 ...   
68    Devon Richardson, Jeffery Hillmer, Juan Madrigal, Robert Hutson, Zerrick Johnson   
69          Jeffery Hillmer, Juan Madrigal, Larry Carthan, Mekhi Doby, Zerrick Johnson   
70        Devon Richardson, Gevon Grant, Juan Madrigal, Robert Hutson, Zerrick Johnson   
71       


### Play calls: decode `uww_plays.csv` and `opponent_plays.csv` and join them onto the play-by-play

Both files live in `INPUT_DIR` (one of each, always). Each row is one tagged possession; its **Title** holds the play call in shorthand plus where it was run (`OK State- DHO RS- Pat Miller`, `41- BS LS- Reject`, `Blob-Box-Curl`).

* **Decode:** every title becomes situation (Half court / BLOB / SLOB / ATO / Opening set), set or formation, actions in order, primary action, starting spot, finish spot and any player named, with a `decode_quality` (Clean / Partial / Needs review / No call). The raw title is always kept.
* **Join:** each clip matches one play-by-play event of the offense, by Synergy String first, then player + clock, then clock + result. Written onto `pbp_events` (UWW) and `pbp_events_upcoming` (opponent).
* **Points:** from the play-by-play at that clock stamp when matched (free throws count), else from the Result tag.
* **Exports:** `uww_play_calls`, `uww_play_call_summary`, `uww_play_glossary`; UWW clips also feed `uww_coach_notes` for the app's existing play-call analytics.

In [106]:
# --- Play calls: uww_plays.csv + opponent_plays.csv, decoded and joined onto the play-by-play ---------------
# CONFIRMED CHANGE (requested): play-call data now comes from exactly two files in INPUT_DIR --
#   uww_plays.csv        UW-Whitewater's own tagged offensive possessions, every game this season
#   opponent_plays.csv   the UPCOMING opponent's tagged offensive possessions from their prior games
# Both are the video-tagging tool's clip export (one row per possession: Title, Result, Date, Pd., Clock,
# Player, Team, Synergy String, ...). The play call lives in "Title", typed as shorthand with where on the
# floor it was run ("OK State- DHO RS- Pat Miller", "41- BS LS- Reject", "Blob-Box-Curl"), so every title is
# DECODED below into situation / set / actions / location, and the raw title is always kept beside it.
#
# JOIN. Each clip is matched to one play-by-play event of the team on offense, scored in this order:
#   1. same game date + period, and the clip's Synergy String equals the event's video_description
#      (the _video.mhtml export carries the identical string, so this is an exact fingerprint)
#   2. same player, clock within _PL_CLOCK_TOL seconds, and an event type the clip's Result allows
#   3. clock within 3 seconds and a compatible event type, when the player name is spelled differently
# One event takes at most one clip. Unmatched clips are NOT dropped: they stay in uww_play_calls.csv with
# matched_event=False, so the play-call breakdowns still count them -- only their pbp-derived points are
# replaced by the points the Result tag implies.
#
# POINTS. A matched clip scores what the play-by-play says the offense scored at that clock stamp (made
# shots plus free throws, so and-ones and trips to the line count). An unmatched clip falls back to its
# Result tag, and a foul with no play-by-play behind it is left unknown rather than guessed.
#
# This supersedes the old "*_plays_*.csv" season log path in the recap cell above (that glob is removed).

_PL_UWW_FILE = os.path.join(INPUT_DIR, "uww_plays.csv")
_PL_OPP_FILE = os.path.join(INPUT_DIR, "opponent_plays.csv")
_PL_CLOCK_TOL = 12          # seconds; clip clocks are typed at the moment of the result, give or take a stoppage
_PL_UWW = "UW-Whitewater"
_PL_EVENT_COLS = ["play_call", "play_series", "play_situation", "play_actions", "primary_action",
                  "play_location", "finish_spot", "play_title", "play_decode_quality",
                  "defense_type", "defense_press", "defense_press_formation", "defense_coverage",
                  "possession_side", "defense_faced", "defense_played", "coverage_faced", "coverage_played"]
_pl_problems = []

# --- Play-title decoder ------------------------------------------------------------------------------------
# The "Title" column is the tagger's shorthand, typed live and inconsistently:
#   "OK State- DHO RS- Sci Sc- STS -RE"   "Blob-Box-Curl"   "5 out- pass 5 -flair"   "41- BS LS- Reject"
# Structure, when there is one: [situation] - [formation / named set] - [action] - [action] ... - [spot]
# Hyphens are the separator, EXCEPT inside formations ("4-1", "2-1-1") and "Hi-Lo", which are protected first.
# Every clip keeps its raw title next to the decode, and decode_quality says how much to trust it.

_PD_SITUATIONS = [
    (r"\bblob\b", "BLOB"), (r"\bslob\b", "SLOB"), (r"\bato\b", "ATO"), (r"\bopener\b", "Opener"),
]
# Named sets / formations. kind "formation" is an alignment ("5 Out", "4-1"); kind "set" is a named play
# ("Panther", "OK State") and outranks a formation when naming the call.
_PD_SETS = [
    (r"\b5\s*out\b", "5 Out", "formation"),
    (r"\b4~1\b|\b41\b", "4-1", "formation"),
    (r"\b3~2\b|\b32\b", "3-2", "formation"),
    (r"\b2~3\b|\b23\b", "2-3", "formation"),
    (r"\b2~1~1\b", "2-1-1", "formation"),
    (r"\b33\b", "33", "formation"),
    (r"\bhilo\b", "Hi-Lo", "formation"),
    (r"\bhorns\b", "Horns", "formation"),
    (r"\bdiamond\b", "Diamond", "formation"),
    (r"\bbox\b", "Box", "formation"),
    (r"\bline\b|\blline\b|\bl\b", "Line", "formation"),
    (r"\bstairs\b", "Stairs", "set"),
    (r"\bokstate\b", "OK State", "set"),
    (r"\bpanther\b", "Panther", "set"),
    (r"\bcheetah\b", "Cheetah", "set"),
    (r"\bflop\b", "Flop", "set"),
    (r"\bhighway\b", "Highway", "set"),
    (r"\bpistol\b", "Pistol", "set"),
    (r"\bmonty\b", "Monty", "set"),
]
# Actions. rank 1 = a named/signature action that identifies the play; 2 = a screen or handoff action;
# 3 = connective movement (pass, swing, follow) that describes HOW the ball got there, not what the play is.
_PD_ACTIONS = [
    (r"\bpat\s*miller\b", "Pat Miller", 1), (r"\bgren(?:ade|dae)\b", "Grenade", 1),
    (r"\bhammer\b", "Hammer", 1), (r"\bzoom\b", "Zoom", 1), (r"\bt?twirl\b", "Twirl", 1),
    (r"\bbreddy\b", "Breddy", 1), (r"\bricky\b|\bric\b", "Ricky", 1), (r"\bivo\b", "IVO", 1),
    (r"\bspain\b", "Spain", 1), (r"\bmotion\b", "Motion", 1), (r"\blob\b", "Lob", 1), (r"\brip\b", "Rip", 1),
    (r"\bxai?vi?er\s*screen\b", "Xavier Screen", 1), (r"\bshoulder\s*screen\b", "Shoulder Screen", 1),
    (r"\bbig\s*on\s*big\b", "Big-on-Big", 1), (r"\bexh?ac?h?n?g?e?\b|\bexchange\b", "Exchange", 2),
    (r"\bdho\b|\bdh\b|\bhandoff\b|\bhand\s*off\b", "DHO", 2),
    (r"\bbs\s*scree?n?\b|\bball\s*screens?\b|\bbs\b", "Ball Screen", 2),
    (r"\bdown\s*screens?\b|\bds\b", "Down Screen", 2), (r"\bup\s*screen\b", "Up Screen", 2),
    (r"\bstagger\b", "Stagger", 2), (r"\bsts\b", "Screen the Screener", 2),
    (r"\bsci(?:ssors)?\b(?:\s*sc\b)?", "Scissors", 2), (r"\b\d\s*screen\s*\d\b", "Screen", 2),
    (r"\bfla(?:i|)re?\b|\bflairs?\b|\bflares?\b", "Flare", 2), (r"\bcurl\b", "Curl", 2),
    (r"\bdd\b", "Double Drag", 2), (r"\breject\b|\brj\b", "Reject", 2), (r"\bslip\b", "Slip", 2),
    (r"\bpop\b", "Pop", 2), (r"\bget\b", "Get", 2), (r"\bgive\b", "Give", 2),
    (r"\biso\b", "ISO", 2), (r"\bpost\b|\bpt\b", "Post Touch", 2), (r"\bclear\b", "Clear", 3),
    (r"\bbc\s*cut\b", "Backcut", 2), (r"\bcut\b", "Cut", 3), (r"\bflash\b", "Flash", 3),
    (r"\bfollow\b", "Follow", 3), (r"\bswing\b", "Swing", 3), (r"\bpass\b", "Pass", 3),
    (r"\bdribble\b|\bpush\s*thru\b", "Dribble Entry", 3), (r"\bstand\b", "Stand", 3),
]
_PD_LOCATIONS = [
    (r"\blw\b", "Left Wing"), (r"\brw\b", "Right Wing"), (r"\blc\b", "Left Corner"), (r"\brc\b", "Right Corner"),
    (r"\bls\b|\bleft\s*side\b|\bleft\b", "Left Side"), (r"\brs\b|\bright\s*side\b|\bright\b", "Right Side"),
    (r"\blb\b", "Left Block"), (r"\brb\b", "Right Block"), (r"\ble\b", "Left Elbow"), (r"\bre\b", "Right Elbow"),
    (r"\btop(?:\s*key)?\b", "Top"), (r"\bother\s*side\b", "Opposite Side"), (r"\bwing\b", "Wing"),
]
_PD_REVIEW = r"\brewatch\b|\btbd\b|\bmess\s*up\b|\?"
_PD_IGNORE = r"\bset\b|\bhalf\b|\bplay\b|\bgl\b|\bout\b|\baround\b"

# --- Defense-tag decoder (NEW: coaches' updated Title logic) ------------------------------------------------
# Titles now also carry the DEFENSE this team's offense faced on the clip -- e.g.
#   "5 OUT- PASS- DS- CURL: M2M SOFT HEDGE"   "UWW M2M D- BLOB"   "FLOW: M2M PRESS- SWITCH"
#   "5 OUT: M2M-SWITCH"   "OVER-CHEETAH: M2M D- DROP"   "Flow: M2M: Deny"
# In uww_plays.csv that's the OPPONENT's defense while UWW ran the play; in opponent_plays.csv it's the
# defense the upcoming opponent faced in their own prior game. Extracted as its own set of fields and
# blanked out of the working text before the offense decoder below runs, so "m2m", "press", "switch" etc.
# never show up as unrecognized leftover text (which used to knock otherwise-clean titles down to Partial).
_PD_PRESS_FORMATIONS = [
    (r"\b1\s*-\s*2\s*-\s*1\s*-\s*1\b", "1-2-1-1 Press"),
    (r"\b1\s*-\s*2\s*-\s*2\b", "1-2-2 Press"),
    # Fixed-width look-around so a stray "2-1-2?" (an unrelated, already-flagged-for-review guess) doesn't
    # get misread as a "1-2" press just because "1-2" appears as a substring of a longer digit run.
    (r"(?<!\d-)\b1\s*-\s*2\b(?!-\d)", "1-2 Press"),
]
# Checked in order: a named coverage before the bare word it's built from ("soft hedge" before "hedge").
_PD_COVERAGE = [
    (r"\bsoft\s*hedge\b", "Soft Hedge"), (r"\bhedge\b", "Hedge"), (r"\bswitch\b", "Switch"),
    (r"\bice\b", "Ice"), (r"\bdrop\b", "Drop"), (r"\bdeny\b", "Deny"), (r"\bjam\b", "Jam"),
]
_PD_PRESS_WORD = r"\bpress\b|\bpres\b|\bprss\b|\bpresss\b"
# Pure marker noise that only ever rides along with a defense tag ("UWW M2M D", "5 OUT- M2M D") -- the
# team-name prefix and a bare trailing "D" (for "Defense") add nothing once defense_type is captured.
_PD_DEFENSE_NOISE = r"\buww\b|\bd\b"


def decode_defense_tag(title):
    """Pull the opponent-defense fields out of a raw Title: defense_type (Man-to-Man / N-N Zone / Zone),
    defense_press (bool), defense_press_formation (named press alignment, if stated), and defense_coverage
    (ball-screen/on-ball call: Switch, Hedge, Soft Hedge, Ice, Drop, Deny, Jam -- pipe-joined if more than
    one is tagged). Returns (fields dict, remaining text with every matched token blanked to a space) so the
    caller can run the ordinary offense decode on what's left."""
    t = " " + re.sub(r"\s+", " ", str(title or "")).lower().strip() + " "
    out = {"defense_type": None, "defense_press": False, "defense_press_formation": None, "defense_coverage": ""}

    # Zone subtype ("2-3 zone", "3-2 zone", "1-3-1 zone") before the bare word, else Man-to-Man.
    zm = re.search(r"\b(\d)\s*-\s*(\d)(?:\s*-\s*(\d))?\s*zone\b", t)
    if zm:
        out["defense_type"] = "-".join(g for g in zm.groups() if g) + " Zone"
        t = t[:zm.start()] + " " * (zm.end() - zm.start()) + t[zm.end():]
    elif re.search(r"\bzone\b", t):
        out["defense_type"] = "Zone"
        t = re.sub(r"\bzone\b", " ", t)
    if re.search(r"\bm2m\b", t):
        out["defense_type"] = out["defense_type"] or "Man-to-Man"
        t = re.sub(r"\bm2m\b", " ", t)

    for pat, name in _PD_PRESS_FORMATIONS:
        pm = re.search(pat, t)
        if pm:
            out["defense_press"], out["defense_press_formation"] = True, name
            t = t[:pm.start()] + " " * (pm.end() - pm.start()) + t[pm.end():]
            break
    if re.search(_PD_PRESS_WORD, t):
        out["defense_press"] = True
        t = re.sub(_PD_PRESS_WORD, " ", t)

    coverage = []
    for pat, name in _PD_COVERAGE:
        if re.search(pat, t):
            coverage.append(name)
            t = re.sub(pat, " ", t)
    out["defense_coverage"] = " | ".join(dict.fromkeys(coverage))

    if out["defense_type"] or out["defense_press"] or coverage:
        t = re.sub(_PD_DEFENSE_NOISE, " ", t)
    # The colon/semicolon (and an occasional "+" joining two coverages, e.g. "Drop+Press") are the new
    # tagging convention's own delimiter between offense and defense -- strip them here rather than let
    # them fall through to the offense decoder as unrecognized punctuation (which used to knock an
    # otherwise-clean title down to "Partial" on every single title that used the new format).
    t = re.sub(r"[:;+]", " ", t)
    return out, re.sub(r"\s+", " ", t).strip()


def _pd_prep(title):
    t = " " + str(title or "").lower().strip() + " "
    # Only real alignments are protected -- "33- 5 BS" is a set name, a separator, then the five man.
    t = re.sub(r"(?<!\d)([1-3])\s*-\s*([1-3])\s*-\s*([1-3])(?!\d)", r"\1~\2~\3", t)
    t = re.sub(r"(?<!\d)(4\s*-\s*1|3\s*-\s*2|2\s*-\s*3|1\s*-\s*4)(?!\d)",
               lambda m: re.sub(r"\s*-\s*", "~", m.group(1)), t)
    t = re.sub(r"\bhi\s*-?\s*lo(?:w)?\b", "hilo", t)
    t = re.sub(r"\bok\s*st(?:ate)?\b", "okstate", t)
    t = re.sub(r"\b5\s*out\b", "5 out", t)
    t = re.sub(r"\bbig\s+sci\b", "sci", t)
    return t


def decode_play_title(title, known_players=()):
    """Decode one Title into structured fields. Never raises; an undecodable title comes back with
    decode_quality "No call" or "Needs review" and its raw text intact."""
    raw = re.sub(r"\s+", " ", str(title or "")).strip()
    out = {"play_title": raw, "play_call": None, "play_series": None, "play_situation": "Half court",
           "play_formation": None, "play_set": None, "play_actions": "", "primary_action": None,
           "play_location": None, "finish_spot": None, "featured_player": None, "decode_quality": "Clean",
           "decode_note": "", "defense_type": None, "defense_press": False, "defense_press_formation": None,
           "defense_coverage": ""}
    if not raw:
        out.update(decode_quality="No call", decode_note="blank title")
        return out
    low = raw.lower()
    # A title that is only a player's name is the tagger marking who, not what.
    players = {str(p).lower(): str(p) for p in known_players if str(p).strip()}
    if low.strip(" -") in players:
        out.update(decode_quality="No call", featured_player=players[low.strip(" -")],
                   decode_note="title is a player name, not a play")
        return out
    review = bool(re.search(_PD_REVIEW, low))
    # Defense fields come out of the RAW title first (order-independent of where they sit -- before or
    # after a colon, or the whole title), and every matched token is blanked before the offense decoder
    # below ever sees the text, so "m2m", "press", "switch" etc. never land in play_actions or notes.
    defense_fields, defense_stripped = decode_defense_tag(raw)
    out.update(defense_fields)
    prepped = _pd_prep(defense_stripped)
    segments = [s.strip() for s in prepped.split("-") if s.strip()]

    situations, formations, sets, actions, locations, notes = [], [], [], [], [], []
    for seg in segments:
        rest = " " + seg + " "
        found = []  # (position, kind, value, extra)
        for pat, name in _PD_SITUATIONS:
            for m in re.finditer(pat, rest):
                found.append((m.start(), "situation", name, None)); rest = rest[:m.start()] + " " * (m.end() - m.start()) + rest[m.end():]
        for pat, name, kind in _PD_SETS:
            for m in re.finditer(pat, rest):
                found.append((m.start(), kind, name, None)); rest = rest[:m.start()] + " " * (m.end() - m.start()) + rest[m.end():]
        for pat, name, rank in _PD_ACTIONS:
            for m in re.finditer(pat, rest):
                found.append((m.start(), "action", name, rank)); rest = rest[:m.start()] + " " * (m.end() - m.start()) + rest[m.end():]
        for pat, name in _PD_LOCATIONS:
            for m in re.finditer(pat, rest):
                found.append((m.start(), "location", name, None)); rest = rest[:m.start()] + " " * (m.end() - m.start()) + rest[m.end():]
        # A trailing "3" after a spot is the finish ("LW 3" = left-wing three).
        three = re.search(r"\b3\b", rest)
        if three:
            found.append((three.start(), "three", "3", None)); rest = rest[:three.start()] + " " + rest[three.end():]
        rest = re.sub(_PD_REVIEW + "|" + _PD_IGNORE, " ", rest)
        # The five/four man referenced inside an action ("pass 5", "5 BS", "DHO 4") -- kept as context only.
        rest = re.sub(r"\b[1-5]\b", " ", rest)
        leftover = rest.strip()
        if leftover:
            # Anything left is either a player's name ("Madson", "PT BROCK") or shorthand we don't know.
            hit = next((players[p] for p in players
                        if any(w and re.search(r"\b" + re.escape(w) + r"\b", leftover) for w in p.split())), None)
            if hit:
                out["featured_player"] = out["featured_player"] or hit
            else:
                notes.append(leftover)
        seg_kinds = {f[1] for f in found}
        is_last = seg is segments[-1]
        if is_last and seg_kinds and seg_kinds <= {"location", "three"} and not leftover:
            spots = [f[2] for f in sorted(found, key=lambda f: f[0]) if f[1] == "location"]
            if spots:
                out["finish_spot"] = spots[-1]
        for _, kind, value, extra in sorted(found, key=lambda f: f[0]):
            if kind == "situation":
                situations.append(value)
            elif kind == "formation":
                formations.append(value)
            elif kind == "set":
                sets.append(value)
            elif kind == "action":
                actions.append((value, extra))
            elif kind == "location":
                locations.append(value)
            elif kind == "three" and locations:
                out["finish_spot"] = f"{locations[-1]} 3"
            elif kind == "three":
                out["finish_spot"] = "3"

    dedupe = lambda xs: list(dict.fromkeys(xs))  # noqa: E731
    situations, formations, sets = dedupe(situations), dedupe(formations), dedupe(sets)
    oob = next((s for s in situations if s in ("BLOB", "SLOB")), None)
    if oob:
        out["play_situation"] = oob + (" (ATO)" if "ATO" in situations else "")
    elif "ATO" in situations:
        out["play_situation"] = "ATO"
    elif "Opener" in situations:
        out["play_situation"] = "Opening set"
    # "Line" and "Box" only mean a formation under an inbounds play; elsewhere "L" is noise.
    if not oob:
        formations = [f for f in formations if f not in ("Line", "Box")]
    out["play_formation"] = formations[0] if formations else None
    out["play_set"] = sets[0] if sets else None
    out["play_actions"] = " | ".join(dedupe(a for a, _ in actions))
    ranked = sorted(((r, i, a) for i, (a, r) in enumerate(actions)), key=lambda x: (x[0], x[1]))
    out["primary_action"] = ranked[0][2] if ranked and ranked[0][0] <= 2 else (ranked[0][2] if ranked else None)
    out["play_location"] = locations[0] if locations else None

    # Name the call: [OOB] + (named set, else formation) + primary action when it adds something.
    base = out["play_set"] or out["play_formation"]
    parts = []
    if oob:
        parts.append(oob)
    if base:
        parts.append(base)
    if out["primary_action"] and out["primary_action"] != base and (not base or ranked[0][0] <= 2):
        # A named set carries its own identity ("Panther"); only add an action to it when it's a signature one.
        if not (out["play_set"] and ranked[0][0] > 1):
            parts.append(out["primary_action"])
    out["play_call"] = " ".join(parts) if parts else None
    out["play_series"] = (oob if oob else None) or out["play_set"] or out["play_formation"] or (
        "Motion / no set" if actions else None)

    if out["play_call"] is None:
        if situations:
            out["play_call"] = f"{out['play_situation']} (unspecified)"
            out["play_series"] = out["play_situation"].replace(" (ATO)", "")
            out["decode_quality"] = "Partial"
        else:
            out["decode_quality"] = "No call"
    if notes:
        out["decode_note"] = "unrecognized: " + ", ".join(notes)
        if out["decode_quality"] == "Clean":
            out["decode_quality"] = "Partial"
    if review:
        out["decode_quality"] = "Needs review"
        out["decode_note"] = ("tagger flagged it (rewatch/TBD/?/mess up)" + ("; " + out["decode_note"] if out["decode_note"] else ""))
    return out


# Plain-English meaning for every shorthand the decoder knows, and how sure that reading is. Exported so the
# brief and the app can print it -- a coach should be able to see exactly how "RIC" became "Ricky".
PLAY_GLOSSARY = pd.DataFrame([
    ("BLOB / SLOB", "Baseline / sideline out-of-bounds play", "Standard"),
    ("ATO", "After-timeout play", "Standard"),
    ("Opener", "First set of a half", "Inferred"),
    ("5 Out / 4-1 (41) / 3-2 (32) / 2-3 (23)", "Floor alignment: perimeter-post split", "Standard"),
    ("33", "Named set (alignment not stated)", "Inferred"),
    ("HiLo / Hi-Low", "High-low post alignment", "Standard"),
    ("Horns", "Two bigs at the elbows", "Standard"),
    ("Box / Line / Stairs / 2-1-1", "Inbounds alignments", "Standard"),
    ("OK State, Panther, Cheetah, Flop, Highway, Pistol, Monty", "Named sets from the staff's playbook", "Standard"),
    ("DHO / DH / handoff", "Dribble handoff", "Standard"),
    ("BS", "Ball screen", "Standard"),
    ("DS", "Down screen", "Standard"),
    ("STS", "Screen the screener", "Standard"),
    ("Sci / Sci Sc", "Scissors cut off the post", "Inferred"),
    ("Flair / Flare", "Flare screen", "Standard"),
    ("RJ / Reject", "Ball handler rejects the screen", "Standard"),
    ("DD", "Double drag ball screen", "Inferred"),
    ("IVO", "Iverson cut", "Inferred"),
    ("RIC / Ricky", "Named action (ricochet screen) -- confirm with staff", "Inferred"),
    ("Zoom / Twirl / Grenade / Hammer / Pat Miller / Breddy / Rip / Spain", "Named actions", "Standard"),
    ("PT", "Post touch -- confirm with staff", "Inferred"),
    ("BC Cut", "Backdoor cut", "Inferred"),
    ("LW RW LC RC LS RS LB RB LE RE Top", "Left/right wing, corner, side, block, elbow; top of the key", "Standard"),
    ("LW 3 / top key 3", "Where the three-point shot came from", "Standard"),
    ("Rewatch / TBD / ? / mess up", "Tagger flagged the clip for review -- excluded from set rankings", "Standard"),
    # Defense faced (new tagging logic, after the colon: "... : M2M SOFT HEDGE")
    ("M2M", "Man-to-man defense", "Standard"),
    ("2-3 / 3-2 / 1-3-1 Zone / Zone", "Zone defense, with subtype when stated", "Standard"),
    ("Press / 1-2-1-1 / 1-2-2 / 1-2", "Full-court press, with formation when named", "Standard"),
    ("Switch / Hedge / Soft Hedge / Ice / Drop", "Ball-screen coverage call", "Standard"),
    ("Deny / Jam", "On-ball or off-ball denial call", "Inferred"),
], columns=["shorthand", "meaning", "confidence"])


def _pl_norm(text):
    return re.sub(r"[^a-z0-9]", "", str(text or "").lower())


def _pl_result_points(result):
    """Points a Result tag implies, or None when it doesn't say (a drawn foul, 'No Violation')."""
    t = re.sub(r"\s+", " ", str(result or "")).strip().lower()
    if t == "make 3 pts":
        return 3
    if t == "make 2 pts":
        return 2
    m = re.match(r"^(\d) pts$", t)
    if m:
        return int(m.group(1))
    if t.startswith("miss") or t in ("turnover", "shot clock violation", "kicked ball"):
        return 0
    return None


def _pl_compatible(result, event_type):
    t = str(result or "").lower()
    if t.startswith("make"):
        return event_type == "made_shot"
    if t.startswith("miss"):
        return event_type == "missed_shot"
    if "turnover" in t or "violation" in t:
        return event_type == "turnover"
    if "foul" in t or re.match(r"^\d pts$", t):
        return event_type in ("free_throw_made", "free_throw_missed", "made_shot", "foul")
    return False


def _pl_load(path, label):
    if not os.path.exists(path):
        _pl_problems.append(f"{os.path.basename(path)} not found in INPUT_DIR -- {label} play calls skipped")
        return pd.DataFrame()
    try:
        df = pd.read_csv(path)
    except Exception as e:
        _pl_problems.append(f"{os.path.basename(path)}: could not read ({e})")
        return pd.DataFrame()
    need = {"Title", "Result", "Date", "Pd.", "Clock", "Player", "Team"}
    if need - set(df.columns):
        _pl_problems.append(f"{os.path.basename(path)}: missing column(s) {sorted(need - set(df.columns))}")
        return pd.DataFrame()
    df = df[df["Player"].notna() & df["Title"].notna()].copy()
    df["game_date"] = pd.to_datetime(df["Date"], errors="coerce").dt.date
    _future = df["game_date"].notna() & (df["game_date"] >= reference_date.date())
    if int(_future.sum()):
        print(f"  {os.path.basename(path)}: dropped {int(_future.sum())} clip(s) dated on/after {reference_date_str}.")
    df = df[~_future]
    df["period"] = df["Pd."].apply(_recap_period_label)
    df["time_remaining_seconds"] = df["Clock"].apply(_clock_to_seconds)
    df["player"] = df["Player"].astype(str).str.strip()
    df["result"] = df["Result"].astype(str).str.strip()
    df["game_code"] = df.get("Game", pd.Series(index=df.index, dtype=object))
    ss = df.get("Synergy String", pd.Series(index=df.index, dtype=object))
    df["synergy_string"] = ss
    # First step after "<jersey> <name> > " -- how the possession ENDED in Synergy's terms (Spot-Up, P&R Ball
    # Handler, ...). Different from the play call, which is what was drawn up.
    df["synergy_play_type"] = ss.astype(str).str.extract(r"^\s*\d*\s*[^>]+>\s*([^>]+?)\s*(?:>|$)")[0]
    df["clip_number"] = df.get("#")
    return df.reset_index(drop=True)


def _pl_team_label(csv_team, labels):
    """Map a clip's Team ("Aurora University") onto the play-by-play's label ("Aurora Spartans") by word
    overlap. None when nothing overlaps -- a guess here would attribute a possession to the wrong team."""
    words = {w for w in re.findall(r"[a-z]+", str(csv_team).lower()) if len(w) > 2 and w not in ("university", "college", "the")}
    if "whitewater" in words:
        return _PL_UWW if _PL_UWW in labels else None
    best, score = None, 0
    for lab in labels:
        lw = {w for w in re.findall(r"[a-z]+", str(lab).lower()) if len(w) > 2}
        s = len(words & lw)
        if s > score:
            best, score = lab, s
    return best


def _pl_match(clips, events, offense_label_for):
    """Attach event_index / matched_by / pbp points to each clip. `offense_label_for(clip_row, game_labels)`
    returns the play-by-play team label for the clip's offense."""
    clips = clips.copy()
    clips["event_index"] = pd.NA
    clips["matched_by"] = None
    clips["points"] = None
    clips["points_source"] = None
    if clips.empty:
        return clips
    # CONFIRMED BUG (fixed here): the offense team was only resolved for clips whose game had play-by-play, so
    # a game with no _pbp file dropped every clip out of the team's scouting (offense_team was blank). Resolve
    # it from every label we know about, play-by-play or not.
    _all_labels = set(events["team"].dropna().unique()) if not events.empty else set()
    _all_labels |= {_PL_UWW} | ({upcoming_opponent_short} if upcoming_opponent_short else set())
    clips["team"] = [offense_label_for(c, _all_labels) for _, c in clips.iterrows()]
    ev = (events[events["team"].notna() & events["player"].notna()].copy() if not events.empty
          else pd.DataFrame(columns=list(events.columns) + ["team", "player"]))
    ev["_desc"] = ev["video_description"].apply(_pl_norm) if "video_description" in ev.columns else ""
    by_game = {g: grp for g, grp in ev.groupby("game_date")} if not ev.empty else {}
    used = set()
    for i, c in clips.iterrows():
        grp = by_game.get(c["game_date"])
        if grp is None:
            continue
        team = offense_label_for(c, set(grp["team"].dropna().unique())) or c["team"]
        clips.at[i, "team"] = team
        if team is None:
            continue
        cand = grp[(grp["team"] == team) & (grp["period"] == c["period"])]
        cand = cand[~cand.index.isin(used)]
        if cand.empty:
            continue
        sig = _pl_norm(c.get("synergy_string"))
        secs = c["time_remaining_seconds"]
        best, how = None, None
        if sig:
            hit = cand[cand["_desc"] == sig]
            if not hit.empty:
                if secs is not None and pd.notna(secs):
                    hit = hit.assign(_dt=(hit["time_remaining_seconds"] - secs).abs()).sort_values("_dt")
                best, how = hit.index[0], "synergy string"
        if best is None and secs is not None and pd.notna(secs):
            cand = cand.assign(_dt=(cand["time_remaining_seconds"] - secs).abs(),
                               _ok=cand["event_type"].apply(lambda e: _pl_compatible(c["result"], e)))
            same = cand[(cand["player"].str.lower() == c["player"].lower()) & (cand["_dt"] <= _PL_CLOCK_TOL)]
            same = same.sort_values(["_ok", "_dt"], ascending=[False, True])
            if not same.empty:
                best, how = same.index[0], "player + clock"
            else:
                near = cand[cand["_ok"] & (cand["_dt"] <= 3)].sort_values("_dt")
                if not near.empty:
                    best, how = near.index[0], "clock + result"
        if best is not None:
            used.add(best)
            clips.at[i, "event_index"] = best
            clips.at[i, "matched_by"] = how
    # Points from the play-by-play: everything the offense scored at the matched event's clock stamp.
    scoring = ev[ev["event_type"].isin(["made_shot", "free_throw_made"])].copy() if not ev.empty else pd.DataFrame()
    pts_at = {}
    if not scoring.empty:
        scoring["_pts"] = scoring.apply(
            lambda r: 1 if r["event_type"] == "free_throw_made"
            else int(pd.to_numeric(pd.Series([r.get("shot_type")]), errors="coerce").fillna(2).iloc[0]), axis=1)
        pts_at = scoring.groupby(["game_date", "team", "period", "time_remaining_seconds"])["_pts"].sum().to_dict()
    for i, c in clips.iterrows():
        if pd.notna(c["event_index"]):
            e = ev.loc[c["event_index"]]
            clips.at[i, "points"] = int(pts_at.get((e["game_date"], e["team"], e["period"], e["time_remaining_seconds"]), 0))
            clips.at[i, "points_source"] = "play-by-play"
        else:
            p = _pl_result_points(c["result"])
            if p is not None:
                clips.at[i, "points"] = p
                clips.at[i, "points_source"] = "result tag"
    return clips


def _pl_lineup_for(clips, events, lineup_col):
    """The 5-man unit on the floor at the matched pbp event, pulled onto each clip -- not decoded from the
    clip itself, so this runs after _pl_match assigns event_index, using whichever per-event lineup column
    that side's events carry (see the "On-court 5-man lineups" cells: uww_lineup for our own games,
    self_lineup for the opponent's prior games). None for an unmatched clip; there's no pbp moment to read
    a lineup from, and guessing one from context wouldn't be honest about what's actually known."""
    out = pd.Series([None] * len(clips), index=clips.index, dtype=object)
    if lineup_col not in events.columns:
        return out
    matched = clips["event_index"].notna()
    if matched.any():
        out.loc[matched] = clips.loc[matched, "event_index"].map(events[lineup_col])
    return out


def _pl_decode_all(clips, roster_names):
    if clips.empty:
        return clips
    decoded = pd.DataFrame([decode_play_title(t, roster_names) for t in clips["Title"]], index=clips.index)
    return pd.concat([clips, decoded], axis=1)


def _pl_attach(events, clips):
    """Write the decoded fields onto matched events. An existing play_call (from a coach recap) is kept
    where this file has nothing for that event."""
    events = events.copy()
    for col in _PL_EVENT_COLS:
        if col not in events.columns:
            events[col] = None
    matched = clips[clips["event_index"].notna()]
    for _, c in matched.iterrows():
        idx = c["event_index"]
        for col in _PL_EVENT_COLS:
            src = "decode_quality" if col == "play_decode_quality" else col
            val = c.get(src)
            if val is not None and not (isinstance(val, float) and pd.isna(val)) and val != "":
                events.at[idx, col] = val
    return events


# ---- Shot clock, estimated from the game clock (no shot-clock column exists in the play-by-play) --------
# CONFIRMED CHANGE (requested): estimated from when each possession started and when it ended, using NCAA
# men's basketball rules (Division III plays the same shot clock as D1/D2): 30 seconds on any change of
# possession -- a make, a turnover, a defensive rebound, the last free throw of a trip, the start of a
# period -- and a 20-second reset when the SAME team keeps the ball off its own offensive rebound (the
# 2019-20 NCAA rule change). There is no shot-clock column to read, so this reconstructs the clock's state
# at every event by walking each game/period in order and tracking whose possession it is and when that
# possession began; the game-clock reading at the moment a possession starts stands in for the shot-clock
# reset, and the moment of the matched event (usually the shot or turnover) gives the reading it ended at.
# This is an ESTIMATE, not a read of an actual shot-clock display -- see decode_note-style caveats below on
# where it can be off, and _SC_QUALITY_NOTE, exported alongside the numbers.
_SC_END_TEAM_EVENTS = {"made_shot", "turnover"}       # possession moves to the other team immediately
_SC_MISS_EVENTS = {"missed_shot", "free_throw_missed"}  # possession is undecided until the rebound
_SC_OFF_REBOUND = {"rebound_offensive", "team_deadball_rebound_offensive"}   # same team keeps it -- reset to 20
_SC_DEF_REBOUND = {"rebound_defensive", "team_deadball_rebound_defensive"}   # ball changes hands -- reset to 30
_SC_FT_MADE = {"free_throw_made"}                      # treated as ending the trip -- see the caveat below
_SC_QUALITY_NOTE = (
    "Estimated from the game clock using NCAA men's shot-clock rules (30 sec on a change of possession, "
    "20 sec after an offensive rebound), not read from an actual shot-clock display. Known simplifications: "
    "treats an inbound as happening at the same instant as the made basket/turnover that preceded it "
    "(actually a second or two later); can't always tell the LAST free throw of a multi-shot trip from the "
    "raw event stream, so an intermediate free throw occasionally starts the reset a beat early; and the "
    "very first possession the parser can see in a period is timed from whatever event it first observes, "
    "not the actual tip-off/inbound moment."
)


def estimate_shot_clock(events):
    """Returns (shot_clock_used, shot_clock_max) as two Series aligned to `events`' index -- seconds run off
    the shot clock when each event happened, and the reset ceiling (30 or 20) that applied to the
    possession it happened during. None where the game clock or event type is missing."""
    used = pd.Series([None] * len(events), index=events.index, dtype=object)
    cmax = pd.Series([None] * len(events), index=events.index, dtype=object)
    if events.empty or "team" not in events.columns:
        return used, cmax
    ev = events[events["team"].notna() & events["time_remaining_seconds"].notna()
               & events["event_type"].notna()].sort_values("event_order")
    if ev.empty:
        return used, cmax
    for _keys, grp in ev.groupby(GAME_KEYS + ["period"], dropna=False):
        poss_team, poss_start, poss_max = None, None, 30
        for idx, row in grp.sort_values("event_order").iterrows():
            et, clock, team = row["event_type"], row["time_remaining_seconds"], row["team"]
            if poss_team is None:
                # Possession was just resolved (or this is the period's first visible event) -- whoever
                # this event belongs to has it now, starting fresh from this moment.
                poss_team, poss_start, poss_max = team, clock, 30
            elif team != poss_team and et not in (_SC_OFF_REBOUND | _SC_DEF_REBOUND | _SC_FT_MADE):
                # An event from the OTHER team with no rebound/free-throw to explain the change (a loose-
                # ball foul, a jump ball, ...) -- resync to avoid charging the wrong team's shot clock.
                poss_team, poss_start, poss_max = team, clock, 30
            elapsed = None
            if poss_start is not None and clock is not None:
                elapsed = min(max(poss_start - clock, 0), poss_max)
            used.at[idx] = elapsed
            cmax.at[idx] = poss_max

            if et in _SC_END_TEAM_EVENTS or et in _SC_FT_MADE:
                poss_team, poss_start, poss_max = None, None, 30
            elif et in _SC_OFF_REBOUND:
                poss_team, poss_start, poss_max = team, clock, 20
            elif et in _SC_DEF_REBOUND:
                poss_team, poss_start, poss_max = team, clock, 30
            elif et == "period_marker":
                poss_team, poss_start, poss_max = None, None, 30
            # missed_shot / free_throw_missed: leave possession open, waiting for the rebound event.
    return used, cmax


for _sc_df_name in ("pbp_events", "pbp_events_upcoming"):
    _sc_df = globals()[_sc_df_name]
    if not _sc_df.empty:
        _sc_used, _sc_max = estimate_shot_clock(_sc_df)
        _sc_df["shot_clock_used"] = _sc_used
        _sc_df["shot_clock_max"] = _sc_max
        globals()[_sc_df_name] = _sc_df

# ---- Personnel grouping TYPE (two bigs / base five / small-ball), not the literal 5-man unit -------------
# CONFIRMED CHANGE (requested): the literal on-court lineup is already broken out, player by player, in
# Top Lineups -- Personnel Grouping Tendencies should instead pool tendencies across every lineup that
# shares a personnel PROFILE (how many traditional bigs are on the floor), the way a coach actually talks
# about it ("their two-big lineup runs more post touches"), not repeat the same five names again.
#
# "Big" here means one of a team's two highest-rebounding players by tagged rebounds in the play-by-play --
# a self-contained stand-in for position/height that needs nothing beyond the play-by-play already built.
# It's an approximation (a high-rebounding wing could get swept in, and a team's true second big could be
# a close third) -- good enough to separate a traditional frontcourt from a small-ball group, not a
# roster-accurate positional breakdown.
_PG_TYPE_LABELS = ("Two-big lineup", "Base five (one big)", "Small / shooting five (no true big)")
_PG_REBOUND_EVENTS = ["rebound_offensive", "rebound_defensive", "team_deadball_rebound_offensive",
                      "team_deadball_rebound_defensive"]


def _pg_rebound_leaders(events, team_label, top_n=2):
    if events.empty or "team" not in events.columns or not team_label:
        return set()
    reb = events[(events["team"] == team_label) & events["event_type"].isin(_PG_REBOUND_EVENTS)
                & events["player"].notna()]
    if reb.empty:
        return set()
    return set(reb["player"].value_counts().head(top_n).index)


def _pg_grouping_type(lineup_str, bigs_set):
    if not lineup_str or not bigs_set or not isinstance(lineup_str, str):
        return None
    members = [p.strip() for p in lineup_str.split(",") if p.strip()]
    if not members:
        return None
    n_big = sum(1 for m in members if m in bigs_set)
    return _PG_TYPE_LABELS[0] if n_big >= 2 else _PG_TYPE_LABELS[1] if n_big == 1 else _PG_TYPE_LABELS[2]


_pg_uww_bigs = _pg_rebound_leaders(pbp_events, _PL_UWW)
_pg_opp_bigs = _pg_rebound_leaders(pbp_events_upcoming, upcoming_opponent_short)

# ---- Granular grouping labels from real roster positions (requested) ---------------------------------
# "Two bigs / one big / no big" is coarse -- it can't tell a three-guard look from a four-guard one. The
# roster-page scrape now supplies each player's listed position, so a lineup can be described the way a coach
# says it out loud: "3G-1W-1B". Guard/Wing/Big is the grouping that survives FastScout's position strings
# ("G", "G/F", "F/C", "W"); the first letter decides, with F treated as a wing only when paired with a guard.
# Falls back to the rebound-based two-big/one-big/no-big label for a team with no roster positions on file,
# so nothing depends on the scrape having run.
_PG_POS_GROUP = {"G": "G", "W": "W", "F": "W", "C": "B"}


def _pg_position_map():
    ros = globals().get("live_rosters")
    out = {}
    if ros is None or getattr(ros, "empty", True) or "position" not in ros.columns:
        return out
    for _, r in ros.iterrows():
        pos = str(r.get("position") or "").strip().upper()
        if not pos:
            continue
        first, parts = pos[0], [p for p in re.split(r"[/-]", pos) if p]
        group = _PG_POS_GROUP.get(first, "W")
        if first == "F":
            group = "B" if any(p.startswith("C") for p in parts) else "W"
        out[_ros_norm_name(r.get("name")) if "_ros_norm_name" in globals()
            else re.sub(r"\s+", " ", str(r.get("name"))).strip().lower()] = group
    return out


_pg_positions = _pg_position_map()


def _pg_shape_label(lineup_str):
    """"3G-1W-1B" when every player's position is known, else None."""
    if not lineup_str or not isinstance(lineup_str, str) or not _pg_positions:
        return None
    members = [p.strip() for p in lineup_str.split(",") if p.strip()]
    groups = [_pg_positions.get(re.sub(r"\s+", " ", m).strip().lower()) for m in members]
    if not members or any(g is None for g in groups):
        return None
    counts = {g: groups.count(g) for g in ("G", "W", "B")}
    return "-".join(f"{counts[g]}{g}" for g in ("G", "W", "B") if counts[g])


def _pg_grouping_label(lineup_str, bigs_set):
    return _pg_shape_label(lineup_str) or _pg_grouping_type(lineup_str, bigs_set)

# ---- Game situation (leading/trailing big, or clutch), reusing the SAME clutch definition already used
# elsewhere in this pipeline (see the "Clutch-time event log" cell: last 5 minutes of the 2nd half or any
# overtime, score within 8) rather than inventing a second one. Both pbp_events and pbp_events_upcoming carry
# "uww_score"/"opp_score" columns holding the score of whoever's game this is (the team_label / self_team
# passed into build_pbp_events) vs the other team -- so the identical formula applies to either dataframe
# unchanged; for pbp_events_upcoming these are the SCOUTED opponent's own score and their opponent's score
# that game, not literally UWW's.
_GS_CLUTCH_MARGIN = 8
_GS_CLUTCH_SECONDS = 300
_GS_BLOWOUT_MARGIN = 10


def _gs_situation(row):
    if pd.isna(row.get("uww_score")) or pd.isna(row.get("opp_score")):
        return None
    margin = row["uww_score"] - row["opp_score"]
    if (row.get("period") != "H1" and pd.notna(row.get("time_remaining_seconds"))
            and row["time_remaining_seconds"] <= _GS_CLUTCH_SECONDS and abs(margin) <= _GS_CLUTCH_MARGIN):
        return f"Clutch (last 5 min, margin \u2264 {_GS_CLUTCH_MARGIN})"
    if margin >= _GS_BLOWOUT_MARGIN:
        return f"Leading by {_GS_BLOWOUT_MARGIN}+"
    if margin <= -_GS_BLOWOUT_MARGIN:
        return f"Trailing by {_GS_BLOWOUT_MARGIN}+"
    return None  # a comfortable middle -- not one of the situations this breakdown is built to flag


for _gs_df_name in ("pbp_events", "pbp_events_upcoming"):
    _gs_df = globals()[_gs_df_name]
    if not _gs_df.empty and {"uww_score", "opp_score", "period", "time_remaining_seconds"}.issubset(_gs_df.columns):
        _gs_df["game_situation"] = _gs_df.apply(_gs_situation, axis=1)
        globals()[_gs_df_name] = _gs_df

# ---- Offensive vs DEFENSIVE possessions ----------------------------------------------------------------
# Until now every clip in these files was an OFFENSIVE possession, so "defense_type" unambiguously meant
# "the defense this team faced". Once defensive possessions are tagged too, that same field means the
# OPPOSITE thing on half the rows -- the defense this team PLAYED -- and nothing downstream would notice:
# "Aurora scored 1.15 PPP against man" would quietly average together with "Aurora's opponents scored 1.15
# against Aurora's man". Both numbers look plausible. That is exactly the two-things-one-name drift that
# has bitten this project before, so the ambiguity is resolved HERE, once, and the ambiguous field is split
# into two explicitly-named ones that can never be confused downstream.
def _pl_possession_side(df, own_team):
    """Offense when the clip's own team has the ball, Defense when it doesn't. Falls back to Offense with a
    loud problem note when the team can't be read, because silently guessing is what this function exists
    to prevent."""
    if df.empty:
        return df
    df = df.copy()
    if "team" not in df.columns:
        df["possession_side"] = "Offense"
        _pl_problems.append(f"{own_team} plays file has no readable Team column -- every clip assumed to be "
                            f"an OFFENSIVE possession. If defensive possessions are tagged in it, every "
                            f"defense-split table is wrong. Fix the export's Team column.")
        return df
    _own = df["team"].astype(str) == str(own_team)
    _unknown = df["team"].isna() | (df["team"].astype(str).str.strip() == "")
    df["possession_side"] = _own.map({True: "Offense", False: "Defense"})
    df.loc[_unknown, "possession_side"] = "Offense"
    if int(_unknown.sum()):
        _pl_problems.append(f"{own_team} plays file: {int(_unknown.sum())} clip(s) have no Team value -- "
                            f"assumed OFFENSIVE possessions. Check them before trusting defense splits.")
    # The split. defense_faced is what the tagged team's OFFENSE saw; defense_played is what that team's
    # own DEFENSE was in. Exactly one of the two is populated on any given clip.
    _is_off = df["possession_side"] == "Offense"
    for _src, _faced, _played in (("defense_type", "defense_faced", "defense_played"),
                                  ("defense_coverage", "coverage_faced", "coverage_played"),
                                  ("defense_press", "press_faced", "press_played"),
                                  ("defense_press_formation", "press_formation_faced", "press_formation_played")):
        if _src not in df.columns:
            continue
        df[_faced] = df[_src].where(_is_off)
        df[_played] = df[_src].where(~_is_off)
    return df


# ---- UW-Whitewater ------------------------------------------------------------------------------------
_pl_uww_raw = _pl_load(_PL_UWW_FILE, "UW-Whitewater")
_pl_roster = set(pbp_events["player"].dropna().astype(str)) if not pbp_events.empty else set()
_pl_uww = _pl_decode_all(_pl_uww_raw, _pl_roster | set(_pl_uww_raw.get("player", [])))
if not _pl_uww.empty:
    _pl_uww = _pl_match(_pl_uww, pbp_events, lambda c, labels: _pl_team_label(c["Team"], labels) or _PL_UWW)
    _pl_uww["side"] = "UWW"
    _pl_uww = _pl_possession_side(_pl_uww, _PL_UWW)
    # UWW's opponent in that game, from the play-by-play itself.
    _pl_date_opp = (pbp_events.dropna(subset=["game_date"]).groupby("game_date")["opponent"].first().to_dict()
                    if not pbp_events.empty else {})
    _pl_uww["opponent"] = _pl_uww["game_date"].map(_pl_date_opp)
    # Who actually had the ball -- no longer "always us". On a defensive possession the opponent is the
    # offense and we are the defense.
    _pl_uww_off = _pl_uww["possession_side"] == "Offense"
    _pl_uww["offense_team"] = _pl_uww["opponent"].where(~_pl_uww_off, _PL_UWW)
    _pl_uww["defense_team"] = _pl_uww["opponent"].where(_pl_uww_off, _PL_UWW)
    _pl_uww["on_court_lineup"] = _pl_lineup_for(_pl_uww, pbp_events, "uww_lineup")
    _pl_uww["personnel_grouping_type"] = _pl_uww["on_court_lineup"].apply(lambda lu: _pg_grouping_label(lu, _pg_uww_bigs))
    _pl_uww["game_situation"] = _pl_lineup_for(_pl_uww, pbp_events, "game_situation")
    _pl_uww["shot_clock_used"] = _pl_lineup_for(_pl_uww, pbp_events, "shot_clock_used")
    _pl_uww["shot_clock_max"] = _pl_lineup_for(_pl_uww, pbp_events, "shot_clock_max")
    pbp_events = _pl_attach(pbp_events, _pl_uww)

# ---- Upcoming opponent --------------------------------------------------------------------------------
_pl_opp_raw = _pl_load(_PL_OPP_FILE, "opponent")
_pl_opp_roster = (set(pbp_events_upcoming["player"].dropna().astype(str))
                  if not pbp_events_upcoming.empty else set())
_pl_opp = _pl_decode_all(_pl_opp_raw, _pl_opp_roster | set(_pl_opp_raw.get("player", [])))
if not _pl_opp.empty:
    # The file is supposed to be THIS week's opponent. If none of its Team values overlap the upcoming
    # opponent's name, it is last week's file -- say so rather than scouting the wrong team.
    _pl_teams = _pl_opp["Team"].dropna().unique()
    if upcoming_opponent_short and not any(_pl_team_label(t, {upcoming_opponent_short}) for t in _pl_teams):
        _pl_problems.append(f"opponent_plays.csv is for {list(_pl_teams)}, not {upcoming_opponent_short} -- "
                            f"replace it with this week's export. Opponent play calls skipped.")
        _pl_opp = pd.DataFrame()
if not _pl_opp.empty:
    _pl_opp = _pl_match(_pl_opp, pbp_events_upcoming,
                        lambda c, labels: _pl_team_label(c["Team"], labels))
    _pl_opp["side"] = "Opponent"
    _pl_opp = _pl_possession_side(_pl_opp, upcoming_opponent_short)
    _pl_opp_off = _pl_opp["possession_side"] == "Offense"
    # In pbp_events_upcoming, `opponent` is the THIRD PARTY they played that game.
    _pl_up_opp = (pbp_events_upcoming.dropna(subset=["game_date"]).groupby("game_date")["opponent"].first().to_dict()
                  if not pbp_events_upcoming.empty else {})
    _pl_opp["opponent"] = _pl_opp["game_date"].map(_pl_up_opp)
    _pl_opp["offense_team"] = _pl_opp["team"].where(_pl_opp_off, _pl_opp["opponent"])
    _pl_opp["defense_team"] = _pl_opp["opponent"].where(_pl_opp_off, upcoming_opponent_short)
    _pl_other = _pl_opp["team"].notna() & (_pl_opp["team"] != upcoming_opponent_short)
    if int(_pl_other.sum()):
        print(f"  opponent_plays.csv: {int(_pl_other.sum())} clip(s) are the OTHER team's offense -- kept, "
              f"but excluded from {upcoming_opponent_short}'s play-call scouting.")
    _pl_opp["on_court_lineup"] = _pl_lineup_for(_pl_opp, pbp_events_upcoming, "self_lineup")
    _pl_opp["personnel_grouping_type"] = _pl_opp["on_court_lineup"].apply(lambda lu: _pg_grouping_label(lu, _pg_opp_bigs))
    _pl_opp["game_situation"] = _pl_lineup_for(_pl_opp, pbp_events_upcoming, "game_situation")
    _pl_opp["shot_clock_used"] = _pl_lineup_for(_pl_opp, pbp_events_upcoming, "shot_clock_used")
    _pl_opp["shot_clock_max"] = _pl_lineup_for(_pl_opp, pbp_events_upcoming, "shot_clock_max")
    pbp_events_upcoming = _pl_attach(pbp_events_upcoming, _pl_opp)

for _df_name in ("pbp_events", "pbp_events_upcoming"):
    _df = globals()[_df_name]
    for _col in _PL_EVENT_COLS:
        if _col not in _df.columns:
            _df[_col] = None

# ---- One table of every clip -------------------------------------------------------------------------
PLAY_CALL_COLS = [
    "side", "scouted_opponent", "offense_team", "defense_team", "opponent", "game_date", "game_code", "period",
    "time_remaining_seconds", "player", "result", "points", "points_source", "matched_event", "matched_by",
    "play_title", "play_call", "play_series", "play_situation", "play_formation", "play_set", "play_actions",
    "primary_action", "play_location", "finish_spot", "featured_player", "decode_quality", "decode_note",
    "synergy_play_type", "synergy_string", "on_court_lineup", "personnel_grouping_type", "game_situation",
    "shot_clock_used", "shot_clock_max", "shot_clock_situation",
    # NEW: the defense this offense faced on the clip, from the coaches' updated Title tagging (see
    # decode_defense_tag). "defense_team" above is WHICH team was on defense; these are WHAT they played.
    "defense_type", "defense_press", "defense_press_formation", "defense_coverage",
    # Offense vs Defense possession, and the unambiguous split of the four fields above (see
    # _pl_possession_side). Prefer defense_faced / defense_played over the raw defense_type downstream:
    # the raw field means opposite things on offensive and defensive possessions.
    "possession_side", "defense_faced", "defense_played", "coverage_faced", "coverage_played",
    "press_faced", "press_played", "press_formation_faced", "press_formation_played",
]
_pl_frames = []
for _f in (_pl_uww, _pl_opp):
    if _f is None or _f.empty:
        continue
    _f = _f.copy()
    _f["scouted_opponent"] = upcoming_opponent_short
    _f["matched_event"] = _f["event_index"].notna()
    # Bucketed for the "Shot clock tendencies" breakdown. Boundaries are seconds USED (not remaining),
    # scaled to the standard 30-second clock -- an offensive-rebound putback (20-second max) can only ever
    # land in Early or Organized, which is correct: it genuinely can't have used more than 20 seconds.
    _sc_used = pd.to_numeric(_f.get("shot_clock_used"), errors="coerce")
    _f["shot_clock_situation"] = pd.cut(
        _sc_used, bins=[-0.01, 9, 19, 100],
        labels=["Early clock (0-9 sec used)", "Organized offense (10-19 sec used)", "Late clock (20+ sec used)"],
    ).astype(object).where(_sc_used.notna(), None)
    _pl_frames.append(_f.reindex(columns=PLAY_CALL_COLS))
play_calls = pd.concat(_pl_frames, ignore_index=True) if _pl_frames else pd.DataFrame(columns=PLAY_CALL_COLS)

# ---- Summaries: one row per (side, level, name) --------------------------------------------------------
# Built here so the brief and the app read the same numbers. A clip flagged Rewatch/TBD or with no call is
# counted in the totals but never ranked as a set.
_PL_LEVELS = [("play_call", "Play call"), ("play_series", "Series"), ("primary_action", "Primary action"),
              ("play_situation", "Situation"), ("play_location", "Location"), ("synergy_play_type", "Finish type"),
              # Which 5-man unit was on the floor, from the play-by-play's own lineup reconstruction (see
              # "On-court 5-man lineups" / the upcoming-opponent lineup cell) -- not decoded from the clip,
              # pulled from the matched pbp event, so this level only has rows for MATCHED clips.
              ("on_court_lineup", "Personnel grouping"),
              # Grouped by personnel TYPE (two bigs / base five / small-ball) rather than the literal five
              # names -- see _pg_grouping_type above. This is what Personnel Grouping Tendencies reads.
              ("personnel_grouping_type", "Personnel grouping type"),
              # Leading/trailing by 10+, or clutch (same clutch definition as the "Clutch-time event log"
              # cell elsewhere in this notebook: last 5 min of the 2nd half or OT, margin <= 8) -- only
              # populated for matched clips, and only for one of these three situations; a comfortable
              # middle possession gets no game_situation value and isn't part of this breakdown.
              ("game_situation", "Game situation"),
              # Estimated shot-clock bucket (see estimate_shot_clock above) -- also only populated for
              # matched clips, since the estimate needs an actual pbp moment to read the game clock from.
              ("shot_clock_situation", "Shot clock"),
              # NEW: the defense faced, decoded straight from the Title (see decode_defense_tag) -- unlike
              # the levels above this doesn't need a matched pbp event, so a clip tagged only "M2M" with no
              # named play still counts here even though it has no "Play call" row. Fills the real data
              # behind what used to be the DEFENSE TYPE BY SITUATION / BALL SCREEN COVERAGE sample tables.
              # Faced vs played, kept separate (see _pl_possession_side) -- "Defense faced" is what this
              # team's OFFENSE saw, "Defense played" is what its own defense was in. The raw defense_type
              # level is deliberately NOT summarized any more: on a mixed offense/defense file it would
              # average the two together into a number that means nothing.
              ("defense_faced", "Defense faced"), ("coverage_faced", "Ball screen coverage faced"),
              ("defense_played", "Defense played"), ("coverage_played", "Ball screen coverage played")]


def _pl_summarize(df, side):
    rows = []
    if df.empty:
        return rows
    df = df.copy()
    df["_pts"] = pd.to_numeric(df["points"], errors="coerce")
    res = df["result"].astype(str).str.lower()
    df["_fga"] = res.str.startswith(("make", "miss")) & res.str.contains("2|3", regex=True)
    df["_fgm"] = res.str.startswith("make") & res.str.contains("2|3", regex=True)
    df["_3pa"] = res.str.contains("3 pts")
    df["_3pm"] = res.str.startswith("make 3")
    df["_to"] = res.str.contains("turnover|violation|kicked", regex=True)
    df["_fd"] = res.str.contains("foul")
    df = df[df["decode_quality"] != "Needs review"]
    total_known = df["_pts"].notna().sum()
    team_ppp = (df["_pts"].sum() / total_known) if total_known else None
    for col, level in _PL_LEVELS:
        g = df[df[col].notna() & (df[col].astype(str).str.strip() != "")]
        for name, grp in g.groupby(col):
            known = grp["_pts"].notna().sum()
            ppp = grp["_pts"].sum() / known if known else None
            fga = int(grp["_fga"].sum())
            top = grp["player"].value_counts()
            rows.append({
                "side": side, "scouted_opponent": upcoming_opponent_short, "level": level, "name": name,
                "uses": len(grp), "games": grp["game_date"].nunique(), "poss_with_points": int(known),
                "points": float(grp["_pts"].sum()), "ppp": round(ppp, 2) if ppp is not None else None,
                "team_ppp": round(team_ppp, 2) if team_ppp is not None else None,
                "fgm": int(grp["_fgm"].sum()), "fga": fga,
                "fg_pct": round(100 * grp["_fgm"].sum() / fga, 1) if fga else None,
                "fg3m": int(grp["_3pm"].sum()), "fg3a": int(grp["_3pa"].sum()),
                "turnovers": int(grp["_to"].sum()), "fouls_drawn": int(grp["_fd"].sum()),
                "top_player": top.index[0] if len(top) else None,
                "top_player_uses": int(top.iloc[0]) if len(top) else 0,
                # Which series this set belongs to (mode) -- lets the brief nest sets under their series
                # without re-deriving the mapping from the clips.
                "series": (grp["play_series"].dropna().value_counts().index[0]
                           if col != "play_series" and grp["play_series"].notna().any() else
                           (name if col == "play_series" else None)),
                "top_action": (grp["primary_action"].dropna().value_counts().index[0]
                               if col != "primary_action" and grp["primary_action"].notna().any() else None),
                "top_location": (grp["play_location"].dropna().value_counts().index[0]
                                 if col != "play_location" and grp["play_location"].notna().any() else None),
                # How many times that spot came up -- the brief shows "Right Side (x4)" rather than implying
                # every use of the set started there.
                "top_location_uses": (int(grp["play_location"].dropna().value_counts().iloc[0])
                                      if col != "play_location" and grp["play_location"].notna().any() else 0),
                "situation": (grp["play_situation"].value_counts().index[0] if col != "play_situation" else name),
                # Same mode-attachment pattern as "situation" above, for the three other splits (requested):
                # lets a "Play call" or "Series" row also be grouped under whichever personnel/shot-clock/
                # game-situation bucket it most often ran in, so those splits can nest sets the same way
                # Half court / BLOB / SLOB / ATO already do, instead of a flat one-row-per-bucket summary.
                "personnel_grouping_mode": (
                    grp["personnel_grouping_type"].dropna().value_counts().index[0]
                    if col != "personnel_grouping_type" and "personnel_grouping_type" in grp.columns
                    and grp["personnel_grouping_type"].notna().any()
                    else (name if col == "personnel_grouping_type" else None)),
                "shot_clock_mode": (
                    grp["shot_clock_situation"].dropna().value_counts().index[0]
                    if col != "shot_clock_situation" and "shot_clock_situation" in grp.columns
                    and grp["shot_clock_situation"].notna().any()
                    else (name if col == "shot_clock_situation" else None)),
                "game_situation_mode": (
                    grp["game_situation"].dropna().value_counts().index[0]
                    if col != "game_situation" and "game_situation" in grp.columns
                    and grp["game_situation"].notna().any()
                    else (name if col == "game_situation" else None)),
                # Same pattern once more, for the defense the offense faced (requested) -- lets "Best against
                # each defense" on HOW WE RUN OFFENSE use the exact same tier-h + series/set pivot as every
                # other split on that card, instead of a bespoke summary table.
                # defense_FACED, not the raw ambiguous defense_type -- this rides on "Best against each
                # defense" in HOW WE RUN OFFENSE, which is about what our offense saw.
                "defense_type_mode": (
                    grp["defense_faced"].dropna().value_counts().index[0]
                    if col != "defense_faced" and "defense_faced" in grp.columns
                    and grp["defense_faced"].notna().any()
                    else (name if col == "defense_faced" else None)),
                "example_titles": " | ".join(grp["play_title"].astype(str).value_counts().index[:3]),
                "game_codes": " ".join(sorted(grp["game_code"].dropna().astype(str).unique())),
            })
    return rows


_pl_sum_rows = []
if not play_calls.empty:
    # Summarized SEPARATELY for offensive and defensive possessions, and every row is stamped with which
    # it came from.
    #
    # CONFIRMED BUG (fixed here): once defensive possessions started appearing in uww_plays.csv,
    # side == "UWW" stopped meaning "our offense" -- it means "a clip from our file", which now includes
    # possessions where the opponent had the ball. Summarizing them together put opposing players in our
    # own offensive tables: a SLOB credited to CJ Brown, a Ripon player, showed up under our sets. The
    # play_call / series levels are only meaningful for the side that had the ball, so they are built per
    # possession_side, and consumers filter on it.
    def _pl_sum_side(df, side_label):
        rows = []
        if df.empty:
            return rows
        _ps = df.get("possession_side")
        for _poss in ("Offense", "Defense"):
            _part = df[_ps.astype(str) == _poss] if _ps is not None else (df if _poss == "Offense" else df.iloc[0:0])
            if _part.empty:
                continue
            for _r in _pl_summarize(_part, side_label):
                _r["possession_side"] = _poss
                rows.append(_r)
        return rows

    _pl_sum_rows += _pl_sum_side(play_calls[play_calls["side"] == "UWW"], "UWW")
    # For the opponent, an OFFENSIVE possession is one where they had the ball; their defensive clips keep
    # their own team label, so filter on possession_side rather than on offense_team alone.
    _opp_clips = play_calls[play_calls["side"] == "Opponent"]
    if "possession_side" in _opp_clips.columns:
        _opp_clips = _opp_clips[((_opp_clips["possession_side"].astype(str) == "Offense")
                                 & (_opp_clips["offense_team"] == upcoming_opponent_short))
                                | (_opp_clips["possession_side"].astype(str) == "Defense")]
    else:
        _opp_clips = _opp_clips[_opp_clips["offense_team"] == upcoming_opponent_short]
    _pl_sum_rows += _pl_sum_side(_opp_clips, "Opponent")
play_call_summary = pd.DataFrame(_pl_sum_rows)
if not play_call_summary.empty and "possession_side" in play_call_summary.columns:
    print(f"  play_call_summary: {int((play_call_summary['possession_side'] == 'Offense').sum())} offensive "
          f"row(s), {int((play_call_summary['possession_side'] == 'Defense').sum())} defensive.")

# One lineup -> grouping map, exported so the game-plan cell aggregates MINUTES on exactly this rule rather
# than classifying lineups a second time (which is how minutes and tagged clips once landed on different
# groupings). Every five-man unit either side has on film, whether or not any clip matched it.
_lg_rows = []
for _lg_df, _lg_side, _lg_team, _lg_bigs, _lg_col in (
        (pbp_events, "UWW", _PL_UWW, _pg_uww_bigs, "uww_lineup"),
        (globals().get("upcoming_lineup_season"), "Opponent", upcoming_opponent_short, _pg_opp_bigs, "lineup")):
    if _lg_df is None or getattr(_lg_df, "empty", True) or _lg_col not in getattr(_lg_df, "columns", []):
        continue
    for _lu in _lg_df[_lg_col].dropna().astype(str).unique():
        _lg_rows.append({"side": _lg_side, "team": _lg_team, "lineup": _lu,
                         "grouping": _pg_grouping_label(_lu, _lg_bigs),
                         "shape": _pg_shape_label(_lu), "scouted_opponent": upcoming_opponent_short})
lineup_grouping = pd.DataFrame(_lg_rows, columns=["side", "team", "lineup", "grouping", "shape",
                                                  "scouted_opponent"])

# ---- Feed UWW's clips into coach_notes, which the app's existing play-call analytics already read -------
if _pl_uww is not None and not _pl_uww.empty:
    _pl_cn = _pl_uww[_pl_uww["play_call"].notna()].copy()
    _pl_cn["coach_note"] = None
    _pl_cn["clip_side"] = "Offense"
    _pl_cn["team"] = _PL_UWW
    _cn_extra = ["game_date", "points", "play_series", "play_title", "play_actions", "play_location", "play_situation"]
    for _c in _cn_extra:
        if _c not in coach_notes.columns:
            coach_notes[_c] = None
    _pl_cn = _pl_cn[[c for c in coach_notes.columns if c in _pl_cn.columns]]
    # The old season play log is retired, so any previous play_call-only rows are superseded by this file.
    _old_log_rows = coach_notes["play_call"].notna() & coach_notes["coach_note"].isna()
    coach_notes = pd.concat([coach_notes[~_old_log_rows], _pl_cn], ignore_index=True)

# ---- Report ---------------------------------------------------------------------------------------------
for _label, _f in (("uww_plays.csv", _pl_uww), ("opponent_plays.csv", _pl_opp)):
    if _f is None or _f.empty:
        continue
    _q = _f["decode_quality"].value_counts().to_dict()
    _m = int(_f["event_index"].notna().sum())
    print(f"{_label}: {len(_f)} clip(s) across {_f['game_date'].nunique()} game(s); decoded {_q}; "
          f"matched to play-by-play {_m}/{len(_f)} "
          f"({_f['matched_by'].value_counts().to_dict()}).")
    _unm = _f[_f["event_index"].isna()]
    if not _unm.empty:
        print("  First unmatched clips (date, period, clock, player, result):")
        print(_unm[["game_date", "period", "Clock", "player", "result"]].head(6).to_string(index=False))
    _top = _f[_f["decode_quality"] != "Needs review"]["play_call"].value_counts().head(6)
    print(f"  Most-called: {_top.to_dict()}")
    _sc_cov = _f["shot_clock_used"].notna().sum()
    if _sc_cov:
        print(f"  Shot clock estimated for {_sc_cov}/{len(_f)} clip(s) (needs a matched pbp event).")
    _def_cov = _f["defense_type"].notna().sum()
    if _def_cov:
        print(f"  Defense tagged on {_def_cov}/{len(_f)} clip(s): "
              f"{_f['defense_type'].value_counts().to_dict()}; "
              f"press on {int(_f['defense_press'].sum())}, coverage called on "
              f"{int((_f['defense_coverage'].astype(str).str.len() > 0).sum())}.")
        # WHICH SIDE those tags landed on decides what every defense table downstream can say. A file
        # where every tag sits on an offensive possession can describe what this team FACED but can say
        # nothing about the defense they PLAY -- which is the difference between opp_defense filling in
        # and staying sample.
        if "possession_side" in _f.columns:
            _off_n = int((_f["possession_side"] == "Offense").sum())
            _def_n = int((_f["possession_side"] == "Defense").sum())
            _faced = int(_f["defense_faced"].notna().sum()) if "defense_faced" in _f.columns else 0
            _played = int(_f["defense_played"].notna().sum()) if "defense_played" in _f.columns else 0
            print(f"    Possessions: {_off_n} offensive, {_def_n} defensive "
                  f"-> {_faced} defense-faced tag(s), {_played} defense-played tag(s).")
            if _played == 0 and _faced:
                print("    NOTE: no defensive possessions in this file, so nothing here can describe the "
                      "defense this team PLAYS -- only what its offense faced. If you tagged their "
                      "defense, check whether the Team column names the team being scouted rather than "
                      "the team with the ball.")
if _pl_problems:
    print("Play-call problems:")
    for _p in _pl_problems:
        print(f"  - {_p}")


  opponent_plays.csv: 404 clip(s) are the OTHER team's offense -- kept, but excluded from Aurora Spartans's play-call scouting.
  play_call_summary: 254 offensive row(s), 177 defensive.
uww_plays.csv: 457 clip(s) across 3 game(s); decoded {'No call': 328, 'Clean': 102, 'Partial': 21, 'Needs review': 6}; matched to play-by-play 408/457 ({'synergy string': 335, 'player + clock': 61, 'clock + result': 12}).
  First unmatched clips (date, period, clock, player, result):
 game_date period Clock         player            result
2025-11-14     H1   NaN Corey Thompson Non Shooting Foul
2025-11-14     H1   NaN   Isaac Verges      No Violation
2025-11-14     H1   NaN      Luke Bara Non Shooting Foul
2025-11-14     H1   NaN      Luke Bara Non Shooting Foul
2025-11-14     H1   NaN  Angel Johnson Non Shooting Foul
2025-11-14     H1   NaN  Angel Johnson      No Violation
  Most-called: {'BLOB': 21, 'SLOB': 12, '5 Out': 12, 'Panther': 11, 'BLOB Line': 11, 'BLOB Line Curl': 9}
  Shot clock estimated f

C:\Users\frits\AppData\Local\Temp\ipykernel_27492\277713935.py:1076: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  coach_notes = pd.concat([coach_notes[~_old_log_rows], _pl_cn], ignore_index=True)



### Compute biggest scoring runs and largest lead/deficit, with lineups

Finds each game's biggest scoring run and largest lead/deficit for both teams, along with which 5-man lineup was on the floor for each.

In [108]:
# --- Scoring runs and largest lead/deficit per game, with the 5-man lineups on the floor for each ------------
scoring_events = pbp_events[pbp_events["event_type"].isin(["made_shot", "free_throw_made"])].copy()
scoring_events["points"] = scoring_events.apply(
    lambda row: int(row["shot_type"]) if row["event_type"] == "made_shot" else 1, axis=1
)

def detailed_runs(group):
    """Same run-detection as before, but keeps each run's start/end event_order so the lineup on the floor can be looked up."""
    runs, cur_team, cur_pts, cur_start, cur_end = [], None, 0, None, None
    for _, row in group.sort_values("event_order").iterrows():
        if row["team"] == cur_team:
            cur_pts += row["points"]
            cur_end = row["event_order"]
        else:
            if cur_team is not None:
                runs.append({"team": cur_team, "run_points": cur_pts, "start_event_order": cur_start, "end_event_order": cur_end})
            cur_team, cur_pts, cur_start, cur_end = row["team"], row["points"], row["event_order"], row["event_order"]
    if cur_team is not None:
        runs.append({"team": cur_team, "run_points": cur_pts, "start_event_order": cur_start, "end_event_order": cur_end})
    return pd.DataFrame(runs)

def lineups_during(opponent, game_date, start_eo, end_eo):
    """Unique UWW/opponent lineups seen across [start_eo, end_eo] of ONE game -- normally just one of
    each, unless a sub happened mid-window. `game_date` is required: event_order restarts every game,
    so an opponent-only filter would sweep in the same event_order range from a rematch too."""
    window = pbp_events[
        (pbp_events["opponent"] == opponent) & (pbp_events["game_date"] == game_date)
        & (pbp_events["event_order"] >= start_eo) & (pbp_events["event_order"] <= end_eo)
    ]
    uww_l = window["uww_lineup"].dropna().unique().tolist()
    opp_l = window["opp_lineup"].dropna().unique().tolist()
    return (uww_l[0] if len(uww_l) == 1 else (" / ".join(uww_l) or None)), (opp_l[0] if len(opp_l) == 1 else (" / ".join(opp_l) or None))

run_rows = []
for (opponent, game_date), group in scoring_events.groupby(GAME_KEYS, dropna=False):
    runs_df = detailed_runs(group)
    biggest_by_team = runs_df.loc[runs_df.groupby("team")["run_points"].idxmax()].set_index("team") if not runs_df.empty else runs_df

    margins_full = pbp_events[(pbp_events["opponent"] == opponent) & (pbp_events["game_date"] == game_date)].dropna(subset=["uww_score"]).copy()
    margins_full["margin"] = margins_full["uww_score"] - margins_full["opp_score"]
    uww_lead_row = margins_full.loc[margins_full["margin"].idxmax()] if not margins_full.empty else None
    opp_lead_row = margins_full.loc[margins_full["margin"].idxmin()] if not margins_full.empty else None

    uww_run = biggest_by_team.loc["UW-Whitewater"] if "UW-Whitewater" in biggest_by_team.index else None
    opp_run = biggest_by_team.loc[opponent] if opponent in biggest_by_team.index else None
    uww_run_lineups = lineups_during(opponent, game_date, uww_run["start_event_order"], uww_run["end_event_order"]) if uww_run is not None else (None, None)
    opp_run_lineups = lineups_during(opponent, game_date, opp_run["start_event_order"], opp_run["end_event_order"]) if opp_run is not None else (None, None)

    run_rows.append({
        "opponent": opponent,
        "game_date": game_date,
        "uww_biggest_run": int(uww_run["run_points"]) if uww_run is not None else 0,
        "uww_run_uww_lineup": uww_run_lineups[0], "uww_run_opp_lineup": uww_run_lineups[1],
        "opponent_biggest_run": int(opp_run["run_points"]) if opp_run is not None else 0,
        "opp_run_uww_lineup": opp_run_lineups[0], "opp_run_opp_lineup": opp_run_lineups[1],
        "uww_largest_lead": int(uww_lead_row["margin"]) if uww_lead_row is not None else 0,
        "uww_lead_uww_lineup": uww_lead_row["uww_lineup"] if uww_lead_row is not None else None,
        "uww_lead_opp_lineup": uww_lead_row["opp_lineup"] if uww_lead_row is not None else None,
        "opponent_largest_lead": int(-opp_lead_row["margin"]) if opp_lead_row is not None else 0,
        "opp_lead_uww_lineup": opp_lead_row["uww_lineup"] if opp_lead_row is not None else None,
        "opp_lead_opp_lineup": opp_lead_row["opp_lineup"] if opp_lead_row is not None else None,
    })
scoring_runs = pd.DataFrame(run_rows)
print(scoring_runs)

print("Scoring run & lead summary, with the 5-man lineups on the floor for each:\n")
for _, row in scoring_runs.iterrows():
    print(f"{row['opponent']}: UWW's biggest run = {row['uww_biggest_run']} pts, {row['opponent']}'s biggest run = {row['opponent_biggest_run']} pts")
    print(f"    During UWW's run -- UWW: {row['uww_run_uww_lineup']} | {row['opponent']}: {row['uww_run_opp_lineup']}")
    print(f"    During {row['opponent']}'s run -- UWW: {row['opp_run_uww_lineup']} | {row['opponent']}: {row['opp_run_opp_lineup']}")
    print(f"    Largest UWW lead: {row['uww_largest_lead']} pts (UWW: {row['uww_lead_uww_lineup']} | {row['opponent']}: {row['uww_lead_opp_lineup']})")
    print(f"    Largest {row['opponent']} lead: {row['opponent_largest_lead']} pts (UWW: {row['opp_lead_uww_lineup']} | {row['opponent']}: {row['opp_lead_opp_lineup']})\n")

                opponent   game_date  uww_biggest_run  \
0      Eureka Red Devils  2025-11-15               12   
1        Ripon Red Hawks  2025-11-07                7   
2  St. Thomas (TX) Celts  2025-11-14                6   

                                                                                                                                       uww_run_uww_lineup  \
0  Darius Chestnut, JR Lukenbill, Jake Quast, Joey Berezowitz, Rashad Rogers / Darius Chestnut, Jake Quast, Joey Berezowitz, Rashad Rogers, Richie Warren   
1                                                                                     Brock Marino, Collin Madson, Isaac Verges, Luke Bara, Richie Warren   
2      Collin Madson, Jake Quast, Joey Berezowitz, Kelton McEwen, Richie Warren / JR Lukenbill, Jake Quast, Joey Berezowitz, Kelton McEwen, Richie Warren   

                                                                                                                                     uww_run_o


### Surface clutch-time events with lineup context

Filters `pbp_events` down to the last 5 minutes of the 2nd half or any overtime with the score within 8 points, alongside the lineups on the floor for each event.

In [110]:
# --- Clutch-time event log: last 5 minutes of the 2nd half or any overtime, with the score within 8 points ----
clutch_mask = (
    (pbp_events["period"] != "H1")
    & (pbp_events["time_remaining_seconds"] <= 300)
    & ((pbp_events["uww_score"] - pbp_events["opp_score"]).abs() <= 8)
)
clutch_events = pbp_events[clutch_mask & pbp_events["event_type"].isin([
    "made_shot", "missed_shot", "free_throw_made", "free_throw_missed", "turnover", "steal", "foul", "block", "assist",
])].copy()

if clutch_events.empty:
    print("No clutch-time stretches yet -- no game so far has been within 8 points in the last 5 minutes of the "
          "2nd half or later. This will populate automatically as closer games are uploaded.")
else:
    print(clutch_events[[
        "opponent", "period", "time_remaining", "team", "player", "event_type", "raw_text", "uww_score", "opp_score",
        "uww_lineup", "opp_lineup",
    ]])
    clutch_points = clutch_events[clutch_events["event_type"].isin(["made_shot", "free_throw_made"])].copy()
    clutch_points["points"] = clutch_points.apply(
        lambda row: int(row["shot_type"]) if row["event_type"] == "made_shot" else 1, axis=1
    )
    print("\nClutch-time points scored, by team:")
    print(clutch_points.groupby(GAME_KEYS + ["team"])["points"].sum().reset_index())

    print("\nUWW's 5-man lineup(s) used in clutch time, by game:")
    for (opponent, game_date), group in clutch_events.groupby(GAME_KEYS, dropna=False):
        print(f"  {opponent} {game_date}: {sorted(group['uww_lineup'].dropna().unique().tolist())}")

                  opponent period time_remaining           team     player  \
442  St. Thomas (TX) Celts     H2     00:19 (H2)  UW-Whitewater  Luke Bara   

    event_type                               raw_text  uww_score  opp_score  \
442  made_shot  Luke Bara Makes 3PT Pull Up Jump Shot       73.0       81.0   

                                                               uww_lineup  \
442  Collin Madson, JR Lukenbill, Kelton McEwen, Luke Bara, Richie Warren   

                                                                           opp_lineup  
442  Angel Johnson, Charles Gitonga, Corey Thompson, Nathan Kongolo, Nicholas Buffalo  

Clutch-time points scored, by team:
                opponent   game_date           team  points
0  St. Thomas (TX) Celts  2025-11-14  UW-Whitewater       3

UWW's 5-man lineup(s) used in clutch time, by game:
  St. Thomas (TX) Celts 2025-11-14: ['Collin Madson, JR Lukenbill, Kelton McEwen, Luke Bara, Richie Warren']



### Cross-reference actual PBP turnovers against the Keys-to-Victory splits, by lineup

Compares each game's actual PBP-derived turnover counts (and margin) against UWW's season averages and against whether turnovers were called out as a pre-game key, then breaks the win/loss record down by who actually won the turnover battle.

In [112]:
# --- Cross-reference ACTUAL PBP turnover counts against the Keys-to-Victory splits -----------------------
pbp_turnovers = (
    pbp_events[pbp_events["event_type"] == "turnover"]
    .groupby(["opponent", "team"])
    .size()
    .reset_index(name="turnovers")
)

uww_season_avg_to = float(uww_team_totals["TO"]) if uww_team_totals is not None else None
uww_season_avg_to_forced = float(uww_opp_totals["TO"]) if uww_opp_totals is not None else None

turnover_crossref_rows = []
for opponent in pbp_events["opponent"].dropna().unique():
    uww_to = pbp_turnovers[(pbp_turnovers["opponent"] == opponent) & (pbp_turnovers["team"] == "UW-Whitewater")]["turnovers"]
    opp_to = pbp_turnovers[(pbp_turnovers["opponent"] == opponent) & (pbp_turnovers["team"] == opponent)]["turnovers"]
    uww_to_val = int(uww_to.iloc[0]) if not uww_to.empty else 0
    opp_to_val = int(opp_to.iloc[0]) if not opp_to.empty else 0

    game_row = scouted_game_comparison[scouted_game_comparison["opponent"] == opponent]
    outcome = game_row["outcome"].iloc[0] if not game_row.empty else None
    was_key = "Ball Security / Turnovers" in game_categories.loc[game_categories["opponent"] == opponent, "category"].tolist()

    turnover_crossref_rows.append({
        "opponent": opponent, "outcome": outcome, "ball_security_was_a_key": was_key,
        "uww_actual_turnovers": uww_to_val, "uww_season_avg_turnovers": uww_season_avg_to,
        "uww_vs_season_avg": round(uww_to_val - uww_season_avg_to, 1) if uww_season_avg_to is not None else None,
        "opponent_actual_turnovers": opp_to_val, "uww_season_avg_turnovers_forced": uww_season_avg_to_forced,
        "turnover_margin_uww_favor": opp_to_val - uww_to_val,
    })

turnover_crossref = pd.DataFrame(turnover_crossref_rows)
print(turnover_crossref)

print("Actual PBP turnovers vs. the Keys-to-Victory 'Ball Security / Turnovers' emphasis:\n")
for _, row in turnover_crossref.iterrows():
    key_note = "WAS" if row["ball_security_was_a_key"] else "was NOT"
    print(f"{row['opponent']} ({row['outcome']}): ball security {key_note} called out as a pre-game key.")
    avg_to = row['uww_season_avg_turnovers']
    vs_avg = row['uww_vs_season_avg']
    avg_forced = row['uww_season_avg_turnovers_forced']
    avg_to_str = f"{avg_to:.1f}" if avg_to is not None else "N/A"
    vs_avg_str = f"{vs_avg:+.1f}" if vs_avg is not None else "N/A"
    avg_forced_str = f"{avg_forced:.1f}" if avg_forced is not None else "N/A"
    print(f"    UWW committed {row['uww_actual_turnovers']} turnovers this game vs. their {avg_to_str} season "
          f"average ({vs_avg_str}).")
    print(f"    {row['opponent']} committed {row['opponent_actual_turnovers']} turnovers (UWW forces "
          f"{avg_forced_str}/gm on average).")
    margin_desc = "in UWW's favor" if row["turnover_margin_uww_favor"] > 0 else ("against UWW" if row["turnover_margin_uww_favor"] < 0 else "even")
    print(f"    Turnover margin (opponent TOs minus UWW TOs): {row['turnover_margin_uww_favor']:+d} ({margin_desc})\n")

if len(turnover_crossref) > 1:
    pbp_turnover_split = (
        turnover_crossref.assign(won_turnover_battle=lambda d: d["turnover_margin_uww_favor"] > 0)
        .groupby("won_turnover_battle")["outcome"]
        .agg(games="count", wins=lambda s: (s == "W").sum(), losses=lambda s: (s == "L").sum())
        .reset_index()
    )
    pbp_turnover_split["win_pct"] = (pbp_turnover_split["wins"] / pbp_turnover_split["games"]).round(3)
    print("Win/loss record by who actually won the turnover battle (PBP-verified, once enough games have PBP data):")
    print(pbp_turnover_split)
else:
    print("Only 1 game has play-by-play data so far -- a real win/loss split by turnover-battle outcome will "
          "populate automatically once more games are uploaded.")

to_events = pbp_events[pbp_events["event_type"] == "turnover"]
uww_to_by_lineup = (
    to_events[to_events["team"] == "UW-Whitewater"]
    .groupby(["opponent", "uww_lineup"]).size().reset_index(name="turnovers")
    .sort_values("turnovers", ascending=False)
)
opp_to_by_lineup = (
    to_events[to_events["team"] != "UW-Whitewater"]
    .groupby(["opponent", "opp_lineup"]).size().reset_index(name="turnovers")
    .sort_values("turnovers", ascending=False)
)
print("\nUWW turnovers by 5-man lineup on the floor:")
print(uww_to_by_lineup)
print("\nOpponent turnovers by their 5-man lineup on the floor:")
print(opp_to_by_lineup)

                opponent outcome  ball_security_was_a_key  \
0  St. Thomas (TX) Celts       L                     True   
1      Eureka Red Devils       W                    False   
2        Ripon Red Hawks       W                    False   

   uww_actual_turnovers uww_season_avg_turnovers uww_vs_season_avg  \
0                    15                     None              None   
1                    10                     None              None   
2                    12                     None              None   

   opponent_actual_turnovers uww_season_avg_turnovers_forced  \
0                          8                            None   
1                         17                            None   
2                         14                            None   

   turnover_margin_uww_favor  
0                         -7  
1                          7  
2                          2  
Actual PBP turnovers vs. the Keys-to-Victory 'Ball Security / Turnovers' emphasis:

St. Thoma


### Cross-reference actual PBP rebounding against the Keys-to-Victory splits, by lineup

Same cross-reference as the turnover cell above, but for rebounding -- actual PBP rebound totals/margin vs. season averages and the pre-game "Rebounding" key, plus the win/loss split by who won the rebound battle, and rebounds broken down by 5-man lineup.

In [114]:
# --- Cross-reference ACTUAL PBP rebounding totals against the Keys-to-Victory splits ----------------------
rebound_team_totals = pbp_box_score.groupby(["opponent", "team"])[["REB", "OREB", "DREB"]].sum().reset_index()

uww_season_avg_reb = {c: float(uww_team_totals[c]) for c in ["REB", "ORB", "DRB"]} if uww_team_totals is not None else None
uww_season_avg_reb_allowed = {c: float(uww_opp_totals[c]) for c in ["REB", "ORB", "DRB"]} if uww_opp_totals is not None else None

rebound_crossref_rows = []
for opponent in pbp_events["opponent"].dropna().unique():
    uww_row = rebound_team_totals[(rebound_team_totals["opponent"] == opponent) & (rebound_team_totals["team"] == "UW-Whitewater")]
    opp_row = rebound_team_totals[(rebound_team_totals["opponent"] == opponent) & (rebound_team_totals["team"] == opponent)]
    uww_reb = int(uww_row["REB"].iloc[0]) if not uww_row.empty else 0
    uww_oreb = int(uww_row["OREB"].iloc[0]) if not uww_row.empty else 0
    uww_dreb = int(uww_row["DREB"].iloc[0]) if not uww_row.empty else 0
    opp_reb = int(opp_row["REB"].iloc[0]) if not opp_row.empty else 0
    opp_oreb = int(opp_row["OREB"].iloc[0]) if not opp_row.empty else 0
    opp_dreb = int(opp_row["DREB"].iloc[0]) if not opp_row.empty else 0

    game_row = scouted_game_comparison[scouted_game_comparison["opponent"] == opponent]
    outcome = game_row["outcome"].iloc[0] if not game_row.empty else None
    was_key = "Rebounding" in game_categories.loc[game_categories["opponent"] == opponent, "category"].tolist()

    rebound_crossref_rows.append({
        "opponent": opponent, "outcome": outcome, "rebounding_was_a_key": was_key,
        "uww_actual_reb": uww_reb, "uww_season_avg_reb": uww_season_avg_reb["REB"] if uww_season_avg_reb else None,
        "uww_vs_season_avg_reb": round(uww_reb - uww_season_avg_reb["REB"], 1) if uww_season_avg_reb else None,
        "uww_actual_oreb": uww_oreb, "uww_actual_dreb": uww_dreb,
        "opponent_actual_reb": opp_reb, "uww_season_avg_reb_allowed": uww_season_avg_reb_allowed["REB"] if uww_season_avg_reb_allowed else None,
        "opponent_actual_oreb": opp_oreb, "opponent_actual_dreb": opp_dreb,
        "rebound_margin_uww_favor": uww_reb - opp_reb,
    })

rebound_crossref = pd.DataFrame(rebound_crossref_rows)
print(rebound_crossref)

print("Actual PBP rebounding vs. the Keys-to-Victory 'Rebounding' emphasis:\n")
for _, row in rebound_crossref.iterrows():
    key_note = "WAS" if row["rebounding_was_a_key"] else "was NOT"
    print(f"{row['opponent']} ({row['outcome']}): rebounding {key_note} called out as a pre-game key.")
    avg_reb = row['uww_season_avg_reb']
    vs_avg_reb = row['uww_vs_season_avg_reb']
    avg_allowed = row['uww_season_avg_reb_allowed']
    avg_reb_str = f"{avg_reb:.1f}" if avg_reb is not None else "N/A"
    vs_avg_reb_str = f"{vs_avg_reb:+.1f}" if vs_avg_reb is not None else "N/A"
    avg_allowed_str = f"{avg_allowed:.1f}" if avg_allowed is not None else "N/A"
    print(f"    UWW grabbed {row['uww_actual_reb']} rebounds ({row['uww_actual_oreb']} off. / {row['uww_actual_dreb']} def.) this game vs. "
          f"their {avg_reb_str} season average ({vs_avg_reb_str}).")
    print(f"    {row['opponent']} grabbed {row['opponent_actual_reb']} rebounds ({row['opponent_actual_oreb']} off. / "
          f"{row['opponent_actual_dreb']} def.) -- UWW allows {avg_allowed_str}/gm on average.")
    margin_desc = "in UWW's favor" if row["rebound_margin_uww_favor"] > 0 else ("against UWW" if row["rebound_margin_uww_favor"] < 0 else "even")
    print(f"    Rebound margin (UWW minus opponent): {row['rebound_margin_uww_favor']:+d} ({margin_desc})\n")

if len(rebound_crossref) > 1:
    pbp_rebound_split = (
        rebound_crossref.assign(won_rebound_battle=lambda d: d["rebound_margin_uww_favor"] > 0)
        .groupby("won_rebound_battle")["outcome"]
        .agg(games="count", wins=lambda s: (s == "W").sum(), losses=lambda s: (s == "L").sum())
        .reset_index()
    )
    pbp_rebound_split["win_pct"] = (pbp_rebound_split["wins"] / pbp_rebound_split["games"]).round(3)
    print("Win/loss record by who actually won the rebound battle (PBP-verified, once enough games have PBP data):")
    print(pbp_rebound_split)
else:
    print("Only 1 game has play-by-play data so far -- a real win/loss split by rebound-battle outcome will "
          "populate automatically once more games are uploaded.")

reb_events = pbp_events[pbp_events["event_type"].isin(["rebound_offensive", "rebound_defensive"])]
uww_reb_by_lineup = (
    reb_events[reb_events["team"] == "UW-Whitewater"]
    .groupby(["opponent", "uww_lineup"]).size().reset_index(name="rebounds")
    .sort_values("rebounds", ascending=False)
)
opp_reb_by_lineup = (
    reb_events[reb_events["team"] != "UW-Whitewater"]
    .groupby(["opponent", "opp_lineup"]).size().reset_index(name="rebounds")
    .sort_values("rebounds", ascending=False)
)
print("\nUWW rebounds by 5-man lineup on the floor:")
print(uww_reb_by_lineup)
print("\nOpponent rebounds by their 5-man lineup on the floor:")
print(opp_reb_by_lineup)

                opponent outcome  rebounding_was_a_key  uww_actual_reb  \
0  St. Thomas (TX) Celts       L                  True              27   
1      Eureka Red Devils       W                  True              38   
2        Ripon Red Hawks       W                  True              36   

  uww_season_avg_reb uww_vs_season_avg_reb  uww_actual_oreb  uww_actual_dreb  \
0               None                  None               10               17   
1               None                  None               12               26   
2               None                  None               12               24   

   opponent_actual_reb uww_season_avg_reb_allowed  opponent_actual_oreb  \
0                   25                       None                    10   
1                   22                       None                     6   
2                   18                       None                     5   

   opponent_actual_dreb  rebound_margin_uww_favor  
0                    15      


### Compute scoring margin and minutes played by 5-man lineup combination

Detects lineup-stint boundaries (any substitution for either team) across all of UWW's games, then aggregates minutes played and net scoring margin for each distinct lineup combination.

In [116]:
# --- Scoring margin and minutes played by 5-man lineup combination -----------------------------------------
pbp_scoreable = pbp_events[pbp_events["event_type"] != "period_marker"].sort_values(GAME_KEYS + ["event_order"]).copy()

# Every grouping here is on GAME_KEYS, not "opponent" -- see GAME_KEYS. `event_order` restarts at 0
# for each game, so sorting/grouping by opponent alone interleaves a home-and-home's events and the
# clock diff below re-counts the same seconds over and over (observed: 9x-33x inflated minutes).
prev_uww_lineup = pbp_scoreable.groupby(GAME_KEYS, dropna=False)["uww_lineup"].shift(1)
prev_opp_lineup = pbp_scoreable.groupby(GAME_KEYS, dropna=False)["opp_lineup"].shift(1)
lineup_changed = (pbp_scoreable["uww_lineup"] != prev_uww_lineup) | (pbp_scoreable["opp_lineup"] != prev_opp_lineup)
pbp_scoreable["stint_num"] = lineup_changed.fillna(True).groupby([pbp_scoreable[k] for k in GAME_KEYS]).cumsum()

pbp_scoreable["prev_uww_score"] = pbp_scoreable.groupby(GAME_KEYS, dropna=False)["uww_score"].shift(1).fillna(0)
pbp_scoreable["prev_opp_score"] = pbp_scoreable.groupby(GAME_KEYS, dropna=False)["opp_score"].shift(1).fillna(0)

# Monotone clock, then diff -- see the matching comment in the lineup-box-score cell above.
pbp_scoreable["clock"] = pbp_scoreable.groupby(GAME_KEYS + ["period"], dropna=False)["time_remaining_seconds"].cummin()
pbp_scoreable["prev_time_remaining_seconds"] = pbp_scoreable.groupby(GAME_KEYS + ["period"], dropna=False)["clock"].shift(1)
pbp_scoreable["seconds_elapsed"] = (pbp_scoreable["prev_time_remaining_seconds"] - pbp_scoreable["clock"]).clip(lower=0).fillna(0)

lineup_stints = pbp_scoreable.groupby(GAME_KEYS + ["stint_num", "uww_lineup", "opp_lineup"]).agg(
    start_event_order=("event_order", "min"), end_event_order=("event_order", "max"),
    end_uww_score=("uww_score", "last"), end_opp_score=("opp_score", "last"),
    start_prev_uww_score=("prev_uww_score", "first"), start_prev_opp_score=("prev_opp_score", "first"),
    stint_seconds=("seconds_elapsed", "sum"), n_events=("event_order", "count"),
).reset_index()
lineup_stints["uww_margin_change"] = (
    (lineup_stints["end_uww_score"] - lineup_stints["start_prev_uww_score"])
    - (lineup_stints["end_opp_score"] - lineup_stints["start_prev_opp_score"])
)
lineup_stints["stint_minutes"] = (lineup_stints["stint_seconds"] / 60).round(2)
print(lineup_stints[[
    "opponent", "stint_num", "uww_lineup", "opp_lineup", "stint_minutes", "uww_margin_change", "n_events",
]].sort_values(["opponent", "stint_num"]))

# Per-game sanity check: five players x 40 minutes means each game must total about 200 minutes.
_clock_check = lineup_stints.groupby(GAME_KEYS)["stint_minutes"].sum().round(1)
_clock_bad = _clock_check[(_clock_check < 35) | (_clock_check > 65)]
if len(_clock_bad):
    print(f"WARNING: {len(_clock_bad)} game(s) have an implausible total elapsed clock "
          f"(expected ~40 min per game):\n{_clock_bad.to_string()}\n")
else:
    print(f"Clock check: all {len(_clock_check)} game(s) between 35 and 65 elapsed minutes.\n")

uww_lineup_summary = (
    lineup_stints.groupby(["opponent", "uww_lineup"])
    .agg(stints=("stint_num", "count"), total_minutes=("stint_minutes", "sum"), net_margin=("uww_margin_change", "sum"))
    .reset_index()
)
uww_lineup_summary["margin_per_min"] = (uww_lineup_summary["net_margin"] / uww_lineup_summary["total_minutes"]).round(2)
uww_lineup_summary = uww_lineup_summary.sort_values("net_margin", ascending=False)

opp_lineup_summary = (
    lineup_stints.groupby(["opponent", "opp_lineup"])
    .agg(stints=("stint_num", "count"), total_minutes=("stint_minutes", "sum"), net_margin_for_uww=("uww_margin_change", "sum"))
    .reset_index()
)
opp_lineup_summary["margin_per_min_for_uww"] = (opp_lineup_summary["net_margin_for_uww"] / opp_lineup_summary["total_minutes"]).round(2)
opp_lineup_summary = opp_lineup_summary.sort_values("net_margin_for_uww", ascending=True)

print("UWW's 5-man lineups, ranked by net scoring margin while on the floor:\n")
for _, row in uww_lineup_summary.iterrows():
    print(f"[{row['opponent']}] {row['uww_lineup']}")
    print(f"    {row['total_minutes']:.1f} min over {row['stints']} stint(s), net margin {row['net_margin']:+.0f} "
          f"({row['margin_per_min']:+.2f}/min)\n")
print(uww_lineup_summary)

print(f"\n{pbp_events['opponent'].dropna().unique().tolist()} lineups, ranked by how they fared AGAINST UWW (most negative net_margin_for_uww = best for them):\n")
for _, row in opp_lineup_summary.iterrows():
    print(f"[{row['opponent']}] {row['opp_lineup']}")
    print(f"    {row['total_minutes']:.1f} min over {row['stints']} stint(s), UWW's net margin vs. this lineup "
          f"{row['net_margin_for_uww']:+.0f} ({row['margin_per_min_for_uww']:+.2f}/min)\n")
print(opp_lineup_summary)

                  opponent  stint_num  \
0        Eureka Red Devils          1   
1        Eureka Red Devils          2   
2        Eureka Red Devils          3   
3        Eureka Red Devils          4   
4        Eureka Red Devils          5   
..                     ...        ...   
123  St. Thomas (TX) Celts         33   
124  St. Thomas (TX) Celts         34   
125  St. Thomas (TX) Celts         35   
126  St. Thomas (TX) Celts         36   
127  St. Thomas (TX) Celts         37   

                                                                     uww_lineup  \
0            Brock Marino, Collin Madson, Isaac Verges, JR Lukenbill, Luke Bara   
1              Brock Marino, Collin Madson, Isaac Verges, Jake Quast, Luke Bara   
2    Collin Madson, Darius Chestnut, Jake Quast, Joey Berezowitz, Rashad Rogers   
3    Collin Madson, Darius Chestnut, Jake Quast, Joey Berezowitz, Rashad Rogers   
4    Collin Madson, Darius Chestnut, Jake Quast, Joey Berezowitz, Rashad Rogers   
..       


### Add minutes played onto the reconstructed box score

pbp_box_score (built above, before lineup_stints existed) aggregates scoring/rebounding/etc. events, none
of which map to a player's own on-court time by themselves -- so it never had a MIN column. lineup_stints
already reconstructs exactly that (via substitution tracking, the same 5-man units the Lineup Simulator and
Lineup Scouting features use), so minutes played per player per game is just those stint minutes summed up
for every stint that player's name appears in.

In [118]:
# --- Derive per-player minutes played from lineup_stints and merge onto pbp_box_score ------------------
_minutes_rows = []
for _, _stint in lineup_stints.iterrows():
    if pd.notna(_stint["uww_lineup"]):
        for _player in str(_stint["uww_lineup"]).split(", "):
            _minutes_rows.append({"opponent": _stint["opponent"], "game_date": _stint["game_date"], "team": "UW-Whitewater", "player": _player, "minutes": _stint["stint_minutes"]})
    if pd.notna(_stint["opp_lineup"]):
        for _player in str(_stint["opp_lineup"]).split(", "):
            _minutes_rows.append({"opponent": _stint["opponent"], "game_date": _stint["game_date"], "team": _stint["opponent"], "player": _player, "minutes": _stint["stint_minutes"]})

if _minutes_rows:
    player_minutes = pd.DataFrame(_minutes_rows).groupby(GAME_KEYS + ["team", "player"])["minutes"].sum().reset_index()
    player_minutes["MIN"] = player_minutes["minutes"].round(1)
    if "MIN" in pbp_box_score.columns:
        pbp_box_score = pbp_box_score.drop(columns=["MIN"])
    # Merged on the game, not just the opponent. Merging on (opponent, team, player) attached ONE
    # combined minutes figure to EVERY meeting's row against that opponent.
    pbp_box_score = pbp_box_score.merge(player_minutes[GAME_KEYS + ["team", "player", "MIN"]], on=GAME_KEYS + ["team", "player"], how="left")
    _n_matched = int(pbp_box_score["MIN"].notna().sum())
    print(f"Matched minutes played for {_n_matched} of {len(pbp_box_score)} pbp_box_score row(s).")
else:
    pbp_box_score["MIN"] = None
    print("No lineup stints available yet -- pbp_box_score.MIN left empty.")


Matched minutes played for 90 of 91 pbp_box_score row(s).



### Identify the lineups on court during each team's biggest scoring run

For each game's biggest scoring run (computed earlier), lists every lineup substitution either team made mid-run.

In [120]:
# --- Which lineup was on the floor for each team's biggest scoring run -------------------------------------
scoring_events_detail = pbp_events[pbp_events["event_type"].isin(["made_shot", "free_throw_made"])].copy()
scoring_events_detail["points"] = scoring_events_detail.apply(
    lambda row: int(row["shot_type"]) if row["event_type"] == "made_shot" else 1, axis=1
)

def detailed_runs(group):
    runs, cur_team, cur_pts, cur_start, cur_end = [], None, 0, None, None
    for _, row in group.sort_values("event_order").iterrows():
        if row["team"] == cur_team:
            cur_pts += row["points"]
            cur_end = row["event_order"]
        else:
            if cur_team is not None:
                runs.append({"team": cur_team, "run_points": cur_pts, "start_event_order": cur_start, "end_event_order": cur_end})
            cur_team, cur_pts, cur_start, cur_end = row["team"], row["points"], row["event_order"], row["event_order"]
    if cur_team is not None:
        runs.append({"team": cur_team, "run_points": cur_pts, "start_event_order": cur_start, "end_event_order": cur_end})
    return pd.DataFrame(runs)

all_runs_detail = []
for (opponent, game_date), group in scoring_events_detail.groupby(GAME_KEYS, dropna=False):
    rd = detailed_runs(group)
    rd["opponent"] = opponent
    rd["game_date"] = game_date
    all_runs_detail.append(rd)

# CONFIRMED BUG (fixed here): if scoring_events_detail has zero rows -- e.g. reference_date is set to
# ON OR BEFORE your most recently played game rather than after it, which would filter that game's
# events entirely out of pbp_events (see the "Play-by-play (PBP) data" cell) -- the loop above never
# appends anything, and pd.concat([]) raised "ValueError: No objects to concatenate". Worth
# double-checking reference_date for that reason if this fires unexpectedly with games actually played.
# An empty-but-correctly-shaped frame here lets the groupby/first() and print loop below run through
# cleanly with zero rows instead of crashing.
if all_runs_detail:
    all_runs_detail = pd.concat(all_runs_detail, ignore_index=True)
else:
    all_runs_detail = pd.DataFrame(columns=["team", "run_points", "start_event_order", "end_event_order", "opponent", "game_date"])

biggest_runs_detail = (
    all_runs_detail.sort_values("run_points", ascending=False)
    .groupby(GAME_KEYS + ["team"], as_index=False)
    .first()
)

print("Lineups on the floor during each team's biggest scoring run:\n")
for _, run in biggest_runs_detail.iterrows():
    window = pbp_events[
        (pbp_events["opponent"] == run["opponent"])
        & (pbp_events["game_date"] == run["game_date"])
        & (pbp_events["event_order"] >= run["start_event_order"])
        & (pbp_events["event_order"] <= run["end_event_order"])
    ].sort_values("event_order")
    uww_lineups_during = window["uww_lineup"].dropna().unique().tolist()
    opp_lineups_during = window["opp_lineup"].dropna().unique().tolist()
    start_row, end_row = window.iloc[0], window.iloc[-1]

    print(f"[{run['opponent']}] {run['team']}'s biggest run: {run['run_points']} points "
          f"({start_row['period']} {start_row['time_remaining']} -> {end_row['period']} {end_row['time_remaining']})")
    if len(uww_lineups_during) == 1:
        print(f"    UWW lineup on the floor the whole run: {uww_lineups_during[0]}")
    else:
        print(f"    UWW made a substitution mid-run -- lineups on the floor: {uww_lineups_during}")
    if len(opp_lineups_during) == 1:
        print(f"    {run['opponent']} lineup on the floor the whole run: {opp_lineups_during[0]}")
    else:
        print(f"    {run['opponent']} made a substitution mid-run -- lineups on the floor: {opp_lineups_during}")
    print()

print(biggest_runs_detail)

Lineups on the floor during each team's biggest scoring run:

[Eureka Red Devils] Eureka Red Devils's biggest run: 5 points (H2 19:12 (H2) -> H2 18:32 (H2))
    UWW lineup on the floor the whole run: Brock Marino, Collin Madson, Isaac Verges, JR Lukenbill, Luke Bara
    Eureka Red Devils lineup on the floor the whole run: Andrew Coker, Ben Carter, Colin DeLaere, Jaxson Provost, Micah Bruer

[Eureka Red Devils] UW-Whitewater's biggest run: 12 points (H1 12:55 (H1) -> H1 10:00 (H1))
    UWW made a substitution mid-run -- lineups on the floor: ['Darius Chestnut, JR Lukenbill, Jake Quast, Joey Berezowitz, Rashad Rogers', 'Darius Chestnut, Jake Quast, Joey Berezowitz, Rashad Rogers, Richie Warren']
    Eureka Red Devils made a substitution mid-run -- lineups on the floor: ['Blake Logsdon, Damuzha Moore, Jacob Gonzalez, Jonah Lauff, Max Richardson', 'Andrew Wells, Damuzha Moore, Jacob Gonzalez, Jaxson Provost, Jonah Lauff']

[Ripon Red Hawks] Ripon Red Hawks's biggest run: 10 points (H1 19:2


### Season-wide 5-man lineup analysis across all games

Combines the lineup-stint data (minutes, margin) across every game played so far into one season-wide view of UWW's most-used and most-effective 5-man combinations.

In [122]:
# --- Season-wide UWW 5-man lineup analysis, combining minutes/margin across ALL games played so far -----------
season_uww_lineups = (
    lineup_stints.groupby("uww_lineup")
    .agg(
        games=("game_date", "nunique"),   # distinct GAMES, not distinct opponents -- a home-and-home is 2
        opponents=("opponent", lambda s: sorted(s.unique())),
        stints=("stint_num", "count"),
        total_minutes=("stint_minutes", "sum"),
        net_margin=("uww_margin_change", "sum"),
    )
    .reset_index()
)
season_uww_lineups["margin_per_min"] = (season_uww_lineups["net_margin"] / season_uww_lineups["total_minutes"]).round(2)
season_uww_lineups = season_uww_lineups.sort_values("total_minutes", ascending=False)

n_games_with_pbp = pbp_events.dropna(subset=["opponent"])[GAME_KEYS].drop_duplicates().shape[0]
print(f"Season-wide UWW 5-man lineups across {n_games_with_pbp} game(s) with play-by-play data "
      f"({season_uww_lineups.shape[0]} distinct lineups used):\n")
print("Most-used lineups overall (by total minutes on the floor):")
print(season_uww_lineups.head(10))

MEANINGFUL_MIN_MINUTES = 2.0
meaningful = season_uww_lineups[season_uww_lineups["total_minutes"] >= MEANINGFUL_MIN_MINUTES]
print(f"\nBest net-margin-per-minute UWW lineups season-wide (min {MEANINGFUL_MIN_MINUTES} minutes played):")
print(meaningful.sort_values("margin_per_min", ascending=False).head(10))
print(f"\nWorst net-margin-per-minute UWW lineups season-wide (min {MEANINGFUL_MIN_MINUTES} minutes played):")
print(meaningful.sort_values("margin_per_min", ascending=True).head(10))

recurring = season_uww_lineups[season_uww_lineups["games"] > 1].sort_values("games", ascending=False)
if recurring.empty:
    print("\nNo single 5-man lineup has repeated across multiple games yet -- each game so far has drawn from a "
          "distinct set of on-court combinations. This will populate as more games are added.")
else:
    print(f"\nLineups that have appeared in MORE than one game ({len(recurring)}):")
    print(recurring)

game_starting_lineups = (
    pbp_scoreable.sort_values(GAME_KEYS + ["event_order"])
    .groupby(GAME_KEYS, dropna=False)["uww_lineup"].first()
    .reset_index(name="starting_lineup")
)
print("\nUWW's starting (tip-off) lineup, by game:")
print(game_starting_lineups)
if game_starting_lineups["starting_lineup"].nunique() == 1:
    print("\nSame starting five has been used in EVERY game so far.")
else:
    print(f"\nStarting five has changed across games -- {game_starting_lineups['starting_lineup'].nunique()} "
          f"different starting combinations used over {n_games_with_pbp} game(s).")

lineups_per_game = lineup_stints.groupby(GAME_KEYS, dropna=False)["uww_lineup"].nunique().reset_index(name="distinct_lineups_used")
print("\nDistinct UWW 5-man combinations used, by game:")
print(lineups_per_game)

Season-wide UWW 5-man lineups across 3 game(s) with play-by-play data (53 distinct lineups used):

Most-used lineups overall (by total minutes on the floor):
                                                                         uww_lineup  \
21               Brock Marino, Collin Madson, Isaac Verges, JR Lukenbill, Luke Bara   
14         Brock Marino, Collin Madson, Darius Chestnut, Isaac Verges, JR Lukenbill   
47       Darius Chestnut, Jake Quast, Joey Berezowitz, Rashad Rogers, Richie Warren   
23              Brock Marino, Collin Madson, Isaac Verges, Luke Bara, Richie Warren   
22                 Brock Marino, Collin Madson, Isaac Verges, Jake Quast, Luke Bara   
26            Brock Marino, Collin Madson, Jake Quast, Kelton McEwen, Richie Warren   
30             Brock Marino, Darius Chestnut, Isaac Verges, JR Lukenbill, Luke Bara   
12  Austin Ambrose, Isaiah Robinson, Kelton McEwen, Trent Grunewald, Tristian Lynch   
11      Austin Ambrose, Darius Chestnut, Jake Quast, Joey B


### UWW performance vs. opponent lineup profile and video-tagged play types, season-wide

Merges lineup stints with each opponent lineup's scouting-tag classification (Starter/Bench role mix) to see how UWW's net margin varies against different opponent lineup profiles, season-wide.

In [124]:
# --- UWW performance vs. OPPONENT lineup profile, and video-tagged PLAY TYPES -- both season-wide -------------
opp_lineup_class = pbp_events[GAME_KEYS + ["opp_lineup", "opp_lineup_summary"]].drop_duplicates()
stints_with_opp_class = lineup_stints.merge(opp_lineup_class, on=GAME_KEYS + ["opp_lineup"], how="left")

def opp_role_bucket(summary):
    if pd.isna(summary):
        return "Unknown (no scouting match)"
    m = re.search(r"(\d+) Starter", summary)
    if not m:
        return "Unknown (no scouting match)"
    starters = int(m.group(1))
    if starters >= 4:
        return "Starter-heavy (4-5 starters)"
    if starters <= 1:
        return "Bench-heavy (0-1 starters)"
    return "Mixed (2-3 starters)"

stints_with_opp_class["opp_role_bucket"] = stints_with_opp_class["opp_lineup_summary"].apply(opp_role_bucket)
role_bucket_summary = (
    stints_with_opp_class.groupby("opp_role_bucket")
    .agg(stints=("stint_num", "count"), total_minutes=("stint_minutes", "sum"), net_margin_for_uww=("uww_margin_change", "sum"))
    .reset_index()
)
role_bucket_summary["margin_per_min"] = (role_bucket_summary["net_margin_for_uww"] / role_bucket_summary["total_minutes"]).round(2)
role_bucket_summary = role_bucket_summary.sort_values("total_minutes", ascending=False)
print("UWW's season-wide net margin by OPPONENT lineup role composition (Starter/Bench mix on the floor):\n")
print(role_bucket_summary)

def extract_style_tags(summary):
    if pd.isna(summary):
        return []
    m = re.search(r"Style: ([^|]+)", summary)
    if not m or "no tagged traits" in m.group(1):
        return []
    return [t.split(" x")[0].strip() for t in m.group(1).split(",")]

stints_with_opp_class["opp_style_tags"] = stints_with_opp_class["opp_lineup_summary"].apply(extract_style_tags)
style_summary = (
    stints_with_opp_class.explode("opp_style_tags").dropna(subset=["opp_style_tags"])
    .groupby("opp_style_tags")
    .agg(stints=("stint_num", "count"), total_minutes=("stint_minutes", "sum"), net_margin_for_uww=("uww_margin_change", "sum"))
    .reset_index()
)
style_summary["margin_per_min"] = (style_summary["net_margin_for_uww"] / style_summary["total_minutes"]).round(2)
style_summary = style_summary.sort_values("total_minutes", ascending=False)
print("\nUWW's season-wide net margin by the OPPONENT lineup's dominant scouted playing style (a lineup can "
      "count toward more than one tag):\n")
print(style_summary)

def extract_play_type(description, player):
    if pd.isna(description) or pd.isna(player):
        return None
    segments = [s.strip() for s in description.split(" > ")]
    player_norm = normalize_player_name(player)
    last_player_idx = None
    for idx, seg in enumerate(segments):
        m = re.match(r"^\d+\s+(.+)$", seg)
        if m and normalize_player_name(m.group(1)) == player_norm:
            last_player_idx = idx
    if last_player_idx is not None and last_player_idx + 1 < len(segments):
        return segments[last_player_idx + 1]
    return segments[1] if len(segments) > 1 else None

uww_shot_rows = pbp_events[
    (pbp_events["team"] == "UW-Whitewater")
    & pbp_events["event_type"].isin(["made_shot", "missed_shot"])
    & pbp_events["video_description"].notna()
].copy()
uww_shot_rows["play_type"] = uww_shot_rows.apply(lambda r: extract_play_type(r["video_description"], r["player"]), axis=1)
uww_shot_rows["made"] = uww_shot_rows["event_type"] == "made_shot"

play_type_summary = (
    uww_shot_rows.groupby("play_type")
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
play_type_summary["fg_pct"] = (100 * play_type_summary["makes"] / play_type_summary["attempts"]).round(1)
play_type_summary = play_type_summary.sort_values("attempts", ascending=False)
print(f"\nUWW's video-tagged shot-attempt play types, season-wide ({len(uww_shot_rows)} video-matched attempts "
      f"across {uww_shot_rows['opponent'].nunique()} game(s)):\n")
print(play_type_summary)

play_type_by_lineup = (
    uww_shot_rows.groupby(["play_type", "uww_lineup"])
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
play_type_by_lineup["fg_pct"] = (100 * play_type_by_lineup["makes"] / play_type_by_lineup["attempts"]).round(1)
top_play_types = play_type_summary.head(5)["play_type"].tolist()
print(f"\nFor the top {len(top_play_types)} most-attempted play types, which UWW lineup ran them most:\n")
print(
    play_type_by_lineup[play_type_by_lineup["play_type"].isin(top_play_types)]
    .sort_values(["play_type", "attempts"], ascending=[True, False])
)

play_type_by_player = (
    uww_shot_rows.groupby(["player", "play_type"])
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
play_type_by_player["fg_pct"] = (100 * play_type_by_player["makes"] / play_type_by_player["attempts"]).round(1)
print("\nUWW's video-tagged play-type efficiency by individual player, season-wide (every player/play-type "
      "combination with at least one attempt):\n")
print(play_type_by_player.sort_values(["player", "attempts"], ascending=[True, False]))

player_top_play_type = (
    play_type_by_player.sort_values("attempts", ascending=False)
    .groupby("player", as_index=False)
    .first()
    .rename(columns={"play_type": "most_used_play_type", "attempts": "attempts_of_that_type", "makes": "makes_of_that_type"})
)
player_overall = (
    uww_shot_rows.groupby("player")
    .agg(total_attempts=("made", "count"), total_makes=("made", "sum"))
    .reset_index()
)
player_overall["overall_fg_pct"] = (100 * player_overall["total_makes"] / player_overall["total_attempts"]).round(1)
player_summary = player_overall.merge(player_top_play_type, on="player", how="left").sort_values(
    "total_attempts", ascending=False
)
print("\nPer-player summary -- overall video-tagged volume/efficiency plus each player's single most-used play "
      "type, season-wide:\n")
print(player_summary[[
    "player", "total_attempts", "total_makes", "overall_fg_pct",
    "most_used_play_type", "attempts_of_that_type", "makes_of_that_type",
]])

starter_flags = pbp_box_score[pbp_box_score["team"] == "UW-Whitewater"][["opponent", "player", "started"]].drop_duplicates()
uww_shot_rows_roles = uww_shot_rows.merge(starter_flags, on=["opponent", "player"], how="left")
uww_shot_rows_roles["role"] = uww_shot_rows_roles["started"].map({True: "Starter", False: "Bench"}).fillna("Unknown")
n_unknown_role = (uww_shot_rows_roles["role"] == "Unknown").sum()
if n_unknown_role:
    print(f"NOTE: {n_unknown_role} attempt(s) could not be matched to a started/bench flag in pbp_box_score "
          f"and are grouped as 'Unknown' below.")

role_overall = (
    uww_shot_rows_roles.groupby("role")
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
role_overall["fg_pct"] = (100 * role_overall["makes"] / role_overall["attempts"]).round(1)
print("\nStarters vs. Bench: overall video-tagged shooting, season-wide:\n")
print(role_overall)

role_by_play_type = (
    uww_shot_rows_roles.groupby(["role", "play_type"])
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
role_by_play_type["fg_pct"] = (100 * role_by_play_type["makes"] / role_by_play_type["attempts"]).round(1)
role_by_play_type = role_by_play_type.sort_values(["role", "attempts"], ascending=[True, False])
print("\nStarters vs. Bench shooting, broken down by play type, season-wide:\n")
print(role_by_play_type)

UWW's season-wide net margin by OPPONENT lineup role composition (Starter/Bench mix on the floor):

                opp_role_bucket  stints  total_minutes  net_margin_for_uww  \
2  Starter-heavy (4-5 starters)      35          51.59                22.0   
1          Mixed (2-3 starters)      68          48.87                13.0   
3   Unknown (no scouting match)      17          14.47                 8.0   
0    Bench-heavy (0-1 starters)       8           4.09                10.0   

   margin_per_min  
2            0.43  
1            0.27  
3            0.55  
0            2.44  

UWW's season-wide net margin by the OPPONENT lineup's dominant scouted playing style (a lineup can count toward more than one tag):

        opp_style_tags  stints  total_minutes  net_margin_for_uww  \
6  three_point_shooter      90          84.80                25.0   
5       slasher_driver      60          59.06                16.0   
4            rebounder      35          35.35                29.0   


### Diagnose Spot-Up struggles: shot quality vs. shooter-specific

Breaks UWW's season-wide Spot-Up shooting down by shot mechanic (catch-and-shoot vs. pull-up vs. drive) and by contest level (guarded vs. open), to see whether struggles are about shot quality/shot selection rather than any one shooter.

In [126]:
# --- Diagnosing UWW's Spot-Up struggles: is it shot QUALITY (contested/long attempts) or SHOOTER-specific? ----
# The tagger's own vocabulary doesn't name every shot; these two labels mark where it runs out. Kept as
# named constants so the "best shot type" logic can recognise a residual bucket instead of presenting it as
# a real shot type.
UNCLASSIFIED_SHOT_MECHANIC = "Unclassified (no mechanic tag)"
NO_CONTEST_TAG = "Not tagged (contest recorded only on catch-and-shoot)"


def extract_shot_mechanic(description):
    """Which kind of shot this was, from the video tagger's own chained description.

    The first three tests are the tagger's SHOT MECHANIC vocabulary. Everything else used to fall through to
    a bucket called "Other" -- which was 19% of all tagged shots and, at 58.9%, the most efficient bucket on
    the board, so it kept winning "best shot type" while telling a coach nothing. Reading the raw tags, it was
    cuts to the rim, putbacks off the offensive glass, and post-ups.

    Those three are tested AFTER the mechanic tests, not before: they describe how a shot was CREATED rather
    than how it was released, and a post-up that finishes as a jumper should still count as a jumper. Checked
    against real tagged data -- "Cut" and "Offensive Rebound" appear in zero already-classified shots, and
    "Post-Up" in 169, all of which keep their existing (more specific) label under this ordering. Adding the
    tier shrinks the residual from 586 shots to 14.
    """
    if pd.isna(description):
        return None
    d = str(description)
    if "No Dribble Jumper" in d:
        return "Catch-and-shoot"
    if "Dribble Jumper" in d:
        return "Pull-up off the dribble"
    if "To Basket" in d:
        return "Drive to the basket"
    # --- fallback tier: shot ORIGIN, for tags carrying no mechanic keyword at all ---
    if "Offensive Rebound" in d:
        return "Putback off the offensive glass"
    if "Cut" in d:
        return "Cut to the basket"
    if "Post-Up" in d:
        return "Post-up"
    return UNCLASSIFIED_SHOT_MECHANIC


def extract_contest(description):
    """Defender contest, which the tagger records ONLY on catch-and-shoot jumpers.

    Verified across 3,039 tagged shots: every one of the 1,069 catch-and-shoot attempts carries Guarded or
    Open, and not one of the other 1,970 does. So a missing contest tag does not mean the shot was a drive --
    the previous label said "(drive, no contest tag)", which mislabelled every cut, putback and post-up as a
    drive. It means the contest dimension simply does not apply to this shot type.
    """
    if pd.isna(description):
        return None
    d = str(description)
    if "Guarded" in d:
        return "Guarded"
    if "Open" in d:
        return "Open"
    return NO_CONTEST_TAG

def extract_distance(description):
    if pd.isna(description):
        return None
    for tag in ["Long/3pt", "Medium/17' to <3p", "Short to < 17'"]:
        if tag in description:
            return tag
    return "N/A"

spotup = uww_shot_rows[uww_shot_rows["play_type"] == "Spot-Up"].copy()
spotup["shot_mechanic"] = spotup["video_description"].apply(extract_shot_mechanic)
spotup["contest"] = spotup["video_description"].apply(extract_contest)
spotup["distance"] = spotup["video_description"].apply(extract_distance)

print(f"Spot-Up shot-quality breakdown, season-wide ({len(spotup)} video-matched attempts):\n")

mechanic_summary = spotup.groupby("shot_mechanic").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
mechanic_summary["fg_pct"] = (100 * mechanic_summary["makes"] / mechanic_summary["attempts"]).round(1)
print("By shot mechanic:")
print(mechanic_summary.sort_values("attempts", ascending=False))

contest_summary = spotup.groupby("contest").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
contest_summary["fg_pct"] = (100 * contest_summary["makes"] / contest_summary["attempts"]).round(1)
print("\nBy contest level:")
print(contest_summary.sort_values("attempts", ascending=False))

distance_summary = spotup.groupby("distance").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
distance_summary["fg_pct"] = (100 * distance_summary["makes"] / distance_summary["attempts"]).round(1)
print("\nBy shot distance:")
print(distance_summary.sort_values("attempts", ascending=False))

catch_shoot = spotup[spotup["shot_mechanic"] == "Catch-and-shoot"]
cs_combo = catch_shoot.groupby(["contest", "distance"]).agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
cs_combo["fg_pct"] = (100 * cs_combo["makes"] / cs_combo["attempts"]).round(1)
print(f"\nCatch-and-shoot Spot-Up jumpers only ({len(catch_shoot)} attempts) -- contest x distance:")
print(cs_combo.sort_values("attempts", ascending=False))

player_spotup = spotup.groupby("player").agg(
    attempts=("made", "count"), makes=("made", "sum"),
    pct_catch_and_shoot=("shot_mechanic", lambda s: round(100 * (s == "Catch-and-shoot").mean(), 1)),
    pct_guarded=("contest", lambda s: round(100 * (s == "Guarded").mean(), 1)),
).reset_index()
player_spotup["fg_pct"] = (100 * player_spotup["makes"] / player_spotup["attempts"]).round(1)
player_spotup = player_spotup.sort_values("attempts", ascending=False)
print("\nPer-player Spot-Up volume/efficiency, with their catch-and-shoot% and guarded% of those attempts:")
print(player_spotup[["player", "attempts", "makes", "fg_pct", "pct_catch_and_shoot", "pct_guarded"]])

Spot-Up shot-quality breakdown, season-wide (46 video-matched attempts):

By shot mechanic:
             shot_mechanic  attempts  makes  fg_pct
0          Catch-and-shoot        35     19    54.3
1      Drive to the basket         8      2    25.0
2  Pull-up off the dribble         3      1    33.3

By contest level:
                                                 contest  attempts  makes  \
0                                                Guarded        18     11   
2                                                   Open        17      8   
1  Not tagged (contest recorded only on catch-and-shoot)        11      3   

   fg_pct  
0    61.1  
2    47.1  
1    27.3  

By shot distance:
            distance  attempts  makes  fg_pct
0           Long/3pt        36     20    55.6
2                N/A         8      2    25.0
1  Medium/17' to <3p         1      0     0.0
3     Short to < 17'         1      0     0.0

Catch-and-shoot Spot-Up jumpers only (35 attempts) -- contest x distance:



### Check contest/mechanic patterns across ALL play types, not just Spot-Up

Extends the same shot-mechanic/contest-level breakdown from the cell above beyond Spot-Up, to every play type -- checking whether the same pattern holds more broadly or is specific to Spot-Up situations.

In [128]:
# --- Does the guarded/open contest pattern hold for catch-and-shoot jumpers on NON-SPOT-UP play types too? ----
uww_shot_rows["shot_mechanic"] = uww_shot_rows["video_description"].apply(extract_shot_mechanic)
uww_shot_rows["contest"] = uww_shot_rows["video_description"].apply(extract_contest)
uww_shot_rows["distance"] = uww_shot_rows["video_description"].apply(extract_distance)

catch_and_shoot_all = uww_shot_rows[uww_shot_rows["shot_mechanic"] == "Catch-and-shoot"]
print(f"ALL catch-and-shoot jumpers, every play type, season-wide ({len(catch_and_shoot_all)} attempts):\n")
by_play_type_mechanic = (
    catch_and_shoot_all.groupby("play_type")
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
by_play_type_mechanic["fg_pct"] = (100 * by_play_type_mechanic["makes"] / by_play_type_mechanic["attempts"]).round(1)
print("Which play types produce catch-and-shoot jumpers, and their efficiency:")
print(by_play_type_mechanic.sort_values("attempts", ascending=False))

contest_all = catch_and_shoot_all.groupby("contest").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
contest_all["fg_pct"] = (100 * contest_all["makes"] / contest_all["attempts"]).round(1)
print("\nGuarded vs. Open, ALL catch-and-shoot jumpers (every play type combined), season-wide:")
print(contest_all.sort_values("attempts", ascending=False))

guarded_all = (
    catch_and_shoot_all[catch_and_shoot_all["contest"] == "Guarded"].groupby("player")
    .agg(guarded_attempts=("made", "count"), guarded_makes=("made", "sum"))
    .reset_index()
)
guarded_all["guarded_fg_pct"] = (100 * guarded_all["guarded_makes"] / guarded_all["guarded_attempts"]).round(1)
open_all = (
    catch_and_shoot_all[catch_and_shoot_all["contest"] == "Open"].groupby("player")
    .agg(open_attempts=("made", "count"), open_makes=("made", "sum"))
    .reset_index()
)
open_all["open_fg_pct"] = (100 * open_all["open_makes"] / open_all["open_attempts"]).round(1)
compare_all = guarded_all.merge(open_all, on="player", how="outer")
compare_all["total_attempts"] = compare_all[["guarded_attempts", "open_attempts"]].sum(axis=1, skipna=True)
compare_all = compare_all.sort_values("total_attempts", ascending=False)
print("\nGuarded vs. Open catch-and-shoot jumpers, by player, ALL play types combined, season-wide:")
print(compare_all[[
    "player", "guarded_attempts", "guarded_makes", "guarded_fg_pct",
    "open_attempts", "open_makes", "open_fg_pct", "total_attempts",
]])

non_jumper = uww_shot_rows[uww_shot_rows["shot_mechanic"] != "Catch-and-shoot"]
non_jumper_summary = (
    non_jumper.groupby(["play_type", "shot_mechanic"])
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
non_jumper_summary["fg_pct"] = (100 * non_jumper_summary["makes"] / non_jumper_summary["attempts"]).round(1)
print("\nNon-catch-and-shoot attempts (drives, pull-ups, post moves, etc.), by play type and mechanic:")
print(non_jumper_summary.sort_values("attempts", ascending=False))

ALL catch-and-shoot jumpers, every play type, season-wide (54 attempts):

Which play types produce catch-and-shoot jumpers, and their efficiency:
          play_type  attempts  makes  fg_pct
6           Spot-Up        35     19    54.3
7        Transition         8      2    25.0
2        Off Screen         5      2    40.0
4      P&R Roll Man         2      1    50.0
0          Hand Off         1      0     0.0
1               ISO         1      0     0.0
3  P&R Ball Handler         1      0     0.0
5           Post-Up         1      0     0.0

Guarded vs. Open, ALL catch-and-shoot jumpers (every play type combined), season-wide:
   contest  attempts  makes  fg_pct
0  Guarded        28     14    50.0
1     Open        26     10    38.5

Guarded vs. Open catch-and-shoot jumpers, by player, ALL play types combined, season-wide:
             player  guarded_attempts  guarded_makes  guarded_fg_pct  \
9         Luke Bara               6.0            3.0            50.0   
4   Darius Chestn


### Contest-level (guarded vs. open) Spot-Up breakdown for all shooters

Same guarded-vs-open contest breakdown as the flagged-player diagnosis, but run across every shooter on the roster rather than just the one player originally flagged.

In [130]:
# --- Same contest-level breakdown as the flagged-player note above, generalized to EVERY shooter with Spot-Up
# volume -- so the pattern (missing mostly on guarded/contested looks vs. mostly on open looks) can be checked
# player by player, not just for one flagged player.
player_contest = (
    spotup.groupby(["player", "contest"])
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
player_contest["fg_pct"] = (100 * player_contest["makes"] / player_contest["attempts"]).round(1)
player_contest = player_contest.sort_values(["player", "attempts"], ascending=[True, False])
print("Spot-Up shooting by player and contest level, season-wide (every player/contest combination with at "
      "least one attempt):\n")
print(player_contest)

guarded_summary = (
    spotup[spotup["contest"] == "Guarded"].groupby("player")
    .agg(guarded_attempts=("made", "count"), guarded_makes=("made", "sum"))
    .reset_index()
)
guarded_summary["guarded_fg_pct"] = (100 * guarded_summary["guarded_makes"] / guarded_summary["guarded_attempts"]).round(1)

open_summary = (
    spotup[spotup["contest"] == "Open"].groupby("player")
    .agg(open_attempts=("made", "count"), open_makes=("made", "sum"))
    .reset_index()
)
open_summary["open_fg_pct"] = (100 * open_summary["open_makes"] / open_summary["open_attempts"]).round(1)

contest_compare = guarded_summary.merge(open_summary, on="player", how="outer")
contest_compare["total_attempts"] = contest_compare[["guarded_attempts", "open_attempts"]].sum(axis=1, skipna=True)
contest_compare = contest_compare.sort_values("total_attempts", ascending=False)
print("\nGuarded vs. Open Spot-Up shooting side-by-side, by player (season-wide):\n")
print(contest_compare[[
    "player", "guarded_attempts", "guarded_makes", "guarded_fg_pct",
    "open_attempts", "open_makes", "open_fg_pct", "total_attempts",
]])

Spot-Up shooting by player and contest level, season-wide (every player/contest combination with at least one attempt):

             player                                                contest  \
0   Agape Keyes Jr.                                                Guarded   
1    Austin Ambrose                                                Guarded   
2      Brock Marino                                                Guarded   
3      Brock Marino  Not tagged (contest recorded only on catch-and-shoot)   
5     Collin Madson  Not tagged (contest recorded only on catch-and-shoot)   
4     Collin Madson                                                Guarded   
6     Collin Madson                                                   Open   
8   Darius Chestnut                                                   Open   
7   Darius Chestnut                                                Guarded   
10     Isaac Verges  Not tagged (contest recorded only on catch-and-shoot)   
11     Isaac Verges  

### Coaching Flag: contested Spot-Up 3PT rate

**Season-wide (video-tagged data):** review the `player_contest` and `contest_compare` output above for any
player whose Spot-Up attempts skew heavily toward guarded catch-and-shoot looks with little or no open volume.
A skew like this may not reflect poor shooting on the contested look itself (compare the guarded FG% to that
player's season 3P%) -- the flag is more about shot-diet allocation: has he logged any OPEN catch-and-shoot
looks at all this season, or has every attempt come standstill against a set defense.

**Recommendation for coaching staff:** where this pattern shows up, review off-ball actions (relocation,
screens, drive-and-kick reads) to spring that player for easier looks. Treat as a small-sample signal -- 
re-check as more games are logged. (See the structured `coaching_flags_df` a few cells below for the
rule-based, automatically-updating version of this check across the full roster.)


### Investigate why guarded catch-and-shoot jumpers outperform open ones

Digs into the counter-intuitive pattern flagged in the previous note -- guarded catch-and-shoot attempts showing a higher make rate than open ones -- to determine whether it's a real effect or a small-sample artifact.

In [133]:
# --- Why might GUARDED catch-and-shoot jumpers outperform OPEN ones, season-wide? ---------------------------
# Three possible explanations to test: (1) it's just small-sample noise, (2) it's driven by one game/opponent,
# or (3) it's a SHOOTER-QUALITY CONFOUND -- the defense keys on UWW's best shooters (more Guarded attempts from
# good shooters), while lesser shooters get left Open more often, so the two buckets aren't comparing the same
# shooters to begin with.
import math

# CONFIRMED BUG (fixed here): this z-test assumes BOTH a Guarded and an Open catch-and-shoot bucket
# already exist with at least one attempt each -- true "season-wide", but not necessarily true this
# early (e.g. the first game or two might have logged only Guarded looks so far, with zero Open ones,
# or vice versa). contest_all.loc[...].iloc[0] on a bucket with no rows raised "IndexError: single
# positional indexer is out-of-bounds" instead of just saying there isn't enough data yet to compare.
def _bucket_n(contest):
    rows = contest_all.loc[contest_all["contest"] == contest]
    return (int(rows["attempts"].iloc[0]), int(rows["makes"].iloc[0])) if not rows.empty else (0, 0)

guarded_n, guarded_makes_n = _bucket_n("Guarded")
open_n, open_makes_n = _bucket_n("Open")

if guarded_n == 0 or open_n == 0:
    print(f"Not enough catch-and-shoot data yet to compare Guarded (n={guarded_n}) vs. Open (n={open_n}) -- "
          f"need at least one attempt logged in each bucket. Skipping the significance test for now; the "
          f"game-by-game and shooter-quality breakdowns below still run on whatever data exists.")
else:
    p1, p2 = guarded_makes_n / guarded_n, open_makes_n / open_n
    p_pool = (guarded_makes_n + open_makes_n) / (guarded_n + open_n)
    se = (p_pool * (1 - p_pool) * (1 / guarded_n + 1 / open_n)) ** 0.5
    z = (p1 - p2) / se
    p_value = 2 * (1 - 0.5 * (1 + math.erf(abs(z) / math.sqrt(2))))
    print(f"Two-proportion z-test, Guarded ({p1:.1%}, n={guarded_n}) vs. Open ({p2:.1%}, n={open_n}): "
          f"z={z:.2f}, p-value={p_value:.3f}")
    print("(p > 0.05 means this gap is NOT statistically distinguishable from random chance at this sample size)\n")

by_game = (
    catch_and_shoot_all.groupby(GAME_KEYS + ["contest"])
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
by_game["fg_pct"] = (100 * by_game["makes"] / by_game["attempts"]).round(1)
print("Guarded vs. Open catch-and-shoot FG%, broken out by game:\n")
print(by_game.sort_values(["opponent", "contest"]))

# CONFIRMED BUG (fixed here): normalize_player_name below was actually relying on the SAME-NAMED
# function accidentally left in the global namespace by the `for` loop in the earlier video-tagging
# cell (a `for` loop doesn't create its own scope in Python, so a `def` inside one leaks into module/
# global scope once the loop body has run at least once). That caused two problems, one a crash and one
# silent: (1) if that loop's file list was empty -- e.g. no video-tagging files yet for the first game
# of the season, or none passing the reference_date filter -- normalize_player_name was never defined
# at all, raising "NameError: name 'normalize_player_name' is not defined" right here. (2) even when it
# WAS defined, it only knew the roster of whichever single OPPONENT'S game that loop happened to process
# LAST -- not UWW's own roster, which is what THIS cell actually needs to match UWW player names against
# the season-stats page. That's wrong regardless of whether the loop ran, and would have been silently
# mismatching UWW players against the wrong known-name set any time it didn't happen to crash. Built
# fresh and self-contained here instead, scoped correctly to UWW's own known players across every game.
_uww_known_players = set(pbp_events.loc[pbp_events["team"] == "UW-Whitewater", "player"].dropna().unique())

def normalize_player_name(name):
    if name in _uww_known_players:
        return name
    for p in _uww_known_players:
        if p.casefold() == str(name).casefold():
            return p
    return name

season_3pt = stats[["PLAYER", "3P%"]].copy()
season_3pt["season_3p_pct"] = pd.to_numeric(season_3pt["3P%"].str.rstrip("%"), errors="coerce")
season_3pt["player_norm"] = season_3pt["PLAYER"].apply(normalize_player_name)

cs_long3 = catch_and_shoot_all[catch_and_shoot_all["distance"] == "Long/3pt"].copy()
cs_long3["player_norm"] = cs_long3["player"].apply(normalize_player_name)
cs_long3 = cs_long3.merge(season_3pt[["player_norm", "season_3p_pct"]], on="player_norm", how="left")

quality_by_contest = (
    cs_long3.groupby("contest")
    .agg(attempts=("made", "count"), avg_shooter_season_3p_pct=("season_3p_pct", "mean"))
    .reset_index()
)
quality_by_contest["avg_shooter_season_3p_pct"] = quality_by_contest["avg_shooter_season_3p_pct"].round(1)
print("\nAverage SEASON-LONG 3P% of the shooter taking the shot, by contest level (catch-and-shoot 3s only) -- "
      "tests whether the defense keys on UWW's better shooters, inflating the Guarded bucket's shooter-quality "
      "mix relative to Open:\n")
print(quality_by_contest)

Two-proportion z-test, Guarded (50.0%, n=28) vs. Open (38.5%, n=26): z=0.85, p-value=0.394
(p > 0.05 means this gap is NOT statistically distinguishable from random chance at this sample size)

Guarded vs. Open catch-and-shoot FG%, broken out by game:

                opponent   game_date  contest  attempts  makes  fg_pct
0      Eureka Red Devils  2025-11-15  Guarded         8      4    50.0
1      Eureka Red Devils  2025-11-15     Open        16      9    56.2
2        Ripon Red Hawks  2025-11-07  Guarded         7      3    42.9
3        Ripon Red Hawks  2025-11-07     Open         8      0     0.0
4  St. Thomas (TX) Celts  2025-11-14  Guarded        13      7    53.8
5  St. Thomas (TX) Celts  2025-11-14     Open         2      1    50.0

Average SEASON-LONG 3P% of the shooter taking the shot, by contest level (catch-and-shoot 3s only) -- tests whether the defense keys on UWW's better shooters, inflating the Guarded bucket's shooter-quality mix relative to Open:

   contest  attempts

### Correction: "Guarded beats Open" is NOT a real team-wide pattern

An earlier pass through this notebook noted that UWW's catch-and-shoot jumpers were hitting at a higher rate
when **guarded** than when **open**, across every play type combined. The investigation above shows this does
**not** reliably hold up as a real tactical signal:

* **Check statistical significance** -- the two-proportion z-test above will show whether a gap this size is
  distinguishable from random chance at the current sample size (re-run as more games are logged; with only a
  few dozen total attempts, most gaps will not be significant).
* **Check whether it's driven by one specific game** -- the `by_game` breakdown above shows whether the pattern
  holds consistently across every logged game, or is really just one cold/hot shooting night skewing the total.
* **Check for a shooter-quality confound** -- the `quality_by_contest` breakdown compares the season-long 3P%
  of the shooters taking Guarded vs. Open looks; if Open shooters actually have a *higher* average season 3P%,
  that's evidence AGAINST guarded shots being intrinsically "better", not for it.

**Takeaway:** treat any aggregate "guarded > open" figure with suspicion until it clears all three checks above.
It should not be cited as evidence that contested shots are preferable to open ones -- individual player
patterns stand on their own volume/allocation evidence, not on this kind of small-sample team-wide claim.
Re-check once more games are logged and the sample grows.


### Structured coaching flags & recommendations database, per player

Consolidates every diagnosis surfaced above (Spot-Up quality, contest-level patterns, and the corrected guarded/open finding) into one structured, per-player table of coaching flags and recommendations.

**Season-stat flags (FG%/3P%/FT%/PPG/MPG/GP-GS) no longer leak games from after reference_date.** They previously read straight from `stats` -- FastScout's live cumulative season-stats page -- which has no date granularity at all and reflects the season total at scrape time. Caught via a real example: Brock Marino's free-throw flag showed "~112 attempts" even though `reference_date` is 2025-11-15 -- 112 FTA across 28 games is a near-full-season total, not something that could have happened by mid-November. These rules now read from `season_asof`, rebuilt from `pbp_box_score`'s real per-game lines and totalled only over games before `reference_date` (that restriction already exists upstream, in the "Play-by-play (PBP) data" cell). The tradeoff: a player whose early games weren't PBP-logged now shows a thinner (or empty) pre-reference-date sample instead of a precise-looking full-season number -- the correct tradeoff for a pre-game report. Team roster membership (names/jersey numbers) is unaffected -- that isn't the same kind of leak as a performance number, so it's still read from the season-stats page.

**Confidence is computed from real sample size, not a fixed label.** Every flag used to carry one of two hardcoded confidence strings -- "High (season-long sample)" on every season-stat-based flag, "Medium (small sample...)" on every per-shot one -- regardless of how many actual attempts backed the number. `confidence_label()` now computes an honest tier (Low/Medium/High) from the real attempt count behind each figure, and a flag whose sample is too thin to mean anything is suppressed rather than shown with false confidence.

**Role no longer falls back to a leaky data source.** An earlier version filled in role "Unknown" for a player with no PBP data by falling back to the season-stats page's GP-GS column -- but that page has the same reference_date leak described above, so that fallback was removed. A player with no PBP-tagged games before reference_date genuinely has an unknown role as of that date, and "Unknown" is the honest answer. Role sort order was also fixed to put Starters first, Bench second, Unknown last, instead of relying on alphabetical string order.

**The generic "seeing the floor" catch-all is a little less generic.** It previously read almost identically for a heavy-minutes rotation player and a deep-bench one -- now includes season PPG (as of reference_date) and splits the framing by usage level.


In [136]:
# --- Structured coaching-flags DATABASE: one row per (player, flag) -- replaces one-off markdown notes and
# inline comments with a rule-based, queryable table that recomputes automatically as more games get logged.
# Every rostered player gets at least one row (falls back to season box-score context when a player's
# logged shot volume is too thin for a play-type diagnosis).
# KNOWN_NAME_ALIASES reconciles known name-spelling mismatches BETWEEN data sources -- e.g. the play-by-play
# pipeline may spell a player differently than the official schedule-page season stats.
# Loaded from data/name_aliases.json (shared with streamlit_app.py) instead of a hardcoded dict here, so a
# newly-discovered mismatch only needs to be added in ONE place rather than kept in sync across the app and
# this notebook. Falls back to the one known mismatch if the file isn't found (e.g. first run before the app
# repo's data/ folder exists locally).
def _load_known_name_aliases():
    alias_path = os.path.join(OUTPUT_DIR, "name_aliases.json")
    if os.path.exists(alias_path):
        try:
            with open(alias_path) as _af:
                raw = json.load(_af)
            return {k.lower(): v.lower() for k, v in raw.items() if not k.startswith("_")}
        except Exception as _e:
            print(f"WARNING: could not read {alias_path} ({_e}); using inline fallback alias.")
    return {"mauryon turner": "maurquis turner"}

KNOWN_NAME_ALIASES = _load_known_name_aliases()

# CONFIRMED BUG (fixed here): this cell used to call a bare normalize_player_name(name) -- which isn't
# actually defined anywhere at the top level of this notebook. It only ever exists as a function defined
# INSIDE a `for` loop in the video-tagging cell above, and Python `for` loops don't create their own
# scope, so that `def` leaks into the global namespace once the loop body has run at least once. If that
# loop's file list is empty -- e.g. no video-tagging files yet, or reference_date is before every game
# so every file gets filtered out -- normalize_player_name was never defined at all, and this cell would
# raise "NameError: name 'normalize_player_name' is not defined" the same way the player-comparison cell
# did before its own fix. Built fresh and self-contained here instead, scoped to UWW's own known players
# across every game in pbp_events (the same fix already applied where this exact problem first surfaced).
_coaching_flags_known_players = set(pbp_events.loc[pbp_events["team"] == "UW-Whitewater", "player"].dropna().unique())

def normalize_player_name(name):
    if name in _coaching_flags_known_players:
        return name
    for p in _coaching_flags_known_players:
        if p.casefold() == str(name).casefold():
            return p
    return name

def resolve_player_key(name):
    # normalize_player_name() only folds CASING/spelling to UWW's own known-player set (and preserves the
    # original string untouched when no case-insensitive match exists there) -- it does NOT lowercase, so a
    # dict lookup against it needs an explicit .lower() to be reliably case-insensitive across sources whose
    # canonical spelling itself differs.
    norm = normalize_player_name(name).lower()
    return KNOWN_NAME_ALIASES.get(norm, norm)

coaching_flags = []

def add_flag(player, player_key, category, flag, evidence, recommendation, confidence, sentiment):
    coaching_flags.append({
        "player": player, "player_key": player_key, "category": category, "flag": flag,
        "evidence": evidence, "recommendation": recommendation, "confidence": confidence,
        "sentiment": sentiment,
    })

# --- CONFIRMED BUG (fixed here): every flag below used to carry a hardcoded confidence STRING regardless of
# how many actual attempts backed the number -- "High (season-long sample)" on every season-stat rule,
# "Medium (small sample -- re-check as more games are logged)" on every per-shot rule. Checked against
# real season output (cell 10's printed `stats` table): Maurquis Turner's "Below-average free-throw shooter"
# flag was tagged "High (season-long sample)" off 0.0-0.7 FTA/gm over a 3-0 (GP-GS) season -- ~2 total FT
# attempts, not a real signal. Agape Keyes Jr.'s "Respectable 3-point shooter" flag was also labelled "High"
# off ~7 total 3PT attempts (0.5-1.2/gm over 6 games) -- identically worded to Kelton McEwen's same flag,
# which is backed by ~53 attempts. The two aren't remotely equivalent, but the old code couldn't tell them
# apart because it never computed a real attempt count, only checked a per-game RATE against a fixed
# threshold. confidence_label() below computes a tier from the actual attempt count behind each number, and
# MIN_ATTEMPTS_TO_FLAG suppresses a rate-stat flag entirely when that count is too thin to mean anything --
# Turner's ~2 FT attempts (0-for-2) no longer produce a flag at all, rather than a misleadingly confident one.
MIN_ATTEMPTS_TO_FLAG = 5  # absolute floor: below this, a season FG%/3P%/FT% flag doesn't fire at all

# CONFIRMED BUG (fixed here): the per-shot rules passed low_max=5 / medium_max=11, so 12 attempts already read
# "High". Brock Marino's "Highly efficient scorer" flag was printed as HIGH confidence off 18 shots -- a
# 14-for-18 stretch that one cold night erases. "High" now needs a genuinely season-sized denominator no
# matter which rule is asking: _CONF_HIGH_FLOOR attempts, applied inside confidence_label itself so an
# individual rule can't hand out "High" cheaply again.
_CONF_HIGH_FLOOR = 30

def confidence_label(n, noun="attempts", low_max=9, medium_max=29):
    """n = the actual count of attempts behind a percentage. A rate is only as trustworthy as its
    denominator, so this replaces a single fixed string with a tier computed from real sample size."""
    if n is None or pd.isna(n):
        return "Unknown (sample size not available)"
    n = int(round(n))
    if n <= low_max:
        return f"Low (n={n} {noun} -- small sample, re-check as more games are played)"
    if n <= medium_max or n < _CONF_HIGH_FLOOR:
        return f"Medium (n={n} {noun})"
    return f"High (n={n} {noun}, season-long sample)"

def format_rate_pair(makes, attempts, games_played):
    """Recreate a FastScout-style 'M-A per game' display string (e.g. '0.4-1.2') from raw totals, since
    season_asof (below) carries totals, not the pre-formatted per-game strings the old `stats` page had."""
    if not games_played:
        return "-"
    return f"{makes / games_played:.1f}-{attempts / games_played:.1f}"

# Fresh, self-contained recomputation of UWW's logged shot attempts with play type / mechanic / contest /
# distance tags (mirrors the logic in the cells above; kept self-contained so this cell doesn't depend on the
# execution order of earlier ones). No extra date filter needed here: pbp_events itself was already cut down
# to game_date < reference_date by the fix in the "Play-by-play (PBP) data" cell above, and uww_shots is
# built directly from pbp_events, so that restriction already carries through.
uww_shots = pbp_events[
    (pbp_events["team"] == "UW-Whitewater")
    & pbp_events["event_type"].isin(["made_shot", "missed_shot"])
    & pbp_events["video_description"].notna()
].copy()
uww_shots["play_type"] = uww_shots.apply(lambda r: extract_play_type(r["video_description"], r["player"]), axis=1)
uww_shots["made"] = uww_shots["event_type"] == "made_shot"
uww_shots["shot_mechanic"] = uww_shots["video_description"].apply(extract_shot_mechanic)
uww_shots["contest"] = uww_shots["video_description"].apply(extract_contest)
uww_shots["distance"] = uww_shots["video_description"].apply(extract_distance)
uww_shots["player_key"] = uww_shots["player"].apply(resolve_player_key)

# Full ROSTER identity only (names, jersey numbers, player_key) -- who's on the team doesn't change
# game-to-game the way cumulative stats do, so this is safe to pull from the official season-stats page
# regardless of reference_date. Deliberately does NOT carry FG%/3P%/FT%/MIN/PTS/GP-GS from that page --
# see season_asof below for why.
roster = stats[~stats["PLAYER"].str.contains("Team|Opponent", case=False, na=False)].copy()
roster["player_key"] = roster["PLAYER"].apply(resolve_player_key)
# The season-stats source table itself has a duplicate-row quirk for at least one jersey number (a placeholder
# row with all "-" stats under one name spelling, alongside a real-stats row under the corrected spelling).
# When the alias/key resolution above merges such rows onto the same player_key, prefer whichever row actually
# has a real GP-GS entry over an all-placeholder one.
roster["_has_real_row"] = roster["GP-GS"].astype(str) != "-"
roster = roster.sort_values("_has_real_row", ascending=False).drop_duplicates("player_key", keep="first").drop(columns="_has_real_row")

# --- CONFIRMED BUG (fixed here): every season-stat rule below (FG%/3P%/FT%/PPG/MPG/GP-GS) previously read
# straight from `stats` -- FastScout's live CUMULATIVE season-stats page (scraped in cell 10). That page has
# no date granularity at all: it's whatever the season total is AT SCRAPE TIME, with no way to ask "as of
# reference_date". Caught via a real example: Brock Marino's free-throw flag showed "~112 attempts" even
# though reference_date is 2025-11-15 -- 112 FTA across 28 games is a near-full-season total, not something
# that could have happened by mid-November. `pbp_events` (and everything built from it, including
# `pbp_box_score`) is already correctly restricted to `game_date < reference_date` by the fix in the
# "Play-by-play (PBP) data" cell above -- so every season-stat rule below now reads from `season_asof`,
# rebuilt here from `pbp_box_score`'s real per-game shooting lines, totalled only over games that had
# actually been played as of reference_date.
# TRADEOFF: pbp_box_score only covers games with a parsed play-by-play file, so a player whose
# pre-reference-date games weren't logged that way will show a thinner (or empty) sample here than the
# season-to-date page would have shown. That's the correct tradeoff for a pre-game report -- an honest
# thin/no sample (which MIN_ATTEMPTS_TO_FLAG and confidence_label() already handle gracefully) beats a
# precise-looking number partly built from games that, as of reference_date, hadn't been played yet.
uww_box_asof = pbp_box_score[
    (pbp_box_score["team"] == "UW-Whitewater") & (pbp_box_score["player"] != "TEAM")
].copy()
uww_box_asof["player_key"] = uww_box_asof["player"].apply(resolve_player_key)
uww_box_asof["MIN"] = pd.to_numeric(uww_box_asof["MIN"], errors="coerce")
season_asof = uww_box_asof.groupby("player_key").agg(
    FGM=("FGM", "sum"), FGA=("FGA", "sum"), FG3M=("FG3M", "sum"), FG3A=("FG3A", "sum"),
    FTM=("FTM", "sum"), FTA=("FTA", "sum"), PTS=("PTS", "sum"), MIN=("MIN", "sum"),
    games_played=("game_date", "nunique"), games_started=("started", "sum"),
).reset_index()
season_asof["fg_pct_season"] = (100 * season_asof["FGM"] / season_asof["FGA"]).round(1)
season_asof["three_pt_pct_season"] = (100 * season_asof["FG3M"] / season_asof["FG3A"]).round(1)
season_asof["ft_pct_season"] = (100 * season_asof["FTM"] / season_asof["FTA"]).round(1)
season_asof["mpg_season"] = (season_asof["MIN"] / season_asof["games_played"]).round(1)
season_asof["pts_season"] = (season_asof["PTS"] / season_asof["games_played"]).round(1)
# These are now EXACT totals from real per-game box scores, not the M-A-per-game-times-GP estimate the old
# `stats`-page version needed (that page never exposed a raw attempt count, only a rounded per-game rate).
season_asof["fga_total_est"] = season_asof["FGA"]
season_asof["tpa_total_est"] = season_asof["FG3A"]
season_asof["fta_total_est"] = season_asof["FTA"]

# Season-wide Starter/Bench role, from the per-game `started` flag reconstructed from the PBP.
# CONFIRMED BUG (fixed here, superseding an earlier "fix"): a previous version of this cell filled in role
# "Unknown" for a player with no PBP data by falling back to the official season-stats page's GP-GS column
# (games started > 0 => Starter). That page has the exact same reference_date leak described above -- a
# player's season-long "started 24 of 29 games" is just as much of a leak as their season-long FT% is, so
# that fallback has been removed. A player with no PBP-tagged games before reference_date genuinely has an
# unknown role AS OF reference_date, and "Unknown" is the honest answer, not a bug to paper over.
starter_flags = pbp_box_score[pbp_box_score["team"] == "UW-Whitewater"][["player", "started"]].drop_duplicates()
starter_flags["player_key"] = starter_flags["player"].apply(resolve_player_key)
player_role = starter_flags.groupby("player_key")["started"].any().map({True: "Starter", False: "Bench"})

# Canonical DISPLAY name per resolved key -- prefer the official roster spelling when available (the athletic
# department's own roster page), otherwise whatever spelling shows up in the logged PBP data. Names only --
# no performance numbers -- so this is unaffected by the reference_date fix above.
canonical_name_by_key = {}
for _, row in roster.iterrows():
    canonical_name_by_key.setdefault(row["player_key"], row["PLAYER"])
for name in uww_shots["player"].unique():
    canonical_name_by_key.setdefault(resolve_player_key(name), name)

roster_keys = sorted(set(roster["player_key"]) | set(uww_shots["player_key"]))

for player_key in roster_keys:
    player = canonical_name_by_key[player_key]
    p_shots = uww_shots[uww_shots["player_key"] == player_key]
    asof_row = season_asof[season_asof["player_key"] == player_key]
    total_attempts = len(p_shots)
    has_positive_flag = False

    cs = p_shots[p_shots["shot_mechanic"] == "Catch-and-shoot"]
    guarded_cs = cs[cs["contest"] == "Guarded"]
    open_cs = cs[cs["contest"] == "Open"]

    # Rule 1: shot diet is skewed heavily toward contested catch-and-shoot looks. Requires guarded to be at
    # least 80% of catch-and-shoot volume, with at most 1 stray open attempt, so a single outlier doesn't mask
    # an otherwise heavily-contested shot diet, while still requiring a real majority skew.
    # CONFIRMED BUG (fixed here): this fired on Brock Marino at 3-for-3 on guarded catch-and-shoots and filed
    # it under "Clean up". A player MAKING every contested look is not a problem to fix this week. The rule now
    # also requires the guarded looks to be underperforming (_CS_GUARDED_MAX_PCT or worse) -- it is a flag
    # about lost points, not about shot diet in the abstract.
    _CS_GUARDED_MAX_PCT = 40
    cs_total = len(guarded_cs) + len(open_cs)
    g_pct_rule1 = (100 * guarded_cs["made"].sum() / len(guarded_cs)) if len(guarded_cs) else None
    if (len(guarded_cs) >= 3 and len(open_cs) <= 1 and cs_total and (len(open_cs) / cs_total) <= 0.2
            and g_pct_rule1 is not None and g_pct_rule1 <= _CS_GUARDED_MAX_PCT):
        g_makes, g_att, o_att = int(guarded_cs["made"].sum()), len(guarded_cs), len(open_cs)
        add_flag(
            player, player_key, "Shot selection",
            "Shot diet skewed heavily toward contested catch-and-shoot looks",
            f"{g_makes}-for-{g_att} ({100 * g_makes / g_att:.0f}%) on guarded catch-and-shoot attempts this "
            f"season, vs. only {o_att} open catch-and-shoot attempt(s).",
            "Design more actions to create separation before the catch (relocation, screens, drive-and-kick "
            "reads) rather than relying on standstill catches against a set defense.",
            confidence_label(cs_total, noun="catch-and-shoot attempts", low_max=5, medium_max=11), "Negative",
        )

    # Rule 2: misses concentrate on OPEN catch-and-shoot looks specifically -- a shooter-specific issue, not a
    # shot-quality one.
    if len(open_cs) >= 3:
        o_makes, o_att = int(open_cs["made"].sum()), len(open_cs)
        o_pct = 100 * o_makes / o_att
        if o_pct <= 35:
            season_3p = None
            if not asof_row.empty and pd.notna(asof_row["three_pt_pct_season"].iloc[0]):
                season_3p = asof_row["three_pt_pct_season"].iloc[0]
            evidence = f"{o_makes}-for-{o_att} ({o_pct:.0f}%) on open catch-and-shoot attempts this season"
            evidence += f" -- below his season 3P% of {season_3p:.1f}%." if season_3p is not None else "."
            add_flag(
                player, player_key, "Shooting efficiency",
                "Missing predominantly OPEN catch-and-shoot looks",
                evidence,
                "Since these are UNCONTESTED misses, treat as a shooting-mechanics/rhythm issue -- prioritize "
                "catch-and-shoot reps in practice rather than trying to generate better shot quality in-game.",
                confidence_label(o_att, noun="open catch-and-shoot attempts", low_max=5, medium_max=11), "Negative",
            )

    # Rule 3 / 4: efficiency on the player's single most-attempted play type (their "go-to" action).
    if total_attempts:
        top_pt = p_shots["play_type"].value_counts().idxmax()
        top_rows = p_shots[p_shots["play_type"] == top_pt]
        top_att, top_makes = len(top_rows), int(top_rows["made"].sum())
        top_pct = 100 * top_makes / top_att if top_att else None
        if top_att >= 4 and top_pct is not None:
            if top_pct <= 30:
                add_flag(
                    player, player_key, "Play-type efficiency",
                    f"Struggles specifically in his most-used action ({top_pt})",
                    f"{top_makes}-for-{top_att} ({top_pct:.0f}%) on {top_pt} -- his single most-attempted "
                    f"action this season, well below his overall shooting split.",
                    f"Reduce reliance on {top_pt} as his primary look, or work on the specific mechanics/reads "
                    f"for that action in practice.",
                    confidence_label(top_att, noun=f"{top_pt} attempts", low_max=5, medium_max=11), "Negative",
                )
            elif top_pct >= 70:
                add_flag(
                    player, player_key, "Play-type efficiency",
                    f"Highly efficient in his most-used action ({top_pt}) -- underused upside",
                    f"{top_makes}-for-{top_att} ({top_pct:.0f}%) on {top_pt}, his most-attempted "
                    f"action this season.",
                    f"Consider increasing his usage/touches in {top_pt} sets -- the efficiency supports more "
                    f"volume there.",
                    confidence_label(top_att, noun=f"{top_pt} attempts", low_max=5, medium_max=11), "Positive",
                )
                has_positive_flag = True

    # Rule 5: overall shot efficiency, for a broader positive signal independent of a single play type.
    if total_attempts >= 8:
        overall_pct = 100 * p_shots["made"].sum() / total_attempts
        if overall_pct >= 70:
            add_flag(
                player, player_key, "Overall efficiency",
                "Highly efficient scorer, season-wide",
                f"{int(p_shots['made'].sum())}-for-{total_attempts} ({overall_pct:.0f}%) across all "
                f"shot attempts this season.",
                "A clear, efficient scoring option -- consider featuring him more prominently in the "
                "half-court offense.",
                confidence_label(total_attempts, noun="shot attempts", low_max=5, medium_max=11), "Positive",
            )
            has_positive_flag = True

    # Rule 6: free-throw shooting as of reference_date (independent of shot logging -- covers every rostered
    # player). Fires only once season_asof gives us a real attempt count to trust (MIN_ATTEMPTS_TO_FLAG floor).
    if not asof_row.empty:
        r6 = asof_row.iloc[0]
        ft_pct, fta_total = r6["ft_pct_season"], r6["fta_total_est"]
        if pd.notna(ft_pct) and pd.notna(fta_total) and fta_total >= MIN_ATTEMPTS_TO_FLAG and ft_pct <= 60:
            # CONFIRMED BUG (fixed here): evidence read "(1.3-4.7 per game, 14 attempts)" -- a FastScout-style
            # makes-attempts pair that looks like a range. Spelled out instead.
            _gp6 = int(r6["games_played"]) if pd.notna(r6["games_played"]) else 0
            add_flag(
                player, player_key, "Free-throw shooting",
                "Below-average free-throw shooter with meaningful attempt volume",
                f"{ft_pct:.1f}% FT this season -- {int(r6['FTM'])}-for-{int(fta_total)} over {_gp6} "
                f"game{'s' if _gp6 != 1 else ''} ({fta_total / max(_gp6, 1):.1f} attempts a game).",
                "Target free-throw shooting in individual workouts -- meaningful attempt volume means this is "
                "costing points.",
                confidence_label(fta_total, noun="FT attempts"), "Negative",
            )

    # --- POSITIVE-signal cascade: guarantees at least one Positive flag per player wherever the data supports
    # one, independent of whether the rules above already found a negative issue for him. Evaluated in
    # priority order; the first candidate with real supporting data is used.
    if not has_positive_flag:
        pt_stats = p_shots.groupby("play_type").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
        pt_stats["pct"] = 100 * pt_stats["makes"] / pt_stats["attempts"]
        best_pt = pt_stats[pt_stats["attempts"] >= 3].sort_values("pct", ascending=False).head(1)
        if not best_pt.empty and best_pt["pct"].iloc[0] >= 65:
            r = best_pt.iloc[0]
            add_flag(
                player, player_key, "Play-type efficiency",
                f"Efficient secondary action: {r['play_type']}",
                f"{int(r['makes'])}-for-{int(r['attempts'])} ({r['pct']:.0f}%) on {r['play_type']} this season.",
                "A reliable look worth calling more often, even if not his primary action.",
                confidence_label(r["attempts"], noun=f"{r['play_type']} attempts", low_max=5, medium_max=11), "Positive",
            )
            has_positive_flag = True

    if not has_positive_flag and total_attempts >= 4:
        overall_pct = 100 * p_shots["made"].sum() / total_attempts
        if overall_pct >= 55:
            add_flag(
                player, player_key, "Overall efficiency",
                "Solid overall shooting, season-wide",
                f"{int(p_shots['made'].sum())}-for-{total_attempts} ({overall_pct:.0f}%) across all "
                f"shot attempts this season.",
                "A dependable scoring option in the offense.",
                confidence_label(total_attempts, noun="shot attempts", low_max=5, medium_max=11), "Positive",
            )
            has_positive_flag = True

    if not has_positive_flag and not asof_row.empty:
        r = asof_row.iloc[0]
        if (
            pd.notna(r["fg_pct_season"]) and r["fg_pct_season"] >= 45
            and pd.notna(r["mpg_season"]) and r["mpg_season"] >= 3
            and pd.notna(r["fga_total_est"]) and r["fga_total_est"] >= MIN_ATTEMPTS_TO_FLAG
        ):
            fgm_a = format_rate_pair(r["FGM"], r["FGA"], r["games_played"])
            add_flag(
                player, player_key, "Season shooting",
                "Solid season field-goal percentage",
                f"{r['fg_pct_season']:.1f}% FG this season ({fgm_a} per game, {r['mpg_season']:.0f} MPG, "
                f"{int(r['fga_total_est'])} attempts).",
                "A reasonably efficient finisher for his role -- keep him involved in the offense.",
                confidence_label(r["fga_total_est"], noun="FGA"), "Positive",
            )
            has_positive_flag = True
        elif (
            pd.notna(r["ft_pct_season"]) and r["ft_pct_season"] >= 70
            and pd.notna(r["fta_total_est"]) and r["fta_total_est"] >= MIN_ATTEMPTS_TO_FLAG
        ):
            ftm_a = format_rate_pair(r["FTM"], r["FTA"], r["games_played"])
            add_flag(
                player, player_key, "Free-throw shooting",
                "Reliable free-throw shooter",
                f"{r['ft_pct_season']:.1f}% FT this season ({ftm_a} per game, {int(r['fta_total_est'])} attempts).",
                "A safe option to have on the floor in late-game free-throw situations.",
                confidence_label(r["fta_total_est"], noun="FT attempts"), "Positive",
            )
            has_positive_flag = True
        elif (
            pd.notna(r["three_pt_pct_season"]) and r["three_pt_pct_season"] >= 33
            and pd.notna(r["tpa_total_est"]) and r["tpa_total_est"] >= MIN_ATTEMPTS_TO_FLAG
        ):
            tpm_a = format_rate_pair(r["FG3M"], r["FG3A"], r["games_played"])
            add_flag(
                player, player_key, "Season shooting",
                "Respectable 3-point shooter this season",
                f"{r['three_pt_pct_season']:.1f}% 3PT this season ({tpm_a} per game, {int(r['tpa_total_est'])} "
                f"attempts).",
                "Worth designing catch-and-shoot looks for him specifically.",
                confidence_label(r["tpa_total_est"], noun="3PT attempts"), "Positive",
            )
            has_positive_flag = True
        elif pd.notna(r["mpg_season"]) and r["mpg_season"] > 0:
            # This catch-all used to say the same sentence, differing only by MPG/GP-GS, for every player with
            # real minutes but no threshold-crossing shooting number. Adding season PPG and splitting the
            # framing by usage level gives a coach something to actually distinguish those cases on.
            pts_str = f", {r['pts_season']:.1f} PPG" if pd.notna(r["pts_season"]) else ""
            gp_gs = f"{int(r['games_played'])}-{int(r['games_started'])}"
            if r["mpg_season"] >= 10:
                flag_text = "Regular rotation minutes without a standout shooting number yet"
                usage_note = "a real rotation role"
            else:
                flag_text = "Seeing early/situational game action"
                usage_note = "limited minutes"
            add_flag(
                player, player_key, "General",
                flag_text,
                f"Averaging {r['mpg_season']:.0f} MPG{pts_str} over {gp_gs} (GP-GS) as of reference_date -- no "
                f"shooting percentage has crossed a flag threshold yet, but he's logging {usage_note}.",
                "Keep tracking as more games are played for a clearer efficiency signal.",
                "Low (no standout stat yet)", "Positive",
            )
            has_positive_flag = True

    if not has_positive_flag:
        add_flag(
            player, player_key, "General / limited data",
            "No performance data yet to flag positively",
            "No recorded minutes, shot attempts, or season stats found for him as of reference_date.",
            "Re-evaluate once he sees game action and stats are recorded.",
            "Low (no data)", "Neutral",
        )

coaching_flags_df = pd.DataFrame(coaching_flags)
coaching_flags_df["role"] = coaching_flags_df["player_key"].map(player_role).fillna("Unknown")
coaching_flags_df = coaching_flags_df[
    ["player", "player_key", "role", "sentiment", "category", "flag", "evidence", "recommendation", "confidence"]
]
_sentiment_order = {"Positive": 0, "Negative": 1, "Neutral": 2}
coaching_flags_df["_sentiment_order"] = coaching_flags_df["sentiment"].map(_sentiment_order)
# CONFIRMED BUG (fixed here): role was previously sorted with ascending=False on the raw string, which only
# put Starters ahead of Bench by alphabetical coincidence -- "Unknown" > "Starter" > "Bench" in reverse
# alphabetical order, so any Unknown-role player actually sorted to the TOP of the table, ahead of every
# starter. Explicit priority order below puts Starters first, Bench second, and any genuinely unresolved
# players last, which is what a coach opening this table actually wants.
_role_order = {"Starter": 0, "Bench": 1, "Unknown": 2}
coaching_flags_df["_role_order"] = coaching_flags_df["role"].map(_role_order).fillna(3)
coaching_flags_df = coaching_flags_df.sort_values(
    ["_role_order", "player", "_sentiment_order", "category"], ascending=[True, True, True, True]
).drop(columns=["_sentiment_order", "_role_order"])

n_players_with_positive = coaching_flags_df.loc[coaching_flags_df["sentiment"] == "Positive", "player"].nunique()
n_no_asof_data = sum(1 for k in roster_keys if season_asof[season_asof["player_key"] == k].empty)
print(f"Coaching flags database: {len(coaching_flags_df)} flag(s) across {coaching_flags_df['player'].nunique()} "
      f"player(s) (full roster: {len(roster_keys)}).")
print(f"{n_players_with_positive} of {len(roster_keys)} players have at least one Positive flag.")
print(f"All season-stat rules (FG%/3P%/FT%/PPG/MPG/GP-GS) are now built from pbp_box_score restricted to "
      f"game_date < reference_date ({reference_date_str}), not the ungated season-to-date page -- "
      f"{n_no_asof_data} of {len(roster_keys)} rostered player(s) have no play-by-play data at all before "
      f"reference_date and fall back to the 'no data yet' flag rather than a leaked full-season number.\n")
print("Name reconciliation applied via KNOWN_NAME_ALIASES where the play-by-play spelling and the "
      "official season-stats spelling of a player's name differ.\n")
print(coaching_flags_df)


Coaching flags database: 30 flag(s) across 22 player(s) (full roster: 22).
19 of 22 players have at least one Positive flag.
All season-stat rules (FG%/3P%/FT%/PPG/MPG/GP-GS) are now built from pbp_box_score restricted to game_date < reference_date (2025-11-19), not the ungated season-to-date page -- 3 of 22 rostered player(s) have no play-by-play data at all before reference_date and fall back to the 'no data yet' flag rather than a leaked full-season number.

Name reconciliation applied via KNOWN_NAME_ALIASES where the play-by-play spelling and the official season-stats spelling of a player's name differ.

                    player              player_key     role sentiment  \
3             Brock Marino            brock marino  Starter  Positive   
4             Brock Marino            brock marino  Starter  Negative   
6            Collin Madson           collin madson  Starter  Positive   
5            Collin Madson           collin madson  Starter  Negative   
10            Isaac


### Expected box score for the upcoming game

Projects an expected box score for the next scouted opponent, combining UWW's season performance, the opponent's own lineup/tendency profile, and the coaching flags database above.

In [138]:
# --- Expected box score for the upcoming game (Elmhurst) ------------------------------------------------------
# Combines every signal already computed elsewhere in this notebook rather than inventing a new data source:
#   1. TEAM-LEVEL SCORE: a log5-style blend of each team's own scoring average with the OTHER side's actual
#      defensive output this season -- using ACTUAL results from completed games (not just season averages)
#      as the best available evidence of how UWW performs against this tier of competition.
#   2. OPPONENT PLAYER LINES: each player's own season per-game average, blended with the ACTUAL box-score
#      line their tag+stat-similarity comparable player (from the `best_matches` player-comparison cell) put
#      up in their real game against this same UWW defense -- a matchup-specific signal a season average
#      alone can't capture. Individual point projections are then scaled so they sum to the team-level
#      projection above.
#   3. UWW PLAYER LINES: each player's own season per-game average, scaled by the same team-level pace/quality
#      adjustment, preserving each player's share of the offense.
# NOTE: this cell is written for the specific upcoming opponent ("Elmhurst") named below -- update the
# opponent name if/when the upcoming matchup changes again. Elmhurst's PDF boxscore has no "Opponent" row (no
# points-allowed figure for them), so there's no way to log5 that side directly -- the model leans on their
# season scoring average instead.
UPCOMING_MATCHUP_OPPONENT = upcoming_opponent_short

def pct_to_float(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    return None if s in ("", "-") else float(s.rstrip("%"))

# --- 1. Team-level projected score --------------------------------------------------------------------------
uww_team_pts_season = float(stats.loc[stats["PLAYER"] == "Team Total", "PTS"].iloc[0])

played = schedule[schedule["outcome"].notna()]
uww_actual_pts_avg = played["team_score"].mean()
opp_actual_pts_avg = played["opponent_score"].mean()  # what UWW's actual scouted opponents have scored on them

# CONFIRMED BUG (fixed here): team_totals["team_ppg"] is now ALWAYS None -- team/player statistics from
# the scouting report are no longer used at all (see the blanking in the "Cross-reference each scouted
# opponent" cell). float(None) crashed here outright. Digging further, this wasn't just a "handle the
# missing value" fix: tier_avg_ppg needs a real team_ppg for MANY scouted opponents at once to compute a
# "typical opponent" baseline, and that's no longer computable at all now that team_ppg only ever gets a
# real number from a PBP-derived override, applied to exactly ONE opponent (whoever is upcoming) -- there
# is no longer a "tier" of opponents to average. Also, that PBP override happens in a LATER cell than
# this one runs, so even the upcoming opponent's own team_ppg is never populated by the time this line
# runs, regardless of the tier problem. Fixed by computing the upcoming opponent's own PPG directly from
# pbp_box_score_upcoming here (the same real, current-season data the later override cell uses), and
# dropping the no-longer-computable tier normalization in favor of the same plain 50/50 blend already
# used for UWW's own side two lines below -- both sides of the projection now rest on exactly the same
# kind of evidence (a season rate blended with real actual-game evidence), instead of one side secretly
# depending on scouting-report data the other side never used to begin with.
# CONFIRMED BUG (fixed here, on top of the fix above): the first attempt at this used
# pbp_box_score_upcoming for the same purpose -- but that variable has the EXACT same forward-reference
# problem team_totals had: it's built in a LATER cell ("Single-game box score aggregated by 5-MAN
# LINEUP..." / the box-score reconstruction cell), so it raised "NameError: name
# 'pbp_box_score_upcoming' is not defined" the very next run, for the identical reason as before --
# just one variable deeper. pbp_events_upcoming (built two cells after prev_games, well before this one)
# has everything needed instead: uww_score is the running score for the upcoming opponent's own side in
# each of their prior games (self_team=upcoming_opponent_short when these events were built), so its
# LAST value per game (sorted by event_order) is that game's final score for them -- averaging that
# across their games gives the same real, PBP-derived PPG without depending on anything built later.
opponent_team_ppg = None
if not pbp_events_upcoming.empty and UPCOMING_MATCHUP_OPPONENT is not None:
    _ebs_final_scores = (
        pbp_events_upcoming.sort_values("event_order")
        .groupby(["opponent", "game_date"])["uww_score"]
        .last()
    )
    if not _ebs_final_scores.empty:
        opponent_team_ppg = round(_ebs_final_scores.mean(), 2)

if opponent_team_ppg is None:
    print(f"Not enough real game data yet for {UPCOMING_MATCHUP_OPPONENT} to project an expected box "
          f"score -- scouting-report statistics are no longer used, and there's no prior-game "
          f"play-by-play data available yet for this opponent. Skipping the projected box score for "
          f"now; it will populate automatically once real game data exists.")
    projected_uww_box = pd.DataFrame(columns=["PLAYER", "MIN", "projected_PTS", "projected_REB", "projected_AST", "FG%", "3P%", "FT%", "projection_basis"])
    projected_opponent_box = pd.DataFrame(columns=["name", "jersey_number", "role", "position", "MIN", "projected_PTS", "projected_REB", "projected_AST", "comp_used", "comp_from_game", "similarity_score", "projection_basis"])
else:
    # UWW's own scoring average, blended 50/50 with their ACTUAL scoring average against comparable competition
    # this season (mean of the completed games) -- real matchup evidence, not just a season-long number.
    expected_uww_pts = round(0.5 * uww_team_pts_season + 0.5 * uww_actual_pts_avg, 1)
    # Same 50/50 blend, mirrored for the opponent's side: their own real PBP-derived scoring average with
    # UWW's real actual PTS/gm allowed to scouted opponents this season.
    expected_opponent_pts = round(0.5 * opponent_team_ppg + 0.5 * opp_actual_pts_avg, 1)

    print(f"Projected final score: UW-Whitewater {expected_uww_pts:.0f} - {UPCOMING_MATCHUP_OPPONENT} {expected_opponent_pts:.0f} "
          f"(margin {expected_uww_pts - expected_opponent_pts:+.0f})")
    print(f"  UWW inputs: season PTS/gm={uww_team_pts_season:.1f}, actual PTS/gm vs scouted opponents={uww_actual_pts_avg:.1f}")
    print(f"  {UPCOMING_MATCHUP_OPPONENT} inputs: PBP-derived PTS/gm={opponent_team_ppg:.1f}, UWW's actual PTS/gm allowed to scouted "
          f"opponents={opp_actual_pts_avg:.1f}")

    # --- 2. Opponent projected player box score -------------------------------------------------------------------
    opponent_players = player_profiles[player_profiles["opponent"] == UPCOMING_MATCHUP_OPPONENT].copy()
    for col in ["PTS", "REB", "AST", "MIN"]:
        opponent_players[col] = pd.to_numeric(opponent_players[col], errors="coerce")

    # CONFIRMED BUG (fixed here): same root cause as the player-comparison display cell above -- for the
    # first scouted game of the season, best_matches comes back empty and columnless (no prior opponent to
    # compare against yet), so filtering it on "target_opponent" raised a KeyError instead of just yielding
    # no matches. An empty frame WITH the right column names lets everything below (the merges, and
    # blend_stat()'s existing NaN-comp fallback to each player's own season average) work exactly as it
    # already does for any individual player with no match found -- no separate empty-season code path needed.
    _bm_cols = ["target_player", "target_opponent", "compared_player", "compared_opponent", "similarity_score"]
    if best_matches.empty or not set(_bm_cols) <= set(best_matches.columns):
        opponent_matches = pd.DataFrame(columns=_bm_cols)
    else:
        opponent_matches = best_matches[best_matches["target_opponent"] == UPCOMING_MATCHUP_OPPONENT]
    opp_actuals = pbp_box_score[pbp_box_score["team"] != "UW-Whitewater"][
        ["opponent", "player", "PTS", "REB", "AST"]
    ].rename(columns={"opponent": "compared_opponent", "player": "compared_player",
                       "PTS": "comp_actual_PTS", "REB": "comp_actual_REB", "AST": "comp_actual_AST"})
    comp_actuals = opponent_matches.merge(opp_actuals, on=["compared_opponent", "compared_player"], how="left")

    opponent_players = opponent_players.merge(
        comp_actuals[["target_player", "compared_player", "compared_opponent", "similarity_score",
                      "comp_actual_PTS", "comp_actual_REB", "comp_actual_AST"]],
        left_on="name", right_on="target_player", how="left",
    )

    OWN_WEIGHT = 0.6  # own season average is the more direct predictor; the comp's actual game is a secondary signal

    def blend_stat(row, own_col, comp_col):
        own, comp = row[own_col], row[comp_col]
        if pd.isna(comp):
            return own
        if pd.isna(own):
            return comp
        return OWN_WEIGHT * own + (1 - OWN_WEIGHT) * comp

    for stat in ["PTS", "REB", "AST"]:
        opponent_players[f"blended_{stat}"] = opponent_players.apply(lambda r, s=stat: blend_stat(r, s, f"comp_actual_{s}"), axis=1)

    blended_total_pts = opponent_players["blended_PTS"].sum()
    opponent_scale = expected_opponent_pts / blended_total_pts if blended_total_pts else 1.0
    opponent_players["projected_PTS"] = (opponent_players["blended_PTS"] * opponent_scale).round(1)
    opponent_players["projected_REB"] = opponent_players["blended_REB"].round(1)
    opponent_players["projected_AST"] = opponent_players["blended_AST"].round(1)

    # Per-player projection basis -- a plain-text explanation of exactly how each line was derived. Intended to be
    # surfaced as a hover tooltip/bubble wherever this table is displayed (e.g. an info icon next to each row in
    # the app), rather than as its own always-visible column.
    def opponent_projection_basis(row):
        if pd.isna(row["PTS"]):
            return "No season stats recorded for this player yet -- insufficient data to project."
        basis = f"Own season avg: {row['PTS']:.1f} PTS, {row['REB']:.1f} REB, {row['AST']:.1f} AST/gm"
        if pd.notna(row["comp_actual_PTS"]):
            basis += (
                f" | Blended {OWN_WEIGHT:.0%} own avg / {1 - OWN_WEIGHT:.0%} actual game -- comp match: "
                f"{row['compared_player']} ({row['compared_opponent']}, similarity {row['similarity_score']:.1f}) "
                f"actually posted {row['comp_actual_PTS']:.0f} PTS, {row['comp_actual_REB']:.0f} REB, "
                f"{row['comp_actual_AST']:.0f} AST vs this same UWW defense"
            )
        else:
            basis += " | No PBP data available for this player's comp match -- used own season average only"
        basis += f" | Scaled x{opponent_scale:.2f} so the roster sums to the projected team total ({expected_opponent_pts:.0f} pts)"
        return basis

    opponent_players["projection_basis"] = opponent_players.apply(opponent_projection_basis, axis=1)

    # MIN was previously left out of this selection even though opponent_players already carries the opponent's
    # own season-average minutes (same source uww_player_profiles.MIN the app already reads elsewhere) -- added
    # so the opponent's side of Projected Box Score isn't missing minutes played while UWW's own side has it.
    projected_opponent_box = opponent_players[
        [c for c in ["name", "jersey_number", "role", "position", "MIN", "projected_PTS", "projected_REB", "projected_AST",
         "compared_player", "compared_opponent", "similarity_score", "projection_basis"] if c in opponent_players.columns]
    ].rename(columns={"compared_player": "comp_used", "compared_opponent": "comp_from_game"}).sort_values("projected_PTS", ascending=False).reset_index(drop=True)

    print(f"\n{UPCOMING_MATCHUP_OPPONENT} projected team total: {projected_opponent_box['projected_PTS'].sum():.1f} pts "
          f"(scaled from a {OWN_WEIGHT:.0%} own-season-average / {1 - OWN_WEIGHT:.0%} comp-actual-game blend)")
    print(projected_opponent_box)

    # --- 3. UWW projected player box score -----------------------------------------------------------------------
    # Scale against the SUM of individual player PTS averages, not the season "Team Total" row -- per-player
    # averages are each computed over that player's OWN games played (GP-GS varies by player), so they don't sum
    # exactly to the team's average PPG (a pre-existing quirk of the scraped season stats, not introduced here).
    # Scaling to the raw individual sum keeps this table's total consistent with the printed team-level projection.
    uww_player_rows = stats[~stats["PLAYER"].isin(["Team Total", "Opponent"])].copy()
    for col in ["PTS", "REB", "AST", "MIN"]:
        uww_player_rows[col] = pd.to_numeric(uww_player_rows[col], errors="coerce")
    raw_individual_pts_sum = uww_player_rows["PTS"].sum()
    uww_scale = expected_uww_pts / raw_individual_pts_sum if raw_individual_pts_sum else 1.0
    uww_player_rows["projected_PTS"] = (uww_player_rows["PTS"] * uww_scale).round(1)
    uww_player_rows["projected_REB"] = uww_player_rows["REB"]
    uww_player_rows["projected_AST"] = uww_player_rows["AST"]

    # Per-player projection basis -- same idea as the opponent's: a plain-text explanation meant for a hover tooltip/bubble.
    def uww_projection_basis(row):
        if pd.isna(row["PTS"]):
            return "No season stats recorded for this player yet -- insufficient data to project."
        min_str = f"{row['MIN']:.0f}" if pd.notna(row["MIN"]) else "?"
        return (
            f"Own season avg: {row['PTS']:.1f} PTS, {row['REB']:.1f} REB, {row['AST']:.1f} AST/gm over {min_str} min "
            f"| Scaled x{uww_scale:.2f} for this matchup's projected pace (UWW projected team total {expected_uww_pts:.0f} pts: "
            f"50% season PPG {uww_team_pts_season:.1f} + 50% actual PPG vs scouted opponents this season {uww_actual_pts_avg:.1f})"
        )

    uww_player_rows["projection_basis"] = uww_player_rows.apply(uww_projection_basis, axis=1)

    projected_uww_box = uww_player_rows[
        ["PLAYER", "MIN", "projected_PTS", "projected_REB", "projected_AST", "FG%", "3P%", "FT%", "projection_basis"]
    ].sort_values("projected_PTS", ascending=False).reset_index(drop=True)

    # --- Stamp WHICH GAME this projection is for -------------------------------------------------------
    # Both projection tables described "the upcoming game" and carried nothing saying which game that was,
    # so once the next parser run overwrote them there was no way to line an old projection up against the
    # game it was made for. The app's Previous Games page could only compare a past game against whatever
    # projection happened to be on disk. These two columns plus the append-on-export below (see the CSV
    # export cell) turn the projections into a permanent per-game record.
    _proj_opp = upcoming_opponent if upcoming_game is not None else UPCOMING_MATCHUP_OPPONENT
    _proj_date = upcoming_game["date"] if upcoming_game is not None else None
    for _pf in (projected_uww_box, projected_opponent_box):
        _pf.insert(0, "game_date", _proj_date)
        _pf.insert(0, "opponent", _proj_opp)

    print(f"\nUW-Whitewater projected team total: {projected_uww_box['projected_PTS'].sum():.1f} pts "
          f"(season averages scaled x{uww_scale:.2f} for pace/matchup)")
    print(projected_uww_box)

    # --- Narrative context from the opponent's game plan + relevant coaching flags on UWW's top projected scorers -
    opponent_ktv = all_game_plans.loc[
        (all_game_plans["opponent"] == UPCOMING_MATCHUP_OPPONENT) & (all_game_plans["topic"] == "KEYS TO VICTORY"), "notes"
    ]
    opponent_strengths = all_game_plans.loc[
        (all_game_plans["opponent"] == UPCOMING_MATCHUP_OPPONENT) & (all_game_plans["topic"] == "TEAM STRENGTHS"), "notes"
    ]
    print(f"\n{UPCOMING_MATCHUP_OPPONENT}'s pre-game keys to victory:", opponent_ktv.iloc[0] if not opponent_ktv.empty else "n/a")
    print(f"{UPCOMING_MATCHUP_OPPONENT}'s team strengths:", opponent_strengths.iloc[0] if not opponent_strengths.empty else "n/a")

    top_scorers = set(projected_uww_box.head(5)["PLAYER"].str.lower())
    relevant_flags = coaching_flags_df[coaching_flags_df["player_key"].isin(top_scorers)]
    if not relevant_flags.empty:
        print("\nCoaching flags on UWW's top-5 projected scorers:")
        for _, f in relevant_flags.iterrows():
            print(f"  [{f['sentiment']}] {f['player']} -- {f['flag']} ({f['evidence']})")

Projected final score: UW-Whitewater 76 - Aurora Spartans 76 (margin -1)
  UWW inputs: season PTS/gm=78.4, actual PTS/gm vs scouted opponents=72.9
  Aurora Spartans inputs: PBP-derived PTS/gm=76.0, UWW's actual PTS/gm allowed to scouted opponents=76.5

Aurora Spartans projected team total: 0.0 pts (scaled from a 60% own-season-average / 40% comp-actual-game blend)
Empty DataFrame
Columns: [name, jersey_number, role, position, MIN, projected_PTS, projected_REB, projected_AST, comp_used, comp_from_game, similarity_score, projection_basis]
Index: []

UW-Whitewater projected team total: 75.5 pts (season averages scaled x0.83 for pace/matchup)
           opponent    game_date                  PLAYER   MIN  projected_PTS  \
0   Aurora Spartans  Wed, Nov 19           Collin Madson  33.0           15.0   
1   Aurora Spartans  Wed, Nov 19               Luke Bara  24.0            8.5   
2   Aurora Spartans  Wed, Nov 19            Isaac Verges  23.0            8.1   
3   Aurora Spartans  Wed, Nov


### Export tables as CSVs into the app's bundled data directory

Writes every table this notebook produces (schedule, box scores, scouting reports, PBP/lineup analytics, player comparisons, coaching flags, etc.) out as CSVs into `OUTPUT_DIR`, for the Streamlit app to read.


### Reconstruct a box score for the opponent's games before facing UWW

The shot-by-shot video-tagged data for these games already exists (pbp_events_upcoming, built above, and
already exported as uww_opponent_prior_games_pbp for the shot-selection analysis) -- this just aggregates
it into a per-player box score, the exact same way uww_pbp_box_score already does for UWW's own games
(same event_type -> stat mapping, same groupby). This is what lets the app's Last Five Games page show a
full box score when a coach clicks one of the opponent's own results, not just UWW's.

In [141]:
# --- Reconstruct a box score from pbp_events_upcoming, same event_type -> stat mapping as pbp_box_score ---
# CONFIRMED CHANGE (requested): the aggregation below used to be written inline against
# pbp_events_upcoming. It is now a function, because a second caller exists: the cell that back-fills
# PAST scouted opponents' stats from their own prior-game PBP. Two copies of an event_type -> stat mapping
# is precisely the drift this notebook has been bitten by before -- one copy gets a fix, the other doesn't,
# and two tables that claim to mean the same thing quietly stop agreeing. One function, two callers.
def box_score_from_pbp_events(events, label="the upcoming opponent's prior game(s)", verbose=True):
    """Player-level box score from parsed PBP events. Empty-but-well-formed frame when there's nothing.

    Columns are keyed (opponent, game_date, team, player) -- note `opponent` here means the THIRD PARTY in
    that game, and `team` is the side the player was on, the same convention pbp_events uses everywhere
    else in this notebook.
    """
    box_cols = ["opponent", "game_date", "team", "player", "PTS", "FGM", "FGA", "FG3M", "FG3A",
                "FTM", "FTA", "OREB", "DREB", "AST", "STL", "BLK", "TO", "PF", "REB", "FG%", "3P%", "FT%"]
    if events is None or events.empty:
        if verbose:
            print(f"No PBP data available for {label} -- box score is empty but well-formed.")
        return pd.DataFrame(columns=box_cols)

    player_events = events[
        events["player"].notna() & (~events["event_type"].isin(TEAM_LEVEL_EVENT_TYPES_BOX))
    ].copy()
    if player_events.empty:
        if verbose:
            print(f"No player-attributed PBP events for {label} -- box score is empty but well-formed.")
        return pd.DataFrame(columns=box_cols)

    player_events["points"] = player_events.apply(
        lambda row: int(row["shot_type"]) if row["event_type"] == "made_shot" else (1 if row["event_type"] == "free_throw_made" else 0),
        axis=1,
    )
    player_events["is_fgm"] = player_events["event_type"] == "made_shot"
    player_events["is_fga"] = player_events["event_type"].isin(["made_shot", "missed_shot"])
    player_events["is_3pm"] = player_events["is_fgm"] & (player_events["shot_type"] == "3")
    player_events["is_3pa"] = player_events["is_fga"] & (player_events["shot_type"] == "3")
    player_events["is_ftm"] = player_events["event_type"] == "free_throw_made"
    player_events["is_fta"] = player_events["event_type"].isin(["free_throw_made", "free_throw_missed"])
    player_events["is_oreb"] = player_events["event_type"] == "rebound_offensive"
    player_events["is_dreb"] = player_events["event_type"] == "rebound_defensive"
    player_events["is_ast"] = player_events["event_type"] == "assist"
    player_events["is_stl"] = player_events["event_type"] == "steal"
    player_events["is_blk"] = player_events["event_type"] == "block"
    player_events["is_to"] = player_events["event_type"] == "turnover"
    player_events["is_pf"] = player_events["event_type"] == "foul"

    box = player_events.groupby(["opponent", "game_date", "team", "player"]).agg(
        PTS=("points", "sum"), FGM=("is_fgm", "sum"), FGA=("is_fga", "sum"),
        FG3M=("is_3pm", "sum"), FG3A=("is_3pa", "sum"), FTM=("is_ftm", "sum"), FTA=("is_fta", "sum"),
        OREB=("is_oreb", "sum"), DREB=("is_dreb", "sum"), AST=("is_ast", "sum"), STL=("is_stl", "sum"),
        BLK=("is_blk", "sum"), TO=("is_to", "sum"), PF=("is_pf", "sum"),
    ).reset_index()

    # Same gap, same fix as pbp_box_score above: a bare team-level turnover (no player attached) is correctly
    # excluded from player_events via the player.notna() filter, but that means it never appeared anywhere
    # in this box score either. Surfaced as a synthetic "TEAM" row instead of silently dropped.
    team_level_turnovers = events[events["player"].isna() & (events["event_type"] == "turnover")]
    if not team_level_turnovers.empty:
        team_to_rows = team_level_turnovers.groupby(["opponent", "game_date", "team"]).size().reset_index(name="TO")
        team_to_rows["player"] = "TEAM"
        for stat_col in ["PTS", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA", "OREB", "DREB", "AST", "STL", "BLK", "PF"]:
            team_to_rows[stat_col] = 0
        box = pd.concat([box, team_to_rows], ignore_index=True)
        if verbose:
            print(f"Added {len(team_to_rows)} synthetic TEAM row(s) for {int(team_to_rows['TO'].sum())} bare team-level turnover(s) in {label}.")

    box["REB"] = box["OREB"] + box["DREB"]
    box["FG%"] = (100 * box["FGM"] / box["FGA"]).round(1)
    box["3P%"] = (100 * box["FG3M"] / box["FG3A"]).round(1)
    box["FT%"] = (100 * box["FTM"] / box["FTA"]).round(1)
    if verbose:
        print(f"Reconstructed box score for {box['opponent'].nunique()} of {label}, {len(box)} player-game row(s) total.")
    return box


pbp_box_score_upcoming = box_score_from_pbp_events(pbp_events_upcoming)


Added 2 synthetic TEAM row(s) for 4 bare team-level turnover(s) in the upcoming opponent's prior game(s).
Reconstructed box score for 4 of the upcoming opponent's prior game(s), 88 player-game row(s) total.



### Minutes played for the upcoming opponent's prior games

Reuses the lineup reconstruction and stints already built two cells up (`self_lineup`/`their_lineup`,
`stint_src`, `stints`, `minutes_margin`) instead of redoing that same substitution-tracking pass a second
time -- an earlier version of this cell duplicated that whole algorithm from scratch under different
column names, which was pure waste and a real risk of the two copies quietly drifting apart over time.
`stints`/`minutes_margin` above only cover the upcoming opponent's OWN lineups (that's all the season-box
cell needed); the one genuinely new piece here is doing the same for `their_lineup` (whichever third party
they played each game), so both sides of `pbp_box_score_upcoming` get a MIN column, not just one.

In [143]:
# --- Minutes played for BOTH sides of the upcoming opponent's prior games, reusing cell above's work --------
if pbp_events_upcoming.empty or "stint_src" not in globals():
    pbp_box_score_upcoming["MIN"] = None
    lineup_stints_upcoming = pd.DataFrame(columns=["opponent", "lineup", "MIN", "+/-"])
    print("No prior-game lineup data available yet -- MIN and lineup_stints_upcoming left empty.")
else:
    # Third-party side: same stint_seconds already computed in stint_src above, just grouped by their_lineup
    # instead of self_lineup (which is all the season-box cell needed and kept).
    third_party_minutes_by_lineup = stint_src.groupby(GAME_KEYS + ["their_lineup"])["seconds_elapsed"].sum().reset_index()
    third_party_minutes_by_lineup["MIN"] = (third_party_minutes_by_lineup["seconds_elapsed"] / 60).round(2)

    def _lineup_minutes_to_player_minutes(df, lineup_col, team_col_value_fn):
        rows = []
        for _, r in df.iterrows():
            if pd.isna(r[lineup_col]):
                continue
            for player in str(r[lineup_col]).split(", "):
                rows.append({"opponent": r["opponent"], "game_date": r["game_date"], "team": team_col_value_fn(r), "player": player, "minutes": r["MIN"]})
        return rows

    _self_rows = _lineup_minutes_to_player_minutes(
        minutes_margin.rename(columns={"lineup": "self_lineup"}), "self_lineup", lambda r: upcoming_opponent_short,
    )
    _third_party_rows = _lineup_minutes_to_player_minutes(
        third_party_minutes_by_lineup, "their_lineup", lambda r: r["opponent"],
    )

    _pu_minutes_rows = _self_rows + _third_party_rows
    if _pu_minutes_rows:
        _pu_player_minutes = pd.DataFrame(_pu_minutes_rows).groupby(GAME_KEYS + ["team", "player"])["minutes"].sum().reset_index()
        _pu_player_minutes["MIN"] = _pu_player_minutes["minutes"].round(1)
        if "MIN" in pbp_box_score_upcoming.columns:
            pbp_box_score_upcoming = pbp_box_score_upcoming.drop(columns=["MIN"])
        pbp_box_score_upcoming = pbp_box_score_upcoming.merge(
            _pu_player_minutes[GAME_KEYS + ["team", "player", "MIN"]], on=GAME_KEYS + ["team", "player"], how="left",
        )
        _pu_n_matched = int(pbp_box_score_upcoming["MIN"].notna().sum())
        print(f"Matched minutes played for {_pu_n_matched} of {len(pbp_box_score_upcoming)} pbp_box_score_upcoming row(s).")
    else:
        pbp_box_score_upcoming["MIN"] = None
        print("No lineup stints available yet -- pbp_box_score_upcoming.MIN left empty.")

    # Per-game lineup stints for the upcoming opponent's OWN lineups, already reconstructed above (`stints`) --
    # exported directly rather than rebuilt, matching upcoming_lineup_season's own scope (their own lineups
    # only; third-party lineup stints aren't exported standalone since nothing in the app needs that yet).
    lineup_stints_upcoming = stints.rename(columns={"self_lineup": "lineup"})[GAME_KEYS + ["lineup", "stint_minutes", "margin_change"]]

# --- Who STARTED each of the upcoming opponent's prior games -------------------------------------------------
# CONFIRMED CHANGE (requested): the brief's personnel pages now split Starters from the bench tiers the app
# uses. UWW's own box score already carries `started` (read from the official box-score asterisks), but the
# opponent's prior-game box score is rebuilt from play-by-play and never had one -- so with before_scout="yes"
# (no scouting-report roles) there was no way to say who starts. The play-by-play's own lineup reconstruction
# already knows the first five-man unit on the floor in each game (self_lineup, persisted onto
# pbp_events_upcoming by the season-lineup cell), and those five are the starters by definition.
pbp_box_score_upcoming["started"] = False
if not pbp_events_upcoming.empty and "self_lineup" in pbp_events_upcoming.columns:
    _st_src = pbp_events_upcoming.dropna(subset=["self_lineup"])
    if "event_order" in _st_src.columns:
        _st_src = _st_src.sort_values("event_order")
    _st_first = _st_src.groupby(GAME_KEYS)["self_lineup"].first()
    _st_by_game = {k: {p.strip() for p in str(v).split(",") if p.strip()} for k, v in _st_first.items()}
    pbp_box_score_upcoming["started"] = pbp_box_score_upcoming.apply(
        lambda r: r["team"] == upcoming_opponent_short
        and r["player"] in _st_by_game.get((r["opponent"], r["game_date"]), set()),
        axis=1,
    )
    print(f"Starters detected for {len(_st_by_game)} of {upcoming_opponent_short}'s prior game(s).")


Matched minutes played for 86 of 88 pbp_box_score_upcoming row(s).
Starters detected for 4 of Aurora Spartans's prior game(s).



### Replace the upcoming opponent's PDF-scouting-report stats with PBP-derived ones

player_profiles' PTS/REB/AST/etc for a scouted opponent normally come from that opponent's own scouting-
report PDF (a static document, captured once). For the CURRENT upcoming opponent specifically, `pbp_box_
score_upcoming` now holds a fully reconstructed box score for every one of their games before facing UWW --
the exact same games `reference_date` scopes everything else in this notebook to. Overriding their
player_profiles stats with an aggregate of THAT instead makes those numbers correctly reference-date-aware,
the same way everything else already had to be fixed to be.

**Scope, explicitly**: this only replaces the CURRENT upcoming opponent's stats. For an opponent UWW has
ALREADY PLAYED, the only PBP data this notebook has for them is their single game against UWW (from
pbp_box_score) -- not a season's worth of games. Using that as a season-stat substitute would trade one
real problem (a stale PDF snapshot) for a different one (a single noisy game standing in for a season
average), silently. Already-played opponents keep their PDF-sourced stats for now -- a genuine, open
limitation, not something this cell quietly papers over.

In [145]:
# --- Override the upcoming opponent's player_profiles stats with a PBP-derived aggregate ---------------------
# CONFIRMED CHANGE (requested): reference_date can be set before the opponent has played ANY game at all
# -- confirmed directly via their OWN schedule, prev_games, which is now correctly reference-date-gated
# (see the prev_games fix above). In that case, a PDF scout report's stats (if one already exists, dated
# from whenever it was actually compiled in the real world) aren't just "not yet overridden" -- they're
# KNOWN to describe games that, as of reference_date, haven't happened yet. Leaving them in place is
# exactly the leak reported live: "Opponent Scoring Reliance" showing real numbers for an opponent who
# hasn't played a game this year. Blank the stat columns outright here, rather than leaving PDF-sourced
# numbers in place, since their own schedule is direct proof no legitimate stats can exist yet.
if prev_games.empty and upcoming_opponent_short is not None:
    _pu_upcoming_mask = player_profiles["opponent"] == upcoming_opponent_short
    _pu_blank_cols = ["MIN", "FG%", "3PM-A", "3P%", "FTM-A", "FT%", "REB", "OREB", "DREB", "AST", "TO", "STL", "BLK", "PTS", "games_played"]
    for _pu_col in _pu_blank_cols:
        if _pu_col in player_profiles.columns:
            player_profiles.loc[_pu_upcoming_mask, _pu_col] = None
    print(f"{upcoming_opponent_short} has no games on record before reference_date ({reference_date_str}) -- "
          f"blanked any PDF-sourced season stats for their {int(_pu_upcoming_mask.sum())} player_profiles "
          f"row(s) rather than showing numbers from games that haven't happened yet.")
elif pbp_box_score_upcoming.empty or upcoming_opponent_short is None:
    print("No prior-game box score available yet -- upcoming opponent's player_profiles stats left as-is (PDF-sourced, if any).")
else:
    _pu_own_box = pbp_box_score_upcoming[pbp_box_score_upcoming["team"] == upcoming_opponent_short].copy()
    _pu_own_box = _pu_own_box[_pu_own_box["player"] != "TEAM"]  # exclude the synthetic bare-turnover row
    if _pu_own_box.empty:
        print(f"No player-level PBP box score data available yet for {upcoming_opponent_short} -- player_profiles stats left as-is.")
    else:
        _pu_agg = _pu_own_box.groupby("player").agg(
            games=("game_date", "nunique"),
            PTS_total=("PTS", "sum"), REB_total=("REB", "sum"), MIN_total=("MIN", "sum"),
            OREB_total=("OREB", "sum"), DREB_total=("DREB", "sum"),
            FGM=("FGM", "sum"), FGA=("FGA", "sum"), FG3M=("FG3M", "sum"), FG3A=("FG3A", "sum"),
            FTM=("FTM", "sum"), FTA=("FTA", "sum"),
            AST=("AST", "sum"), STL=("STL", "sum"), BLK=("BLK", "sum"), TO=("TO", "sum"),
        ).reset_index()

        # PTS/REB/MIN/OREB/DREB: per-game averages (matches player_profiles' existing convention for PTS/REB/
        # MIN; OREB/DREB are new here -- the PDF-sourced player_profiles table never had a rebound split at
        # all, only combined REB, so these two columns didn't exist on this table before this override).
        # AST/STL/BLK/TO: left as SEASON TOTALS on purpose -- get_opponent_games_played() divides these
        # downstream throughout the app; pre-dividing here would double-divide them.
        _pu_agg["PTS"] = (_pu_agg["PTS_total"] / _pu_agg["games"]).round(1)
        _pu_agg["REB"] = (_pu_agg["REB_total"] / _pu_agg["games"]).round(1)
        _pu_agg["MIN"] = (_pu_agg["MIN_total"] / _pu_agg["games"]).round(1)
        _pu_agg["OREB"] = (_pu_agg["OREB_total"] / _pu_agg["games"]).round(1)
        _pu_agg["DREB"] = (_pu_agg["DREB_total"] / _pu_agg["games"]).round(1)
        # CONFIRMED BUG (fixed here): the original `.astype(str).replace("nan", "")` approach for a
        # zero-attempts player (FGA/FG3A/FTA == 0, a real case -- e.g. a player who saw the floor but never
        # attempted a 3) doesn't actually work in this pandas version: .astype(str) on a float NaN keeps it
        # as an actual null rather than stringifying it to the literal text "nan", so .replace("nan", "") has
        # nothing to match and the NaN survives all the way through, then poisons the "+ \'%\'" concatenation
        # into another NaN. Verified directly (not assumed) before shipping this fix -- the old code would
        # have left a real, silent NaN in this column for exactly the players this was meant to handle
        # gracefully. Using an explicit per-value formatter instead of a dtype-fragile string trick.
        def _pct_str(v):
            return f"{v}%" if pd.notna(v) else "-"
        _pu_agg["FG%"] = (100 * _pu_agg["FGM"] / _pu_agg["FGA"]).round(1).apply(_pct_str)
        _pu_agg["3P%"] = (100 * _pu_agg["FG3M"] / _pu_agg["FG3A"]).round(1).apply(_pct_str)
        _pu_agg["FT%"] = (100 * _pu_agg["FTM"] / _pu_agg["FTA"]).round(1).apply(_pct_str)
        _pu_agg["3PM-A"] = _pu_agg["FG3M"].astype(int).astype(str) + "-" + _pu_agg["FG3A"].astype(int).astype(str)
        _pu_agg["FTM-A"] = _pu_agg["FTM"].astype(int).astype(str) + "-" + _pu_agg["FTA"].astype(int).astype(str)

        # AST/STL/BLK/TO stay SEASON TOTALS (the app divides them) -- but the app was dividing by the
        # TEAM's games played, which understates anyone who missed time: Gavin Sarvis' 73 assists in the 24
        # games he actually played rendered as 2.6/gm instead of 3.0 because the divisor was Loras' 28.
        # Ship each player's own games-played alongside the totals so the app can divide correctly.
        _pu_agg["games_played"] = _pu_agg["games"]
        _pu_stat_cols = ["MIN", "FG%", "3PM-A", "3P%", "FTM-A", "FT%", "REB", "OREB", "DREB", "AST", "TO", "STL", "BLK", "PTS", "games_played"]
        if "games_played" not in player_profiles.columns:
            player_profiles["games_played"] = pd.NA
        _pu_replacement = _pu_agg[["player"] + _pu_stat_cols].rename(columns={"player": "name"})

        # Match by player name (pbp_box_score_upcoming has no jersey_number) -- case-insensitive, since roster
        # names elsewhere in this notebook are sometimes cased slightly differently between sources.
        _pu_upcoming_mask = player_profiles["opponent"] == upcoming_opponent_short
        _pu_name_to_row = {str(n).strip().casefold(): row for n, row in zip(_pu_replacement["name"], _pu_replacement.to_dict("records"))}
        _pu_n_matched = 0
        for _pu_idx in player_profiles[_pu_upcoming_mask].index:
            _pu_key = str(player_profiles.at[_pu_idx, "name"]).strip().casefold()
            if _pu_key in _pu_name_to_row:
                for _pu_col in _pu_stat_cols:
                    player_profiles.at[_pu_idx, _pu_col] = _pu_name_to_row[_pu_key][_pu_col]
                _pu_n_matched += 1
        print(f"Replaced PDF-sourced stats with PBP-derived stats for {_pu_n_matched} of {_pu_upcoming_mask.sum()} "
              f"{upcoming_opponent_short} player_profiles row(s), from {_pu_agg['games'].max()} prior game(s) of PBP data.")

        # --- Add PBP-derived players who have NO scout-report entry at all ---------------------------------
        # CONFIRMED BUG (fixed here): the loop above only ever UPDATES rows that already exist in
        # player_profiles, so a player who shows up in the opponent's prior-game PBP but was never in the
        # scout report was dropped on the floor entirely -- and with him, his contribution to every
        # team-level sum the app builds off this table.
        #
        # Confirmed case: Andrew Wells had 2 blocks for Eureka vs Dominican (IL), but "Wells" appears nowhere
        # in Eureka's scout report -- not in the roster AND not in its season BOXSCORE table, so the
        # `box_only` backfill in the player_profiles cell can't recover him either (that one only rescues
        # players the scout report's own boxscore lists). The app therefore summed 3 of Eureka's 5 blocks and
        # rendered 1.0 BPG (3/3) instead of 1.67 (5/3). Note the failing number was the NUMERATOR -- the
        # games-played denominator was correct all along.
        #
        # These additions get the same treatment `box_only` gives its own: role="Bench",
        # has_scouting_report=False, null demographics. jersey_number stays null because
        # pbp_box_score_upcoming has no jersey column (that's why the update loop above matches on name).
        _pu_existing = {str(n).strip().casefold() for n in player_profiles.loc[_pu_upcoming_mask, "name"].dropna()}
        _pu_add = _pu_replacement[
            ~_pu_replacement["name"].astype(str).str.strip().str.casefold().isin(_pu_existing)
        ].copy()
        if not _pu_add.empty:
            _pu_add["opponent"] = upcoming_opponent_short
            _pu_add["game_date"] = game_date_for(upcoming_opponent_short)
            _pu_add["jersey_number"] = None
            _pu_add["role"] = "Bench"
            _pu_add["has_scouting_report"] = False
            _pu_add["position_group"] = "Unknown"
            _pu_add["player_notes"] = ""
            _pu_add["keys_to_defending"] = ""
            _pu_add["notes_tags_display"] = ""
            _pu_add["keys_tags_display"] = ""
            for _pu_missing_col in player_profiles.columns:
                if _pu_missing_col not in _pu_add.columns:
                    _pu_add[_pu_missing_col] = None
            # Assigned AFTER the None-fill loop above on purpose: these two columns hold real sets (other
            # cells call set operations on them directly), and a column of shared Nones would break that.
            _pu_add["notes_tags"] = [set() for _ in range(len(_pu_add))]
            _pu_add["keys_tags"] = [set() for _ in range(len(_pu_add))]
            player_profiles = pd.concat(
                [player_profiles, _pu_add[player_profiles.columns.tolist()]], ignore_index=True
            )
            # _pu_upcoming_mask was built against the PRE-concat frame; reusing a now-too-short boolean mask
            # against the grown frame is a pandas IndexingError, so rebuild it before the diagnostics below.
            _pu_upcoming_mask = player_profiles["opponent"] == upcoming_opponent_short
            print(f"  Added {len(_pu_add)} PBP-only player(s) with no scout-report entry at all "
                  f"(role=Bench): {sorted(_pu_add['name'].astype(str))}")

            # Mirror the same additions into all_rosters, which is exported as uww_opponent_rosters and is
            # what the app's ROSTER panel reads -- without this a PBP-only player shows up in the team-stat
            # sums (player_profiles) but is invisible in the roster list, which reads as a data bug to whoever
            # is checking the numbers by hand. Kept to roster_cols only: all_rosters is the pre-enrichment
            # table (no stat columns, no tag columns).
            _pu_roster_add = _pu_add[[c for c in roster_cols if c in _pu_add.columns]].copy()
            for _pu_missing_roster_col in roster_cols:
                if _pu_missing_roster_col not in _pu_roster_add.columns:
                    _pu_roster_add[_pu_missing_roster_col] = None
            # Independent dedupe rather than relying on the enclosing `if not _pu_add.empty` guard: that guard
            # is driven by player_profiles, so re-running the player-profiles cell WITHOUT re-running the
            # roster cell would otherwise append a second copy of the same player here.
            _pu_roster_have = {
                (str(o).strip().casefold(), str(n).strip().casefold())
                for o, n in zip(all_rosters.get("opponent", []), all_rosters.get("name", []))
            }
            _pu_roster_add = _pu_roster_add[[
                (str(r["opponent"]).strip().casefold(), str(r["name"]).strip().casefold()) not in _pu_roster_have
                for _, r in _pu_roster_add.iterrows()
            ]]
            if not _pu_roster_add.empty:
                all_rosters = pd.concat([all_rosters, _pu_roster_add[roster_cols]], ignore_index=True)
                print(f"  Also added {len(_pu_roster_add)} of them to all_rosters (uww_opponent_rosters) so "
                      f"they appear in the app's ROSTER panel, not just the team-stat sums.")
        else:
            print("  No PBP-only players to add -- every player in the prior-game PBP already had a "
                  "player_profiles row.")
        _pu_roster_names = set(player_profiles.loc[_pu_upcoming_mask, "name"].astype(str))
        _pu_pbp_names = set(_pu_replacement["name"].astype(str))
        _pu_unmatched_names = _pu_roster_names - _pu_pbp_names
        if _pu_unmatched_names:
            print(f"  Roster player(s) with no matching PBP box-score row yet (kept their PDF-sourced stats, if any): {sorted(_pu_unmatched_names)}")
        # STRENGTHENED DIAGNOSTIC -- this exact override has now been reported as "still showing the old
        # number" once already; rather than guess a third time, print everything needed to tell apart the
        # three real possibilities in one look: (a) a genuine name-spelling mismatch between the roster and
        # what got parsed off the raw PBP text (b) the override ran but the app is reading a stale/un-
        # redeployed CSV (c) this cell genuinely hasn't been re-run since the fix went in.
        print(f"\n  Roster names for {upcoming_opponent_short} ({len(_pu_roster_names)}): {sorted(_pu_roster_names)}")
        print(f"  PBP-derived names found ({len(_pu_pbp_names)}): {sorted(_pu_pbp_names)}")
        print("  Per-player PTS after this override (spot-check against what the app is showing):")
        print(player_profiles.loc[_pu_upcoming_mask, ["name", "PTS"]].sort_values("PTS", ascending=False).to_string(index=False))


Replaced PDF-sourced stats with PBP-derived stats for 0 of 0 Aurora Spartans player_profiles row(s), from 4 prior game(s) of PBP data.
  Added 13 PBP-only player(s) with no scout-report entry at all (role=Bench): ['Adam Awender', 'Alex Ross', 'Bryden Gryzmala', 'Cullen Rauls', 'Devon Richardson', 'Gevon Grant', 'Isaiah Thompson', 'Jeffery Hillmer', 'Juan Madrigal', 'Larry Carthan', 'Mekhi Doby', 'Robert Hutson', 'Zerrick Johnson']
  Also added 13 of them to all_rosters (uww_opponent_rosters) so they appear in the app's ROSTER panel, not just the team-stat sums.

  Roster names for Aurora Spartans (13): ['Adam Awender', 'Alex Ross', 'Bryden Gryzmala', 'Cullen Rauls', 'Devon Richardson', 'Gevon Grant', 'Isaiah Thompson', 'Jeffery Hillmer', 'Juan Madrigal', 'Larry Carthan', 'Mekhi Doby', 'Robert Hutson', 'Zerrick Johnson']
  PBP-derived names found (13): ['Adam Awender', 'Alex Ross', 'Bryden Gryzmala', 'Cullen Rauls', 'Devon Richardson', 'Gevon Grant', 'Isaiah Thompson', 'Jeffery Hillmer'

C:\Users\frits\AppData\Local\Temp\ipykernel_27492\1717124111.py:124: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  player_profiles = pd.concat(



### Replace the upcoming opponent's TEAM totals with PBP-derived ones too

`team_totals` (team_ppg / opp_ppg_allowed, exported as uww_opponent_team_totals) has the exact same problem
the player-level override two cells up already fixed: `extract_team_totals_from_pdf` reads a "Team Total"
row straight off the same static scouting-report PDF, with no reference_date awareness at all. Same fix,
same scope limit: only the CURRENT upcoming opponent gets overridden, using `pbp_box_score_upcoming`'s own
"TEAM"/per-player rows for their games before facing UWW.

In [147]:
# --- Override the upcoming opponent's team_totals (PPG / opponent-PPG-allowed) with a PBP-derived aggregate --
# NOTE: no team-level BLK here on purpose. Checked before building it: the app's Team Stats panel already
# computes team Blocks by summing player_profiles' own BLK column and dividing by get_opponent_games_played()
# -- which are both already fixed (the player-level PBP override two cells up, and the games-played scoping
# fix from earlier this project). Sum of player blocks IS the team total, by definition of what a box score
# is, so a separate team-level BLK column here would just be unused dead weight -- confirmed nothing in the
# app reads one before adding it.
# CONFIRMED CHANGE (requested): same fix as the player_profiles override above, and the same underlying
# leak -- prev_games (now reference-date-gated) proves this opponent hasn't played any game yet, so any
# PDF-sourced team_ppg/opp_ppg_allowed is known to describe games that haven't happened as of
# reference_date. Blanked here rather than left in place.
if prev_games.empty and upcoming_opponent_short is not None:
    if not team_totals.empty and upcoming_opponent_short in set(team_totals["opponent"]):
        _tt_mask = team_totals["opponent"] == upcoming_opponent_short
        team_totals.loc[_tt_mask, ["team_ppg", "opp_ppg_allowed"]] = None
        print(f"{upcoming_opponent_short} has no games on record before reference_date ({reference_date_str}) "
              f"-- blanked their PDF-sourced team_totals row rather than showing numbers from games that "
              f"haven't happened yet.")
    else:
        print(f"{upcoming_opponent_short} has no games on record before reference_date ({reference_date_str}) "
              f"and no existing team_totals row to blank -- nothing to do.")
elif pbp_box_score_upcoming.empty or upcoming_opponent_short is None:
    print("No prior-game box score available yet -- upcoming opponent's team_totals left as-is (PDF-sourced, if any).")
else:
    _tt_own = pbp_box_score_upcoming[pbp_box_score_upcoming["team"] == upcoming_opponent_short]
    _tt_opp = pbp_box_score_upcoming[pbp_box_score_upcoming["team"] != upcoming_opponent_short]
    _tt_n_games = pbp_box_score_upcoming["game_date"].nunique()
    if _tt_own.empty or _tt_n_games == 0:
        print(f"No PBP box score data available yet for {upcoming_opponent_short} -- team_totals left as-is.")
    else:
        _tt_team_ppg = round(_tt_own["PTS"].sum() / _tt_n_games, 2)
        _tt_opp_ppg_allowed = round(_tt_opp["PTS"].sum() / _tt_n_games, 2) if not _tt_opp.empty else None

        if team_totals.empty or upcoming_opponent_short not in set(team_totals["opponent"]):
            team_totals = pd.concat([team_totals, pd.DataFrame([{
                "opponent": upcoming_opponent_short, "team_ppg": _tt_team_ppg, "opp_ppg_allowed": _tt_opp_ppg_allowed,
            }])], ignore_index=True)
            print(f"Added a new PBP-derived team_totals row for {upcoming_opponent_short} (none existed from a PDF).")
        else:
            _tt_mask = team_totals["opponent"] == upcoming_opponent_short
            team_totals.loc[_tt_mask, "team_ppg"] = _tt_team_ppg
            if _tt_opp_ppg_allowed is not None:
                team_totals.loc[_tt_mask, "opp_ppg_allowed"] = _tt_opp_ppg_allowed
            print(f"Replaced PDF-sourced team_totals for {upcoming_opponent_short}: team_ppg={_tt_team_ppg}, "
                  f"opp_ppg_allowed={_tt_opp_ppg_allowed} (from {_tt_n_games} prior game(s) of PBP data).")


Added a new PBP-derived team_totals row for Aurora Spartans (none existed from a PDF).


C:\Users\frits\AppData\Local\Temp\ipykernel_27492\3558686415.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  team_totals = pd.concat([team_totals, pd.DataFrame([{



### Back-fill PAST scouted opponents' player stats from their own prior-game PBP

The override two cells up only touches the **upcoming** opponent. Every previously-scouted opponent keeps
the all-null stat columns set in the player-profiles cell, because scouting-report stats are disabled
parser-wide (they're last-season numbers). That is why every row of `uww_player_comparisons` came out with
an empty `compared_PTS` and a `comparison_method` of "scouting notes, no stats" — the compared side of the
comparison had no stats to compare, for any opponent, ever.

Their prior-game PBP files are already on disk (e.g. Elmhurst's five games before Dec 2). This cell runs
the same chain the upcoming opponent gets — own schedule → games before facing UWW → `_pbp` files →
`build_pbp_events` → `box_score_from_pbp_events` — once per past opponent, and fills their
`player_profiles` stat columns from it. Same cutoff rule as `prev_games`: strictly before the earlier of
their UWW matchup and `reference_date`, so nothing a team did after we played them can leak backwards into
what they looked like when we played them.


In [149]:
# --- Back-fill PAST scouted opponents' player_profiles stats from their own prior-game PBP -----------------
# CONFIRMED BUG (fixed here): player_profiles' stat columns are set all-null for EVERY opponent in the
# player-profiles cell (scouting-report stats are last-season data, disabled parser-wide). The only thing
# that ever filled them back in was the PBP override a couple of cells up, whose mask is
# `player_profiles["opponent"] == upcoming_opponent_short`. So exactly one team per run had stats and every
# previously-scouted opponent kept nulls forever.
#
# Downstream that surfaced as a comparison bug rather than a data bug, which is why it took a while to see:
# player_comparison.py's stat_similarity() returns None when either side has no numbers, similarity_score()
# then labels the pair "scouting notes, no stats", and EVERY row of uww_player_comparisons carried that
# label with an empty compared_PTS -- e.g. Damyen Jackson (Loras) compared to EJ Marshall (Elmhurst) on
# notes alone, despite Elmhurst's five pre-Dec-2 games sitting in INPUT_DIR the whole time. The stat_weight
# term (3 of a possible 11.5 evidence points) was silently unavailable for every comparison ever made.
#
# The cutoff is the same rule prev_games uses, applied per opponent: strictly before the EARLIER of their
# own UWW matchup and reference_date. A past opponent has played plenty of games since we faced them, and
# folding those in would describe a team we never played.
#
# Scope limits, on purpose:
#   - Only fills rows that already exist. Unlike the upcoming-opponent override, this does NOT add
#     PBP-only players to player_profiles/all_rosters: those feed roster panels and team-level sums for
#     the team being prepared for, and quietly growing every past opponent's roster is a bigger change
#     than the bug being fixed here.
#   - MIN is left alone. Minutes come from lineup stints (the cell above), which are only computed for the
#     upcoming opponent; fabricating a minutes column here would be inventing data.
_pp_targets = [o for o in scout_reports if o != upcoming_opponent_short]
_pp_filled_rows, _pp_summary, _pp_skipped = 0, [], []

if not _pp_targets:
    print("No previously-scouted opponents to back-fill.")
else:
    def _pp_own_schedule(opp_name):
        """This opponent's OWN schedule -- in-memory parse first, local backup file second.

        Mirrors the upcoming-opponent path (the team_schedules lookup and the _schedule_file_for fallback),
        rather than reimplementing the naming tolerance those two already carry.
        """
        for ts in team_schedules:
            if ts.empty:
                continue
            team = str(ts["team"].iloc[0])
            if team.lower() in opp_name.lower() or opp_name.lower() in team.lower():
                out = pd.DataFrame({
                    "date": ts["date"],
                    "opponent": ts["opponent"],
                })
                out["game_date"] = out["date"].apply(parse_schedule_date)
                return out
        path = _schedule_file_for(opp_name, opp_name, volume_dir)
        if not os.path.exists(path):
            return None
        raw = pd.read_html(StringIO(str(BeautifulSoup(load_html_snapshot(path), "lxml").find_all("table")[0])))[0]
        out = pd.DataFrame({
            "date": raw["Date"],
            "opponent": raw["Opponent"].apply(lambda x: split_opponent(x)[0]),
        })
        out["game_date"] = out["date"].apply(parse_schedule_date)
        return out

    def _pp_self_column(raw_df, known_names):
        """Which raw PBP column holds this team's own events, resolved by roster-name hits.

        Defined here rather than reusing cell 57's resolve_self_column: that one is created inside an
        `if not opp_schedule.empty:` branch, so on a run with no upcoming opponent it does not exist at
        all and this cell would die with a NameError on a code path that has nothing to do with the
        upcoming game.
        """
        uww_hits = sum(any(n in str(t) for n in known_names) for t in raw_df["uww_text"].dropna())
        opp_hits = sum(any(n in str(t) for n in known_names) for t in raw_df["opp_text"].dropna())
        return "uww_text" if uww_hits >= opp_hits else "opp_text"

    def _pp_uww_matchup_date(opp_name):
        """The date UWW played them, read off UWW's own schedule -- same source upcoming_game comes from."""
        rows = schedule[schedule["opponent"].astype(str).str.contains(re.escape(opp_name), case=False, na=False)]
        if rows.empty:
            return None
        return parse_schedule_date(rows.iloc[0]["date"], uww_season_start_year)

    for _pp_opp in _pp_targets:
        _pp_uww_date = _pp_uww_matchup_date(_pp_opp)
        if _pp_uww_date is None:
            _pp_skipped.append(f"{_pp_opp}: not resolvable on UWW's schedule")
            continue

        _pp_sched = _pp_own_schedule(_pp_opp)
        if _pp_sched is None or _pp_sched.empty:
            _pp_skipped.append(f"{_pp_opp}: no own-schedule file ('{_pp_opp} - Schedule.html/.mhtml')")
            continue

        _pp_cutoff = min(_pp_uww_date, reference_date.date())
        _pp_prev = _pp_sched[_pp_sched["game_date"].notna() & (_pp_sched["game_date"] < _pp_cutoff)]
        if _pp_prev.empty:
            _pp_skipped.append(f"{_pp_opp}: no games before {_pp_cutoff}")
            continue

        # Same column-resolution problem the upcoming opponent has: which raw PBP column holds THIS team's
        # own events depends on whose account exported the snapshot, so resolve it per file against their
        # known roster names rather than assuming.
        _pp_known = set(player_profiles.loc[player_profiles["opponent"] == _pp_opp, "name"].dropna())

        _pp_events_list = []
        _pp_games_found = 0
        for _, _pp_row in _pp_prev.iterrows():
            _pp_date_str = f"{_pp_row['game_date'].month}_{_pp_row['game_date'].day}_{_pp_row['game_date'].strftime('%y')}"
            _pp_files = glob.glob(f"{volume_dir}/{_pp_date_str}*{_pp_opp}*_pbp.*")
            if not _pp_files:
                continue
            _pp_games_found += 1
            for _pp_path in _pp_files:
                _pp_raw = parse_pbp_mhtml(_pp_path)
                _pp_events_list.append(build_pbp_events(
                    _pp_raw, _pp_row["opponent"], _pp_row["game_date"],
                    self_team=_pp_opp, self_column=_pp_self_column(_pp_raw, _pp_known),
                ))

        if not _pp_events_list:
            _pp_skipped.append(f"{_pp_opp}: {len(_pp_prev)} prior game(s) on their schedule, no _pbp file for any of them")
            continue

        _pp_box = box_score_from_pbp_events(
            pd.concat(_pp_events_list, ignore_index=True), label=f"{_pp_opp}'s games before UWW", verbose=False)
        _pp_own = _pp_box[(_pp_box["team"] == _pp_opp) & (_pp_box["player"] != "TEAM")]
        if _pp_own.empty:
            _pp_skipped.append(f"{_pp_opp}: PBP parsed but no rows attributed to them (self-column resolution?)")
            continue

        # Aggregate exactly the way the upcoming-opponent override does, so a number means the same thing
        # whichever side of the comparison it lands on:
        #   PTS/REB/OREB/DREB -> per-game averages
        #   AST/STL/BLK/TO    -> SEASON TOTALS (the app divides these by games_played downstream)
        _pp_agg = _pp_own.groupby("player").agg(
            games=("game_date", "nunique"),
            PTS_total=("PTS", "sum"), REB_total=("REB", "sum"),
            OREB_total=("OREB", "sum"), DREB_total=("DREB", "sum"),
            FGM=("FGM", "sum"), FGA=("FGA", "sum"), FG3M=("FG3M", "sum"), FG3A=("FG3A", "sum"),
            FTM=("FTM", "sum"), FTA=("FTA", "sum"),
            AST=("AST", "sum"), STL=("STL", "sum"), BLK=("BLK", "sum"), TO=("TO", "sum"),
        ).reset_index()
        for _pp_stat in ("PTS", "REB", "OREB", "DREB"):
            _pp_agg[_pp_stat] = (_pp_agg[f"{_pp_stat}_total"] / _pp_agg["games"]).round(1)

        def _pp_pct(v):
            return f"{v}%" if pd.notna(v) else "-"
        _pp_agg["FG%"] = (100 * _pp_agg["FGM"] / _pp_agg["FGA"]).round(1).apply(_pp_pct)
        _pp_agg["3P%"] = (100 * _pp_agg["FG3M"] / _pp_agg["FG3A"]).round(1).apply(_pp_pct)
        _pp_agg["FT%"] = (100 * _pp_agg["FTM"] / _pp_agg["FTA"]).round(1).apply(_pp_pct)
        _pp_agg["3PM-A"] = _pp_agg["FG3M"].astype(int).astype(str) + "-" + _pp_agg["FG3A"].astype(int).astype(str)
        _pp_agg["FTM-A"] = _pp_agg["FTM"].astype(int).astype(str) + "-" + _pp_agg["FTA"].astype(int).astype(str)
        _pp_agg["games_played"] = _pp_agg["games"]

        _pp_cols = ["FG%", "3PM-A", "3P%", "FTM-A", "FT%", "REB", "OREB", "DREB",
                    "AST", "TO", "STL", "BLK", "PTS", "games_played"]
        for _pp_col in _pp_cols:
            if _pp_col not in player_profiles.columns:
                player_profiles[_pp_col] = pd.NA

        _pp_by_name = {str(r["player"]).strip().casefold(): r for r in _pp_agg.to_dict("records")}
        _pp_mask = player_profiles["opponent"] == _pp_opp
        _pp_matched = 0
        for _pp_idx in player_profiles[_pp_mask].index:
            _pp_key = str(player_profiles.at[_pp_idx, "name"]).strip().casefold()
            if _pp_key in _pp_by_name:
                for _pp_col in _pp_cols:
                    player_profiles.at[_pp_idx, _pp_col] = _pp_by_name[_pp_key][_pp_col]
                _pp_matched += 1
        _pp_filled_rows += _pp_matched

        _pp_unmatched = sorted(
            {str(n) for n in player_profiles.loc[_pp_mask, "name"].dropna()}
            - {str(p) for p in _pp_agg["player"]}
        )
        _pp_summary.append({
            "opponent": _pp_opp,
            "UWW matchup": _pp_uww_date,
            "prior games on schedule": len(_pp_prev),
            "with _pbp file": _pp_games_found,
            "players filled": _pp_matched,
            "roster rows": int(_pp_mask.sum()),
            "no PBP match": len(_pp_unmatched),
        })
        if _pp_unmatched:
            print(f"  {_pp_opp}: no PBP box-score rows for {_pp_unmatched} (left blank rather than guessed)")

    print(f"\nBack-filled PBP-derived stats for {_pp_filled_rows} player_profiles row(s) across "
          f"{len(_pp_summary)} past opponent(s).")
    if _pp_summary:
        print(pd.DataFrame(_pp_summary).to_string(index=False))
    if _pp_skipped:
        print("\nSkipped (no data, not an error -- these keep blank stats and their comparisons stay "
              "notes-only):")
        for _pp_line in _pp_skipped:
            print(f"  {_pp_line}")

    # Spot-check the reported case directly rather than trusting the counts above.
    _pp_check = player_profiles[player_profiles["opponent"].astype(str).str.contains("Elmhurst", case=False, na=False)]
    if not _pp_check.empty:
        print("\nElmhurst spot-check (season stats BEFORE they played UWW on their matchup date):")
        print(_pp_check[["name", "games_played", "PTS", "REB", "AST", "FG%", "3P%"]].to_string(index=False))



Back-filled PBP-derived stats for 26 player_profiles row(s) across 2 past opponent(s).
             opponent UWW matchup  prior games on schedule  with _pbp file  players filled  roster rows  no PBP match
St. Thomas (TX) Celts  2025-11-14                        2               2              12           12             0
    Eureka Red Devils  2025-11-15                        3               3              14           14             0

Skipped (no data, not an error -- these keep blank stats and their comparisons stay notes-only):
  Ripon Red Hawks: no games before 2025-11-07



### Rebuild player comparisons with PBP-derived stats

The comparison cell near the top runs before the upcoming opponent's stats are replaced with PBP-derived ones. Bench players have no PDF season stats, so their comparisons were decided on size and position alone. Re-running the same function here, after the override, gives them real production evidence.


In [151]:
# --- Rebuild player comparisons now that the upcoming opponent's stats are PBP-derived ----------------------
# CONFIRMED BUG (fixed here): the comparison cell near the top of this notebook runs BEFORE the override a
# few cells up, which replaces the upcoming opponent's player_profiles stats with PBP-derived per-game
# aggregates. Order matters more than it looks:
#
#   - A scouted starter has PTS/REB/AST in the PDF, so his comparison had stat evidence either way.
#   - A bench player who was never written up has NO PDF season stats. At comparison time his stat columns
#     were empty, stat_similarity came back None, and his "comparable player" was decided by position,
#     height and role alone -- a size guess wearing the same score as a real match. Observed directly: an
#     un-scouted bench player scored 5.17 against a fully-scouted starter's 5.25.
#   - The PBP override then fills exactly those missing numbers, from the opponent's own prior games...
#     several cells too late for anyone to use them.
#
# Re-running the same function here, unchanged, with the now-populated player_profiles gives bench players
# real production evidence. Nothing else about the algorithm changes, and the earlier run stays where it is
# so its output can still be inspected above.
if target_opponent is None or not previous_opponents:
    print("No comparison target or no previously-scouted opponent -- leaving the earlier comparisons as they are.")
else:
    _pre_rebuild = best_matches.copy() if isinstance(best_matches, pd.DataFrame) else pd.DataFrame()

    _rebuilt = build_player_comparison_artifacts(
        schedule=schedule,
        scout_reports=scout_reports,
        player_profiles=player_profiles,   # now carries PBP-derived stats for the upcoming opponent
        cache_path=os.path.join(OUTPUT_DIR, "_cache", "llm_player_comparison_cache.jsonl"),
        use_llm=False,   # the LLM pass compares scouting NOTES, which the override didn't touch -- rerunning
                         # it would spend calls to reproduce the same answers. The earlier LLM artifacts are
                         # left untouched in the globals from the first run.
        upcoming_opponent=upcoming_opponent_short,
    )
    _new_best = _rebuilt["best_matches"]

    if _new_best.empty:
        print("Rebuild produced no comparisons -- keeping the earlier result rather than exporting an empty table.")
    else:
        best_matches = _new_best
        player_similarity = _rebuilt["player_similarity"]

        _gained = _stat_backed = 0
        if not _pre_rebuild.empty and "stat_similarity" in _pre_rebuild.columns:
            _before = _pre_rebuild.set_index("target_player")["stat_similarity"].to_dict()
            for _, _r in best_matches.iterrows():
                _was = _before.get(_r["target_player"])
                if pd.isna(_was) and pd.notna(_r.get("stat_similarity")):
                    _gained += 1
        _stat_backed = int(best_matches["stat_similarity"].notna().sum())

        print(f"Rebuilt player comparisons for {target_opponent} using PBP-derived stats.")
        print(f"  {_stat_backed} of {len(best_matches)} comparisons now have stat evidence "
              f"({_gained} gained it in this rebuild -- those were size-and-position guesses before).")

        _weak = best_matches[best_matches["evidence_coverage"] < 0.7]
        if not _weak.empty:
            print(f"\n  Still thin ({len(_weak)}) -- the app labels these so they aren't read as real matches:")
            for _, _r in _weak.iterrows():
                print(f"    {_r['target_player']} -> {_r['compared_player']} "
                      f"({_r['evidence_coverage']:.0%} evidence) -- {_r['comparison_method']}")

        print("\n  Comparisons for players with no scouting-report entry:")
        _unscouted = best_matches[~best_matches["target_has_scouting_report"].fillna(True)]
        if _unscouted.empty:
            print("    (none -- every rostered player has a scouting-report entry)")
        else:
            for _, _r in _unscouted.iterrows():
                print(f"    {_r['target_player']} -> {_r['compared_player']} of {_r['compared_opponent']} "
                      f"(score {_r['similarity_score']}, {_r['evidence_coverage']:.0%} evidence)")


LLM comparison skipped (use_llm=False) -- using tag-based similarity only.
Rebuilt player comparisons for Aurora Spartans using PBP-derived stats.
  13 of 13 comparisons now have stat evidence (0 gained it in this rebuild -- those were size-and-position guesses before).

  Still thin (13) -- the app labels these so they aren't read as real matches:
    Adam Awender -> Dylan Logsdon (26% evidence) -- stats and size (no scouting notes for this player)
    Alex Ross -> Dylan Logsdon (26% evidence) -- stats and size (no scouting notes for this player)
    Isaiah Thompson -> Dylan Logsdon (26% evidence) -- stats and size (no scouting notes for this player)
    Robert Hutson -> Tony Mabon (26% evidence) -- stats and size (no scouting notes for this player)
    Jeffery Hillmer -> Reyce Allen (63% evidence) -- stats and size (no scouting notes for this player)
    Cullen Rauls -> Tony Mabon (26% evidence) -- stats and size (no scouting notes for this player)
    Zerrick Johnson -> Reyce Allen 

In [152]:
# --- OPTIONAL: re-derive the coach-note theme taxonomy with an LLM ----------------------------------------
# The app groups free-text clip notes into themes by PHRASE MATCHING against data/note_themes.json. That is
# deliberate: classifying a note costs nothing and needs no network, so every new note the parser writes is
# grouped the moment the app reads it, forever.
#
# This cell is the other half of that arrangement: the one place a model is allowed to touch the taxonomy.
# Run it when the staff's vocabulary has drifted far enough that notes are landing in "no theme" -- it reads
# the notes THIS parser just produced, asks the model to propose themes and the phrases that identify them,
# and rewrites note_themes.json. Nothing at runtime changes; the app still only does string matching.
#
# Guards, because a bad rewrite here silently degrades every theme in the app:
#   * USE_LLM must be True AND REBUILD_NOTE_THEMES must be True -- neither alone does anything.
#   * The existing file is backed up next to itself before being replaced.
#   * The model's output is validated (shape, theme count, phrase count) and REJECTED wholesale on anything
#     unexpected, leaving the current file untouched.
#   * Existing phrases are MERGED IN rather than replaced, so a hand-tuned phrase never disappears because
#     the model didn't think of it this time.
REBUILD_NOTE_THEMES = False   # set True (with USE_LLM=True) to re-derive themes from the current notes

# APP_DATA_DIR is assigned in the CSV-export cell further down; this cell runs before it, so resolve
# it here rather than depending on run order (both point at OUTPUT_DIR either way).
APP_DATA_DIR = globals().get("APP_DATA_DIR") or OUTPUT_DIR

NOTE_THEMES_PATH = os.path.join(APP_DATA_DIR, "note_themes.json")


def _strip_note_play_call(text):
    """Same play-call strip the app applies before classifying -- the play name is its own column, and
    leaving it in makes every note from one set look thematically identical."""
    t = re.sub(r"^[A-Z][A-Z0-9\-&' ]{1,24}?\s+EXECUTION\b[.,:=]*\s*", "", str(text).strip())
    t = re.sub(r"^[A-Z0-9\-' ]{2,20}=\s*", "", t.strip())
    return t.strip(" ,.=")


def rebuild_note_themes(notes_series, path=NOTE_THEMES_PATH, max_notes=400):
    """Ask the model for a theme taxonomy over these notes; write it only if it validates."""
    if not USE_LLM or not REBUILD_NOTE_THEMES:
        print("Note-theme rebuild skipped (needs USE_LLM=True and REBUILD_NOTE_THEMES=True). "
              "The app keeps using the existing data/note_themes.json -- classification is unaffected.")
        return None

    bodies = [_strip_note_play_call(n) for n in notes_series.dropna().astype(str)]
    bodies = [b for b in bodies if len(b) > 3][:max_notes]
    if len(bodies) < 20:
        print(f"Only {len(bodies)} usable note(s) -- too few to derive themes from. Leaving the file alone.")
        return None

    existing = {}
    if os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as fh:
                existing = {t["theme"]: t for t in json.load(fh).get("themes", [])}
        except Exception as read_error:
            print(f"  Could not read the existing theme file ({type(read_error).__name__}) -- treating it as empty.")

    prompt = (
        "You are organizing a college basketball staff's own video-clip notes into themes so a coach can "
        "group similar corrections together.\n\n"
        "Return 12-18 themes covering the notes below. For each theme give:\n"
        '  "theme": a short noun phrase a coach would recognize (e.g. "Switching & Screen Coverage")\n'
        '  "side": one of "Offense", "Defense", "Both"\n'
        '  "phrases": 10-30 lowercase words/phrases that IDENTIFY the theme in note text. Use the staff\'s '
        "own vocabulary and abbreviations exactly as written (e.g. \"trans. def\", \"oreb\", \"wall up\", "
        "\"x-out\"). Prefer distinctive multi-word phrases; avoid generic words like \"good\", \"the\", "
        "\"play\" that would match everything.\n\n"
        "These phrases are used for literal substring matching later -- no model runs at classification "
        "time -- so they must be words that actually appear in notes of that theme.\n\n"
        'Respond with ONLY a JSON object: {"themes": [{"theme": "...", "side": "...", "phrases": ["..."]}]}\n\n'
        "NOTES:\n" + "\n".join(f"- {b}" for b in bodies)
    )
    try:
        from openai import OpenAI

        client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL") or None)
        response = client.chat.completions.create(
            model=os.environ.get("AI_MODEL", "gpt-4o-mini"),
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
        )
        parsed = json.loads(response.choices[0].message.content)
    except Exception as llm_error:
        print(f"Theme rebuild FAILED ({type(llm_error).__name__}: {llm_error}) -- existing file left untouched.")
        return None

    themes = parsed.get("themes")
    if not isinstance(themes, list) or not (8 <= len(themes) <= 30):
        print(f"Rejected the model's response: expected 8-30 themes, got {len(themes) if isinstance(themes, list) else 'none'}. "
              "Existing file left untouched.")
        return None
    clean = []
    for t in themes:
        name = str(t.get("theme", "")).strip()
        phrases = [str(p).strip().lower() for p in t.get("phrases", []) if str(p).strip()]
        phrases = [p for p in phrases if 2 < len(p) <= 40]
        if not name or len(phrases) < 4:
            print(f"  Dropping theme {name!r}: {len(phrases)} usable phrase(s), need 4+.")
            continue
        side = str(t.get("side", "Both")).strip().title()
        if side not in ("Offense", "Defense", "Both"):
            side = "Both"
        # Merge, never replace: a phrase someone added by hand survives a rebuild.
        if name in existing:
            phrases = sorted(set(phrases) | set(existing[name].get("phrases", [])))
        clean.append({"theme": name, "side": side, "phrases": sorted(set(phrases))})
    if len(clean) < 8:
        print(f"Rejected: only {len(clean)} theme(s) survived validation. Existing file left untouched.")
        return None

    # Any theme the model dropped this run is KEPT -- notes already classified under it would otherwise
    # silently become unclassified.
    kept = [existing[n] for n in existing if n not in {c["theme"] for c in clean}]
    if kept:
        print(f"Keeping {len(kept)} existing theme(s) the model didn't propose this run: "
              f"{', '.join(t['theme'] for t in kept)}")
    clean.extend(kept)

    if os.path.exists(path):
        backup = path.replace(".json", f".backup-{datetime.now().strftime('%Y%m%d-%H%M%S')}.json")
        try:
            os.replace(path, backup)
            print(f"Backed up the previous taxonomy to {os.path.basename(backup)}")
        except Exception as backup_error:
            print(f"  Could not back up the existing file ({type(backup_error).__name__}) -- aborting rather than overwriting it.")
            return None

    payload = {
        "version": 1,
        "generated": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "notes": ("Lexical theme taxonomy for coach-note grouping. Runtime classification in the app is pure "
                  "string matching -- no model call, no tokens. Edit phrases here to retune."),
        "themes": clean,
    }
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(payload, fh, indent=2)
    print(f"Wrote {len(clean)} theme(s), {sum(len(t['phrases']) for t in clean)} phrase(s) to {path}")
    for t in clean:
        print(f"  {t['side']:<8} {t['theme']} ({len(t['phrases'])} phrases)")
    return payload


_note_theme_result = rebuild_note_themes(
    coach_notes["coach_note"] if "coach_note" in coach_notes.columns else pd.Series(dtype=object)
)


Note-theme rebuild skipped (needs USE_LLM=True and REBUILD_NOTE_THEMES=True). The app keeps using the existing data/note_themes.json -- classification is unaffected.


In [153]:
# --- UW-Whitewater player headshots from the team's own roster page ----------------------------------------
# The app shows a photo on each UWW player card, looked up by filename in data/uww_player_pictures/. Those
# were being placed there by hand. This reads the saved roster page (a browser "Save as Webpage, Single
# File" export of uwwsports.com's roster) that lives in INPUT_DIR alongside every other snapshot, and fills
# that folder automatically.
#
# Two sources per player, in order:
#   1. the full-size image on the athletics site (the page only links an 80px thumbnail, so the query
#      string is dropped to ask for the original), and
#   2. the thumbnail bytes EMBEDDED in the .mhtml itself -- which is why this still works with no network
#      at all, just at 80px.
UWW_ROSTER_FORCE_REFRESH = False   # True to re-fetch photos that are already on disk

# APP_DATA_DIR is assigned in the CSV-export cell further down; this cell runs before it, so resolve
# it here rather than depending on run order (both point at OUTPUT_DIR either way).
APP_DATA_DIR = globals().get("APP_DATA_DIR") or OUTPUT_DIR

_ROSTER_PIC_DIR = os.path.join(APP_DATA_DIR, "uww_player_pictures")
os.makedirs(_ROSTER_PIC_DIR, exist_ok=True)

_roster_paths = sorted(
    p for p in glob.glob(f"{INPUT_DIR}/*.mhtml") + glob.glob(f"{INPUT_DIR}/*.html")
    if re.search(r"roster", os.path.basename(p), re.IGNORECASE)
)
print(f"Found {len(_roster_paths)} roster snapshot(s):")
for _p in _roster_paths:
    print(" -", os.path.basename(_p))


def _mhtml_embedded_parts(path):
    """{Content-Location -> raw bytes} for every image part inside an MHTML archive. The archive keeps the
    images the browser had already loaded, so a lazy-loaded photo further down the page may be absent --
    hence the download attempt first."""
    if not path.lower().endswith(".mhtml"):
        return {}
    import email as _email
    from email import policy as _policy

    with open(path, "rb") as fh:
        msg = _email.message_from_bytes(fh.read(), policy=_policy.default)
    out = {}
    for part in msg.walk():
        if part.get_content_type().startswith("image/"):
            loc = part.get("Content-Location", "")
            if loc:
                out[loc] = (part.get_payload(decode=True) or b"", part.get_content_subtype())
    return out


def _roster_file_stem(name):
    """"Tyshawn Teague-Johnson" -> "uww_Tyshawn_Teague-Johnson", matching what the app's picture lookup
    expects (it strips the "uww_" prefix and splits the rest on underscores)."""
    cleaned = re.sub(r"[^A-Za-z0-9 '\-\.]", "", str(name)).strip()
    cleaned = re.sub(r"\s+", "_", cleaned).replace("'", "").replace(".", "")
    return f"uww_{cleaned}"


_roster_saved, _roster_skipped, _roster_failed = [], [], []
for _rpath in _roster_paths:
    try:
        _rhtml = load_html_snapshot(_rpath)
    except Exception as _rerr:
        print(f"  Could not read {os.path.basename(_rpath)}: {type(_rerr).__name__}: {_rerr}")
        continue
    if not _rhtml:
        continue
    _embedded = _mhtml_embedded_parts(_rpath)

    # Every roster tile renders <img alt="<Name> - View Profile">; that alt is the most reliable place the
    # player's name appears next to their photo (the surrounding markup changes with the site's template).
    _soup = BeautifulSoup(_rhtml, "html.parser")
    _players = []
    for _img in _soup.find_all("img"):
        _alt = str(_img.get("alt", ""))
        _m = re.match(r"^(.*?)\s*-\s*View Profile\s*$", _alt, re.IGNORECASE)
        if not _m:
            continue
        _name = _m.group(1).strip()
        _url = _img.get("data-src") or _img.get("src") or ""
        if _name and _url:
            _players.append((_name, _url))
    print(f"  {os.path.basename(_rpath)}: {len(_players)} player photo(s) referenced")

    for _name, _url in _players:
        _abs = _url if _url.startswith("http") else urljoin("https://uwwsports.com/", _url)
        _stem = _roster_file_stem(_name)
        _existing = glob.glob(os.path.join(_ROSTER_PIC_DIR, _stem + ".*"))
        if _existing and not UWW_ROSTER_FORCE_REFRESH:
            _roster_skipped.append(_name)
            continue

        _data, _ext = None, "jpg"
        # 1. full size from the site -- the page links "?width=80", so ask for the unresized original
        _full = _abs.split("?")[0]
        try:
            _req = urllib.request.Request(_full, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(_req, timeout=15) as _resp:
                _data = _resp.read()
                _ctype = _resp.headers.get("Content-Type", "")
                _ext = {"image/jpeg": "jpg", "image/png": "png", "image/webp": "webp"}.get(_ctype.split(";")[0], "jpg")
        except Exception as _derr:
            # 2. the thumbnail the archive already carries -- no network needed
            _hit = _embedded.get(_abs) or _embedded.get(_url)
            if _hit and _hit[0]:
                _data, _ext = _hit[0], _hit[1] or "webp"
                print(f"    {_name}: download failed ({type(_derr).__name__}) -- using the 80px copy embedded in the archive")
        if not _data:
            _roster_failed.append(_name)
            continue
        _dest = os.path.join(_ROSTER_PIC_DIR, f"{_stem}.{_ext}")
        try:
            with open(_dest, "wb") as _fh:
                _fh.write(_data)
            _roster_saved.append((_name, os.path.basename(_dest), len(_data)))
        except Exception as _werr:
            print(f"    {_name}: could not write {os.path.basename(_dest)} ({type(_werr).__name__})")
            _roster_failed.append(_name)

print(f"\nHeadshots: {len(_roster_saved)} saved, {len(_roster_skipped)} already on disk, {len(_roster_failed)} failed."
      f"  ->  {_ROSTER_PIC_DIR}")
for _n, _f, _b in _roster_saved:
    print(f"  {_n:<28} {_f:<40} {_b/1024:.0f} KB")
if _roster_failed:
    print("  Failed (no download and nothing embedded for them): " + ", ".join(_roster_failed))
if not _roster_paths:
    print("  No roster snapshot found. Save the team roster page as a single-file .mhtml into INPUT_DIR "
          "with 'Roster' in the filename, then re-run this cell.")


Found 6 roster snapshot(s):
 - 2025-26 Men's Basketball Roster - University of Wisconsin-Whitewater Athletics.mhtml
 - Aurora Spartans - Roster_2025.html
 - Eureka Red Devils - Roster_2025.html
 - Ripon Red Hawks - Roster_2025.html
 - St. Thomas (TX) Celts - Roster_2025.html
 - UW-Whitewater Warhawks - Roster_2025.html
  2025-26 Men's Basketball Roster - University of Wisconsin-Whitewater Athletics.mhtml: 21 player photo(s) referenced
  Aurora Spartans - Roster_2025.html: 0 player photo(s) referenced
  Eureka Red Devils - Roster_2025.html: 0 player photo(s) referenced
  Ripon Red Hawks - Roster_2025.html: 0 player photo(s) referenced
  St. Thomas (TX) Celts - Roster_2025.html: 0 player photo(s) referenced
  UW-Whitewater Warhawks - Roster_2025.html: 0 player photo(s) referenced

Headshots: 0 saved, 21 already on disk, 0 failed.  ->  ../data\uww_player_pictures



### Head-to-head: previous meetings with the upcoming opponent

Opponents recur — conference teams twice a season, most of them again the next year — so the brief and the app open with how the last meeting(s) actually went.

Sources, in order: **this season's schedule** (already parsed), then **local schedule snapshots for earlier seasons** (captures are already saved season-suffixed, so a previous year's file on disk needs no scraping), then a **live scrape of any earlier season with no local file** — FastScout's team URL takes a season code, and the result is saved season-suffixed so it's a once-per-season cost.

Exports `uww_head_to_head.csv`.

In [155]:
# --- Head-to-head: every previous meeting with the upcoming opponent, this season and earlier ---------------
# CONFIRMED CHANGE (requested): opponents recur -- conference teams twice a season, and most of them again the
# following year -- so the brief opens with how the last meeting(s) actually went. Three sources, in this order:
#   1. THIS season's schedule (uww_schedule, already parsed above): any earlier meeting that has been played.
#   2. Local schedule snapshots for OTHER seasons. Schedule captures are already saved per season (see
#      _add_season_suffix_to_path -- "UW-Whitewater - Schedule_2024.html"), so a previous year's file that is
#      already on disk is read straight off it, with no scraping at all.
#   3. A live scrape of previous seasons, only for the seasons with no local file. FastScout's own team URL
#      takes a season code (FASTSCOUT_DOCS_SEASON is "25" for 2025-26), so "24" is 2024-25; the scraped HTML is
#      saved season-suffixed, which means this is a once-per-season cost, not a once-per-run one.
# Every meeting found is exported to uww_head_to_head.csv for the brief and the app to render.
_H2H_UWW = "UW-Whitewater"
_H2H_SEASONS_BACK = 2          # how many earlier seasons to look for (2024-25 and 2023-24 when it's 2025-26)
_h2h_problems = []


def _h2h_schedule_from_html(html, source_label):
    """Every game on one of UWW's own schedule pages, unfiltered.

    build_team_schedule_from_html() above deliberately narrows UWW's own schedule to scouted opponents plus the
    upcoming game -- right for the scouting pipeline, wrong here, where the point is to find a meeting that was
    never scouted (a previous season's game, or an early-season one before scouting started). This repeats only
    the parsing half of that function, reusing its own helpers (split_opponent, split_result,
    extract_season_start_year, parse_schedule_date), and skips the filtering half."""
    page_soup = BeautifulSoup(html, "lxml")
    tables = page_soup.find_all("table")
    if not tables:
        raise ValueError(f"{source_label}: no schedule table on the page")
    raw = pd.read_html(StringIO(str(tables[0])))[0]
    # The box-score link per row -- the entry point scrape_pbp_live() needs to pull a previous meeting's
    # full play-by-play (see the pbp cell). Same extraction build_team_schedule_from_html does.
    game_urls = []
    for row_el in [r for r in tables[0].find_all("tr") if r.find_all("td")]:
        hrefs = [a["href"] for a in row_el.find_all("a", href=True)]
        box = [h for h in hrefs if "/teams/" in h and "/boxscore" in h]
        game_urls.append(_resolve_team_link(box[0]) if box else None)
    raw["game_url"] = game_urls[:len(raw)] + [None] * max(0, len(raw) - len(game_urls))
    team_name_raw = page_soup.find("h1").get_text(strip=True)
    team_name = re.match(r"^([^\d]+)", team_name_raw).group(1).strip()
    out = pd.DataFrame()
    out["date"] = raw["Date"]
    out["opponent"] = raw["Opponent"].apply(lambda v: split_opponent(v)[0])
    out["team"] = team_name
    out["location"] = raw["Location"]
    res = raw["Result"].apply(split_result)
    out["outcome"] = res.apply(lambda x: x[0])
    out["team_score"] = res.apply(lambda x: x[1])
    out["opponent_score"] = res.apply(lambda x: x[2])
    out["game_url"] = raw["game_url"]
    year = extract_season_start_year(page_soup, fallback=_DEFAULT_SEASON_START_YEAR)
    out["season"] = f"{year}-{str(year + 1)[-2:]}"
    out["season_start_year"] = year
    out["_parsed_date"] = out["date"].apply(lambda d: parse_schedule_date(d, year))
    return out[out["outcome"].astype(str).str.upper().isin(["W", "L"])]


def _h2h_same_team(a, b):
    """Do two schedule spellings mean the same program? Mascots and abbreviations drift between seasons
    ("Aurora Spartans" vs "Aurora University" vs "Aurora"), so this compares the distinctive words rather than
    the whole string."""
    # CONFIRMED BUG (fixed here): matching on ANY shared word paired "Eureka Red Devils" with "Ripon Red
    # Hawks" -- both are "Red" -- and would have put another team's game in the head-to-head table. Mascot and
    # colour words are ignored, so only the school's own name can make a match.
    _MASCOT = {"red", "blue", "green", "black", "white", "gold", "golden", "purple", "crimson", "scarlet",
               "fighting", "flying", "big", "little", "hawks", "devils", "eagles", "spartans", "warriors",
               "warhawks", "tigers", "lions", "bears", "wolves", "panthers", "cardinals", "knights", "titans",
               "pioneers", "raiders", "falcons", "vikings", "bulldogs", "wildcats", "cougars", "yellowjackets",
               "blugolds", "pointers", "gusties", "celts", "foresters"}
    stop = {"university", "college", "the", "of", "state", "saint", "st"} | _MASCOT
    words = lambda t: {w for w in re.findall(r"[a-z]+", str(t).lower()) if len(w) > 2 and w not in stop}  # noqa: E731
    wa, wb = words(a), words(b)
    return bool(wa & wb) if wa and wb else False


def _h2h_local_schedule_files():
    """UWW's own schedule snapshots on disk, newest season first, one file per season."""
    hits = {}
    for path in sorted(glob.glob(f"{schedules_dir}/*.html") + glob.glob(f"{schedules_dir}/*.mhtml")):
        base = os.path.basename(path)
        if "schedule" not in base.lower() or "whitewater" not in base.lower():
            continue
        year = re.search(r"_(\d{4})\.[^.]+$", base)
        hits[int(year.group(1)) if year else _DEFAULT_SEASON_START_YEAR] = path
    return hits


def _h2h_scrape_season(page, season_code, year, timeout_ms=30000):
    """One earlier season's schedule, via FastScout's own season parameter on the team URL."""
    url = f"{FASTSCOUT_ORIGIN}/teams/myTeam?league={FASTSCOUT_DOCS_LEAGUE}&season={season_code}"
    print(f"    [scrape] {year}-{str(year + 1)[-2:]} schedule: {url}")
    _goto_with_auth_retry(page, url, "text=SCHEDULE", timeout_ms)
    _click_tab_by_text(page, "SCHEDULE", timeout_ms)
    page.wait_for_selector("#myTeamSchedule", timeout=timeout_ms)
    page.wait_for_selector("#myTeamSchedule tr", timeout=timeout_ms)
    html = page.content()
    _save_scraped_html(html, os.path.join(schedules_dir, f"UW-Whitewater - Schedule_{year}.html"),
                       f"UW-Whitewater's {year}-{str(year + 1)[-2:]} schedule")
    return html


_h2h_rows = []
_h2h_current_year = None
try:
    _h2h_current_year = int(str(uww_team_schedule["season"].dropna().iloc[0]).split("-")[0])
except Exception:
    _h2h_current_year = _DEFAULT_SEASON_START_YEAR

# ---- 1. this season's earlier meetings ---------------------------------------------------------------
if upcoming_opponent_short and not uww_team_schedule.empty:
    _played = uww_team_schedule[uww_team_schedule["outcome"].astype(str).str.upper().isin(["W", "L"])]
    for _, g in _played.iterrows():
        if _h2h_same_team(g.get("opponent"), upcoming_opponent_short):
            _h2h_rows.append({"season": g.get("season") or f"{_h2h_current_year}-{str(_h2h_current_year + 1)[-2:]}",
                              "date": g.get("date"), "location": g.get("location"), "outcome": g.get("outcome"),
                              "team_score": g.get("team_score"), "opponent_score": g.get("opponent_score"),
                              "opponent_as_listed": g.get("opponent"), "game_url": g.get("game_url"),
                              "source": "this season's schedule"})

# ---- 2 and 3. earlier seasons: local files first, scrape only what's missing -------------------------
_h2h_local = _h2h_local_schedule_files()
_h2h_want = [_h2h_current_year - n for n in range(1, _H2H_SEASONS_BACK + 1)]
_h2h_html_by_year = {y: _h2h_local[y] for y in _h2h_want if y in _h2h_local}
_h2h_missing = [y for y in _h2h_want if y not in _h2h_html_by_year]

if _h2h_missing and fastscout_username and fastscout_password:
    def _run_h2h_scrape(login_page):
        found = {}
        for _y in _h2h_missing:
            _code = str(int(FASTSCOUT_DOCS_SEASON) - (_h2h_current_year - _y))
            try:
                found[_y] = _h2h_scrape_season(login_page, _code, _y)
            except Exception as e:
                _h2h_problems.append(f"{_y}-{str(_y + 1)[-2:]} schedule: {type(e).__name__}: {e}")
        return found

    try:
        for _y, _html in run_in_fastscout_session(_run_h2h_scrape).items():
            _h2h_html_by_year[_y] = _html
    except Exception as _e:
        _h2h_problems.append(f"could not open a FastScout session for earlier seasons: {type(_e).__name__}: {_e}")
elif _h2h_missing:
    _h2h_problems.append(f"no local schedule file for {_h2h_missing} and no FastScout credentials -- earlier "
                         f"seasons skipped.")

for _y, _src in sorted(_h2h_html_by_year.items(), reverse=True):
    try:
        _html = load_html_snapshot(_src) if isinstance(_src, str) and os.path.exists(str(_src)) else _src
        _sched = _h2h_schedule_from_html(_html, f"{_y}-{str(_y + 1)[-2:]} schedule")
    except Exception as e:
        _h2h_problems.append(f"{_y}-{str(_y + 1)[-2:]} schedule could not be parsed: {type(e).__name__}: {e}")
        continue
    for _, g in _sched.iterrows():
        if _h2h_same_team(g.get("opponent"), upcoming_opponent_short):
            _h2h_rows.append({"season": g.get("season"), "date": g.get("date"), "location": g.get("location"),
                              "outcome": g.get("outcome"), "team_score": g.get("team_score"),
                              "opponent_score": g.get("opponent_score"),
                              "opponent_as_listed": g.get("opponent"), "game_url": g.get("game_url"),
                              "source": f"{_y}-{str(_y + 1)[-2:]} schedule"})

head_to_head = pd.DataFrame(_h2h_rows, columns=["season", "date", "location", "outcome", "team_score",
                                                "opponent_score", "opponent_as_listed", "game_url", "source"])
if not head_to_head.empty:
    head_to_head = head_to_head.drop_duplicates(subset=["season", "date"])
    head_to_head["team_score"] = pd.to_numeric(head_to_head["team_score"], errors="coerce")
    head_to_head["opponent_score"] = pd.to_numeric(head_to_head["opponent_score"], errors="coerce")
    head_to_head["margin"] = head_to_head["team_score"] - head_to_head["opponent_score"]
    head_to_head["home_away"] = head_to_head["location"].astype(str).str.strip().str.lower().map(
        {"home": "Home", "away": "Away", "neutral": "Neutral"}).fillna(head_to_head["location"])

    # A real calendar date for each meeting. "date" is the schedule's display string ("Wed, Nov 20") with no
    # year, so nothing could be JOINED to it -- in particular the tagged play calls, which the brief now reads
    # per meeting (what we ran, what defense, did it work). Resolved once here with the season's start year.
    def _h2h_iso(row):
        try:
            _p = parse_schedule_date(row["date"], int(str(row["season"]).split("-")[0]))
        except Exception:
            return None
        _p = pd.Timestamp(_p) if _p is not None else None
        return None if _p is None or pd.isna(_p) else _p.strftime("%Y-%m-%d")
    head_to_head["game_date"] = head_to_head.apply(_h2h_iso, axis=1)

    # Who led each side in a meeting we have a box score for. Only games this pipeline parsed play-by-play
    # for will have one -- a previous season's game usually won't, and that column is simply left blank
    # rather than filled with something from the wrong game.
    # CONFIRMED BUG (fixed here): this referenced pbp_box_score directly and raised NameError -- the cell had
    # been placed right after the upcoming opponent is identified, which is long before any box score is built.
    # The cell now sits after the box-score cells (see the notebook order), and this reads the table through
    # globals() anyway, so running it early degrades to "no leading scorers" instead of crashing the run.
    _h2h_box = globals().get("pbp_box_score")
    if _h2h_box is None:
        _h2h_box = pd.DataFrame()

    def _h2h_leaders(row):
        if _h2h_box.empty or "game_date" not in _h2h_box.columns:
            return pd.Series({"uww_leader": None, "opp_leader": None})
        # CONFIRMED BUG (fixed here): this called .date() on parse_schedule_date()'s return value, which is a
        # datetime.date already (not a datetime), so it raised AttributeError. Normalised through
        # pd.Timestamp so either kind of return value compares correctly.
        parsed = parse_schedule_date(row["date"], int(str(row["season"]).split("-")[0]))
        parsed = pd.Timestamp(parsed) if parsed is not None else None
        if parsed is None or pd.isna(parsed):
            return pd.Series({"uww_leader": None, "opp_leader": None})
        game = _h2h_box[pd.to_datetime(_h2h_box["game_date"], errors="coerce").dt.date == parsed.date()]
        game = game[game["player"].astype(str) != "TEAM"]
        if game.empty:
            return pd.Series({"uww_leader": None, "opp_leader": None})
        out = {}
        for key, mask in (("uww_leader", game["team"].astype(str).str.contains("Whitewater", case=False, na=False)),
                          ("opp_leader", ~game["team"].astype(str).str.contains("Whitewater", case=False, na=False))):
            side = game[mask]
            if side.empty:
                out[key] = None
                continue
            top = side.loc[pd.to_numeric(side["PTS"], errors="coerce").idxmax()]
            out[key] = f"{top['player']} {int(pd.to_numeric(top['PTS'], errors='coerce'))}"
        return pd.Series(out)

    head_to_head = pd.concat([head_to_head, head_to_head.apply(_h2h_leaders, axis=1)], axis=1)
    head_to_head = head_to_head.sort_values(["season", "date"], ascending=[False, False]).reset_index(drop=True)

head_to_head.to_csv(os.path.join(APP_DATA_DIR, "uww_head_to_head.csv"), index=False)

# ==========================================================================================================
# The previous meeting in full, plus how that opponent's roster has changed since (requested)
# ==========================================================================================================
# Each meeting's own play-by-play is pulled from its box-score link the same way every other game in this
# notebook is (scrape_pbp_live -> build_pbp_events -> box_score_from_pbp_events), cached to disk so a season
# is scraped once. Whatever can't be fetched degrades to the scoreline already shown above.
_h2h_box_rows = []
_h2h_meetings_parsed = []
if not head_to_head.empty:
    _h2h_want_pbp = head_to_head[head_to_head["game_url"].notna()]

    def _h2h_pbp_path(row):
        _safe = re.sub(r"[^\w]+", "_", f"{row['season']} {row['date']}").strip("_")
        return os.path.join(schedules_dir, f"{upcoming_opponent_short} - PBP {_safe}.html")

    def _h2h_parse_meeting(row, html_or_raw):
        """One meeting -> player box-score lines for both sides."""
        raw = parse_pbp_html(html_or_raw) if isinstance(html_or_raw, str) else html_or_raw
        _date = pd.Timestamp(parse_schedule_date(row["date"], int(str(row["season"]).split("-")[0])))
        events = build_pbp_events(raw, row["opponent_as_listed"], _date.date(), self_team=_H2H_UWW)
        box = box_score_from_pbp_events(events, label=f"{row['season']} meeting", verbose=False)
        if box.empty:
            return []
        box = box[box["player"].astype(str) != "TEAM"].copy()
        box["season"] = row["season"]
        box["meeting_date"] = row["date"]
        return box.to_dict("records")

    _h2h_need_scrape = []
    for _, _row in _h2h_want_pbp.iterrows():
        _path = _h2h_pbp_path(_row)
        if os.path.exists(_path):
            try:
                _h2h_box_rows += _h2h_parse_meeting(_row, load_html_snapshot(_path))
                _h2h_meetings_parsed.append(f"{_row['season']} {_row['date']} (cached)")
            except Exception as e:
                _h2h_problems.append(f"{_row['season']} {_row['date']} play-by-play could not be parsed: "
                                     f"{type(e).__name__}: {e}")
        else:
            _h2h_need_scrape.append(_row)

    if _h2h_need_scrape and fastscout_username and fastscout_password:
        def _run_h2h_pbp(login_page):
            got = {}
            for _row in _h2h_need_scrape:
                try:
                    got[(_row["season"], _row["date"])] = scrape_pbp_live(
                        login_page, _row["game_url"], save_path=_h2h_pbp_path(_row))
                except Exception as e:
                    _h2h_problems.append(f"{_row['season']} {_row['date']} play-by-play: {type(e).__name__}: {e}")
            return got

        try:
            for _key, _raw in run_in_fastscout_session(_run_h2h_pbp).items():
                _match = [r for _, r in _h2h_want_pbp.iterrows() if (r["season"], r["date"]) == _key]
                if _match:
                    try:
                        _h2h_box_rows += _h2h_parse_meeting(_match[0], _raw)
                        _h2h_meetings_parsed.append(f"{_key[0]} {_key[1]} (scraped)")
                    except Exception as e:
                        _h2h_problems.append(f"{_key[0]} {_key[1]}: {type(e).__name__}: {e}")
        except Exception as e:
            _h2h_problems.append(f"could not open a FastScout session for meeting play-by-play: "
                                 f"{type(e).__name__}: {e}")
    elif _h2h_need_scrape:
        _h2h_problems.append(f"{len(_h2h_need_scrape)} meeting(s) have no cached play-by-play and no FastScout "
                             f"credentials -- the brief shows their scoreline only.")

head_to_head_box = pd.DataFrame(_h2h_box_rows)
head_to_head_box.to_csv(os.path.join(APP_DATA_DIR, "uww_head_to_head_box.csv"), index=False)

# ---- how their roster has changed since that meeting -------------------------------------------------
# Who played in the previous meeting, and whether they are still on this year's team: returning players are
# matched against this season's roster and box score, so "returning" means genuinely available now, not just
# listed. Production is stated in that game's own terms (what they did TO US), which is the number a coach
# actually wants -- "their leading scorer against us last year is gone" is the useful sentence.
_h2h_change_rows = []
if not head_to_head_box.empty and upcoming_opponent_short:
    _prev = head_to_head_box[head_to_head_box["team"].astype(str).apply(
        lambda t: _h2h_same_team(t, upcoming_opponent_short))]
    # This year's squad: the live roster page and this season's box scores, keyed by a normalised name so a
    # spelling difference between the two sources doesn't read as a different player.
    _cur_by_key = {}
    for _src in (globals().get("live_rosters"), globals().get("pbp_box_score_upcoming")):
        if _src is None or getattr(_src, "empty", True):
            continue
        _col = "name" if "name" in _src.columns else "player"
        _rows = _src
        if "team" in _src.columns:
            _rows = _src[_src["team"].astype(str).apply(lambda t: _h2h_same_team(t, upcoming_opponent_short))]
        for _n in _rows[_col].dropna():
            if str(_n).strip().upper() == "TEAM":
                continue
            _cur_by_key.setdefault(re.sub(r"\s+", " ", str(_n)).strip().lower(), str(_n).strip())
    _cur_names = set(_cur_by_key)

    _prev_totals = (_prev.groupby("player")[["PTS", "REB", "AST"]].sum().reset_index()
                    if not _prev.empty else pd.DataFrame())
    for _, r in _prev_totals.sort_values("PTS", ascending=False).iterrows():
        _key = re.sub(r"\s+", " ", str(r["player"])).strip().lower()
        _h2h_change_rows.append({"opponent": upcoming_opponent_short, "player": r["player"],
                                 "status": "Returning" if _key in _cur_names else "Gone",
                                 "prev_pts": r["PTS"], "prev_reb": r["REB"], "prev_ast": r["AST"]})
    _played_then = {re.sub(r"\s+", " ", str(p)).strip().lower() for p in _prev_totals.get("player", [])}
    for _name in sorted(_cur_names - _played_then):
        _h2h_change_rows.append({"opponent": upcoming_opponent_short, "player": _cur_by_key.get(_name, _name.title()),
                                 "status": "New since then",
                                 "prev_pts": None, "prev_reb": None, "prev_ast": None})

head_to_head_roster_change = pd.DataFrame(_h2h_change_rows,
                                          columns=["opponent", "player", "status", "prev_pts", "prev_reb", "prev_ast"])
head_to_head_roster_change.to_csv(os.path.join(APP_DATA_DIR, "uww_head_to_head_roster_change.csv"), index=False)
if _h2h_meetings_parsed:
    print(f"  Meeting play-by-play parsed: {', '.join(_h2h_meetings_parsed)}.")
    if not head_to_head_roster_change.empty:
        _ret = head_to_head_roster_change[head_to_head_roster_change["status"] == "Returning"]
        _gone = head_to_head_roster_change[head_to_head_roster_change["status"] == "Gone"]
        print(f"  Roster continuity: {len(_ret)} of {len(_ret) + len(_gone)} players who faced us are back; "
              f"{_gone['prev_pts'].sum():.0f} of their points against us are gone.")

if head_to_head.empty:
    print(f"No previous meetings with {upcoming_opponent_short or 'the upcoming opponent'} found "
          f"(this season or the last {_H2H_SEASONS_BACK}).")
else:
    _w = int((head_to_head["outcome"].astype(str).str.upper() == "W").sum())
    print(f"Head-to-head vs {upcoming_opponent_short}: {len(head_to_head)} previous meeting(s), "
          f"{_w}-{len(head_to_head) - _w}.")
    print(head_to_head[["season", "date", "home_away", "outcome", "team_score", "opponent_score"]].to_string(index=False))
if _h2h_problems:
    print("Head-to-head problems:")
    for _p in _h2h_problems:
        print(f"  - {_p}")


  Meeting play-by-play parsed: 2024-25 Wed, Nov 20 (cached).
  Roster continuity: 3 of 8 players who faced us are back; 55 of their points against us are gone.
Head-to-head vs Aurora Spartans: 1 previous meeting(s), 1-0.
 season        date home_away outcome  team_score  opponent_score
2024-25 Wed, Nov 20      Away       W          75              72


In [156]:
# --- Export the same tables as CSV files directly into the Streamlit app's source directory, so the app can
# read them as static bundled data (pandas.read_csv) instead of querying a SQL warehouse / Unity Catalog at
# runtime. (The Databricks-only Delta-table export used inside the workspace is skipped in this portable
# version -- there's no Spark session outside Databricks, and this CSV export is the app's actual data source.)
APP_DATA_DIR = OUTPUT_DIR
os.makedirs(APP_DATA_DIR, exist_ok=True)

# opponent_schedules is derived straight from the combined "schedule" frame (which already loops over every
# team's MHTML in schedules_dir -- UWW's own plus any opponent's -- and tags each row with "team").
opponent_schedules = (
    schedule[schedule["team"] != "UW-Whitewater Warhawks"]
    .rename(columns={"team": "opponent", "date": "game_date", "opponent": "vs_opponent"})
    [["opponent", "game_date", "vs_opponent", "location", "outcome", "team_score", "opponent_score", "point_margin"]]
    .reset_index(drop=True)
)

# uww_pbp_events is trimmed to the columns the app actually renders -- the full ~25-column table is unnecessarily
# large to bundle as a static CSV inside the app package.
PBP_EVENTS_EXPORT_COLS = [
    "opponent", "game_date", "event_order", "period", "time_remaining",
    "team", "player", "event_type", "raw_text", "shot_type", "uww_score", "opp_score", "uww_lineup",
    "video_description", "coach_note", "play_call",
    # Decoded play-call fields from uww_plays.csv (see the "Play calls" cell).
    "play_series", "play_situation", "play_actions", "primary_action", "play_location", "finish_spot", "play_title",
]
# shot_type/uww_lineup were previously only exported on the clutch-events slice (CLUTCH_EVENTS_EXPORT_COLS
# below) even though both are already computed on every row of the full pbp_events DataFrame, not just
# clutch-time ones -- added here so the app can answer questions that need which 5-man UWW lineup was on the
# floor for a given shot on ANY possession, not just crunch-time ones (e.g. "which lineup gets our best shot
# type most often").

# uww_clutch_events / uww_scoring_runs -- both already computed earlier in this notebook (the "Clutch-time
# event log" and "Scoring runs and largest lead/deficit" cells) but were previously never added to csv_tables,
# so they never reached the app despite the analysis already existing. clutch_events is a filtered slice of
# pbp_events (same shape), so it needs the same column trim plus shot_type/uww_lineup/opp_lineup so the app can
# show point values and which 5-man units were on the floor during clutch stretches.
CLUTCH_EVENTS_EXPORT_COLS = [
    "opponent", "game_date", "event_order", "period", "time_remaining", "team", "player", "event_type",
    "raw_text", "shot_type", "uww_score", "opp_score", "uww_lineup", "opp_lineup", "video_description",
]
clutch_events_export = clutch_events[[c for c in CLUTCH_EVENTS_EXPORT_COLS if c in clutch_events.columns]]

# opponent column = whoever the upcoming opponent ACTUALLY played in that game (not always Whitewater --
# these are their games BEFORE facing Whitewater), team column = which side that specific row's event
# belongs to (the upcoming opponent's own play, or their opponent-in-that-game's play). Filtering
# team != upcoming_opponent_short gives exactly "what other teams did against this opponent's defense" --
# previously computed inline for print/diagnostic output only (see the "opponent_events" split a few cells
# up) and never exported, so the app had no way to use it at all.
OPPONENT_PRIOR_PBP_EXPORT_COLS = [
    "opponent", "game_date", "event_order", "period", "time_remaining", "time_remaining_seconds",
    "team", "player", "event_type", "raw_text", "shot_type", "video_description",
    # Decoded play-call fields from opponent_plays.csv.
    "play_call", "play_series", "play_situation", "play_actions", "primary_action", "play_location", "finish_spot",
    "play_title",
    # The defense the offense faced on that clip, from the coaches' updated Title tagging (decode_defense_tag
    # in the play-calls cell). These were decoded and carried on the events but never made it into this export
    # list, so the opponent's prior-game pbp silently dropped every defense read -- the app had no way to see
    # defense-by-event for those games even though the data existed upstream.
    "defense_type", "defense_press", "defense_press_formation", "defense_coverage",
]
# NOTE: no per-event lineup columns here (unlike PBP_EVENTS_EXPORT_COLS' "uww_lineup" for UWW's own games) --
# self_lineup/their_lineup only exist on pbp_up, a local copy made a couple cells up for the season-lineup-box
# computation, never attached back onto pbp_events_upcoming itself. Nothing currently needs them at the
# per-event level for the opponent's prior games (uww_opp_lineup_season_box already covers the season-
# aggregate use case, and the minutes-played cell re-derives what it needs from stint_src directly).

csv_tables = {
    "uww_schedule": schedule,
    "uww_season_stats": stats,
    "uww_pbp_events": pbp_events[PBP_EVENTS_EXPORT_COLS],
    "uww_pbp_box_score": pbp_box_score,
    "uww_lineup_stints": lineup_stints,
    "uww_coaching_flags": coaching_flags_df,
    "uww_opponent_rosters": all_rosters,
    "uww_player_profiles": player_profiles,
    "uww_opponent_game_plans": all_game_plans,
    "uww_ktv_splits": splits,
    "uww_ktv_game_categories": game_categories,
    "uww_pbp_derived_keys": pbp_derived_keys,
    "uww_opponent_team_totals": team_totals,
    "uww_projected_box_score": projected_uww_box,
    "uww_opponent_projected_box_score": projected_opponent_box,
    "uww_player_comparisons": best_matches,
    "uww_opp_lineup_season_box": upcoming_lineup_season,
    "uww_opponent_schedules": opponent_schedules,
    "uww_clutch_events": clutch_events_export,
    "uww_scoring_runs": scoring_runs,
    "uww_coach_notes": coach_notes,
    "uww_live_rosters": live_rosters,
    "uww_head_to_head": head_to_head,
    "uww_head_to_head_box": head_to_head_box,
    "uww_head_to_head_roster_change": head_to_head_roster_change,  # previous meetings with the upcoming opponent  # photo/number/position/height/class year only -- see the roster cell above
    # Every tagged clip from uww_plays.csv / opponent_plays.csv, decoded, plus the per-set summaries and the
    # shorthand glossary the brief and the app print.
    "uww_play_calls": play_calls,
    "uww_play_call_summary": play_call_summary,
    "uww_lineup_grouping": lineup_grouping,  # lineup -> grouping label, shared by the brief and the app
    "uww_play_glossary": PLAY_GLOSSARY,
    "uww_plays_catalog": plays_catalog,
    "uww_opponent_prior_games_pbp": pbp_events_upcoming[[c for c in OPPONENT_PRIOR_PBP_EXPORT_COLS if c in pbp_events_upcoming.columns]],
    "uww_opponent_prior_games_box_score": pbp_box_score_upcoming,
    "uww_opponent_prior_games_lineup_stints": lineup_stints_upcoming,
}

# Tables that ACCUMULATE across runs instead of being overwritten. Everything else is a full rebuild of
# current state and must be replaced wholesale; these two are a record of what was predicted before a
# specific game, which is only useful if past rows survive. Rows are keyed by (opponent, game_date) and the
# newest run wins for a key that already exists -- so re-running the parser for the same upcoming game
# corrects that game's projection rather than duplicating it.
APPEND_TABLES = {
    "uww_projected_box_score": ["opponent", "game_date"],
    "uww_opponent_projected_box_score": ["opponent", "game_date"],
}

csv_export_status = []
for name, df in csv_tables.items():
    path = os.path.join(APP_DATA_DIR, f"{name}.csv")
    keys = APPEND_TABLES.get(name)
    if keys and os.path.exists(path) and all(k in df.columns for k in keys):
        prior = pd.read_csv(path)
        if all(k in prior.columns for k in keys):
            # Drop this run's (opponent, game_date) from the archive, then append the fresh rows.
            this_run = set(map(tuple, df[keys].astype(str).drop_duplicates().to_numpy()))
            prior_keys = list(map(tuple, prior[keys].astype(str).to_numpy()))
            prior = prior[[k not in this_run for k in prior_keys]]
            df = pd.concat([prior, df], ignore_index=True)
        else:
            print(f"  {name}: existing CSV predates the (opponent, game_date) stamp -- replacing it. "
                  f"Projections made before this run aren't recoverable.")
    df.to_csv(path, index=False)
    csv_export_status.append((name, len(df), os.path.getsize(path)))

print(pd.DataFrame(csv_export_status, columns=["table", "rows", "csv_bytes"]))

                                     table  rows  csv_bytes
0                             uww_schedule    21      10043
1                         uww_season_stats    25       2707
2                           uww_pbp_events  1422     362233
3                        uww_pbp_box_score    91      10197
4                        uww_lineup_stints   128      28906
5                       uww_coaching_flags    30      10449
6                     uww_opponent_rosters    36       3700
7                      uww_player_profiles    50       9693
8                  uww_opponent_game_plans    41       7907
9                           uww_ktv_splits     5        358
10                 uww_ktv_game_categories     7        521
11                    uww_pbp_derived_keys     3       1049
12                uww_opponent_team_totals     4        128
13                 uww_projected_box_score    46      12933
14        uww_opponent_projected_box_score    26       4886
15                  uww_player_compariso

C:\Users\frits\AppData\Local\Temp\ipykernel_27492\953423779.py:126: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([prior, df], ignore_index=True)



### Extract and save team logos from PBP MHTML files

Pulls each team's logo image out of the play-by-play MHTML snapshots and saves it into the app's `data/logo` directory, for the Streamlit app's UI.

In [158]:
# --- Extract team logos embedded in the PBP/schedule MHTML files and save them into the Streamlit app's
# data/logo/ directory. Each MHTML archive (Chrome "Save as Webpage, Single File") bundles referenced images
# as binary MIME parts. Team logos appear under two URL patterns:
#   1. https://download.fastmodeltechnologies.com/FSimages/logos/NCAAB-III/<Team>.png
#   2. https://stats-assets.fastmodelsports.com/images/teams/<Team>  (alternate, e.g. Simpson)
# The filename becomes "<Team>.png" -- matching the short_opponent names used elsewhere in the app.
from urllib.parse import unquote as _logo_unquote

_LOGO_DIR = os.path.join(APP_DATA_DIR, "logo")
os.makedirs(_LOGO_DIR, exist_ok=True)

_LOGO_URL_PATTERN_1 = "https://download.fastmodeltechnologies.com/FSimages/logos/NCAAB-III/"
_LOGO_URL_PATTERN_2 = "https://stats-assets.fastmodelsports.com/images/teams/"

_logo_mhtml_files = sorted(glob.glob(f"{volume_dir}/*.mhtml"))

_logos_saved = {}
for _mf in _logo_mhtml_files:
    with open(_mf, "rb") as _f:
        _raw = _f.read()
    _msg = email.message_from_bytes(_raw, policy=policy.default)
    for _part in _msg.walk():
        _loc = _part.get("Content-Location", "")
        _ct = _part.get_content_type()
        if not _ct.startswith("image/"):
            continue
        _team_name = None
        if _LOGO_URL_PATTERN_1 in _loc:
            _team_name = _logo_unquote(_loc.replace(_LOGO_URL_PATTERN_1, "").replace(".png", ""))
        elif _LOGO_URL_PATTERN_2 in _loc:
            _team_name = _logo_unquote(_loc.split("/")[-1])
        if _team_name and _team_name not in _logos_saved:
            _payload = _part.get_payload(decode=True)
            if _payload and len(_payload) > 100:
                _filepath = os.path.join(_LOGO_DIR, f"{_team_name}.png")
                with open(_filepath, "wb") as _out:
                    _out.write(_payload)
                _logos_saved[_team_name] = len(_payload)

print(f"Saved {len(_logos_saved)} team logos to {_LOGO_DIR}:")
for _name in sorted(_logos_saved):
    print(f"  - {_name} ({_logos_saved[_name]:,} bytes)")

Saved 21 team logos to ../data\logo:
  - Alma (226,793 bytes)
  - Aurora (136,971 bytes)
  - Carroll (WI) (109,313 bytes)
  - Coe (115,946 bytes)
  - Elmhurst (118,533 bytes)
  - Eureka (150,170 bytes)
  - Hope (66,665 bytes)
  - Lawrence (84,599 bytes)
  - Loras (95,745 bytes)
  - Ripon (87,371 bytes)
  - Simpson (76,108 bytes)
  - St. Thomas (TX) (188,379 bytes)
  - UW-Eau Claire (70,774 bytes)
  - UW-La Crosse (26,203 bytes)
  - UW-Oshkosh (166,949 bytes)
  - UW-Platteville (71,166 bytes)
  - UW-River Falls (89,021 bytes)
  - UW-Stevens Point (139,555 bytes)
  - UW-Stout (194,873 bytes)
  - UW-Whitewater (145,562 bytes)
  - Washington-St. Louis (87,899 bytes)



### Extract player headshots from scouting report PDFs

Pulls each scouted player's headshot image out of their FastScout scouting-report PDF, for the Streamlit app's player-comparison UI.

In [160]:
# --- Extract player headshot images from scouting reports (PDF or HTML) and save them to the Streamlit
# app's data/player_images/ directory.
# For PDFs: identify headshots by eliminating repeated header images, keeping portrait-oriented images on
#   roster pages, and matching to player names by vertical position order.
# For HTMLs: find <img> elements within playerGroup tiles and download from their URLs, matching to the
#   player name parsed from the same tile's player-info-line spans.
import re as _pi_re
import urllib.request
from urllib.parse import urlparse, quote, urlunparse
from collections import Counter as _PICounter


try:
    import fitz  # pymupdf -- only needed for PDF reports
except ModuleNotFoundError:
    fitz = None  # no PDFs to process if pymupdf isn't installed


_PI_OUTPUT_DIR = os.path.join(APP_DATA_DIR, "player_images")
os.makedirs(_PI_OUTPUT_DIR, exist_ok=True)


_PI_PLAYER_PATTERN = _pi_re.compile(r"#\d+\s*[\u2022\u00b7]\s*(.+?)\s*[\u2022\u00b7]\s*[GCFPG/]+\s*[\u2022\u00b7]")




# --- HTML headshot extraction: find <img> in each playerGroup tile, download from URL ---
def _extract_headshots_from_html(html_path, opponent_name, output_dir):
    """Extract player headshots from an HTML scout report by finding <img> elements in playerGroup tiles."""
    with open(html_path, "r", encoding="utf-8") as f:
        html = f.read()
    soup = BeautifulSoup(html, "html.parser")
    printable = soup.find(class_="PrintableNode")
    if printable is None:
        return {}, {}, []


    extracted = {}
    skipped = {}
    _pending_downloads = []  # URLs that need authenticated download via Playwright


    # Diagnostic: count playerGroup tiles found
    _pg_tiles = [t for t in printable.find_all("div") if "Tile" in (t.get("class") or []) and "playerGroup" in (t.get("class") or [])]
    print(f"    [DIAG] Found {len(_pg_tiles)} playerGroup tiles in PrintableNode")
    if not _pg_tiles:
        # Try alternate: find any div with 'player' in class name
        _alt_tiles = [t for t in printable.find_all("div") if any("player" in c.lower() for c in (t.get("class") or []))]
        print(f"    [DIAG] Alternate: {len(_alt_tiles)} divs with 'player' in class")
        if _alt_tiles:
            print(f"    [DIAG] First alt classes: {_alt_tiles[0].get('class')}")
        # Also show all unique Tile classes
        _all_tiles = [t for t in printable.find_all("div") if "Tile" in (t.get("class") or [])]
        _tile_class_sets = set()
        for _t in _all_tiles[:20]:
            _tile_class_sets.add(tuple(sorted(_t.get("class", []))))
        print(f"    [DIAG] Unique Tile class combos (first 20): {list(_tile_class_sets)[:10]}")
    else:
        # Show first tile's structure in detail
        _first = _pg_tiles[0]
        _has_info = _first.find("span", class_="player-info-line")
        _has_img = _first.find("img")
        print(f"    [DIAG] First playerGroup: has player-info-line={_has_info is not None}, has img={_has_img is not None}")
        if _has_img:
            print(f"    [DIAG] First img src: {_has_img.get('src', '')[:100]}")
        if _has_info:
            _info_div = _has_info.find("div", class_=lambda c: c and "display-flex" in c)
            print(f"    [DIAG] info_div found: {_info_div is not None}")
            if _info_div is None:
                # Show what divs exist inside player-info-line
                _inner_divs = _has_info.find_all("div", limit=5)
                print(f"    [DIAG] Divs inside player-info-line: {[(d.get('class'), d.get_text()[:50]) for d in _inner_divs]}")
            else:
                _fspans = _info_div.find_all("span", recursive=False)
                _flds = [s.get_text(" ", strip=True) for s in _fspans]
                _itext = " \u2022 ".join(f for f in _flds if f)
                print(f"    [DIAG] info_text = {repr(_itext[:120])}")
                _nmatch = _PI_PLAYER_PATTERN.search(_itext)
                print(f"    [DIAG] regex match: {_nmatch is not None}")
                if _nmatch:
                    print(f"    [DIAG] captured name: {repr(_nmatch.group(1))}")
        else:
            _spans = _first.find_all("span", limit=5)
            print(f"    [DIAG] First tile spans: {[(s.get('class'), s.get_text()[:40]) for s in _spans]}")


    for tile in printable.find_all("div"):
        classes = tile.get("class", [])
        if "Tile" not in classes or "playerGroup" not in classes:
            continue


        # Extract player name from player-info-line
        info_span = tile.find("span", class_="player-info-line")
        if info_span is None:
            continue
        info_div = info_span.find("div", class_=lambda c: c and "display-flex" in c)
        if info_div is None:
            continue
        field_spans = info_div.find_all("span", recursive=False)
        fields = [s.get_text(" ", strip=True) for s in field_spans]
        info_text = " \u2022 ".join(f for f in fields if f)
        name_match = _PI_PLAYER_PATTERN.search(info_text)
        if not name_match:
            continue
        player_name = name_match.group(1).strip()
        safe_name = _pi_re.sub(r'[^\w\s\-]', '', player_name).strip()


        # Find the headshot <img> in this tile -- prefer the real HTTP headshot URL over
        # inline data: placeholders. FastScout headshot URLs typically contain "personnel",
        # "images/personnel/", "media-attachments", "headshot", "player", or "FSimages".
        img_tag = None
        for img in tile.find_all("img"):
            src = img.get("src", "")
            if not src or src.endswith(".svg") or "logo" in src.lower():
                continue
            # Never let a data: URI overwrite an already-found HTTP img
            if src.startswith("data:") and img_tag is not None:
                continue
            img_tag = img
            if "headshot" in src.lower() or "player" in src.lower() or "personnel" in src.lower() or "FSimages" in src or "media-attachments" in src:
                break  # best candidate found


        if img_tag is None:
            continue
        img_src = img_tag.get("src", "")


        filepath = os.path.join(output_dir, f"{safe_name}.png")
        if os.path.exists(filepath):
            skipped[safe_name] = opponent_name
            continue


        try:
            if img_src.startswith("data:"):
                import base64
                header, b64data = img_src.split(",", 1)
                img_bytes = base64.b64decode(b64data)
            else:
                # URL-encode path segments (spaces, parens, etc.) while keeping scheme/host intact
                _parsed = urlparse(img_src)
                _encoded_url = urlunparse(_parsed._replace(path=quote(_parsed.path, safe="/")))
                req = urllib.request.Request(_encoded_url, headers={"User-Agent": "Mozilla/5.0"})
                with urllib.request.urlopen(req, timeout=15) as resp:
                    img_bytes = resp.read()


            if len(img_bytes) > 500:  # skip tiny placeholders
                with open(filepath, "wb") as out_f:
                    out_f.write(img_bytes)
                extracted[safe_name] = opponent_name
            elif img_src.startswith(("http://", "https://")):
                # CDN returned auth-wall placeholder -- collect for Playwright batch download
                # (only HTTP(S) URLs can be re-fetched with auth; data: URIs are genuinely empty)
                _pending_downloads.append((player_name, safe_name, img_src, filepath, opponent_name))
        except Exception as dl_err:
            print(f"    [WARN] Could not download headshot for {player_name}: {type(dl_err).__name__}: {dl_err}")


    # --- Collect pending downloads (do NOT attempt async here -- the caller batches them) ---
    if _pending_downloads:
        print(f"    [AUTH] {len(_pending_downloads)} headshot(s) need authenticated download (deferred to batch)")


    return extracted, skipped, _pending_downloads




# --- PDF headshot extraction (original PyMuPDF approach) ---
def _extract_headshots_from_pdf(pdf_path, opponent_name, output_dir):
    """Extract player headshots from a PDF scout report using PyMuPDF."""
    if fitz is None:
        print(f"    [SKIP] pymupdf not installed -- cannot process PDF: {os.path.basename(pdf_path)}")
        return {}, {}, {}


    doc = fitz.open(pdf_path)
    extracted = {}
    skipped = {}
    diag = {"file": os.path.basename(pdf_path), "format": "pdf", "pages": doc.page_count,
            "total_images": 0, "candidate_headshots": 0, "names_matched": 0}


    xref_count = _PICounter()
    for pn in range(doc.page_count):
        for img in doc[pn].get_images(full=True):
            xref_count[img[0]] += 1
    header_xrefs = {xref for xref, cnt in xref_count.items() if cnt >= 3}


    for pn in range(doc.page_count):
        page = doc[pn]
        images = page.get_images(full=True)
        diag["total_images"] += len(images)


        headshots = []
        for img in images:
            xref = img[0]
            if xref in header_xrefs:
                continue
            base = doc.extract_image(xref)
            w, h = base['width'], base['height']
            if h >= w * 0.8 and len(base['image']) > 5000:
                rects = page.get_image_rects(xref)
                if rects:
                    headshots.append({'y': rects[0].y0, 'image_data': base['image'], 'ext': base['ext']})


        if not headshots:
            continue
        diag["candidate_headshots"] += len(headshots)
        headshots.sort(key=lambda x: x['y'])


        text = page.get_text("text")
        names = _PI_PLAYER_PATTERN.findall(text)
        diag["names_matched"] += len(names)


        for i in range(min(len(headshots), len(names))):
            name = names[i].strip()
            safe = _pi_re.sub(r'[^\w\s\-]', '', name).strip()
            filename = f"{safe}.{headshots[i]['ext']}"
            filepath = os.path.join(output_dir, filename)
            if os.path.exists(filepath):
                skipped[safe] = opponent_name
            else:
                with open(filepath, 'wb') as out_f:
                    out_f.write(headshots[i]['image_data'])
                extracted[safe] = opponent_name


    doc.close()
    return extracted, skipped, diag




# --- Main loop: process all scout reports (HTML and PDF) ---
_pi_all_extracted = {}
_pi_skipped = {}
_pi_all_pending = []  # (player_name, safe_name, url, filepath, opponent_name) tuples deferred for batch auth download
_pi_diag = []


print(f"Processing {len(scout_report_files)} scout report(s) from scout_report_files...")
for _pi_path in sorted(scout_report_files):
    _pi_file = os.path.basename(_pi_path)
    _pi_opponent = opponent_from_scout_filename(_pi_path)


    if _pi_path.lower().endswith(".html"):
        _ext, _skip, _pending = _extract_headshots_from_html(_pi_path, _pi_opponent, _PI_OUTPUT_DIR)
        _pi_all_extracted.update(_ext)
        _pi_skipped.update(_skip)
        _pi_all_pending.extend(_pending)
        _pi_diag.append({"file": _pi_file, "format": "html", "extracted": len(_ext), "skipped": len(_skip)})
    else:
        _ext, _skip, _d = _extract_headshots_from_pdf(_pi_path, _pi_opponent, _PI_OUTPUT_DIR)
        _pi_all_extracted.update(_ext)
        _pi_skipped.update(_skip)
        if _d:
            _pi_diag.append(_d)


# --- Batch authenticated download of ALL pending headshots in a single Playwright session ---
# run_in_fastscout_session is SYNCHRONOUS (dispatches fn(page) on a dedicated worker thread using
# Playwright's sync API) -- no async/await/event-loop needed here.
if _pi_all_pending:
    print(f"\n[AUTH] Batch downloading {len(_pi_all_pending)} headshot(s) via authenticated Playwright session...")
    try:
        def _batch_download_headshots(page):
            downloaded = 0
            for _pname, _sname, _url, _fpath, _opp in _pi_all_pending:
                try:
                    _resp = page.request.get(_url)
                    if _resp.ok:
                        _body = _resp.body()
                        if len(_body) > 500:
                            with open(_fpath, "wb") as _f:
                                _f.write(_body)
                            _pi_all_extracted[_sname] = _opp
                            downloaded += 1
                        else:
                            print(f"  [SKIP] {_pname}: authenticated resp still only {len(_body)} bytes")
                    else:
                        print(f"  [SKIP] {_pname}: HTTP {_resp.status}")
                except Exception as _e:
                    print(f"  [WARN] {_pname}: {type(_e).__name__}: {_e}")
            return downloaded


        _pi_pw_downloaded = run_in_fastscout_session(_batch_download_headshots)
        if _pi_pw_downloaded:
            print(f"[OK] Downloaded {_pi_pw_downloaded}/{len(_pi_all_pending)} headshot(s) via authenticated session")
        else:
            print(f"[INFO] Playwright session connected but no images exceeded 500 bytes -- "
                  f"headshots may not be uploaded on FastScout for these teams.")
    except Exception as _sess_err:
        print(f"[INFO] Cannot authenticate for headshot download: {type(_sess_err).__name__}: {_sess_err}")
        print(f"[INFO] Set FASTSCOUT_USERNAME/FASTSCOUT_PASSWORD env vars and re-run to download headshots.")
else:
    print("\n[INFO] No headshots required authenticated download.")


print(f"\nExtracted {len(_pi_all_extracted)} new player headshot(s) to {_PI_OUTPUT_DIR}")
if _pi_skipped:
    print(f"Skipped {len(_pi_skipped)} already-on-disk headshot(s)")
for _pi_name, _pi_opp in sorted(_pi_all_extracted.items(), key=lambda x: x[1]):
    print(f"  [NEW] [{_pi_opp}] {_pi_name}")


if not _pi_all_extracted and not _pi_skipped:
    print("\n--- DIAGNOSTIC: 0 headshots found. Per-report breakdown: ---")
    if not _pi_diag:
        print("  scout_pdf_files was EMPTY -- no reports to process.")
    for _d in _pi_diag:
        if _d.get("format") == "html":
            print(f"  {_d['file']}: HTML format, {_d['extracted']} extracted, {_d['skipped']} skipped")
        else:
            print(f"  {_d['file']}: {_d.get('pages', '?')} pages, {_d.get('total_images', 0)} images, "
                  f"{_d.get('candidate_headshots', 0)} candidates, {_d.get('names_matched', 0)} names matched")

# --- before_scout: hide everything else the scouting report left behind in the app's data dir -----------
# Filtering the report out of scout_report_files stops NEW artifacts being produced from it, but two kinds
# of file written by an EARLIER run are still sitting in the app's data directory, and the app happily
# renders both: the player headshots extracted from the report, and the report PDF itself (which drives the
# "Download Scouting Report PDF" button). The report is gone and its by-products aren't -- that's the leak
# this closes, so before_scout="yes" looks the same as an opponent whose report simply hasn't been made yet.
#
# Moved into a "_hidden_before_scout" subfolder rather than deleted. The app reads only the top level of
# each directory, so a subfolder is invisible to it, and flipping before_scout back to "no" restores
# everything on the next run instead of needing a re-extract -- which for the HTML-sourced headshots means
# a re-authenticated Playwright download, not just a local re-parse.
import shutil

_BS_HIDDEN_SUBDIR = "_hidden_before_scout"


def _bs_hide(paths, live_dir):
    """Move `paths` out of `live_dir` into its hidden subfolder. Returns the basenames moved."""
    _moved = []
    for _path in paths:
        if not os.path.isfile(_path):
            continue
        _hidden_dir = os.path.join(live_dir, _BS_HIDDEN_SUBDIR)
        os.makedirs(_hidden_dir, exist_ok=True)
        shutil.move(_path, os.path.join(_hidden_dir, os.path.basename(_path)))
        _moved.append(os.path.basename(_path))
    return sorted(_moved)


def _bs_restore(live_dir):
    """Move everything back out of `live_dir`'s hidden subfolder -- not just the current opponent's, so a
    file hidden during one run doesn't stay hidden once you've moved on to the next game. A hidden file
    whose live copy already exists is left alone rather than overwriting a freshly produced one; it's a
    duplicate of the same artifact either way. Returns (restored_basenames, n_duplicates_left)."""
    _hidden_dir = os.path.join(live_dir, _BS_HIDDEN_SUBDIR)
    if not os.path.isdir(_hidden_dir):
        return [], 0
    _restored, _dupes = [], 0
    for _name in sorted(os.listdir(_hidden_dir)):
        _hidden_path = os.path.join(_hidden_dir, _name)
        if not os.path.isfile(_hidden_path):
            continue
        _dest = os.path.join(live_dir, _name)
        if os.path.exists(_dest):
            _dupes += 1
            continue
        shutil.move(_hidden_path, _dest)
        _restored.append(_name)
    return _restored, _dupes


def _pi_image_paths_for(base_dir, name):
    """Every filename this player's headshot could be under. The extractors write "<sanitized name>.<ext>"
    while the app looks up "<roster name>.<ext>" -- usually identical, but check both rather than assume."""
    _safe = _pi_re.sub(r"[^\w\s\-]", "", str(name)).strip()
    return [
        os.path.join(base_dir, f"{_n}.{_ext}")
        for _n in {str(name).strip(), _safe} if _n
        for _ext in ("png", "jpeg", "jpg")
    ]


# HEADSHOTS -- no longer hidden here. Report-extracted headshots used to be the only photo source for the
# upcoming opponent, so before_scout="yes" had to hide them to avoid leaking a report that's meant to not
# exist yet. That's no longer true: Personnel Details now gets its photos from the live /roster page scrape
# (see the "Roster pages" cell), which is independent of the scouting report and isn't gated by before_scout
# at all -- so there's nothing report-derived left to leak through a photo. Restoring files a PREVIOUS run
# already hid is kept below (harmless cleanup of stragglers); only the forward-hiding action is removed.

# REPORT PDF -- the app finds this with `f.lower().endswith(".pdf") and short_opponent.lower() in f.lower()`
# over data/scouting_reports/. Mirror that test exactly rather than inventing a looser one: the goal is to
# hide precisely what the app would otherwise offer for download, and a file the app can't find needs no
# hiding. Note this directory is hand-maintained -- the parser never writes to it -- so a PDF here survives
# every re-run on its own.
_BS_REPORTS_DIR = os.path.join(APP_DATA_DIR, "scouting_reports")


def _bs_report_pdfs_for(opponent_short):
    if not opponent_short or not os.path.isdir(_BS_REPORTS_DIR):
        return []
    _key = str(opponent_short).strip().lower()
    return [
        os.path.join(_BS_REPORTS_DIR, _f) for _f in sorted(os.listdir(_BS_REPORTS_DIR))
        if _f.lower().endswith(".pdf") and _key in _f.lower()
    ]


if _before_scout_enabled:
    _bs_reports_moved = _bs_hide(_bs_report_pdfs_for(upcoming_opponent_short), _BS_REPORTS_DIR)
    _bs_who = upcoming_opponent_short or "upcoming opponent"
    print(f"[before_scout=yes] Scouting report PDF: "
          + (f"hid {len(_bs_reports_moved)} file(s) from {_BS_REPORTS_DIR} -- {_bs_reports_moved}"
             if _bs_reports_moved else f"none on disk for {_bs_who} to hide."))
    # One-time cleanup: restore any headshots a PREVIOUS run hid, since new runs no longer hide them.
    _bs_restored, _bs_dupes = _bs_restore(_PI_OUTPUT_DIR)
    if _bs_restored or _bs_dupes:
        print(f"[before_scout=yes] Restored {len(_bs_restored)} headshot(s) hidden by an earlier version of "
              f"this cell to {_PI_OUTPUT_DIR}"
              + (f" ({_bs_dupes} left in place -- a live copy already exists)." if _bs_dupes else "."))
else:
    for _bs_dir, _bs_label in ((_PI_OUTPUT_DIR, "headshot"), (_BS_REPORTS_DIR, "scouting report PDF")):
        _bs_restored, _bs_dupes = _bs_restore(_bs_dir)
        if _bs_restored or _bs_dupes:
            print(f"\n[before_scout=no] Restored {len(_bs_restored)} previously hidden {_bs_label}(s) to "
                  f"{_bs_dir}"
                  + (f" ({_bs_dupes} left in place -- a live copy already exists)." if _bs_dupes else "."))


Processing 3 scout report(s) from scout_report_files...
    [DIAG] Found 9 playerGroup tiles in PrintableNode
    [DIAG] First playerGroup: has player-info-line=True, has img=True
    [DIAG] First img src: https://stats-assets.fastmodelsports.com/images/personnel/St. Thomas (TX)/2025-2026/Angel Johnson/17
    [DIAG] info_div found: True
    [DIAG] info_text = '#10 • Angel Johnson • G • 6\'2" • SR'
    [DIAG] regex match: True
    [DIAG] captured name: 'Angel Johnson'
    [DIAG] Found 7 playerGroup tiles in PrintableNode
    [DIAG] First playerGroup: has player-info-line=True, has img=True
    [DIAG] First img src: https://stats-assets.fastmodelsports.com/images/personnel/Eureka/2025-2026/Jaxson Provost/1762669410
    [DIAG] info_div found: True
    [DIAG] info_text = '#12 • Jaxson Provost • G • 5\'10" • 170 lbs • JR'
    [DIAG] regex match: True
    [DIAG] captured name: 'Jaxson Provost'
    [DIAG] Found 7 playerGroup tiles in PrintableNode
    [DIAG] First playerGroup: has player-info


### Assemble every Key to Victory into one table

Builds `uww_ktv_keys.csv` -- one row per key, in display order -- from all seven generators: the data-driven play-by-play keys, the staff's written Keys to Victory and Team Strengths, the four Lineup Scouting keys, and the three video-tagged shot-look keys. Keys are now built in the parser and merely rendered downstream, so the HTML brief and the app's Upcoming Game page can't produce different lists. Read the cell's header comment before validating against the app.


In [162]:
# --- Assemble every Key to Victory into ONE table -----------------------------------------------------------
# The keys on the app's Upcoming Game page used to be built in two places: the data-driven ones here, the
# rest inside streamlit_app.py at render time. Anything else that wanted the keys -- the HTML brief below, a
# printout, an email -- had to re-derive them and could quietly end up with a different list.
#
# This cell is now the single place keys are BUILT. It writes uww_ktv_keys.csv, one row per key in display
# order, and everything downstream just renders it.
#
# All seven generators are here:
#   Data-Driven      pbp_derived_keys; "Feature our best look"; "Attack their weakest look"; "What they go
#                    to most" (the last three from video-tagged shot data)
#   Keys to Victory  the staff's written keys from the scouting report
#   Team Strengths   the staff's read on the opponent, as "Opponent strength: ..." lines
#   Lineup Scouting  attack their worst 5-man unit; counter with ours; attack their worst 3-man combo;
#                    feature our best 3-man combo
#
# READS CSVs, not the in-memory frames above, even though those are right here. Two reasons. The CSVs are
# what the app reads, so the two can't diverge on what a column is called or how it was scoped. And it
# makes this cell re-runnable on its own, which matters while the ported logic is still being checked
# against the app's output.
#
# PORTING NOTES -- worth reading before validating this against the app:
#   * Thresholds are copied verbatim, not re-chosen: 3.0 min for opponent lineups, 5.0 for our 3-man
#     combos, 2.0 in the counter match, 8 attempts for our own shot looks, 5 for the opponent's, 25 for
#     the PPA shrinkage. Each was set against real data in the app; changing one here would make the two
#     lists differ for a reason that looks like a bug.
#   * Play-call resolution is ported in full (catalog canonicalization plus the regex fallback for
#     free-text notes), since dropping the fallback would silently change which play a key names.
#   * The app excludes opponent == "Aurora" from stints because that game's lineup columns are swapped.
#     Carried over verbatim, flagged at _KTV_STINT_EXCLUDE, and it belongs in the stint parser rather than
#     in both consumers.
#   * The app wraps each generator in try/except and moves on. Same here, but the failure is PRINTED
#     rather than swallowed: a key silently missing from a coach's brief is the worse outcome.

from itertools import combinations as _ktv_combos

_KTV_COLS = ["opponent", "game_date", "category", "key_number", "icon", "headline",
             "evidence", "reasoning", "source"]

# Hardcoded in streamlit_app.py as `_stints[_stints["opponent"] != "Aurora"]`. The real fix is upstream,
# where that game's uww_lineup/opp_lineup columns get swapped; until then both consumers have to know.
_KTV_STINT_EXCLUDE = ("Aurora",)

# --- Sample-size floors for the Personnel keys ------------------------------------------------------------
# CONFIRMED BUG (fixed here): "Attack their worst lineup" fired on units with 3.0 minutes together and named a
# -5 in 3.0 min group as the headline -- two possessions of noise. It also picked the three LOWEST +/- units
# even when every qualifying unit was positive, so a +7/40 group could be printed under "attack". The floors
# are raised and a unit now has to actually be losing its minutes to be called a target.
#   NOTE FOR THE APP: the app used to compute these keys itself with the 3.0 floor. It should now render
#   uww_ktv_keys.csv only; if it still recomputes, the two lists will differ by design.
_KTV_OPP_LU_MIN_MINUTES = 8.0     # opponent 5-man unit
_KTV_OPP_3MAN_MIN_MINUTES = 12.0  # opponent 3-man combo
_KTV_UWW_3MAN_MIN_MINUTES = 10.0  # our 3-man combo
_KTV_LOOK_LINEUP_MIN_ATTEMPTS = 5 # a lineup's attempts on a look before "gets it best" is claimed

# --- Categories -------------------------------------------------------------------------------------------
# Offense / Defense / Personnel. Where a key comes from a generator, the category is set explicitly by the
# generator, because the generator knows what the key is FOR -- "attack their weakest look" is an offensive
# instruction even though every number in it describes their defense, and keyword matching on the text gets
# that backwards.
#
# Only the staff's free-text keys have to be classified from words, and those fall back to "General" rather
# than being forced into one of the three. A misfiled key is worse than an unfiled one: a defensive
# instruction sitting in the offensive section is an instruction a coach reads at the wrong moment.
_KTV_CATEGORY_ORDER = ["Offense", "Defense", "Personnel", "General"]
_KTV_CATEGORY_KEYWORDS = {
    "Defense": (r"defen[sc]|guard|deny|contest|close ?out|help|rotate|box ?out|defensive (?:glass|rebound)"
                r"|take away|shut|stop|force|contain|transition d|get back|protect the (?:rim|paint)"
                r"|pack|switch|trap|press|steal|turnover(?:s)? (?:we|forced)|no (?:middle|baseline)"),
    "Offense": (r"attack|score|shoot|shot|three|3(?:'s|s)?|drive|finish|paint touch|ball movement|assist"
                r"|offensive (?:glass|rebound)|second chance|pace|push|run|execute|spacing|post up"
                r"|free ?throw|get to the line|feature|hunt"),
    "Personnel": (r"lineup|rotation|minutes|bench|foul trouble|matchup|substitut|combo|unit|starter"
                  r"|who(?:'s| is) on the floor"),
}


def _ktv_categorize(text):
    """Category for a free-text key, by word-boundary keyword match; "General" when nothing fits.

    Defense is tested first: a written key is usually phrased around what the opponent does, so "take away
    their transition threes" hits an offensive word ("threes") while plainly being a defensive
    instruction. Matching defense first stops the more specific reading losing to the more generic one.
    """
    low = str(text or "").lower()
    for category in ("Defense", "Offense", "Personnel"):
        if re.search(_KTV_CATEGORY_KEYWORDS[category], low):
            return category
    return "General"


_ktv_rows = []
_ktv_problems = []


def _ktv_load(name):
    """One exported table, or an empty frame. Missing tables are normal (early season, before_scout)."""
    try:
        return pd.read_csv(os.path.join(APP_DATA_DIR, f"{name}.csv"))
    except Exception:
        return pd.DataFrame()


def _ktv_add(icon, headline, evidence, reasoning, source, category):
    _ktv_rows.append({
        "opponent": upcoming_opponent_short,
        "category": category,
        "icon": icon,
        "headline": str(headline or "").strip(),
        "evidence": "" if evidence is None else str(evidence).strip(),
        "reasoning": "" if reasoning is None else str(reasoning).strip(),
        "source": source,
    })


def _ktv_last_names(lineup_str):
    """'First Last, First Last, ...' -> 'Last, Last, ...'.

    A lineup field can hold TWO units joined by " / " (a substitution mid-run), so split on that first
    and de-duplicate -- the naive comma split rendered nine names with four repeated.
    """
    names = []
    for unit in str(lineup_str or "").split(" / "):
        for full in unit.split(","):
            full = full.strip()
            if full and full not in names:
                names.append(full)
    return ", ".join(n.split()[-1] if n.split() else n for n in names)


# --- shot-look tagging (ported from the app's extract_shot_mechanic / extract_contest) ------------------
_KTV_UNCLASSIFIED_MECHANIC = "Unclassified (no mechanic tag)"
_KTV_NO_CONTEST_TAG = "Not tagged (contest recorded only on catch-and-shoot)"
_KTV_SHRINKAGE_ATTEMPTS = 25


def _ktv_mechanic(description):
    """Which kind of shot this was, from the tagger's own chained description. The three origin tests run
    AFTER the mechanic tests on purpose: they describe how a shot was created rather than how it was
    released, so a post-up finishing as a jumper still counts as a jumper."""
    if pd.isna(description):
        return None
    d = str(description)
    if "No Dribble Jumper" in d:
        return "Catch-and-shoot"
    if "Dribble Jumper" in d:
        return "Pull-up off the dribble"
    if "To Basket" in d:
        return "Drive to the basket"
    if "Offensive Rebound" in d:
        return "Putback off the offensive glass"
    if "Cut" in d:
        return "Cut to the basket"
    if "Post-Up" in d:
        return "Post-up"
    return _KTV_UNCLASSIFIED_MECHANIC


def _ktv_contest(description):
    """Defender contest, which the tagger records ONLY on catch-and-shoot jumpers -- so a missing tag means
    the dimension doesn't apply to this shot type, not that the shot was a drive."""
    if pd.isna(description):
        return None
    d = str(description)
    if "Guarded" in d:
        return "Guarded"
    if "Open" in d:
        return "Open"
    return _KTV_NO_CONTEST_TAG


def _ktv_shot_looks(shots, min_attempts=8):
    """Per mechanic-by-contest look: volume, FG%, eFG%, points per attempt, and a shrunk PPA to rank on.

    Ranked on points per attempt rather than FG% (a 34% three is worth more than a 44% two), and shrunk
    toward the overall rate by attempts so a lucky 5-of-8 bucket can't top a 30-of-70 that is genuinely
    better. Returns an empty frame when nothing clears min_attempts.
    """
    if shots.empty:
        return pd.DataFrame()
    work = shots[shots["_mechanic"].notna() & shots["_contest"].notna()].copy()
    if work.empty:
        return pd.DataFrame()

    if "shot_type" in work.columns:
        value = pd.to_numeric(work["shot_type"], errors="coerce")
        work["_value"] = value.where(value.isin([2, 3]), 2)
    else:
        work["_value"] = 2
    work["_points"] = work["_value"] * work["_make"].astype(int)
    work["_is_three"] = work["_value"] == 3
    work["_three_make"] = (work["_is_three"] & work["_make"].astype(bool)).astype(int)

    grouped = work.groupby(["_mechanic", "_contest"]).agg(
        Attempts=("_make", "count"), Makes=("_make", "sum"),
        Points=("_points", "sum"), Threes=("_is_three", "sum"), ThreeMakes=("_three_make", "sum"),
    ).reset_index()
    grouped = grouped[grouped["Attempts"] >= min_attempts]
    if grouped.empty:
        return pd.DataFrame()

    total_attempts = float(work["_make"].count())
    baseline_ppa = float(work["_points"].sum()) / total_attempts if total_attempts else 0.0
    grouped["FG%"] = 100 * grouped["Makes"] / grouped["Attempts"]
    grouped["eFG%"] = 100 * (grouped["Makes"] + 0.5 * grouped["ThreeMakes"]) / grouped["Attempts"]
    grouped["PPA"] = grouped["Points"] / grouped["Attempts"]
    grouped["Share"] = 100 * grouped["Attempts"] / total_attempts if total_attempts else 0.0
    grouped["ThreeRate"] = 100 * grouped["Threes"] / grouped["Attempts"]
    credibility = grouped["Attempts"] / (grouped["Attempts"] + _KTV_SHRINKAGE_ATTEMPTS)
    grouped["PPA_adj"] = grouped["PPA"] * credibility + baseline_ppa * (1 - credibility)
    grouped.attrs["baseline_ppa"] = baseline_ppa
    return grouped.sort_values("PPA_adj", ascending=False).reset_index(drop=True)


def _ktv_shot_stat_line(row, baseline_ppa=None):
    parts = [
        f"{int(row['Makes'])}/{int(row['Attempts'])} ({row['FG%']:.0f}% FG, {row['eFG%']:.0f}% eFG)",
        f"{row['PPA']:.2f} pts/attempt",
    ]
    if baseline_ppa:
        parts.append(f"{row['PPA'] - baseline_ppa:+.2f} vs all looks ({baseline_ppa:.2f})")
    parts.append(f"{row['Share']:.0f}% of attempts")
    # CONFIRMED BUG (fixed here): printed "100% from three", which reads as a make rate and sat next to
    # "50% FG" on the same line. ThreeRate is the SHARE of these attempts that are threes.
    if row.get("ThreeRate", 0) >= 99.5:
        parts.append("all threes")
    elif row.get("ThreeRate", 0) >= 1:
        parts.append(f"{row['ThreeRate']:.0f}% of them threes")
    return " \u00b7 ".join(parts)


def _ktv_describe_look(mechanic, contest=None):
    """Plain-language name for a shot type. The contest dimension only exists for catch-and-shoot, so for
    every other shot the honest rendering is to say nothing about it rather than paste on a caveat about
    the tagging system as if it described the shot."""
    if mechanic is None or (isinstance(mechanic, float) and mechanic != mechanic):
        return "untagged shots"
    m = str(mechanic)
    if m == _KTV_UNCLASSIFIED_MECHANIC:
        m = "shots with no mechanic tag"
    if contest in ("Guarded", "Open"):
        return f"{m} ({str(contest).lower()})"
    return m


def _ktv_tag_shots(events, team=None, exclude_team=None):
    """Shot events with _mechanic/_contest/_make attached, filtered to tagged video only."""
    if events.empty or "event_type" not in events.columns:
        return pd.DataFrame()
    shots = events[events["event_type"].isin(["made_shot", "missed_shot"])].copy()
    if team is not None:
        shots = shots[shots["team"] == team]
    if exclude_team is not None:
        shots = shots[shots["team"].notna() & (shots["team"] != exclude_team)]
    if shots.empty or "video_description" not in shots.columns:
        return pd.DataFrame()
    shots = shots[shots["video_description"].notna()]
    if shots.empty:
        return shots
    shots["_mechanic"] = shots["video_description"].apply(_ktv_mechanic)
    shots["_contest"] = shots["video_description"].apply(_ktv_contest)
    shots["_make"] = shots["event_type"] == "made_shot"
    return shots


# --- play-call resolution (ported in full) --------------------------------------------------------------
_KTV_NON_PLAY_CALL_PATTERNS = (
    r"^end\s+of\b",
    r"^(half|halftime|game|period|quarter|ot\d*|overtime)$",
    r"^(time\s*out|timeout|to)$",
    r"^(dead\s*ball|jump\s*ball|tip\s*off|tipoff)$",
    r"^(shot\s*clock|clock)\b",
    r"^(free\s*throws?|ft)$",
    r"^(n/?a|none|unknown|tbd|misc|other|untagged)$",
)
_KTV_QUALIFIER_WORDS = {
    "good", "bad", "great", "ok", "okay", "nice", "poor",
    "make", "made", "makes", "miss", "missed", "misses", "score", "scored", "bucket",
    "and1", "and-1", "foul", "fouled", "to", "turnover", "tov", "execution", "exec",
}
_KTV_PLAY_ACRONYMS = {"ELOB", "SLOB", "BLOB", "ATO", "DHO", "ISO", "PNR", "OB", "UCLA", "STS"}


def _ktv_play_norm(text):
    return re.sub(r"[^A-Z0-9]", "", str(text).upper())


_ktv_play_catalog = {}
_ktv_cat = _ktv_load("uww_plays_catalog")
if not _ktv_cat.empty and "play_name" in _ktv_cat.columns:
    for _, _cr in _ktv_cat.iterrows():
        _ks = [k for k in str(_cr.get("match_keys", "")).split("|") if k] or [_ktv_play_norm(_cr["play_name"])]
        for _k in _ks:
            _ktv_play_catalog.setdefault(_k, str(_cr["play_name"]))


def _ktv_is_non_play(name):
    t = re.sub(r"\s+", " ", str(name or "")).strip().lower()
    return True if not t else any(re.search(p, t) for p in _KTV_NON_PLAY_CALL_PATTERNS)


def _ktv_strip_qualifiers(name):
    """Only whole trailing tokens from a known list, only from the END -- a 'longest prefix that matches
    the catalog' rule would turn the real play 'Twins Swirl' into the different real play 'Twins'."""
    toks = str(name or "").strip().split()
    while len(toks) > 1 and toks[-1].strip("().,+-\"'").lower() in _KTV_QUALIFIER_WORDS:
        toks.pop()
    return " ".join(toks)


def _ktv_title_case(name):
    """Consistent spelling for a call the catalog doesn't list, so 'TWINS SWIRL' and 'Twins Swirl' don't
    group as two plays. Short or digit-bearing tokens stay as typed so DHO and P-4 survive."""
    out = []
    for tok in re.split(r"(\s+)", str(name).strip()):
        if not tok or tok.isspace():
            out.append(tok)
        elif tok.upper().strip("()\"'") in _KTV_PLAY_ACRONYMS:
            out.append(tok.upper())
        elif len(tok) <= 3 or any(ch.isdigit() for ch in tok):
            out.append(tok.upper() if tok.isupper() else tok)
        else:
            out.append(tok[:1].upper() + tok[1:].lower())
    return "".join(out)


def _ktv_canonical_play(name):
    if name is None or (isinstance(name, float) and name != name):
        return name
    hit = _ktv_play_catalog.get(_ktv_play_norm(name))
    if hit:
        return hit
    stripped = _ktv_strip_qualifiers(name)
    if stripped and _ktv_play_norm(stripped) != _ktv_play_norm(name):
        hit = _ktv_play_catalog.get(_ktv_play_norm(stripped))
        if hit:
            return hit
    if _ktv_is_non_play(stripped or name):
        return None
    return _ktv_title_case(stripped or name)


def _ktv_extract_play_call(note):
    """A short, mostly-uppercase leading phrase before the word EXECUTION -- the one consistent signal in
    this staff's notation. Anything else returns None rather than guessing a name."""
    if pd.isna(note):
        return None
    m = re.match(r"^([A-Z][A-Z0-9\-&' ]{1,24}?)\s+EXECUTION\b", str(note).strip())
    return m.group(1).strip() if m else None


def _ktv_resolve_play_calls(df):
    """Prefer the parser's structured play_call column; fall back to the regex on free-text coach notes for
    rows the play log doesn't cover; canonicalize both against the catalog so one play has one name."""
    if "coach_note" in df.columns and _KTV_USE_COACH_NOTE_PLAY_CALLS:
        regex_fallback = df["coach_note"].apply(_ktv_extract_play_call)
    else:
        regex_fallback = pd.Series([None] * len(df), index=df.index)
    if "play_call" not in df.columns:
        return regex_fallback.apply(_ktv_canonical_play)
    has_real = df["play_call"].notna() & (df["play_call"].astype(str).str.strip() != "")
    return df["play_call"].where(has_real, regex_fallback).apply(_ktv_canonical_play)


# "Usually" is a claim about a pattern, so it needs enough repetitions to be one. Without these floors a
# single tagged possession produced "Usually comes off Twins Right Swirl (1x this season)" -- a sentence a
# coach would reasonably act on, resting on one clip. The share floor covers the other half of the problem:
# 3 of 40 is a real count and still not what they usually do.
_KTV_MIN_PLAY_CALLS = 3
_KTV_MIN_PLAY_SHARE = 0.25

# The play log is the reliable source. The regex fallback reads a play name out of free-text coach notes
# ("TWINS RIGHT SWIRL EXECUTION = ..."), which is how a name can still appear after the structured
# play_call data has been cleared out. Set this False to use the play log only.
_KTV_USE_COACH_NOTE_PLAY_CALLS = True


def _ktv_top_play_text(rows, suffix=""):
    if rows.empty or not ({"coach_note", "play_call"} & set(rows.columns)):
        return None
    calls = _ktv_resolve_play_calls(rows).dropna()
    if calls.empty:
        return None
    counts = calls.value_counts()
    top, n = counts.index[0], int(counts.iloc[0])
    if n < _KTV_MIN_PLAY_CALLS or (n / len(calls)) < _KTV_MIN_PLAY_SHARE:
        return None
    return (f"Usually comes off {top}{suffix} "
            f"({n} of {len(calls)} tagged possessions this season)")


# --- lineup profile matching (ported from lineup_profile / profile_distance / counter_lineups) ----------
_KTV_STYLE_TAGS = ("three_point_shooter", "slasher_driver", "post_scorer", "playmaker",
                   "rebounder", "catch_and_shoot")
_KTV_POSITION_SLOTS = ("Guard", "Wing", "Forward/Post")
_KTV_PROFILE_WEIGHTS = {**{f"pos_{p}": 1.0 for p in _KTV_POSITION_SLOTS},
                        "starters": 0.5,
                        **{f"tag_{t}": 0.5 for t in _KTV_STYLE_TAGS}}


def _ktv_safe_float(val):
    try:
        return float(val)
    except (TypeError, ValueError):
        return None


_ktv_player_lookup = {}
_ktv_prof = _ktv_load("uww_player_profiles")
if not _ktv_prof.empty and "name" in _ktv_prof.columns:
    for _, _pr in _ktv_prof.iterrows():
        _nm = str(_pr.get("name", "")).strip()
        if not _nm:
            continue
        _tags = str(_pr.get("notes_tags_display", "") or "")
        _ktv_player_lookup[(str(_pr.get("opponent", "")).strip(), _nm.casefold())] = {
            "position_group": str(_pr.get("position_group", "") or "Unknown"),
            "role": str(_pr.get("role", "") or ""),
            "height_inches": _ktv_safe_float(_pr.get("height_inches")),
            "tags": {t.strip() for t in _tags.split(",") if t.strip()},
        }


def _ktv_lineup_profile(opponent, lineup_str):
    """None when fewer than three of the five players resolve -- a profile built from two known players
    describes the gaps in the scouting data more than it describes the lineup."""
    names = [n.strip() for n in str(lineup_str).split(",") if n.strip()]
    known = [p for p in (_ktv_player_lookup.get((str(opponent).strip(), n.casefold())) for n in names) if p]
    if len(known) < 3:
        return None
    profile = {f"pos_{slot}": 0.0 for slot in _KTV_POSITION_SLOTS}
    for player in known:
        key = f"pos_{player['position_group']}"
        if key in profile:
            profile[key] += 1.0
    heights = [p["height_inches"] for p in known if p["height_inches"]]
    profile["height"] = (sum(heights) / len(heights)) if heights else None
    profile["starters"] = float(sum(1 for p in known if p["role"] == "Starter"))
    for tag in _KTV_STYLE_TAGS:
        profile[f"tag_{tag}"] = float(sum(1 for p in known if tag in p["tags"]))
    return profile


def _ktv_profile_distance(a, b):
    """Height compared in 3-inch units so a three-inch difference counts about the same as one position
    slot differing; in raw inches it would swamp every other feature."""
    if not a or not b:
        return float("inf")
    total = sum(w * (a.get(k, 0.0) - b.get(k, 0.0)) ** 2 for k, w in _KTV_PROFILE_WEIGHTS.items())
    if a.get("height") and b.get("height"):
        total += ((a["height"] - b["height"]) / 3.0) ** 2
    return total ** 0.5


def _ktv_describe_profile(profile):
    if not profile:
        return ""
    bits = []
    mix = [f"{int(profile['pos_' + slot])} {slot.split('/')[0].lower()}"
           for slot in _KTV_POSITION_SLOTS if profile.get(f"pos_{slot}")]
    if mix:
        bits.append(", ".join(mix))
    if profile.get("height"):
        inches = profile["height"]
        bits.append(f"avg {int(inches // 12)}'{int(round(inches % 12))}\"")
    traits = [t.replace("_", " ") for t in _KTV_STYLE_TAGS if profile.get(f"tag_{t}", 0) >= 2]
    if traits:
        bits.append(" / ".join(traits))
    return " \u00b7 ".join(bits)


def _ktv_counter_lineups(short, target_lineup, stints_df, min_matched_minutes=40.0, min_lineup_minutes=2.0):
    """Our 5-man units by net margin against opponent lineups that RESEMBLE target_lineup.

    Walks outward from the most similar unit we've faced until enough floor time is gathered, so the
    sample adapts to how much comparable basketball has been played rather than using a fixed cutoff.
    Returns (table, matched_minutes, n_similar, target_description); table is None whenever the match
    can't be made, so the caller says so rather than showing a season-wide ranking under a
    matchup-specific heading.
    """
    target = _ktv_lineup_profile(short, target_lineup)
    needed = {"opponent", "opp_lineup", "uww_lineup", "stint_minutes", "uww_margin_change"}
    if target is None or stints_df is None or stints_df.empty or not needed <= set(stints_df.columns):
        return None, 0.0, 0, _ktv_describe_profile(target)

    faced = stints_df[["opponent", "opp_lineup"]].dropna().drop_duplicates()
    scored = []
    for opp, lineup in faced.itertuples(index=False):
        distance = _ktv_profile_distance(target, _ktv_lineup_profile(opp, lineup))
        if distance != float("inf"):
            scored.append((distance, opp, lineup))
    if not scored:
        return None, 0.0, 0, _ktv_describe_profile(target)
    scored.sort(key=lambda row: row[0])

    minutes_by_unit = stints_df.groupby(["opponent", "opp_lineup"])["stint_minutes"].sum()
    keep, matched_minutes = set(), 0.0
    for _, opp, lineup in scored:
        keep.add((opp, lineup))
        matched_minutes += float(minutes_by_unit.get((opp, lineup), 0.0))
        if matched_minutes >= min_matched_minutes:
            break
    if matched_minutes <= 0:
        return None, 0.0, 0, _ktv_describe_profile(target)

    matched = stints_df[[(o, l) in keep for o, l in zip(stints_df["opponent"], stints_df["opp_lineup"])]]
    agg = (matched.groupby("uww_lineup")
                  .agg(MIN=("stint_minutes", "sum"), net=("uww_margin_change", "sum")).reset_index())
    agg = agg[agg["MIN"] >= min_lineup_minutes]
    if agg.empty:
        return None, matched_minutes, len(keep), _ktv_describe_profile(target)
    agg["rate"] = agg["net"] / agg["MIN"]
    return agg.sort_values("rate", ascending=False), matched_minutes, len(keep), _ktv_describe_profile(target)


# ========================================================================================================
# Build the keys
# ========================================================================================================
_ktv_game_date = None
try:
    _ktv_game_date = game_date_for(upcoming_opponent_short)
except Exception:
    pass

_ktv_short = upcoming_opponent_short

# --- A. Data-driven keys already built above from play-by-play -----------------------------------------
try:
    _dk = _ktv_load("uww_pbp_derived_keys")
    if not _dk.empty and "opponent" in _dk.columns and _ktv_short:
        _dk = _dk[_dk["opponent"].astype(str) == str(_ktv_short)]
        if "key_number" in _dk.columns:
            _dk = _dk.sort_values("key_number")
        for _, _k in _dk.iterrows():
            # Both derived keys are instructions about what to take away, i.e. defensive.
            _ktv_add("\U0001f4ca", _k.get("title"), _k.get("supporting_stats"),
                     _k.get("recommendation"), "Data-Driven", "Defense")
except Exception as _e:
    _ktv_problems.append(f"Data-Driven (pbp_derived_keys): {_e}")

# --- B/C. The staff's written report --------------------------------------------------------------------
# Both topics store their items as one "|"-joined string with "1. " numbering baked in.
try:
    _gp = _ktv_load("uww_opponent_game_plans")
    if not _gp.empty and {"opponent", "topic", "notes"}.issubset(_gp.columns) and _ktv_short:
        _plan = _gp[_gp["opponent"].astype(str) == str(_ktv_short)]
        # Team Strengths are classified as Defense outright rather than by keywords: an opponent strength
        # is by definition a thing we have to take away, whatever words the staff used to describe it.
        # That also dodges a real false positive -- "two playmaking guards" matched the defensive keyword
        # "guard" by accident, and "second-chance points" matched nothing at all.
        for _topic, _icon, _source, _prefix, _fixed_cat in (
            ("KEYS TO VICTORY", "\U0001f4cb", "Keys to Victory", "", None),
            ("TEAM STRENGTHS", "\u26a0\ufe0f", "Team Strengths", "Opponent strength: ", "Defense"),
        ):
            _rows = _plan[_plan["topic"].astype(str).str.strip().str.upper() == _topic]
            if _rows.empty:
                continue
            for _item in str(_rows.iloc[0]["notes"]).split("|"):
                _item = re.sub(r"^\d+\.\s*", "", _item.strip())
                if _item and _item.lower() != "nan":
                    # The staff's written Keys to Victory are the only ones read off the text.
                    _ktv_add(_icon, f"{_prefix}{_item}", None, None, _source,
                             _fixed_cat or _ktv_categorize(_item))
except Exception as _e:
    _ktv_problems.append(f"Keys to Victory / Team Strengths (game plans): {_e}")

# --- lineup data, shared by the four Lineup Scouting keys ------------------------------------------------
_ktv_stints = _ktv_load("uww_lineup_stints")
_ktv_uww_lu = None
if not _ktv_stints.empty:
    if "opponent" in _ktv_stints.columns:
        _ktv_stints = _ktv_stints[~_ktv_stints["opponent"].isin(_KTV_STINT_EXCLUDE)].copy()
    if {"end_uww_score", "start_prev_uww_score"}.issubset(_ktv_stints.columns):
        _ktv_stints["uww_pts"] = _ktv_stints["end_uww_score"] - _ktv_stints["start_prev_uww_score"]
    if {"uww_lineup", "stint_minutes", "uww_margin_change"}.issubset(_ktv_stints.columns):
        _ktv_uww_lu = _ktv_stints.groupby("uww_lineup").agg(
            MIN=("stint_minutes", "sum"),
            PTS=("uww_pts", "sum") if "uww_pts" in _ktv_stints.columns else ("stint_minutes", "size"),
            plus_minus=("uww_margin_change", "sum"),
            GP=("game_date", "nunique"),
        ).reset_index().rename(columns={"uww_lineup": "lineup", "plus_minus": "+/-"})

# The opponent lineup table carries no opponent column, so validate it actually belongs to THIS opponent
# by checking its player names against their roster. Without this, a stale table left from the previous
# opponent gets scouted as if it were theirs.
_ktv_opp_lu = _ktv_load("uww_opp_lineup_season_box")
if _ktv_opp_lu.empty or "MIN" not in _ktv_opp_lu.columns:
    _ktv_opp_lu = None
else:
    _names = set()
    for _tbl in ("uww_player_profiles", "uww_opponent_rosters"):
        _t = _ktv_load(_tbl)
        if not _t.empty and "opponent" in _t.columns and _ktv_short:
            _names = set(_t[_t["opponent"].astype(str) == str(_ktv_short)]["name"].dropna())
        if _names:
            break
    if _names and "lineup" in _ktv_opp_lu.columns:
        _lu_players = set()
        for _lu in _ktv_opp_lu["lineup"].dropna():
            _lu_players.update(p.strip() for p in str(_lu).split(","))
        if not (_lu_players & _names):
            _ktv_opp_lu = None
    if _ktv_opp_lu is not None:
        for _c in ("MIN", "PTS", "+/-", "GP"):
            if _c in _ktv_opp_lu.columns:
                _ktv_opp_lu[_c] = pd.to_numeric(_ktv_opp_lu[_c], errors="coerce").fillna(0)

# --- D. Attack their worst 5-man unit --------------------------------------------------------------------
try:
    if _ktv_opp_lu is not None and not _ktv_opp_lu.empty:
        _vl = _ktv_opp_lu.copy()
        _vl["_pm_fg"] = pd.to_numeric(_vl["FG%"], errors="coerce").fillna(0) if "FG%" in _vl.columns else 0
        if "TO" in _vl.columns:
            _vl["_to_rate"] = (_vl["TO"] / _vl["MIN"].replace(0, float("nan")) * 40).round(1)
        _vl_qual = _vl[_vl["MIN"] >= _KTV_OPP_LU_MIN_MINUTES]
        _worst = _vl_qual[_vl_qual["+/-"] < 0].nsmallest(3, "+/-")
        if not _worst.empty:
            _lines = ["Worst +/- lineups:"]
            for _, _r in _worst.iterrows():
                _fg = f", {_r['_pm_fg']:.0f}% FG" if _r.get("_pm_fg", 0) > 0 else ""
                _lines.append(f"{_r['+/-']:+.1f} in {_r['MIN']:.1f} min{_fg} \u2014 {_ktv_last_names(_r['lineup'])}")
            if "_to_rate" in _vl_qual.columns:
                _high_to = _vl_qual.nlargest(2, "_to_rate")
                if not _high_to.empty:
                    _lines.append("Highest TO rate (per 40 min):")
                    for _, _r in _high_to.iterrows():
                        _lines.append(f"{_r['_to_rate']:.1f} TO/40 \u2014 {_ktv_last_names(_r['lineup'])}")
            _ktv_add("\U0001f512",
                     f"Attack {_ktv_short}'s {_ktv_last_names(_worst.iloc[0]['lineup'])} lineup",
                     "\n".join(_lines),
                     f"{_ktv_short}'s most exploitable units: their worst net-rating lineups with real "
                     f"minutes this season ({_KTV_OPP_LU_MIN_MINUTES:.0f}+ min, losing their minutes), and "
                     f"the lineups that give the ball away most per 40 "
                     f"minutes. The headline names the worst of them.",
                     "Lineup Scouting", "Personnel")
except Exception as _e:
    _ktv_problems.append(f"Lineup Scouting (attack worst 5-man): {_e}")

# --- E. Counter with our best unit -----------------------------------------------------------------------
# Two modes, and the key always says which: MATCHED is our net margin against opponent lineups that
# resemble the one being prepared for -- a genuine counter. FALLBACK is our best units season-wide, which
# is useful but NOT opponent-specific and is labelled as such rather than dressed up as a matchup call.
try:
    _target = None
    if _ktv_opp_lu is not None and not _ktv_opp_lu.empty:
        _top = _ktv_opp_lu.nlargest(1, "MIN")
        if not _top.empty:
            _target = _top.iloc[0]["lineup"]

    _matched, _matched_min, _n_similar, _target_desc = (None, 0.0, 0, "")
    if _target is not None:
        _matched, _matched_min, _n_similar, _target_desc = _ktv_counter_lineups(
            _ktv_short, _target, _ktv_stints)

    if _matched is not None and not _matched.empty:
        _rows = list(_matched.head(3).iterrows())
        _caption = "\n".join(
            f"{_r['rate']:+.2f}/min ({_r['net']:+.0f} in {_r['MIN']:.1f} min) \u2014 {_ktv_last_names(_r['uww_lineup'])}"
            for _, _r in _rows)
        _reason = [f"UWW's best net margin against opponent units that resemble {_ktv_short}'s most-used "
                   f"lineup ({_ktv_last_names(_target)})."]
        if _target_desc:
            _reason.append(f"Target profile: {_target_desc}.")
        _reason.append(f"Measured across {_n_similar} comparable opponent unit(s), {_matched_min:.0f} min "
                       f"this season.")
        _ktv_add("\U0001f512", f"Counter with {_ktv_last_names(_rows[0][1]['uww_lineup'])}",
                 _caption, " ".join(_reason), "Lineup Scouting", "Personnel")
    elif _ktv_uww_lu is not None and not _ktv_uww_lu.empty:
        _fb = _ktv_uww_lu[_ktv_uww_lu["MIN"] >= _KTV_OPP_LU_MIN_MINUTES].nlargest(3, "+/-")
        if not _fb.empty:
            _rows = list(_fb.iterrows())
            _caption = "\n".join(
                f"{_r['+/-']:+.1f} total ({(_r['+/-'] / _r['MIN'] if _r['MIN'] > 0 else 0.0):+.2f}/min in "
                f"{_r['MIN']:.1f} min) \u2014 {_ktv_last_names(_r['lineup'])}" for _, _r in _rows)
            _why = ("no comparable opponent lineups on record yet" if _target is not None
                    else "no opponent lineup data yet")
            _ktv_add("\U0001f512", f"Counter with {_ktv_last_names(_rows[0][1]['lineup'])}", _caption,
                     f"UWW's best lineups by net margin season-wide \u2014 NOT matchup-specific, because "
                     f"there are {_why}.", "Lineup Scouting", "Personnel")
except Exception as _e:
    _ktv_problems.append(f"Lineup Scouting (counter lineup): {_e}")

# --- F. 3-man combinations -------------------------------------------------------------------------------
# Same idea as the 5-man keys but for the smaller units. The 3-man aggregates only carry MIN/PTS/+/-/GP
# (no FG%/TO), so this is scoped to net margin rather than shooting or turnovers.
# Both reset BEFORE the try: if the first half raised on a re-run, the second frame would otherwise keep its
# value from the previous run and be exported below as if it were current.
_uww_3man = None
_opp_3man = None
try:
    _uww_3man = None
    if _ktv_stints is not None and not _ktv_stints.empty and "uww_lineup" in _ktv_stints.columns:
        _recs = []
        for _, _st in _ktv_stints.iterrows():
            _players = sorted(p.strip() for p in str(_st["uww_lineup"]).split(","))
            for _combo in _ktv_combos(_players, 3):
                _recs.append({"lineup": ", ".join(_combo), "stint_minutes": _st["stint_minutes"],
                              "uww_pts": _st.get("uww_pts", 0),
                              "uww_margin_change": _st["uww_margin_change"],
                              "game_date": _st.get("game_date")})
        if _recs:
            _uww_3man = pd.DataFrame(_recs).groupby("lineup").agg(
                MIN=("stint_minutes", "sum"), PTS=("uww_pts", "sum"),
                plus_minus=("uww_margin_change", "sum"), GP=("game_date", "nunique"),
            ).reset_index().rename(columns={"plus_minus": "+/-"})

    _opp_3man = None
    if _ktv_opp_lu is not None and not _ktv_opp_lu.empty:
        _recs = []
        for _, _r in _ktv_opp_lu.iterrows():
            _players = sorted(p.strip() for p in str(_r["lineup"]).split(","))
            if len(_players) >= 3:
                for _combo in _ktv_combos(_players, 3):
                    _recs.append({"lineup": ", ".join(_combo),
                                  "MIN": float(_r["MIN"]) if pd.notna(_r["MIN"]) else 0,
                                  "PTS": float(_r["PTS"]) if pd.notna(_r.get("PTS")) else 0,
                                  "+/-": float(_r["+/-"]) if pd.notna(_r.get("+/-")) else 0,
                                  "GP": float(_r["GP"]) if pd.notna(_r.get("GP")) else 1})
        if _recs:
            _opp_3man = pd.DataFrame(_recs).groupby("lineup").agg(
                MIN=("MIN", "sum"), PTS=("PTS", "sum"), plus_minus=("+/-", "sum"), GP=("GP", "max"),
            ).reset_index().rename(columns={"plus_minus": "+/-"})

    if _opp_3man is not None and not _opp_3man.empty:
        _worst3 = _opp_3man[(_opp_3man["MIN"] >= _KTV_OPP_3MAN_MIN_MINUTES)
                            & (_opp_3man["+/-"] < 0)].nsmallest(3, "+/-")
        if not _worst3.empty:
            _lines = ["Worst +/- 3-man combos:"] + [
                f"{_r['+/-']:+.1f} in {_r['MIN']:.1f} min \u2014 {_ktv_last_names(_r['lineup'])}"
                for _, _r in _worst3.iterrows()]
            _ktv_add("\U0001f512",
                     f"Attack {_ktv_short}'s {_ktv_last_names(_worst3.iloc[0]['lineup'])} combo",
                     "\n".join(_lines),
                     f"{_ktv_short}'s most exploitable 3-man combinations: their worst net-rating units "
                     f"with real minutes this season ({_KTV_OPP_3MAN_MIN_MINUTES:.0f}+ min).",
                     "Lineup Scouting", "Personnel")

    if _uww_3man is not None and not _uww_3man.empty:
        _best3 = _uww_3man[(_uww_3man["MIN"] >= _KTV_UWW_3MAN_MIN_MINUTES)
                           & (_uww_3man["+/-"] > 0)].nlargest(3, "+/-")
        if not _best3.empty:
            _lines = [f"{_r['+/-']:+.1f} total ({(_r['+/-'] / _r['MIN'] if _r['MIN'] > 0 else 0.0):+.2f}/min "
                      f"in {_r['MIN']:.1f} min) \u2014 {_ktv_last_names(_r['lineup'])}"
                      for _, _r in _best3.iterrows()]
            _ktv_add("\U0001f512", f"Feature the {_ktv_last_names(_best3.iloc[0]['lineup'])} combo",
                     "\n".join(_lines),
                     f"UWW's best 3-man combinations by net margin season-wide ({_KTV_UWW_3MAN_MIN_MINUTES:.0f}+ min) \u2014 not "
                     "matchup-specific like the 5-man counter above, but the smaller units most worth "
                     "leaning on regardless of opponent.", "Lineup Scouting", "Personnel")
except Exception as _e:
    _ktv_problems.append(f"Lineup Scouting (3-man combos): {_e}")

# Export the 3-man tables (requested: show the top three in the brief, like the five-man units). These are
# the SAME frames the "Attack their ... combo" / "Feature the ... combo" keys above are built from, written
# out rather than recomputed elsewhere, so a combo named in a key and the combo table can never disagree.
# GP is only exact for our side: ours is counted from stints; theirs is assembled from five-man season rows
# with no game dates, where max() over the units is a floor, not a count -- so it is exported as a floor.
_three_man_frames = []
if "_uww_3man" in globals() and _uww_3man is not None and not _uww_3man.empty:
    _three_man_frames.append(_uww_3man.assign(side="UWW", scouted_opponent=_ktv_short, gp_exact=True))
if "_opp_3man" in globals() and _opp_3man is not None and not _opp_3man.empty:
    _three_man_frames.append(_opp_3man.assign(side="Opponent", scouted_opponent=_ktv_short, gp_exact=False))
three_man_combos = (pd.concat(_three_man_frames, ignore_index=True) if _three_man_frames
                    else pd.DataFrame(columns=["lineup", "MIN", "PTS", "+/-", "GP", "side",
                                               "scouted_opponent", "gp_exact"]))
three_man_combos.to_csv(os.path.join(APP_DATA_DIR, "uww_three_man_combos.csv"), index=False)
print(f"  3-man combos: {int((three_man_combos['side'] == 'UWW').sum())} ours, "
      f"{int((three_man_combos['side'] == 'Opponent').sum())} {_ktv_short} -> uww_three_man_combos.csv")

# --- G. Feature our best look ----------------------------------------------------------------------------
try:
    _ss = _ktv_tag_shots(_ktv_load("uww_pbp_events"), team="UW-Whitewater")
    # CONFIRMED BUG (fixed here): the headline read "Feature our best look: Catch-and-shoot (guarded)". Guarded
    # vs open is the DEFENSE's choice, not a look we can call, so splitting on it for an offensive instruction
    # recommended something nobody can run. Contest levels are pooled here so the key names a shot type we
    # can actually draw up. (H keeps the split: there it describes how the opponent defends, which is real.)
    if not _ss.empty:
        _ss = _ss.copy()
        _ss["_contest"] = "All"
        _grouped = _ktv_shot_looks(_ss, min_attempts=8)
        if not _grouped.empty:
            _baseline = _grouped.attrs.get("baseline_ppa")
            # A residual bucket is not a shot type. Prefer the best NAMED look; fall back to the
            # unclassified pile only if nothing else clears the floor, and label it plainly.
            _named = _grouped[_grouped["_mechanic"] != _KTV_UNCLASSIFIED_MECHANIC]
            _best = (_named if not _named.empty else _grouped).iloc[0]
            _bm, _bc = _best["_mechanic"], _best["_contest"]
            _best_rows = _ss[(_ss["_mechanic"] == _bm) & (_ss["_contest"] == _bc)]

            _lineup_txt = None
            if "uww_lineup" in _best_rows.columns:
                _lu_rows = _best_rows[_best_rows["uww_lineup"].notna()]
                if not _lu_rows.empty:
                    _lu = _lu_rows.groupby("uww_lineup").agg(
                        Attempts=("_make", "count"), Makes=("_make", "sum")).reset_index()
                    # CONFIRMED BUG (fixed here): a 3-attempt floor crowned a lineup that went 3/3. Raised, and
                    # ties broken by volume so the unit that generates it most wins a tie.
                    _lu = _lu[_lu["Attempts"] >= _KTV_LOOK_LINEUP_MIN_ATTEMPTS]
                    if not _lu.empty:
                        _lu["FG%"] = 100 * _lu["Makes"] / _lu["Attempts"]
                        _bl = _lu.sort_values(["FG%", "Attempts"], ascending=False).iloc[0]
                        _lineup_txt = (f"{_ktv_last_names(_bl['uww_lineup'])} gets it best "
                                       f"({int(_bl['Makes'])}/{int(_bl['Attempts'])}, {_bl['FG%']:.0f}%)")

            _play_txt = _ktv_top_play_text(_best_rows)
            _parts = [p for p in (_lineup_txt, _play_txt) if p]
            _ktv_add("\U0001f3c0", "Feature our best look: " + _ktv_describe_look(_bm, _bc),
                     _ktv_shot_stat_line(_best, _baseline) + " this season",
                     " -- ".join(_parts) if _parts else
                     "Not enough lineup or play-call data linked to these shots yet to say who runs this most.",
                     "Data-Driven", "Offense")
except Exception as _e:
    _ktv_problems.append(f"Data-Driven (feature our best look): {_e}")

# --- H. Attack their weakest look ------------------------------------------------------------------------
# Uses the shots taken BY whoever the opponent played in each game before facing UWW: the look those teams
# were most efficient at is this opponent's worst-defended one. Then cross-referenced against our own
# season-wide shot data for the same look, to say which of our lineups and play calls already generates it.
try:
    _third = _ktv_tag_shots(_ktv_load("uww_opponent_prior_games_pbp"), exclude_team=_ktv_short)
    if _third.empty:
        _ktv_add("\U0001f3af", "Attack Opponent Worst Offensive Shot Selection & Quality", None,
                 f"No video-tagged data yet for teams {_ktv_short} played before facing UWW this season -- "
                 f"needs a local/live-scraped _pbp and _video file for each of those games.",
                 "Data-Driven", "Offense")
    else:
        _grouped = _ktv_shot_looks(_third, min_attempts=5)  # a handful of prior games, not a season
        if _grouped.empty:
            _ktv_add("\U0001f3af", "Attack Opponent Worst Offensive Shot Selection & Quality", None,
                     f"Some video-tagged data exists for teams {_ktv_short} played before UWW, but not "
                     f"enough attempts yet of any one shot type (need 5+) to call one a clear weakness.",
                     "Data-Driven", "Offense")
        else:
            _baseline = _grouped.attrs.get("baseline_ppa")
            _best = _grouped.iloc[0]
            _bm, _bc = _best["_mechanic"], _best["_contest"]
            _our_shots = _ktv_tag_shots(_ktv_load("uww_pbp_events"), team="UW-Whitewater")
            _lineup_txt = _volume_txt = _play_txt = None
            if not _our_shots.empty:
                _match = _our_shots[(_our_shots["_mechanic"] == _bm) & (_our_shots["_contest"] == _bc)]
                if "uww_lineup" in _match.columns:
                    _lu_rows = _match[_match["uww_lineup"].notna()]
                    if not _lu_rows.empty:
                        _lu = _lu_rows.groupby("uww_lineup").agg(
                            Attempts=("_make", "count"), Makes=("_make", "sum")).reset_index()
                        _lu["FG%"] = 100 * _lu["Makes"] / _lu["Attempts"]
                        # Two questions, two lists: ranking by FG% alone rewards a unit that went 3/3, so
                        # the accuracy list keeps a 3+ attempt floor and the volume list answers "who
                        # actually generates this for us" with no floor. A coach needs both.
                        _qual = _lu[_lu["Attempts"] >= _KTV_LOOK_LINEUP_MIN_ATTEMPTS]
                        if not _qual.empty:
                            _top_fg = _qual.sort_values(["FG%", "Attempts"], ascending=False).head(3)
                            _lineup_txt = "\n".join([f"Best on this shot ({_KTV_LOOK_LINEUP_MIN_ATTEMPTS}+ attempts):"] + [
                                f"\u2022 {_ktv_last_names(_r['uww_lineup'])} {int(_r['Makes'])}/"
                                f"{int(_r['Attempts'])} ({_r['FG%']:.0f}%)" for _, _r in _top_fg.iterrows()])
                        _top_vol = _lu.sort_values(["Attempts", "FG%"], ascending=False).head(3)
                        if not _top_vol.empty:
                            _volume_txt = "\n".join(["Runs it most:"] + [
                                f"\u2022 {_ktv_last_names(_r['uww_lineup'])} {int(_r['Attempts'])}x "
                                f"({_r['FG%']:.0f}%)" for _, _r in _top_vol.iterrows()])
                _play_txt = _ktv_top_play_text(_match, suffix=" for us")

            _parts = [p for p in (_lineup_txt, _volume_txt, _play_txt) if p]
            _ktv_add("\U0001f3af",
                     f"Attack their weakest look: {_ktv_describe_look(_bm, _bc)}",
                     # The team count and the tagged-games count used to be appended here. They
                     # describe the sample rather than the shot, and on a printed page they pushed the
                     # number a coach actually needs behind a clause about the tagging process.
                     f"Opponents shot {_ktv_shot_stat_line(_best, _baseline)} on this against {_ktv_short}",
                     "\n".join(_parts) if _parts else
                     "Not enough UWW lineup or play-call data linked to this shot type yet to say who runs "
                     "it most for us.",
                     # Offense: every number describes their defense, but the instruction is what WE run.
                     "Data-Driven", "Offense")
except Exception as _e:
    _ktv_problems.append(f"Data-Driven (attack their weakest look): {_e}")

# --- I. What their offense goes to most ------------------------------------------------------------------
# The complement to H: by VOLUME rather than efficiency, for defensive prep rather than offensive attack.
try:
    _own = _ktv_tag_shots(_ktv_load("uww_opponent_prior_games_pbp"), team=_ktv_short)
    if not _own.empty:
        _g = _own[_own["_mechanic"].notna() & _own["_contest"].notna()].groupby(
            ["_mechanic", "_contest"]).agg(Attempts=("_make", "count"), Makes=("_make", "sum")).reset_index()
        _g = _g[_g["Attempts"] >= 5]
        if not _g.empty:
            _g["FG%"] = 100 * _g["Makes"] / _g["Attempts"]
            _top = _g.nlargest(1, "Attempts").iloc[0]
            _ktv_add("\U0001f6e1\ufe0f",
                     f"What {_ktv_short} goes to most: {_ktv_describe_look(_top['_mechanic'], _top['_contest'])}",
                     f"{int(_top['Attempts'])} attempts, {_top['FG%']:.0f}% -- across their games before UWW",
                     "What their offense goes to most often, regardless of how well it's worked -- worth a "
                     "specific defensive scheme item to take away.",
                     "Data-Driven", "Defense")
except Exception as _e:
    _ktv_problems.append(f"Data-Driven (what they go to most): {_e}")

# --- J/K. Play calls (uww_plays.csv / opponent_plays.csv, decoded by the "Play calls" cell) -------------
# J names the opponent's go-to set and what it produces; K names our most productive set with real volume.
# Both read uww_play_call_summary.csv, which excludes Rewatch/TBD clips from every ranking.
_KTV_PLAY_MIN_USES = 4
try:
    _pcs = _ktv_load("uww_play_call_summary")
    if not _pcs.empty and {"side", "level", "name", "uses"}.issubset(_pcs.columns) and _ktv_short:
        _pcs = _pcs[_pcs["scouted_opponent"].astype(str) == str(_ktv_short)]
        # Rank "best set" on PPP shrunk toward the side's own average by _KTV_PLAY_SHRINK possessions, so a
        # 4-for-4 week can't outrank a set that has worked across twenty trips.
        _KTV_PLAY_SHRINK = 8
        _pcs = _pcs.assign(_ppp_adj=(pd.to_numeric(_pcs["points"], errors="coerce")
                                     + pd.to_numeric(_pcs["team_ppp"], errors="coerce") * _KTV_PLAY_SHRINK)
                           / (pd.to_numeric(_pcs["poss_with_points"], errors="coerce") + _KTV_PLAY_SHRINK))
        _calls = _pcs[(_pcs["level"] == "Play call")
                      & ~_pcs["name"].astype(str).str.contains("unspecified", na=False)
                      & (_pcs["uses"] >= _KTV_PLAY_MIN_USES)]

        _theirs = _calls[_calls["side"] == "Opponent"].sort_values(["uses", "ppp"], ascending=[False, False])
        # "Go-to set" means half court: an inbounds set can top the count just because every game has a dozen
        # baseline out-of-bounds trips. The busiest inbounds set is named in the reasoning instead.
        _theirs_half = _theirs[_theirs["situation"].astype(str) == "Half court"]
        _theirs_oob = _theirs[_theirs["situation"].astype(str).str.match(r"^(BLOB|SLOB)")]
        if not _theirs_half.empty:
            _t = _theirs_half.iloc[0]
            _ev = (f"{int(_t['uses'])} uses in {int(_t['games'])} games"
                   + (f" · {_t['ppp']:.2f} PPP (their overall {_t['team_ppp']:.2f})" if pd.notna(_t.get("ppp")) and pd.notna(_t.get("team_ppp")) else "")
                   + (f" · {int(_t['fgm'])}/{int(_t['fga'])} FG" if int(_t.get("fga") or 0) else "")
                   + (f" · {int(_t['turnovers'])} TO" if int(_t.get("turnovers") or 0) else ""))
            _who = (f"Finished most by {_t['top_player']} ({int(_t['top_player_uses'])}x)"
                    if isinstance(_t.get("top_player"), str) and int(_t.get("top_player_uses") or 0) >= 2 else "")
            _how = " -- ".join(p for p in (
                f"built on {_t['top_action']}" if isinstance(_t.get("top_action"), str) else "",
                f"run to the {_t['top_location'].lower()}" if isinstance(_t.get("top_location"), str) else "") if p)
            _second = _theirs_half.iloc[1] if len(_theirs_half) > 1 else None
            _reason = ". ".join(p for p in (
                _who, _how.capitalize() if _how else "",
                (f"Next most-called: {_second['name']} ({int(_second['uses'])} uses)" if _second is not None else ""),
                (f"Busiest inbounds set: {_theirs_oob.iloc[0]['name']} ({int(_theirs_oob.iloc[0]['uses'])} uses"
                 + (f", {_theirs_oob.iloc[0]['ppp']:.2f} PPP)" if pd.notna(_theirs_oob.iloc[0].get('ppp')) else ")")
                 if not _theirs_oob.empty else ""),
                f"Tagged as: {_t['example_titles']}" if isinstance(_t.get("example_titles"), str) else "") if p)
            _ktv_add("\U0001f4cb", f"Take away their go-to set: {_t['name']}", _ev, _reason,
                     "Data-Driven", "Defense")
            # Their most productive set with volume, when it isn't the same one.
            _eff = _theirs[_theirs["_ppp_adj"].notna()].sort_values("_ppp_adj", ascending=False)
            if not _eff.empty and _eff.iloc[0]["name"] != _t["name"] and pd.notna(_eff.iloc[0].get("team_ppp")) \
                    and _eff.iloc[0]["ppp"] >= _eff.iloc[0]["team_ppp"] + 0.15:
                _e = _eff.iloc[0]
                _ktv_add("\U0001f4cb", f"Know their best set: {_e['name']}",
                         f"{_e['ppp']:.2f} PPP on {int(_e['uses'])} uses vs {_e['team_ppp']:.2f} overall",
                         (f"Finished most by {_e['top_player']}. " if isinstance(_e.get("top_player"), str) and int(_e.get("top_player_uses") or 0) >= 2 else "")
                         + "Their most productive set with real volume -- the one to recognize by its alignment "
                           "and call out before the action starts.",
                         "Data-Driven", "Defense")

        _ours = _calls[(_calls["side"] == "UWW") & _calls["_ppp_adj"].notna()].sort_values(["_ppp_adj", "uses"], ascending=False)
        if not _ours.empty and pd.notna(_ours.iloc[0].get("team_ppp")) and _ours.iloc[0]["_ppp_adj"] > _ours.iloc[0]["team_ppp"]:
            _o = _ours.iloc[0]
            _ktv_add("\U0001f4cb", f"Call our best set: {_o['name']}",
                     f"{_o['ppp']:.2f} PPP on {int(_o['uses'])} uses vs our {_o['team_ppp']:.2f} overall"
                     + (f" · {int(_o['fgm'])}/{int(_o['fga'])} FG" if int(_o.get("fga") or 0) else ""),
                     (f"Finished most by {_o['top_player']} ({int(_o['top_player_uses'])}x). " if isinstance(_o.get("top_player"), str) and int(_o.get("top_player_uses") or 0) >= 2 else "")
                     + f"Our most productive tagged set with {_KTV_PLAY_MIN_USES}+ uses this season.",
                     "Data-Driven", "Offense")
except Exception as _e:
    _ktv_problems.append(f"Data-Driven (play calls): {_e}")

# ========================================================================================================
# REBOUNDING (Data-Driven) -- every miss in their play-by-play paired with the rebound that follows it.
# Defined HERE (and reused by the game-plan cell's REBOUND TENDENCIES table) so the key, the brief's Bottom
# Line and roster reads, and the app's full table all come from one pairing -- not two copies that can drift.
#
# The brief no longer has a rebounding section (requested). Rebounding only reaches it when it's lopsided
# enough to change the plan; these are the thresholds, and the brief's Bottom Line note prints them.
# ========================================================================================================
REBOUND_RULES = {
    "min_paired": 20,     # paired misses before any team-level rate is trusted
    "crash_rate": 35,     # THEIR offensive-rebound % on their own misses at/above this -> box-out key
    "soft_rate": 65,      # THEIR defensive-rebound % on opponent misses at/below this -> crash-the-glass key
    "player_share": 35,   # one player with this % of their offensive rebounds after misses -> roster read
    "player_min": 5,      # ...and at least this many of them
}
_REB_SKIP_STOP = {"made_shot", "missed_shot", "turnover", "free_throw_made", "free_throw_missed", "jump_ball",
                  "period_end", "period_start"}
_REB_OFF_EV = {"rebound_offensive", "team_deadball_rebound_offensive"}
_REB_DEF_EV = {"rebound_defensive", "team_deadball_rebound_defensive"}


def _reb_shot_kind(ev):
    """What kind of miss -- the crash rate after a missed layup is a different number from after a three."""
    if ev["event_type"] == "free_throw_missed":
        return "Missed free throw"
    if str(ev.get("shot_type")) == "3":
        return "Missed three"
    _t = str(ev.get("raw_text") or "").lower()
    if any(w in _t for w in ("layup", "lay-up", "dunk", "tip", "hook", "putback", "put back", "jam")):
        return "Missed layup / dunk"
    return "Missed two-point jumper"


def rebound_pairs(pbp, team):
    """One row per miss: who missed, what kind, and the rebound that followed (or None). Walks forward from
    each miss inside the same game and period and stops at the first rebound; a make, another miss, a
    turnover or a period change first leaves the miss unpaired rather than crediting the wrong board.
    is_them marks misses by `team` -- offensive=True on those is THEIR offensive rebound."""
    cols = ["shooter_team", "kind", "reb_team", "reb_player", "offensive", "is_them"]
    if pbp is None or pbp.empty or not {"event_type", "team", "game_date", "event_order"}.issubset(pbp.columns):
        return pd.DataFrame(columns=cols)
    rp = pbp.copy()
    rp["event_type"] = rp["event_type"].astype(str)
    rp["_ord"] = pd.to_numeric(rp["event_order"], errors="coerce")
    gkey = ["game_date", "opponent"] if "opponent" in rp.columns else ["game_date"]
    out = []
    for _, gm in rp.sort_values(gkey + ["_ord"]).groupby(gkey, dropna=False):
        evs = gm.to_dict("records")
        for i, e in enumerate(evs):
            if e["event_type"] not in ("missed_shot", "free_throw_missed"):
                continue
            reb = None
            for nx in evs[i + 1:i + 6]:
                if nx.get("period") != e.get("period"):
                    break
                if nx["event_type"] in _REB_OFF_EV or nx["event_type"] in _REB_DEF_EV:
                    reb = nx
                    break
                if nx["event_type"] in _REB_SKIP_STOP:
                    break
            out.append({"shooter_team": str(e.get("team")), "kind": _reb_shot_kind(e),
                        "reb_team": None if reb is None else str(reb.get("team")),
                        "reb_player": None if reb is None else reb.get("player"),
                        "offensive": None if reb is None else reb["event_type"] in _REB_OFF_EV})
    df = pd.DataFrame(out, columns=cols[:-1])
    if df.empty:
        return pd.DataFrame(columns=cols)
    df["is_them"] = df["shooter_team"] == str(team)
    if not df["is_them"].any() and str(team).strip():   # names can carry a mascot in the pbp
        df["is_them"] = df["shooter_team"].str.lower().str.contains(str(team).lower().split()[0], na=False)
    return df


_reb_summary_rows = []
try:
    _rpairs = rebound_pairs(_ktv_load("uww_opponent_prior_games_pbp"), _ktv_short)
    if not _rpairs.empty:
        _R = REBOUND_RULES
        _paired = _rpairs[_rpairs["offensive"].notna()].copy()
        _paired["offensive"] = _paired["offensive"].astype(bool)
        _their = _paired[_paired["is_them"]]
        _opp = _paired[~_paired["is_them"]]
        _crashers = (_their[_their["offensive"]]["reb_player"].dropna().astype(str)
                     .loc[lambda s: ~s.str.upper().isin({"TEAM", "", "NAN"})].value_counts())
        _oreb_n = int(_their["offensive"].sum())
        if len(_their):
            _oreb_pct = 100 * _oreb_n / len(_their)
            _reb_summary_rows.append({"metric": "their_oreb_pct", "value": round(_oreb_pct, 1),
                                      "paired": len(_their), "player": None, "count": _oreb_n})
            for _pl, _c in _crashers.head(3).items():
                _reb_summary_rows.append({"metric": "their_oreb_player", "value": round(100 * _c / _oreb_n, 1)
                                          if _oreb_n else None, "paired": len(_their), "player": _pl,
                                          "count": int(_c)})
            if len(_their) >= _R["min_paired"] and _oreb_pct >= _R["crash_rate"]:
                _who = ", ".join(f"{p} ({int(c)})" for p, c in _crashers.head(2).items())
                _ktv_add("", "Finish every defensive possession with a box-out",
                         f"They grab {_oreb_pct:.0f}% of their own misses ({_oreb_n} of {len(_their)} paired "
                         f"misses in their play-by-play)" + (f"; most by {_who}." if _who else "."),
                         f"At {_oreb_pct:.0f}% they turn a real share of misses into second chances -- a stop "
                         "isn't a stop until the rebound is secured. Put a body on "
                         + (_crashers.index[0] if len(_crashers) else "their bigs") + " on every shot.",
                         "Data-Driven", "Defense")
        if len(_opp):
            _dreb_pct = 100 * (~_opp["offensive"]).sum() / len(_opp)
            _reb_summary_rows.append({"metric": "their_dreb_pct", "value": round(_dreb_pct, 1),
                                      "paired": len(_opp), "player": None, "count": int((~_opp["offensive"]).sum())})
            if len(_opp) >= _R["min_paired"] and _dreb_pct <= _R["soft_rate"]:
                _ktv_add("", "Send an extra body to the offensive glass",
                         f"Opponents rebound {100 - _dreb_pct:.0f}% of their own misses against them "
                         f"({int(_opp['offensive'].sum())} of {len(_opp)} paired misses).",
                         f"They secure only {_dreb_pct:.0f}% of defensive boards -- our misses are live balls. "
                         "Crash with a fourth man when the shot goes up from the perimeter.",
                         "Data-Driven", "Offense")
except Exception as _e:
    _ktv_problems.append(f"Data-Driven (rebounding): {_e}")

rebound_summary = pd.DataFrame(_reb_summary_rows, columns=["metric", "value", "paired", "player", "count"])
rebound_summary.insert(0, "opponent", _ktv_short)
rebound_summary.to_csv(os.path.join(APP_DATA_DIR, "uww_rebound_summary.csv"), index=False)

# ========================================================================================================
ktv_keys = pd.DataFrame(_ktv_rows)
if ktv_keys.empty:
    ktv_keys = pd.DataFrame(columns=_KTV_COLS)
else:
    ktv_keys["game_date"] = _ktv_game_date
    # Grouped by category, generator order preserved inside each group, then numbered across the whole
    # sorted list -- so a coach can cite "key 7" and everyone is looking at the same key.
    ktv_keys["_cat_rank"] = ktv_keys["category"].apply(
        lambda c: _KTV_CATEGORY_ORDER.index(c) if c in _KTV_CATEGORY_ORDER else len(_KTV_CATEGORY_ORDER))
    ktv_keys = ktv_keys.sort_values("_cat_rank", kind="stable").drop(columns="_cat_rank")
    ktv_keys["key_number"] = range(1, len(ktv_keys) + 1)
    ktv_keys = ktv_keys[_KTV_COLS]

ktv_keys.to_csv(os.path.join(APP_DATA_DIR, "uww_ktv_keys.csv"), index=False)
print(f"Wrote uww_ktv_keys.csv -- {len(ktv_keys)} key(s) for {_ktv_short or 'no opponent'}"
      + (f": {ktv_keys['category'].value_counts().to_dict()}" if not ktv_keys.empty else ""))
if _ktv_problems:
    # Printed, not swallowed: a key quietly missing from a coach's brief is the worse failure.
    print("  Generators that FAILED (their keys are missing from the table above):")
    for _p in _ktv_problems:
        print(f"    - {_p}")
if ktv_keys.empty and not _ktv_problems:
    print("  (No keys built. Expected with before_scout=\"yes\" for the written ones; the rest need the "
          "opponent's tagged prior games and this season's lineup stints.)")


  3-man combos: 208 ours, 171 Aurora Spartans -> uww_three_man_combos.csv
Wrote uww_ktv_keys.csv -- 13 key(s) for Aurora Spartans: {'Defense': 7, 'Offense': 4, 'Personnel': 2}
  Generators that FAILED (their keys are missing from the table above):
    - Lineup Scouting (counter lineup): cannot convert float NaN to integer



### Style matchups: teams like them, and teams like us

Builds `uww_style_matchups.csv` -- both halves of the app's STYLE MATCHUPS section, computed here so the brief and the app render one answer. One weighted seventeen-feature style model, run in two directions: target the upcoming opponent against the pool of teams UWW has played, and target UWW against the pool of teams the opponent has played. A match score means the same thing in both. See the cell's header comment for the one deliberate omission (the prior-season pool top-up).


In [164]:
# --- Style matchups: teams like them that we've played, and teams like us that have played them -------------
# Both halves of the app's STYLE MATCHUPS section, computed here and written to uww_style_matchups.csv so the
# brief, the app and anything else render one answer instead of each deriving their own.
#
# One model, two directions. The distance is a weighted RMS of robust z-score differences over the
# seventeen-feature style profile, and the match score is 100*exp(-0.7*d) -- so a 78 in one direction means
# exactly what a 78 means in the other. What changes is who is the target and who is the pool:
#   like_them   target = the upcoming opponent, pool = teams UWW has already played this season. Answers
#               "what has actually worked against this style".
#   like_us     target = UWW, pool = the teams the upcoming opponent has already played. Answers "what did
#               teams built like us produce against them".
#
# WHY THE PROFILES COME FROM BOX SCORES. Every team in the pool is described by reconstructed box scores, so
# describing the target from a scouting-report PDF instead would compare two things built by different
# methods and call the difference style. The upcoming opponent is measured from their OWN prior games rather
# than from the one meeting against UWW, since a body of work against several teams describes them better.
# Height and the shooter/post shares are the one block no box score carries; those come from the scouting
# reports, minutes-weighted across the rotation.
#
# SAMPLE SIZE IS REPORTED, NOT BAKED IN. Shrinking each team's rates toward the pool median was tried in the
# app and removed: the target has a season of prior games while most candidates have exactly one, so
# shrinkage systematically favoured whichever candidate happened to be sampled most like the target (a
# planted style-clone with identical raw rates came out 5.7 points apart on sample size alone). The distance
# runs on observed rates and each row carries a confidence instead.
#
# PORTING NOTE -- one deliberate omission. The app tops the like_them pool up with teams UWW played LAST
# season when this season is short of MIN_COMPARABLE_POOL candidates, reading them out of a sibling
# data_<season> archive folder. That archive lookup is an app concept (_discover_available_seasons); this
# parser has no multi-season view, so the pool here is current-season only. In the opening weeks of a season
# that means this table can be thinner than the app's panel, or empty when the app's is not. Everything else
# -- the feature spec, the weights, the scaling, the ranking, the confidence -- is copied verbatim.

import math as _sm_math

_SM_OUT = "uww_style_matchups"

# (key, display label, category, weight). Category weights: offense and defense 0.25 each because preparing
# for a team is equally about both; shot selection 0.15 kept separate from shooting SKILL because how often
# a team shoots threes is a schematic choice a game plan responds to while whether they make them is form;
# tempo 0.12; scoring level 0.10, deliberately low because how good they are is not how much they resemble
# someone; personnel 0.13, the only block that can't come from a box score. Inside the four factors, Dean
# Oliver's own weights.
_SM_FEATURE_SPEC = [
    ("off_efg",       "Their eFG%",             "Their offense",   0.100),
    ("off_tov",       "Their TOV%",             "Their offense",   0.0625),
    ("off_orb",       "Their ORB%",             "Their offense",   0.050),
    ("off_ftr",       "Their FT rate",          "Their offense",   0.0375),
    ("def_efg",       "eFG% they allow",        "Their defense",   0.100),
    ("def_tov",       "TOV% they force",        "Their defense",   0.0625),
    ("def_drb",       "DRB% they secure",       "Their defense",   0.050),
    ("def_ftr",       "FT rate they allow",     "Their defense",   0.0375),
    ("off_3par",      "Their 3PA rate",         "Shot selection",  0.060),
    ("off_astr",      "Their AST per made FG",  "Shot selection",  0.040),
    ("def_3par",      "3PA rate they allow",    "Shot selection",  0.050),
    ("pace",          "Pace (poss/40)",         "Tempo",           0.120),
    ("off_rtg",       "Points per 100",         "Scoring level",   0.050),
    ("def_rtg",       "Points allowed per 100", "Scoring level",   0.050),
    ("height_in",     "Avg height",             "Personnel",       0.070),
    ("share_shooter", "Shooter share",          "Personnel",       0.030),
    ("share_post",    "Post share",             "Personnel",       0.030),
]
_SM_WEIGHTS = {k: w for k, _, _, w in _SM_FEATURE_SPEC}
_SM_LABELS = {k: lbl for k, lbl, _, _ in _SM_FEATURE_SPEC}
_SM_CATEGORIES = {}
for _k, _lbl, _cat, _w in _SM_FEATURE_SPEC:
    _SM_CATEGORIES.setdefault(_cat, []).append(_k)

_SM_CONFIDENCE_GAMES = 2.0     # games at which a candidate's sample carries half weight in confidence
_SM_MIN_FEATURES = 5           # shared, discriminating features a match score needs to mean anything
_SM_TOP_K = 3
_SM_ROTATION_MIN_MPG = 8.0
_SM_JUNK_PLAYER_RE = (r"(?i)^(?:TEAM$|Commits |Turnover|Jump Ball|Subs In|Subs Out|Timeout|Official )"
                      r"|(?: Commits Foul$)")
_SM_DATE_COLS = ("game_date", "date", "iso_date")

_sm_rows = []
# The record behind each panel as NUMBERS (requested: the brief shows these panels only in THE BOTTOM LINE,
# and only when they matter -- which can't be decided by parsing strip_text). Filled per direction below.
_sm_dir_summary = {}
_sm_problems = []


def _sm_load(name):
    try:
        return pd.read_csv(os.path.join(APP_DATA_DIR, f"{name}.csv"))
    except Exception:
        return pd.DataFrame()


def _sm_pretty_date(value):
    """'2025-11-07' -> 'Fri, Nov 7'. Anything unparseable is returned as-is."""
    ts = pd.to_datetime(value, errors="coerce")
    if pd.isna(ts):
        return str(value)
    return f"{ts:%a}, {ts:%b} {ts.day}"


_SM_LOW_CONF = 0.35  # same amber/red cut the card colour uses


def _sm_low_conf_prefix(ranked):
    """Strip lead-in when EVERY card in a panel is red-confidence. A record over three one-game profiles is
    worth showing, but not as if it were a finding."""
    if ranked is None or ranked.empty or "confidence" not in ranked.columns:
        return ""
    conf = pd.to_numeric(ranked["confidence"], errors="coerce")
    if conf.notna().any() and float(conf.max()) < _SM_LOW_CONF:
        return "Low confidence -- every match here rests on a one- or two-game profile. "
    return ""


def _sm_date_col(df):
    for c in _SM_DATE_COLS:
        if c in df.columns:
            return c
    return None


def _sm_possessions(fga, oreb, to, fta):
    return fga - oreb + to + 0.44 * fta


def _sm_rate_profile(team_box, foe_box, games):
    """The fourteen box-derivable features for one team over one set of games.

    Everything here is a RATE: nothing changes when a team plays faster, which is the point -- pace is its
    own feature rather than a hidden contaminant. A component with no denominator returns None rather than
    0, because a zero would be read as a real value and count as agreement in the distance.
    """
    def tot(df, c):
        return pd.to_numeric(df[c], errors="coerce").sum() if c in df.columns else float("nan")

    cols = ("FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA", "OREB", "DREB", "AST", "TO", "PTS")
    o = {c: tot(team_box, c) for c in cols}
    d = {c: tot(foe_box, c) for c in cols}

    def safe(n, dn, scale=100.0):
        return (scale * n / dn) if (pd.notna(n) and pd.notna(dn) and dn > 0) else None

    off_poss = _sm_possessions(o["FGA"], o["OREB"], o["TO"], o["FTA"]) if pd.notna(o["FGA"]) else float("nan")
    def_poss = _sm_possessions(d["FGA"], d["OREB"], d["TO"], d["FTA"]) if pd.notna(d["FGA"]) else float("nan")
    # Averaging the two possession estimates is standard practice -- either side alone is a noisy
    # approximation of the same underlying number, and they should agree.
    if pd.notna(off_poss) and pd.notna(def_poss):
        poss = (off_poss + def_poss) / 2
    elif pd.notna(off_poss):
        poss = off_poss
    elif pd.notna(def_poss):
        poss = def_poss
    else:
        poss = None

    def _den(a, b):
        return (a + b) if (pd.notna(a) and pd.notna(b)) else float("nan")

    return {
        "off_efg": safe(o["FGM"] + 0.5 * o["FG3M"], o["FGA"]) if pd.notna(o["FGM"]) else None,
        "off_tov": safe(o["TO"], off_poss) if pd.notna(off_poss) else None,
        "off_orb": safe(o["OREB"], _den(o["OREB"], d["DREB"])),
        "off_ftr": safe(o["FTA"], o["FGA"]),
        "def_efg": safe(d["FGM"] + 0.5 * d["FG3M"], d["FGA"]) if pd.notna(d["FGM"]) else None,
        "def_tov": safe(d["TO"], def_poss) if pd.notna(def_poss) else None,
        # Stated as the share of available defensive rebounds THEY secured, so higher is better for them --
        # the same direction as every other feature, which keeps the distance interpretable.
        "def_drb": safe(o["DREB"], _den(o["DREB"], d["OREB"])),
        "def_ftr": safe(d["FTA"], d["FGA"]),
        "off_3par": safe(o["FG3A"], o["FGA"]),
        "off_astr": safe(o["AST"], o["FGM"]) if pd.notna(o["AST"]) else None,
        "def_3par": safe(d["FG3A"], d["FGA"]),
        "pace": (poss / games) if (poss is not None and games > 0) else None,
        "off_rtg": safe(o["PTS"], poss) if poss else None,
        "def_rtg": safe(d["PTS"], poss) if poss else None,
    }


def _sm_format_feature(key, value):
    """Display form for one feature. Every rate is already on a 0-100 scale, so the unit suffix is chosen
    by what the number MEANS rather than by its size."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return "-"
    try:
        value = float(value)
    except (TypeError, ValueError):
        return "-"
    if key == "height_in":
        return f"{int(value // 12)}'{int(round(value % 12))}\""
    if key.startswith("share_"):
        return f"{value * 100:.0f}%"
    if key in ("off_rtg", "def_rtg", "pace"):
        return f"{value:.1f}"
    if key == "off_astr":
        return f"{value / 100:.2f}"     # assists per made field goal, stored x100 by safe()
    if key.endswith(("_efg", "_tov", "_orb", "_drb", "_ftr", "_3par")):
        return f"{value:.1f}%"
    return f"{value:.1f}"


def _sm_robust_scale(frame):
    """Median/IQR z-scores, clipped to +/-3.

    Deliberately not min-max: that pins the range to the two most extreme teams, so one bad scrape squashes
    every real difference into a sliver. A feature with no spread becomes NaN rather than 0.0 -- writing 0.0
    makes every team identical on it and counts that as perfect agreement, which quietly inflates match
    scores toward 100 on features nobody varies on.
    """
    scaled = pd.DataFrame(index=frame.index)
    for column in frame.columns:
        values = pd.to_numeric(frame[column], errors="coerce")
        spread = (values.quantile(0.75) - values.quantile(0.25)) / 1.349
        if not spread or pd.isna(spread) or spread == 0:
            spread = values.std(ddof=0)
        if not spread or pd.isna(spread) or spread == 0:
            scaled[column] = float("nan")
            continue
        scaled[column] = ((values - values.median()) / spread).clip(-3, 3)
    return scaled


def _sm_rank(target, candidates, profiles, k=_SM_TOP_K):
    """Rank `candidates` by how closely their style resembles `target`.

    The total is divided by the weight ACTUALLY used, so a team missing a feature is neither rewarded nor
    punished for having fewer things to differ on -- but how much weight was available is reported, so a
    match built on half the profile is visibly that. Returns (ranked frame or None, thin flag).
    """
    keys = [k_ for k_, _, _, _ in _SM_FEATURE_SPEC]
    if profiles.empty or target not in profiles.index:
        return None, False
    pool = [o for o in dict.fromkeys(candidates) if o in profiles.index and o != target]
    if not pool:
        return None, False

    present = [c for c in keys if c in profiles.columns]
    scaled = _sm_robust_scale(profiles[present])
    target_row = scaled.loc[target]
    total_weight = sum(_SM_WEIGHTS[k_] for k_ in present)

    def games_for(name):
        if "_games" not in profiles.columns:
            return float("nan")
        v = pd.to_numeric(pd.Series([profiles.at[name, "_games"]]), errors="coerce").iloc[0]
        return float(v) if pd.notna(v) else float("nan")

    rows = []
    for opponent in pool:
        candidate = scaled.loc[opponent]
        weighted_sq = used_weight = 0.0
        per_category, scored_keys = {}, []
        for category, cat_keys in _SM_CATEGORIES.items():
            cat_sq = cat_weight = 0.0
            for key in [k_ for k_ in cat_keys if k_ in present]:
                a, b = target_row.get(key), candidate.get(key)
                if (pd.isna(a) or pd.isna(b)
                        or pd.isna(profiles.at[target, key]) or pd.isna(profiles.at[opponent, key])):
                    continue
                w = _SM_WEIGHTS[key]
                cat_sq += w * (a - b) ** 2
                cat_weight += w
                scored_keys.append(key)
            if cat_weight > 0:
                per_category[category] = (cat_sq / cat_weight) ** 0.5
                weighted_sq += cat_sq
                used_weight += cat_weight
        if used_weight <= 0:
            continue
        distance = (weighted_sq / used_weight) ** 0.5
        games = games_for(opponent)
        coverage = (used_weight / total_weight) if total_weight else 0.0
        rows.append({
            "opponent": opponent, "distance": distance,
            "match": int(round(100 * _sm_math.exp(-0.7 * distance))),
            "features_scored": len(set(scored_keys)),
            "weight_covered": coverage,
            "games": games,
            "confidence": (coverage * (games / (games + _SM_CONFIDENCE_GAMES))
                           if pd.notna(games) else float("nan")),
            **{f"cat::{c}": v for c, v in per_category.items()},
        })
    if not rows:
        return None, False

    ranked = pd.DataFrame(rows)
    solid = ranked[ranked["features_scored"] >= _SM_MIN_FEATURES]
    chosen = (solid if not solid.empty else ranked).sort_values("distance").head(k).reset_index(drop=True)
    return chosen, bool(solid.empty)


def _sm_card_fields(rank_row, profiles, target_row, target_label):
    """The per-team detail the card shows: which categories are alike, which differs most, and the two
    features that separate them most with both teams' actual numbers -- so a coach can judge the match
    rather than trusting the score."""
    cats = {c.split("::", 1)[1]: rank_row[c] for c in rank_row.index
            if str(c).startswith("cat::") and pd.notna(rank_row.get(c))}
    alike = sorted(cats, key=cats.get)[:2]
    differs = sorted(cats, key=cats.get, reverse=True)[:1]
    gaps = []
    for key, label, _, _ in _SM_FEATURE_SPEC:
        a, b = target_row.get(key), profiles.loc[rank_row["opponent"]].get(key)
        if pd.isna(a) or pd.isna(b):
            continue
        gaps.append((abs(float(a) - float(b)) / (abs(float(a)) + 1e-6), key, label, a, b))
    gap_lines = [f"{label}: {_sm_format_feature(key, b)} vs {_sm_format_feature(key, a)} ({target_label})"
                 for _, key, label, a, b in sorted(gaps, reverse=True)[:2]]
    return ", ".join(alike), ", ".join(differs), " | ".join(gap_lines)


def _sm_add(direction, rank, name, rank_row, profiles, target_row, target_label, meetings,
            strip_label, strip_text, thin):
    alike, differs, gaps = _sm_card_fields(rank_row, profiles, target_row, target_label)
    _sm_rows.append({
        "opponent": upcoming_opponent_short,
        "direction": direction,
        "rank": rank,
        "team": name,
        "match": int(rank_row["match"]),
        "features_scored": int(rank_row["features_scored"]),
        "total_features": len(_SM_FEATURE_SPEC),
        "weight_covered": round(float(rank_row["weight_covered"]), 4),
        "games": (int(rank_row["games"]) if pd.notna(rank_row.get("games")) else None),
        "confidence": (round(float(rank_row["confidence"]), 4)
                       if pd.notna(rank_row.get("confidence")) else None),
        "alike": alike,
        "differs": differs,
        "biggest_gaps": gaps,
        "meetings": " | ".join(meetings),
        # Repeated on every row of a direction rather than kept in a second table: one row is enough to
        # render the panel, and a summary that can go missing when a row is filtered out is worse.
        "strip_label": strip_label,
        "strip_text": strip_text,
        "thin": thin,
    })


# ========================================================================================================
# Shared profile table
# ========================================================================================================
_sm_short = upcoming_opponent_short
_sm_box = _sm_load("uww_pbp_box_score")
_sm_prior = _sm_load("uww_opponent_prior_games_box_score")
_sm_sched = _sm_load("uww_schedule")

# Rate profiles for every opponent UWW has PLAYED, from the reconstructed box scores. These describe how
# that team played AGAINST UWW -- arguably the most relevant sample for a style comparison, but not their
# season at large, which is stated on the panel rather than buried.
_sm_records = {}
if not _sm_box.empty and {"team", "opponent"}.issubset(_sm_box.columns):
    _sm_bcol = _sm_date_col(_sm_box)
    for _opp, _grp in _sm_box.groupby("opponent"):
        _them, _us = _grp[_grp["team"] != "UW-Whitewater"], _grp[_grp["team"] == "UW-Whitewater"]
        if _them.empty or _us.empty:
            continue
        _games = int(_them[_sm_bcol].nunique()) if _sm_bcol else 1
        _rec = _sm_rate_profile(_them, _us, _games or 1)
        _rec["_games"] = _games or 1
        _sm_records[_opp] = _rec

# The upcoming opponent's own prior games win over "how they looked against UWW" if both exist -- a body of
# work against several teams describes them better than one meeting.
if not _sm_prior.empty and "team" in _sm_prior.columns and _sm_short:
    _sm_pcol = _sm_date_col(_sm_prior)
    _them = _sm_prior[_sm_prior["team"].astype(str) == str(_sm_short)]
    _foes = _sm_prior[_sm_prior["team"].astype(str) != str(_sm_short)]
    if not _them.empty and not _foes.empty:
        _games = int(_them[_sm_pcol].nunique()) if _sm_pcol else 1
        _rec = _sm_rate_profile(_them, _foes, _games or 1)
        _rec["_games"] = _games or 1
        _sm_records[_sm_short] = _rec

# Height and style shares -- the three features no box score can produce, minutes-weighted across the
# rotation so a 30-minute starter counts for more than a 4-minute reserve.
_sm_people = {}
_sm_prof_tbl = _sm_load("uww_player_profiles")
if not _sm_prof_tbl.empty and "opponent" in _sm_prof_tbl.columns:
    for _opp, _grp in _sm_prof_tbl.groupby("opponent"):
        if "name" in _grp.columns:
            _grp = _grp[~_grp["name"].astype(str).str.contains(_SM_JUNK_PLAYER_RE, na=False, regex=True)]
        if _grp.empty:
            continue
        _mins = (pd.to_numeric(_grp["MIN"], errors="coerce").fillna(0.0)
                 if "MIN" in _grp.columns else pd.Series(0.0, index=_grp.index))
        _rot = _mins >= _SM_ROTATION_MIN_MPG
        if not _rot.any():
            _rot = _mins > 0
        _wts = _mins.where(_rot, 0.0)
        _total = float(_wts.sum())
        _hts = (pd.to_numeric(_grp["height_inches"], errors="coerce")
                if "height_inches" in _grp.columns else pd.Series(dtype=float))
        _usable = (_hts.notna() & (_wts > 0)) if not _hts.empty else pd.Series(False, index=_grp.index)
        _entry = {"height_in": (float((_hts[_usable] * _wts[_usable]).sum() / _wts[_usable].sum())
                                if _usable.any() else None)}
        _tags = (_grp["notes_tags_display"].astype(str)
                 if "notes_tags_display" in _grp.columns else pd.Series("", index=_grp.index))
        for _sk, _tag in (("share_shooter", "three_point_shooter"), ("share_post", "post_scorer")):
            _has = _tags.str.contains(_tag, na=False)
            _entry[_sk] = float(_wts[_has].sum() / _total) if _total > 0 else None
        _sm_people[_opp] = _entry

_sm_profiles = pd.DataFrame.from_dict(_sm_records, orient="index") if _sm_records else pd.DataFrame()
if _sm_people:
    _sm_ppl = pd.DataFrame.from_dict(_sm_people, orient="index")
    _sm_profiles = _sm_profiles.join(_sm_ppl, how="outer") if not _sm_profiles.empty else _sm_ppl
if not _sm_profiles.empty:
    for _k in _SM_WEIGHTS:
        if _k not in _sm_profiles.columns:
            _sm_profiles[_k] = None
    if "_games" not in _sm_profiles.columns:
        _sm_profiles["_games"] = 1
    _sm_profiles["_games"] = pd.to_numeric(_sm_profiles["_games"], errors="coerce").fillna(1)
    _sm_profiles = _sm_profiles.replace([float("inf"), float("-inf")], pd.NA)

# ========================================================================================================
# Direction 1: teams like THEM that we have played
# ========================================================================================================
try:
    if _sm_profiles.empty or not _sm_short or _sm_short not in _sm_profiles.index:
        _sm_problems.append(f"like_them: no style profile could be built for {_sm_short or 'the opponent'}")
    else:
        # UWW's completed games before the upcoming one. Scoped this way so a result that hasn't happened
        # yet can never appear here.
        _sm_uww_sched = _sm_sched[_sm_sched["team"].astype(str).str.contains("Whitewater", case=False, na=False)] \
            if not _sm_sched.empty and "team" in _sm_sched.columns else pd.DataFrame()
        _sm_played = _sm_uww_sched[_sm_uww_sched["outcome"].notna() & _sm_uww_sched["team_score"].notna()] \
            if not _sm_uww_sched.empty and {"outcome", "team_score"}.issubset(_sm_uww_sched.columns) \
            else pd.DataFrame()

        # Map each played game onto the name the profile table uses, keeping EVERY meeting -- a
        # home-and-home is two separate results and the split may be the most interesting thing about it.
        _sm_index = sorted([str(n) for n in _sm_profiles.index], key=len, reverse=True)
        _sm_games_by = {}
        for _, _g in _sm_played.iterrows():
            _full = str(_g.get("opponent", "")).strip()
            _hit = next((n for n in _sm_index if _full.startswith(n)), None)
            if _hit and _hit != _sm_short:
                _sm_games_by.setdefault(_hit, []).append(_g)

        _sm_ranked, _sm_thin = _sm_rank(_sm_short, list(_sm_games_by), _sm_profiles)
        if _sm_ranked is None or _sm_ranked.empty:
            _sm_problems.append(
                f"like_them: no usable comparison against the {len(_sm_games_by)} previously-played "
                f"opponent(s) -- most likely their profiles are too thin to share features with "
                f"{_sm_short}")
        else:
            # What actually happened against this style -- the reason the panel exists.
            _sm_rows_played = [g for n in _sm_ranked["opponent"] for g in _sm_games_by.get(n, [])]
            _sm_strip = ""
            if _sm_rows_played:
                _sm_df = pd.DataFrame(_sm_rows_played)
                _w = int((_sm_df["outcome"] == "W").sum())
                _l = int((_sm_df["outcome"] == "L").sum())
                _pf = pd.to_numeric(_sm_df["team_score"], errors="coerce").mean()
                _pa = pd.to_numeric(_sm_df["opponent_score"], errors="coerce").mean()
                # CONFIRMED BUG (fixed here): the comparison was against UWW's average over ALL games, and
                # the matched teams are part of that average. With three games played and all three matched,
                # it printed "+0.0 pts scored, +0.0 allowed" -- guaranteed by construction, and read as
                # "this style makes no difference". Compared against the games NOT in the matched set
                # instead; when there aren't any, the delta is left off and the strip says why.
                _matched_names = set(_sm_ranked["opponent"])
                _rest = [g for n, gs in _sm_games_by.items() if n not in _matched_names for g in gs]
                _delta = ""
                if _rest:
                    _rest_df = pd.DataFrame(_rest)
                    _rest_pf = pd.to_numeric(_rest_df["team_score"], errors="coerce").mean()
                    _rest_pa = pd.to_numeric(_rest_df["opponent_score"], errors="coerce").mean()
                    if pd.notna(_rest_pf) and pd.notna(_rest_pa):
                        _delta = (f" ({_pf - _rest_pf:+.1f} pts scored, {_pa - _rest_pa:+.1f} allowed vs our "
                                  f"other {len(_rest)} game{'s' if len(_rest) != 1 else ''})")
                else:
                    _delta = (" -- that is every game we have played, so there is no other style to "
                              "compare it against yet")
                _sm_strip = (f"UWW is {_w}-{_l} against these {len(_sm_ranked)} teams, averaging "
                             f"{_pf:.1f} scored and {_pa:.1f} allowed{_delta}.")
                _sm_dir_summary["like_them"] = {"dir_wins": _w, "dir_losses": _l, "dir_pf": round(float(_pf), 1),
                                                "dir_pa": round(float(_pa), 1)}

                _sm_strip = _sm_low_conf_prefix(_sm_ranked) + _sm_strip

            _sm_target_row = _sm_profiles.loc[_sm_short]
            for _i, (_, _r) in enumerate(_sm_ranked.iterrows(), start=1):
                _name = _r["opponent"]
                _meetings = []
                for _g in _sm_games_by.get(_name, []):
                    _sc = ""
                    if pd.notna(_g.get("team_score")) and pd.notna(_g.get("opponent_score")):
                        _sc = f" {int(_g['team_score'])}-{int(_g['opponent_score'])}"
                    _meetings.append(f"{_g.get('outcome', '')}{_sc}"
                                     + (f" ({_g.get('date')})" if _g.get("date") else ""))
                _sm_add("like_them", _i, _name, _r, _sm_profiles, _sm_target_row,
                        _sm_short, _meetings, "How we did against this style", _sm_strip, _sm_thin)
except Exception as _e:
    _sm_problems.append(f"like_them: {_e}")

# ========================================================================================================
# Direction 2: teams like US that have played them
# ========================================================================================================
# These profiles cannot come from the shared table above: that is built from scouted opponents, and the
# upcoming opponent's other opponents were never scouted. What does exist is the reconstructed box score of
# each of those games, so each team is profiled from the ONE game they played them -- a real limitation,
# which is why the Personnel rows are blank for them and the confidence figure is low.
try:
    if _sm_prior.empty or _sm_box.empty or not _sm_short:
        _sm_problems.append("like_us: needs the opponent's prior-game box scores and UWW's own")
    else:
        _sm_key = _sm_date_col(_sm_prior)
        _sm_ctx, _sm_recs = {}, {}
        for _team, _g in _sm_prior.groupby("team", dropna=True):
            if str(_team) == str(_sm_short):
                continue
            # Multiple meetings with the same team pool into one profile, exactly as they do above.
            _dates = _g[_sm_key].dropna().unique() if _sm_key else []
            _them = (_sm_prior[(_sm_prior["team"].astype(str) == str(_sm_short))
                               & (_sm_prior[_sm_key].isin(_dates))] if len(_dates)
                     else _sm_prior[_sm_prior["team"].astype(str) == str(_sm_short)])
            if _them.empty:
                continue
            _n = int(len(_dates)) or 1
            _rec = _sm_rate_profile(_g, _them, _n)
            _rec["_games"] = _n
            _sm_recs[str(_team)] = _rec

            # Pooling games is right for the PROFILE (more possessions, less noise) and wrong for the
            # RESULT, which is per game by definition -- summing PTS across three meetings produced an
            # impossible "L 225-231" scoreline. Each meeting is scored on its own.
            _meets = []
            for _d in (_dates if len(_dates) else [None]):
                _side = _g if _d is None else _g[_g[_sm_key] == _d]
                _oside = _them if _d is None else _them[_them[_sm_key] == _d]
                _a = float(pd.to_numeric(_side.get("PTS"), errors="coerce").sum())
                _b = float(pd.to_numeric(_oside.get("PTS"), errors="coerce").sum())
                if _a or _b:
                    _meets.append({"pts": _a, "their_pts": _b, "won": _a > _b, "date": _d})
            if _meets:
                _sm_ctx[str(_team)] = {"meetings": _meets}

        # UWW's own profile, built by the same function from the same kind of source.
        _sm_us = _sm_box[_sm_box["team"] == "UW-Whitewater"]
        _sm_foes = _sm_box[_sm_box["team"] != "UW-Whitewater"]
        _sm_ucol = _sm_date_col(_sm_us)
        _sm_ugames = int(_sm_us[_sm_ucol].nunique()) if (_sm_ucol and not _sm_us.empty) else 0
        if _sm_recs and not _sm_us.empty and _sm_ugames:
            _rec = _sm_rate_profile(_sm_us, _sm_foes, _sm_ugames)
            _rec["_games"] = _sm_ugames
            _sm_recs["UW-Whitewater"] = _rec

        if len(_sm_recs) < 2:
            _sm_problems.append(f"like_us: not enough reconstructed box scores from {_sm_short}'s prior games")
        else:
            _sm_tl_profiles = pd.DataFrame.from_dict(_sm_recs, orient="index")
            for _k in _SM_WEIGHTS:
                if _k not in _sm_tl_profiles.columns:
                    _sm_tl_profiles[_k] = None
            _sm_tl_ranked, _sm_tl_thin = _sm_rank("UW-Whitewater", list(_sm_ctx), _sm_tl_profiles)
            if _sm_tl_ranked is None or _sm_tl_ranked.empty:
                _sm_problems.append(f"like_us: none of {_sm_short}'s prior opponents could be compared "
                                    f"against UWW on a usable set of features")
            else:
                # Averaged over GAMES, not over teams -- a team met three times contributes three games to
                # the record and to the averages, which is what the question actually asks.
                _top = [m for n in _sm_tl_ranked["opponent"] for m in _sm_ctx.get(n, {}).get("meetings", [])]
                _all = [m for c in _sm_ctx.values() for m in c.get("meetings", [])]
                _w = sum(1 for m in _top if m["won"])
                _sm_dir_summary["like_us"] = {"dir_wins": _w, "dir_losses": len(_top) - _w,
                                              "dir_pf": round(sum(m["pts"] for m in _top) / len(_top), 1) if _top else None,
                                              "dir_pa": round(sum(m["their_pts"] for m in _top) / len(_top), 1) if _top else None}
                _pf = (sum(m["pts"] for m in _top) / len(_top)) if _top else 0
                _pa = (sum(m["their_pts"] for m in _top) / len(_top)) if _top else 0
                _field = (sum(m["pts"] for m in _all) / len(_all)) if _all else 0
                _strip = (_sm_low_conf_prefix(_sm_tl_ranked)
                          + f"They went {_w}-{len(_top) - _w} against {_sm_short} in {len(_top)} game(s), "
                          f"averaging {_pf:.1f} scored and {_pa:.1f} allowed. Across all {len(_all)} of "
                          f"their games, {_sm_short} allowed {_field:.1f} per game.")

                _sm_me_row = _sm_tl_profiles.loc["UW-Whitewater"]
                for _i, (_, _r) in enumerate(_sm_tl_ranked.iterrows(), start=1):
                    _meets = _sm_ctx.get(_r["opponent"], {}).get("meetings", [])
                    # Dates formatted like the like_them panel ("Fri, Nov 7") -- the two panels sit on the
                    # same page and printed ISO dates on one side and weekday dates on the other.
                    _meetings = [f"{'W' if m['won'] else 'L'} {int(m['pts'])}-{int(m['their_pts'])}"
                                 + (f" ({_sm_pretty_date(m['date'])})" if m.get("date") else "") for m in _meets]
                    _sm_add("like_us", _i, _r["opponent"], _r, _sm_tl_profiles, _sm_me_row,
                            "UWW", _meetings, "How teams like us did against them", _strip, _sm_tl_thin)
except Exception as _e:
    _sm_problems.append(f"like_us: {_e}")

# ========================================================================================================
style_matchups = pd.DataFrame(_sm_rows)
for _dcol in ("dir_wins", "dir_losses", "dir_pf", "dir_pa"):
    style_matchups[_dcol] = (style_matchups["direction"].map(lambda d: _sm_dir_summary.get(d, {}).get(_dcol))
                             if not style_matchups.empty else None)
if style_matchups.empty:
    style_matchups = pd.DataFrame(columns=[
        "opponent", "direction", "rank", "team", "match", "features_scored", "total_features",
        "weight_covered", "games", "confidence", "alike", "differs", "biggest_gaps", "meetings",
        "strip_label", "strip_text", "thin"])
style_matchups.to_csv(os.path.join(APP_DATA_DIR, f"{_SM_OUT}.csv"), index=False)

print(f"Wrote {_SM_OUT}.csv -- {len(style_matchups)} matched team(s) for {_sm_short or 'no opponent'}"
      + (f": {style_matchups['direction'].value_counts().to_dict()}" if not style_matchups.empty else ""))
if not style_matchups.empty and style_matchups["thin"].any():
    print("  THIN: at least one direction's best candidate shares fewer than "
          f"{_SM_MIN_FEATURES} scoring features with the target. Those are the closest available, not a "
          f"confident match.")
if _sm_problems:
    # Printed, not swallowed -- a panel silently absent from a coach's brief is the worse failure.
    print("  Directions that produced nothing:")
    for _p in _sm_problems:
        print(f"    - {_p}")


Wrote uww_style_matchups.csv -- 6 matched team(s) for Aurora Spartans: {'like_them': 3, 'like_us': 3}



### Scouting notes and keys to defending

Builds `uww_scouting_notes.csv`: one row per subject per source, covering both players and five-man units. Coach-written notes from the `_scout.html` report are passed through verbatim; the data-driven notes, keys, strengths and weaknesses are generated from the opponent's own box scores. Players and units carry the same four fields so one renderer draws both sections. Every percentage read carries a volume floor.


In [166]:
# --- Scouting notes and keys to defending, derived from the opponent's own box scores ------------------------
# Writes uww_scouting_notes.csv: one row per SUBJECT per SOURCE, so the coach-written notes from the
# "_scout.html" report and the data-driven ones generated here sit side by side rather than one replacing
# the other. They answer the same question from different evidence and a coach should see both -- when they
# agree that is corroboration, and when they disagree that is the interesting part.
#
#   subject_type = "player"  one of their players
#   subject_type = "lineup"   one of their five-man units
#   source = "Coach"          lifted verbatim from the scouting report (uww_player_profiles / rosters)
#   source = "Data-Driven"    generated below from the reconstructed box scores
#
# Players and units carry the SAME four fields -- notes, keys, strengths, weaknesses -- so one renderer
# draws both and the two sections cannot drift into different shapes. That is why the lineup reads moved
# here from the brief: derived content belongs in the parser, and a read that exists in only one consumer
# is a read the app can never show.
#
# Notes DESCRIBE, keys PRESCRIBE. That split is deliberate: a note is what the numbers say about a player,
# a key is what to do about it on Wednesday. Each key traces back to a note rather than being a separate
# opinion, so a coach can see why the instruction is there.
#
# WHAT THIS WILL NOT DO. Every read is gated on volume, and the phrasing never claims more than the sample
# supports. A player who is 2-of-4 from three is not a shooter; saying he is gets someone run off the line
# for no reason and leaves the paint open. Where the numbers say nothing, the player gets no line rather
# than a filler sentence -- an empty entry is a real answer and reads as one.
#
# Stated against the opponent's OWN team, not a league average: there is no league baseline in this data,
# and inventing one would put a number on the page that nothing here can support. "Takes a quarter of their
# shots" is a claim these tables can actually make.

_PN_OUT = "uww_scouting_notes"

# Volume floors. Low on purpose -- this is a handful of games, not a season -- so they exclude noise
# rather than demanding a big sample.
_PN_MIN_FGA = 12      # field-goal attempts before efficiency or shot profile is worth a word
_PN_MIN_3PA = 6       # three-point attempts before a percentage means anything
_PN_MIN_FTA = 8       # free-throw attempts before a foul-shooting read is worth a word
_PN_MIN_PTS_SHARE = 0.18   # share of team scoring that makes someone a primary option
# CONFIRMED BUG (fixed here): Mekhi Doby played ONE of Aurora's four games and came out with "Inefficient",
# "Gets to the line (13.0 FTA/gm)" and "Let him be the one who shoots it" -- one night's box score presented
# as a scouting identity. A player now needs _PN_MIN_GAME_SHARE of the team's games on film before any
# PRESCRIPTIVE key is written for him; the descriptive notes stay, prefixed with how thin the sample is.
_PN_MIN_GAME_SHARE = 0.5
# Same problem for five-man units: "Their weakest group -- push tempo" off 5.0 minutes together. Below this
# many minutes a unit gets its minutes line and nothing else.
_PN_LU_MIN_MINUTES = 8.0
# Player-level reference to the decoded play-call tags (see the "Play calls" cell): a set needs this many
# tagged uses BY this player before it's worth naming -- low on purpose, same reasoning as the floors above.
_PN_MIN_PLAY_USES = 3

_pn_rows = []
_pn_problems = []


def _pn_load(name):
    try:
        return pd.read_csv(os.path.join(APP_DATA_DIR, f"{name}.csv"))
    except Exception:
        return pd.DataFrame()


def _pn_text(value):
    text = "" if value is None else str(value).strip()
    return "" if text.lower() in ("nan", "none") else text


def _pn_player_reads(p, table, top_pts, top_oreb, top_ast):
    """(strengths, weaknesses) for one player as short phrases.

    Short labels a coach scans, as opposed to the notes above, which are full sentences that explain. Both
    are generated from the same totals so they can never contradict each other -- the split is how much
    room the reader has, not which numbers were used.
    """
    g = max(float(p["games"]), 1.0)
    fga, f3a, fta = float(p["FGA"]), float(p["FG3A"]), float(p["FTA"])
    strengths, weaknesses = [], []

    if p["player"] == top_pts and float(p["PTS"]) > 0:
        strengths.append(f"Leading scorer ({float(p['PTS']) / g:.1f} ppg)")
    if p["player"] == top_oreb and float(p["OREB"]) / g >= 1.5:
        strengths.append(f"Offensive glass ({float(p['OREB']) / g:.1f} orpg)")
    if p["player"] == top_ast and float(p["AST"]) / g >= 2:
        strengths.append(f"Primary creator ({float(p['AST']) / g:.1f} apg)")
    if float(table["REB"].max()) > 0 and float(p["REB"]) == float(table["REB"].max()) and float(p["REB"]) / g >= 4:
        strengths.append(f"Leads them on the glass ({float(p['REB']) / g:.1f} rpg)")
    if float(table["STL"].max()) > 0 and float(p["STL"]) == float(table["STL"].max()) and float(p["STL"]) / g >= 1.2:
        strengths.append(f"Active hands ({float(p['STL']) / g:.1f} spg)")
    if float(table["BLK"].max()) > 0 and float(p["BLK"]) == float(table["BLK"].max()) and float(p["BLK"]) / g >= 0.8:
        strengths.append(f"Rim protection ({float(p['BLK']) / g:.1f} bpg)")

    if f3a >= _PN_MIN_3PA:
        pct = 100 * float(p["FG3M"]) / f3a
        if pct >= 35:
            strengths.append(f"Shooter \u2014 {pct:.0f}% on {int(f3a)} threes")
        elif pct <= 25:
            weaknesses.append(f"Cold from three \u2014 {pct:.0f}% on {int(f3a)}")
    if fga >= _PN_MIN_FGA:
        pct = 100 * float(p["FGM"]) / fga
        if pct >= 50:
            strengths.append(f"Efficient \u2014 {pct:.0f}% from the field")
        elif pct <= 35:
            weaknesses.append(f"Inefficient \u2014 {pct:.0f}% on {int(fga)} shots")
    if fta >= _PN_MIN_FTA:
        if fta / g >= 4:
            strengths.append(f"Gets to the line ({fta / g:.1f} FTA/gm)")
        ft_pct = 100 * float(p["FTM"]) / fta
        if ft_pct <= 60:
            weaknesses.append(f"Poor at the line \u2014 {ft_pct:.0f}%")
    if float(p["TO"]) / g >= 2.5 and float(p["TO"]) >= float(p["AST"]):
        weaknesses.append(f"Gives it away ({float(p['TO']) / g:.1f} TO vs {float(p['AST']) / g:.1f} AST)")
    if float(p["PF"]) / g >= 3.2:
        weaknesses.append(f"Foul-prone ({float(p['PF']) / g:.1f} pf/gm)")
    return strengths, weaknesses


try:
    _pn_short = upcoming_opponent_short
    _pn_box = _pn_load("uww_opponent_prior_games_box_score")
    # Loaded here, not inside Part B's branch below, so it's always defined -- Part C (five-man units)
    # reads it too, and needs a safe empty default on any path where Part B's box-score branch is skipped.
    _pn_calls_ok_df = pd.DataFrame()

    # ---- A. Coach-written notes, straight from the scouting report ------------------------------------
    # Not regenerated or reworded here. A coach's note is evidence in its own right and gets passed
    # through exactly as written.
    _pn_seen = set()
    for _tbl in ("uww_player_profiles", "uww_opponent_rosters"):
        _t = _pn_load(_tbl)
        if _t.empty or not {"opponent", "name"}.issubset(_t.columns) or not _pn_short:
            continue
        for _, _p in _t[_t["opponent"].astype(str) == str(_pn_short)].iterrows():
            _nm = _pn_text(_p.get("name"))
            _notes = _pn_text(_p.get("player_notes"))
            _keys = _pn_text(_p.get("keys_to_defending"))
            if not _nm or (not _notes and not _keys) or _nm in _pn_seen:
                continue
            _pn_seen.add(_nm)
            _pn_rows.append({"opponent": _pn_short, "subject_type": "player", "name": _nm,
                             "source": "Coach", "notes": _notes, "keys_to_defending": _keys,
                             "strengths": "", "weaknesses": ""})

    # ---- B. Data-driven notes and keys ----------------------------------------------------------------
    if _pn_box.empty or "team" not in _pn_box.columns or not _pn_short:
        _pn_problems.append("no prior-game box scores for the upcoming opponent -- data-driven notes skipped")
    else:
        _pn_own = _pn_box[(_pn_box["team"].astype(str) == str(_pn_short))
                          & (_pn_box["player"].astype(str) != "TEAM")]
        # Per-player read of the decoded play-call tags (play_call/primary_action/play_location), joined onto
        # the opponent's clips in the "Play calls" cell. Loaded once, filtered per player inside the loop below.
        _pn_calls_all = _pn_load("uww_play_calls")
        _pn_calls_ok = (not _pn_calls_all.empty
                        and {"side", "scouted_opponent", "offense_team", "player", "play_call",
                             "decode_quality", "points"}.issubset(_pn_calls_all.columns))
        if _pn_calls_ok:
            _pn_calls_ok_df = _pn_calls_all[
                (_pn_calls_all["side"] == "Opponent")
                & (_pn_calls_all["scouted_opponent"].astype(str) == str(_pn_short))
                & (_pn_calls_all["offense_team"].astype(str) == str(_pn_short))
                & (_pn_calls_all["decode_quality"] != "Needs review")
            ]
        else:
            _pn_calls_ok_df = pd.DataFrame()
            _pn_problems.append("no uww_play_calls.csv (or missing columns) -- player play-call references skipped")
        if _pn_own.empty:
            _pn_problems.append(f"no box-score rows for {_pn_short} in their prior games")
        else:
            _pn_cols = ("PTS", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA",
                        "OREB", "DREB", "REB", "AST", "STL", "BLK", "TO", "PF")
            _pn_tot = _pn_own.groupby("player").apply(
                lambda g: pd.Series({c: pd.to_numeric(g[c], errors="coerce").sum()
                                     if c in g.columns else 0 for c in _pn_cols}
                                    | {"games": g["game_date"].nunique()
                                       if "game_date" in g.columns else len(g)})
            ).reset_index()

            _pn_team_pts = float(_pn_tot["PTS"].sum()) or 1.0
            _pn_team_fga = float(_pn_tot["FGA"].sum()) or 1.0
            _pn_top_oreb = _pn_tot.loc[_pn_tot["OREB"].idxmax()]["player"] if _pn_tot["OREB"].max() > 0 else None
            _pn_top_ast = _pn_tot.loc[_pn_tot["AST"].idxmax()]["player"] if _pn_tot["AST"].max() > 0 else None
            _pn_top_pts = _pn_tot.loc[_pn_tot["PTS"].idxmax()]["player"] if _pn_tot["PTS"].max() > 0 else None

            _pn_team_games = (int(_pn_own["game_date"].nunique()) if "game_date" in _pn_own.columns
                              else int(_pn_tot["games"].max()))
            for _, _p in _pn_tot.iterrows():
                _nm = str(_p["player"])
                _g = max(float(_p["games"]), 1.0)
                _thin = _pn_team_games > 1 and _g < _PN_MIN_GAME_SHARE * _pn_team_games
                _fga, _f3a, _fta = float(_p["FGA"]), float(_p["FG3A"]), float(_p["FTA"])
                _notes, _keys = [], []

                _pts_share = float(_p["PTS"]) / _pn_team_pts
                _fga_share = _fga / _pn_team_fga
                # True shooting puts threes and free throws on the same scale as twos, which raw FG%
                # cannot -- a 36% three-point shooter and a 50% finisher are equally efficient and FG%
                # would call one of them bad.
                _ts_den = 2 * (_fga + 0.44 * _fta)
                _ts = (100 * float(_p["PTS"]) / _ts_den) if _ts_den > 0 else None

                # --- usage ---
                if _pts_share >= _PN_MIN_PTS_SHARE or _nm == _pn_top_pts:
                    _notes.append(f"Primary option: {float(_p['PTS']) / _g:.1f} ppg, "
                                  f"{100 * _pts_share:.0f}% of their scoring on "
                                  f"{100 * _fga_share:.0f}% of their shots.")
                    _keys.append("Top priority \u2014 no easy catches, and make a second option beat you.")

                # --- shot profile ---
                if _fga >= _PN_MIN_FGA:
                    _three_rate = _f3a / _fga if _fga else 0
                    _three_pct = (100 * float(_p["FG3M"]) / _f3a) if _f3a > 0 else None
                    if _three_rate >= 0.45 and _f3a >= _PN_MIN_3PA:
                        _notes.append(f"Perimeter-first: {100 * _three_rate:.0f}% of his attempts are "
                                      f"threes ({int(_p['FG3M'])}-of-{int(_f3a)}, {_three_pct:.0f}%).")
                        if _three_pct is not None and _three_pct >= 35:
                            _keys.append("Run him off the line \u2014 close out short and make him put it "
                                         "on the floor.")
                        elif _three_pct is not None and _three_pct <= 28:
                            _keys.append("Live with the three \u2014 go under screens and help off him "
                                         "into the paint.")
                    elif _three_rate <= 0.15:
                        _notes.append(f"Interior player: only {100 * _three_rate:.0f}% of his attempts "
                                      f"come from three.")
                        _keys.append("Nothing easy at the rim \u2014 wall up early rather than reaching, "
                                     "and make him finish over a body.")

                    if _ts is not None:
                        if _ts >= 57:
                            _notes.append(f"Efficient: {_ts:.1f}% true shooting on {int(_fga)} attempts.")
                        elif _ts <= 45:
                            _notes.append(f"Inefficient: {_ts:.1f}% true shooting on {int(_fga)} attempts.")
                            _keys.append("Let him be the one who shoots it \u2014 volume without "
                                         "efficiency is a result we will take.")

                # --- free throws ---
                if _fta >= _PN_MIN_FTA:
                    _ft_rate = _fta / _fga if _fga else 0
                    _ft_pct = 100 * float(_p["FTM"]) / _fta
                    if _ft_rate >= 0.40:
                        _notes.append(f"Gets to the line: {_fta / _g:.1f} attempts a game "
                                      f"({_ft_rate:.2f} per field-goal attempt).")
                        _keys.append("Guard him without fouling \u2014 hands up, no bail-outs.")
                    if _ft_pct <= 60:
                        _notes.append(f"{_ft_pct:.0f}% from the line on {int(_fta)} attempts.")
                        _keys.append("If he has to be fouled late, he is the one to foul.")

                # --- glass, playmaking, giveaways, fouls ---
                if _nm == _pn_top_oreb and float(_p["OREB"]) / _g >= 1.5:
                    _notes.append(f"Their offensive glass: {float(_p['OREB']) / _g:.1f} offensive rebounds "
                                  f"a game, most on the team.")
                    _keys.append("Put a body on him every shot \u2014 he is where their second chances "
                                 "come from.")

                _ast, _to = float(_p["AST"]), float(_p["TO"])
                if _nm == _pn_top_ast and _ast / _g >= 2:
                    _notes.append(f"Initiates for them: {_ast / _g:.1f} assists a game, "
                                  f"{_ast / _to:.1f} per turnover." if _to > 0 else
                                  f"Initiates for them: {_ast / _g:.1f} assists a game.")
                    _keys.append("Get it out of his hands \u2014 deny the first pass and make someone "
                                 "else start the offense.")
                if _to / _g >= 2.5 and _to >= _ast:
                    _notes.append(f"Loose with it: {_to / _g:.1f} turnovers a game against "
                                  f"{_ast / _g:.1f} assists.")
                    _keys.append("Pressure him full-court and trap him off the catch.")

                _pf = float(_p["PF"])
                if _pf / _g >= 3.2:
                    _notes.append(f"Fouls: {_pf / _g:.1f} a game.")
                    _keys.append("Attack him early \u2014 two quick ones changes how he guards.")

                # --- his own tagged play calls (uww_plays.csv / opponent_plays.csv, decoded) ---------------
                if not _pn_calls_ok_df.empty:
                    _my_clips = _pn_calls_ok_df[_pn_calls_ok_df["player"].astype(str) == _nm]
                    if not _my_clips.empty:
                        _named = _my_clips[~_my_clips["play_call"].astype(str).str.contains("unspecified", na=False)
                                           & _my_clips["play_call"].notna()]
                        _call_counts = _named["play_call"].value_counts()
                        _top_calls = _call_counts[_call_counts >= _PN_MIN_PLAY_USES]
                        if not _top_calls.empty:
                            _pts = pd.to_numeric(_my_clips["points"], errors="coerce")
                            _known = int(_pts.notna().sum())
                            _ppp = (_pts.sum() / _known) if _known else None
                            _call_txt = ", ".join(f"{n} ({int(c)}x)" for n, c in _top_calls.head(3).items())
                            _notes.append(
                                f"Tagged film: featured in {_call_txt} out of {len(_my_clips)} tagged "
                                f"possession(s)" + (f" \u2014 {_ppp:.2f} PPP." if _ppp is not None else "."))
                            _top_set = _top_calls.index[0]
                            _set_clips = _my_clips[_my_clips["play_call"] == _top_set]
                            _loc = _set_clips["play_location"].dropna()
                            _act = _set_clips["primary_action"].dropna()
                            _where = (f" to the {_loc.mode().iloc[0].lower()}"
                                      if not _loc.empty and not _loc.mode().empty else "")
                            _how = (f" off {_act.mode().iloc[0]}"
                                    if not _act.empty and not _act.mode().empty
                                    and _act.mode().iloc[0] not in _top_set else "")
                            _keys.append(f"Know {_top_set}{_how}{_where} \u2014 his most-tagged set on film "
                                         f"({int(_top_calls.iloc[0])}x).")

                _strengths, _weaknesses = _pn_player_reads(_p, _pn_tot, _pn_top_pts, _pn_top_oreb,
                                                          _pn_top_ast)
                if _thin:
                    # Describe, don't prescribe: a key is an instruction a player will act on Wednesday.
                    if _notes or _strengths or _weaknesses:
                        _notes.insert(0, f"Only {int(_g)} of {_pn_team_games} games on film \u2014 provisional.")
                    _keys = []
                if _notes or _keys or _strengths or _weaknesses:
                    _pn_rows.append({
                        "opponent": _pn_short, "subject_type": "player", "name": _nm,
                        "source": "Data-Driven",
                        "notes": " ".join(_notes),
                        # Pipe-joined so a renderer can show each as its own line; kept in one field so
                        # the table stays one row per subject per source.
                        "keys_to_defending": " | ".join(_keys),
                        "strengths": " | ".join(_strengths),
                        "weaknesses": " | ".join(_weaknesses),
                    })

    # ---- C. Five-man units, same four fields as the players ------------------------------------------
    # Measured against the opponent's OWN lineup averages and stated as rates -- per 40 minutes or a
    # percentage. Raw totals would rank the units by playing time a second time, so the heaviest-used
    # unit would come out "best at everything" by construction.
    _pn_lu = _pn_load("uww_opp_lineup_season_box")
    if not _pn_lu.empty and "lineup" in _pn_lu.columns and "MIN" in _pn_lu.columns:
        _lu_min = pd.to_numeric(_pn_lu["MIN"], errors="coerce").fillna(0)
        _lu_total_min = float(_lu_min.sum())
        if _lu_total_min > 0:
            def _lu_num(col):
                return (pd.to_numeric(_pn_lu[col], errors="coerce").fillna(0)
                        if col in _pn_lu.columns else pd.Series(0.0, index=_pn_lu.index))

            def _lu_team_rate(col):
                return float(_lu_num(col).sum()) / _lu_total_min * 40

            _lu_team_fg = (100 * _lu_num("FGM").sum() / _lu_num("FGA").sum()
                           if _lu_num("FGA").sum() > 0 else None)
            _lu_team_3p = (100 * _lu_num("FG3M").sum() / _lu_num("FG3A").sum()
                           if _lu_num("FG3A").sum() > 0 else None)
            _lu_team_to40, _lu_team_reb40 = _lu_team_rate("TO"), _lu_team_rate("REB")
            _lu_team_ast40 = _lu_team_rate("AST")
            _lu_busiest = _pn_lu.loc[_lu_min.idxmax(), "lineup"]

            for _i, _u in _pn_lu.iterrows():
                _key = str(_u["lineup"])
                _mins = float(_lu_min.loc[_i])
                if _mins <= 0:
                    continue
                _notes, _keys, _str, _weak = [], [], [], []

                def _val(col):
                    return float(pd.to_numeric(pd.Series([_u.get(col, 0)]), errors="coerce").fillna(0).iloc[0])

                _gp = _val("GP")
                _notes.append(f"{_mins:.1f} minutes together"
                              + (f" over {int(_gp)} game(s)" if _gp else "")
                              + (" \u2014 their most-used unit." if _key == str(_lu_busiest) else "."))
                if _mins < _PN_LU_MIN_MINUTES:
                    _notes.append(f"Under {_PN_LU_MIN_MINUTES:.0f} minutes together \u2014 too few possessions "
                                  f"to read.")
                    _pn_rows.append({"opponent": _pn_short, "subject_type": "lineup", "name": _key,
                                     "source": "Data-Driven", "notes": " ".join(_notes),
                                     "keys_to_defending": "", "strengths": "", "weaknesses": ""})
                    continue

                _margin = _val("+/-")
                _per40 = _margin / _mins * 40
                if abs(_per40) >= 5:
                    _notes.append(f"Net {_per40:+.0f} per 40 minutes on the floor "
                                  f"({_margin:+.0f} in {_mins:.1f}).")
                    if _per40 > 0:
                        _str.append(f"{_per40:+.0f} per 40 on the floor")
                        _keys.append("Their best group \u2014 make sure our own best five is out there "
                                     "with them.")
                    else:
                        _weak.append(f"{_per40:+.0f} per 40 on the floor")
                        _keys.append("Their weakest group \u2014 push tempo and hunt shots while it is "
                                     "on the floor.")

                _fg, _fga = _u.get("FG%"), _val("FGA")
                if pd.notna(_fg) and _lu_team_fg and _fga >= _PN_MIN_FGA:
                    _fg = float(_fg)
                    if _fg - _lu_team_fg >= 4:
                        _str.append(f"Shoots it better than their norm ({_fg:.1f}% vs {_lu_team_fg:.1f}%)")
                    elif _lu_team_fg - _fg >= 4:
                        _weak.append(f"Cold unit ({_fg:.1f}% vs {_lu_team_fg:.1f}%)")

                _tp, _tpa = _u.get("3P%"), _val("FG3A")
                if pd.notna(_tp) and _lu_team_3p and _tpa >= _PN_MIN_3PA:
                    _tp = float(_tp)
                    if _tp - _lu_team_3p >= 5:
                        _str.append(f"Hits threes ({_tp:.1f}% on {int(_tpa)})")
                        _notes.append(f"Shoots {_tp:.1f}% from three on {int(_tpa)} attempts with this "
                                      f"group on the floor.")
                        _keys.append("Stay attached on the perimeter \u2014 no help off this group's "
                                     "shooters.")
                    elif _lu_team_3p - _tp >= 5:
                        _weak.append(f"Won't make threes ({_tp:.1f}% on {int(_tpa)})")
                        _keys.append("Pack the paint against this group and live with the three.")

                for _col, _team_value, _good, _bad, _higher, _key_good, _key_bad in (
                    ("REB", _lu_team_reb40, "Rebounds well", "Gets beaten on the glass", True,
                     None, "Crash the offensive glass while this unit is in."),
                    ("AST", _lu_team_ast40, "Moves the ball", "Stagnant \u2014 few assists", True,
                     None, "Switch and make this group create off the dribble."),
                    ("TO", _lu_team_to40, "Takes care of it", "Turnover-prone", False,
                     None, "Pressure this unit \u2014 they give it away."),
                ):
                    if _team_value is None:
                        continue
                    _rate = _val(_col) / _mins * 40
                    _gap = _rate - _team_value
                    if abs(_gap) < max(1.5, 0.15 * _team_value):
                        continue
                    _better = (_gap > 0) if _higher else (_gap < 0)
                    _phrase = f"{_good if _better else _bad} ({_rate:.1f} vs {_team_value:.1f} per 40)"
                    (_str if _better else _weak).append(_phrase)
                    _instruction = _key_good if _better else _key_bad
                    if _instruction:
                        _keys.append(_instruction)

                # --- this group's own tagged play calls (uww_plays.csv / opponent_plays.csv, decoded) -----
                # There's no lineup tag on a clip itself -- a clip names the possession's ball-handler or
                # finisher, not all five players on the floor -- so this pools every clip whose tagged player
                # is a MEMBER of this group, rather than claiming the five were confirmed on the floor
                # together for each one. Said explicitly in the note so it isn't read as more than it is.
                if _pn_calls_ok_df is not None and not _pn_calls_ok_df.empty:
                    _lu_members = [p.strip() for p in _key.split(",") if p.strip()]
                    _lu_clips = _pn_calls_ok_df[_pn_calls_ok_df["player"].astype(str).isin(_lu_members)]
                    if not _lu_clips.empty:
                        _lu_named = _lu_clips[~_lu_clips["play_call"].astype(str).str.contains("unspecified", na=False)
                                              & _lu_clips["play_call"].notna()]
                        _lu_call_counts = _lu_named["play_call"].value_counts()
                        _lu_top_calls = _lu_call_counts[_lu_call_counts >= _PN_MIN_PLAY_USES]
                        if not _lu_top_calls.empty:
                            _lu_pts = pd.to_numeric(_lu_clips["points"], errors="coerce")
                            _lu_known = int(_lu_pts.notna().sum())
                            _lu_ppp = (_lu_pts.sum() / _lu_known) if _lu_known else None
                            _lu_call_txt = ", ".join(f"{n} ({int(c)}x)" for n, c in _lu_top_calls.head(3).items())
                            _notes.append(
                                f"Tagged film across this group's players: {_lu_call_txt} out of "
                                f"{len(_lu_clips)} tagged possession(s)"
                                + (f" \u2014 {_lu_ppp:.2f} PPP." if _lu_ppp is not None else ".")
                                + " (each clip's tagged player is a member of this group -- not confirmed "
                                  "as all five on the floor together for every one.)")
                            _lu_top_set = _lu_top_calls.index[0]
                            _lu_set_clips = _lu_clips[_lu_clips["play_call"] == _lu_top_set]
                            _lu_loc = _lu_set_clips["play_location"].dropna()
                            _lu_act = _lu_set_clips["primary_action"].dropna()
                            _lu_where = (f" to the {_lu_loc.mode().iloc[0].lower()}"
                                        if not _lu_loc.empty and not _lu_loc.mode().empty else "")
                            _lu_how = (f" off {_lu_act.mode().iloc[0]}"
                                      if not _lu_act.empty and not _lu_act.mode().empty
                                      and _lu_act.mode().iloc[0] not in _lu_top_set else "")
                            _keys.append(f"Know {_lu_top_set}{_lu_how}{_lu_where} \u2014 this group's "
                                        f"most-tagged set on film ({int(_lu_top_calls.iloc[0])}x).")

                if _notes or _keys or _str or _weak:
                    _pn_rows.append({
                        "opponent": _pn_short, "subject_type": "lineup", "name": _key,
                        "source": "Data-Driven",
                        "notes": " ".join(_notes),
                        "keys_to_defending": " | ".join(_keys),
                        "strengths": " | ".join(_str),
                        "weaknesses": " | ".join(_weak),
                    })

except Exception as _e:
    _pn_problems.append(str(_e))

scouting_notes = pd.DataFrame(_pn_rows)
if scouting_notes.empty:
    scouting_notes = pd.DataFrame(columns=["opponent", "subject_type", "name", "source", "notes",
                                           "keys_to_defending", "strengths", "weaknesses"])
scouting_notes.to_csv(os.path.join(APP_DATA_DIR, f"{_PN_OUT}.csv"), index=False)

print(f"Wrote {_PN_OUT}.csv -- {len(scouting_notes)} row(s) for "
      f"{upcoming_opponent_short or 'no opponent'}"
      + (f": {scouting_notes.groupby(['subject_type', 'source']).size().to_dict()}"
         if not scouting_notes.empty else ""))
if _pn_problems:
    print("  Notes:")
    for _p in _pn_problems:
        print(f"    - {_p}")


Wrote uww_scouting_notes.csv -- 79 row(s) for Aurora Spartans: {('lineup', 'Data-Driven'): 70, ('player', 'Data-Driven'): 9}



### Game plan, practice plan and staff inputs (SAMPLE placeholders where the files don't cover it)

Writes one CSV per game-plan section so the app and the brief render the same thing:

* **Real, from the exported tables:** `uww_late_game_foul_list`, `uww_sample_size_warnings`, and the emphasis items inside `uww_practice_plan` (pulled from the Keys to Victory and player flags).
* **SAMPLE until the staff supplies them:** `uww_opp_sets`, `uww_opp_defense`, `uww_opp_personnel`, `uww_opp_shot_zones`, `uww_opp_tendencies`, `uww_matchups`, `uww_scout_team`, `uww_uww_availability`, `uww_film_clips`, and the practice calendar.

Every row carries `is_sample`. Renderers must draw sample rows inside a red **SAMPLE DATA** box. To replace a section with real input, fill the matching template from `OUTPUT_DIR/staff_input_templates/` and save it to `INPUT_DIR/staff_inputs/` — rows for the upcoming opponent replace the sample for that whole section.

In [168]:
# --- Game-plan and practice-plan tables, with SAMPLE placeholders for what the files don't carry yet --------
# The brief answers "who are they, statistically". A staff building Monday's practice also needs HOW they play
# (sets, special situations, defensive scheme), physical details (height, hand), availability, matchups, a
# scout-team plan and a practice calendar. None of that is in play-by-play or video-tagging exports.
#
# This cell writes one CSV per section so the app and the brief render the same thing. Every row carries
# `is_sample`:
#   is_sample = False   real -- derived from the exported tables, or typed in by the staff (see below)
#   is_sample = True    PLACEHOLDER generated here so the layout can be reviewed. Renderers MUST show these
#                       inside a red "SAMPLE DATA" box. They are shaped like the real thing (real player names,
#                       plausible values) precisely so nobody mistakes a blank for "nothing to scout" -- which
#                       is also why they must never be shown unmarked.
#
# REPLACING SAMPLES WITH REAL INPUT. Drop a CSV into INPUT_DIR/staff_inputs/ with the same name and columns as
# the template this cell writes to OUTPUT_DIR/staff_input_templates/ (minus is_sample). Rows whose `opponent`
# matches the upcoming opponent replace the sample rows for that section entirely. Nothing is blended: a
# section is either all staff input or all sample, so a red box never hides one real row among fake ones.
#
# REAL-DATA sections written here (never sample):
#   uww_late_game_foul_list   who to foul / not foul, from their own FT shooting with an attempt floor
#   uww_sample_size_warnings  every number in the brief that rests on a thin sample, in one place
#   uww_practice_plan         the EMPHASIS items are pulled from uww_ktv_keys and uww_coaching_flags (real);
#                             the calendar and minutes are sample until a staff calendar is supplied, and the
#                             row-level `schedule_is_sample` says so.
import hashlib as _gp_hash

_GP_STAFF_DIR = os.path.join(INPUT_DIR, "staff_inputs")
_GP_TEMPLATE_DIR = os.path.join(OUTPUT_DIR, "staff_input_templates")
_GP_UWW = "UW-Whitewater"
_GP_FOUL_MIN_FTA = 8          # FT attempts before a player goes on either foul list
_GP_THIN_GAME_SHARE = 0.5     # same floor as the scouting-notes cell
_GP_THIN_LINEUP_MIN = 8.0     # same floor as the scouting-notes / KTV cells
_gp_short = upcoming_opponent_short
_gp_problems = []


def _gp_load(name):
    try:
        return pd.read_csv(os.path.join(APP_DATA_DIR, f"{name}.csv"))
    except Exception:
        return pd.DataFrame()


def _gp_pick(seed, options):
    """Deterministic 'random' choice, so a sample doesn't change every run and look like new data."""
    h = int(_gp_hash.md5(str(seed).encode()).hexdigest(), 16)
    return options[h % len(options)]


def _gp_last(name):
    parts = str(name).split()
    return parts[-1] if parts else str(name)


def _gp_staff_rows(section, columns):
    """Staff-typed rows for this opponent, or None. A file that exists but has no rows for this opponent is
    treated as absent -- a staff sheet for last week's opponent must not suppress this week's sample."""
    path = os.path.join(_GP_STAFF_DIR, f"{section}.csv")
    if not os.path.exists(path):
        return None
    try:
        df = pd.read_csv(path)
    except Exception as e:
        _gp_problems.append(f"{section}: could not read staff input ({e}) -- using sample")
        return None
    if "opponent" in df.columns and _gp_short:
        df = df[df["opponent"].astype(str).str.strip() == str(_gp_short)]
    if df.empty:
        return None
    for c in columns:
        if c not in df.columns:
            df[c] = None
    df = df[columns].copy()
    df["is_sample"] = False
    return df


def _gp_write(section, columns, sample_rows, real_rows=None):
    """Staff input if present, else rows derived from real data (real_rows), else the sample rows. Also writes
    an empty template the staff can fill. Staff input wins over derived data: it's the staff's call."""
    os.makedirs(_GP_TEMPLATE_DIR, exist_ok=True)
    pd.DataFrame(columns=columns).to_csv(os.path.join(_GP_TEMPLATE_DIR, f"{section}.csv"), index=False)
    staff = _gp_staff_rows(section, columns)
    if staff is not None:
        out = staff
    elif real_rows:
        out = pd.DataFrame(real_rows, columns=columns)
        out["is_sample"] = False
    else:
        out = pd.DataFrame(sample_rows, columns=columns)
        out["is_sample"] = True
    out.to_csv(os.path.join(APP_DATA_DIR, f"uww_{section}.csv"), index=False)
    return out


# ---- per-player season lines, both sides --------------------------------------------------------------
def _gp_player_lines(box, team_value, invert=False):
    if box.empty or "team" not in box.columns:
        return pd.DataFrame()
    mask = box["team"].astype(str) == str(team_value)
    rows = box[~mask if invert else mask]
    rows = rows[rows["player"].astype(str) != "TEAM"]
    if rows.empty:
        return pd.DataFrame()
    cols = [c for c in ("PTS", "MIN", "FGA", "FG3M", "FG3A", "FTM", "FTA", "REB", "OREB", "AST", "BLK", "TO")
            if c in rows.columns]
    work = rows.copy()
    for c in cols:
        work[c] = pd.to_numeric(work[c], errors="coerce").fillna(0)
    agg = work.groupby("player")[cols].sum()
    agg["games"] = work.groupby("player")["game_date"].nunique() if "game_date" in work.columns else 1
    agg = agg.reset_index()
    # Minutes when we have them, points otherwise -- "who plays most" is the question for matchups.
    order = "MIN" if "MIN" in agg.columns and agg["MIN"].sum() > 0 else "PTS"
    return agg.sort_values(order, ascending=False).reset_index(drop=True)


_gp_prior = _gp_load("uww_opponent_prior_games_box_score")
_gp_box = _gp_load("uww_pbp_box_score")
_gp_opp = _gp_player_lines(_gp_prior, _gp_short)
_gp_us = _gp_player_lines(_gp_box, _GP_UWW)
_gp_team_games = (int(_gp_prior[_gp_prior["team"].astype(str) == str(_gp_short)]["game_date"].nunique())
                  if not _gp_prior.empty and "game_date" in _gp_prior.columns else 0)

# Role guesses used ONLY to make sample rows plausible (who a set is "for"). Not exported as facts.
_gp_shooter = _gp_big = _gp_driver = _gp_handler = None
if not _gp_opp.empty:
    _regular = _gp_opp[_gp_opp["games"] >= max(1, _GP_THIN_GAME_SHARE * max(_gp_team_games, 1))]
    _pool = _regular if not _regular.empty else _gp_opp
    _sh = _pool[_pool["FG3A"] >= 6].assign(_p=lambda d: d["FG3M"] / d["FG3A"])
    _gp_shooter = _sh.sort_values("_p", ascending=False)["player"].iloc[0] if not _sh.empty else _pool["player"].iloc[0]
    _gp_big = _pool.sort_values(["REB", "BLK"], ascending=False)["player"].iloc[0]
    _gp_driver = _pool.sort_values("FTA", ascending=False)["player"].iloc[0]
    _gp_handler = _pool.sort_values("AST", ascending=False)["player"].iloc[0]


# ======================================================================================================
# REAL: late-game foul list
# ======================================================================================================
# CONFIRMED BUG (fixed here): the attempt floor was applied to BOTH calls, so a player shooting 6-for-6 was
# left off the list entirely -- a blank cell, when the obvious read is "don't put him on the line". The two
# calls carry different risk, so they no longer share a threshold:
#   FOUL him       -- a deliberate act on our part, so it needs real evidence he is bad: _GP_FOUL_MIN_FTA
#                     attempts AND a shrunk estimate at or under _GP_FOUL_BAD_PCT.
#   DO NOT FOUL    -- avoiding a shooter costs us nothing if we're wrong, so a small but clean sample is
#                     enough: _GP_FOUL_MIN_FTA attempts at _GP_FOUL_GOOD_PCT or better, OR as few as
#                     _GP_FOUL_SMALL_FTA attempts when he hasn't missed much (_GP_FOUL_SMALL_PCT or better).
# Rates are shrunk toward a D3 baseline before the call is made, so 6-for-6 doesn't read as a true 100%
# shooter while still landing on the right side of the line; the raw makes-attempts pair is what's displayed.
_GP_FOUL_BAD_PCT = 62
_GP_FOUL_GOOD_PCT = 75
_GP_FOUL_SMALL_FTA = 4
_GP_FOUL_SMALL_PCT = 85
_GP_FOUL_SHRINK_N = 6      # prior weight, in attempts
_GP_FOUL_PRIOR_PCT = 70.0  # roughly the D3 men's free-throw average

_foul_rows = []
if not _gp_opp.empty and "FTA" in _gp_opp.columns:
    _ft = _gp_opp[_gp_opp["FTA"] >= min(_GP_FOUL_MIN_FTA, _GP_FOUL_SMALL_FTA)].copy()
    _ft["ft_pct"] = (100 * _ft["FTM"] / _ft["FTA"]).round(1)
    _ft["ft_pct_adj"] = ((100 * _ft["FTM"] + _GP_FOUL_PRIOR_PCT * _GP_FOUL_SHRINK_N)
                         / (_ft["FTA"] + _GP_FOUL_SHRINK_N)).round(1)
    for _, r in _ft.sort_values("ft_pct_adj").iterrows():
        _enough_to_foul = r["FTA"] >= _GP_FOUL_MIN_FTA
        _clean_small = r["FTA"] >= _GP_FOUL_SMALL_FTA and r["ft_pct"] >= _GP_FOUL_SMALL_PCT
        if _enough_to_foul and r["ft_pct_adj"] <= _GP_FOUL_BAD_PCT:
            _call = "Foul"
        elif (_enough_to_foul and r["ft_pct_adj"] >= _GP_FOUL_GOOD_PCT) or _clean_small:
            _call = "Do not foul"
        elif _enough_to_foul:
            _call = "Neutral"
        else:
            continue  # too few attempts to say anything either way
        _thin_ft = _gp_team_games > 1 and r["games"] < _GP_THIN_GAME_SHARE * _gp_team_games
        _note = f"Only {int(r['games'])} of {_gp_team_games} games on film" if _thin_ft else ""
        if _clean_small and r["FTA"] < _GP_FOUL_MIN_FTA:
            _note = (f"{int(r['FTM'])}-for-{int(r['FTA'])} -- small sample, but no reason to put him on the "
                     f"line" + (f"; {_note.lower()}" if _note else ""))
        _foul_rows.append({"opponent": _gp_short, "player": r["player"], "call": _call,
                           "ft_pct": r["ft_pct"], "ft_pct_adj": r["ft_pct_adj"],
                           "ftm": int(r["FTM"]), "fta": int(r["FTA"]),
                           "games": int(r["games"]), "note": _note})
late_game_foul_list = pd.DataFrame(_foul_rows, columns=["opponent", "player", "call", "ft_pct", "ft_pct_adj",
                                                        "ftm", "fta", "games", "note"])
late_game_foul_list["is_sample"] = False
late_game_foul_list.to_csv(os.path.join(APP_DATA_DIR, "uww_late_game_foul_list.csv"), index=False)


# ======================================================================================================
# REAL: sample-size warnings -- one place a coach can see what NOT to lean on
# ======================================================================================================
_warn = []
if not _gp_opp.empty and _gp_team_games > 1:
    for _, r in _gp_opp.iterrows():
        if r["games"] < _GP_THIN_GAME_SHARE * _gp_team_games and r["PTS"] / max(r["games"], 1) >= 8:
            _warn.append({"area": "Opponent player", "subject": r["player"],
                          "detail": f"{r['PTS'] / r['games']:.1f} ppg comes from {int(r['games'])} of "
                                    f"{_gp_team_games} games on film -- confirm his status and role."})
_lu = _gp_load("uww_opp_lineup_season_box")
if not _lu.empty and "MIN" in _lu.columns:
    _n_thin = int((pd.to_numeric(_lu["MIN"], errors="coerce") < _GP_THIN_LINEUP_MIN).sum())
    if _n_thin:
        _warn.append({"area": "Opponent lineups", "subject": f"{_n_thin} of {len(_lu)} units",
                      "detail": f"under {_GP_THIN_LINEUP_MIN:.0f} minutes together -- no reads or keys "
                                f"are drawn from them."})
_sm = _gp_load("uww_style_matchups")
if not _sm.empty and "confidence" in _sm.columns and "opponent" in _sm.columns:
    _sm = _sm[_sm["opponent"].astype(str) == str(_gp_short)]
    for _dir, _label in (("like_them", "Teams like them we've played"), ("like_us", "Teams like us")):
        _c = pd.to_numeric(_sm[_sm["direction"] == _dir]["confidence"], errors="coerce")
        if _c.notna().any() and _c.max() < 0.35:
            _warn.append({"area": "Style matchups", "subject": _label,
                          "detail": f"best match confidence is {100 * _c.max():.0f}% -- context, not evidence."})
_fl = _gp_load("uww_coaching_flags")
if not _fl.empty and "confidence" in _fl.columns:
    _low = _fl[_fl["confidence"].astype(str).str.lower().str.startswith("low")]
    if not _low.empty:
        _warn.append({"area": "Our player flags", "subject": f"{len(_low)} flag(s)",
                      "detail": "rest on fewer than 10 attempts -- use for film review, not rotation calls."})
if _gp_team_games and _gp_team_games < 6:
    _warn.append({"area": "Whole brief", "subject": f"{_gp_team_games} opponent games on film",
                  "detail": "every opponent rate here is early-season -- expect several to move by January."})
sample_size_warnings = pd.DataFrame(_warn, columns=["area", "subject", "detail"])
sample_size_warnings.insert(0, "opponent", _gp_short)
sample_size_warnings["is_sample"] = False
sample_size_warnings.to_csv(os.path.join(APP_DATA_DIR, "uww_sample_size_warnings.csv"), index=False)


# ======================================================================================================
# SAMPLE (until staff input exists): how they play
# ======================================================================================================
_S = lambda n: n if n else "their best player"  # noqa: E731

# ---- REAL sets from opponent_plays.csv, when the play-call cell produced them ----------------------------
# Situation, set, who finishes it, its main action and where, how often, how well it worked, and the game
# codes to pull film from. "our_call" is a GENERATED suggestion keyed on the primary action -- the brief labels
# that column "Suggested coverage" so nobody reads it as the staff's scheme.
_GP_COVERAGE_BY_ACTION = {
    "DHO": "Get into the handoff; force it away from the middle",
    "Ball Screen": "Pick one ball-screen coverage and stay in it; no middle",
    "Down Screen": "Lock and trail; bump the cutter off the screen",
    "Stagger": "Trail the first screen, top-lock the second",
    "Screen the Screener": "Talk it early; switch the second screen if we must",
    "Flare": "Top-lock the flare; screener's man shows",
    "Curl": "Trail tight and have the big wall the curl",
    "Pat Miller": "Walk through it Monday -- their signature action",
    "Grenade": "Get into the handoff before it becomes a pick-and-roll",
    "Hammer": "Weak-side corner defender stays home; no help from the corner",
    "Zoom": "Chase over the pin-down and jump the handoff",
    "Twirl": "Trail the twirl cut; big stays connected",
    "IVO": "Deny the Iverson cut across the elbows",
    "Scissors": "Communicate the split; no switching into mismatches",
    "Breddy": "Walk through the inbounds alignment and assignments",
    "Lob": "Help side sits on the rim; inbounder's man turns and faces",
    "Post Touch": "Three-quarter front; dig from the passer",
    "ISO": "Gap help, make him score over a crowd",
    "Double Drag": "Talk both screens; tag the roller",
}
_gp_pcs = _gp_load("uww_play_call_summary")
_gp_pc = _gp_load("uww_play_calls")

# ---- Minimum tagged possessions before a split is reported as real rather than left as sample ----------
# One place for every floor in this cell. These used to be defined inline next to each table, which meant
# four related tables quietly disagreed about how thin a sample is too thin to read. Raise them together
# once there's more film; a row below its floor falls back to sample data rather than reporting a
# confident-looking percentage off two or three clips.
MIN_USES = {
    "defense_by_situation": 4,   # situation x defense cross-tab: possessions in ONE situation
    "shot_clock": 4,             # possessions in one shot-clock bucket
    "ball_screen_coverage": 3,   # possessions against one named coverage
    "personnel_grouping": 4,     # possessions with one personnel type on the floor
    "game_situation": 4,         # possessions in one game-situation bucket
}
_DEF_MIN_USES = MIN_USES["defense_by_situation"]
_SC_MIN_USES = MIN_USES["shot_clock"]
_BSC_MIN_USES = MIN_USES["ball_screen_coverage"]
_PG_MIN_USES = MIN_USES["personnel_grouping"]
_GS_MIN_USES = MIN_USES["game_situation"]

_sets_real = []
_film_real = []
if not _gp_pcs.empty and {"side", "level", "name"}.issubset(_gp_pcs.columns):
    _opp_calls = _gp_pcs[(_gp_pcs["side"] == "Opponent") & (_gp_pcs["level"] == "Play call")
                         & (_gp_pcs["scouted_opponent"].astype(str) == str(_gp_short))]
    _opp_calls = _opp_calls[~_opp_calls["name"].astype(str).str.contains("unspecified", na=False)]
    _opp_calls = _opp_calls.sort_values(["uses", "ppp"], ascending=[False, False])
    # Half-court sets first, then inbounds/ATO, so the table reads the way a walkthrough runs.
    _half = _opp_calls[_opp_calls["situation"].astype(str) == "Half court"].head(8)
    _oob = _opp_calls[_opp_calls["situation"].astype(str) != "Half court"].head(6)
    for _, r in pd.concat([_half, _oob]).iterrows():
        if int(r["uses"]) < 2:
            continue
        _ppp = f", {r['ppp']:.2f} PPP" if pd.notna(r.get("ppp")) else ""
        _where = f" -- {r['top_location']}" if isinstance(r.get("top_location"), str) and r["top_location"] else ""
        _sets_real.append([
            _gp_short, r["situation"], r["name"], r.get("top_player"),
            (str(r.get("top_action") or "") + _where).strip(" -") or "--",
            f"{int(r['uses'])} uses in {int(r['games'])} games{_ppp}",
            _GP_COVERAGE_BY_ACTION.get(str(r.get("top_action")), ""),
            str(r.get("game_codes") or ""),
        ])
    for _, r in pd.concat([_half.head(4), _oob.head(2)]).iterrows():
        _film_real.append([_gp_short, f"{r['name']} ({r['situation']})", int(r["uses"]),
                           "Whole team" if r["situation"] != "Half court" else "Whole team -- scout team runs it"])

_sets_cols = ["opponent", "situation", "set_name", "primary_player", "action", "frequency", "our_call", "film_ref"]
_sets_sample = [
    [_gp_short, "Half court", "5-Out Motion", _S(_gp_driver), "Drive-and-kick off a dribble handoff at the top",
     "Base offense (~40%)", "Gap help, no middle drives, close out short", "Gm 2, 1st half"],
    [_gp_short, "Half court", "Horns Flare", _S(_gp_shooter), "Elbow ball screen, weak-side flare for the shooter",
     "3-4x a half", "Top-lock the flare; switch 4-5 on the elbow screen", "Gm 1, 2nd half"],
    [_gp_short, "Half court", "Post Split", _S(_gp_big), "Entry to the block, split cut above it",
     "2-3x a half", "Three-quarter front; dig from the passer", "Gm 3, 1st half"],
    [_gp_short, "Half court", "Spain PnR", _S(_gp_handler), "Ball screen with a back-screen on the roller's man",
     "Late clock", "Ice the ball screen; talk the back-screen early", "Gm 4, 2nd half"],
    [_gp_short, "BLOB", "Box Up", _S(_gp_shooter), "Stagger to the corner, big seals the rim",
     "Most BLOBs", "Switch everything on the box; bump the sealer", "Gm 2"],
    [_gp_short, "SLOB", "Stack Lob", _S(_gp_big), "Stack at the elbow, back-screen lob to the rim",
     "Under 5:00", "Help side sits on the rim; no top-side denial", "Gm 1"],
    [_gp_short, "ATO", "Elevator", _S(_gp_shooter), "Elevator doors at the top of the key for a catch-and-shoot",
     "Out of timeouts", "Chase over the top; the screener's man sits in the doors", "Gm 3"],
    [_gp_short, "Press break", "4-Across", _S(_gp_handler), "Four across the free-throw line, inbounder runs baseline",
     "vs full-court pressure", "Deny the first pass to the handler; trap the catch", "Gm 4"],
    [_gp_short, "End of game", "Iso-High Ball Screen", _S(_gp_driver), "Clear-out, late high ball screen with 8 on the clock",
     "Last possession", "Show and recover; no fouling on the drive", "Gm 2"],
]
opp_sets = _gp_write("opp_sets", _sets_cols, _sets_sample, real_rows=_sets_real)

_def_cols = ["opponent", "item", "what_they_do", "how_we_attack"]
_def_sample = [
    [_gp_short, "Base defense", "Man-to-man about 80% of possessions; 2-3 zone out of timeouts",
     "Have a zone offense called from the bench on every dead ball"],
    [_gp_short, "Ball-screen coverage", "Hard hedge and recover", "Short roll to the free-throw line; slip early"],
    [_gp_short, "Handoffs", "Switch", "Keep the ball and attack the slower defender; re-screen"],
    [_gp_short, "Post defense", "Play behind, dig from the passer", "Swing it after the dig; shooter relocates"],
    [_gp_short, "Help rules", "Strong-side low man helps on drives", "Corner stays filled; baseline drift"],
    [_gp_short, "Pressure", "1-2-1-1 after made free throws, late halves", "Middle flash, ball reversal, attack 4-on-3"],
]
# ---- REAL: what their defense actually shows, from clips where THEY were the defense --------------
# This table had no real_rows wiring at all -- it was sample-only regardless of what was tagged, which is
# why it stayed empty even after Aurora's defense was tagged. The scheme text (how_we_attack) is still a
# coaching judgement and stays staff-input, but WHAT they do is now read off the film.
# Source is defense_PLAYED on opponent-side clips: possessions where the other team had the ball and this
# opponent was defending. defense_faced would be the other team's scheme under this opponent's name.
_def_real = []
_OPPD_MIN = 5
if not _gp_pc.empty and "defense_played" in _gp_pc.columns:
    _od = _gp_pc[(_gp_pc["side"] == "Opponent")
                 & (_gp_pc["scouted_opponent"].astype(str) == str(_gp_short))
                 & (_gp_pc["decode_quality"] != "Needs review")
                 & _gp_pc["defense_played"].notna()]
    if len(_od) >= _OPPD_MIN:
        _base = _od["defense_played"].astype(str).value_counts()
        _pct = round(100 * int(_base.iloc[0]) / int(_base.sum()))
        _second = (f"; also {_base.index[1]} ({round(100 * int(_base.iloc[1]) / int(_base.sum()))}%)"
                   if len(_base) > 1 else "")
        _def_real.append([_gp_short, "Base defense",
                          f"{_base.index[0]} on {_pct}% of {int(_base.sum())} tagged possessions{_second}",
                          ""])
        if "coverage_played" in _od.columns:
            _cv = _od[_od["coverage_played"].astype(str).str.strip() != ""]["coverage_played"] \
                .astype(str).value_counts()
            if len(_cv):
                _def_real.append([_gp_short, "Ball-screen coverage",
                                  ", ".join(f"{n} ({int(c)}x)" for n, c in _cv.head(3).items()), ""])
        if "press_played" in _od.columns and int(_od["press_played"].sum()):
            _pf = _od[_od["press_played"].astype(bool)]
            _forms = (_pf["press_formation_played"].dropna().astype(str).value_counts()
                      if "press_formation_played" in _pf.columns else pd.Series(dtype=int))
            _def_real.append([_gp_short, "Pressure",
                              f"Pressed on {len(_pf)} of {len(_od)} tagged possessions"
                              + (f" -- {', '.join(_forms.index[:2])}" if len(_forms) else ""), ""])
opp_defense = _gp_write("opp_defense", _def_cols, _def_sample, real_rows=_def_real)

# Diagnostic: say exactly why this table is or isn't real, since "tagged but still empty" has bitten here.
if _gp_pc.empty:
    print("  opp_defense: uww_play_calls is empty -- staying sample.")
elif "defense_played" not in _gp_pc.columns:
    print("  opp_defense: no defense_played column -- re-run from the play-calls cell so possession_side "
          "and the faced/played split get built. Staying sample.")
elif not _def_real:
    _oc = _gp_pc[(_gp_pc["side"] == "Opponent") & (_gp_pc["scouted_opponent"].astype(str) == str(_gp_short))]
    _n_faced = int(_oc["defense_faced"].notna().sum()) if "defense_faced" in _oc.columns else 0
    _n_played = int(_oc["defense_played"].notna().sum()) if "defense_played" in _oc.columns else 0
    print(f"  opp_defense: {_n_played} clip(s) tagged with a defense {_gp_short} PLAYED "
          f"(need {_OPPD_MIN}+), and {_n_faced} tagged with a defense they FACED -- staying sample.")
    if _n_played == 0 and _n_faced > 0:
        print("    ^ Every defense tag in opponent_plays.csv landed on an OFFENSIVE possession "
              f"(Team = {_gp_short}). If those clips are meant to be {_gp_short}'s own defense, the Team "
              "column is naming the team being scouted rather than the team with the ball -- see the "
              "possession_side note in the play-calls cell.")
else:
    print(f"  opp_defense: real data for {len(_def_real)} item(s).")

_pers_cols = ["opponent", "player", "jersey", "position", "height", "hand", "class_year", "status",
              "games_on_film", "photo_url"]
_pers_sample = []
_used_numbers = set()
for _i, r in _gp_opp.head(10).iterrows():
    _num_options = [n for n in list(range(0, 35)) if n not in _used_numbers]
    _jersey = _gp_pick(r["player"] + "#", _num_options)
    _used_numbers.add(_jersey)
    _three_share = (r["FG3A"] / r["FGA"]) if r.get("FGA") else 0
    if r["player"] == _gp_big or (r["REB"] / max(r["games"], 1) >= 6 and _three_share < 0.15):
        _pos, _hts = "F/C", ["6-7", "6-8", "6-9"]
    elif _three_share >= 0.4 or r["AST"] / max(r["games"], 1) >= 2:
        _pos, _hts = "G", ["5-11", "6-1", "6-2", "6-3"]
    else:
        _pos, _hts = "W", ["6-4", "6-5", "6-6"]
    _thin = _gp_team_games > 1 and r["games"] < _GP_THIN_GAME_SHARE * _gp_team_games
    _pers_sample.append([
        _gp_short, r["player"], str(_jersey), _pos,
        _gp_pick(r["player"] + "h", _hts), _gp_pick(r["player"] + "L", ["R", "R", "R", "R", "L"]),
        _gp_pick(r["player"] + "c", ["Fr.", "So.", "Jr.", "Sr."]),
        "Confirm -- limited film" if _thin else "Available",
        # games_on_film is REAL even inside a sample section; the brief labels the column as such.
        f"{int(r['games'])} of {_gp_team_games}" if _gp_team_games else str(int(r["games"])),
        None,  # no photo without a real roster scrape
    ])

# ---- REAL personnel details from the live FastScout roster scrape (see the "Roster pages" cell), when it
# found this opponent. jersey/position/height/class_year/photo are real; hand and status are left BLANK
# rather than guessed -- the roster page doesn't carry either, and this table is either fully real or fully
# sample (never a mix), so a blank here is honest about what the source actually has.
#
# CONFIRMED BUG (fixed here): the roster page lists everyone on the roster, including players who haven't
# recorded a single minute in a game we have box-score data for -- a live run surfaced one by name ("Wifi
# Chin") with an incorrect photo. Photo-to-player pairing is done by POSITION on the page (see the roster
# cell's own docstring on this risk), and a zero-minute/inactive player is exactly the case most likely to
# sit in a different part of the page (a "not active" group) where that positional pairing breaks -- so
# these players are pulled OUT of the table entirely (no photo shown, no chance of a wrong one) and named in
# one summary line underneath instead, rather than risk a bad photo standing next to a real name.
#
# CONFIRMED BUG (fixed here): the roster page renders names ALL CAPS ("DEVON RICHARDSON"), while the box
# score has them title-cased ("Devon Richardson"). Matching on the exact string meant NOBODY matched -- the
# entire roster, active players included, landed in the "no minutes" line. Match on a case/whitespace-
# normalized key instead, and prefer the box score's own casing for display since it's already correct;
# _pers_titlecase is a fallback only for a player the box score has never seen at all.
_PERS_NAME_FIX = {"II": "II", "III": "III", "IV": "IV", "JR": "Jr.", "SR": "Sr."}


def _pers_norm(name):
    return re.sub(r"\s+", " ", str(name)).strip().lower()


def _pers_titlecase(name):
    """ALL CAPS -> Title Case for a player with no box-score entry to borrow proper casing from. Simple
    word-by-word title-casing -- doesn't special-case McEwen-style names, so double-check those by eye."""
    out = []
    for word in str(name).strip().split():
        key = word.rstrip(".").upper()
        out.append(_PERS_NAME_FIX[key] if key in _PERS_NAME_FIX else (word[:1].upper() + word[1:].lower()))
    return " ".join(out)


_live_ros = _gp_load("uww_live_rosters")
_pers_real = []
_pers_no_minutes = []
if not _live_ros.empty and "team" in _live_ros.columns:
    _my_ros = _live_ros[_live_ros["team"].astype(str) == str(_gp_short)]
    if not _my_ros.empty:
        _has_min_col = "MIN" in _gp_opp.columns
        _min_by_norm = ({_pers_norm(p): m for p, m in _gp_opp.set_index("player")["MIN"].items()}
                        if _has_min_col else {})
        _games_by_norm = {_pers_norm(r["player"]): (int(r["games"]), _gp_team_games, r["player"])
                          for _, r in _gp_opp.iterrows()}
        for _, r in _my_ros.iterrows():
            _key = _pers_norm(r["name"])
            _mins = _min_by_norm.get(_key)
            _played = (_mins is not None and _mins > 0) if _has_min_col else (_key in _games_by_norm)
            _gof = _games_by_norm.get(_key)
            _display_name = _gof[2] if _gof else _pers_titlecase(r["name"])
            if not _played:
                _pers_no_minutes.append(_display_name)
                continue
            _pers_real.append([
                _gp_short, _display_name, r.get("jersey_number"), r.get("position"), r.get("height"), "",
                r.get("class_year"), "",
                f"{_gof[0]} of {_gof[1]}" if _gof and _gof[1] else "",
                r.get("photo_url"),
            ])

opp_personnel = _gp_write("opp_personnel", _pers_cols, _pers_sample, real_rows=_pers_real)

# One real row per opponent, always (never sample-gated): the names pulled out of the table above. Rendered
# as a single summary sentence under Personnel Details rather than full rows -- a player is worth naming so
# the staff knows he's on the roster, but a blank/guessed row for him would overstate what's actually known.
roster_no_minutes = pd.DataFrame(
    [{"opponent": _gp_short, "names": ", ".join(_pers_no_minutes), "count": len(_pers_no_minutes)}]
    if _pers_no_minutes else [], columns=["opponent", "names", "count"])
roster_no_minutes.to_csv(os.path.join(APP_DATA_DIR, "uww_roster_no_minutes.csv"), index=False)

_zone_cols = ["opponent", "player", "rim", "paint_non_rim", "midrange", "corner_3", "above_break_3"]
_zone_sample = []
for _, r in _gp_opp.head(5).iterrows():
    _t = round(100 * r["FG3A"] / r["FGA"]) if r.get("FGA") else 0  # three share is REAL
    _c3 = round(_t * _gp_pick(r["player"] + "z", [0.3, 0.4, 0.5]))
    _two = 100 - _t
    _rim = round(_two * _gp_pick(r["player"] + "r", [0.45, 0.55, 0.65]))
    _mid = round((_two - _rim) * 0.5)
    _zone_sample.append([_gp_short, r["player"], _rim, _two - _rim - _mid, _mid, _c3, _t - _c3])
opp_shot_zones = _gp_write("opp_shot_zones", _zone_cols, _zone_sample)

# ---- REAL per-player shot profile, from the play-by-play ----------------------------------------------
# The zone table above is a placeholder: only its three-point share is real, and rim / paint / mid / corner
# are a pseudo-random split. The play-by-play CAN say something real about location, though -- the text of
# every make and miss says whether a two was a layup/dunk/tip or a jumper. So each player gets an honest
# three-way split: AT THE RIM, OTHER TWOS (jumpers, hooks, floaters), and THREES, with FG% in each. Coarser
# than five zones, but every number is real. Feeds the roster's per-player shot line in the brief.
_RIM_WORDS = ("layup", "lay-up", "dunk", "tip", "putback", "put back", "jam", "alley")

def _shot_profile(pbp, team_value, side_label):
    if pbp.empty or not {"event_type", "team", "player"}.issubset(pbp.columns):
        return []
    s = pbp[pbp["event_type"].astype(str).isin(["made_shot", "missed_shot"])
            & (pbp["team"].astype(str) == str(team_value))].copy()
    if s.empty:
        return []
    _txt = s["raw_text"].astype(str).str.lower() if "raw_text" in s.columns else pd.Series("", index=s.index)
    _three = s["shot_type"].astype(str) == "3" if "shot_type" in s.columns else pd.Series(False, index=s.index)
    s["_zone"] = "Other 2"
    s.loc[_txt.apply(lambda t: any(w in t for w in _RIM_WORDS)) & ~_three, "_zone"] = "Rim"
    s.loc[_three, "_zone"] = "Three"
    s["_made"] = s["event_type"].astype(str) == "made_shot"
    rows = []
    for _pl, _g in s.groupby(s["player"].astype(str)):
        if _pl.strip().upper() in ("", "TEAM", "NAN"):
            continue
        _n = len(_g)
        _row = {"opponent": _gp_short, "side": side_label, "player": _pl, "fga": _n}
        for _z, _key in (("Rim", "rim"), ("Other 2", "other2"), ("Three", "three")):
            _zg = _g[_g["_zone"] == _z]
            _row[f"{_key}_att"] = len(_zg)
            _row[f"{_key}_share"] = round(100 * len(_zg) / _n) if _n else None
            _row[f"{_key}_fg"] = round(100 * _zg["_made"].sum() / len(_zg), 1) if len(_zg) else None
        rows.append(_row)
    return rows

_prof_rows = (_shot_profile(_gp_load("uww_opponent_prior_games_pbp"), _gp_short, "Opponent")
              + _shot_profile(_gp_load("uww_pbp_events"), "UW-Whitewater", "UWW"))
player_shot_profile = pd.DataFrame(_prof_rows, columns=[
    "opponent", "side", "player", "fga", "rim_att", "rim_share", "rim_fg", "other2_att", "other2_share",
    "other2_fg", "three_att", "three_share", "three_fg"])
player_shot_profile.to_csv(os.path.join(APP_DATA_DIR, "uww_player_shot_profile.csv"), index=False)
print(f"  player_shot_profile: {int((player_shot_profile['side'] == 'Opponent').sum())} of theirs, "
      f"{int((player_shot_profile['side'] == 'UWW').sum())} of ours (rim / other 2 / three, from play-by-play).")

_tend_cols = ["opponent", "item", "detail"]
_tend_sample = [
    [_gp_short, "Transition", "Push after misses about half the time; walk it up after makes"],
    [_gp_short, "Offensive glass", f"Send 2-3 to the glass; {_S(_gp_big)} crashes every shot"],
    [_gp_short, "Timeouts", "Head coach calls the first one early if down 6+; saves two for the last 4:00"],
    [_gp_short, "End of half", "Hold for one; iso the best scorer at the top with 8 seconds left"],
    [_gp_short, "Officials", "Crew not yet assigned -- check the conference site Tuesday"],
]
opp_tendencies = _gp_write("opp_tendencies", _tend_cols, _tend_sample)

# ======================================================================================================
# SAMPLE (until staff input exists): our side
# ======================================================================================================
_match_cols = ["opponent", "their_player", "our_defender", "backup", "note"]
_match_sample = []
_us_starters = list(_gp_us["player"].head(5)) if not _gp_us.empty else []
_top_threat = None
if not _gp_opp.empty:
    _reg = _gp_opp[_gp_opp["games"] >= _GP_THIN_GAME_SHARE * max(_gp_team_games, 1)]
    _top_threat = (_reg if not _reg.empty else _gp_opp).sort_values("PTS", ascending=False)["player"].iloc[0]
_us_bench = list(_gp_us["player"].iloc[5:10]) if len(_gp_us) > 5 else []


# ======================================================================================================
# SAMPLE ONLY (no path to real data from current tagging): sections that show what richer film tagging
# would unlock. Each needs a field the tagging tool doesn't capture today -- see the note on each table.
# DEFENSE TYPE BY SITUATION and BALL SCREEN COVERAGE used to live in this block too, but the coaches'
# updated Title logic (see decode_defense_tag in the play-calls cell) now tags exactly this, so both moved
# below to the real-data section with everything else derived from uww_play_calls / uww_play_call_summary.
# ======================================================================================================
_def_type_cols = ["opponent", "situation", "primary_defense", "freq_pct", "secondary_defense", "note"]
_def_type_sample = [
    [_gp_short, "Half court", "Man-to-man", 78, "2-3 zone (after a timeout or a long defensive stretch)",
     f"Switches 1-4, doesn't switch {_S(_gp_big)}'s matchup"],
    [_gp_short, "BLOB", "Man-to-man", 60, "Box-and-1 on our best shooter", "Denies the first pass hard"],
    [_gp_short, "SLOB", "Man-to-man", 85, "", "Same coverage as half court, no separate call seen"],
    [_gp_short, "After a made basket (press)", "1-2-1-1 full-court", 30, "Man-to-man (no press)",
     "Presses more in the 4th when trailing"],
]

# ---- REAL defense-by-situation, cross-tabbed straight from the per-clip table (uww_play_calls) rather
# than uww_play_call_summary -- that table only aggregates one column at a time, and this needs TWO
# (situation x defense_type together). A situation needs at least _DEF_MIN_USES tagged possessions with a
# defense_type before it's shown; below that it stays sample rather than reporting a "100%" read on 2 clips.
_def_type_real = []
# defense_PLAYED, not defense_type: this table is what the opponent's own defense runs. The raw
# defense_type on an opponent-side clip is what their OFFENSE faced -- i.e. the other team's defense --
# so reading it here reported the wrong team's scheme under this opponent's name.
if not _gp_pc.empty and {"side", "scouted_opponent", "play_situation", "defense_played", "decode_quality"}.issubset(_gp_pc.columns):
    _dpc = _gp_pc[(_gp_pc["side"] == "Opponent") & (_gp_pc["scouted_opponent"].astype(str) == str(_gp_short))
                  & _gp_pc["defense_played"].notna() & (_gp_pc["decode_quality"] != "Needs review")]
    for _situ, _grp in (_dpc.groupby("play_situation") if not _dpc.empty else []):
        _counts = _grp["defense_played"].value_counts()
        if _counts.empty or int(_counts.sum()) < _DEF_MIN_USES:
            continue
        _freq = round(100 * int(_counts.iloc[0]) / int(_counts.sum()))
        _secondary = _counts.index[1] if len(_counts) > 1 else ""
        _press_n = int(_grp["press_played"].sum()) if "press_played" in _grp.columns else 0
        _note = f"{int(_counts.sum())} tagged possession{'s' if _counts.sum() != 1 else ''}"
        if _press_n:
            _note += f"; press seen on {_press_n}"
        _def_type_real.append([_gp_short, _situ, _counts.index[0], _freq, _secondary, _note])
if _gp_pc.empty:
    print("  defense_by_situation: uww_play_calls is empty -- staying sample.")
elif "defense_played" not in _gp_pc.columns:
    print("  defense_by_situation: uww_play_calls has no defense_type column -- the notebook needs a full "
          "re-run from the play-calls cell so the new Title tagging gets decoded. Staying sample.")
elif not _def_type_real:
    _opp_def_n = int(((_gp_pc["side"] == "Opponent") & (_gp_pc["scouted_opponent"].astype(str) == str(_gp_short))
                      & _gp_pc["defense_played"].notna()).sum())
    print(f"  defense_by_situation: {_opp_def_n} opponent clip(s) with a defense tag for {_gp_short}, none of "
          f"which reached {_DEF_MIN_USES}+ in any one situation -- staying sample. (0 clips usually means "
          f"opponent_plays.csv hasn't been re-tagged with the new Title format yet.)")
else:
    print(f"  defense_by_situation: real data for {len(_def_type_real)} situation(s).")
defense_by_situation = _gp_write("defense_by_situation", _def_type_cols, _def_type_sample, real_rows=_def_type_real)

_clock_cols = ["opponent", "clock_situation", "freq_pct", "ppp", "fg_pct", "note"]
_clock_sample = [
    [_gp_short, "Early clock (0-9 sec used)", 22, 1.18, 54, "Mostly transition and early ball screens"],
    [_gp_short, "Organized offense (10-19 sec used)", 55, 0.94, 41, f"Their half-court sets, {_S(_gp_shooter)} runs off screens here"],
    [_gp_short, "Late clock (20+ sec used)", 23, 0.71, 33, f"{_S(_gp_driver)} isolation when the set breaks down"],
]

# ---- REAL shot-clock tendencies, estimated from the game clock using NCAA men's shot-clock rules and
# cross-referenced against each matched clip's decoded play call (see the "Play calls" cell -- shot_clock_
# situation, level="Shot clock" in uww_play_call_summary). Only matched clips carry an estimate.
_clock_real = []
if not _gp_pcs.empty and {"side", "level", "name", "uses"}.issubset(_gp_pcs.columns):
    _sc_rows_df = _gp_pcs[(_gp_pcs["side"] == "Opponent") & (_gp_pcs["level"] == "Shot clock")
                          & (_gp_pcs["scouted_opponent"].astype(str) == str(_gp_short))]
    _sc_order = ["Early clock (0-9 sec used)", "Organized offense (10-19 sec used)", "Late clock (20+ sec used)"]
    _sc_total_uses = _sc_rows_df["uses"].sum()
    for _bucket in _sc_order:
        _row = _sc_rows_df[_sc_rows_df["name"] == _bucket]
        if _row.empty or int(_row.iloc[0]["uses"]) < _SC_MIN_USES:
            continue
        _r = _row.iloc[0]
        _freq = round(100 * _r["uses"] / _sc_total_uses) if _sc_total_uses else None
        _note = f"{int(_r['uses'])} tagged possessions"
        if isinstance(_r.get("top_action"), str):
            _note += f" -- most often {_r['top_action']}"
        if isinstance(_r.get("top_player"), str):
            _note += f" ({_r['top_player']})"
        _clock_real.append([_gp_short, _bucket, _freq, _r.get("ppp"), _r.get("fg_pct"), _note])

shot_clock_tendencies = _gp_write("shot_clock_tendencies", _clock_cols, _clock_sample, real_rows=_clock_real)

_matchup_hist_cols = ["opponent", "their_player", "defended_by", "possessions", "ppp_allowed", "note"]
_matchup_hist_sample = [
    [_gp_short, _S(_gp_driver), (_us_starters[0] if _us_starters else ""), 18, 0.83, "Best matchup on file -- length bothers his first step"],
    [_gp_short, _S(_gp_driver), (_us_starters[1] if len(_us_starters) > 1 else ""), 9, 1.22, "Got switched onto him twice in the 4th and gave up 6 straight"],
    [_gp_short, _S(_gp_shooter), (_us_starters[2] if len(_us_starters) > 2 else ""), 14, 0.71, "Chases well over screens"],
    [_gp_short, _S(_gp_big), (_us_starters[4] if len(_us_starters) > 4 else ""), 21, 0.95, "Gives up deep position early in the shot clock"],
]
matchup_history = _gp_write("matchup_history", _matchup_hist_cols, _matchup_hist_sample)

_counters_cols = ["opponent", "set", "when_we_take_it_away", "their_counter", "our_adjustment"]
_counters_sample = [
    [_gp_short, "4-1 Ball Screen", "We show hard and force it left", "Reject and drive right instead",
     "Have the weak-side big ready to step up, not just the on-ball defender"],
    [_gp_short, "Hi-Lo DHO", "We deny the handoff", f"{_S(_gp_big)} seals and posts instead",
     "Front early rather than three-quarter once the handoff is denied"],
    [_gp_short, "BLOB Box Curl", "We switch the curl", "They throw the skip pass to the opposite corner",
     "Tag the low man to the corner on the switch"],
]
play_counters = _gp_write("play_counters", _counters_cols, _counters_sample)

_bsc_cols = ["opponent", "coverage", "freq_pct", "ppp", "fg_pct", "note"]
_bsc_sample = [
    [_gp_short, "Drop", 45, 0.87, 39, f"Big sags to the paint -- {_S(_gp_shooter)} pulls up off it"],
    [_gp_short, "Hard hedge", 25, 1.05, 47, "Big shows well but recovers late -- short roll is open behind it"],
    [_gp_short, "Switch", 15, 0.95, 43, f"Switches 1 through 4, not onto {_S(_gp_big)}"],
    [_gp_short, "Ice (side ball screens)", 10, 0.62, 31, "Forces it baseline; help comes from the nail"],
    [_gp_short, "Blitz", 5, 1.30, 50, f"Live-ball turnover risk if {_S(_gp_handler)} splits it"],
]

# ---- REAL ball-screen coverage, from uww_play_call_summary's "Ball screen coverage" level (level built
# in the play-calls cell from decode_defense_tag's defense_coverage field: Switch / Hedge / Soft Hedge /
# Ice / Drop / Deny / Jam -- a clip tagged with more than one, e.g. "Drop+Press", counts under its own
# combined bucket rather than being split). Same shape and threshold pattern as shot_clock_tendencies above.
_bsc_real = []
_bsc_rows_df = pd.DataFrame()
_bsc_cols_ok = not _gp_pcs.empty and {"side", "level", "name", "uses"}.issubset(_gp_pcs.columns)
if _bsc_cols_ok:
    # DEFENSIVE rows: this is the coverage they played, which only exists on possessions where they were
    # defending. play_call_summary now stamps possession_side for exactly this reason.
    _bsc_rows_df = _gp_pcs[(_gp_pcs["side"] == "Opponent") & (_gp_pcs["level"] == "Ball screen coverage played")
                           & (_gp_pcs["scouted_opponent"].astype(str) == str(_gp_short))]
    if "possession_side" in _bsc_rows_df.columns:
        _bsc_rows_df = _bsc_rows_df[_bsc_rows_df["possession_side"].astype(str) == "Defense"]
    _bsc_total_uses = _bsc_rows_df["uses"].sum()
    for _, _r in _bsc_rows_df.sort_values("uses", ascending=False).iterrows():
        if int(_r["uses"]) < _BSC_MIN_USES:
            continue
        _freq = round(100 * _r["uses"] / _bsc_total_uses) if _bsc_total_uses else None
        _note = f"{int(_r['uses'])} tagged possession{'s' if _r['uses'] != 1 else ''}"
        if isinstance(_r.get("top_player"), str):
            _note += f" -- most often against {_r['top_player']}"
        _bsc_real.append([_gp_short, _r["name"], _freq, _r.get("ppp"), _r.get("fg_pct"), _note])
if _gp_pcs.empty:
    print("  ball_screen_coverage: uww_play_call_summary is empty -- staying sample.")
elif not _bsc_cols_ok or "Ball screen coverage played" not in set(_gp_pcs.get("level", [])):
    print("  ball_screen_coverage: no \"Ball screen coverage\" level in uww_play_call_summary -- the notebook "
          "needs a full re-run from the play-calls cell so the new Title tagging gets decoded. Staying sample.")
elif not _bsc_real:
    print(f"  ball_screen_coverage: {int(_bsc_rows_df['uses'].sum()) if not _bsc_rows_df.empty else 0} tagged "
          f"clip(s) with a coverage call for {_gp_short}, none of which reached {_BSC_MIN_USES}+ on any one "
          f"coverage -- staying sample. (0 clips usually means opponent_plays.csv hasn't been re-tagged with "
          f"the new Title format yet.)")
else:
    print(f"  ball_screen_coverage: real data for {len(_bsc_real)} coverage type(s).")
ball_screen_coverage = _gp_write("ball_screen_coverage", _bsc_cols, _bsc_sample, real_rows=_bsc_real)

# ---- Tempo & transition (REAL, from the play-by-play -- needs no play tagging at all) ------------------
# Pre-film question the brief never answered: how fast do they want to play, and how much of their offense
# comes before the defense is set. Possessions are the standard estimate (FGA - OREB + TO + 0.475*FTA),
# averaged per game; transition share comes from the shot-clock estimate already on the tagged clips.
_tempo_cols = ["opponent", "metric", "value", "note"]
_tempo_sample = [
    [_gp_short, "Possessions per game", 71.4, "Sample -- needs their prior-game box scores"],
    [_gp_short, "Points per possession", 1.07, "Sample"],
    [_gp_short, "Early-offense share", 38, "Sample -- % of tagged possessions shooting inside 10 seconds"],
]
_tempo_real = []

def _team_game_totals(box, team):
    """One row per game: the team's summed box score, INCLUDING the synthetic TEAM row.

    CONFIRMED BUG (fixed here): the earlier version used the TEAM row ALONE whenever one existed. That row
    is synthetic -- it only carries bare team-level turnovers (see the pbp box score cell), with FGA, FTA,
    OREB and PTS all zero -- so possessions came out as a couple of turnovers a game and PPP as roughly
    zero. The TEAM row has to be ADDED to the player rows, never used instead of them.
    """
    if box.empty or "team" not in box.columns:
        return pd.DataFrame()
    own = box[box["team"].astype(str) == str(team)]
    if own.empty:
        return own
    cols = [c for c in ("FGA", "OREB", "TO", "FTA", "PTS") if c in own.columns]
    key = "game_date" if "game_date" in own.columns else None
    if not key:
        return pd.DataFrame()
    g = own.groupby(key, dropna=True)[cols].sum(numeric_only=True).reset_index()
    for c in ("FGA", "OREB", "TO", "FTA", "PTS"):
        if c not in g.columns:
            g[c] = pd.NA
    return g

def _possessions(g):
    """Standard estimate. Returns None rather than a wrong number when a required column is missing --
    a possession estimate without OREB overstates pace by the offensive-rebound count every game."""
    if g.empty or g[["FGA", "TO", "FTA", "OREB"]].isna().all().any():
        return None
    return (pd.to_numeric(g["FGA"], errors="coerce").fillna(0)
            - pd.to_numeric(g["OREB"], errors="coerce").fillna(0)
            + pd.to_numeric(g["TO"], errors="coerce").fillna(0)
            + 0.475 * pd.to_numeric(g["FTA"], errors="coerce").fillna(0))

_tb = _team_game_totals(_gp_load("uww_opponent_prior_games_box_score"), _gp_short)
_poss = _possessions(_tb)
if _poss is not None and _poss.sum() > 0:
    _n_games = len(_tb)
    _pts = pd.to_numeric(_tb["PTS"], errors="coerce").fillna(0)
    _tempo_real.append([_gp_short, "Possessions per game", round(float(_poss.mean()), 1),
                        f"Estimated over {_n_games} game(s): FGA - OREB + TO + 0.475*FTA"])
    _tempo_real.append([_gp_short, "Points per possession", round(float(_pts.sum() / _poss.sum()), 2),
                        f"Their scoring over {_n_games} game(s) of film"])
    # Our own pace from our own box, so the number has something to be compared against.
    _ub = _team_game_totals(_gp_load("uww_pbp_box_score"), "UW-Whitewater")
    _up = _possessions(_ub)
    if _up is not None and _up.sum() > 0:
        _tempo_real.append(["UW-Whitewater", "Possessions per game", round(float(_up.mean()), 1),
                            f"Our own pace over {len(_ub)} game(s)"])
    # Sanity bounds: college men's pace lives roughly 60-85. Anything outside is a data problem, not a
    # fast team, so say so on the run rather than let it reach the app looking plausible.
    if not (55 <= float(_poss.mean()) <= 90):
        print(f"  tempo_profile: {_gp_short} computed at {float(_poss.mean()):.1f} possessions/game -- outside "
              "the plausible range; check the prior-games box score before trusting it.")
elif not _tb.empty:
    print("  tempo_profile: box score is missing a column the possession estimate needs (FGA, OREB, TO or "
          "FTA) -- left as sample rather than reporting an inflated pace.")

# possession_side is a column in normal runs; guard with a Series default so a missing column can never
# turn into DataFrame.get() returning the bare string "Offense" (which has no .astype).
_ps = _gp_pc["possession_side"] if "possession_side" in _gp_pc.columns else pd.Series("Offense", index=_gp_pc.index)
if not _gp_pc.empty and "shot_clock_situation" in _gp_pc.columns:
    _tc = _gp_pc[(_gp_pc["side"] == "Opponent")
                 & (_ps.astype(str) != "Defense")
                 & (_gp_pc["offense_team"].astype(str) == str(_gp_short))
                 & _gp_pc["shot_clock_situation"].notna()]
    if len(_tc) >= 10:
        _early = _tc["shot_clock_situation"].astype(str).str.contains("Early", case=False, na=False).sum()
        _tempo_real.append([_gp_short, "Early-offense share", round(100 * _early / len(_tc)),
                            f"% of {len(_tc)} tagged possessions shooting inside 10 seconds"])
tempo_profile = _gp_write("tempo_profile", _tempo_cols, _tempo_sample, real_rows=_tempo_real)

# ---- Are they changing? (REAL) -------------------------------------------------------------------------
# Four games treated as one static profile hides a team that has switched what it does. Split their tagged
# possessions into an earlier half and a recent half and report only splits that actually MOVED. Needs
# enough games to be two-vs-two or better, and says so rather than reporting noise.
_trend_cols = ["opponent", "split", "earlier", "recent", "change", "note"]
_trend_sample = [
    [_gp_short, "Zone share of defense", "20%", "60%", "+40", "Sample -- needs more tagged games"],
]
_trend_real = []
_TREND_MIN_GAMES = 4
_TREND_MIN_MOVE = 10        # percentage points before a change is worth printing
if not _gp_pc.empty and "game_date" in _gp_pc.columns:
    _tr = _gp_pc[(_gp_pc["side"] == "Opponent") & _gp_pc["game_date"].notna()].copy()
    _games = sorted(_tr["game_date"].dropna().astype(str).unique())
    if len(_games) >= _TREND_MIN_GAMES:
        _half = len(_games) // 2
        _early_g, _recent_g = set(_games[:_half]), set(_games[_half:])
        _tr["_era"] = _tr["game_date"].astype(str).map(
            lambda g: "earlier" if g in _early_g else ("recent" if g in _recent_g else None))
        for _label, _col, _test in (
                ("Zone share of their defense", "defense_played",
                 lambda s: s.astype(str).str.contains("zone", case=False, na=False)),
                ("Press share of their defense", "press_played", lambda s: s.astype(bool)),
                ("Early-offense share", "shot_clock_situation",
                 lambda s: s.astype(str).str.contains("Early", case=False, na=False))):
            if _col not in _tr.columns:
                continue
            _vals = {}
            for _era in ("earlier", "recent"):
                _g = _tr[(_tr["_era"] == _era) & _tr[_col].notna()]
                if len(_g) >= 8:
                    _vals[_era] = 100 * _test(_g[_col]).sum() / len(_g)
            if len(_vals) == 2 and abs(_vals["recent"] - _vals["earlier"]) >= _TREND_MIN_MOVE:
                _trend_real.append([_gp_short, _label, f"{_vals['earlier']:.0f}%", f"{_vals['recent']:.0f}%",
                                    f"{_vals['recent'] - _vals['earlier']:+.0f}",
                                    f"First {_half} game(s) vs last {len(_games) - _half}"])
    else:
        print(f"  trend_profile: only {len(_games)} tagged game(s) -- needs {_TREND_MIN_GAMES}+ before an "
              f"earlier-vs-recent split says anything. Staying sample.")
trend_profile = _gp_write("trend_profile", _trend_cols, _trend_sample, real_rows=_trend_real)

_help_cols = ["opponent", "trigger", "who_rotates", "tendency", "note"]
_help_sample = [
    [_gp_short, "Drive to the rim", "Weak-side corner defender", "Rotates late -- 1-count behind the drive",
     "Corner three is there if the extra pass comes quickly"],
    [_gp_short, "Skip pass to the corner", "Nearest wing defender", "Closes out under control, no fly-bys",
     "Better to attack this closeout off the dribble than shoot into it"],
    [_gp_short, "Post entry", f"{_S(_gp_big)}'s man", "Digs hard from the top, occasionally leaves for a full double",
     "Kick to the dig's man when the double comes"],
    [_gp_short, "Roller on a ball screen", "Weak-side big", "Tags the roller consistently, X-out is a beat slow",
     "Second cutter behind the tag is open more than the roller itself"],
]
help_rotation_tendencies = _gp_write("help_rotation_tendencies", _help_cols, _help_sample)

_pg_cols = ["opponent", "personnel_grouping", "minutes_pct", "primary_actions_used", "note"]
_pg_sample = [
    [_gp_short, "Base five (one big)", 72, "4-1 Ball Screen, Hi-Lo DHO", "Their most-used grouping all four games"],
    [_gp_short, "Two-big lineup", 18, "Post touches, offensive rebounding",
     "Comes in for defensive stops, stays if it's working"],
    [_gp_short, "Small / shooting five (no true big)", 10, f"5 Out, ball screens for {_S(_gp_shooter)}",
     "Closing lineup when trailing"],
]

# ---- REAL personnel groupings, aggregated by TYPE (two bigs / base five / small-ball) rather than the
# literal 5-man unit -- the literal lineups are already broken out, player by player with their own keys,
# in Top Lineups, so this pools every lineup sharing a personnel profile instead of repeating those same
# five names. Type comes from the play-calls cell's own classifier (_pg_grouping_type there, based on each
# team's two highest-rebounding players in the play-by-play -- an approximation of "who plays big minutes,"
# not a roster-accurate position; see that cell's comment for the full caveat). Reads
# level="Personnel grouping type" in uww_play_call_summary. A clip only carries a type when it MATCHED a
# play-by-play event, so this covers only the tagged possessions that matched, not every clip.
_pg_real = []
if not _gp_pcs.empty and {"side", "level", "name", "uses"}.issubset(_gp_pcs.columns):
    # CONFIRMED BUG (fixed here): the table showed one grouping at 47% of minutes and nothing else, so more
    # than half their playing time was simply absent. Only types with _PG_MIN_USES tagged clips were listed --
    # a type with plenty of real MINUTES but few matched clips vanished entirely. Every type with minutes is
    # listed now; the tagged columns are blank when there aren't enough clips to say anything about them.
    _pg_rows_df = _gp_pcs[(_gp_pcs["side"] == "Opponent") & (_gp_pcs["level"] == "Personnel grouping type")
                          & (_gp_pcs["scouted_opponent"].astype(str) == str(_gp_short))].sort_values(
                              "uses", ascending=False)
    # Real minutes share by TYPE: classify each literal lineup in the season lineup box score the SAME way
    # (top-2 rebounders = bigs) and sum its real MIN into that type's bucket, rather than guessing.
    # CONFIRMED CHANGE (requested): groupings are now the play-calls cell's own labels -- "3G-2B",
    # "3G-1W-1B" from real roster positions, falling back to two-big/one-big/no-big for a team with no
    # roster positions on file. That cell also exports the lineup -> grouping map (uww_lineup_grouping), so
    # MINUTES are aggregated on exactly the rule the tagged clips were classified with, instead of this cell
    # classifying lineups a second time (which is how minutes and clips once landed on different groupings).
    _pg_map_df = _gp_load("uww_lineup_grouping")
    _pg_map = {}
    if not _pg_map_df.empty and {"lineup", "grouping"}.issubset(_pg_map_df.columns):
        _side_rows = _pg_map_df[_pg_map_df["side"] == "Opponent"] if "side" in _pg_map_df.columns else _pg_map_df
        _pg_map = {str(r["lineup"]): r["grouping"] for _, r in _side_rows.iterrows() if pd.notna(r["grouping"])}

    def _pg_type_for(lineup_str):
        return _pg_map.get(str(lineup_str))

    _pg_min_by_type = {}
    if not _lu.empty and "lineup" in _lu.columns and "MIN" in _lu.columns:
        _lu_typed = _lu.assign(_type=_lu["lineup"].apply(_pg_type_for))
        _pg_total_min = pd.to_numeric(_lu["MIN"], errors="coerce").sum()
        if _pg_total_min:
            _pg_min_by_type = (_lu_typed.assign(_min=pd.to_numeric(_lu_typed["MIN"], errors="coerce"))
                              .dropna(subset=["_type", "_min"]).groupby("_type")["_min"].sum()
                              .div(_pg_total_min).mul(100).round().to_dict())

    _pg_seen = set()
    for _, r in _pg_rows_df.iterrows():
        _type_name = str(r["name"])
        _pg_seen.add(_type_name)
        _calls = (_gp_pc[(_gp_pc["side"] == "Opponent") & (_gp_pc["personnel_grouping_type"] == _type_name)
                         & ~_gp_pc["play_call"].astype(str).str.contains("unspecified", na=False)
                         & _gp_pc["play_call"].notna()]["play_call"].value_counts().head(3)
                  if not _gp_pc.empty and "personnel_grouping_type" in _gp_pc.columns else pd.Series(dtype=int))
        if int(r["uses"]) < _PG_MIN_USES:
            _pg_real.append([_gp_short, _type_name, _pg_min_by_type.get(_type_name), "--",
                             f"Only {int(r['uses'])} tagged possession(s) -- too few to read"])
            continue
        _what = ", ".join(f"{n} ({int(c)}x)" for n, c in _calls.items()) or (r.get("top_action") or "--")
        _note = f"{int(r['uses'])} tagged possessions"
        if pd.notna(r.get("ppp")):
            _note += f", {r['ppp']:.2f} PPP"
        if pd.notna(r.get("team_ppp")):
            _note += f" (their overall {r['team_ppp']:.2f})"
        _pg_real.append([_gp_short, _type_name, _pg_min_by_type.get(_type_name), _what, _note])

    for _type_name, _pct in sorted(_pg_min_by_type.items(), key=lambda kv: -kv[1]):
        if _type_name in _pg_seen:
            continue
        _pg_real.append([_gp_short, _type_name, _pct, "--", "No tagged clips matched this grouping yet"])
    # Keep the table in a fixed, readable order rather than whatever the tagged counts happened to be.
    # Busiest grouping first -- with position shapes ("3G-2B") there's no fixed order to impose.
    _pg_real.sort(key=lambda row: -(row[2] or 0))

personnel_grouping = _gp_write("personnel_grouping", _pg_cols, _pg_sample, real_rows=_pg_real)

# ---- REAL: what each five-man unit actually runs, for the brief's Top Lineups (requested) --------------
# The sets tagged on possessions with that exact unit on the floor, so a lineup row can say what it runs
# rather than only how it scored.
_lu_calls_rows = []
if not _gp_pc.empty and "on_court_lineup" in _gp_pc.columns:
    _named_calls = _gp_pc[_gp_pc["play_call"].notna()
                          & ~_gp_pc["play_call"].astype(str).str.contains("unspecified", na=False)
                          & (_gp_pc["decode_quality"] != "Needs review")]
    for (_side_val, _lu), _grp in _named_calls.groupby(["side", "on_court_lineup"]):
        _vc = _grp["play_call"].value_counts().head(3)
        _pts = pd.to_numeric(_grp["points"], errors="coerce")
        _lu_calls_rows.append({
            "scouted_opponent": _gp_short, "side": _side_val, "lineup": str(_lu),
            "top_calls": ", ".join(f"{n} ({int(c)}x)" for n, c in _vc.items()),
            "tagged_possessions": len(_grp),
            "ppp": round(_pts.sum() / _pts.notna().sum(), 2) if _pts.notna().any() else None,
            "grouping": _pg_map.get(str(_lu)) if "_pg_map" in dir() else None})
lineup_play_calls = pd.DataFrame(_lu_calls_rows, columns=["scouted_opponent", "side", "lineup", "top_calls",
                                                          "tagged_possessions", "ppp", "grouping"])
lineup_play_calls.to_csv(os.path.join(APP_DATA_DIR, "uww_lineup_play_calls.csv"), index=False)

_reb_cols = ["opponent", "situation", "crash_pct", "who_crashes", "note"]
_reb_sample = [
    [_gp_short, "Miss by a non-shooter (paint attempt)", 65, _S(_gp_big) + " + ball-side wing",
     "Second-chance points mostly come from this situation"],
    [_gp_short, "Miss by their primary shooter (three)", 20, _S(_gp_big) + " only",
     "Everyone else gets back -- transition D opportunity for us"],
    [_gp_short, "Missed free throw", 40, _S(_gp_big), "Lines up early, boxes out inconsistently"],
]

# ---- REAL: who gets the rebound after a miss, straight from the play-by-play -------------------------
# Every miss is paired with the rebound that follows it (requested). No tagging needed: the play-by-play
# already records missed_shot / free_throw_missed and then rebound_offensive / rebound_defensive with the
# rebounder's name. Pairing walks forward from the miss inside the same game and stops at the first
# rebound; if another shot, a turnover or the next period comes first, that miss is left unpaired rather
# than credited to the wrong board. Team (deadball) rebounds count toward the rate but not toward "who".
#
# Two directions, because they answer different questions:
#   their misses   -> how hard THEY crash the offensive glass, and who does it
#   opponent misses -> who on their side secures the defensive board (who we have to box out)
_reb_real = []
_REB_MIN_MISSES = 8
# Pairing is rebound_pairs() from the Keys to Victory cell -- ONE implementation shared by the key, the
# brief's Bottom Line / roster reads and this table, so they can't disagree about who got a rebound.
_reb_pbp = _gp_load("uww_opponent_prior_games_pbp")
if not _reb_pbp.empty and {"event_type", "team", "game_date", "event_order"}.issubset(_reb_pbp.columns):
    _pairs = rebound_pairs(_reb_pbp, _gp_short)
    if not _pairs.empty:
        _is_them = _pairs["is_them"].astype(bool)

        def _who(df):
            _p = df["reb_player"].dropna().astype(str)
            _p = _p[~_p.str.upper().isin({"TEAM", ""})].value_counts()
            return ", ".join(f"{_S(n)} ({int(c)})" for n, c in _p.head(3).items())

        _order = ["Missed layup / dunk", "Missed two-point jumper", "Missed three", "Missed free throw"]
        for _dir, _mask, _want_off, _prefix in (
                ("their", _is_them, True, "Their"),
                ("opp", ~_is_them, False, "Opponent")):
            _d = _pairs[_mask]
            for _kind in _order + ["All misses"]:
                _k = _d if _kind == "All misses" else _d[_d["kind"] == _kind]
                _paired = _k[_k["offensive"].notna()]
                if len(_k) < _REB_MIN_MISSES or _paired.empty:
                    continue
                if _want_off:
                    # Their miss: an OFFENSIVE rebound is theirs -- that's their crash rate.
                    _got = _paired[_paired["offensive"] == True]
                    _pct = round(100 * len(_got) / len(_paired))
                    _label = f"{_prefix} {_kind.lower()}" if _kind != "All misses" else "All their misses"
                    _note = (f"{_pct}% offensive rebound on {len(_paired)} of {len(_k)} misses "
                             f"({len(_k) - len(_paired)} unpaired)")
                else:
                    # Opponent's miss: a DEFENSIVE rebound is theirs -- who secures it.
                    _got = _paired[_paired["offensive"] == False]
                    _pct = round(100 * len(_got) / len(_paired))
                    _label = f"{_prefix} {_kind.lower()}" if _kind != "All misses" else "All opponent misses"
                    _note = (f"They secure {_pct}% of {len(_paired)} opponent misses "
                             f"({len(_k) - len(_paired)} unpaired)")
                _reb_real.append([_gp_short, _label, _pct, _who(_got), _note])
        print(f"  rebound_tendencies: {len(_pairs)} misses in their play-by-play, "
              f"{int(_pairs['offensive'].notna().sum())} paired to a rebound -> {len(_reb_real)} row(s).")
else:
    print("  rebound_tendencies: no opponent play-by-play with event_order -- staying sample.")
rebound_tendencies = _gp_write("rebound_tendencies", _reb_cols, _reb_sample, real_rows=_reb_real)

_gs_cols = ["opponent", "situation", "ppp", "primary_actions", "note"]
_gs_sample = [
    [_gp_short, "Leading by 10+", 0.85, "Slows down, more Hi-Lo DHO", "Stops running in transition"],
    [_gp_short, "Trailing by 10+", 1.15, "More ball screens, faster pace", f"{_S(_gp_driver)} takes over possessions"],
    [_gp_short, "Clutch (last 5 min, margin \u2264 8)", 0.97, "Isolation for " + _S(_gp_driver),
     "Goes away from their sets late"],
]

# ---- REAL game situation splits, reusing the exact clutch definition already used elsewhere in this
# pipeline (see the "Clutch-time event log" cell and the "Game situation" tag in the Play calls cell) --
# not a new definition invented for this table. Reads level="Game situation" in uww_play_call_summary.
_gs_real = []
if not _gp_pcs.empty and {"side", "level", "name", "uses"}.issubset(_gp_pcs.columns):
    _gs_rows_df = _gp_pcs[(_gp_pcs["side"] == "Opponent") & (_gp_pcs["level"] == "Game situation")
                          & (_gp_pcs["scouted_opponent"].astype(str) == str(_gp_short))
                          & (_gp_pcs["uses"] >= _GS_MIN_USES)]
    # Same left-to-right order as the sample rows, when present.
    _gs_order = ["Leading by 10+", "Trailing by 10+", "Clutch (last 5 min, margin \u2264 8)"]
    _gs_rows_df = _gs_rows_df.assign(_ord=_gs_rows_df["name"].apply(
        lambda n: _gs_order.index(n) if n in _gs_order else len(_gs_order))).sort_values("_ord")
    for _, r in _gs_rows_df.iterrows():
        _situ = str(r["name"])
        _calls = (_gp_pc[(_gp_pc["side"] == "Opponent") & (_gp_pc["game_situation"] == _situ)
                         & ~_gp_pc["play_call"].astype(str).str.contains("unspecified", na=False)
                         & _gp_pc["play_call"].notna()]["play_call"].value_counts().head(2)
                  if not _gp_pc.empty and "game_situation" in _gp_pc.columns else pd.Series(dtype=int))
        _what = ", ".join(f"{n} ({int(c)}x)" for n, c in _calls.items()) or (r.get("top_action") or "--")
        _note = f"{int(r['uses'])} tagged possessions"
        if pd.notna(r.get("team_ppp")):
            _note += f" (their overall {r['team_ppp']:.2f} PPP)"
        _gs_real.append([_gp_short, _situ, r.get("ppp"), _what, _note])

game_situation_splits = _gp_write("game_situation_splits", _gs_cols, _gs_sample, real_rows=_gs_real)

_sq_cols = ["opponent", "contest_level", "freq_pct", "fg_pct", "note"]
_sq_sample = [
    [_gp_short, "Wide open (no closeout)", 15, 58, "Mostly transition and scramble situations"],
    [_gp_short, "Open (closeout, no contest)", 30, 44, f"{_S(_gp_shooter)} shoots this well above the others"],
    [_gp_short, "Contested (hand up, on time)", 40, 33, "Team average drops hard here"],
    [_gp_short, "Tightly contested (late or rushed)", 15, 19, "Mostly late-clock possessions"],
]
shot_quality_by_contest = _gp_write("shot_quality_by_contest", _sq_cols, _sq_sample)

_dt_cols = ["opponent", "trigger", "from_where", "escape_read", "note"]
_dt_sample = [
    [_gp_short, f"{_S(_gp_big)} post touch below the block", "Baseline dig", "Kicks out to the corner",
     "Corner shooter is the read -- deny that pass first"],
    [_gp_short, f"{_S(_gp_driver)} isolation, dribbles into the lane", "Nail help", "Drives through it more than passing out",
     "Live-ball turnover risk if we trap instead of just digging"],
]
double_team_tendencies = _gp_write("double_team_tendencies", _dt_cols, _dt_sample)

_osn_cols = ["opponent", "screen_type", "technique", "freq_pct", "note"]
_osn_sample = [
    [_gp_short, "Down screen / pin down", "Trail (go over)", 60, f"Run {_S(_gp_shooter)} off these -- they chase, don't switch"],
    [_gp_short, "Flare screen", "Switch", 55, "Safer to attack the mismatch than the shot itself"],
    [_gp_short, "Stagger (two screens)", "Fight through first, switch second", 50, "Confusion point -- late defender is open man's man"],
]
offball_screen_navigation = _gp_write("offball_screen_navigation", _osn_cols, _osn_sample)

for _i, r in _gp_opp.head(5).iterrows():
    _d = _us_starters[_i] if _i < len(_us_starters) else ""
    _b = _us_bench[_i] if _i < len(_us_bench) else ""
    _match_sample.append([_gp_short, r["player"], _d, _b,
                          "Leading scorer -- top priority" if r["player"] == _top_threat else ""])
matchups = _gp_write("matchups", _match_cols, _match_sample)

_scout_cols = ["opponent", "scout_player", "plays_as", "imitate"]
_scout_sample = []
_bench_pool = list(_gp_us["player"].iloc[5:]) if len(_gp_us) > 5 else list(_gp_us["player"])
for _i, r in _gp_opp.head(5).iterrows():
    if _i >= len(_bench_pool):
        break
    _role = ("Post seals and offensive glass" if r["player"] == _gp_big else
             "Downhill drives, draw contact" if r["player"] == _gp_driver else
             "Catch-and-shoot off flares" if r["player"] == _gp_shooter else
             "Initiate sets, first pass" if r["player"] == _gp_handler else "Spot up, cut hard")
    _scout_sample.append([_gp_short, _bench_pool[_i], r["player"], _role])
scout_team = _gp_write("scout_team", _scout_cols, _scout_sample)

_avail_cols = ["opponent", "player", "status", "note"]
_avail_sample = [[_gp_short, p, "Available", ""] for p in _us_starters]
if _avail_sample:
    _avail_sample[-1] = [_gp_short, _avail_sample[-1][1], "Limited", "Non-contact Monday, full Tuesday"]
uww_availability = _gp_write("uww_availability", _avail_cols, _avail_sample)

_film_cols = ["opponent", "clip_group", "clips", "who_watches"]
_film_sample = [
    [_gp_short, "Their base offense (5-Out Motion)", 6, "Whole team"],
    [_gp_short, f"{_S(_gp_big)} post touches and offensive rebounds", 5, "Bigs"],
    [_gp_short, "BLOB / SLOB / ATO", 8, "Whole team"],
    [_gp_short, "Their ball-screen coverage vs us-type guards", 5, "Guards"],
    [_gp_short, "Press break and 1-2-1-1", 4, "Whole team"],
]
film_clips = _gp_write("film_clips", _film_cols, _film_sample, real_rows=_film_real)


# ======================================================================================================
# PRACTICE PLAN -- emphasis items REAL (keys + flags), calendar SAMPLE until a staff calendar exists
# ======================================================================================================
def _gp_parse_display_date(text):
    """'Wed, Nov 19' -> datetime, using reference_date's season for the year (Nov-Dec vs Jan-Mar)."""
    ts = pd.to_datetime(str(text), errors="coerce", format="%a, %b %d")
    if pd.isna(ts):
        ts = pd.to_datetime(str(text), errors="coerce")
    if pd.isna(ts):
        return None
    ref = globals().get("reference_date") or datetime.now()
    year = ref.year if (ts.month >= 7) == (ref.month >= 7) else (ref.year + 1 if ref.month >= 7 else ref.year - 1)
    return ts.replace(year=year).to_pydatetime()


_ktv = _gp_load("uww_ktv_keys")
if not _ktv.empty and "opponent" in _ktv.columns:
    _ktv = _ktv[_ktv["opponent"].astype(str) == str(_gp_short)].sort_values("key_number")


def _gp_keys(category, n):
    if _ktv.empty:
        return []
    return [(int(r["key_number"]), str(r["headline"])) for _, r in _ktv[_ktv["category"] == category].head(n).iterrows()]


_flags = _gp_load("uww_coaching_flags")
_cleanup_players = []
if not _flags.empty and "sentiment" in _flags.columns:
    _cleanup_players = list(dict.fromkeys(_flags[_flags["sentiment"].astype(str) == "Negative"]["player"].head(3)))

_sched = _gp_load("uww_schedule")
_game_dt = _prev_dt = None
if not _sched.empty and "Upcoming" in _sched.columns:
    _uww_rows = _sched[_sched["team"].astype(str).str.contains("Whitewater", case=False, na=False)]
    _up = _uww_rows[_uww_rows["Upcoming"].astype(str).str.strip().str.lower() == "yes"]
    if not _up.empty:
        _game_dt = _gp_parse_display_date(_up.iloc[0].get("date"))
        _played = _uww_rows.loc[:_up.index[0]].iloc[:-1]
        _played = _played[_played["outcome"].astype(str).str.upper().isin(["W", "L"])]
        if not _played.empty:
            _prev_dt = _gp_parse_display_date(_played.iloc[-1].get("date"))

_plan_cols = ["opponent", "day", "date", "segment", "minutes", "detail", "ties_to", "schedule_is_sample"]

# ---- Recommendations the practice plan consumes (REAL) -------------------------------------------------
# Which defense hurts them least, and whether pressure actually costs them the ball. Same math the brief's
# WHAT TO PLAY THEM IN section prints, computed once here so the plan and the brief can't disagree.
# defense_FACED, never the raw defense_type -- on a mixed offense/defense file the raw field flips meaning.
_def_rec_text = ""
_press_rec_text = ""
_REC_MIN = 6
if not _gp_pc.empty and "defense_faced" in _gp_pc.columns:
    _rc = _gp_pc[(_gp_pc["side"] == "Opponent")
                 & (_gp_pc["offense_team"].astype(str) == str(_gp_short))
                 & (_gp_pc["decode_quality"] != "Needs review")].copy()
    if not _rc.empty:
        _rc["_pts"] = pd.to_numeric(_rc.get("points"), errors="coerce")

        def _rec_family(r):
            if bool(r.get("press_faced")):
                return "press"
            d = str(r.get("defense_faced") or "")
            return "zone" if "zone" in d.lower() else ("man" if "man" in d.lower() else "")

        _rc["_fam"] = _rc.apply(_rec_family, axis=1)
        _fams = {f: g for f, g in _rc[_rc["_fam"] != ""].groupby("_fam") if len(g) >= _REC_MIN}
        if len(_fams) >= 2:
            _scored = {f: (g["_pts"].sum() / g["_pts"].notna().sum()) for f, g in _fams.items()
                       if g["_pts"].notna().sum()}
            if _scored:
                _best_d = min(_scored, key=_scored.get)
                _def_rec_text = (f"Reps in {_best_d} -- they scored {_scored[_best_d]:.2f} PPP against it on "
                                 f"{len(_fams[_best_d])} tagged possessions, their lowest")
        _pr = _rc[_rc["press_faced"].astype(bool)] if "press_faced" in _rc.columns else pd.DataFrame()
        _np = _rc[~_rc["press_faced"].astype(bool)] if "press_faced" in _rc.columns else pd.DataFrame()
        if len(_pr) >= _REC_MIN and len(_np) >= _REC_MIN:
            _res_p = _pr.get("result", pd.Series("", index=_pr.index)).astype(str).str.lower()
            _res_n = _np.get("result", pd.Series("", index=_np.index)).astype(str).str.lower()
            _to_p = 100 * _res_p.str.contains("turnover|violation|kicked", regex=True).sum() / len(_pr)
            _to_n = 100 * _res_n.str.contains("turnover|violation|kicked", regex=True).sum() / len(_np)
            _press_rec_text = (
                f"Press reps -- pressure moved their turnover rate {_to_p - _to_n:+.1f} points "
                f"({_to_n:.0f}% to {_to_p:.0f}%) over {len(_pr)} pressed possessions"
                if _to_p - _to_n >= 5 else
                f"Skip the press -- pressure only moved their turnover rate {_to_p - _to_n:+.1f} points; "
                f"work half-court defense instead")

_plan = []
_def_keys, _off_keys, _pers_keys = _gp_keys("Defense", 3), _gp_keys("Offense", 2), _gp_keys("Personnel", 1)
_foul_names = ", ".join(_gp_last(p) for p in late_game_foul_list[late_game_foul_list["call"] == "Foul"]["player"])
_set_names = ", ".join(opp_sets[opp_sets["situation"] == "Half court"]["set_name"].head(3))
_so_names = ", ".join(opp_sets[opp_sets["situation"].astype(str).str.match(r"^(BLOB|SLOB|ATO)")]["set_name"].head(3))

if _game_dt is not None:
    from datetime import timedelta as _gp_td
    _start = (_prev_dt + _gp_td(days=1)) if _prev_dt else (_game_dt - _gp_td(days=3))
    _start = max(_start, _game_dt - _gp_td(days=5))
    _days = []
    _d = _start
    while _d < _game_dt:
        _days.append(_d)
        _d += _gp_td(days=1)
    _label = lambda d: f"{d:%a}, {d:%b} {d.day}"  # noqa: E731

    def _seg(day, date, segment, minutes, detail, ties_to=""):
        _plan.append([_gp_short, day, _label(date), segment, minutes, detail, ties_to, True])

    _practice_no = 0
    for _i, _d in enumerate(_days):
        _is_first, _is_last = _i == 0, _i == len(_days) - 1
        if _is_first and len(_days) >= 3:
            _seg("Recovery + film", _d, "Film", 40, f"Scout film: {_set_names or 'their base offense'}",
                 "Opponent sets")
            _seg("Recovery + film", _d, "Shooting", 20, "Form and free throws, no contact",
                 ", ".join(_cleanup_players) if _cleanup_players else "")
            continue
        _heavy = not _is_last
        _practice_no += 1
        _pday = f"Practice {_practice_no}" + ("" if _heavy else " (light)")
        _seg(_pday, _d, "Warm-up / dynamic", 10, "")
        if _cleanup_players and _heavy:
            _seg(_pday, _d, "Individual", 12, "Catch-and-shoot and free-throw reps",
                 "Player flags: " + ", ".join(_gp_last(p) for p in _cleanup_players))
        for _n, _h in (_def_keys if _heavy else _def_keys[:1]):
            _seg(_pday, _d, "Defensive install" if _heavy else "Defensive walkthrough",
                 12 if _heavy else 10, _h, f"Key {_n}")
        _seg(_pday, _d, "Scout team", 15,
             f"Scout team runs {_set_names or 'their base sets'}", "Opponent sets")
        # Close the loop (requested): the brief now COMPUTES which defense hurts them and whether pressing
        # is worth it, so the plan asks for reps at that instead of leaving the staff to re-derive it.
        if _def_rec_text and _heavy:
            _seg(_pday, _d, "Defensive emphasis", 10, _def_rec_text, "What to play them in")
        if _press_rec_text and _heavy:
            _seg(_pday, _d, "Press reps", 8, _press_rec_text, "Pressure response")
        for _n, _h in (_off_keys if _heavy else _off_keys[:1]):
            _seg(_pday, _d, "Offensive emphasis", 10, _h, f"Key {_n}")
        _seg(_pday, _d, "Special situations", 8,
             f"Defend {_so_names or 'their BLOB/SLOB/ATO'}", "Opponent sets")
        if _heavy:
            _seg(_pday, _d, "Late game", 8,
                 f"Foul-list reps -- foul: {_foul_names}" if _foul_names else "End-of-game situations",
                 "Late-game foul list" if _foul_names else "")
            _seg(_pday, _d, "Live 5-on-5", 15,
                 "Scout team in their personnel; stop on key violations"
                 + (f" -- {_pers_keys[0][1][0].lower()}{_pers_keys[0][1][1:]}" if _pers_keys else ""),
                 f"Key {_pers_keys[0][0]}" if _pers_keys else "")
    _seg("Game day", _game_dt, "Shootaround", 45,
         "Walk their sets, matchups and the foul list; 50 game-speed shots each", "Matchups")

practice_plan = _gp_write("practice_plan", _plan_cols, _plan)
# is_sample for the practice plan means "the whole plan came from the generator". The emphasis text inside it
# is real regardless -- schedule_is_sample carries the narrower claim, so a renderer can say exactly that.

# ======================================================================================================
# REAL: personnel tiers for BOTH teams -- Starters, then the three bench tiers the app's Personnel tab uses
# ======================================================================================================
# CONFIRMED CHANGE (requested): the brief's personnel pages are grouped the same way as the app -- a Starters
# page, then "Bench -- rotation", "Bench -- limited minutes" and "Bench -- no minutes in last N game(s)".
# Built here, once, for both teams, so the brief renders a table instead of re-deriving tiers itself.
#   Bench tiers: the app's own rule (opponent_bench_tiers in streamlit_app.py) -- a player with no minutes in
#     the last _PT_RECENT_WINDOW games is "no minutes"; otherwise under _PT_ROTATION_MIN_MPG minutes a game is
#     "limited"; otherwise "rotation". Minutes, not box-score appearances: a row of zeros isn't playing.
#   Starters: the scouting report's Starter role when one exists for the opponent (the staff's word wins);
#     otherwise players who started at least half of the recent games (`started` -- official box-score
#     asterisks for UWW, the first five-man unit on the floor for the opponent), up to five. With no start
#     data at all, the five highest minutes-per-game players over the recent window, and the brief says so.
_PT_RECENT_WINDOW = 3      # RECENT_GAMES_WINDOW in streamlit_app.py
_PT_ROTATION_MIN_MPG = 8.0  # ROTATION_MIN_MPG in streamlit_app.py
_PT_TIER_ORDER = ["Starter", "Bench \u2014 rotation", "Bench \u2014 limited minutes", "Bench \u2014 no minutes"]


def _pt_tiers(box, team_value, side, report_starters=()):
    if box.empty or "team" not in box.columns or "player" not in box.columns:
        return []
    own = box[(box["team"].astype(str) == str(team_value)) & (box["player"].astype(str) != "TEAM")].copy()
    if own.empty or "game_date" not in own.columns:
        return []
    own["_d"] = pd.to_datetime(own["game_date"], errors="coerce")
    games = sorted(own["_d"].dropna().unique(), reverse=True)
    if not games:
        return []
    recent = set(games[:_PT_RECENT_WINDOW])
    own["_min"] = pd.to_numeric(own["MIN"], errors="coerce") if "MIN" in own.columns else float("nan")
    has_min = own["_min"].notna().any()
    played = own[own["_min"].fillna(0) > 0] if has_min else own
    own["_started"] = own["started"].astype(str).str.lower().isin(["true", "1"]) if "started" in own.columns else False
    rows = []
    for name in own["player"].astype(str).unique():
        p_rows = played[played["player"].astype(str) == name]
        appearances = set(p_rows["_d"].dropna().unique())
        recent_count = len(appearances & recent)
        mpg = float(p_rows["_min"].mean()) if has_min and not p_rows.empty else None
        missed = 0
        for g in games:
            if g in appearances:
                break
            missed += 1
        starts_recent = int(own[(own["player"].astype(str) == name) & own["_d"].isin(recent)]["_started"].sum())
        rows.append({"side": side, "team": team_value, "player": name, "games": len(appearances),
                     "mpg": round(mpg, 1) if mpg is not None else None, "recent_count": recent_count,
                     "missed_recent": missed, "starts_recent": starts_recent, "recent_n": len(recent)})
    df = pd.DataFrame(rows)
    # Starters.
    starter_basis = ""
    report_starters = {str(n).strip().lower() for n in report_starters if str(n).strip()}
    if report_starters:
        starters = set(df[df["player"].str.lower().isin(report_starters)]["player"])
        starter_basis = "scouting report role"
    elif df["starts_recent"].sum() > 0:
        cand = df[df["starts_recent"] >= max(1, -(-len(recent) // 2))]
        starters = set(cand.sort_values(["starts_recent", "mpg"], ascending=False).head(5)["player"])
        starter_basis = f"started {max(1, -(-len(recent) // 2))}+ of the last {len(recent)} game(s)"
    else:
        starters = set(df[df["recent_count"] > 0].sort_values("mpg", ascending=False).head(5)["player"])
        starter_basis = f"no start data -- top five in minutes over the last {len(recent)} game(s)"

    def tier(r):
        if r["player"] in starters:
            return "Starter"
        if r["recent_count"] == 0:
            return "Bench \u2014 no minutes"
        if r["mpg"] is not None and r["mpg"] < _PT_ROTATION_MIN_MPG:
            return "Bench \u2014 limited minutes"
        return "Bench \u2014 rotation"

    df["tier"] = df.apply(tier, axis=1)
    df["tier_order"] = df["tier"].map(_PT_TIER_ORDER.index)
    df["starter_basis"] = starter_basis
    return df.to_dict("records")


_pt_report_starters = []
for _pt_src_name in ("uww_opponent_rosters", "uww_player_profiles"):
    _pt_src = _gp_load(_pt_src_name)
    if not _pt_src.empty and {"opponent", "name", "role"}.issubset(_pt_src.columns):
        _pt_report_starters = list(_pt_src[(_pt_src["opponent"].astype(str) == str(_gp_short))
                                           & (_pt_src["role"].astype(str).str.lower() == "starter")]["name"])
        if _pt_report_starters:
            break

personnel_tiers = pd.DataFrame(
    _pt_tiers(_gp_prior, _gp_short, "Opponent", _pt_report_starters) + _pt_tiers(_gp_box, _GP_UWW, "UWW"),
    columns=["side", "team", "player", "tier", "tier_order", "games", "mpg", "recent_count", "missed_recent",
             "starts_recent", "recent_n", "starter_basis"])
personnel_tiers.insert(0, "scouted_opponent", _gp_short)
personnel_tiers.to_csv(os.path.join(APP_DATA_DIR, "uww_personnel_tiers.csv"), index=False)

# ======================================================================================================
# REAL: UWW's own season five-man units, so the brief's UWW section can mirror the opponent's Top Lineups
# ======================================================================================================
_uwl = _gp_load("uww_lineup_stints")
uww_lineup_season = pd.DataFrame(columns=["lineup", "GP", "MIN", "+/-"])
if not _uwl.empty and "uww_lineup" in _uwl.columns:
    _uwl = _uwl.dropna(subset=["uww_lineup"]).copy()
    _uwl["_min"] = pd.to_numeric(_uwl.get("stint_minutes"), errors="coerce")
    _uwl["_pm"] = pd.to_numeric(_uwl.get("uww_margin_change"), errors="coerce")
    uww_lineup_season = (_uwl.groupby("uww_lineup")
                         .agg(GP=("game_date", "nunique"), MIN=("_min", "sum"), **{"+/-": ("_pm", "sum")})
                         .reset_index().rename(columns={"uww_lineup": "lineup"}))
    uww_lineup_season["MIN"] = uww_lineup_season["MIN"].round(1)
    uww_lineup_season = uww_lineup_season.sort_values("MIN", ascending=False)
uww_lineup_season.to_csv(os.path.join(APP_DATA_DIR, "uww_uww_lineup_season.csv"), index=False)

print(f"Wrote game-plan tables for {_gp_short or 'no opponent'}:")
for _name, _df in (("uww_late_game_foul_list", late_game_foul_list), ("uww_sample_size_warnings", sample_size_warnings),
                   ("uww_opp_sets", opp_sets), ("uww_opp_defense", opp_defense),
                   ("uww_opp_personnel", opp_personnel), ("uww_opp_shot_zones", opp_shot_zones),
                   ("uww_opp_tendencies", opp_tendencies), ("uww_matchups", matchups),
                   ("uww_scout_team", scout_team), ("uww_uww_availability", uww_availability),
                   ("uww_film_clips", film_clips), ("uww_shot_clock_tendencies", shot_clock_tendencies),
                   ("uww_defense_by_situation", defense_by_situation),
                   ("uww_ball_screen_coverage", ball_screen_coverage),
                   ("uww_help_rotation_tendencies", help_rotation_tendencies),
                   ("uww_personnel_grouping", personnel_grouping), ("uww_rebound_tendencies", rebound_tendencies),
                   ("uww_game_situation_splits", game_situation_splits),
                   ("uww_shot_quality_by_contest", shot_quality_by_contest),
                   ("uww_double_team_tendencies", double_team_tendencies),
                   ("uww_offball_screen_navigation", offball_screen_navigation),
                   ("uww_matchup_history", matchup_history), ("uww_play_counters", play_counters),
                   ("uww_practice_plan", practice_plan), ("uww_personnel_tiers", personnel_tiers),
                   ("uww_tempo_profile", tempo_profile), ("uww_trend_profile", trend_profile),
                   ("uww_uww_lineup_season", uww_lineup_season)):
    _src = ("SAMPLE" if (not _df.empty and "is_sample" in _df.columns and bool(_df["is_sample"].all()))
            else ("real" if not _df.empty else "empty"))
    print(f"  {_name:28s} {len(_df):3d} row(s)  [{_src}]")
print(f"  Staff input templates: {os.path.abspath(_GP_TEMPLATE_DIR)}")
if _gp_problems:
    for _p in _gp_problems:
        print(f"  - {_p}")


  opp_defense: real data for 3 item(s).
  player_shot_profile: 12 of theirs, 18 of ours (rim / other 2 / three, from play-by-play).
  defense_by_situation: real data for 2 situation(s).
  ball_screen_coverage: no "Ball screen coverage" level in uww_play_call_summary -- the notebook needs a full re-run from the play-calls cell so the new Title tagging gets decoded. Staying sample.
  rebound_tendencies: 320 misses in their play-by-play, 317 paired to a rebound -> 8 row(s).
Wrote game-plan tables for Aurora Spartans:
  uww_late_game_foul_list        7 row(s)  [real]
  uww_sample_size_warnings       6 row(s)  [real]
  uww_opp_sets                   6 row(s)  [real]
  uww_opp_defense                3 row(s)  [real]
  uww_opp_personnel             13 row(s)  [real]
  uww_opp_shot_zones             5 row(s)  [SAMPLE]
  uww_opp_tendencies             5 row(s)  [SAMPLE]
  uww_matchups                   5 row(s)  [SAMPLE]
  uww_scout_team                 5 row(s)  [SAMPLE]
  uww_uww_availability


### Build an HTML scouting brief for the upcoming opponent

A one-file executive summary for the coaching staff to read before they write their own scouting reports: how the two teams stack up, where the game is won, their personnel and how to guard them, the five-man units they trust, their recent form, our own watch-outs, and the projection. Built from the CSVs the export cell above just wrote -- the same files the Streamlit app reads -- so the brief and the app can't disagree about a number. Lands in `<OUTPUT_DIR>/scouting_briefs/`.


In [170]:
# --- Build a one-file HTML scouting brief for the upcoming opponent ----------------------------------------
# An executive summary the staff can read (or print, or forward) BEFORE they sit down to write their own
# scouting reports -- the handful of things from across this whole notebook that change how you prepare,
# not a dump of every table.
#
# Reads the CSVs the export cell just wrote rather than the in-memory DataFrames above. That is deliberate:
# those CSVs are exactly what the Streamlit app reads, so the brief and the app can never quietly disagree
# about a number -- and a coach who checks a figure in the app against the emailed brief gets the same
# answer. It also means this cell can be re-run on its own without re-running the notebook.
#
# Output is a single self-contained file: no images, no scripts, no external CSS, so it survives being sent
# as an email attachment and prints cleanly.
#
# Every section stands or falls on its own. With before_scout="yes" there is no game plan, no player notes
# and no tag-based comparisons, so those sections drop out and the footer names the tables that were empty
# -- a coach should never have to wonder whether data was missing or the brief was broken.
import html as _sb_html

_SB_OUT_DIR = os.path.join(APP_DATA_DIR, "scouting_briefs")

# ---- THE BOTTOM LINE: when a conditional sentence is printed ------------------------------------------
# Some Bottom Line sentences only appear past a threshold -- below it the difference isn't big enough to
# change how we prepare, and printing it anyway trains the staff to skim the section. These are the ONLY
# place the thresholds live: the rule text in each sentence's note (which the brief links to, and the app
# prints) is generated from them, so changing a number here updates the explanation too.
# Roster layout. "table" = one wide row per player (the original). "stacked" = a header bar per player
# (photo, name, bio) with the full width below for the box-score line, WHERE HE SHOOTS FROM (from the
# play-by-play) and the reads. Stacked replaces the separate SHOT LOCATIONS section, whose zone split is a
# placeholder; with "table" that section stays where it was.
ROSTER_LAYOUT = "table"

BOTTOM_LINE_RULES = {
    # Tempo: printed when EITHER is true.
    "tempo_pace_gap": 5,          # their possessions per game minus ours, either direction
    "tempo_early_share": 40,      # % of their tagged possessions that shoot inside 10 seconds
    # Four factors: printed when BOTH are true, so it names one clear story rather than a close call.
    "ff_min_weighted": 1.0,       # the top factor's weighted gap, absolute
    "ff_dominance": 2.0,          # ...and at least this many times the second-largest factor's
    # Style matchups (TEAMS LIKE ... panels, no longer their own sections): printed when ALL are true.
    "style_min_games": 3,         # games behind the record
    "style_min_confidence": 0.35, # best match in the panel at least this confident (the app's amber/red cut)
    "style_lopsided": 0.75,       # win share at/above this, or at/below 1 minus this
    # Rebounding thresholds live in REBOUND_RULES (Keys to Victory cell), because the key, this sentence
    # and the roster read must all fire on the same numbers. Read from there, never restated here.
}


_SB_UWW = "UW-Whitewater"


# --------------------------------------------------------------------------------------------
# Loading
# --------------------------------------------------------------------------------------------

class _SbData:
    """CSV access that never raises on a missing or empty table.

    A scouting report assembled from twenty-odd tables should not fail to build because one of
    them hasn't been produced yet -- an early-season run legitimately has no lineup data, and a
    before_scout run legitimately has no game plan. Missing is recorded, not fatal.
    """

    def __init__(self, data_dir):
        self.dir = data_dir
        self.missing = []
        self._cache = {}

    def __call__(self, name):
        if name not in self._cache:
            path = os.path.join(self.dir, f"{name}.csv")
            try:
                df = pd.read_csv(path)
            except Exception:
                df = pd.DataFrame()
            if df.empty:
                self.missing.append(name)
            self._cache[name] = df
        return self._cache[name]


def _sb_num(series):
    return pd.to_numeric(series, errors="coerce")


def _sb_pct(made, att):
    made, att = float(made or 0), float(att or 0)
    return round(100 * made / att, 1) if att else None


def _sb_fmt(value, digits=1, dash="--"):
    if value is None or (isinstance(value, float) and pd.isna(value)) or value == "":
        return dash
    if isinstance(value, (int, float)):
        return f"{value:.{digits}f}" if digits else f"{value:.0f}"
    return str(value)


def _sb_esc(value):
    return _sb_html.escape("" if value is None else str(value))


def _sb_clean(value):
    """One scalar's worth of the same test _sb_has_text applies to a column."""
    text = "" if value is None else str(value).strip()
    return "" if text.lower() in ("nan", "none") else text


def _sb_split(value):
    """Split a pipe-joined field into its items, dropping anything blank.

    Reads go through here rather than `str(value or "").split(" | ")`, which looks safe and isn't: an
    empty CSV cell arrives from pandas as a float NaN, and NaN is TRUTHY, so the `or ""` never fires and
    str(NaN) yields the literal text "nan" -- which then renders as a bullet point reading "nan".
    """
    text = _sb_clean(value)
    return [part.strip() for part in text.split(" | ") if part.strip()] if text else []


def _sb_has_text(series):
    """True where a column holds something a coach would actually read.

    A missing scouting note arrives here three different ways -- a real NaN, an empty string, or
    the literal text "nan" left behind by an earlier astype(str) -- and all three have to count as
    empty, or a player with no notes gets his own write-up block containing nothing.
    """
    # fillna BEFORE astype(str): in this pandas version .astype(str) on a float NaN leaves a real
    # null rather than the text "nan", so every subsequent comparison returns True and a player
    # with no notes passes the filter. The same trap is documented in the parser itself.
    cleaned = series.fillna("").astype(str).str.strip()
    return cleaned.ne("") & ~cleaned.str.lower().isin(["nan", "none"])


# --------------------------------------------------------------------------------------------
# Who are we playing
# --------------------------------------------------------------------------------------------

def _sb_resolve_matchup(_sb_d):
    """The upcoming game, and the opponent's short name as every other table spells it.

    uww_schedule holds every team's rows (_SB_UWW's own plus each opponent's), so filter to _SB_UWW's
    before looking for the Upcoming flag. The short name is then resolved against the tables that
    actually key on it, rather than assumed to equal the schedule's own opponent text -- those two
    agree in the normal case, but the whole point of pinning it down here is so that they can't
    quietly disagree in the report.
    """
    sched = _sb_d("uww_schedule")
    if sched.empty or "Upcoming" not in sched.columns:
        return None, None, None

    uww_rows = sched[sched["team"].astype(str).str.contains("Whitewater", case=False, na=False)]
    upcoming = uww_rows[uww_rows["Upcoming"].astype(str).str.strip().str.lower() == "yes"]
    if upcoming.empty:
        return None, None, None
    game = upcoming.iloc[0]
    scheduled_name = str(game["opponent"]).strip()

    candidates = set()
    for table, col in (("uww_player_profiles", "opponent"), ("uww_opponent_team_totals", "opponent"),
                       ("uww_opponent_rosters", "opponent")):
        df = _sb_d(table)
        if not df.empty and col in df.columns:
            candidates |= set(df[col].dropna().astype(str))

    short = next(
        (c for c in sorted(candidates, key=len, reverse=True)
         if c.lower() in scheduled_name.lower() or scheduled_name.lower() in c.lower()),
        scheduled_name,
    )
    return game, scheduled_name, short


# --------------------------------------------------------------------------------------------
# Team-level numbers
# --------------------------------------------------------------------------------------------

def _sb_team_line(box, team_col, team_value, invert=False):
    """Per-game team averages straight from a play-by-play-derived box score.

    Deliberately NOT read off uww_player_profiles: that table mixes conventions (PTS/REB/MIN are
    per game, AST/STL/BLK/TO are season totals awaiting a games-played divisor), which is exactly
    the kind of thing that produces a report where two numbers on the same row mean different
    things. A raw box score has one convention: totals. Divide once, here.
    """
    if box.empty or team_col not in box.columns:
        return {}
    mask = box[team_col].astype(str) == str(team_value)
    rows = box[~mask] if invert else box[mask]
    if rows.empty:
        return {}

    games = rows["game_date"].nunique() if "game_date" in rows.columns else 0
    if not games:
        return {}

    total = {c: _sb_num(rows[c]).sum() if c in rows.columns else 0
             for c in ("PTS", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA",
                       "REB", "OREB", "DREB", "AST", "STL", "BLK", "TO")}
    return {
        "games": games,
        "PTS": total["PTS"] / games,
        "REB": total["REB"] / games,
        "AST": total["AST"] / games,
        "TO": total["TO"] / games,
        "STL": total["STL"] / games,
        "BLK": total["BLK"] / games,
        "FG%": _sb_pct(total["FGM"], total["FGA"]),
        "3P%": _sb_pct(total["FG3M"], total["FG3A"]),
        "FT%": _sb_pct(total["FTM"], total["FTA"]),
        "3PA": total["FG3A"] / games,
        "FGA": total["FGA"] / games,
        # Pace: possessions per game, same estimate the app's tempo table uses (FGA - OREB + TO +
        # 0.475*FTA) over the same rows -- players AND the synthetic TEAM row, which carries team
        # turnovers. Left out entirely when the box has no OREB column: the sum above would silently
        # treat it as zero and overstate pace by every offensive rebound.
        "PACE": ((total["FGA"] - total["OREB"] + total["TO"] + 0.475 * total["FTA"]) / games
                 if "OREB" in rows.columns else None),
    }


# --- Four Factors ---------------------------------------------------------------------------------------
# Dean Oliver's own weighting -- shooting matters most, free throws least. Copied from the app rather than
# re-chosen, so the brief's table and the app's dialog can't rank the factors differently.
_SB_FF_WEIGHTS = {"eFG%": 0.40, "TOV%": 0.25, "ORB%": 0.20, "FT Rate": 0.15}
# Higher is better for three of them; TOV% is the exception -- a turnover is a lost possession.
_SB_FF_HIGHER_IS_BETTER = {"eFG%": True, "TOV%": False, "ORB%": True, "FT Rate": True}


def _sb_possessions(fga, oreb, to, fta):
    """The standard possession estimate. A box score carries no possession count, so this is the widely
    used approximation: a possession ends on a made shot, a defensive rebound, a turnover, or the last free
    throw of a trip -- 0.44 approximating how often an FTA is the last of its trip."""
    return fga - oreb + to + 0.44 * fta


def _sb_four_factors(team_box, opp_box):
    """eFG%, TOV%, ORB% and FT Rate for team_box's side. opp_box (the other side over the same games) is
    required for ORB%, whose denominator is the opponent's defensive rebounds."""
    def total(df, col):
        return _sb_num(df[col]).sum() if col in df.columns else 0

    fgm, fga = total(team_box, "FGM"), total(team_box, "FGA")
    fg3m, to = total(team_box, "FG3M"), total(team_box, "TO")
    fta, oreb = total(team_box, "FTA"), total(team_box, "OREB")
    opp_dreb = total(opp_box, "DREB")
    poss = _sb_possessions(fga, oreb, to, fta)
    return {
        "eFG%": ((fgm + 0.5 * fg3m) / fga * 100) if fga > 0 else None,
        "TOV%": (to / poss * 100) if poss > 0 else None,
        "ORB%": (oreb / (oreb + opp_dreb) * 100) if (oreb + opp_dreb) > 0 else None,
        "FT Rate": (fta / fga * 100) if fga > 0 else None,
    }


def _sb_player_table(_sb_d, short):
    """One row per opponent player: identity from the roster, production from their own film."""
    box = _sb_d("uww_opponent_prior_games_box_score")
    profiles = _sb_d("uww_player_profiles")

    rows = []
    if not box.empty and "team" in box.columns:
        own = box[box["team"].astype(str) == str(short)]
        own = own[own["player"].astype(str) != "TEAM"]
        if not own.empty:
            grouped = own.groupby("player")
            for name, g in grouped:
                games = g["game_date"].nunique() if "game_date" in g.columns else len(g)
                total = {c: _sb_num(g[c]).sum() if c in g.columns else 0
                         for c in ("PTS", "REB", "AST", "STL", "BLK", "TO", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA")}
                rows.append({
                    "name": str(name),
                    "games": games,
                    "PTS": total["PTS"] / games if games else None,
                    "REB": total["REB"] / games if games else None,
                    "AST": total["AST"] / games if games else None,
                    "STL": total["STL"] / games if games else None,
                    "BLK": total["BLK"] / games if games else None,
                    "TO": total["TO"] / games if games else None,
                    "FG%": _sb_pct(total["FGM"], total["FGA"]),
                    "3P%": _sb_pct(total["FG3M"], total["FG3A"]),
                    "FT%": _sb_pct(total["FTM"], total["FTA"]),
                    # Season totals kept alongside the per-game rates: the reads below need volume to
                    # decide whether a percentage means anything, and a rate alone can't say that.
                    "FGA": total["FGA"],
                    "3PA": total["FG3A"],
                    "FTA": total["FTA"],
                    "PTS_total": total["PTS"],
                })

    table = pd.DataFrame(rows)
    if table.empty:
        return table

    ident_cols = ["name", "jersey_number", "position", "height", "class_year", "role",
                  "player_notes", "keys_to_defending", "notes_tags_display", "keys_tags_display"]
    # Two identity sources, tried in order, the second only filling gaps the first left. Both are
    # built from the scouting report, so with before_scout="yes" neither exists and this is a no-op
    # -- which is why the jersey fallback below reads the play-by-play name itself.
    for ident_table in (profiles, _sb_d("uww_opponent_rosters")):
        if ident_table.empty or not {"opponent", "name"}.issubset(ident_table.columns):
            continue
        ident = ident_table[ident_table["opponent"].astype(str) == str(short)]
        ident = ident[[c for c in ident_cols if c in ident.columns]].drop_duplicates("name")
        if ident.empty:
            continue
        table = table.merge(ident, on="name", how="left", suffixes=("", "_alt"))
        for col in ident_cols:
            alt = f"{col}_alt"
            if alt in table.columns:
                table[col] = table[col].where(_sb_has_text(table[col]), table[alt])
                table = table.drop(columns=[alt])

    for col in ident_cols:
        if col not in table.columns:
            table[col] = None

    # Some play-by-play feeds prefix the jersey onto the name ("3 Jalen Ward"); others carry no
    # number at all. Take one off the front when it's there, and strip it from the displayed name so
    # it isn't printed twice. When neither the roster nor the feed has a number the column stays
    # blank, which is the honest answer -- better than inventing one.
    parsed = table["name"].astype(str).str.extract(r"^#?\s*(\d{1,2})\s+(.*)$")
    table["jersey_number"] = table["jersey_number"].where(
        _sb_has_text(table["jersey_number"]), parsed[0])
    table["name"] = parsed[1].where(parsed[1].notna(), table["name"])

    table["role"] = table["role"].fillna("Bench")
    # Season volume first: ordering by PPG put a one-game 15-point night above a four-game 13.5 ppg starter.
    return table.sort_values(["PTS_total", "PTS"], ascending=False).reset_index(drop=True)


# --------------------------------------------------------------------------------------------
# HTML
# --------------------------------------------------------------------------------------------

# --------------------------------------------------------------------------------------------
# Presentation -- deliberately the app's own visual language
# --------------------------------------------------------------------------------------------
# Every value below is lifted from streamlit_app.py rather than invented here: Montserrat and the
# --warhawk-* variables from its global stylesheet, #1a1a2e / 10px radius / #9DAAAC labels from the
# Upcoming Game banner, the 1px #e0e0e0 + 8px radius bordered box from section_header(), and the
# "N. <headline>" + dimmed 0.8rem evidence line from the Keys to Victory renderer. A coach
# reading this next to the app should not be able to tell they came from different code.

_SB_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Montserrat:wght@400;500;600;700;800&display=swap');

:root {
  --warhawk-purple: #4E2A84;
  --warhawk-gray: #9DAAAC;
  --warhawk-light: #F0EDF5;
  --banner: #1a1a2e;
  --edge: #e0e0e0;
  --body: #262730;
  --dim: #666;
  --win: #2e7d32;
  --loss: #c62828;
}
* { box-sizing: border-box; }
body {
  margin: 0; padding: 18px 18px 56px;
  background: #fff; color: var(--body);
  font-family: 'Montserrat', -apple-system, "Segoe UI", Helvetica, Arial, sans-serif;
  font-size: 15px; line-height: 1.5;
  -webkit-text-size-adjust: 100%;
}
.page { max-width: 1000px; margin: 0 auto; }
p { margin: 0 0 0.65rem; }

/* section_header() */
.sect { border: 1px solid var(--edge); border-radius: 8px; padding: 12px 16px; margin: 1.5rem 0 0.75rem; }
.sect .t { font-weight: 800; font-size: 1.05rem; letter-spacing: 0.5px; color: var(--warhawk-purple); }
.card { border: 1px solid var(--edge); border-radius: 8px; padding: 12px 16px; margin: 0 0 0.75rem; }
.card .hd { font-weight: 800; font-size: 1.05rem; letter-spacing: 0.5px; margin-bottom: 8px; }

/* Two panels side by side, the way the app pairs Team Stats with Last Five Games. min-width:0 is
   load-bearing: without it a flex child refuses to shrink below its content's intrinsic width, and
   the roster table's eight columns push the pair wider than the page instead of compressing. */
.cols { display: flex; gap: 16px; align-items: flex-start; }
.cols > .col { flex: 1; min-width: 0; }
.cols .sect { margin-top: 0; }
.cols table { font-size: 0.78rem; }
.cols th, .cols td { padding-left: 5px; padding-right: 5px; }
.lede { font-size: 0.95rem; }
ul.bl { margin: 0; padding-left: 1.1rem; }
ul.bl li { font-size: 0.95rem; margin-bottom: 5px; }
ul.bl li::marker { color: var(--warhawk-purple); }
ul.bl li:last-child { margin-bottom: 0; }

/* banner */
.banner { background: var(--banner); border-radius: 10px; padding: 22px 32px; margin-bottom: 0.75rem;
          display: flex; align-items: center; justify-content: space-between; }
.banner .side { text-align: center; flex: 1; display: flex; flex-direction: column; align-items: center; }
.banner .mid { justify-content: center; }
.banner .crest { height: 64px; display: flex; align-items: center; justify-content: center; margin-bottom: 8px; }
.banner .crest img { max-height: 64px; max-width: 90px; object-fit: contain; }
.banner .team { color: #fff; font-weight: 800; font-size: 1.4rem; letter-spacing: 0.5px; }
.banner .rec { color: var(--warhawk-gray); font-size: 1.05rem; font-weight: 600; margin-top: 3px; }
.banner .streak { color: #aabbcc; font-size: 0.8rem; font-style: italic; margin-top: 2px; }
.banner .when { color: var(--warhawk-gray); font-size: 1rem; font-weight: 500; }
.banner .vs { color: #fff; font-size: 1.6rem; font-weight: 700; margin: 4px 0; }
.banner .where { color: var(--warhawk-gray); font-size: 0.95rem; }

/* team stats: UWW left, stat centre, opponent right */
.cmp .row { display: flex; align-items: center; justify-content: space-between;
            padding: 7px 4px; border-bottom: 1px solid #f0f0f0; }
.cmp .row:last-child { border-bottom: 0; }
.cmp .us, .cmp .them { flex: 1; font-size: 0.95rem; font-weight: 700; font-variant-numeric: tabular-nums; }
.cmp .us { color: var(--warhawk-purple); }
.cmp .them { color: #222; text-align: right; }
.cmp .stat { flex: 1.1; text-align: center; font-size: 0.8rem; color: var(--dim); font-weight: 600; }
.cmp .who { display: flex; justify-content: space-between; align-items: center;
            margin-bottom: 8px; padding: 0 4px; }
.cmp .who .l { font-size: 0.95rem; font-weight: 700; color: var(--warhawk-purple); }
.cmp .who .c { font-size: 0.85rem; color: #888; }
.cmp .who .r { font-size: 0.95rem; font-weight: 700; color: #222; }
.lead { font-size: 0.68rem; font-weight: 700; padding: 1px 6px; border-radius: 8px; margin-left: 6px;
        vertical-align: middle; }
.lead.us { background: var(--warhawk-light); color: var(--warhawk-purple); }
.lead.them { background: #f1f1f1; color: #555; }

.ff-lede { font-size: 0.88rem; margin: 0 0 6px; }
td.ff-us { color: var(--warhawk-purple); font-weight: 700; }
td.ff-them { color: var(--loss); font-weight: 700; }

/* keys */
.ktvcat { font-size: 0.72rem; font-weight: 800; letter-spacing: 0.8px; text-transform: uppercase;
          color: var(--warhawk-purple); border-bottom: 2px solid var(--warhawk-light);
          padding-bottom: 4px; margin: 14px 0 10px; }
.ktvcat:first-child { margin-top: 0; }
.key { margin-bottom: 14px; }
.key .hl { font-size: 0.95rem; font-weight: 700; }
/* The lineup and shot-look keys carry multi-line captions ("Worst +/- lineups:" then one unit per
   line). pre-line keeps those breaks without turning the text into a <pre> block. */
.key .ev { font-size: 0.8rem; color: var(--dim); margin: 0 0 6px 18px; white-space: pre-line; }
.key .why { font-size: 0.88rem; margin: 0 0 0 18px; font-style: italic; color: #333;
            white-space: pre-line; }
.key .src { border: 1px solid #666; background: #fff; font-size: 0.65rem; font-weight: 600;
            padding: 1px 7px; border-radius: 8px; margin-left: 4px; white-space: nowrap; }

/* tables */
table { width: 100%; border-collapse: collapse; font-size: 0.85rem; }
th { text-align: right; font-weight: 700; color: var(--dim); font-size: 0.72rem; letter-spacing: 0.3px;
     padding: 0 8px 6px; border-bottom: 1px solid var(--edge); white-space: nowrap; }
td { text-align: right; padding: 7px 8px; border-bottom: 1px solid #f0f0f0; white-space: nowrap;
     font-variant-numeric: tabular-nums; }
tr:last-child td { border-bottom: 0; }
th:first-child, td:first-child { text-align: left; white-space: normal; }
td.wrap { text-align: left; white-space: normal; font-variant-numeric: normal; }
th.read, td.read { text-align: left; white-space: normal; font-variant-numeric: normal; min-width: 130px; }
.dot { display: block; font-size: 0.76rem; line-height: 1.35; padding-left: 10px; position: relative; }
.dot::before { content: "\u25aa"; position: absolute; left: 0; }
.dot.good { color: #1b5e20; }
.dot.bad { color: #8c1d2c; }
tr.starter td { font-weight: 700; }
td.jersey { color: var(--warhawk-purple); font-weight: 700; }
th.num, td.num { text-align: right; width: 2.4rem; }
.meta { color: #888; font-weight: 500; font-size: 0.78rem; margin-left: 7px; }

/* player notes, mirroring the Player Details dialog */
.player { border: 1px solid var(--edge); border-radius: 8px; padding: 12px 16px; margin-bottom: 10px; }
.player .nm { font-weight: 700; font-size: 1rem; }
.player .role { font-size: 0.6rem; font-weight: 700; letter-spacing: 0.4px; text-transform: uppercase;
                background: var(--warhawk-light); color: var(--warhawk-purple); border-radius: 8px;
                padding: 2px 7px; margin-left: 6px; vertical-align: middle; }
.player .statline { font-size: 0.8rem; color: var(--dim); font-variant-numeric: tabular-nums;
                    margin-top: 2px; }
.player .reads { margin-top: 6px; }
.player .ln { color: #888; font-weight: 500; font-size: 0.8rem; margin-left: 8px; }
.player .lbl { font-weight: 700; font-size: 0.85rem; }
.player .notes { font-style: italic; color: #333; font-size: 0.88rem; margin: 4px 0 0; }
.player .keys { font-size: 0.88rem; margin: 4px 0 0; }
.srcblock { border-left: 3px solid var(--edge); padding: 2px 0 2px 10px; margin: 8px 0 0; }
.srcblock.coach { border-left-color: var(--warhawk-purple); }
.srcblock.derived { border-left-color: #37474f; }
.srclabel { font-size: 0.65rem; font-weight: 700; letter-spacing: 0.4px; text-transform: uppercase;
            color: #888; margin-bottom: 2px; }
.kd { margin-top: 4px; }
.kd .lbl { font-weight: 700; font-size: 0.8rem; display: block; }
.kdline { font-size: 0.85rem; padding-left: 10px; position: relative; }
.kdline::before { content: "\u25aa"; position: absolute; left: 0;
                 color: var(--warhawk-purple); }

/* last five */
.five { display: flex; flex-wrap: wrap; gap: 10px; }
.five .g { border: 1px solid var(--edge); border-radius: 8px; padding: 10px 14px; min-width: 150px; flex: 1; }
.five .d { color: #888; font-size: 0.78rem; }
.five .o { font-size: 0.92rem; font-weight: 600; margin: 2px 0; }
.five .r { font-weight: 700; font-size: 0.92rem; }
.five .r.w { color: var(--win); }
.five .r.l { color: var(--loss); }

/* style matchup cards */
.matchrow { display: flex; gap: 10px; align-items: stretch; }
.matchrow > .match { flex: 1; min-width: 0; margin-bottom: 0; }
.match { border: 1px solid #eee; border-radius: 8px; padding: 10px 12px; margin-bottom: 8px; }
.match .nm { font-weight: 700; font-size: 0.95rem; color: var(--warhawk-purple); }
.match .score { font-size: 1.6rem; font-weight: 800; line-height: 1.1; }
.match .score .of { font-size: 0.7rem; color: #888; font-weight: 600; }
.match .sub { font-size: 0.68rem; color: #999; }
.match .conf { font-size: 0.68rem; font-weight: 700; }
.match .conf.hi { color: #2e7d32; }
.match .conf.mid { color: #8a6d3b; }
.match .conf.lo { color: #b3261e; }
.match .meet { font-size: 0.85rem; margin-top: 2px; }
.match .meet .w { color: var(--win); font-weight: 700; }
.match .meet .l { color: var(--loss); font-weight: 700; }
.match .alike { font-size: 0.75rem; color: #2e7d32; margin-top: 6px; }
.match .differs { font-size: 0.75rem; color: #c62828; }
.match .gap { font-size: 0.72rem; color: #666; margin-top: 2px; }
.strip { border: 1px solid #eee; border-radius: 8px; padding: 10px 12px; margin-top: 8px; font-size: 0.85rem; }
.strip .lb { font-size: 0.7rem; font-weight: 700; letter-spacing: 0.4px; color: var(--warhawk-purple);
             text-transform: uppercase; margin-bottom: 2px; }
.thin { font-size: 0.78rem; color: #8a6d3b; background: #fdf6e3; border-radius: 6px; padding: 8px 10px;
        margin: 0 0 8px; }

.flag { margin-bottom: 10px; }
.flag .fl { font-size: 0.9rem; }
.flag .conf-tag { font-size: 0.62rem; font-weight: 700; text-transform: uppercase; letter-spacing: 0.4px;
                  color: #888; border: 1px solid var(--edge); border-radius: 8px; padding: 1px 6px;
                  margin-left: 4px; white-space: nowrap; }

.note { font-size: 0.78rem; color: var(--dim); margin: 8px 0 0; }

/* SAMPLE DATA boxes. Anything generated as a placeholder (is_sample=True in its CSV) is drawn inside one of
   these and nowhere else, so a coach can never mistake a placeholder for scouting. */
.sample { border: 2px solid #c62828; background: #fff5f5; border-radius: 8px; padding: 10px 14px 12px;
          margin: 0 0 0.75rem; }
.sample .sample-tag { display: inline-block; background: #c62828; color: #fff; font-size: 0.66rem;
                      font-weight: 800; letter-spacing: 0.6px; text-transform: uppercase; border-radius: 6px;
                      padding: 2px 8px; margin-bottom: 6px; }
.sample .sample-why { font-size: 0.78rem; color: #8c1d2c; margin: 0 0 8px; }
.sample table td { border-bottom-color: #f6dada; }
.sect.sample-sect { border: 2px solid #c62828; background: #fff5f5; }
.pc-grid { display: flex; gap: 12px; flex-wrap: wrap; }
/* ---- condensed layout (#6) ---- */
.pb { break-after: page; page-break-after: always; height: 0; }
.divider { margin: 18px 0 10px; padding: 10px 14px; border-radius: 8px; background: var(--banner); color: #fff; }
.divider .dt { font-size: 1.05rem; font-weight: 800; letter-spacing: 0.6px; }
.divider .ds { font-size: 0.78rem; color: var(--warhawk-gray); margin-top: 2px; }
.linkbar { font-size: 0.8rem; margin: -4px 0 10px; }
a.applink { color: var(--warhawk-purple); font-weight: 700; text-decoration: none; white-space: nowrap; }
a.applink:hover { text-decoration: underline; }
.applink.off { color: var(--dim); font-weight: 600; font-style: italic; }
a.nmlink { color: inherit; text-decoration: none; border-bottom: 1px dotted #b9a9d3; }
sup.fn { font-size: 0.62rem; color: var(--warhawk-purple); font-weight: 700; margin-left: 2px; }
sup.fn a.fnlink { color: inherit; text-decoration: none; border-bottom: 1px dotted #b9a9d3; }
.grid2 { display: grid; grid-template-columns: 1fr 1fr; gap: 0 12px; align-items: start; }
/* Newspaper-style flow: cards fill the first column then the second, so a short card never strands the rest
   of a page the way a grid row does in print. */
.flow2 { column-count: 2; column-gap: 12px; }
.flow2 > .player, .flow2 > .gcell { display: inline-block; width: 100%; break-inside: avoid; }
.grid3 { display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 0 10px; align-items: start; }
.player, .key, .match { break-inside: avoid; page-break-inside: avoid; }
.chips { display: flex; flex-wrap: wrap; gap: 4px 6px; margin-top: 6px; }
.chip { font-size: 0.68rem; border-radius: 10px; padding: 1px 8px; background: #f3f0f8; color: var(--body); }
.chip.warn { background: #fdf6e3; color: #8a6d3b; }
.ktvline { display: flex; gap: 6px; align-items: baseline; padding: 3px 0; border-bottom: 1px solid #f2f2f2; font-size: 0.9rem; }
.ktvline:last-child { border-bottom: 0; }
.ktvline .n { font-weight: 800; color: var(--warhawk-purple); min-width: 1.6em; }
.ktvline .src { font-size: 0.6rem; font-weight: 700; letter-spacing: 0.3px; text-transform: uppercase; border: 1px solid; border-radius: 8px; padding: 0 6px; margin-left: 4px; white-space: nowrap; vertical-align: middle; }
.tier-h { font-size: 0.8rem; font-weight: 800; letter-spacing: 0.5px; text-transform: uppercase; color: var(--warhawk-purple); margin: 12px 0 2px; }
.tier-c { font-size: 0.76rem; color: var(--dim); margin: 0 0 6px; }
table.compact td, table.compact th { padding: 4px 6px; font-size: 0.78rem; }
td.nm { font-weight: 700; white-space: nowrap; }
.mini { font-size: 0.74rem; color: var(--dim); }
.results { display: flex; gap: 6px; flex-wrap: wrap; margin-bottom: 8px; }
.foul-icon { font-size: 1rem; text-align: center; }
tr.serrow td { font-weight: 700; background: var(--warhawk-light); }
tr.setrow td:first-child { padding-left: 12px; }
td.foul-icon { font-size: 0.95rem; font-weight: 700; }
.tbl-bul { margin: 0; padding-left: 14px; }
.tbl-bul li { margin: 0 0 1px; }
/* Defense-family table: THEIR offense and OUR defense sit under separate labelled bands with different
   tints, so a coach can't mistake "PPP allowed" (ours) for "PPP" (theirs) at a glance. */
table.dfam .dfam-band th { font-size: 0.6rem; letter-spacing: 0.5px; text-transform: uppercase;
                            text-align: center; padding: 3px 4px; border-bottom: 0; }
table.dfam th.band-them, table.dfam td.band-them { background: #f6f3fa; }
table.dfam th.band-us, table.dfam td.band-us { background: #eaf3ea; }
table.dfam .dfam-band th.band-them { color: var(--warhawk-purple); border-top: 2px solid var(--warhawk-purple); }
table.dfam .dfam-band th.band-us { color: #2e7d32; border-top: 2px solid #2e7d32; }
/* ---- stacked roster (ROSTER_LAYOUT = "stacked") ---- */
.pcard { border: 1px solid var(--edge); border-radius: 8px; margin: 0 0 10px; overflow: hidden;
         break-inside: avoid; page-break-inside: avoid; }
.pcard .ph { display: flex; align-items: center; gap: 10px; background: var(--warhawk-light);
             border-bottom: 2px solid var(--warhawk-purple); padding: 6px 10px; }
.pcard .ph img, .pcard .ph .photo-thumb { width: 42px; height: 42px; }
.pcard .ph-name { font-weight: 800; font-size: 0.95rem; }
.pcard .ph-bio { font-weight: 400; font-size: 0.72rem; color: var(--dim); }
.pcard .ph-foul { margin-left: auto; font-size: 0.72rem; font-weight: 700; white-space: nowrap; }
.pcard table.pstats { width: 100%; margin: 4px 0 0; }
.pcard table.pstats th, .pcard table.pstats td { text-align: center; }
.pcard table.sz-row { width: 100%; border-collapse: collapse; table-layout: fixed; margin: 2px 0 0; }
.pcard table.sz-row td { border: 0; padding: 3px 10px; vertical-align: top; text-align: left; }
.pcard .sz-cap { font-size: 0.62rem; font-weight: 800; text-transform: uppercase; letter-spacing: 0.4px;
                 color: var(--warhawk-purple); padding-bottom: 0 !important; }
.pcard .sz-cap .mini { text-transform: none; letter-spacing: 0; font-weight: 400; }
.pcard .sz-l { font-size: 0.66rem; color: var(--dim); font-weight: 400; }
.pcard .sz-bar { height: 6px; background: #ece8f3; border-radius: 3px; overflow: hidden; margin: 2px 0; }
.pcard .sz-bar span { display: block; height: 100%; background: var(--warhawk-purple); }
.pcard .sz-v { font-size: 0.74rem; font-weight: 400; }
.pcard .p-reads { padding: 2px 10px 6px; font-size: 0.76rem; }
.teamrow { display: flex; flex-wrap: wrap; gap: 6px; margin-bottom: 4px; }
/* ---- alternative offense layout #2 (_TEST): usage x efficiency quadrants ---- */
/* Flex, not CSS grid: the brief gets printed and PDF'd, and older print/WebKit engines silently collapse
   `display:grid` to a single stacked column -- which turns a 2x2 quadrant into a meaningless list. */
.quad { display: flex; flex-wrap: wrap; margin: 0 -4px; }
/* The UWW section carries five labelled lines per set; at half width those wrap into an unreadable
   column, so that section stacks its cells full width instead. */
.quad.quad-rich .quad-cell { width: 100%; }
.quad-cell { width: 50%; box-sizing: border-box; margin: 0 0 8px; padding: 7px 9px;
             border: 1px solid var(--edge); border-radius: 6px; border-top-width: 3px;
             break-inside: avoid; float: left; }
.quad-cell.q-good { border-top-color: #2e7d32; }
.quad-cell.q-up   { border-top-color: #1565c0; }
.quad-cell.q-bad  { border-top-color: #c62828; }
.quad-cell.q-dim  { border-top-color: #9e9e9e; }
.quad-h { font-size: 0.8rem; font-weight: 800; text-transform: uppercase; letter-spacing: 0.4px;
          display: flex; align-items: baseline; }
.quad-n { margin-left: auto; font-size: 0.68rem; font-weight: 700; color: var(--dim); }
.quad-sub { font-size: 0.68rem; color: var(--dim); margin-bottom: 5px; }
.quad-row { border-top: 1px dotted var(--edge); padding: 3px 0 2px; }
.quad-row:first-of-type { border-top: 0; }
.quad-set { font-weight: 700; font-size: 0.8rem; }
.quad-num { float: right; font-size: 0.78rem; font-weight: 700; }
.quad-meta { font-size: 0.68rem; color: var(--dim); clear: both; line-height: 1.35; }
.quad-meta .qk { display: inline-block; min-width: 54px; font-weight: 700; color: var(--warhawk-purple);
                 text-transform: uppercase; font-size: 0.58rem; letter-spacing: 0.3px; }
.quad-cov { font-size: 0.68rem; color: var(--warhawk-purple); font-weight: 600; clear: both; }
.quad-empty, .quad-more { font-size: 0.7rem; color: var(--dim); font-style: italic; padding-top: 3px; }
/* ---- alternative offense layout (_TEST) ---- */
.ot-situ { display: flex; align-items: baseline; gap: 8px; margin: 10px 0 4px;
           border-bottom: 2px solid var(--warhawk-purple); padding-bottom: 2px; }
.ot-situ-name { font-size: 0.82rem; font-weight: 800; text-transform: uppercase;
                letter-spacing: 0.5px; color: var(--warhawk-purple); }
.ot-situ-n { font-size: 0.68rem; color: var(--dim); margin-left: auto; }
.ot-row { margin: 5px 0 7px; break-inside: avoid; }
.ot-head { display: flex; align-items: baseline; gap: 8px; }
.ot-name { font-weight: 700; font-size: 0.84rem; }
.ot-num { margin-left: auto; font-size: 0.78rem; font-weight: 700; white-space: nowrap; }
.ot-bar { height: 7px; background: #ece8f3; border-radius: 4px; overflow: hidden; margin: 2px 0 3px; }
.ot-fill { display: block; height: 100%; background: var(--warhawk-purple); }
.ot-fill.ot-good { background: #2e7d32; }
.ot-fill.ot-bad { background: #c62828; }
.ot-num.ot-good { color: #2e7d32; }
.ot-num.ot-bad { color: #c62828; }
.ot-detail { font-size: 0.78rem; padding-left: 2px; }
.ot-cov { font-size: 0.72rem; color: var(--dim); }
/* Horizontal team-stat strip: one narrow column per stat, running left to right across the full width.
   Columns are allowed to shrink but not wrap mid-stat, so the strip stays one scannable row. */
.hstat { display: flex; align-items: stretch; gap: 0; width: 100%; overflow: hidden; }
.hs-col { flex: 1 1 0; min-width: 0; text-align: center; padding: 3px 2px; border-left: 1px solid var(--edge); }
.hs-col:first-child { border-left: 0; }
.hs-key { flex: 0 0 auto; text-align: right; padding-right: 8px; color: var(--dim); font-weight: 700; }
.hs-l { font-size: 0.62rem; text-transform: uppercase; letter-spacing: 0.3px; color: var(--dim);
        white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
.hs-v { font-size: 0.86rem; line-height: 1.5; white-space: nowrap; }
.hs-win { font-weight: 800; color: var(--warhawk-purple); }
.fnhold { float: right; }
.tchip { border: 1px solid var(--edge); border-radius: 12px; padding: 2px 10px; font-size: 0.8rem; font-weight: 600; }
.strip-line { font-size: 0.8rem; margin: 4px 0; }
.results .rs { border: 1px solid var(--edge); border-radius: 6px; padding: 2px 8px; font-size: 0.74rem; }
.results .rs b.w { color: var(--win); } .results .rs b.l { color: var(--loss); }
.dayg { display: grid; grid-template-columns: repeat(auto-fit, minmax(170px, 1fr)); gap: 8px; }
.dayg .day { border: 1px solid var(--edge); border-radius: 6px; padding: 6px 8px; font-size: 0.74rem; break-inside: avoid; }
.sample .dayg .day { border-color: #f6dada; background: #fff; }
.dayg .day .dh { font-weight: 800; color: var(--warhawk-purple); margin-bottom: 4px; }
.dayg .day .sg { margin: 2px 0; } .dayg .day .sg .tie { color: var(--dim); }
.cols .matchrow { flex-direction: column; }
@media print {
  body { font-size: 10pt; padding: 0; }
  .card, .sample { padding: 6px 10px; margin-bottom: 6px; }
  .sect { padding: 4px 10px; margin: 8px 0 4px; }
  .player { padding: 6px 10px; }
  a.applink, a.nmlink { color: var(--warhawk-purple); }
  @page { size: letter; margin: 0.45in; }
}
.pc-grid > div { flex: 1; min-width: 260px; }
.pc-sub { font-size: 0.72rem; font-weight: 800; letter-spacing: 0.6px; text-transform: uppercase;
          color: var(--warhawk-purple); margin: 10px 0 4px; }
td.ppp-good { color: #1b5e20; font-weight: 700; }
td.ppp-bad { color: #8c1d2c; font-weight: 700; }
.pc-raw { color: #888; font-size: 0.74rem; }
.photo-thumb { width: 34px; height: 34px; border-radius: 50%; object-fit: cover; display: block; }
.photo-thumb-blank { background: #e0dbea; border: 1px solid var(--edge); }
.bio { display: flex; align-items: center; gap: 10px; margin: 6px 0 8px; padding: 4px 6px; border-radius: 6px; }
.bio.sample-bio { background: #fff5f5; border: 1px solid #f6dada; }
.bio-text { font-size: 0.82rem; color: var(--dim); }
details.gloss summary { cursor: pointer; font-size: 0.8rem; color: var(--warhawk-purple); font-weight: 700; margin-top: 8px; }
details.gloss table { margin-top: 6px; }
.sect.sample-sect .t { color: #c62828; }
.legend { font-size: 0.8rem; color: var(--dim); margin: 0 0 0.75rem; display: flex; gap: 14px; flex-wrap: wrap; }
.legend .sw { display: inline-block; width: 12px; height: 12px; border-radius: 3px; vertical-align: -2px;
              margin-right: 5px; }
.legend .sw.real { border: 1px solid var(--edge); background: #fff; }
.legend .sw.fake { border: 2px solid #c62828; background: #fff5f5; }
.pill-thin { font-size: 0.62rem; font-weight: 700; letter-spacing: 0.3px; text-transform: uppercase;
             color: #8a6d3b; background: #fdf6e3; border-radius: 8px; padding: 1px 7px; margin-left: 6px;
             vertical-align: middle; }
.caution { background: #fdf6e3; border: 1px solid #f0dca8; }
.caution .row { font-size: 0.85rem; padding: 4px 0; border-bottom: 1px solid #f3e6c4; }
.caution .row:last-child { border-bottom: 0; }
.caution .area { font-weight: 700; color: #8a6d3b; margin-right: 6px; }
td.call-foul { color: #1b5e20; font-weight: 700; }
td.call-nofoul { color: #8c1d2c; font-weight: 700; }
tr.dayhead td { background: var(--warhawk-light); color: var(--warhawk-purple); font-weight: 800;
                font-size: 0.78rem; letter-spacing: 0.4px; text-transform: uppercase; }
.sample tr.dayhead td { background: #fde3e3; color: #8c1d2c; }
.foot { margin-top: 1.5rem; padding-top: 12px; border-top: 1px solid var(--edge);
        font-size: 0.75rem; color: var(--dim); }

@media screen and (max-width: 820px) {
  .cols { display: block; }
  .matchrow { display: block; }
  .matchrow > .match { margin-bottom: 8px; }
  .cols .col + .col .sect { margin-top: 1.5rem; }
}
@media screen and (max-width: 720px) {
  .banner { padding: 16px; }
  .banner .team { font-size: 1rem; }
  .banner .crest, .banner .crest img { height: 44px; max-height: 44px; }
  .five .g { min-width: 130px; }
}
@media print {
  /* CONFIRMED BUG (fixed here): the responsive rules above used to be plain (max-width: ...) queries, and a
     Letter page is narrower than 820px -- so in print every side-by-side layout collapsed to one column.
     This block also used to forbid ANY card or table from splitting across pages, which left half-empty
     pages wherever a tall section didn't fit. Only small, self-contained blocks are kept whole now. */
  body { padding: 0; font-size: 9.5pt; line-height: 1.35; }
  .banner, .divider, .sample, .sect, .chip, td.ppp-good, td.ppp-bad { -webkit-print-color-adjust: exact; print-color-adjust: exact; }
  .player, .key, .match, .gcell, .dayg .day, tr, .ktvline { break-inside: avoid; }
  .sect { break-after: avoid; }
  .banner { padding: 10px 14px; }
  .banner .crest, .banner .crest img { height: 40px; max-height: 40px; }
  .card, .sample { padding: 6px 10px; margin-bottom: 6px; }
  .sect { padding: 3px 10px; margin: 6px 0 4px; }
  .player { padding: 6px 10px; margin-bottom: 6px; }
  .photo-thumb { width: 28px; height: 28px; }
  /* Starter cards: reads as wrapping inline chips, note/key text a size down, so five fit one page. */
  .flow2 .player .reads { display: flex; flex-wrap: wrap; gap: 0 10px; margin: 2px 0; }
  .flow2 .player .reads .dot { font-size: 7.5pt; }
  .flow2 .player .notes { font-size: 8pt; line-height: 1.3; margin: 2px 0; }
  .flow2 .player .kdline { font-size: 8pt; line-height: 1.3; }
  .flow2 .player .statline, .flow2 .player .bio-text { font-size: 7.5pt; }
  .flow2 .player .bio { margin: 2px 0 4px; padding: 2px 4px; }
  .flow2 .player .srcblock { margin-top: 4px; padding-left: 8px; }
  .flow2 .player .nm { font-size: 10pt; }
}
"""


def _sb_headshot_b64(name):
    """A player's headshot from data/player_images as base64, or "" -- the same folder and filename variants
    the app's _get_player_img_b64() uses. This is the fallback when the FastScout roster page's own photo URL
    isn't available for a player (no live scrape yet for that team, or the scrape found no photo for him),
    which is why our own players showed no picture while the opponent's did: opponent headshots also get
    extracted from scouting-report PDFs into this folder, and UWW's don't come from a report at all.
    Embedded rather than linked so the brief stays a single self-contained file, like the crests."""
    import base64 as _sb_b64
    img_dir = os.path.join(APP_DATA_DIR, "player_images")
    if not name or not os.path.isdir(img_dir):
        return ""
    raw = re.sub(r"\s+", " ", str(name)).strip()
    variants = {raw, re.sub(r"[^\w\s\-]", "", raw).strip()}
    for stem in variants:
        for ext in ("png", "jpeg", "jpg"):
            path = os.path.join(img_dir, f"{stem}.{ext}")
            if os.path.isfile(path):
                with open(path, "rb") as f:
                    return _sb_b64.b64encode(f.read()).decode()
    # Last resort: case-insensitive match on the directory listing.
    lower = {f.lower(): f for f in os.listdir(img_dir)}
    for stem in variants:
        for ext in ("png", "jpeg", "jpg"):
            hit = lower.get(f"{stem}.{ext}".lower())
            if hit:
                with open(os.path.join(img_dir, hit), "rb") as f:
                    return _sb_b64.b64encode(f.read()).decode()
    return ""


def _sb_logo_b64(*candidates):
    """Team logo as base64, using the same lookup the app's find_logo_b64() uses.

    The banner is the most recognisable thing on the Upcoming Game page and the crests are most of
    why -- a brief with empty squares where the app has logos reads as a different document. Same
    data/logo directory, same exact/prefix/reverse-prefix order, so a team that has a crest in the
    app has one here.
    """
    import base64 as _sb_b64
    logo_dir = os.path.join(APP_DATA_DIR, "logo")
    if not os.path.isdir(logo_dir):
        return ""
    stems = sorted((os.path.splitext(f)[0] for f in os.listdir(logo_dir) if f.lower().endswith(".png")),
                   key=len, reverse=True)

    def _read(stem):
        with open(os.path.join(logo_dir, f"{stem}.png"), "rb") as f:
            return _sb_b64.b64encode(f.read()).decode()

    for name in candidates:
        if not name or (isinstance(name, float) and pd.isna(name)):
            continue
        name = str(name).strip()
        if not name:
            continue
        if os.path.exists(os.path.join(logo_dir, f"{name}.png")):
            return _read(name)
        for stem in stems:
            if name.startswith(stem) and len(stem) >= 3:
                return _read(stem)
        for stem in stems:
            if stem.startswith(name) and len(name) >= 3:
                return _read(stem)
    return ""


def _sb_record_and_streak(games):
    """Win-loss record and current streak from a set of completed games, newest last."""
    if games.empty or "outcome" not in games.columns:
        return "", ""
    played = games[games["outcome"].astype(str).str.upper().isin(["W", "L"])]
    if played.empty:
        return "", ""
    wins = int((played["outcome"].astype(str).str.upper() == "W").sum())
    losses = int((played["outcome"].astype(str).str.upper() == "L").sum())
    count, kind = 0, ""
    for outcome in played["outcome"].astype(str).str.upper().iloc[::-1]:
        if count == 0:
            kind, count = outcome, 1
        elif outcome == kind:
            count += 1
        else:
            break
    streak = f"{count}-game {'win' if kind == 'W' else 'loss'} streak" if count else ""
    return f"{wins}-{losses}", streak


def _sb_strip_mascot(name):
    """"Aurora Spartans" -> "Aurora", the way the banner shows it.

    The app has a full known-mascot list for this; a brief only needs the banner line, so this
    trims the trailing mascot word(s) conservatively and leaves anything it isn't sure about alone
    rather than mangling a real team name.
    """
    text = str(name or "").strip()
    if not text or text.endswith(")"):
        return text
    words = text.split()
    if len(words) >= 3 and words[0].lower().startswith("uw"):
        return " ".join(words[:1])
    if len(words) >= 2 and words[-1][:1].isupper():
        return " ".join(words[:-1])
    return text


# Filled in by _sb_build_html so the run can report which players ended up with no headshot -- a silent
# blank circle is how our own players' pictures went missing without anyone noticing.
_sb_photo_report = {}

# Filled in by _sb_build_html with the methodology notes, in the order they were registered. The brief only
# prints the ⓘ markers now (they deep-link into the app), so this is what the app reads to show the text --
# and the numbering here is what those markers point at, so it has to stay in registration order.
_sb_notes_report = []


def _sb_build_html(_sb_d, game, scheduled_name, short):
    parts = []
    # Output goes to whichever list is on top of _sinks, so a block can be rendered into a side-by-side
    # column (see capture()) with the exact same code that renders it full width.
    _sinks = [parts]

    def add(html_text):
        _sinks[-1].append(html_text)

    def capture(fn, *args, **kwargs):
        _sinks.append([])
        try:
            fn(*args, **kwargs)
        finally:
            out = _sinks.pop()
        return out

    # ---- links into the app (#7) -------------------------------------------------------------------
    # CONFIRMED CHANGE (requested): the brief is condensed, and every place detail was cut links into the
    # app instead. Links carry query parameters the app reads on load (see handle_brief_deeplink in
    # streamlit_app.py): page, tab, and optionally key / player+team / set / section. The hosted URL comes
    # from APP_BASE_URL (set it in the config cell) or the UWW_APP_URL environment variable; with neither,
    # links degrade to plain "(in the app: ...)" text rather than pointing nowhere.
    from urllib.parse import urlencode as _sb_urlencode

    def _sb_normalize_app_url(raw):
        """Force an absolute https:// URL, or nothing.

        A bare host ("uwwmensbball-new.streamlit.app") in an href is a RELATIVE url: the browser resolves
        it against wherever the brief happens to sit, so a link opens
        file:///C:/.../scouting_briefs/uwwmensbball-new.streamlit.app/?page=... instead of the app. The
        brief is emailed around and opened from disk, so this fails silently for every reader. Normalizing
        happens HERE, at the point of use, rather than only in the config cell -- app_href re-reads the raw
        global/env value, so a fix applied only at config time is bypassed whenever the value arrives by
        another route (env var, a re-run config cell, an edit made after this cell ran).
        """
        u = str(raw or "").strip().strip('"\'').rstrip("/")
        if not u:
            return ""
        # Typos that would otherwise sail through as "has no scheme": https//host, https:/host, http:host.
        u = re.sub(r"^(https?)(?::/{0,2}|/{1,2})", r"\1://", u, flags=re.I)
        if not re.match(r"^https?://", u, flags=re.I):
            u = "https://" + u
        return u.rstrip("/")

    _app_url = _sb_normalize_app_url(globals().get("APP_BASE_URL") or os.environ.get("UWW_APP_URL"))

    # Every app link opens in a NEW TAB (requested). rel="noopener" goes with target="_blank" as a matter
    # of course: without it the opened page gets a handle on the opener via window.opener, and "noreferrer"
    # keeps the local file:// path of the brief out of the referrer header when it is opened from disk.
    _SB_NEWTAB = ' target="_blank" rel="noopener noreferrer"'

    def app_href(**params):
        q = _sb_urlencode({k: v for k, v in params.items() if v not in (None, "")})
        return f"{_app_url}/?{q}" if _app_url else ""

    def app_link(label, **params):
        href = app_href(**params)
        if href:
            return f'<a class="applink" href="{_sb_esc(href)}"{_SB_NEWTAB}>{_sb_esc(label)} &rarr;</a>'
        return f'<span class="applink off">{_sb_esc(label)} (in the app)</span>'

    def name_link(text, **params):
        href = app_href(**params)
        return f'<a class="nmlink" href="{_sb_esc(href)}"{_SB_NEWTAB}>{_sb_esc(text)}</a>' if href else _sb_esc(text)

    def _name_two_lines(name):
        """First name, line break, last name -- for the compact roster table, where the name column has to
        stay narrow. Splits on the first space only, so a suffix or two-word last name ("Agape Keyes Jr.")
        stays together on the second line."""
        first, sep, rest = str(name).partition(" ")
        return f"{_sb_esc(first)}<br>{_sb_esc(rest)}" if sep else _sb_esc(name)

    def name_link_lines(name, **params):
        href = app_href(**params)
        label = _name_two_lines(name)
        return f'<a class="nmlink" href="{_sb_esc(href)}"{_SB_NEWTAB}>{label}</a>' if href else label

    # ---- footnotes (#6) ----------------------------------------------------------------------------
    # Long methodology paragraphs used to sit under each table, then on a Notes page at the end of the
    # brief. They live in the APP now (requested) -- the brief keeps only the small numbered ⓘ at each
    # section title, hyperlinked to that note in the app, so the explanation is one tap away without
    # spending brief pages on it. Still collected here because the numbering has to match what the app
    # prints, and because the export below is what the app reads.
    footnotes = []

    def footnote(text):
        text = _sb_clean(text)
        if not text:
            return ""
        footnotes.append(text)
        _n = len(footnotes)
        _href = app_href(page="upcoming", tab="notes", note=_n)
        if not _href:
            # No app URL configured -- a numbered marker pointing at a page the reader can't reach (and
            # that isn't printed here any more) is worse than no marker at all.
            return ""
        return (f' <sup class="fn"><a class="fnlink" href="{_sb_esc(_href)}"{_SB_NEWTAB} '
                f'title="Why this number is what it is \u2014 opens the app">\u24d8{_n}</a></sup>')

    def page_break():
        add('<div class="pb"></div>')

    def divider(title, sub=""):
        add(f'<div class="divider"><div class="dt">{title}</div>'
            + (f'<div class="ds">{sub}</div>' if sub else "") + "</div>")

    def section_html(title):
        return f'<div class="sect"><div class="t">{title}</div></div>'

    def section(title):
        add(section_html(title))

    # ---- banner (same markup and values as render_upcoming_game) ---------------------------
    sched = _sb_d("uww_schedule")
    uww_games = sched[sched["team"].astype(str).str.contains("Whitewater", case=False, na=False)] \
        if not sched.empty else pd.DataFrame()
    uww_before = uww_games.loc[:game.name].iloc[:-1] if not uww_games.empty else pd.DataFrame()
    uww_record, uww_streak = _sb_record_and_streak(uww_before)

    opp_sched = _sb_d("uww_opponent_schedules")
    opp_before = pd.DataFrame()
    if not opp_sched.empty and "opponent" in opp_sched.columns:
        opp_before = opp_sched[opp_sched["opponent"].astype(str).str.strip() == str(short).strip()]
    opp_record, opp_streak = _sb_record_and_streak(opp_before)

    opp_display = _sb_strip_mascot(short)
    uww_logo = _sb_logo_b64("UW-Whitewater")
    opp_logo = _sb_logo_b64(short, scheduled_name, opp_display)

    def crest(b64):
        return (f'<div class="crest"><img src="data:image/png;base64,{b64}" alt=""></div>'
                if b64 else '<div class="crest"></div>')

    add('<div class="banner">')
    add(f'<div class="side">{crest(uww_logo)}'
        f'<div class="team">UW-WHITEWATER</div>'
        f'<div class="rec">{_sb_esc(uww_record)}</div>'
        + (f'<div class="streak">{_sb_esc(uww_streak)}</div>' if uww_streak else "")
        + "</div>")
    add(f'<div class="side mid"><div class="when">{_sb_esc(game.get("date", "-"))}</div>'
        f'<div class="vs">VS</div>'
        f'<div class="where">{_sb_esc(game.get("location", "-"))}</div></div>')
    add(f'<div class="side">{crest(opp_logo)}'
        f'<div class="team">{_sb_esc(str(opp_display).upper())}</div>'
        f'<div class="rec">{_sb_esc(opp_record)}</div>'
        + (f'<div class="streak">{_sb_esc(opp_streak)}</div>' if opp_streak else "")
        + "</div>")
    add("</div>")
    add('<div class="legend"><span><span class="sw real"></span>From your play-by-play, video and box-score '
        'files</span><span><span class="sw fake"></span>SAMPLE DATA \u2014 placeholder until the staff '
        'fills it in (INPUT_DIR/staff_inputs)</span></div>')

    # ---- gather ---------------------------------------------------------------------------
    us = _sb_team_line(_sb_d("uww_pbp_box_score"), "team", _SB_UWW)
    them = _sb_team_line(_sb_d("uww_opponent_prior_games_box_score"), "team", short)
    them_allowed = _sb_team_line(_sb_d("uww_opponent_prior_games_box_score"), "team", short, invert=True)
    us_allowed = _sb_team_line(_sb_d("uww_pbp_box_score"), "team", _SB_UWW, invert=True)

    totals = _sb_d("uww_opponent_team_totals")
    if not totals.empty and "opponent" in totals.columns:
        row = totals[totals["opponent"].astype(str) == str(short)]
        if not row.empty:
            them.setdefault("PTS", _sb_num(row["team_ppg"]).iloc[0])
            them_allowed.setdefault("PTS", _sb_num(row["opp_ppg_allowed"]).iloc[0])

    players = _sb_player_table(_sb_d, short)

    # ---- staff-input / sample sections ---------------------------------------------------------
    # One renderer for every table the game-plan cell writes. When the table is SAMPLE (is_sample True on
    # every row), the card is drawn as a red box with a SAMPLE DATA tag and a line saying what real input
    # would replace it. When the staff has supplied the file, the same table renders as an ordinary card.
    def staff_rows(table_name):
        df = _sb_d(table_name)
        if df.empty:
            return df, False
        if "opponent" in df.columns:
            df = df[df["opponent"].astype(str) == str(short)]
        is_sample = bool("is_sample" in df.columns and not df.empty
                         and df["is_sample"].astype(str).str.lower().isin(["true", "1"]).all())
        return df, is_sample

    def open_card(is_sample, why):
        if is_sample:
            add('<div class="sample"><div class="sample-tag">Sample data \u2014 not from your files</div>')
            if why:
                add(f'<p class="sample-why">{_sb_esc(why)}</p>')
        else:
            add('<div class="card">')

    def staff_cell(value):
        if isinstance(value, float) and not pd.isna(value) and float(value).is_integer():
            return str(int(value))
        return _sb_clean(value)

    def staff_table(table_name, title, columns, why_sample, note=None):
        df, is_sample = staff_rows(table_name)
        if df.empty:
            return
        add('<div class="gcell">')
        add(f'<div class="sect{" sample-sect" if is_sample else ""}"><div class="t">{title}'
            f'{footnote(note) if note else ""}</div></div>')
        open_card(is_sample, why_sample)
        add("<table><thead><tr>" + "".join(
            f'<th style="text-align:left">{_sb_esc(label)}</th>' for _, label in columns) + "</tr></thead><tbody>")
        for _, r in df.iterrows():
            add("<tr>" + "".join(f'<td class="wrap">{_sb_esc(staff_cell(r.get(col)))}</td>'
                                 for col, _ in columns) + "</tr>")
        add("</tbody></table>")
        add("</div></div>")

    # ---- team stats and four factors, side by side ----------------------------------------
    # Collected into lists first rather than written straight out, so the top two can be dropped into
    # one flex row. Either panel can be absent -- no box score, or not enough of one to compute the
    # factors -- and a lone panel is emitted full width instead of stranded in half a row.
    team_parts, ff_parts = [], []
    t_add, f_add = team_parts.append, ff_parts.append

    # ---- team stats -----------------------------------------------------------------------
    # Laid out HORIZONTALLY (requested): each stat is its own narrow column reading top-to-bottom
    # (label, our number, theirs), and the columns run left to right across the full width of the
    # section. The old "cmp" layout stacked one stat per line down the page, which made this a tall
    # narrow list instead of a strip a coach can scan in one pass.
    if us and them:
        t_add('<span class="fnhold">' + footnote(
            "Team stats: both sides are built the same way, from play-by-play box scores rather than a scouting "
            "report, so they compare like with like. Their figures cover only the games before this matchup.")
            + "</span>")
        comparisons = [
            # Pace sits right after scoring (requested): points per game only mean something next to how
            # many possessions produced them. Neither direction is "better", so it is never bolded.
            ("Points", "PTS", True), (None, None, None), ("Pace (poss/g)", "PACE", None),
            ("Field Goal %", "FG%", True), ("3PT %", "3P%", True), ("3PA", "3PA", None),
            ("Rebounds", "REB", True), ("Assists", "AST", True),
            ("Turnovers", "TO", False), ("Steals", "STL", True), ("Blocks", "BLK", True),
        ]
        _cells = []
        for label, key, higher_is_better in comparisons:
            if key is None:
                label, a, b, higher_is_better = "Points Against", us_allowed.get("PTS"), them_allowed.get("PTS"), False
            else:
                a, b = us.get(key), them.get(key)
            if a is None and b is None:
                continue
            _us_cls, _them_cls = "hs-v", "hs-v"
            if higher_is_better is not None and a is not None and b is not None and abs(float(a) - float(b)) >= 0.05:
                if (float(a) > float(b)) == bool(higher_is_better):
                    _us_cls += " hs-win"
                else:
                    _them_cls += " hs-win"
            _cells.append(f'<div class="hs-col"><div class="hs-l">{_sb_esc(label)}</div>'
                          f'<div class="{_us_cls}">{_sb_fmt(a)}</div>'
                          f'<div class="{_them_cls}">{_sb_fmt(b)}</div></div>')
        t_add('<div class="hstat"><div class="hs-col hs-key"><div class="hs-l">&nbsp;</div>'
              f'<div class="hs-v">UWW</div><div class="hs-v">{_sb_esc(str(opp_display).upper())}</div></div>'
              + "".join(_cells) + "</div>")
        t_add('<p class="mini">Per game, before this matchup. Bold is the better number.</p>')

    # ---- four factors ---------------------------------------------------------------------
    # Same computation and the same weighted ranking as the app's Four Factors dialog. Ours come from
    # our own games, theirs from their prior games, so neither side is adjusted for who they played
    # -- stated under the table rather than left for a coach to assume one way or the other.
    _ff_box = _sb_d("uww_pbp_box_score")
    _ff_prior = _sb_d("uww_opponent_prior_games_box_score")
    _ff_top, _ff_rows = None, []
    if not _ff_box.empty and not _ff_prior.empty and "team" in _ff_box.columns and "team" in _ff_prior.columns:
        _ff_us = _sb_four_factors(_ff_box[_ff_box["team"] == _SB_UWW],
                                  _ff_box[_ff_box["team"] != _SB_UWW])
        _ff_them = _sb_four_factors(_ff_prior[_ff_prior["team"].astype(str) == str(short)],
                                    _ff_prior[_ff_prior["team"].astype(str) != str(short)])
        _ff_rows = []
        for _factor, _weight in _SB_FF_WEIGHTS.items():
            _a, _b = _ff_us.get(_factor), _ff_them.get(_factor)
            if _a is None or _b is None:
                continue
            # Edge is stated so positive always favours us -- which for turnovers means a LOWER rate.
            _edge = (_a - _b) if _SB_FF_HIGHER_IS_BETTER[_factor] else (_b - _a)
            _ff_rows.append({"factor": _factor, "uww": _a, "opp": _b, "edge": _edge,
                             "weight": _weight, "weighted": _edge * _weight})
        if _ff_rows:
            _ff_rows.sort(key=lambda r: abs(r["weighted"]), reverse=True)
            _ff_top = _ff_rows[0]
            # The table itself is gone from the brief (requested) -- every factor is one tap away in the app's
            # Four Factors dialog. The methodology note is registered on THE BOTTOM LINE sentence itself (when
            # it prints), not here: a note registered here had no marker anywhere in the brief pointing to it.

    def two_up(left, right):
        """Two panels in one flex row, or whichever one exists full width. A lone panel stranded in
        half a row reads as a rendering failure rather than as missing data."""
        if left and right:
            return ('<div class="cols"><div class="col">' + "\n".join(left) + "</div>"
                    + '<div class="col">' + "\n".join(right) + "</div></div>")
        return "\n".join(left + right)


    # =====================================================================================================
    # CONDENSED LAYOUT (requested): Page 1 game plan -> opponent (glance, starters, bench, lineups, offense,
    # context, tagging-dependent) -> UWW mirror (starters, bench, lineups, offense, context, our week) -> notes.
    # Nothing was removed: detail that no longer fits a page is one link away in the app, and every
    # methodology paragraph is on the Notes page under its ⓘ number.
    # =====================================================================================================
    OPP_UP = _sb_esc(str(opp_display).upper())
    _team_gp = int(them.get("games") or 0)
    add('<div class="linkbar">' + app_link("Open this matchup in the app", page="upcoming") + "</div>")

    # ---- shared data --------------------------------------------------------------------------------
    _notes_tbl = _sb_d("uww_scouting_notes")
    if not _notes_tbl.empty and "opponent" in _notes_tbl.columns:
        _notes_tbl = _notes_tbl[_notes_tbl["opponent"].astype(str) == str(short)]
    _tiers = _sb_d("uww_personnel_tiers")
    if not _tiers.empty and "scouted_opponent" in _tiers.columns:
        _tiers = _tiers[_tiers["scouted_opponent"].astype(str) == str(short)]
    _pcs_all = _sb_d("uww_play_call_summary")
    _pcc_all = _sb_d("uww_play_calls")
    if not _pcs_all.empty and "scouted_opponent" in _pcs_all.columns:
        _pcs_all = _pcs_all[_pcs_all["scouted_opponent"].astype(str) == str(short)]

    def _offense_rows(side):
        """play_call_summary rows for one side's OWN OFFENSE.

        side alone is not enough any more: it names the FILE a clip came from, and those files now carry
        defensive possessions too. Without the possession_side filter our offensive tables pick up
        possessions where the opponent had the ball -- which is how a SLOB finished by a Ripon player
        turned up under our own sets. Rows written before possession_side existed have no value for it and
        are treated as offensive, which is what they were.
        """
        if _pcs_all.empty or "side" not in _pcs_all.columns:
            return pd.DataFrame()
        rows = _pcs_all[_pcs_all["side"] == side]
        if "possession_side" in rows.columns:
            rows = rows[rows["possession_side"].astype(str) != "Defense"]
        return rows
    if not _pcc_all.empty and "scouted_opponent" in _pcc_all.columns:
        _pcc_all = _pcc_all[_pcc_all["scouted_opponent"].astype(str) == str(short)]
    _flags = _sb_d("uww_coaching_flags")
    _live_ros = _sb_d("uww_live_rosters")
    _avail, _avail_sample = staff_rows("uww_uww_availability")

    def _norm(n):
        return re.sub(r"\s+", " ", str(n)).strip().lower()

    def _sb_title(n):
        """ALL-CAPS roster names read as shouting in running prose. Title-case them, but leave a name that
        already has mixed case alone -- "McEwen" and "Mur Martin" are correct as tagged and .title() would
        wreck them."""
        n = re.sub(r"\s+", " ", str(n)).strip()
        if not n or n != n.upper():
            return n
        return " ".join(w[:1].upper() + w[1:].lower() if w else w for w in n.split(" "))

    def players_for(box_name, team_value):
        """Per-game lines for any team's players -- same math as _sb_player_table, which is opponent-only."""
        box = _sb_d(box_name)
        if box.empty or "team" not in box.columns:
            return pd.DataFrame()
        own = box[(box["team"].astype(str) == str(team_value)) & (box["player"].astype(str) != "TEAM")]
        rows = []
        for name, g in own.groupby("player"):
            games = g["game_date"].nunique() if "game_date" in g.columns else len(g)
            t = {c: _sb_num(g[c]).sum() if c in g.columns else 0
                 for c in ("PTS", "REB", "AST", "STL", "BLK", "TO", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA", "MIN")}
            rows.append({"name": str(name), "games": games,
                         **{k: (t[k] / games if games else None) for k in ("PTS", "REB", "AST", "STL", "BLK", "TO", "MIN")},
                         "FG%": _sb_pct(t["FGM"], t["FGA"]), "3P%": _sb_pct(t["FG3M"], t["FG3A"]),
                         "FT%": _sb_pct(t["FTM"], t["FTA"]), "FTM": t["FTM"], "FTA": t["FTA"],
                         "PTS_total": t["PTS"]})
        return pd.DataFrame(rows)

    def stat_line(p):
        return (f'{_sb_fmt(p["games"], 0)} GP &middot; {_sb_fmt(p.get("MIN"))} MIN &middot; {_sb_fmt(p["PTS"])} PTS &middot; '
                f'{_sb_fmt(p["REB"])} REB &middot; {_sb_fmt(p["AST"])} AST &middot; {_sb_fmt(p["TO"])} TO &middot; '
                f'{_sb_fmt(p["FG%"])}% FG &middot; {_sb_fmt(p["3P%"])}% 3P &middot; {_sb_fmt(p["FT%"])}% FT')

    def tier_groups(side, player_names):
        """[(label, caption, [names])] in the app's order. Falls back to one group when no tiers exist."""
        t = _tiers[_tiers["side"] == side] if not _tiers.empty and "side" in _tiers.columns else pd.DataFrame()
        if t.empty:
            return None, [("Players", "No personnel tiers on file yet -- ordered by production.", list(player_names))], "", []
        n = int(t["recent_n"].max() or 0)
        basis = _sb_clean(t["starter_basis"].iloc[0])
        by = lambda label: list(t[t["tier"] == label].sort_values("mpg", ascending=False)["player"])  # noqa: E731
        starters = by("Starter")
        # "Bench -- limited minutes" is no longer printed (requested): under-8-minute players filled a
        # third of the roster pages without changing how anyone prepares. They are still tiered in the
        # data and still in the app -- the list comes back here so the brief can say how many were left
        # out and point the staff at them, rather than silently shortening the roster.
        # "No minutes in last N games" is no longer printed either (requested) -- it lives in the app. Both
        # omitted tiers are returned so the roster's single app link can say how many were left out.
        _limited = by("Bench \u2014 limited minutes") + by("Bench \u2014 no minutes")
        bench = [
            ("Bench \u2014 rotation", f"Played in the last {n} game(s) and averaging 8+ minutes.", by("Bench \u2014 rotation")),
        ]
        return starters, bench, basis, _limited

    # ---- personnel identity (photo / # / pos / ht / yr) ------------------------------------------------
    _pers_tbl, _pers_is_sample = staff_rows("uww_opp_personnel")
    _opp_bio = {_norm(r["player"]): r for _, r in _pers_tbl.iterrows()} if not _pers_tbl.empty else {}
    _uww_bio = {}
    if not _live_ros.empty and "team" in _live_ros.columns:
        _uww_labels = {_norm(_SB_UWW)}
        _sched_all = _sb_d("uww_schedule")
        if not _sched_all.empty and "team" in _sched_all.columns:
            _uww_labels |= {_norm(t) for t in _sched_all["team"].dropna().unique() if "whitewater" in str(t).lower()}
        _uww_ros_rows = _live_ros[_live_ros["team"].astype(str).str.contains("whitewater", case=False, na=False)
                                  | _live_ros["team"].astype(str).map(_norm).isin(_uww_labels)]
        for _, r in _uww_ros_rows.iterrows():
            _uww_bio[_norm(r["name"])] = {"player": r["name"], "jersey": r.get("jersey_number"),
                                          "position": r.get("position"), "height": r.get("height"),
                                          "class_year": r.get("class_year"), "photo_url": r.get("photo_url")}
    _avail_by = {_norm(r["player"]): r for _, r in _avail.iterrows()} if not _avail.empty else {}

    _photo_stats = {"Opponent": [0, []], "UWW": [0, []]}

    def photo_cell(url, name=None, side=None):
        url = _sb_clean(url)
        if not url and name:
            b64 = _sb_headshot_b64(name)
            if b64:
                url = f"data:image/png;base64,{b64}"
        if side in _photo_stats and name:
            if url:
                _photo_stats[side][0] += 1
            else:
                _photo_stats[side][1].append(str(name))
        return (f'<img src="{_sb_esc(url)}" class="photo-thumb" alt="">' if url
                else '<div class="photo-thumb photo-thumb-blank"></div>')

    def bio_bits(r, extra=None):
        bits = []
        if r is not None:
            if _sb_clean(r.get("jersey")):
                bits.append(f"#{_sb_esc(_sb_clean(r.get('jersey')))}")
            for c in ("position", "height", "hand", "class_year", "status"):
                if _sb_clean(r.get(c)):
                    bits.append(_sb_esc(_sb_clean(r.get(c))))
        return bits + (extra or [])

    # ---- one full card (starters) ----------------------------------------------------------------------
    def opp_card(name, p):
        rows = (_notes_tbl[(_notes_tbl["subject_type"] == "player") & (_notes_tbl["name"] == name)]
                if not _notes_tbl.empty else pd.DataFrame())
        bio = _opp_bio.get(_norm(name))
        title = name_link(name, page="upcoming", tab="personnel", team="opponent", player=name)
        if p is not None and _team_gp > 1 and float(p["games"]) < 0.5 * _team_gp:
            title += f' <span class="pill-thin">{int(p["games"])} of {_team_gp} games</span>'
        add('<div class="player">')
        add(f'<div class="nm">{title}</div>')
        gof = _sb_clean(bio.get("games_on_film")) if bio is not None else ""
        add(f'<div class="bio{" sample-bio" if _pers_is_sample else ""}">'
            f'{photo_cell(bio.get("photo_url") if bio is not None else None, name, "Opponent")}'
            f'<div class="bio-text">{" &middot; ".join(bio_bits(bio, [f"{_sb_esc(gof)} games on film"] if gof else []))}'
            f'</div></div>')
        if p is not None:
            add(f'<div class="statline">{stat_line(p)}</div>')
        good, bad = [], []
        for _, r in rows.iterrows():
            good += _sb_split(r.get("strengths"))
            bad += _sb_split(r.get("weaknesses"))
        if good or bad:
            add('<div class="reads">' + "".join(f'<span class="dot good">{_sb_esc(i)}</span>' for i in good)
                + "".join(f'<span class="dot bad">{_sb_esc(i)}</span>' for i in bad) + "</div>")
        for src in ("Coach", "Data-Driven"):
            r = rows[rows["source"] == src] if not rows.empty else pd.DataFrame()
            if r.empty:
                continue
            r = r.iloc[0]
            notes, keys = _sb_clean(r.get("notes")), _sb_clean(r.get("keys_to_defending"))
            if not notes and not keys:
                continue
            add(f'<div class="srcblock {"coach" if src == "Coach" else "derived"}"><div class="srclabel">{_sb_esc(src)}</div>')
            if notes:
                add(f'<p class="notes">{_sb_esc(notes)}</p>')
            if keys:
                add('<div class="kd"><span class="lbl">Keys to Defending</span>'
                    + "".join(f'<div class="kdline">{_sb_esc(k)}</div>' for k in _sb_split(keys)) + "</div>")
            add("</div>")
        add("</div>")

    def uww_player_calls(name):
        if _pcc_all.empty or "side" not in _pcc_all.columns:
            return ""
        c = _pcc_all[(_pcc_all["side"] == "UWW") & (_pcc_all["player"].astype(str).map(_norm) == _norm(name))
                     & (_pcc_all["decode_quality"] != "Needs review")]
        named = c[c["play_call"].notna() & ~c["play_call"].astype(str).str.contains("unspecified", na=False)]
        vc = named["play_call"].value_counts()
        vc = vc[vc >= 3]
        if vc.empty:
            return ""
        pts = pd.to_numeric(c["points"], errors="coerce")
        ppp = pts.sum() / pts.notna().sum() if pts.notna().any() else None
        return (f"Tagged film: featured in " + ", ".join(f"{n} ({int(k)}x)" for n, k in vc.head(3).items())
                + f" out of {len(c)} tagged possession(s)" + (f" \u2014 {ppp:.2f} PPP." if ppp is not None else "."))

    def uww_card(name, p):
        bio = _uww_bio.get(_norm(name))
        av = _avail_by.get(_norm(name))
        extra = []
        if av is not None and _sb_clean(av.get("status")):
            extra.append(_sb_esc(_sb_clean(av.get("status"))) + (" (sample)" if _avail_sample else "")
                         + (f" \u2014 {_sb_esc(_sb_clean(av.get('note')))}" if _sb_clean(av.get("note")) else ""))
        add('<div class="player">')
        add(f'<div class="nm">{name_link(name, page="upcoming", tab="personnel", team="uww", player=name)}</div>')
        add(f'<div class="bio">{photo_cell(bio.get("photo_url") if bio else None, name, "UWW")}'
            f'<div class="bio-text">{" &middot; ".join(bio_bits(bio, extra)) or "No roster-page details on file"}</div></div>')
        if p is not None:
            add(f'<div class="statline">{stat_line(p)}</div>')
        pf = (_flags[_flags["player"].astype(str).map(_norm) == _norm(name)]
              if not _flags.empty and "player" in _flags.columns else pd.DataFrame())
        if not pf.empty:
            for label, sentiment in (("Lean on", "positive"), ("Clean up", "negative")):
                sel = pf[pf["sentiment"].astype(str).str.lower() == sentiment]
                if sel.empty:
                    continue
                add(f'<div class="srcblock derived"><div class="srclabel">{label}</div>')
                for _, f in sel.iterrows():
                    # Confidence labels read like "Low (n=3 Off Screen attempts -- small sample, re-check...)"; the
                    # tier and the count are what a card needs -- the full sentence stays in the app.
                    _conf = _sb_clean(f.get("confidence"))
                    _cm = re.match(r"^(\w+)\s*\((n=\d+)", _conf)
                    _conf = f"{_cm.group(1)}, {_cm.group(2)}" if _cm else _conf
                    add(f'<p class="notes"><strong>{_sb_esc(_sb_clean(f.get("flag")))}</strong>'
                        + (f' <span class="conf-tag">{_sb_esc(_conf)}</span>' if _conf else "")
                        + f' \u2014 {_sb_esc(_sb_clean(f.get("evidence")))}</p>')
                    if _sb_clean(f.get("recommendation")):
                        add(f'<div class="kdline">{_sb_esc(_sb_clean(f.get("recommendation")))}</div>')
                add("</div>")
        calls = uww_player_calls(name)
        if calls:
            add(f'<div class="srcblock derived"><div class="srclabel">Play calls</div><p class="notes">{_sb_esc(calls)}</p></div>')
        add("</div>")

    # CONFIRMED CHANGE (requested): the standalone Late-Game Foul List is gone; its call now rides along in
    # the opponent's roster table beside each player's FT%, which is where a coach reads it anyway.
    _foul_tbl, _ = staff_rows("uww_late_game_foul_list")
    _foul_by = {_norm(r["player"]): r for _, r in _foul_tbl.iterrows()} if not _foul_tbl.empty else {}

    def foul_cell(name):
        r = _foul_by.get(_norm(name))
        if r is None:
            return '<td class="mini">--</td>'
        call = _sb_clean(r.get("call"))
        # Icon rather than words (requested): a check means foul him, a cross means don't. The words stay in
        # the cell's tooltip and are spelled out under the table, so the icon is never the only explanation.
        icon, cls = {"Foul": ("\u2714", "call-foul"),
                     "Do not foul": ("\u2718", "call-nofoul")}.get(call, ("\u2013", "mini"))
        return (f'<td class="{cls} foul-icon" title="{_sb_esc(call)} \u2014 {_sb_fmt(r.get("ft_pct"))}% FT">'
                f'{icon}</td>')

    # ---- compact roster rows ---------------------------------------------------------------------------
    # One table shape for every tier, with fixed column widths so the starters, rotation, limited-minutes and
    # no-minutes tables all line up down the page instead of each sizing itself to its own content.
    _ROSTER_COLS = ('<colgroup><col style="width:30px"><col style="width:13%"><col style="width:9%">'
                    '<col style="width:4%"><col style="width:5%"><col style="width:5%"><col style="width:5%">'
                    '<col style="width:5%"><col style="width:5%"><col style="width:5%"><col style="width:5%">'
                    '<col style="width:7%"><col style="width:32%"></colgroup>')

    # Who dominates their offensive glass, if anyone -- keyed by normalised name for the roster read above.
    _reb_crasher = {}
    _RRr = globals().get("REBOUND_RULES") or {"min_paired": 20, "player_share": 35, "player_min": 5}
    _rsr = _sb_d("uww_rebound_summary")
    if not _rsr.empty and {"opponent", "metric", "player", "value", "count", "paired"}.issubset(_rsr.columns):
        _rsr = _rsr[(_rsr["opponent"].astype(str) == str(short)) & (_rsr["metric"] == "their_oreb_player")]
        for _, _cr in _rsr.iterrows():
            _share = pd.to_numeric(_cr["value"], errors="coerce")
            _cnt = int(pd.to_numeric(_cr["count"], errors="coerce") or 0)
            if (pd.notna(_share) and _share >= _RRr["player_share"] and _cnt >= _RRr["player_min"]
                    and int(pd.to_numeric(_cr["paired"], errors="coerce") or 0) >= _RRr["min_paired"]):
                _reb_crasher[_norm(_cr["player"])] = (f"Crashes the offensive glass \u2014 {_cnt} of their "
                                                      f"offensive rebounds after misses ({_share:.0f}%)")

    def _roster_player(name, stats_df, side, bullets):
        """Everything one roster entry needs, assembled once so the table layout and the stacked layout
        can't show different reads, keys or flags for the same player."""
        s_ = stats_df[stats_df["name"].map(_norm) == _norm(name)] if not stats_df.empty else pd.DataFrame()
        p = s_.iloc[0] if not s_.empty else None
        if side == "Opponent":
            bio = _opp_bio.get(_norm(name))
            rows = (_notes_tbl[(_notes_tbl["subject_type"] == "player") & (_notes_tbl["name"] == name)]
                    if not _notes_tbl.empty else pd.DataFrame())
            reads = [x for _, r in rows.iterrows() for x in _sb_split(r.get("strengths")) + _sb_split(r.get("weaknesses"))]
            keys = [x for _, r in rows.iterrows() for x in _sb_split(r.get("keys_to_defending"))]
            items = reads[:2] + keys[:1]
            # Rebounding read (requested): only for the one player who dominates their offensive glass, by
            # the same REBOUND_RULES as the key and the Bottom Line. Goes FIRST so it survives the top-three
            # cut -- if he's doing this, it's the thing to know about him.
            _crash = _reb_crasher.get(_norm(name))
            if _crash:
                items = [_crash] + items
            if bullets and len(items) < 3:
                # Fill to three from whatever is left, keys first -- an instruction beats another label.
                items += [x for x in keys[1:] + reads[2:] if x not in items][:3 - len(items)]
            team = "opponent"
        else:
            bio = _uww_bio.get(_norm(name))
            pf = (_flags[_flags["player"].astype(str).map(_norm) == _norm(name)]
                  if not _flags.empty and "player" in _flags.columns else pd.DataFrame())
            items = [_sb_clean(f.get("flag")) for _, f in pf.iterrows()]
            calls = uww_player_calls(name)
            items += [calls] if calls else []
            team = "uww"
        detail = (("<ul class=\"tbl-bul\">" + "".join(f"<li>{_sb_esc(x)}</li>" for x in items[:3]) + "</ul>")
                  if bullets else " \u00b7 ".join(_sb_esc(x) for x in items))
        bits = " ".join(bio_bits(bio)) if bio is not None else ""
        return {"p": p, "bio": bio, "bits": bits, "items": items, "detail": detail, "team": team}

    def _ft_cell(p):
        # A bare "100.0" invites exactly the question asked about Hillmer's free throws -- real, or one
        # attempt? The makes-attempts pair rides along whenever the sample is thin -- unless it's already
        # 100%, where the fraction adds nothing a coach needs at a glance.
        return (f'<td title="{int(p.get("FTM") or 0)}-for-{int(p.get("FTA") or 0)}">{_sb_fmt(p["FT%"])}'
                + (f' <span class="mini">({int(p.get("FTM") or 0)}-{int(p.get("FTA") or 0)})</span>'
                   if (p.get("FTA") or 0) and p["FTA"] < 10
                   and round(float(p.get("FT%") or 0), 1) != 100.0 else "") + "</td>")

    def bench_table(names, stats_df, side, bullets=False):
        """bullets=True (starters): the last column holds up to three bulleted items instead of a one-line
        summary -- the same table the bench uses, with room for more of each starter's read."""
        if ROSTER_LAYOUT == "stacked":
            return stacked_roster(names, stats_df, side, bullets)
        add('<table class="compact roster">' + _ROSTER_COLS
            + '<thead><tr><th></th><th style="text-align:left">Player</th>'
            '<th style="text-align:left">Bio</th><th>GP</th><th>MIN</th><th>PTS</th><th>REB</th><th>AST</th>'
            '<th>FG%</th><th>3P%</th><th>FT%</th><th style="text-align:left">Late game foul</th>'
            '<th style="text-align:left">'
            + ("Top read / key" if side == "Opponent" else "Flag / play calls") + "</th></tr></thead><tbody>")
        for name in names:
            r = _roster_player(name, stats_df, side, bullets)
            p, bio = r["p"], r["bio"]
            add(f'<tr><td>{photo_cell(bio.get("photo_url") if bio is not None else None, name, side)}</td>'
                f'<td class="nm">{name_link_lines(name, page="upcoming", tab="personnel", team=r["team"], player=name)}</td>'
                f'<td class="wrap mini">{r["bits"]}</td>'
                + (f'<td>{_sb_fmt(p["games"], 0)}</td><td>{_sb_fmt(p.get("MIN"))}</td><td>{_sb_fmt(p["PTS"])}</td>'
                   f'<td>{_sb_fmt(p["REB"])}</td><td>{_sb_fmt(p["AST"])}</td><td>{_sb_fmt(p["FG%"])}</td>'
                   f'<td>{_sb_fmt(p["3P%"])}</td>' + _ft_cell(p)
                   if p is not None else '<td colspan="8" class="mini">no box-score line</td>')
                + (foul_cell(name) if side == "Opponent" else '<td class="mini">--</td>')
                + f'<td class="wrap mini">{r["detail"]}</td></tr>')
        add("</tbody></table>")

    # ---- STACKED roster layout (requested example) ---------------------------------------------------
    # Each player gets a HEADER BAR -- photo, name, bio, late-game foul call -- and the full width below
    # it goes to the numbers: the box-score line, then the shot profile (where he shoots from and how well,
    # from the play-by-play), then the reads. The table layout above has to fit all of that into one row,
    # which is why name and bio were squeezed into narrow columns and there was no room for shot data.
    _shot_prof = _sb_d("uww_player_shot_profile")

    def _shot_line(name, side):
        if _shot_prof.empty or "player" not in _shot_prof.columns:
            return ""
        sp = _shot_prof[(_shot_prof["side"] == side) & (_shot_prof["player"].map(_norm) == _norm(name))]
        if side == "Opponent" and "opponent" in sp.columns:
            sp = sp[sp["opponent"].astype(str) == str(short)]
        if sp.empty:
            return ""
        r = sp.iloc[0]
        fga = int(pd.to_numeric(r.get("fga"), errors="coerce") or 0)
        if fga < 5:
            return ""
        # A TABLE, not flex: the brief is printed and PDF'd, and older print engines lay a wrapping flex row
        # out unpredictably -- the caption and the three zones collapsed into one squeezed line in testing.
        cells = []
        for key, label in (("rim", "At the rim"), ("other2", "Other 2s"), ("three", "Threes")):
            share = pd.to_numeric(r.get(f"{key}_share"), errors="coerce")
            fg = pd.to_numeric(r.get(f"{key}_fg"), errors="coerce")
            att = int(pd.to_numeric(r.get(f"{key}_att"), errors="coerce") or 0)
            share = int(share) if pd.notna(share) else 0
            cells.append(f'<td class="sz"><div class="sz-l">{label}</div>'
                         f'<div class="sz-bar"><span style="width:{max(share, 2)}%"></span></div>'
                         f'<div class="sz-v"><strong>{share}%</strong> of shots &middot; '
                         + (f'{fg:.0f}% FG' if pd.notna(fg) else "no attempts" if not att else "--")
                         + (f' <span class="mini">({att})</span>' if att else "") + "</div></td>")
        return (f'<table class="sz-row"><tr><td class="sz-cap" colspan="3">Where he shoots '
                f'<span class="mini">{fga} FGA, from the play-by-play</span></td></tr><tr>'
                + "".join(cells) + "</tr></table>")

    def stacked_roster(names, stats_df, side, bullets=False):
        for name in names:
            r = _roster_player(name, stats_df, side, bullets)
            p, bio = r["p"], r["bio"]
            # The header has room for words, which are clearer than the table's check / cross icon.
            _foul = ""
            _fr = _foul_by.get(_norm(name)) if side == "Opponent" else None
            if _fr is not None:
                _call = _sb_clean(_fr.get("call"))
                if _call in ("Foul", "Do not foul"):
                    # Built outside the f-string: a backslash escape inside an f-string EXPRESSION is a
                    # SyntaxError before Python 3.12.
                    _fl = "Foul him late" if _call == "Foul" else "Don\u2019t foul late"
                    _fc = "call-foul" if _call == "Foul" else "call-nofoul"
                    _foul = (f'<span class="ph-foul {_fc}">{_fl} '
                             f'<span class="mini">({_sb_fmt(_fr.get("ft_pct"))}% FT)</span></span>')
            add('<div class="pcard">')
            add(f'<div class="ph">{photo_cell(bio.get("photo_url") if bio is not None else None, name, side)}'
                f'<div class="ph-name">{name_link(name, page="upcoming", tab="personnel", team=r["team"], player=name)}'
                f'<div class="ph-bio">{r["bits"]}</div></div>{_foul}</div>')
            if p is not None:
                add('<table class="compact pstats"><thead><tr><th>GP</th><th>MIN</th><th>PTS</th><th>REB</th>'
                    '<th>AST</th><th>FG%</th><th>3P%</th><th>FT%</th></tr></thead><tbody><tr>'
                    f'<td>{_sb_fmt(p["games"], 0)}</td><td>{_sb_fmt(p.get("MIN"))}</td><td>{_sb_fmt(p["PTS"])}</td>'
                    f'<td>{_sb_fmt(p["REB"])}</td><td>{_sb_fmt(p["AST"])}</td><td>{_sb_fmt(p["FG%"])}</td>'
                    f'<td>{_sb_fmt(p["3P%"])}</td>' + _ft_cell(p) + "</tr></tbody></table>")
            else:
                add('<p class="mini">No box-score line.</p>')
            add(_shot_line(name, side))
            if r["items"]:
                add('<div class="p-reads">' + ("<ul class=\"tbl-bul\">" + "".join(
                    f"<li>{_sb_esc(x)}</li>" for x in r["items"][:3]) + "</ul>") + "</div>")
            add("</div>")

    def personnel_pages(side, team_title, stats_df, card_fn, extra_no_min="", break_first=True, lead=None):
        names_by_prod = list(stats_df.sort_values("PTS_total", ascending=False)["name"]) if not stats_df.empty else []
        starters, bench, basis, limited = tier_groups(side, names_by_prod)
        if starters is None:
            starters, bench = names_by_prod[:5], [("Bench", "No personnel tiers on file yet.", names_by_prod[5:])]
            basis = "no tier table -- top five by production"
        if not starters and not any(g[2] for g in bench):
            return
        if break_first:
            page_break()
        if lead:
            lead()  # e.g. the team divider, kept on the same page as the starters it introduces
        section(f"\U0001f465 {team_title} - ROSTER"
                + footnote(f"{team_title} starters: {basis}. Per-game averages from their own film. "
                           + ("Photo, jersey, position, height and class come from the FastScout roster page when "
                              "a live scrape has run; hand and status aren't on that page and stay blank. Coach "
                              "notes pass through as written; data-driven lines are stated against the team's own "
                              "averages with attempt floors." if side == "Opponent" else
                              "Flags come from our own season film (highest-confidence first in the app). Play-call "
                              "lines need 3+ tagged uses of a set.")))
        # CONFIRMED CHANGE (requested): starters and every bench tier live in ONE roster section, in the same
        # table shape. Starters get up to three bullets in the last column; each player's full card (every
        # read, note and key) is one click away in the app.
        add('<div class="card">')
        add('<div class="tier-h">Starters</div>')
        bench_table(starters, stats_df, side, bullets=True)
        for label, caption, names in bench:
            if not names:
                continue
            add(f'<div class="tier-h">{_sb_esc(label)}</div><p class="tier-c">{_sb_esc(caption)}</p>')
            bench_table(names, stats_df, side)
        # Players never on the floor in any game on film are also app-only now; count them with the rest.
        _extra_n = len([x for x in str(extra_no_min or "").split(",") if x.strip()])
        if side == "Opponent" and _foul_by:
            add('<p class="mini">Late game foul: <span class="call-foul">\u2714</span> foul him '
                '(62% or worse on 8+ attempts) &middot; <span class="call-nofoul">\u2718</span> do not foul '
                '(75% or better, or 85%+ on as few as 4 attempts) &middot; \u2013 neutral or too few '
                'attempts to say.</p>')
        # ONE link out of the roster, not two (requested). Under-8-minute players are tiered but no longer
        # printed, so the count rides on this same line rather than earning a second identical hyperlink.
        _omitted = len(limited) + _extra_n
        add('<p class="mini">'
            + (f'{_omitted} more player(s) \u2014 under 8 minutes a game or not playing recently \u2014 are '
               f'not listed here. ' if _omitted else "")
            + app_link("Full card for any player", page="upcoming", tab="personnel",
                       team="uww" if side == "UWW" else "opponent") + "</p>")
        add("</div>")

    # ---- compact tables used by several sections -------------------------------------------------------
    def pc_table(rows, cols, team_ppp=None, row_class=""):
        out = ["<table class=\"compact\"><thead><tr>"
               + "".join(f'<th style="text-align:{"left" if k in ("name", "top_player", "situation") else "right"}">{_sb_esc(l)}</th>' for k, l in cols)
               + "</tr></thead><tbody>"]
        _rc = f' class="{row_class}"' if row_class else ""
        for _, r in rows.iterrows():
            cells = []
            for k, _l in cols:
                v = r.get(k)
                if k == "ppp":
                    cls = ""
                    if pd.notna(v) and team_ppp is not None and pd.notna(team_ppp):
                        cls = "ppp-good" if v >= team_ppp + 0.15 else ("ppp-bad" if v <= team_ppp - 0.15 else "")
                    cells.append(f'<td class="{cls}">{_sb_fmt(v, 2)}</td>')
                elif k == "fg":
                    cells.append(f'<td>{int(r["fgm"])}/{int(r["fga"])}</td>' if int(r.get("fga") or 0) else "<td>--</td>")
                elif k in ("name", "top_player", "situation"):
                    cells.append(f'<td class="wrap">{_sb_esc(_sb_clean(v))}</td>')
                else:
                    cells.append(f"<td>{_sb_fmt(v, 0)}</td>")
            out.append(f"<tr{_rc}>" + "".join(cells) + "</tr>")
        out.append("</tbody></table>")
        return "".join(out)

    def lineups_table(lu_df, side, note_rows=True):
        if lu_df.empty or "lineup" not in lu_df.columns:
            return
        top = lu_df.sort_values("MIN", ascending=False).head(3)
        # PPP per five-man unit, from the play-call summary's own per-lineup rows (level "Personnel grouping"),
        # so it means the same thing here as everywhere else in the brief.
        _lu_ppp = {}
        if not _pcs_all.empty and "level" in _pcs_all.columns:
            _lp = _pcs_all[(_pcs_all["side"] == side) & (_pcs_all["level"] == "Personnel grouping")]
            _lu_ppp = {str(r["name"]): (r.get("ppp"), r.get("uses")) for _, r in _lp.iterrows()}
        # What each unit actually runs (requested), from the parser's per-lineup play-call table, plus its
        # grouping shape so the row says who is on the floor as well as what they call.
        _lu_calls, _lu_group = {}, {}
        _lpc = _sb_d("uww_lineup_play_calls")
        if not _lpc.empty and "lineup" in _lpc.columns:
            _lpc = _lpc[_lpc["side"] == side] if "side" in _lpc.columns else _lpc
            _lu_calls = {str(r["lineup"]): _sb_clean(r.get("top_calls")) for _, r in _lpc.iterrows()}
            _lu_group = {str(r["lineup"]): _sb_clean(r.get("grouping")) for _, r in _lpc.iterrows()}
        add('<table class="compact"><thead><tr><th style="text-align:left">Lineup</th>'
            '<th style="text-align:left">Group</th><th>GP</th><th>MIN</th>'
            '<th>+/-</th><th>PPP</th>' + ("<th>FG%</th><th>3P%</th>" if "FG%" in lu_df.columns else "")
            + '<th style="text-align:left">What they run</th>'
            + ('<th style="text-align:left">Key</th>' if note_rows else "") + "</tr></thead><tbody>")
        for _, u in top.iterrows():
            key = ""
            if note_rows and not _notes_tbl.empty:
                nr = _notes_tbl[(_notes_tbl["subject_type"] == "lineup") & (_notes_tbl["name"] == str(u["lineup"]))]
                ks = [k for _, r in nr.iterrows() for k in _sb_split(r.get("keys_to_defending"))]
                key = ks[0] if ks else ""
            _ppp, _ppp_n = _lu_ppp.get(str(u["lineup"]), (None, None))
            add(f'<tr><td class="wrap">{_sb_esc(u["lineup"])}</td>'
                f'<td class="mini">{_sb_esc(_lu_group.get(str(u["lineup"]), ""))}</td>'
                f'<td>{_sb_fmt(u.get("GP"), 0)}</td>'
                f'<td>{_sb_fmt(u.get("MIN"))}</td><td>{_sb_fmt(u.get("+/-"), 0)}</td>'
                + (f'<td title="{int(_ppp_n)} tagged possessions">{_sb_fmt(_ppp, 2)}</td>'
                   if _ppp is not None and pd.notna(_ppp) else '<td class="mini">--</td>')
                + (f'<td>{_sb_fmt(u.get("FG%"))}</td><td>{_sb_fmt(u.get("3P%"))}</td>' if "FG%" in lu_df.columns else "")
                + f'<td class="wrap mini">{_sb_esc(_lu_calls.get(str(u["lineup"]), "--"))}</td>'
                + (f'<td class="wrap mini">{_sb_esc(key)}</td>' if note_rows else "") + "</tr>")
        add("</tbody></table>")

    def combos_table(side):
        """Top three 3-man combos by minutes (requested), alongside the five-man units.

        Same table the Keys to Victory's combo keys are built from (uww_three_man_combos, exported by the
        KTV cell), so a combo named in a key and the combo shown here can never disagree. PPP and "What they
        run" come from tagged possessions with all three on the floor; per-100 margin is shown next to raw
        +/- because a trio's minutes vary far more than a five's, and raw +/- mostly measures playing time.
        """
        df = _sb_d("uww_three_man_combos")
        if df.empty or "lineup" not in df.columns or "side" not in df.columns:
            return False
        df = df[df["side"] == side]
        if "scouted_opponent" in df.columns:
            df = df[df["scouted_opponent"].astype(str) == str(short)]
        if df.empty:
            return False
        df = df.assign(_min=pd.to_numeric(df["MIN"], errors="coerce"),
                       _pm=pd.to_numeric(df["+/-"], errors="coerce"))
        top = df.sort_values("_min", ascending=False).head(3)
        _gp_exact = bool(top["gp_exact"].astype(str).str.lower().eq("true").all()) if "gp_exact" in top.columns else True

        # Tagged possessions with all three on the floor -- offensive possessions only, matched on
        # normalised names so "Jake Quast" and "JAKE QUAST" count as the same player.
        _clips = pd.DataFrame()
        if not _pcc_all.empty and "on_court_lineup" in _pcc_all.columns:
            _clips = _pcc_all[(_pcc_all["side"] == side) & (_pcc_all["decode_quality"] != "Needs review")
                              & _pcc_all["on_court_lineup"].notna()].copy()
            if "possession_side" in _clips.columns:
                _clips = _clips[_clips["possession_side"].astype(str) != "Defense"]
            if not _clips.empty:
                _clips["_five"] = _clips["on_court_lineup"].astype(str).map(
                    lambda s: {_norm(x) for x in re.split(r"[,/|]", s) if x.strip()})
                _clips["_pts"] = pd.to_numeric(_clips.get("points"), errors="coerce")

        add('<table class="compact"><thead><tr><th style="text-align:left">Combo</th>'
            + ("<th>GP</th>" if _gp_exact else "")
            + "<th>MIN</th><th>+/-</th><th>per 40</th><th>PPP</th>"
            + '<th style="text-align:left">What they run</th></tr></thead><tbody>')
        for _, u in top.iterrows():
            _trio = {_norm(x) for x in str(u["lineup"]).split(",") if x.strip()}
            _ppp_cell, _run = '<td class="mini">--</td>', "--"
            if not _clips.empty and len(_trio) == 3:
                _on = _clips[_clips["_five"].map(lambda f: _trio <= f)]
                _k = _on["_pts"].notna().sum() if not _on.empty else 0
                if _k:
                    _ppp_cell = (f'<td title="{int(_k)} tagged possessions">'
                                 f'{_sb_fmt(_on["_pts"].sum() / _k, 2)}</td>')
                if not _on.empty and "play_call" in _on.columns:
                    _vc = (_on["play_call"].dropna().astype(str)
                           .loc[lambda x: ~x.str.contains("unspecified", na=False)].value_counts())
                    if len(_vc):
                        _run = ", ".join(f"{n} ({int(c)}x)" for n, c in _vc.head(2).items())
            _per40 = (u["_pm"] / u["_min"] * 40) if pd.notna(u["_min"]) and u["_min"] > 0 and pd.notna(u["_pm"]) else None
            add(f'<tr><td class="wrap">{_sb_esc(", ".join(_sb_title(x.strip()) for x in str(u["lineup"]).split(",")))}</td>'
                + (f'<td>{_sb_fmt(u.get("GP"), 0)}</td>' if _gp_exact else "")
                + f'<td>{_sb_fmt(u["_min"])}</td><td>{("+" if pd.notna(u["_pm"]) and u["_pm"] > 0 else "") + _sb_fmt(u["_pm"], 0)}</td>'
                + f'<td>{("+" if _per40 is not None and _per40 > 0 else "") + _sb_fmt(_per40)}</td>'
                + _ppp_cell
                + f'<td class="wrap mini">{_sb_esc(_run)}</td></tr>')
        add("</tbody></table>")
        return True

    # Reconciliation-only now (requested: the "shows its top set..." explanation moved to a visible line
    # under each section header instead of staying a footnote -- see the description added right after
    # section() in the offense sections below). This one stays a footnote: it's a caveat about missing
    # data, not a description of what's shown.
    _SERIES_PIVOT_NOTE = (
        "Not every tagged possession in HOW {OPP} RUNS OFFENSE or HOW WE RUN OFFENSE ran a named set -- some "
        "are tagged with the situation only (untagged, flagged for rewatch, or the title named only a player). "
        "The full list of named sets and their clips is in the app.")

    def _film_clip_lookup():
        """normalized set name -> clip count, from uww_film_clips -- only ever populated for the Opponent
        side (see the game-plan cell). Used to badge the matching rows in HOW {OPP} RUNS OFFENSE instead of
        listing them again in their own FILM SESSION section (requested). clip_group arrives as
        "Set Name (Situation)"; the situation is dropped and counts are summed, so a set tagged in more than
        one situation badges with its full clip count wherever it appears."""
        df, _ = staff_rows("uww_film_clips")
        out = {}
        if df.empty or "clip_group" not in df.columns:
            return out
        for _, r in df.iterrows():
            _group = _sb_clean(r.get("clip_group"))
            m = re.match(r"^(.*) \((.*)\)$", _group)
            _name = m.group(1) if m else _group
            if not _name:
                continue
            out[_norm(_name)] = out.get(_norm(_name), 0) + int(pd.to_numeric(r.get("clips"), errors="coerce") or 0)
        return out


    _CLOCK_ORDER = ["Early clock (0-9 sec used)", "Organized offense (10-19 sec used)", "Late clock (20+ sec used)"]
    _SIT_ORDER = ["Leading by 10+", "Trailing by 10+", "Clutch (last 5 min, margin \u2264 8)"]

    # =====================================================================================================
    # PAGE 1 -- GAME PLAN
    # =====================================================================================================
    lede = []
    if them.get("games"):
        lede.append(f"Scoring {_sb_fmt(them.get('PTS'))} a game and allowing {_sb_fmt(them_allowed.get('PTS'))}, "
                    f"over {int(them['games'])} game{'s' if them['games'] != 1 else ''} of film.")
    if not players.empty:
        _tg = int(them.get("games") or players["games"].max() or 1)
        _regular = players[players["games"] >= max(1, 0.5 * _tg)].sort_values("PTS_total", ascending=False)
        _team_pts_total = float(players["PTS_total"].sum()) or None
        if not _regular.empty:
            top = _regular.iloc[0]
            share = round(100 * float(top["PTS_total"]) / _team_pts_total) if _team_pts_total else None
            lede.append(f"{top['name']} leads them at {_sb_fmt(top['PTS'])} a night"
                        + (f" \u2014 {share}% of their points on film." if share else "."))
            if len(_regular) > 1:
                second = _regular.iloc[1]
                lede.append(f"{second['name']} is the next threat at {_sb_fmt(second['PTS'])} "
                            f"({_sb_fmt(second['REB'])} reb, {_sb_fmt(second['AST'])} ast).")
    if them.get("3P%") is not None and them.get("3PA") is not None:
        lede.append(f"They shoot {_sb_fmt(them['3P%'])}% from three on {_sb_fmt(them['3PA'])} attempts a game.")
    # Pace moved out of its own section (requested) -- the full tempo table stays in the app. It earns a
    # Bottom Line sentence only when it would change preparation; the thresholds are BOTTOM_LINE_RULES at
    # the top of this cell, and the sentence carries a note stating them so the staff can see the rule.
    _R = BOTTOM_LINE_RULES
    _tp, _tp_sample = staff_rows("uww_tempo_profile")
    if not _tp.empty and not _tp_sample and {"opponent", "metric", "value"}.issubset(_tp.columns):
        def _tv(team, metric):
            _r = _tp[(_tp["opponent"].astype(str) == str(team)) & (_tp["metric"].astype(str) == metric)]
            return pd.to_numeric(_r["value"].iloc[0], errors="coerce") if not _r.empty else None
        _their_pace = _tv(short, "Possessions per game")
        _our_pace = _tv("UW-Whitewater", "Possessions per game")
        _early = _tv(short, "Early-offense share")
        _bits = []
        if _their_pace is not None and _our_pace is not None and pd.notna(_their_pace) and pd.notna(_our_pace) \
                and abs(_their_pace - _our_pace) >= _R["tempo_pace_gap"]:
            _bits.append(f"they play {'faster' if _their_pace > _our_pace else 'slower'} than we do "
                         f"({_their_pace:.0f} possessions a game to our {_our_pace:.0f})")
        if _early is not None and pd.notna(_early) and _early >= _R["tempo_early_share"]:
            _bits.append(f"{_early:.0f}% of their tagged possessions shoot inside 10 seconds, so transition "
                         "defense needs its own work")
        if _bits:
            lede.append("Tempo: " + "; ".join(_bits) + ".")
    # Four Factors no longer gets its own table (requested) -- call out the biggest weighted gap here instead,
    # but only when it's genuinely decisive: the top factor's weighted edge has to clear both an absolute floor
    # (so a game with four small, similar factors stays quiet) and a relative one (at least double the next
    # factor, so it reads as THE story, not just nominally first).
    if _ff_top is not None and len(_ff_rows) > 1:
        _ff_second = _ff_rows[1]
        _ff_others_sum = sum(abs(r["weighted"]) for r in _ff_rows[1:])
        if (abs(_ff_top["weighted"]) >= BOTTOM_LINE_RULES["ff_min_weighted"]
                and abs(_ff_top["weighted"]) >= BOTTOM_LINE_RULES["ff_dominance"] * abs(_ff_second["weighted"])):
            _whose = "our" if _ff_top["edge"] > 0 else f"{_sb_clean(opp_display)}'s"
            _tail = (", more than the other three factors combined."
                     if abs(_ff_top["weighted"]) > _ff_others_sum
                     else f", well clear of the next-largest factor ({_ff_second['factor']}, "
                          f"{_ff_second['weighted']:+.2f}).")
            lede.append(f"The clearest statistical edge is {_ff_top['factor']} \u2014 {_whose} edge "
                        f"(UWW {_ff_top['uww']:.1f} vs {_ff_top['opp']:.1f}), a {_ff_top['edge']:+.1f} point gap "
                        f"worth {_ff_top['weighted']:+.2f} weighted" + _tail)

    # ---- Rebounding: only when lopsided (REBOUND_RULES, shared with the key and the roster read) -------
    _RR = globals().get("REBOUND_RULES") or {"min_paired": 20, "crash_rate": 35, "soft_rate": 65,
                                              "player_share": 35, "player_min": 5}
    _rs = _sb_d("uww_rebound_summary")
    if not _rs.empty and "opponent" in _rs.columns:
        _rs = _rs[_rs["opponent"].astype(str) == str(short)]
    _reb_bits = []
    if not _rs.empty and "metric" in _rs.columns:
        def _rv(metric):
            _r = _rs[_rs["metric"] == metric]
            return _r.iloc[0] if not _r.empty else None
        _o, _d = _rv("their_oreb_pct"), _rv("their_dreb_pct")
        if _o is not None and int(_o["paired"]) >= _RR["min_paired"] and float(_o["value"]) >= _RR["crash_rate"]:
            _top = _rs[_rs["metric"] == "their_oreb_player"].sort_values("count", ascending=False)
            _reb_bits.append(f"they grab {float(_o['value']):.0f}% of their own misses"
                             + (f", {_sb_title(_top.iloc[0]['player'])} the most ({int(_top.iloc[0]['count'])})"
                                if not _top.empty else ""))
        if _d is not None and int(_d["paired"]) >= _RR["min_paired"] and float(_d["value"]) <= _RR["soft_rate"]:
            _reb_bits.append(f"they secure only {float(_d['value']):.0f}% of opponents' misses, so our misses are "
                             "live balls")
    if _reb_bits:
        lede.append("Rebounding: " + "; ".join(_reb_bits) + ".")

    # ---- Style matchups: only when the record is lopsided, on enough games, at real confidence ---------
    _sm = _sb_d("uww_style_matchups")
    if not _sm.empty and {"opponent", "direction", "dir_wins", "dir_losses", "confidence"}.issubset(_sm.columns):
        _sm = _sm[_sm["opponent"].astype(str) == str(short)]
        for _dir in ("like_them", "like_us"):
            _g = _sm[_sm["direction"] == _dir]
            if _g.empty:
                continue
            _w = pd.to_numeric(_g["dir_wins"], errors="coerce").iloc[0]
            _l = pd.to_numeric(_g["dir_losses"], errors="coerce").iloc[0]
            _conf = pd.to_numeric(_g["confidence"], errors="coerce").max()
            if pd.isna(_w) or pd.isna(_l) or pd.isna(_conf):
                continue
            _n = int(_w) + int(_l)
            _share = (_w / _n) if _n else 0.5
            if (_n < BOTTOM_LINE_RULES["style_min_games"] or _conf < BOTTOM_LINE_RULES["style_min_confidence"]
                    or (1 - BOTTOM_LINE_RULES["style_lopsided"]) < _share < BOTTOM_LINE_RULES["style_lopsided"]):
                continue
            _teams = ", ".join(_sb_strip_mascot(t) for t in _g.sort_values("rank")["team"].head(3))
            _pf = pd.to_numeric(_g["dir_pf"], errors="coerce").iloc[0] if "dir_pf" in _g.columns else None
            _pa = pd.to_numeric(_g["dir_pa"], errors="coerce").iloc[0] if "dir_pa" in _g.columns else None
            _sc = f", averaging {_pf:.0f}-{_pa:.0f}" if _pf is not None and _pa is not None and pd.notna(_pf) \
                and pd.notna(_pa) else ""
            if _dir == "like_them":
                lede.append(f"Against teams that play like them ({_teams}) we're {int(_w)}-{int(_l)}{_sc}.")
            else:
                lede.append(f"Teams that play like us ({_teams}) went {int(_w)}-{int(_l)} against them{_sc}.")
    # Read-with-caution items are rendered at the very END of the brief (requested) -- they qualify numbers
    # throughout, so they read better as a closing caveat than as a wall of warnings on the game-plan page.
    _warn_df, _ = staff_rows("uww_sample_size_warnings")
    if lede:
        # The rules for the CONDITIONAL lines are documented on the section header, not on each sentence.
        # A per-sentence note only exists when its sentence prints -- so "why is there no tempo line?" would
        # have no answer at exactly the moment it gets asked. The header note is always present, and its
        # numbers come straight from BOTTOM_LINE_RULES, so it can't drift from what the code does.
        _R = BOTTOM_LINE_RULES
        section("\U0001f4cc THE BOTTOM LINE" + footnote(
            "How THE BOTTOM LINE is built. Scoring, leading scorers and three-point shooting always appear. Two "
            "lines are CONDITIONAL and only print when the difference is big enough to change how we prepare -- "
            "if one is missing, the difference was checked and was small, not skipped. "
            f"TEMPO: shown when their pace differs from ours by {_R['tempo_pace_gap']}+ possessions a game, OR "
            f"{_R['tempo_early_share']}%+ of their tagged possessions shoot inside 10 seconds. Possessions are "
            "estimated as FGA - OREB + TO + 0.475*FTA from the box score. "
            f"FOUR FACTORS: shown when the biggest weighted gap is at least {_R['ff_min_weighted']:g} AND at "
            f"least {_R['ff_dominance']:g}x the second-largest, so it names one clear story rather than the "
            "nominal leader of a close call. Weights are Dean Oliver's -- shooting 40%, turnovers 25%, offensive "
            "rebounding 20%, free throws 15% -- and a positive edge always favours UWW (a lower turnover rate is "
            "an edge). Neither side is adjusted for strength of schedule. "
            f"REBOUNDING (every miss in their play-by-play paired with the rebound that follows): shown when, on "
            f"{_RR['min_paired']}+ paired misses, they rebound {_RR['crash_rate']}%+ of their own misses, OR secure "
            f"{_RR['soft_rate']}% or less of their opponents' misses. The same rules create the matching Keys to "
            f"Victory key, and a player with {_RR['player_share']}%+ of their offensive rebounds "
            f"({_RR['player_min']}+ boards) gets a roster read. "
            f"STYLE MATCHUPS: shown when the record against teams that play like them (or of teams like us "
            f"against them) is at least {BOTTOM_LINE_RULES['style_lopsided']:.0%} one way, over "
            f"{BOTTOM_LINE_RULES['style_min_games']}+ games, with the best match at least "
            f"{BOTTOM_LINE_RULES['style_min_confidence']:.2f} confidence. "
            "Full tempo, four factors, rebounding and style-matchup detail are in the app."))
        add('<div class="card">')
        # An item is plain text, or (text, marker) when the sentence is conditional and carries a note
        # explaining when it appears. The text is escaped; the marker is already-built HTML.
        add('<ul class="bl">' + "".join(
            f"<li>{_sb_esc(l[0])}{l[1]}</li>" if isinstance(l, tuple) else f"<li>{_sb_esc(l)}</li>"
            for l in lede) + "</ul>")
        add("</div>")

    # ---- head-to-head: how the previous meeting(s) went (requested) ------------------------------------
    _h2h = _sb_d("uww_head_to_head")
    if not _h2h.empty:
        _h2h_w = int((_h2h["outcome"].astype(str).str.upper() == "W").sum())
        section(f"\U0001f501 WHEN WE'VE PLAYED {OPP_UP}" + footnote(
            "Previous meetings with this opponent, from our own schedules -- this season's earlier games plus the "
            "last two seasons. Each meeting's play-by-play is pulled from its own box-score link and cached, so "
            "the player lines below are that game's real box score. \"How this team is different now\" compares "
            "who played in that meeting against this year's roster and box scores -- points shown in brackets "
            "are what that player scored against US in the meeting, not a season average."))
        add('<div class="card">')
        add(f'<p class="ff-lede">{_h2h_w}-{len(_h2h) - _h2h_w} in {len(_h2h)} previous meeting'
            f'{"s" if len(_h2h) != 1 else ""}.</p>')
        add('<table class="compact"><thead><tr><th style="text-align:left">Season</th>'
            '<th style="text-align:left">Date</th><th style="text-align:left">Site</th>'
            '<th style="text-align:left">Result</th><th>Margin</th>'
            '<th style="text-align:left">Our leader</th><th style="text-align:left">Their leader</th>'
            "</tr></thead><tbody>")
        for _, _g in _h2h.iterrows():
            _oc = _sb_clean(_g.get("outcome")).upper()
            _score = (f'{int(_g["team_score"])}-{int(_g["opponent_score"])}'
                      if pd.notna(_g.get("team_score")) and pd.notna(_g.get("opponent_score")) else "")
            _m = _g.get("margin")
            add(f'<tr><td>{_sb_esc(_sb_clean(_g.get("season")))}</td>'
                f'<td>{_sb_esc(_sb_clean(_g.get("date")))}</td>'
                f'<td>{_sb_esc(_sb_clean(_g.get("home_away")))}</td>'
                f'<td class="{"call-foul" if _oc == "W" else "call-nofoul"}">{_sb_esc(_oc)} {_sb_esc(_score)}</td>'
                f'<td>{("+" if pd.notna(_m) and _m > 0 else "") + _sb_fmt(_m, 0)}</td>'
                f'<td class="wrap mini">{_sb_esc(_sb_clean(_g.get("uww_leader")))}</td>'
                f'<td class="wrap mini">{_sb_esc(_sb_clean(_g.get("opp_leader")))}</td></tr>')
        add("</tbody></table>")

        # ---- what we ran in each meeting, what defense, and whether it worked (requested) ----------------
        # From the tagged clips of that game, joined on the meeting's real date (head_to_head.game_date).
        # "Worked" is measured against our own season average, not an absolute number: 0.95 PPP is good
        # for some teams and bad for others, and the question is whether the plan beat what we normally do.
        # Most meetings are from earlier seasons that were never tagged -- when NONE of them are, this says
        # so and shows a clearly-labelled sample of what the block will look like, so the staff knows the
        # section exists and what tagging it needs, rather than it silently not appearing.
        def _h2h_clip_view():
            if _pcc_all.empty or "game_date" not in _h2h.columns or "game_date" not in _pcc_all.columns:
                return []
            _c = _pcc_all[(_pcc_all["side"] == "UWW") & (_pcc_all["decode_quality"] != "Needs review")].copy()
            if _c.empty:
                return []
            _c["_d"] = pd.to_datetime(_c["game_date"], errors="coerce").dt.strftime("%Y-%m-%d")
            _c["_pts"] = pd.to_numeric(_c.get("points"), errors="coerce")
            _ps = _c["possession_side"].astype(str) if "possession_side" in _c.columns \
                else pd.Series("Offense", index=_c.index)
            _c["_off"] = _ps != "Defense"

            def _ppp(g):
                k = g["_pts"].notna().sum()
                return (g["_pts"].sum() / k) if k else None

            # Season baselines -- what "worked" is measured against.
            _base_off = _ppp(_c[_c["_off"]])
            _base_def = _ppp(_c[~_c["_off"]])
            views = []
            for _, _g in _h2h.iterrows():
                _gd = _sb_clean(_g.get("game_date"))
                if not _gd:
                    continue
                _m = _c[_c["_d"] == _gd]
                if _m.empty:
                    continue
                views.append((_g, _m[_m["_off"]], _m[~_m["_off"]], _base_off, _base_def, _ppp))
            return views

        def _verdict(v, base, lower_is_better=False):
            if v is None or base is None or pd.isna(v) or pd.isna(base):
                return ""
            d = v - base
            good = (d <= -0.10) if lower_is_better else (d >= 0.10)
            bad = (d >= 0.10) if lower_is_better else (d <= -0.10)
            tag = "worked" if good else ("didn't work" if bad else "about normal")
            cls = "ppp-good" if good else ("ppp-bad" if bad else "")
            return f'<span class="{cls}"><strong>{tag}</strong></span> ({v:.2f} vs our usual {base:.2f})'

        def _top_calls(g, _ppp, n=3):
            if g.empty or "play_call" not in g.columns:
                return ""
            g = g[g["play_call"].notna() & ~g["play_call"].astype(str).str.contains("unspecified", na=False)]
            out = []
            for _name, _cg in g.groupby(g["play_call"].astype(str)):
                out.append((_name, len(_cg), _ppp(_cg)))
            out.sort(key=lambda x: -x[1])
            return ", ".join(f"{_sb_esc(nm)} {k}x" + (f" ({p:.2f})" if p is not None else "")
                             for nm, k, p in out[:n])

        def _def_mix(g, col):
            if g.empty or col not in g.columns or g[col].isna().all():
                return ""
            vc = g[col].dropna().astype(str).value_counts()
            return ", ".join(f"{_sb_esc(k)} {round(100 * int(v) / int(vc.sum()))}%" for k, v in vc.head(2).items())

        _views = _h2h_clip_view()
        add('<div class="pc-sub">What we ran, and did it work</div>')
        if _views:
            add('<table class="compact"><thead><tr><th style="text-align:left">Meeting</th>'
                '<th style="text-align:left">Our offense</th><th style="text-align:left">Our defense</th>'
                "</tr></thead><tbody>")
            for _g, _off, _def, _bo, _bd, _ppp in _views:
                _off_ppp, _def_ppp = _ppp(_off), _ppp(_def)
                _o = []
                if len(_off):
                    _o.append(f"{len(_off)} tagged possessions \u2014 {_verdict(_off_ppp, _bo)}")
                    _tc = _top_calls(_off, _ppp)
                    if _tc:
                        _o.append(f"Ran most: {_tc}")
                    _dm = _def_mix(_off, "defense_faced")
                    if _dm:
                        _o.append(f"They played: {_dm}")
                else:
                    _o.append('<span class="mini">not tagged</span>')
                _d = []
                if len(_def):
                    _d.append(f"{len(_def)} tagged possessions \u2014 {_verdict(_def_ppp, _bd, lower_is_better=True)}")
                    _dm = _def_mix(_def, "defense_played")
                    if _dm:
                        _d.append(f"We played: {_dm}")
                    _tc = _top_calls(_def, _ppp)
                    if _tc:
                        _d.append(f"They ran: {_tc}")
                else:
                    _d.append('<span class="mini">not tagged</span>')
                add(f'<tr><td class="wrap">{_sb_esc(_sb_clean(_g.get("date")))}<br>'
                    f'<span class="mini">{_sb_esc(_sb_clean(_g.get("season")))} &middot; '
                    f'{_sb_esc(_sb_clean(_g.get("outcome")).upper())}</span></td>'
                    f'<td class="wrap mini">{"<br>".join(_o)}</td>'
                    f'<td class="wrap mini">{"<br>".join(_d)}</td></tr>')
            add("</tbody></table>")
            add('<p class="mini">"Worked" means 0.10+ PPP better than our season average (on defense, 0.10+ '
                'fewer allowed); numbers in brackets are PPP. Meetings without tagged clips are left out.</p>')
        else:
            # Honest fallback: say it isn't tagged, then show a labelled sample so the section's shape and
            # purpose are visible. Every value below is invented and marked as such.
            add('<div class="sample"><span class="sample-tag">SAMPLE DATA \u2014 NOT FROM YOUR FILES</span>'
                f'<p class="sample-why">None of our previous meetings with {_sb_esc(_sb_clean(opp_display))} have '
                'tagged play calls yet, so what we ran and whether it worked can\'t be shown. Tag the meeting in '
                'uww_plays.csv (offense and defense) and this fills in automatically. What it will look '
                'like:</p>'
                '<table class="compact"><thead><tr><th style="text-align:left">Meeting</th>'
                '<th style="text-align:left">Our offense</th><th style="text-align:left">Our defense</th>'
                '</tr></thead><tbody><tr><td class="wrap">Wed, Nov 20<br><span class="mini">2024-25 &middot; W'
                '</span></td><td class="wrap mini">61 tagged possessions \u2014 <strong>worked</strong> (1.18 vs '
                'our usual 1.04)<br>Ran most: 5 Out Ball Screen 12x (1.33), Chin 9x (1.22), Horns Flare 7x '
                '(0.71)<br>They played: Man-to-Man 82%, 2-3 Zone 18%</td><td class="wrap mini">58 tagged '
                'possessions \u2014 <strong>about normal</strong> (0.97 vs our usual 0.99)<br>We played: '
                'Man-to-Man 90%, 2-3 Zone 10%<br>They ran: 4-1 Ball Screen 11x (0.64), Hi-Lo DHO 8x '
                '(1.50)</td></tr></tbody></table></div>')

        # ---- the meeting in full, and how that team has changed since (requested) ----------------------
        _h2h_box = _sb_d("uww_head_to_head_box")
        _h2h_chg = _sb_d("uww_head_to_head_roster_change")
        if not _h2h_chg.empty and "opponent" in _h2h_chg.columns:
            _h2h_chg = _h2h_chg[_h2h_chg["opponent"].astype(str) == str(short)]
        if not _h2h_box.empty or not _h2h_chg.empty:
            add('<div class="grid2"><div>')
        if not _h2h_box.empty:
            add('<div class="pc-sub">Who did the damage</div>')
            _last = _h2h_box[_h2h_box["meeting_date"] == _h2h_box["meeting_date"].iloc[0]] \
                if "meeting_date" in _h2h_box.columns else _h2h_box
            for _team_val, _grp in _last.groupby("team"):
                _grp = _grp.assign(_p=pd.to_numeric(_grp["PTS"], errors="coerce")).sort_values("_p", ascending=False)
                _who = ", ".join(f"{r['player']} {int(r['_p'])}" for _, r in _grp.head(4).iterrows()
                                 if pd.notna(r["_p"]))
                add(f'<p class="mini"><strong>{_sb_esc(_sb_strip_mascot(_team_val))}:</strong> {_sb_esc(_who)}</p>')
        add("</div><div>")
        if not _h2h_chg.empty:
            _ret = _h2h_chg[_h2h_chg["status"] == "Returning"]
            _gone = _h2h_chg[_h2h_chg["status"] == "Gone"]
            _new = _h2h_chg[_h2h_chg["status"] == "New since then"]
            _ret_pts = pd.to_numeric(_ret["prev_pts"], errors="coerce").sum()
            _gone_pts = pd.to_numeric(_gone["prev_pts"], errors="coerce").sum()
            add('<div class="pc-sub">How this team is different now</div>')
            add(f'<p class="mini">{len(_ret)} of the {len(_ret) + len(_gone)} players who faced us are back '
                f'({_ret_pts:.0f} of the {_ret_pts + _gone_pts:.0f} points they scored on us); {len(_gone)} gone, '
                f'{len(_new)} new since.</p>')
            for _label, _sel in (("Back", _ret), ("Gone", _gone)):
                if _sel.empty:
                    continue
                _names = ", ".join(
                    f"{r['player']}" + (f" ({int(pd.to_numeric(r['prev_pts'], errors='coerce'))})"
                                        if pd.notna(r.get("prev_pts")) else "")
                    for _, r in _sel.iterrows())
                add(f'<p class="mini"><strong>{_label}:</strong> {_sb_esc(_names)}</p>')
            if not _new.empty:
                # Dumping all ~24 newcomers in raw uppercase told a coach nothing -- most are walk-ons who
                # never play. Rank them by the minutes they're actually getting this season and name only
                # the ones worth knowing, with their scoring; the rest collapse to a count.
                _new_names = [str(x) for x in _new["player"].astype(str)]
                # players_for, not _opp_stats -- that's built further down the document, after this section.
                _new_box = players_for("uww_opponent_prior_games_box_score", short)
                _cur = {}
                if not _new_box.empty and "name" in _new_box.columns:
                    _cur = {_norm(r["name"]): r for _, r in _new_box.iterrows()}
                _ranked = []
                for _nm in _new_names:
                    _row = _cur.get(_norm(_nm))
                    _min = pd.to_numeric(_row.get("MIN"), errors="coerce") if _row is not None else None
                    _pts = pd.to_numeric(_row.get("PTS"), errors="coerce") if _row is not None else None
                    _ranked.append((_nm, _min, _pts))
                # Anyone averaging real minutes, or scoring, is worth naming. A newcomer with no line at
                # all in their prior games hasn't played and stays in the count.
                _worth = [x for x in _ranked
                          if (pd.notna(x[1]) and x[1] >= 8) or (pd.notna(x[2]) and x[2] >= 4)]
                _worth.sort(key=lambda x: (-(x[1] if pd.notna(x[1]) else 0), -(x[2] if pd.notna(x[2]) else 0)))
                _worth = _worth[:6]
                if _worth:
                    _shown = ", ".join(
                        _sb_title(_nm) + (f" ({_pts:.1f} ppg)" if pd.notna(_pts) else "")
                        for _nm, _min, _pts in _worth)
                    _rest = len(_new_names) - len(_worth)
                    add(f'<p class="mini"><strong>New since then:</strong> {_sb_esc(_shown)}'
                        + (f' <span class="mini">+ {_rest} more who barely play</span>' if _rest > 0 else "")
                        + "</p>")
                else:
                    add(f'<p class="mini"><strong>New since then:</strong> {len(_new_names)} newcomers, '
                        f'none averaging real minutes yet.</p>')
        if not _h2h_box.empty or not _h2h_chg.empty:
            add("</div></div>")
        add("</div>")

    _SB_SOURCE_COLORS = {"Data-Driven": "#37474f", "Keys to Victory": "#4E2A84", "Team Strengths": "#c62828",
                         "Lineup Scouting": "#5d4037", "Coach Notes": "#00695c"}

    def source_badge(source):
        source = _sb_clean(source)
        if not source:
            return ""
        color = _SB_SOURCE_COLORS.get(source, "#666")
        return f' <span class="src" style="border-color:{color};color:{color};">{_sb_esc(source)}</span>'

    ktv = _sb_d("uww_ktv_keys")
    if not ktv.empty and "opponent" in ktv.columns:
        ktv = ktv[ktv["opponent"].astype(str) == str(short)]
        if "key_number" in ktv.columns:
            ktv = ktv.sort_values("key_number")
    if not ktv.empty:
        section("\U0001f511 KEYS TO VICTORY" + footnote(
            "Keys are built by the parser into uww_ktv_keys.csv and rendered identically by the app. The brief shows "
            "headlines only; each key's evidence (the numbers behind it) and reasoning (why it matters and what to do) "
            "are on the app's Keys to Victory tab -- click a key to open it there."))
        # One column per category, side by side (requested).
        _cats = [c for c in ("Offense", "Defense", "Personnel", "General")
                 if c in set(ktv["category"].astype(str))] or ["Keys"]
        _cats += [c for c in ktv["category"].astype(str).unique() if c not in _cats]
        add('<div class="card"><div class="grid%d">' % min(max(len(_cats), 1), 3))
        for _cat in _cats:
            _rows = ktv[ktv["category"].astype(str) == _cat] if _cat != "Keys" else ktv
            if _rows.empty:
                continue
            add(f'<div class="gcell"><div class="ktvcat">{_sb_esc(_cat)}</div>')
            for _, k in _rows.iterrows():
                _n = int(k["key_number"]) if pd.notna(k.get("key_number")) else ""
                # No per-key icon (requested): the number and the category heading already organise the
                # list, and an emoji on every line was decoration competing with the headline. The icon
                # column is still in uww_ktv_keys for the app.
                add(f'<div class="ktvline"><span class="n">{_n}.</span><span>'
                    f'{name_link(_sb_clean(k.get("headline")), page="upcoming", tab="keys", key=_n)}'
                    f'{source_badge(k.get("source"))}</span></div>')
            add("</div>")
        add("</div>")
        add('<p class="mini">' + app_link("Evidence and reasoning for every key", page="upcoming", tab="keys") + "</p>")
        add("</div>")

    def foul_list_block():
        _foul, _ = staff_rows("uww_late_game_foul_list")
        if _foul.empty:
            return
        add('<div class="gcell">')
        section("\u23f1\ufe0f LATE-GAME FOUL LIST" + footnote(
            "From their own free-throw shooting on film. The two calls need different evidence: FOUL him takes 8+ "
            "attempts and a rate at 62% or worse, because it's a deliberate act; DO NOT FOUL takes 8+ attempts "
            "at 75% or better, OR as few as 4 attempts when he hasn't missed (85%+) -- avoiding a shooter "
            "costs nothing if we're wrong. Rates are shrunk toward a 70% baseline before the call, so 6-for-6 "
            "doesn't read as a true 100% shooter; the makes-attempts pair shown is raw. Check who is actually "
            "on the floor before the call."))
        add('<div class="card"><table class="compact"><thead><tr><th style="text-align:left">Player</th>'
            '<th style="text-align:left">Call</th><th>FT%</th><th>FTM-FTA</th></tr></thead><tbody>')
        for _, _f in _foul.iterrows():
            _cls = {"Foul": "call-foul", "Do not foul": "call-nofoul"}.get(_sb_clean(_f.get("call")), "")
            # Games-played detail removed from this table (requested) -- thin samples are flagged on page 1.
            add(f'<tr><td class="nm">{_sb_esc(_f["player"])}</td>'
                f'<td class="{_cls}">{_sb_esc(_f["call"])}</td><td>{_sb_fmt(_f["ft_pct"])}</td>'
                f'<td>{int(_f["ftm"])}-{int(_f["fta"])}</td></tr>')
        add("</tbody></table></div></div>")

    def matchups_block():
        staff_table("uww_matchups", "\U0001f91d MATCHUPS",
                    [("their_player", "Their player"), ("our_defender", "Our defender"), ("backup", "Backup"),
                     ("note", "Note")],
                    "Placeholder pairings by minutes played; names are real, assignments are not a recommendation.",
                    note="Matchups: paired by minutes played, not by who has actually guarded whom. To collect: the "
                         "same on-ball-defender tag Matchup History needs would turn this into a recommendation "
                         "backed by real head-to-head possessions. Value: High.")

    # ---- team stats, four factors and the style comparisons, side by side above the Aurora section --------
    if team_parts:
        section("\U0001f4ca TEAM STATS")
        add('<div class="card">' + "".join(team_parts) + "</div>")
    # TEAMS LIKE ... sections removed from the brief (requested) -- the full panels stay in the app's Tools
    # tab. The record only reaches the brief as a Bottom Line sentence, and only when it's lopsided on
    # enough games at non-low confidence (BOTTOM_LINE_RULES "style_*").

    # =====================================================================================================
    # OPPONENT
    # =====================================================================================================
    page_break()
    divider(f"{OPP_UP}", "Scouting the opponent")

    # ---- recent results ------------------------------------------------------------------------------
    if not opp_before.empty:
        section(f"\U0001f4c5 {OPP_UP} RECENT RESULTS")
        add('<div class="card">')
        add('<div class="results">')
        for _, g in opp_before.tail(5).iterrows():
            outcome = _sb_clean(g.get("outcome")).upper()
            score = (f' {int(g["team_score"])}-{int(g["opponent_score"])}'
                     if pd.notna(g.get("team_score")) and pd.notna(g.get("opponent_score")) else "")
            home = _sb_clean(g.get("location")).lower() == "home"
            add(f'<span class="rs">{_sb_esc(g.get("game_date"))} {"vs" if home else "@"} '
                f'{_sb_esc(_sb_strip_mascot(g.get("vs_opponent")))} <b class="{"w" if outcome == "W" else "l"}">'
                f'{_sb_esc(outcome)}{score}</b></span>')
        add("</div></div>")

    _sm_unused = None
    if not _sm.empty and "opponent" in _sm.columns:
        _sm = _sm[_sm["opponent"].astype(str) == str(short)]


    # ---- personnel: starters page + bench page (#3) ---------------------------------------------------
    _no_min = _sb_d("uww_roster_no_minutes")
    _no_min_names = ""
    if not _no_min.empty and "opponent" in _no_min.columns:
        _no_min = _no_min[_no_min["opponent"].astype(str) == str(short)]
        if not _no_min.empty and int(_no_min.iloc[0].get("count") or 0):
            _no_min_names = _sb_clean(_no_min.iloc[0]["names"])
    _opp_stats = players_for("uww_opponent_prior_games_box_score", short)
    personnel_pages("Opponent", OPP_UP, _opp_stats, opp_card, extra_no_min=_no_min_names)

    # ---- lineups, offense, context --------------------------------------------------------------------
    page_break()
    lineups = _sb_d("uww_opp_lineup_season_box")
    if not lineups.empty and "lineup" in lineups.columns:
        section("\U0001f501 TOP LINEUPS & 3-MAN COMBOS" + footnote(
            "Top three five-man units and top three 3-man combos, each by minutes. GP, MIN and +/- are totals "
            "across the games on film; a combo's minutes are every stint with all three on the floor, whoever the "
            "other two were. \"Per 40\" is +/- scaled to 40 minutes -- combos vary so much in playing time that "
            "raw +/- mostly measures minutes. GP isn't shown for their combos: their lineup data has no game dates, "
            "so games played can't be counted exactly. The key shown "
            "is the first data-driven key for that unit. \"What they run\" is the sets tagged on possessions with "
            "that exact five on the floor, and \"Group\" is its position shape (3G-2B = three guards and two "
            "bigs, from roster positions). Every lineup's full strengths, weaknesses and keys are in the app."))
        add('<div class="card">')
        lineups_table(lineups, "Opponent")
        # Top three 3-man combos under the five-man units (requested). Captured first so the sub-heading
        # only prints when there is a table to put under it.
        _combo_html = capture(combos_table, "Opponent")
        if _combo_html:
            add('<div class="pc-sub">Top 3-man combos</div>')
            add("".join(_combo_html))
        add('<p class="mini">' + app_link("All lineups and combos with full reads", page="upcoming",
                                          tab="personnel", section="lineups") + "</p>")
        add("</div>")

    # =====================================================================================================
    # WHAT TO PLAY THEM IN -- everything here is only possible now that offense AND defense are tagged on
    # the same possession. It answers the questions a staff argues about BEFORE anyone watches film.
    #
    # In opponent_plays.csv the tagged defense is the defense the OPPONENT FACED, so these read as "how
    # Aurora's offense did against X", not "what Aurora plays on defense" (that's DEFENSE TYPE BY SITUATION
    # further down). Everything aggregates at defense FAMILY level (man / zone / press), not specific
    # coverage: with ~120 tagged possessions a specific set against a specific coverage is 2-3 clips, which
    # is noise. Clip counts are printed on every row so nobody reads a percentage off a sample of three.
    # =====================================================================================================
    _DEF_FAMILY_MIN = 6      # possessions before a family is reported at all
    _DEF_SPLIT_MIN = 4       # possessions before a per-player / per-series split is reported

    def _def_family(row):
        """man / zone / press, from the decoded defense fields. Press is its own family because it's a
        separate decision and a separate practice block -- a pressing man team lands in 'press'."""
        if bool(row.get("press_faced")):
            return "Press"
        d = _sb_clean(row.get("defense_faced"))
        if not d:
            return ""
        return "Zone" if "zone" in d.lower() else ("Man" if "man" in d.lower() else d)

    def _pf_clips(side_val, team_val=None):
        """Decoded, reviewable clips for one side with a defense read on them."""
        if _pcc_all.empty or "defense_faced" not in _pcc_all.columns:
            return pd.DataFrame()
        # OFFENSIVE possessions only. On a mixed file a defensive clip's defense tag is what this team
        # PLAYED, which would silently invert every row in this section.
        d = _pcc_all[(_pcc_all["side"] == side_val) & (_pcc_all["decode_quality"] != "Needs review")].copy()
        if "possession_side" in d.columns:
            d = d[d["possession_side"].astype(str) != "Defense"]
        if team_val is not None and "offense_team" in d.columns:
            d = d[d["offense_team"].astype(str) == str(team_val)]
        if d.empty:
            return d
        d["_fam"] = d.apply(_def_family, axis=1)
        d = d[d["_fam"].astype(str) != ""]
        if d.empty:
            return d
        _res = d["result"].astype(str).str.lower() if "result" in d.columns else pd.Series("", index=d.index)
        d["_pts"] = pd.to_numeric(d.get("points"), errors="coerce")
        d["_to"] = _res.str.contains("turnover|violation|kicked", regex=True)
        d["_fga"] = _res.str.startswith(("make", "miss")) & _res.str.contains("2|3", regex=True)
        d["_fgm"] = _res.str.startswith("make") & _res.str.contains("2|3", regex=True)
        return d

    def _ppp_of(grp):
        _known = grp["_pts"].notna().sum()
        return (grp["_pts"].sum() / _known) if _known else None

    def _ppp_cls(v, base):
        if v is None or base is None or pd.isna(v) or pd.isna(base):
            return ""
        return "ppp-good" if v >= base + 0.15 else ("ppp-bad" if v <= base - 0.15 else "")

    _pf_opp = _pf_clips("Opponent", short)
    _pf_uww = _pf_clips("UWW")
    _pf_any = (not _pf_opp.empty) or (not _pf_uww.empty)

    if _pf_any:
        page_break()
        section(f"\U0001f3af WHAT TO PLAY {OPP_UP} IN" + footnote(
            "Every row here is the defense the opponent's offense FACED on a tagged possession, so it reads "
            "as how they did against that look -- not what they run on defense themselves. Man, Zone and "
            "Press are families, not specific coverages: at this sample size a named coverage is a handful "
            "of clips. PPP is points per tagged possession. Treat these as things to CONFIRM on film, not "
            "as settled reads -- the clip count on each row is how much weight it carries."))
        add('<div class="card">')

        # ---- 1+2. Each defense, AND whether pressing is worth it -- one table ----------------------
        # These used to be two tables, and the second ("Do they handle pressure") repeated the Press row
        # of the first (requested: combine). Press is already a row here; the only thing the second table
        # added was the turnover-rate COMPARISON against not being pressed, so that is now a sentence under
        # the table instead of its own grid.
        #
        # Their columns and ours are grouped under two labelled header bands (requested: make it clear which
        # columns are OUR defense) -- a bare "We allow" sitting next to "PPP" was easy to read as theirs.
        if not _pf_opp.empty:
            _base = _ppp_of(_pf_opp)
            _ours = pd.DataFrame()
            if not _pcc_all.empty and "defense_played" in _pcc_all.columns:
                _ours = _pcc_all[(_pcc_all["side"] == "UWW")
                                 & (_pcc_all["decode_quality"] != "Needs review")
                                 & _pcc_all["defense_played"].notna()].copy()
                if not _ours.empty:
                    _ours["_pts"] = pd.to_numeric(_ours.get("points"), errors="coerce")
                    _ores = _ours["result"].astype(str).str.lower() if "result" in _ours.columns \
                        else pd.Series("", index=_ours.index)
                    _ours["_to"] = _ores.str.contains("turnover|violation|kicked", regex=True)
                    _ours["_fam"] = _ours.apply(
                        lambda r: "Press" if bool(r.get("press_played")) else
                        ("Zone" if "zone" in str(r.get("defense_played") or "").lower() else
                         ("Man" if "man" in str(r.get("defense_played") or "").lower() else "")), axis=1)
                    _ours = _ours[_ours["_fam"] != ""]
            _have_ours = not _ours.empty

            add('<div class="tier-h">How they score against each defense</div>')
            add('<table class="compact dfam"><thead>'
                '<tr class="dfam-band"><th></th>'
                f'<th colspan="4" class="band-them">{_sb_esc(OPP_UP)} OFFENSE vs this defense</th>'
                + ('<th colspan="3" class="band-us">OUR DEFENSE when we play it</th>' if _have_ours else "")
                + '<th></th></tr>'
                '<tr><th style="text-align:left">Defense</th>'
                '<th class="band-them">Poss</th><th class="band-them">PPP</th>'
                '<th class="band-them">FG%</th><th class="band-them">TO%</th>'
                + ('<th class="band-us">Poss</th><th class="band-us">PPP allowed</th>'
                   '<th class="band-us">TO% forced</th>' if _have_ours else "")
                + '<th style="text-align:left">What they went to most</th>'
                "</tr></thead><tbody>")
            _fam_rows = []
            for _fam, _g in _pf_opp.groupby("_fam"):
                if len(_g) < _DEF_FAMILY_MIN:
                    continue
                _fam_rows.append((_fam, _g, _ppp_of(_g)))
            # Fixed order so Press is always last and reads as "and when you pressure them".
            _order = {"Man": 0, "Zone": 1, "Press": 2}
            for _fam, _g, _p in sorted(_fam_rows, key=lambda x: (_order.get(x[0], 9), -len(x[1]))):
                _fga, _fgm = int(_g["_fga"].sum()), int(_g["_fgm"].sum())
                _top = ""
                if "play_series" in _g.columns and _g["play_series"].notna().any():
                    _tc = _g["play_series"].dropna().astype(str).value_counts()
                    if len(_tc):
                        _top = f"{_tc.index[0]} ({int(_tc.iloc[0])}x)"
                _ours_cells = ""
                if _have_ours:
                    _og = _ours[_ours["_fam"] == _fam]
                    if len(_og):
                        _op = _ppp_of(_og)
                        _ours_cells = (f'<td class="band-us">{len(_og)}</td>'
                                       f'<td class="band-us">{_sb_fmt(_op, 2) if _op is not None else "--"}</td>'
                                       f'<td class="band-us">{_sb_fmt(100 * _og["_to"].sum() / len(_og), 1)}</td>')
                    else:
                        _ours_cells = '<td class="band-us" colspan="3"><span class="mini">not run yet</span></td>'
                add(f'<tr class="serrow"><td class="wrap">{_sb_esc(_fam)}</td>'
                    f'<td class="band-them">{len(_g)}</td>'
                    f'<td class="band-them {_ppp_cls(_p, _base)}">{_sb_fmt(_p, 2)}</td>'
                    f'<td class="band-them">{_sb_fmt(100 * _fgm / _fga, 1) if _fga else "--"}</td>'
                    f'<td class="band-them">{_sb_fmt(100 * _g["_to"].sum() / len(_g), 1)}</td>'
                    + _ours_cells
                    + f'<td class="wrap mini">{_sb_esc(_top)}</td></tr>')
            if not _fam_rows:
                add(f'<tr><td colspan="{9 if _have_ours else 6}" class="mini">No defense family has reached '
                    f'{_DEF_FAMILY_MIN} tagged possessions yet.</td></tr>')
            add("</tbody></table>")

            _notes = []
            if _fam_rows:
                _best = min(_fam_rows, key=lambda x: (x[2] if x[2] is not None else 9))
                _worst = max(_fam_rows, key=lambda x: (x[2] if x[2] is not None else -9))
                if _best[0] != _worst[0] and _best[2] is not None and _worst[2] is not None:
                    _notes.append(f"They struggle most against <strong>{_sb_esc(_best[0])}</strong> "
                                  f"({_sb_fmt(_best[2], 2)} PPP on {len(_best[1])}) and hurt you most against "
                                  f"<strong>{_sb_esc(_worst[0])}</strong> ({_sb_fmt(_worst[2], 2)} PPP on "
                                  f"{len(_worst[1])}).")
            # The pressure verdict -- what the separate table used to say, in one line.
            if "press_faced" in _pf_opp.columns:
                _pressed = _pf_opp[_pf_opp["press_faced"].astype(bool)]
                _unpressed = _pf_opp[~_pf_opp["press_faced"].astype(bool)]
                if len(_pressed) >= _DEF_FAMILY_MIN and len(_unpressed) >= _DEF_FAMILY_MIN:
                    _to_p = 100 * _pressed["_to"].sum() / len(_pressed)
                    _to_u = 100 * _unpressed["_to"].sum() / len(_unpressed)
                    _forms = (_pressed["press_formation_faced"].dropna().astype(str).value_counts()
                              if "press_formation_faced" in _pressed.columns else pd.Series(dtype=int))
                    _notes.append(
                        f"<strong>Pressure</strong> moves their turnover rate {_to_p - _to_u:+.1f} points "
                        f"({_to_u:.1f}% unpressed \u2192 {_to_p:.1f}% pressed) \u2014 "
                        + ("worth pressing." if _to_p - _to_u >= 5 else "not enough to be worth the risk.")
                        + (f" Presses seen: {_sb_esc(', '.join(_forms.index[:3]))}." if len(_forms) else ""))
            for _n in _notes:
                add(f'<p class="strip-line">{_n}</p>')

        # ---- 3. Counters: what a coverage change makes them do ------------------------------------
        if not _pf_opp.empty and "coverage_faced" in _pf_opp.columns:
            _cov = _pf_opp[_pf_opp["coverage_faced"].astype(str).str.strip() != ""]
            if not _cov.empty:
                _cov_rows = []
                for _cname, _g in _cov.groupby(_cov["coverage_faced"].astype(str)):
                    if len(_g) < _DEF_SPLIT_MIN:
                        continue
                    _went = ""
                    if "play_call" in _g.columns and _g["play_call"].notna().any():
                        _wc = _g["play_call"].dropna().astype(str)
                        _wc = _wc[~_wc.str.contains("unspecified", na=False)].value_counts()
                        if len(_wc):
                            _went = ", ".join(f"{n} ({int(c)}x)" for n, c in _wc.head(2).items())
                    _cov_rows.append((_cname, _g, _ppp_of(_g), _went))
                if _cov_rows:
                    _cbase = _ppp_of(_cov)
                    add('<div class="tier-h">When you change the coverage, what do they go to</div>')
                    add('<table class="compact"><thead><tr><th style="text-align:left">Coverage they saw</th>'
                        '<th>Poss</th><th>PPP</th><th style="text-align:left">What they called</th>'
                        "</tr></thead><tbody>")
                    for _cname, _g, _p, _went in sorted(_cov_rows, key=lambda x: -len(x[1])):
                        add(f'<tr class="serrow"><td class="wrap">{_sb_esc(_cname)}</td><td>{len(_g)}</td>'
                            f'<td class="{_ppp_cls(_p, _cbase)}">{_sb_fmt(_p, 2)}</td>'
                            f'<td class="wrap mini">{_sb_esc(_went)}</td></tr>')
                    add("</tbody></table>")

        # ---- 4. Personnel x defense: who the zone-buster is ---------------------------------------
        if not _pf_opp.empty and "player" in _pf_opp.columns:
            _pl = _pf_opp[_pf_opp["player"].astype(str).str.strip() != ""]
            _rows4 = []
            for _who, _g in _pl.groupby(_pl["player"].astype(str)):
                if len(_g) < _DEF_SPLIT_MIN * 2:
                    continue
                _by = {}
                for _fam, _fg in _g.groupby("_fam"):
                    if len(_fg) >= _DEF_SPLIT_MIN:
                        _by[_fam] = (_ppp_of(_fg), len(_fg))
                if len(_by) < 2:
                    continue
                _hi = max(_by.items(), key=lambda kv: (kv[1][0] if kv[1][0] is not None else -9))
                _lo = min(_by.items(), key=lambda kv: (kv[1][0] if kv[1][0] is not None else 9))
                if _hi[0] == _lo[0] or _hi[1][0] is None or _lo[1][0] is None:
                    continue
                _rows4.append((_who, _hi, _lo, _hi[1][0] - _lo[1][0]))
            if _rows4:
                add('<div class="tier-h">Who changes by defense' + footnote(
                    "Only players with enough possessions against two different defense families are listed, "
                    "and only where the gap is real. This is who to help off and who to stay attached to "
                    "when you change looks.") + "</div>")
                add('<table class="compact"><thead><tr><th style="text-align:left">Player</th>'
                    '<th style="text-align:left">Best against</th><th style="text-align:left">Worst against</th>'
                    "<th>Gap</th></tr></thead><tbody>")
                for _who, _hi, _lo, _gap in sorted(_rows4, key=lambda x: -x[3])[:6]:
                    add(f'<tr class="serrow"><td class="wrap">{_sb_esc(_sb_title(_who))}</td>'
                        f'<td class="wrap">{_sb_esc(_hi[0])} \u2014 {_sb_fmt(_hi[1][0], 2)} ({_hi[1][1]})</td>'
                        f'<td class="wrap">{_sb_esc(_lo[0])} \u2014 {_sb_fmt(_lo[1][0], 2)} ({_lo[1][1]})</td>'
                        f'<td>{_gap:+.2f}</td></tr>')
                add("</tbody></table>")

        # ---- 5. Confirm on film -------------------------------------------------------------------
        # The brief's job before film is to hand over HYPOTHESES to confirm, not conclusions. Ordered by
        # how much the claim would change the plan, with the clip count behind each one stated.
        _checks = []
        if not _pf_opp.empty:
            _fams = {f: g for f, g in _pf_opp.groupby("_fam") if len(g) >= _DEF_FAMILY_MIN}
            if len(_fams) >= 2:
                _ranked = sorted(_fams.items(), key=lambda kv: (_ppp_of(kv[1]) or 9))
                _b, _w = _ranked[0], _ranked[-1]
                _checks.append((abs((_ppp_of(_w[1]) or 0) - (_ppp_of(_b[1]) or 0)),
                                f"They score {_sb_fmt(_ppp_of(_b[1]), 2)} PPP against {_b[0]} vs "
                                f"{_sb_fmt(_ppp_of(_w[1]), 2)} against {_w[0]} \u2014 watch whether that's the "
                                f"defense or just shot-making.", len(_b[1]) + len(_w[1])))
            _pr = _pf_opp[_pf_opp["press_faced"].astype(bool)] if "press_faced" in _pf_opp.columns else pd.DataFrame()
            if len(_pr) >= _DEF_FAMILY_MIN:
                _rest = _pf_opp[~_pf_opp["press_faced"].astype(bool)]
                if len(_rest):
                    _d = 100 * _pr["_to"].sum() / len(_pr) - 100 * _rest["_to"].sum() / len(_rest)
                    _checks.append((abs(_d) / 10,
                                    f"Their turnover rate moves {_d:+.1f} points under pressure \u2014 check "
                                    f"who is actually bringing it up and whether they have a second handler.",
                                    len(_pr)))
        if not _pf_uww.empty:
            _ufams = {f: g for f, g in _pf_uww.groupby("_fam") if len(g) >= _DEF_FAMILY_MIN}
            if len(_ufams) >= 2:
                _ur = sorted(_ufams.items(), key=lambda kv: -(_ppp_of(kv[1]) or 0))
                _checks.append((abs((_ppp_of(_ur[0][1]) or 0) - (_ppp_of(_ur[-1][1]) or 0)),
                                f"Our own offense scores best against {_ur[0][0]} "
                                f"({_sb_fmt(_ppp_of(_ur[0][1]), 2)} PPP) \u2014 confirm they'll actually show "
                                f"it before building around it.", len(_ur[0][1])))
        if _checks:
            add('<div class="tier-h">Confirm these on film</div>')
            add('<ul class="bl">')
            for _w8, _text, _n in sorted(_checks, key=lambda x: -x[0])[:5]:
                add(f'<li>{_sb_esc(_text)} <span class="mini">({_n} tagged possessions)</span></li>')
            add("</ul>")
        add("</div>")




    # =====================================================================================================
    # HOW THEY RUN OFFENSE -- the quadrant view, built for the defensive side of the ball. Replaced an
    # other side of the ball. The axes are the same (how often x how well), but the QUESTION flips: on our
    # own sets the answer is "call it more / stop calling it", which is meaningless for an opponent we
    # don't coach. Here the same two axes answer "what do we have to take away, and what can we live with",
    # so the quadrant names, the ordering and the per-row detail are all defensive. The scary cell is
    # low-usage / high-efficiency -- the thing they don't run often enough to have shown up on film, which
    # is exactly what beats an under-prepared defense.
    # =====================================================================================================
    def offense_section_test2(side, title):
        sm = _offense_rows(side)
        if sm.empty:
            return
        calls = sm[(sm["level"] == "Play call")
                   & ~sm["name"].astype(str).str.contains("unspecified", na=False)].copy()
        if calls.empty:
            return
        calls["_uses"] = pd.to_numeric(calls["uses"], errors="coerce").fillna(0)
        calls["_ppp"] = pd.to_numeric(calls["ppp"], errors="coerce")
        _QSHOW = 3     # plays listed per quadrant; the rest collapse to "+ N more"
        _QMIN = 2      # lower than our own side: a set they ran twice and scored on is still a warning
        _rated = calls[(calls["_uses"] >= _QMIN) & calls["_ppp"].notna()]
        if _rated.empty:
            return
        team_ppp = calls["team_ppp"].dropna().iloc[0] if calls["team_ppp"].notna().any() else _rated["_ppp"].mean()
        _use_cut = _rated["_uses"].median()
        _meta_df, _meta_sample = staff_rows("uww_opp_sets")
        _meta = {_norm(r["set_name"]): r for _, r in _meta_df.iterrows()} if not _meta_df.empty else {}
        _film = _film_clip_lookup() if side == "Opponent" else {}

        section(title + footnote(
            "Same tagged possessions as the sections above, sorted by how often they run a set against how "
            f"well it scores. The usage line is the median ({_use_cut:.0f} uses) and the efficiency line is "
            f"their own average ({team_ppp:.2f} PPP), so this is relative to them, not to the league. Sets "
            f"with fewer than {_QMIN} tagged possessions are left out. \"Haven't seen it enough\" is the "
            "cell to read twice: those are efficient looks with a thin sample, which is what beats a "
            "defense that only prepared for the obvious."))
        add('<div class="card">')

        _quads = [
            ("Take this away", "They run it a lot and it works", "q-bad",
             lambda r: r["_uses"] >= _use_cut and r["_ppp"] >= team_ppp, "ppp"),
            ("Haven't seen it enough", "Efficient, but thin sample \u2014 don't get surprised", "q-up",
             lambda r: r["_uses"] < _use_cut and r["_ppp"] >= team_ppp, "ppp"),
            # Sorted by USAGE, not PPP: the point of this cell is "they keep going to it and it doesn't
            # work", so their most-run sets lead. Sorting by PPP here put the least-bad sets first and cut
            # their two most-run sets (4-1 Ball Screen, OK State Pat Miller) off the bottom.
            ("Make them keep running it", "They go to it often and it doesn't score", "q-good",
             lambda r: r["_uses"] >= _use_cut and r["_ppp"] < team_ppp, "uses"),
            ("Live with it", "Rare and ineffective \u2014 don't spend practice on it", "q-dim",
             lambda r: r["_uses"] < _use_cut and r["_ppp"] < team_ppp, "ppp"),
        ]
        add('<div class="quad">')
        for _qname, _qsub, _qcls, _test, _sort in _quads:
            # Most dangerous first by default; "Make them keep running it" leads with what they run most.
            _order = ["_uses", "_ppp"] if _sort == "uses" else ["_ppp", "_uses"]
            _rows = _rated[_rated.apply(_test, axis=1)].sort_values(_order, ascending=[False, False])
            add(f'<div class="quad-cell {_qcls}">'
                f'<div class="quad-h">{_sb_esc(_qname)}<span class="quad-n">{len(_rows)}</span></div>'
                f'<div class="quad-sub">{_sb_esc(_qsub)}</div>')
            if _rows.empty:
                add('<div class="quad-empty">Nothing here yet.</div>')
            for _, r in _rows.head(_QSHOW).iterrows():
                # Defensive detail: who finishes it, where it happens, and our coverage call -- the three
                # things a defender needs. (Our own version showed "best vs defense", which is an offensive
                # read and has no meaning for a team we don't call plays for.)
                _who = _sb_clean(r.get("top_player"))
                _where = _sb_clean(r.get("top_location"))
                _m = _meta.get(_norm(r["name"]))
                _cov = _sb_clean(_m.get("our_call")) if _m is not None else ""
                _cn = _film.get(_norm(r["name"]))
                _meta_bits = " &middot; ".join(x for x in (_who, _where) if x)
                add(f'<div class="quad-row"><span class="quad-set">{_sb_esc(_sb_clean(r["name"]))}</span>'
                    + (f' <span class="mini">\U0001f3ac {_cn}</span>' if _cn else "")
                    + f'<span class="quad-num">{r["_ppp"]:.2f}<span class="mini"> / {int(r["_uses"])}x</span></span>'
                    + (f'<div class="quad-meta">{_sb_esc(_meta_bits)}</div>' if _meta_bits else "")
                    + (f'<div class="quad-cov">{_sb_esc(_cov)}</div>' if _cov else "")
                    + "</div>")
            if len(_rows) > _QSHOW:
                add(f'<div class="quad-more">+ {len(_rows) - _QSHOW} more</div>')
            add("</div>")
        add("</div>")
        if _meta_sample:
            add('<p class="mini">Coverage calls are a generated placeholder until '
                'staff_inputs/opp_sets.csv exists.</p>')
        add('<p class="mini">' + app_link("Every set, clip and spot", page="upcoming", tab="game_plan",
                                          section=f"play_calls_{side.lower()}") + "</p>")
        add("</div>")

    offense_section_test2("Opponent", f"\U0001f3ac HOW {OPP_UP} RUNS OFFENSE")

    # ---- sections that need tagging or staff input (kept, laid out two-up) ---------------------------------
    page_break()
    section("\U0001f52c NEEDS TAGGING OR STAFF INPUT" + footnote(
        "Red boxes are SAMPLE DATA generated so the layout can be reviewed. Each section's own ⓘ note says what to tag "
        "and how much it would add. Controlled vocabulary for play titles: every title decoded correctly only because "
        "the decoder absorbs spelling variance (\"Blob-Box-Curl\" vs \"BLOB- Box- Curl\"); a dropdown or autocomplete "
        "built from the playbook would remove that fragility. Value: process, not new data."))
    add('<div class="flow2">')
    # THEIR DEFENSE & HOW WE ATTACK IT, DEFENSE TYPE BY SITUATION and BALL SCREEN COVERAGE moved out of
    # this block: they are real now (defense tagging) and all three are combined into "What they will play
    # against us" at the top of HOW WE RUN OFFENSE. The tables are still exported for the app.
    staff_table("uww_help_rotation_tendencies", "\U0001f504 HELP &amp; ROTATION",
                [("trigger", "Trigger"), ("who_rotates", "Who rotates"), ("tendency", "Tendency")],
                "Off-ball rotations aren't tagged today.",
                note="Help & rotation: on defensive clips note which off-ball defender rotated and whether the "
                     "closeout was on time. Value: High, but a judgment call -- add once coverage tagging is routine.")
    # MATCHUP HISTORY removed from the brief (requested) -- still exported for the app.
    staff_table("uww_play_counters", "\U0001f504 WHEN WE TAKE AWAY THEIR SET",
                [("set", "Set"), ("their_counter", "Their counter"), ("our_adjustment", "Our adjustment")],
                "A title captures one action, not a sequence.",
                note="Counters: tag a second clip when a denied action leads into another, linked to the first clip's "
                     "number. Value: High, but hardest to tag consistently -- a season-two addition.")
    if ROSTER_LAYOUT != "stacked":   # stacked roster carries a real per-player shot line instead
        staff_table("uww_opp_shot_zones", "\U0001f3af SHOT LOCATIONS \u2014 TOP FIVE",
                [("player", "Player"), ("rim", "Rim"), ("paint_non_rim", "Paint"), ("midrange", "Mid"),
                 ("corner_3", "C3"), ("above_break_3", "AB3")],
                "Three-point share is real; the zone split is a placeholder.",
                note="Shot locations: tag a zone (rim / paint / mid / corner 3 / above-break 3) on every shot. The decoder "
                     "reads some spots from titles (\"LW 3\") but never for twos. Value: High.")
    staff_table("uww_opp_tendencies", "\U0001f3c3 TRANSITION, GLASS &amp; BENCH",
                [("item", "Area"), ("detail", "What they do")],
                "Placeholders (staff_inputs/opp_tendencies.csv).",
                note="Transition/glass/bench: tag \"Transition\" as its own situation (Early clock is only a rough "
                     "stand-in); glass crashers need a rebounder tag; timeouts and officials aren't in the play-by-play. "
                     "Value: Medium.")
    # REBOUND TENDENCIES removed as a section (requested). The full table stays in the app; the brief
    # carries rebounding only when it's lopsided -- a Keys to Victory key (built in the KTV cell), a Bottom
    # Line sentence, and a roster read on the player doing the crashing. Rules: REBOUND_RULES.
    staff_table("uww_shot_quality_by_contest", "\U0001f3af SHOT QUALITY BY CONTEST",
                [("contest_level", "Contest"), ("freq_pct", "%"), ("fg_pct", "FG%")],
                "Contest level isn't tagged beyond Synergy's guarded/open.",
                note="Shot quality: tag wide open / open / contested / tightly contested on every shot. Value: Medium.")
    staff_table("uww_double_team_tendencies", "\U0001f465 DOUBLE-TEAMS",
                [("trigger", "Trigger"), ("from_where", "From"), ("escape_read", "Escape")],
                "Double teams aren't tagged.",
                note="Double teams: tag when one occurs, from where, and the read. Value: Medium, narrow scope.")
    staff_table("uww_offball_screen_navigation", "\U0001f504 OFF-BALL SCREEN NAVIGATION",
                [("screen_type", "Screen"), ("technique", "Technique"), ("freq_pct", "%")],
                "Off-ball screen technique isn't tagged.",
                note="Off-ball screens: tag over/under/switch/fight-through on down screens, pin downs and staggers. "
                     "Value: Medium-to-low; pairs with Ball Screen Coverage.")
    add("</div>")

    # =====================================================================================================
    # UW-WHITEWATER -- mirrors the opponent section
    # =====================================================================================================
    _uww_stats = players_for("uww_pbp_box_score", _SB_UWW)
    personnel_pages("UWW", "UW-WHITEWATER", _uww_stats, uww_card, break_first=True,
                    lead=lambda: divider("UW-WHITEWATER", "Our own personnel, offense and week"))

    page_break()
    matchups_block()  # our defenders against their players -- our side of the game plan (requested)
    _uww_lu = _sb_d("uww_uww_lineup_season")
    if not _uww_lu.empty:
        section("\U0001f501 OUR TOP LINEUPS & 3-MAN COMBOS" + footnote(
            "Our top three five-man units and top three 3-man combos by minutes this season, from the "
            "play-by-play's own lineup stints. +/- is the scoring margin while that unit was on the floor; a "
            "combo's minutes are every stint with all three on the floor. \"Per 40\" is +/- scaled to 40 "
            "minutes."))
        add('<div class="card">')
        lineups_table(_uww_lu, "UWW", note_rows=False)
        _combo_html = capture(combos_table, "UWW")
        if _combo_html:
            add('<div class="pc-sub">Top 3-man combos</div>')
            add("".join(_combo_html))
        add('<p class="mini">' + app_link("All of our lineups and combos", page="upcoming", tab="stats",
                                          section="uww_lineups") + "</p></div>")

    # =====================================================================================================
    # HOW WE RUN OFFENSE -- the same quadrant idea as the opponent's section above, but the question flips:
    # table and the bar layout used for the opponent. Scouting THEM is about recognition ("what are they
    # about to run"); scouting OURSELVES is a different question -- what should we call more, and what
    # should we stop calling. So this one drops the situation-by-situation structure entirely and sorts
    # every set into a usage x efficiency quadrant, which turns the table into a decision instead of a
    # reference. Cut whichever of the three loses.
    # =====================================================================================================
    def offense_section_test_uww(side, title):
        sm = _offense_rows(side)
        if sm.empty:
            return
        calls = sm[(sm["level"] == "Play call")
                   & ~sm["name"].astype(str).str.contains("unspecified", na=False)].copy()
        if calls.empty:
            return
        calls["_uses"] = pd.to_numeric(calls["uses"], errors="coerce").fillna(0)
        calls["_ppp"] = pd.to_numeric(calls["ppp"], errors="coerce")
        # A set needs enough reps for its PPP to mean anything before it can be told to call it more or less.
        _QMIN = 3
        _QSHOW = 3     # sets listed per quadrant; the rest collapse to "+ N more"
        _rated = calls[(calls["_uses"] >= _QMIN) & calls["_ppp"].notna()]
        if _rated.empty:
            return
        team_ppp = calls["team_ppp"].dropna().iloc[0] if calls["team_ppp"].notna().any() else _rated["_ppp"].mean()
        # Split on the median so the quadrants always have something in them -- a fixed usage cutoff would
        # empty out early in the season and overflow late.
        _use_cut = _rated["_uses"].median()

        r_uses_cut = _use_cut

        def _q_context(set_name):
            """The four things a coach asked for on every quadrant row, each computed from that set's own
            clips: who finishes it (and how often, so "Jake Quast" can't be misread as a recommendation),
            which personnel group and clock window it lives in, and WHY it landed in this quadrant --
            shooting, turnovers, or simply how rarely we call it."""
            blank = {"who": "", "lineup": "", "clock": "", "driver": "", "unit": ""}
            if _pcc_all.empty or "play_call" not in _pcc_all.columns:
                return blank
            d = _pcc_all[(_pcc_all["side"] == side)
                        & (_pcc_all["play_call"].astype(str) == str(set_name))
                        & (_pcc_all["decode_quality"] != "Needs review")].copy()
            if "possession_side" in d.columns:
                d = d[d["possession_side"].astype(str) != "Defense"]
            if d.empty:
                return blank
            n = len(d)
            d["_pts"] = pd.to_numeric(d.get("points"), errors="coerce")
            _res = d["result"].astype(str).str.lower() if "result" in d.columns else pd.Series("", index=d.index)
            d["_to"] = _res.str.contains("turnover|violation|kicked", regex=True)
            d["_fga"] = _res.str.startswith(("make", "miss")) & _res.str.contains("2|3", regex=True)
            d["_fgm"] = _res.str.startswith("make") & _res.str.contains("2|3", regex=True)
            out = dict(blank)

            # WHO -- stated as observed usage, never as a recommendation.
            if "player" in d.columns and d["player"].notna().any():
                _vc = d["player"].dropna().astype(str).value_counts()
                if len(_vc):
                    out["who"] = f"Finished by {_sb_title(_vc.index[0])} on {int(_vc.iloc[0])} of {n}"

            def _split_best(col, label_fmt):
                """Best-scoring bucket of a split, only when there is something to compare it against."""
                if col not in d.columns or d[col].isna().all():
                    return ""
                g = d[d[col].notna() & (d[col].astype(str).str.strip() != "")]
                if g.empty:
                    return ""
                agg = g.groupby(g[col].astype(str)).agg(
                    n=("_pts", "size"), pts=("_pts", "sum"), known=("_pts", lambda x: x.notna().sum()))
                agg = agg[agg["n"] >= 2]
                if agg.empty:
                    return ""
                agg["ppp"] = agg["pts"] / agg["known"].replace(0, pd.NA)
                agg = agg.dropna(subset=["ppp"]).sort_values("ppp", ascending=False)
                if agg.empty:
                    return ""
                _top = agg.iloc[0]
                # With only one bucket there is no contrast to report -- say where it runs, not "best in".
                if len(agg) == 1:
                    return label_fmt.format(kind="All", name=agg.index[0], ppp=_top["ppp"], n=int(_top["n"]))
                _bot = agg.iloc[-1]
                return (label_fmt.format(kind="Best", name=agg.index[0], ppp=_top["ppp"], n=int(_top["n"]))
                        + f" &middot; worst {_bot.name} {_bot['ppp']:.2f}")

            out["lineup"] = _split_best("personnel_grouping_type",
                                        "{kind} in {name} \u2014 {ppp:.2f} ({n})")
            out["clock"] = _split_best("shot_clock_situation",
                                       "{kind} on {name} \u2014 {ppp:.2f} ({n})")

            # UNIT -- the five- or three-man group it runs with most. on_court_lineup is a full five;
            # trimmed to three names so it fits a quadrant cell without wrapping to four lines.
            if "on_court_lineup" in d.columns and d["on_court_lineup"].notna().any():
                _lu = d["on_court_lineup"].dropna().astype(str).value_counts()
                if len(_lu) and int(_lu.iloc[0]) >= 2:
                    _names = [x.strip() for x in re.split(r"[,/|]", _lu.index[0]) if x.strip()]
                    _trio = ", ".join(_sb_title(x) for x in _names[:3])
                    out["unit"] = (f"Most with {_trio}"
                                   + (f" +{len(_names) - 3}" if len(_names) > 3 else "")
                                   + f" ({int(_lu.iloc[0])}x)")

            # DRIVER -- why this set sits in this quadrant, stated against our own team averages.
            _fga, _fgm = int(d["_fga"].sum()), int(d["_fgm"].sum())
            _to_rate = 100 * d["_to"].sum() / n
            _bits = []
            if _fga:
                _bits.append(f"{100 * _fgm / _fga:.0f}% FG on {_fga}")
            if _to_rate >= 20:
                _bits.append(f"turned over {_to_rate:.0f}%")
            if r_uses_cut is not None and n < r_uses_cut:
                _bits.append("small sample")
            out["driver"] = " &middot; ".join(_bits)
            return out

        def _q_best_defense(set_name):
            """Highest-PPP defense this set scored against. Same rule as the main section's "Best vs.
            defense" column."""
            if _pcc_all.empty or "defense_faced" not in _pcc_all.columns or "play_call" not in _pcc_all.columns:
                return ""
            d = _pcc_all[(_pcc_all["side"] == side)
                        & (_pcc_all["play_call"].astype(str) == str(set_name))
                        & _pcc_all["defense_faced"].notna()
                        & (_pcc_all["decode_quality"] != "Needs review")].copy()
            if "possession_side" in d.columns:
                d = d[d["possession_side"].astype(str) != "Defense"]
            if d.empty:
                return ""
            d["_pts"] = pd.to_numeric(d.get("points"), errors="coerce")
            by = d.groupby("defense_faced").agg(uses=("defense_faced", "size"), pts=("_pts", "sum"),
                                                known=("_pts", lambda s: s.notna().sum()))
            by["ppp"] = by["pts"] / by["known"].replace(0, pd.NA)
            if len(by) == 1:
                return _sb_clean(by.index[0])
            by = by[by["uses"] >= 2].dropna(subset=["ppp"])
            return _sb_clean(by.sort_values("ppp", ascending=False).index[0]) if not by.empty else ""

        section(title + footnote(
            "Same tagged possessions as HOW WE RUN OFFENSE above, sorted by how often we call a set against "
            f"how well it scores. The usage line is the median ({_use_cut:.0f} uses) and the efficiency line "
            f"is our own average ({team_ppp:.2f} PPP), so this is relative to us, not to the league. Sets "
            f"with fewer than {_QMIN} tagged possessions are left out -- not enough reps to judge. Read it "
            "as a call sheet, not a verdict: a low-efficiency set may still be there to set up something "
            "else."))
        add('<div class="card">')

        _quads = [
            ("Keep calling it", "Called often, scores well", "q-good",
             lambda r: r["_uses"] >= _use_cut and r["_ppp"] >= team_ppp),
            ("Call it more", "Scores well, we barely call it", "q-up",
             lambda r: r["_uses"] < _use_cut and r["_ppp"] >= team_ppp),
            ("Fix it or cut it", "Called often, does not score", "q-bad",
             lambda r: r["_uses"] >= _use_cut and r["_ppp"] < team_ppp),
            ("Shelve it", "Rarely called, does not score", "q-dim",
             lambda r: r["_uses"] < _use_cut and r["_ppp"] < team_ppp),
        ]
        # ---- What they will play against us -- replaces three separate sections --------------------
        # THEIR DEFENSE & HOW WE ATTACK IT, DEFENSE TYPE BY SITUATION and BALL SCREEN COVERAGE all
        # described the same thing -- the opponent's own defense -- three ways, and each partly restated
        # the others (requested: combine). It lives HERE rather than in WHAT TO PLAY THEM IN because that
        # section is about THEIR OFFENSE; this is the other side of the ball, and it is exactly what our
        # sets below have to beat. Computed straight from the clips (defense_PLAYED on the opponent's
        # defensive possessions) so the base-defense, by-situation and coverage numbers can't disagree.
        _td = pd.DataFrame()
        if not _pcc_all.empty and "defense_played" in _pcc_all.columns:
            _td = _pcc_all[(_pcc_all["side"] == "Opponent")
                           & (_pcc_all["decode_quality"] != "Needs review")
                           & _pcc_all["defense_played"].notna()].copy()
            if "possession_side" in _td.columns:
                _td = _td[_td["possession_side"].astype(str) == "Defense"]
        _TD_MIN = 4
        if len(_td) >= _TD_MIN:
            _td["_pts"] = pd.to_numeric(_td.get("points"), errors="coerce")
            add(f'<div class="tier-h">What {_sb_esc(OPP_UP)} will play against us</div>')
            add('<table class="compact"><thead><tr><th style="text-align:left">Situation</th><th>Poss</th>'
                '<th style="text-align:left">Base defense</th><th style="text-align:left">Changes to</th>'
                '<th>PPP allowed</th></tr></thead><tbody>')
            # Overall first, then each situation that has enough possessions to say something.
            _groups = [("All", _td)]
            if "play_situation" in _td.columns:
                _so = ["Half court", "BLOB", "SLOB", "ATO"]
                for _sit in _so + [x for x in _td["play_situation"].dropna().astype(str).unique() if x not in _so]:
                    _sg = _td[_td["play_situation"].astype(str) == _sit]
                    if len(_sg) >= _TD_MIN:
                        _groups.append((_sit, _sg))
            for _lbl, _g in _groups:
                _vc = _g["defense_played"].astype(str).value_counts()
                _base_txt = f"{_vc.index[0]} ({round(100 * int(_vc.iloc[0]) / len(_g))}%)"
                _chg = (f"{_vc.index[1]} ({round(100 * int(_vc.iloc[1]) / len(_g))}%)" if len(_vc) > 1 else "--")
                _known = _g["_pts"].notna().sum()
                _ppp = (_g["_pts"].sum() / _known) if _known else None
                add(f'<tr class="{"serrow" if _lbl == "All" else "setrow"}"><td class="wrap">{_sb_esc(_lbl)}</td>'
                    f'<td>{len(_g)}</td><td class="wrap">{_sb_esc(_base_txt)}</td>'
                    f'<td class="wrap mini">{_sb_esc(_chg)}</td><td>{_sb_fmt(_ppp, 2)}</td></tr>')
            add("</tbody></table>")

            _lines = []
            # Ball-screen coverage, with what it gives up -- the coverage that leaks most is the one to attack.
            if "coverage_played" in _td.columns:
                _cv = _td[_td["coverage_played"].astype(str).str.strip() != ""]
                if not _cv.empty:
                    _cvs = []
                    for _cn, _cg in _cv.groupby(_cv["coverage_played"].astype(str)):
                        _k = _cg["_pts"].notna().sum()
                        _cvs.append((_cn, len(_cg), (_cg["_pts"].sum() / _k) if _k else None))
                    _cvs.sort(key=lambda x: -x[1])
                    _lines.append("<strong>Ball screens:</strong> " + ", ".join(
                        f"{_sb_esc(n)} {c}x" + (f" ({p:.2f} allowed)" if p is not None else "")
                        for n, c, p in _cvs[:4]))
                    _leak = [x for x in _cvs if x[1] >= 3 and x[2] is not None]
                    if len(_leak) >= 2:
                        _worst_cov = max(_leak, key=lambda x: x[2])
                        _lines[-1] += (f" \u2014 attack <strong>{_sb_esc(_worst_cov[0])}</strong>, "
                                       f"it gives up the most.")
            if "press_played" in _td.columns and int(_td["press_played"].astype(bool).sum()):
                _pr = _td[_td["press_played"].astype(bool)]
                _pf = (_pr["press_formation_played"].dropna().astype(str).value_counts()
                       if "press_formation_played" in _pr.columns else pd.Series(dtype=int))
                _lines.append(f"<strong>Pressure:</strong> pressed on {len(_pr)} of {len(_td)} possessions"
                              + (f" ({_sb_esc(', '.join(_pf.index[:2]))})" if len(_pf) else "") + ".")
            # Staff's "how we attack it" notes carry over when the staff has actually written them -- the
            # generated sample text never does.
            _sd, _sd_sample = staff_rows("uww_opp_defense")
            if not _sd.empty and not _sd_sample and "how_we_attack" in _sd.columns:
                for _, _sr in _sd.iterrows():
                    _how = _sb_clean(_sr.get("how_we_attack"))
                    if _how:
                        _lines.append(f"<strong>{_sb_esc(_sb_clean(_sr.get('item')))}:</strong> "
                                      f"{_sb_esc(_how)}")
            for _l in _lines:
                add(f'<p class="strip-line">{_l}</p>')
            add('<p class="mini">Each set below shows which defense it beat \u2014 read the two together.</p>')
        else:
            add(f'<p class="mini">{_sb_esc(OPP_UP)}\'s own defense isn\'t tagged yet, so what they\'ll play '
                'against us isn\'t shown. It appears here once their defensive possessions are tagged.</p>')

        add('<div class="quad quad-rich">')
        for _qname, _qsub, _qcls, _test in _quads:
            _rows = _rated[_rated.apply(_test, axis=1)].sort_values(
                ["_ppp", "_uses"], ascending=[False, False])
            add(f'<div class="quad-cell {_qcls}">'
                f'<div class="quad-h">{_sb_esc(_qname)}<span class="quad-n">{len(_rows)}</span></div>'
                f'<div class="quad-sub">{_sb_esc(_qsub)}</div>')
            if _rows.empty:
                add('<div class="quad-empty">Nothing here yet.</div>')
            for _, r in _rows.head(_QSHOW).iterrows():
                _vs = _q_best_defense(r["name"])
                _cx = _q_context(r["name"])
                # Each line answers a different question, so they are labelled rather than run together:
                # who actually finishes it, what it runs with, when it runs, and why it scores the way it
                # does. The "who" line in particular is phrased as observed usage -- a bare name read as
                # "we should run this for him", which is not what the number says.
                _lines = []
                if _cx["who"]:
                    _lines.append(("Who", _cx["who"] + (f" &middot; best vs {_sb_esc(_vs)}" if _vs else "")))
                elif _vs:
                    _lines.append(("Who", f"best vs {_sb_esc(_vs)}"))
                if _cx["unit"]:
                    _lines.append(("Unit", _cx["unit"]))
                if _cx["lineup"]:
                    _lines.append(("Personnel", _cx["lineup"]))
                if _cx["clock"]:
                    _lines.append(("Clock", _cx["clock"]))
                if _cx["driver"]:
                    _lines.append(("Why", _cx["driver"]))
                add(f'<div class="quad-row"><span class="quad-set">{_sb_esc(_sb_clean(r["name"]))}</span>'
                    f'<span class="quad-num">{r["_ppp"]:.2f}<span class="mini"> / {int(r["_uses"])}x</span></span>'
                    + "".join(f'<div class="quad-meta"><span class="qk">{k}</span> {v}</div>'
                              for k, v in _lines)
                    + "</div>")
            if len(_rows) > _QSHOW:
                add(f'<div class="quad-more">+ {len(_rows) - _QSHOW} more</div>')
            add("</div>")
        add("</div>")
        add('<p class="mini">' + app_link("Every set, clip and spot", page="upcoming", tab="game_plan",
                                          section="play_calls_uww") + "</p>")
        add("</div>")

    offense_section_test_uww("UWW", "\U0001f3c0 HOW WE RUN OFFENSE")

    # ---- our week -------------------------------------------------------------------------------------------
    # FILM SESSION dropped from here (requested) -- its rows now badge the matching sets directly in
    # HOW {OPP} RUNS OFFENSE (see _film_clip_lookup), so this row only needs the two remaining cards.
    section("\U0001f4c5 OUR WEEK")
    add('<div class="grid2">')
    staff_table("uww_uww_availability", "\U0001fa7a AVAILABILITY",
                [("player", "Player"), ("status", "Status"), ("note", "Note")],
                "Placeholder statuses (staff_inputs/uww_availability.csv).")
    staff_table("uww_scout_team", "\U0001f3ad SCOUT TEAM",
                [("scout_player", "Our player"), ("plays_as", "Plays as"), ("imitate", "Imitate")],
                "Placeholder casting (staff_inputs/scout_team.csv).")
    add("</div>")

    # PRACTICE PLAN removed from the brief (requested) -- still built and shown in full in the app's Game
    # Plan tab. The brief's defensive recommendation and press verdict still feed it there.

    # =====================================================================================================
    # NOTES
    # =====================================================================================================
    _sb_photo_report.clear()
    _sb_photo_report.update({k: (v[0], list(v[1])) for k, v in _photo_stats.items()})
    _sb_notes_report.clear()
    _sb_notes_report.extend(footnotes)

    gl = _sb_d("uww_play_glossary")
    # The Notes list and the play-title glossary live in the APP now (requested). READ WITH CAUTION stays
    # -- it's a warning about the numbers on THIS page, not reference material, so it has to travel with
    # the brief. What's left here is a pointer: the ⓘ markers throughout already deep-link to individual
    # notes, and this card covers the two whole collections.
    if not _warn_df.empty or footnotes or not gl.empty:
        page_break()
        if not _warn_df.empty:
            section("\u26a0\ufe0f READ WITH CAUTION")
            add('<div class="card caution">')
            for _, _w in _warn_df.iterrows():
                add(f'<div class="row"><span class="area">{_sb_esc(_w.get("area"))}</span>'
                    f'<strong>{_sb_esc(_w.get("subject"))}</strong> \u2014 {_sb_esc(_w.get("detail"))}</div>')
            add("</div>")
        if footnotes or not gl.empty:
            section("\u24d8 METHODOLOGY")
            add('<div class="card"><ul class="bl">')
            if footnotes:
                add("<li>" + app_link(f"Notes ({len(footnotes)}) \u2014 how every number on this brief was built",
                                      page="upcoming", tab="notes")
                    + " \u2014 each \u24d8 in this brief links straight to its own note.</li>")
            if not gl.empty:
                add("<li>" + app_link(f"How the play titles were decoded ({len(gl)} shorthand terms)",
                                      page="upcoming", tab="notes", section="glossary")
                    + " \u2014 what each tagging shorthand reads as, and how confident that read is.</li>")
            add("</ul></div>")

    # ---- footer ---------------------------------------------------------------------------
    try:
        generated = datetime.now().strftime("%B %-d, %Y at %-I:%M %p")
    except ValueError:
        # "%-d"/"%-I" are a glibc strftime extension that Windows' CRT rejects outright -- the same
        # portability trap the play-by-play cells already hit with "%-m"/"%-d". Fall back rather
        # than let a timestamp take down the whole report.
        generated = datetime.now().strftime("%B %d, %Y at %I:%M %p")
    add('<div class="foot">')
    add(f"<p>Generated {_sb_esc(generated)} from the scouting parser's exported data \u2014 the same "
        f"tables the scouting app reads. Everything outside a red SAMPLE DATA box is reconstructed "
        f"from play-by-play, video-tagging and box-score exports.</p>")
    if _sb_d.missing:
        add(f'<p>Not available for this build: {_sb_esc(", ".join(sorted(set(_sb_d.missing))))}. '
            f'Sections relying on those sources were left out rather than filled with estimates.</p>')
    add("<p>Prepared ahead of the staff's own scouting reports. Numbers will move as more film is "
        "tagged.</p></div>")

    body = "\n".join(parts)
    return (
        '<!DOCTYPE html>\n<html lang="en">\n<head>\n<meta charset="utf-8">\n'
        '<meta name="viewport" content="width=device-width, initial-scale=1">\n'
        f"<title>{_sb_esc(_SB_UWW)} vs {_sb_esc(short)} &mdash; scouting brief</title>\n"
        f"<style>{_SB_CSS}</style>\n</head>\n<body>\n<div class=\"page\">\n{body}\n</div>\n</body>\n</html>\n"
    )


# --- Run it ------------------------------------------------------------------------------------------------
_sb_data = _SbData(APP_DATA_DIR)
_sb_game, _sb_scheduled_name, _sb_short = _sb_resolve_matchup(_sb_data)

# Prefer the name this notebook already resolved (see the "Identify the upcoming opponent" cell, including
# its no-scout-report fallback) over the one derived from the CSVs -- one source of truth for who we are
# playing, rather than two that agree right up until they don't.
_sb_notebook_short = globals().get("upcoming_opponent_short")
if _sb_notebook_short:
    _sb_short = _sb_notebook_short

if _sb_game is None:
    print("No upcoming game found in uww_schedule.csv -- no brief written. Check that the CSV export cell "
          "ran, and that reference_date falls on or before the next game you want a brief for.")
else:
    _sb_document = _sb_build_html(_sb_data, _sb_game, _sb_scheduled_name, _sb_short)
    os.makedirs(_SB_OUT_DIR, exist_ok=True)
    _sb_slug = re.sub(r"[^\w]+", "_", str(_sb_short)).strip("_") or "opponent"
    _sb_path = os.path.join(_SB_OUT_DIR, f"scouting_brief_{_sb_slug}.html")
    with open(_sb_path, "w", encoding="utf-8") as _sb_f:
        _sb_f.write(_sb_document)
    print(f"Wrote {os.path.abspath(_sb_path)} ({len(_sb_document):,} bytes) -- open it in a browser, or "
          f"attach it to an email as-is.")
    # The Notes page moved out of the brief and into the app (requested), so the app needs the text. Note
    # numbers are the deep-link target the brief's ⓘ markers point at, so they're stored explicitly rather
    # than left to row order.
    _sb_notes_df = pd.DataFrame({"note": range(1, len(_sb_notes_report) + 1),
                                 "opponent": str(_sb_short),
                                 "text": _sb_notes_report})
    _sb_notes_df.to_csv(os.path.join(APP_DATA_DIR, "uww_brief_notes.csv"), index=False)
    print(f"  {len(_sb_notes_df)} methodology note(s) -> uww_brief_notes.csv (the brief links to these "
          f"instead of printing them).")
    if not str(globals().get("APP_BASE_URL") or os.environ.get("UWW_APP_URL") or "").strip():
        print("  NOTE: APP_BASE_URL is blank, so the brief's \u24d8 markers and the Methodology links are "
              "omitted entirely -- with the notes no longer printed in the brief, there is currently no "
              "way for a reader to reach them. Set APP_BASE_URL in the config cell.")
    for _side, (_hit, _miss) in _sb_photo_report.items():
        if _hit or _miss:
            print(f"  {_side} headshots: {_hit} found, {len(_miss)} missing"
                  + (f" -- {', '.join(_miss[:6])}{' ...' if len(_miss) > 6 else ''}" if _miss else "")
                  + ("" if _hit else " (no live roster scrape for this team yet, and no files in "
                                    "data/player_images -- see the Roster pages cell)"))
    if _sb_data.missing:
        print("  Tables that were empty or missing (their sections were left out of the brief): "
              + ", ".join(sorted(set(_sb_data.missing))))


Wrote C:\Users\frits\OneDrive\Documents\GitHub\uwwmensbball\data\scouting_briefs\scouting_brief_Aurora_Spartans.html (474,649 bytes) -- open it in a browser, or attach it to an email as-is.
  21 methodology note(s) -> uww_brief_notes.csv (the brief links to these instead of printing them).
  Opponent headshots: 10 found, 0 missing
  UWW headshots: 11 found, 0 missing
